In [1]:
# ============================================================
# PROJECT 10 — CELL 0
# SAFE PARALLEL-RUNTIME BOOTSTRAP
#
# Run only in the NEW Project 10 notebook.
#
# Reads:
# - completion registry
# - Project 9 progress/status
#
# Writes only:
# - Results/Aggregated/project_10_selection/
#
# Does NOT modify:
# - Project 9
# - Projects 1–8
# - completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import os
import socket
import time

import pandas as pd


# ------------------------------------------------------------
# 1. MOUNT DRIVE IF THE UI MOUNT WAS NOT COMPLETED
# ------------------------------------------------------------

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)


# Project 9 paths are READ-ONLY in this notebook.
PROJECT_9_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "camunda__camunda-bpm-platform"
)

PROJECT_9_STATUS_PATH = (
    PROJECT_9_DIR
    / "camunda_step5a_status.json"
)

PROJECT_9_PROGRESS_PATH = (
    PROJECT_9_DIR
    / "camunda_full_run_control"
    / "camunda_full_run_progress.json"
)


# The only persistent directory this cell may write.
PROJECT_10_BOOTSTRAP_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "project_10_selection"
)

PROJECT_10_BOOTSTRAP_PATH = (
    PROJECT_10_BOOTSTRAP_DIR
    / "project_10_runtime_bootstrap.json"
)


PASS_STATUS = (
    "PASS_PROJECT_10_SAFE_PARALLEL_RUNTIME_BOOTSTRAP"
)


print("=" * 108)
print("=== PROJECT 10 CELL 0: SAFE PARALLEL-RUNTIME BOOTSTRAP ===")
print("=" * 108)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def read_json_with_retry(
    path,
    attempts=8,
    delay_seconds=0.5,
):
    path = Path(path)

    last_error = None

    for _ in range(attempts):

        try:

            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:

            last_error = error

            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON file.\n"
        f"Path: {path}\n"
        f"Last error: {type(last_error).__name__}: {last_error}"
    )


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


# ------------------------------------------------------------
# 4. VERIFY THESIS ROOT
# ------------------------------------------------------------

required_paths = [
    THESIS_ROOT,
    NOTES_DIR,
    RESULTS_DIR,
    REGISTRY_PATH,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(
        "Required thesis paths are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


# ------------------------------------------------------------
# 5. VERIFY THE COMPLETION REGISTRY READ-ONLY
# ------------------------------------------------------------

registry_sha256_before = calculate_sha256(
    REGISTRY_PATH
)


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


if "ProjectNumber" not in registry.columns:

    raise AssertionError(
        "Registry lacks ProjectNumber column."
    )


project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if len(registry) != 8:

    raise AssertionError(
        "Registry must currently contain eight completed projects.\n"
        f"Actual rows: {len(registry)}"
    )


if set(project_numbers) != set(range(1, 9)):

    raise AssertionError(
        "Registry must contain exactly Projects 1–8."
    )


if project_numbers.eq(9).any():

    raise AssertionError(
        "Project 9 was unexpectedly registered before finalisation."
    )


if project_numbers.eq(10).any():

    raise AssertionError(
        "Project 10 is already present in the completion registry."
    )


# ------------------------------------------------------------
# 6. READ PROJECT 9 STATUS — NO WRITES
# ------------------------------------------------------------

project_9_status = {}

project_9_progress = {}


if PROJECT_9_STATUS_PATH.exists():

    project_9_status = read_json_with_retry(
        PROJECT_9_STATUS_PATH
    )


if PROJECT_9_PROGRESS_PATH.exists():

    project_9_progress = read_json_with_retry(
        PROJECT_9_PROGRESS_PATH
    )


project_9_status_value = (
    project_9_progress.get(
        "Status",
        project_9_status.get(
            "Status",
            "STATUS_FILE_NOT_YET_AVAILABLE",
        ),
    )
)


project_9_completed_conditions = (
    project_9_progress.get(
        "CompletedConditions",
        project_9_status.get(
            "CompletedConditions",
        ),
    )
)


project_9_pending_conditions = (
    project_9_progress.get(
        "PendingConditions",
        project_9_status.get(
            "PendingConditions",
        ),
    )
)


project_9_last_completed_condition = (
    project_9_progress.get(
        "LastCompletedCondition",
        project_9_status.get(
            "LastCompletedCondition",
        ),
    )
)


# ------------------------------------------------------------
# 7. RECORD PROJECT 10 RUNTIME IDENTITY
# ------------------------------------------------------------

runtime_hostname = socket.gethostname()

runtime_process_id = os.getpid()


boot_id_path = Path(
    "/proc/sys/kernel/random/boot_id"
)


if boot_id_path.exists():

    runtime_boot_id = (
        boot_id_path.read_text(
            encoding="utf-8"
        )
        .strip()
    )

else:

    runtime_boot_id = "UNAVAILABLE"


created_at_utc = datetime.now(
    timezone.utc
).isoformat()


# Local runtime marker. This stays only inside this Project 10 VM.
local_marker_path = Path(
    "/content/project_10_runtime_marker.json"
)


atomic_write_json(
    local_marker_path,
    {
        "Worker":
            "PROJECT_10",

        "Hostname":
            runtime_hostname,

        "ProcessID":
            runtime_process_id,

        "RuntimeBootID":
            runtime_boot_id,

        "CreatedAtUTC":
            created_at_utc,
    },
)


# ------------------------------------------------------------
# 8. WRITE ONLY THE ISOLATED PROJECT 10 BOOTSTRAP
# ------------------------------------------------------------

# Safety checks against accidental Project 9 writes.
if PROJECT_9_DIR in PROJECT_10_BOOTSTRAP_PATH.parents:

    raise AssertionError(
        "Project 10 bootstrap path overlaps Project 9."
    )


if "project_10_selection" not in str(
    PROJECT_10_BOOTSTRAP_PATH
):

    raise AssertionError(
        "Project 10 bootstrap path is not isolated."
    )


bootstrap_payload = {
    "ProjectNumber":
        10,

    "Worker":
        "PROJECT_10_PARALLEL_WORKER",

    "Status":
        PASS_STATUS,

    "RuntimeHostname":
        runtime_hostname,

    "RuntimeProcessID":
        runtime_process_id,

    "RuntimeBootID":
        runtime_boot_id,

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistryRows":
        len(
            registry
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "RegisteredProjects":
        sorted(
            project_numbers.tolist()
        ),

    "Project9Status":
        project_9_status_value,

    "Project9CompletedConditions":
        project_9_completed_conditions,

    "Project9PendingConditions":
        project_9_pending_conditions,

    "Project9LastCompletedCondition":
        project_9_last_completed_condition,

    "Project9Modified":
        False,

    "Projects1To8Modified":
        False,

    "CompletionRegistryModified":
        False,

    "Project10BootstrapDirectory":
        str(
            PROJECT_10_BOOTSTRAP_DIR
        ),

    "CreatedAtUTC":
        created_at_utc,
}


atomic_write_json(
    PROJECT_10_BOOTSTRAP_PATH,
    bootstrap_payload,
)


bootstrap_readback = read_json_with_retry(
    PROJECT_10_BOOTSTRAP_PATH
)


if bootstrap_readback.get(
    "Status"
) != PASS_STATUS:

    raise AssertionError(
        "Project 10 bootstrap readback failed."
    )


# ------------------------------------------------------------
# 9. CONFIRM THE REGISTRY WAS NOT MODIFIED
# ------------------------------------------------------------

registry_sha256_after = calculate_sha256(
    REGISTRY_PATH
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:

    raise AssertionError(
        "Completion registry changed during Project 10 bootstrap."
    )


# ------------------------------------------------------------
# 10. RESULT
# ------------------------------------------------------------

print("\nProject 10 runtime identity:")

print(
    "Hostname:",
    runtime_hostname,
)

print(
    "Process ID:",
    runtime_process_id,
)

print(
    "Runtime boot ID:",
    runtime_boot_id,
)


print("\nCompletion registry:")

print(
    "Registered projects:",
    sorted(
        project_numbers.tolist()
    ),
)

print(
    "Registry unchanged:",
    registry_unchanged,
)


print("\nProject 9 read-only snapshot:")

print(
    "Status:",
    project_9_status_value,
)

print(
    "Completed conditions:",
    project_9_completed_conditions,
)

print(
    "Pending conditions:",
    project_9_pending_conditions,
)

print(
    "Last completed condition:",
    project_9_last_completed_condition,
)


print("\nProject 10 bootstrap:")

print(
    PROJECT_10_BOOTSTRAP_PATH
)

print(
    "Bootstrap SHA-256:",
    calculate_sha256(
        PROJECT_10_BOOTSTRAP_PATH
    ),
)


print("\nProject 9 modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)

print("\nCompletion registry modified:")
print(0)


print(
    "\nSTATUS:",
    PASS_STATUS,
)

print("=" * 108)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== PROJECT 10 CELL 0: SAFE PARALLEL-RUNTIME BOOTSTRAP ===

Project 10 runtime identity:
Hostname: 2451a5221ded
Process ID: 2705
Runtime boot ID: 686350ca-5944-4e1b-94f7-5c06cf7cb442

Completion registry:
Registered projects: [1, 2, 3, 4, 5, 6, 7, 8]
Registry unchanged: True

Project 9 read-only snapshot:
Status: RUNNING_PROJECT_9_FULL_270_CONDITION_EXPERIMENT
Completed conditions: 196
Pending conditions: 74
Last completed condition: noise_30__seed_22

Project 10 bootstrap:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/project_10_selection/project_10_runtime_bootstrap.json
Bootstrap SHA-256: d98f0270c4ec0279c1c84c025546a7baa0e1f5bcc8f0d3a19211344f5e2e009c

Project 9 modified:
0

Projects 1–8 modified:
0

Completion registry modified:
0

STATUS: PASS_PROJECT_10_SAFE_PARALLEL_RUNTIME_BOOTSTRAP


In [2]:
# ============================================================
# PROJECT 10 — CELL 1 / STEP 1A
# LOCAL SOURCE EXTRACTION AND PROVISIONAL SELECTION
#
# This cell reuses the frozen Project 9 candidate scan.
#
# Expected provisional Project 10:
#   spring-cloud@spring-cloud-dataflow
#
# Writes only:
#   Results/Aggregated/project_10_selection/
#
# Local runtime writes:
#   /content/project_10_local/
#   /content/datasets/spring-cloud@spring-cloud-dataflow/
#
# Does NOT:
# - modify Project 9
# - modify Projects 1–8
# - modify the completion registry
# - finalise Project 10
# ============================================================

from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import shutil
import tarfile
import tempfile
import time

import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 10

EXPECTED_BOOTSTRAP_STATUS = (
    "PASS_PROJECT_10_SAFE_PARALLEL_RUNTIME_BOOTSTRAP"
)

PASS_STATUS = (
    "PASS_PROJECT_10_LOCAL_SOURCE_AND_PROVISIONAL_SELECTION_READY"
)

EXPECTED_PROJECT_9 = (
    "camunda@camunda-bpm-platform"
)

EXPECTED_PROJECT_10 = (
    "spring-cloud@spring-cloud-dataflow"
)

EXPECTED_PROJECT_10_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

EXPECTED_ARCHIVE_MD5 = (
    "728804085c757ff5357aa165b4b6384f"
)


REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "id_map.csv",
    "entity_change_history.csv",
]


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

ARCHIVE_DRIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)


PROJECT_9_SELECTION_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "project_09_selection"
)

PROJECT_9_RANKED_CANDIDATES_PATH = (
    PROJECT_9_SELECTION_DIR
    / "project_09_eligible_candidates_ranked.csv"
)

PROJECT_9_PROVISIONAL_SELECTION_PATH = (
    PROJECT_9_SELECTION_DIR
    / "project_09_provisional_selection.json"
)

PROJECT_9_SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_selection_checkpoint.json"
)


PROJECT_10_SELECTION_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "project_10_selection"
)

PROJECT_10_BOOTSTRAP_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_runtime_bootstrap.json"
)

PROJECT_10_AVAILABLE_CANDIDATES_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_available_candidates_ranked.csv"
)

PROJECT_10_SOURCE_MANIFEST_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_local_source_manifest.csv"
)

PROJECT_10_PROVISIONAL_SELECTION_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_provisional_selection.json"
)

PROJECT_10_STEP1A_REPORT_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_step1a_report.json"
)

PROJECT_10_STEP1A_STATUS_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_step1a_status.json"
)


LOCAL_WORK_ROOT = Path(
    "/content/project_10_local"
)

LOCAL_ARCHIVE_PATH = (
    LOCAL_WORK_ROOT
    / "TCP-CI-main-dataset.tar.gz"
)

LOCAL_DATASETS_ROOT = Path(
    "/content/datasets"
)


print("=" * 116)
print("=== PROJECT 10 CELL 1 / STEP 1A: LOCAL SOURCE EXTRACTION AND PROVISIONAL SELECTION ===")
print("=" * 116)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open(
        "rb"
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def atomic_write_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json_with_retry(
    path,
    attempts=8,
    delay_seconds=0.5,
):
    path = Path(
        path
    )

    last_error = None

    for _ in range(
        attempts
    ):

        try:

            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:

            last_error = error

            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON.\n"
        f"Path: {path}\n"
        f"Error: {type(last_error).__name__}: {last_error}"
    )


def normalise_project_slug(
    project_name,
):
    return (
        str(
            project_name
        )
        .replace(
            "@",
            "__",
        )
        .replace(
            "/",
            "__",
        )
    )


def create_source_manifest(
    source_directory,
):
    source_directory = Path(
        source_directory
    )

    source_files = sorted(
        [
            path
            for path in source_directory.rglob(
                "*"
            )
            if path.is_file()
        ],
        key=lambda path:
            path.relative_to(
                source_directory
            ).as_posix(),
    )

    records = []

    root_digest = hashlib.sha256()

    for file_order, path in enumerate(
        source_files,
        start=1,
    ):

        relative_path = (
            path.relative_to(
                source_directory
            ).as_posix()
        )

        size_bytes = int(
            path.stat().st_size
        )

        file_sha256 = calculate_hash(
            path,
            algorithm="sha256",
        )

        records.append({
            "FileOrder":
                file_order,

            "RelativePath":
                relative_path,

            "SizeBytes":
                size_bytes,

            "SHA256":
                file_sha256,
        })

        root_line = (
            f"{relative_path}\0"
            f"{size_bytes}\0"
            f"{file_sha256}\n"
        )

        root_digest.update(
            root_line.encode(
                "utf-8"
            )
        )

    return (
        pd.DataFrame(
            records
        ),
        root_digest.hexdigest(),
    )


def safe_extract_selected_project(
    archive_path,
    project_name,
    destination_root,
):
    archive_path = Path(
        archive_path
    )

    destination_root = Path(
        destination_root
    )

    extraction_temp = Path(
        tempfile.mkdtemp(
            prefix="project10_extract_",
            dir="/content",
        )
    )

    selected_members = []

    try:

        with tarfile.open(
            archive_path,
            mode="r:gz",
        ) as archive:

            for member in archive.getmembers():

                member_path = PurePosixPath(
                    member.name
                )

                member_parts = (
                    member_path.parts
                )

                if project_name not in member_parts:
                    continue

                if member_path.is_absolute():

                    raise RuntimeError(
                        "Archive contains an absolute path."
                    )

                if ".." in member_parts:

                    raise RuntimeError(
                        "Archive contains path traversal."
                    )

                if member.issym() or member.islnk():

                    raise RuntimeError(
                        "Project archive contains a symbolic or hard link."
                    )

                if not (
                    member.isfile()
                    or member.isdir()
                ):

                    continue

                selected_members.append(
                    member
                )

            if not selected_members:

                raise FileNotFoundError(
                    "The selected project directory was not found "
                    "inside the dataset archive.\n"
                    f"Project: {project_name}"
                )

            archive.extractall(
                path=extraction_temp,
                members=selected_members,
                filter="data",
            )

        extracted_candidates = [
            path
            for path in extraction_temp.rglob(
                "*"
            )
            if (
                path.is_dir()
                and path.name == project_name
            )
        ]

        if len(
            extracted_candidates
        ) != 1:

            raise RuntimeError(
                "Could not locate exactly one extracted project directory.\n"
                f"Matches: {[str(path) for path in extracted_candidates]}"
            )

        extracted_source = (
            extracted_candidates[
                0
            ]
        )

        target_source = (
            destination_root
            / project_name
        )

        target_source.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if target_source.exists():

            shutil.rmtree(
                target_source
            )

        shutil.copytree(
            extracted_source,
            target_source,
        )

        return (
            target_source,
            len(
                selected_members
            ),
        )

    finally:

        if extraction_temp.exists():

            shutil.rmtree(
                extraction_temp,
                ignore_errors=True,
            )


# ------------------------------------------------------------
# 4. VALIDATE REQUIRED INPUTS
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    ARCHIVE_DRIVE_PATH,
    PROJECT_10_BOOTSTRAP_PATH,
    PROJECT_9_RANKED_CANDIDATES_PATH,
    PROJECT_9_PROVISIONAL_SELECTION_PATH,
    PROJECT_9_SELECTION_CHECKPOINT_PATH,
]


missing_inputs = [
    str(
        path
    )
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:

    raise FileNotFoundError(
        "Required Project 10 inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


bootstrap = read_json_with_retry(
    PROJECT_10_BOOTSTRAP_PATH
)


if bootstrap.get(
    "Status"
) != EXPECTED_BOOTSTRAP_STATUS:

    raise AssertionError(
        "Project 10 bootstrap status differs.\n"
        f"Expected: {EXPECTED_BOOTSTRAP_STATUS}\n"
        f"Actual:   {bootstrap.get('Status')}"
    )


# ------------------------------------------------------------
# 5. READ REGISTRY WITHOUT MODIFYING IT
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


if "ProjectNumber" not in registry.columns:

    raise AssertionError(
        "Registry lacks ProjectNumber."
    )


registered_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != 8
    or set(
        registered_numbers
    ) != set(
        range(
            1,
            9,
        )
    )
):

    raise AssertionError(
        "Registry must contain exactly Projects 1–8."
    )


registry_project_column = None


for candidate_column in [
    "Project",
    "ProjectName",
    "project",
]:

    if candidate_column in registry.columns:

        registry_project_column = (
            candidate_column
        )

        break


if registry_project_column is None:

    raise AssertionError(
        "Could not resolve the project-name column in the registry."
    )


registered_project_names = set(
    registry[
        registry_project_column
    ].astype(str)
)


# ------------------------------------------------------------
# 6. VERIFY PROJECT 9 FROZEN SELECTION
# ------------------------------------------------------------

project_9_provisional = read_json_with_retry(
    PROJECT_9_PROVISIONAL_SELECTION_PATH
)

project_9_checkpoint = read_json_with_retry(
    PROJECT_9_SELECTION_CHECKPOINT_PATH
)


project_9_name = (
    project_9_checkpoint.get(
        "Project"
    )
    or project_9_provisional.get(
        "Project"
    )
)


if project_9_name != EXPECTED_PROJECT_9:

    raise AssertionError(
        "Frozen Project 9 selection differs.\n"
        f"Expected: {EXPECTED_PROJECT_9}\n"
        f"Actual:   {project_9_name}"
    )


project_9_candidate_file_sha256 = (
    calculate_hash(
        PROJECT_9_RANKED_CANDIDATES_PATH,
        algorithm="sha256",
    )
)

project_9_selection_checkpoint_sha256 = (
    calculate_hash(
        PROJECT_9_SELECTION_CHECKPOINT_PATH,
        algorithm="sha256",
    )
)


# ------------------------------------------------------------
# 7. SELECT THE NEXT AVAILABLE RANKED CANDIDATE
# ------------------------------------------------------------

ranked_candidates = pd.read_csv(
    PROJECT_9_RANKED_CANDIDATES_PATH,
    low_memory=False,
)


required_candidate_columns = {
    "CandidateRank",
    "Project",
    "ProjectSlug",
}


missing_candidate_columns = (
    required_candidate_columns
    - set(
        ranked_candidates.columns
    )
)


if missing_candidate_columns:

    raise AssertionError(
        "Ranked candidate file lacks required columns:\n"
        + "\n".join(
            sorted(
                missing_candidate_columns
            )
        )
    )


ranked_candidates[
    "CandidateRank"
] = pd.to_numeric(
    ranked_candidates[
        "CandidateRank"
    ],
    errors="raise",
).astype(int)


ranked_candidates = (
    ranked_candidates.sort_values(
        "CandidateRank",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


excluded_projects = (
    registered_project_names
    | {
        project_9_name,
    }
)


available_candidates = (
    ranked_candidates[
        ~ranked_candidates[
            "Project"
        ].astype(str).isin(
            excluded_projects
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


available_candidates.insert(
    0,
    "Project10AvailableRank",
    range(
        1,
        len(
            available_candidates
        ) + 1,
    ),
)


if available_candidates.empty:

    raise RuntimeError(
        "No ranked candidates remain for Project 10."
    )


selected_candidate = (
    available_candidates.iloc[
        0
    ]
)


selected_project = str(
    selected_candidate[
        "Project"
    ]
)

selected_project_slug = str(
    selected_candidate[
        "ProjectSlug"
    ]
)


if selected_project != EXPECTED_PROJECT_10:

    raise AssertionError(
        "The next deterministic candidate differs.\n"
        f"Expected: {EXPECTED_PROJECT_10}\n"
        f"Actual:   {selected_project}"
    )


if (
    selected_project_slug
    != EXPECTED_PROJECT_10_SLUG
):

    raise AssertionError(
        "Project 10 slug differs.\n"
        f"Expected: {EXPECTED_PROJECT_10_SLUG}\n"
        f"Actual:   {selected_project_slug}"
    )


if (
    normalise_project_slug(
        selected_project
    )
    != selected_project_slug
):

    raise AssertionError(
        "Project name and slug are inconsistent."
    )


atomic_write_csv(
    PROJECT_10_AVAILABLE_CANDIDATES_PATH,
    available_candidates,
)


print("\nProvisional Project 10 selection:")

print(
    "Original candidate rank:",
    int(
        selected_candidate[
            "CandidateRank"
        ]
    ),
)

print(
    "Project 10 available rank:",
    int(
        selected_candidate[
            "Project10AvailableRank"
        ]
    ),
)

print(
    "Project:",
    selected_project,
)

print(
    "Project slug:",
    selected_project_slug,
)


# ------------------------------------------------------------
# 8. COPY THE IMMUTABLE ARCHIVE LOCALLY
# ------------------------------------------------------------

LOCAL_WORK_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

LOCAL_DATASETS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


drive_archive_md5 = calculate_hash(
    ARCHIVE_DRIVE_PATH,
    algorithm="md5",
)


if drive_archive_md5 != EXPECTED_ARCHIVE_MD5:

    raise AssertionError(
        "Google Drive dataset archive MD5 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_MD5}\n"
        f"Actual:   {drive_archive_md5}"
    )


copy_required = True


if LOCAL_ARCHIVE_PATH.exists():

    local_existing_md5 = calculate_hash(
        LOCAL_ARCHIVE_PATH,
        algorithm="md5",
    )

    copy_required = (
        local_existing_md5
        != EXPECTED_ARCHIVE_MD5
    )


if copy_required:

    print(
        "\nCopying immutable archive from Drive to "
        "the Project 10 local runtime..."
    )

    temporary_local_archive = (
        LOCAL_ARCHIVE_PATH.with_name(
            LOCAL_ARCHIVE_PATH.name
            + ".copying"
        )
    )

    if temporary_local_archive.exists():

        temporary_local_archive.unlink()

    shutil.copy2(
        ARCHIVE_DRIVE_PATH,
        temporary_local_archive,
    )

    copied_md5 = calculate_hash(
        temporary_local_archive,
        algorithm="md5",
    )

    if copied_md5 != EXPECTED_ARCHIVE_MD5:

        temporary_local_archive.unlink(
            missing_ok=True
        )

        raise AssertionError(
            "Locally copied archive MD5 differs."
        )

    os.replace(
        temporary_local_archive,
        LOCAL_ARCHIVE_PATH,
    )


local_archive_md5 = calculate_hash(
    LOCAL_ARCHIVE_PATH,
    algorithm="md5",
)


if local_archive_md5 != EXPECTED_ARCHIVE_MD5:

    raise AssertionError(
        "Local archive MD5 differs after copy."
    )


local_archive_sha256 = calculate_hash(
    LOCAL_ARCHIVE_PATH,
    algorithm="sha256",
)


print(
    "Archive MD5:",
    local_archive_md5,
)

print(
    "Archive SHA-256:",
    local_archive_sha256,
)


# ------------------------------------------------------------
# 9. EXTRACT ONLY THE PROJECT 10 DIRECTORY LOCALLY
# ------------------------------------------------------------

local_project_source = (
    LOCAL_DATASETS_ROOT
    / selected_project
)


existing_required_files = all(
    (
        local_project_source
        / filename
    ).exists()
    for filename in REQUIRED_SOURCE_FILES
)


if existing_required_files:

    selected_archive_members = None

    print(
        "\nValid local Project 10 source already exists. "
        "Extraction skipped."
    )

else:

    print(
        "\nExtracting only the Project 10 source directory..."
    )

    (
        local_project_source,
        selected_archive_members,
    ) = safe_extract_selected_project(
        archive_path=(
            LOCAL_ARCHIVE_PATH
        ),

        project_name=(
            selected_project
        ),

        destination_root=(
            LOCAL_DATASETS_ROOT
        ),
    )


if (
    LOCAL_DATASETS_ROOT
    not in local_project_source.parents
):

    raise AssertionError(
        "Local Project 10 source escaped /content/datasets."
    )


missing_required_source_files = [
    filename
    for filename in REQUIRED_SOURCE_FILES
    if not (
        local_project_source
        / filename
    ).exists()
]


if missing_required_source_files:

    raise FileNotFoundError(
        "Project 10 is missing required source files:\n"
        + "\n".join(
            missing_required_source_files
        )
    )


print(
    "Local source:",
    local_project_source,
)

print(
    "Required source files:",
    len(
        REQUIRED_SOURCE_FILES
    ),
    "/",
    len(
        REQUIRED_SOURCE_FILES
    ),
)


# ------------------------------------------------------------
# 10. CREATE SOURCE MANIFEST AND ROOT HASH
# ------------------------------------------------------------

(
    source_manifest,
    source_root_sha256,
) = create_source_manifest(
    local_project_source
)


if source_manifest.empty:

    raise AssertionError(
        "Project 10 source manifest is empty."
    )


source_file_count = len(
    source_manifest
)

source_total_bytes = int(
    source_manifest[
        "SizeBytes"
    ].sum()
)


atomic_write_csv(
    PROJECT_10_SOURCE_MANIFEST_PATH,
    source_manifest,
)


manifest_readback = pd.read_csv(
    PROJECT_10_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


if len(
    manifest_readback
) != source_file_count:

    raise AssertionError(
        "Source manifest readback row count differs."
    )


# ------------------------------------------------------------
# 11. WRITE PROVISIONAL SELECTION
# ------------------------------------------------------------

candidate_payload = {}


for column in available_candidates.columns:

    value = selected_candidate[
        column
    ]

    if isinstance(
        value,
        (
            int,
            float,
        ),
    ):

        if pd.isna(
            value
        ):

            candidate_payload[
                column
            ] = None

        else:

            candidate_payload[
                column
            ] = value.item() if hasattr(
                value,
                "item"
            ) else value

    else:

        candidate_payload[
            column
        ] = str(
            value
        )


provisional_selection_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        selected_project,

    "ProjectSlug":
        selected_project_slug,

    "Status":
        "PROVISIONAL_PROJECT_10_SELECTION",

    "SelectionBasis":
        (
            "First remaining eligible candidate from the "
            "frozen Project 9 candidate ranking after excluding "
            "registered Projects 1–8 and active Project 9."
        ),

    "OriginalCandidateRank":
        int(
            selected_candidate[
                "CandidateRank"
            ]
        ),

    "Project10AvailableRank":
        int(
            selected_candidate[
                "Project10AvailableRank"
            ]
        ),

    "CandidateMetrics":
        candidate_payload,

    "LocalSourceDirectory":
        str(
            local_project_source
        ),

    "RequiredSourceFiles":
        REQUIRED_SOURCE_FILES,

    "SourceFileCount":
        source_file_count,

    "SourceBytes":
        source_total_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "SourceManifest":
        str(
            PROJECT_10_SOURCE_MANIFEST_PATH
        ),

    "SourceManifestSHA256":
        calculate_hash(
            PROJECT_10_SOURCE_MANIFEST_PATH,
            algorithm="sha256",
        ),

    "DatasetArchiveDrivePath":
        str(
            ARCHIVE_DRIVE_PATH
        ),

    "DatasetArchiveLocalPath":
        str(
            LOCAL_ARCHIVE_PATH
        ),

    "DatasetArchiveMD5":
        local_archive_md5,

    "DatasetArchiveSHA256":
        local_archive_sha256,

    "Project9":
        project_9_name,

    "Project9SelectionCheckpointSHA256":
        project_9_selection_checkpoint_sha256,

    "Project9CandidateRankingSHA256":
        project_9_candidate_file_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "CompletionRegistryModified":
        False,

    "Project9Modified":
        False,

    "Projects1To8Modified":
        False,

    "CreatedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    PROJECT_10_PROVISIONAL_SELECTION_PATH,
    provisional_selection_payload,
)


# ------------------------------------------------------------
# 12. VALIDATE READBACK
# ------------------------------------------------------------

provisional_readback = read_json_with_retry(
    PROJECT_10_PROVISIONAL_SELECTION_PATH
)


if provisional_readback.get(
    "Project"
) != EXPECTED_PROJECT_10:

    raise AssertionError(
        "Provisional Project 10 readback differs."
    )


if provisional_readback.get(
    "SourceRootSHA256"
) != source_root_sha256:

    raise AssertionError(
        "Provisional source-root hash readback differs."
    )


registry_sha256_after = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:

    raise AssertionError(
        "Completion registry changed during Project 10 Step 1A."
    )


# Project 9 frozen selection files must remain unchanged.
if (
    calculate_hash(
        PROJECT_9_RANKED_CANDIDATES_PATH,
        algorithm="sha256",
    )
    != project_9_candidate_file_sha256
):

    raise AssertionError(
        "Project 9 candidate ranking changed."
    )


if (
    calculate_hash(
        PROJECT_9_SELECTION_CHECKPOINT_PATH,
        algorithm="sha256",
    )
    != project_9_selection_checkpoint_sha256
):

    raise AssertionError(
        "Project 9 selection checkpoint changed."
    )


# ------------------------------------------------------------
# 13. REPORT AND STATUS
# ------------------------------------------------------------

report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        selected_project,

    "ProjectSlug":
        selected_project_slug,

    "Status":
        PASS_STATUS,

    "OriginalCandidateRank":
        int(
            selected_candidate[
                "CandidateRank"
            ]
        ),

    "AvailableCandidates":
        len(
            available_candidates
        ),

    "LocalSourceDirectory":
        str(
            local_project_source
        ),

    "SourceFileCount":
        source_file_count,

    "SourceBytes":
        source_total_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "RequiredFilesPresent":
        True,

    "ArchiveMD5":
        local_archive_md5,

    "ArchiveSHA256":
        local_archive_sha256,

    "RegistryUnchanged":
        registry_unchanged,

    "Project9Modified":
        False,

    "Projects1To8Modified":
        False,

    "CompletionRegistryModified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    PROJECT_10_STEP1A_REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        selected_project,

    "ProjectSlug":
        selected_project_slug,

    "Status":
        PASS_STATUS,

    "LocalSourceDirectory":
        str(
            local_project_source
        ),

    "SourceRootSHA256":
        source_root_sha256,

    "ProvisionalSelection":
        str(
            PROJECT_10_PROVISIONAL_SELECTION_PATH
        ),

    "ProvisionalSelectionSHA256":
        calculate_hash(
            PROJECT_10_PROVISIONAL_SELECTION_PATH,
            algorithm="sha256",
        ),

    "CompletionRegistryModified":
        False,

    "Project9Modified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    PROJECT_10_STEP1A_STATUS_PATH,
    status_payload,
)


status_readback = read_json_with_retry(
    PROJECT_10_STEP1A_STATUS_PATH
)


if status_readback.get(
    "Status"
) != PASS_STATUS:

    raise AssertionError(
        "Project 10 Step 1A status readback differs."
    )


# ------------------------------------------------------------
# 14. DISPLAY
# ------------------------------------------------------------

print("\nRemaining ranked candidates:")

display(
    available_candidates.head(
        10
    )
)


print("\nProject 10 source manifest:")

display(
    source_manifest
)


print("\n")
print("=" * 116)
print("=== PROJECT 10 CELL 1 / STEP 1A RESULT ===")
print("=" * 116)

print("\nProvisional Project 10:")

print(
    "Project:",
    selected_project,
)

print(
    "Project slug:",
    selected_project_slug,
)

print(
    "Original candidate rank:",
    int(
        selected_candidate[
            "CandidateRank"
        ]
    ),
)

print(
    "Project 10 available rank:",
    int(
        selected_candidate[
            "Project10AvailableRank"
        ]
    ),
)


print("\nCandidate dimensions:")

dimension_columns = [
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingEvaluationBuilds",
]


for column in dimension_columns:

    if column in selected_candidate.index:

        print(
            f"{column}:",
            selected_candidate[
                column
            ],
        )


print("\nLocal source:")

print(
    local_project_source
)

print(
    "Source files:",
    source_file_count,
)

print(
    "Source bytes:",
    source_total_bytes,
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


print("\nDataset archive:")

print(
    "MD5:",
    local_archive_md5,
)

print(
    "SHA-256:",
    local_archive_sha256,
)


print("\nSaved outputs:")

for output_path in [
    PROJECT_10_AVAILABLE_CANDIDATES_PATH,
    PROJECT_10_SOURCE_MANIFEST_PATH,
    PROJECT_10_PROVISIONAL_SELECTION_PATH,
    PROJECT_10_STEP1A_REPORT_PATH,
    PROJECT_10_STEP1A_STATUS_PATH,
]:

    print(
        output_path
    )


print("\nProject 9 modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)

print("\nCompletion registry modified:")
print(0)


print(
    "\nSTATUS:",
    PASS_STATUS,
)

print("=" * 116)

=== PROJECT 10 CELL 1 / STEP 1A: LOCAL SOURCE EXTRACTION AND PROVISIONAL SELECTION ===

Provisional Project 10 selection:
Original candidate rank: 2
Project 10 available rank: 1
Project: spring-cloud@spring-cloud-dataflow
Project slug: spring-cloud__spring-cloud-dataflow

Copying immutable archive from Drive to the Project 10 local runtime...
Archive MD5: 728804085c757ff5357aa165b4b6384f
Archive SHA-256: 92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e

Extracting only the Project 10 source directory...
Local source: /content/datasets/spring-cloud@spring-cloud-dataflow
Required source files: 5 / 5

Remaining ranked candidates:


,Project10AvailableRank,CandidateRank,Project,ProjectSlug,SourceDirectory,BuildsPath,ExecutionsPath,DatasetPath,IDMapPath,EntityHistoryPath,...,ModelFailingEvaluationBuilds,ModelObservedVerdicts,TimestampParseFailures,ProtocolEligible,EligibilityReason,InspectionStatus,InspectionError,ElapsedSeconds,MinimumEvaluationFailureSupport,MinimumTrainingFailureSupport
0,1,2,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,/content/datasets/spring-cloud@spring-cloud-da...,/content/datasets/spring-cloud@spring-cloud-da...,/content/datasets/spring-cloud@spring-cloud-da...,/content/datasets/spring-cloud@spring-cloud-da...,/content/datasets/spring-cloud@spring-cloud-da...,/content/datasets/spring-cloud@spring-cloud-da...,...,27,"0.0,1.0,2.0",0,True,Essential fixed-holdout protocol requirements ...,ELIGIBLE,NaN,0.178105,213,63
1,2,3,apache@shardingsphere,apache__shardingsphere,/content/datasets/apache@shardingsphere,/content/datasets/apache@shardingsphere/builds...,/content/datasets/apache@shardingsphere/exe.csv,/content/datasets/apache@shardingsphere/datase...,/content/datasets/apache@shardingsphere/id_map...,/content/datasets/apache@shardingsphere/entity...,...,25,"0.0,1.0,2.0",0,True,Essential fixed-holdout protocol requirements ...,ELIGIBLE,NaN,1.598458,171,1188
2,3,4,zolyfarkas@spf4j,zolyfarkas__spf4j,/content/datasets/zolyfarkas@spf4j,/content/datasets/zolyfarkas@spf4j/builds.csv,/content/datasets/zolyfarkas@spf4j/exe.csv,/content/datasets/zolyfarkas@spf4j/dataset.csv,/content/datasets/zolyfarkas@spf4j/id_map.csv,/content/datasets/zolyfarkas@spf4j/entity_chan...,...,91,"0.0,1.0,2.0",0,True,Essential fixed-holdout protocol requirements ...,ELIGIBLE,NaN,0.358227,101,296
3,4,5,jcabi@jcabi-github,jcabi__jcabi-github,/content/datasets/jcabi@jcabi-github,/content/datasets/jcabi@jcabi-github/builds.csv,/content/datasets/jcabi@jcabi-github/exe.csv,/content/datasets/jcabi@jcabi-github/dataset.csv,/content/datasets/jcabi@jcabi-github/id_map.csv,/content/datasets/jcabi@jcabi-github/entity_ch...,...,44,"0.0,1.0,2.0",0,True,Essential fixed-holdout protocol requirements ...,ELIGIBLE,NaN,0.293697,78,90
4,5,6,JMRI@JMRI,JMRI__JMRI,/content/datasets/JMRI@JMRI,/content/datasets/JMRI@JMRI/builds.csv,/content/datasets/JMRI@JMRI/exe.csv,/content/datasets/JMRI@JMRI/dataset.csv,/content/datasets/JMRI@JMRI/id_map.csv,/content/datasets/JMRI@JMRI/entity_change_hist...,...,24,"0.0,1.0,2.0",0,True,Essential fixed-holdout protocol requirements ...,ELIGIBLE,NaN,11.830855,73,239
5,6,7,EMResearch@EvoMaster,EMResearch__EvoMaster,/content/datasets/EMResearch@EvoMaster,/content/datasets/EMResearch@EvoMaster/builds.csv,/content/datasets/EMResearch@EvoMaster/exe.csv,/content/datasets/EMResearch@EvoMaster/dataset...,/content/datasets/EMResearch@EvoMaster/id_map.csv,/content/datasets/EMResearch@EvoMaster/entity_...,...,41,"0.0,1.0,2.0",0,True,Essential fixed-holdout protocol requirements ...,ELIGIBLE,NaN,1.196755,68,284
6,7,8,apache@sling,apache__sling,/content/datasets/apache@sling,/content/datasets/apache@sling/builds.csv,/content/datasets/apache@sling/exe.csv,/content/datasets/apache@sling/dataset.csv,/content/datasets/apache@sling/id_map.csv,/content/datasets/apache@sling/entity_change_h...,...,48,"0.0,1.0,2.0",0,True,Essential fixed-holdout protocol requirements ...,ELIGIBLE,NaN,1.787232,49,765
7,8,9,yamcs@Yamcs,yamcs__Yamcs,/content/datasets/yamcs@Yamcs,/content/datasets/yamcs@Yamcs/builds.csv,/content/datasets/yamcs@Yamcs/exe.csv,/content/datasets/yamcs@Yamcs/dataset.csv,/content/datasets/yamcs@Yamcs/id_map.csv,/content/datasets/yamcs@Yamcs/entity_change_hi...,...,10,"0.0,1.0,2.0",0,True,Essential fixed-holdout protocol requirements ...,ELIGIBLE,NaN,0.163804,45,145
8,9,10,apache@logging-log4j2,apache__logging-log4j2,/content/datasets/apache@logging-log4j2,/content/datasets/apache@logging-log4j2/builds...,/content/datasets/apache@logging-log4j2/exe.csv,/content/datasets/apache@logging-log4j2/datase...,/content/datasets/apache@logging-log4j2/id_ma


Project 10 source manifest:


,FileOrder,RelativePath,SizeBytes,SHA256
0,1,builds.csv,213888,e2a4865751d9af85ec165a1ea48a7e4db80694fd7dc0b8...
1,2,contributors.csv,8403,b6561b98ee98f4767f85b9a09c4731d05ce7a20115ffef...
2,3,dataset.csv,6779791,8dea3771bc77921912964fcec72a580e783f8aa767d15d...
3,4,entity_change_history.csv,2219944,e565a8e2b5325ea4d3063af2cde51f43fe1544e83f5c4e...
4,5,exe.csv,1462659,872cf1a33f6461ab18201f0c723f59b3189f4164f7aff0...
5,6,id_map.csv,366239,dad8e26aa42bab28d4d4c1175e9699663064fcc9b90662...




=== PROJECT 10 CELL 1 / STEP 1A RESULT ===

Provisional Project 10:
Project: spring-cloud@spring-cloud-dataflow
Project slug: spring-cloud__spring-cloud-dataflow
Original candidate rank: 2
Project 10 available rank: 1

Candidate dimensions:
Builds: 408
TrainingBuilds: 306
EvaluationBuilds: 102
RawExecutionRows: 47094
RawTrainFailures: 65
RawEvaluationFailures: 213
ModelReadyRows: 8706
ModelTrainingRows: 6095
ModelEvaluationRows: 2611
ModelTrainFailures: 63
ModelEvaluationFailures: 213
ModelFailingEvaluationBuilds: 27

Local source:
/content/datasets/spring-cloud@spring-cloud-dataflow
Source files: 6
Source bytes: 11050924
Source root SHA-256: 582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e

Dataset archive:
MD5: 728804085c757ff5357aa165b4b6384f
SHA-256: 92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e

Saved outputs:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/project_10_selection/project_10_available_candidates_ranked.csv
/content/dri

In [3]:
# ============================================================
# PROJECT 10 — CELL 2 / STEP 1B
# SELECTION LOCK, SOURCE FREEZE AND CHRONOLOGICAL SPLIT
#
# PROJECT:
#   spring-cloud@spring-cloud-dataflow
#
# This cell:
# - validates the Step 1A provisional selection
# - recomputes every source-file hash
# - independently reconstructs build chronology
# - applies the frozen chronology rule:
#       timestamp ascending
#       Build ID descending for equal timestamps
# - creates the chronological 75/25 split
# - independently recomputes all candidate dimensions
# - freezes Project 10's selection checkpoint
#
# This cell does NOT:
# - modify Project 9
# - modify Projects 1–8
# - modify the completion registry
# - inject noise
# - train models
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)


EXPECTED_BOOTSTRAP_STATUS = (
    "PASS_PROJECT_10_SAFE_PARALLEL_RUNTIME_BOOTSTRAP"
)

EXPECTED_STEP1A_STATUS = (
    "PASS_PROJECT_10_LOCAL_SOURCE_AND_PROVISIONAL_SELECTION_READY"
)

STEP1B_PASS_STATUS = (
    "PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e"
)

EXPECTED_ARCHIVE_MD5 = (
    "728804085c757ff5357aa165b4b6384f"
)


EXPECTED_BUILDS = 408
EXPECTED_TRAINING_BUILDS = 306
EXPECTED_EVALUATION_BUILDS = 102

EXPECTED_RAW_EXECUTION_ROWS = 47094
EXPECTED_RAW_TRAIN_FAILURES = 65
EXPECTED_RAW_EVALUATION_FAILURES = 213

EXPECTED_MODEL_READY_ROWS = 8706
EXPECTED_MODEL_TRAINING_ROWS = 6095
EXPECTED_MODEL_EVALUATION_ROWS = 2611
EXPECTED_MODEL_TRAIN_FAILURES = 63
EXPECTED_MODEL_EVALUATION_FAILURES = 213
EXPECTED_MODEL_FAILING_EVALUATION_BUILDS = 27


TRAINING_FRACTION = 0.75


REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "id_map.csv",
    "entity_change_history.csv",
]


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)


PROJECT_10_SELECTION_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "project_10_selection"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)


BOOTSTRAP_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_runtime_bootstrap.json"
)

STEP1A_STATUS_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_step1a_status.json"
)

PROVISIONAL_SELECTION_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_provisional_selection.json"
)

STEP1A_SOURCE_MANIFEST_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_local_source_manifest.csv"
)

AVAILABLE_CANDIDATES_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_available_candidates_ranked.csv"
)


FIXED_SPLIT_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_fixed_chronological_split.csv"
)

CHRONOLOGY_AUDIT_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_chronology_audit.csv"
)

PARTITION_AUDIT_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_partition_audit.csv"
)

DIMENSION_AUDIT_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_dimension_audit.csv"
)

SOURCE_MANIFEST_FROZEN_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_frozen_source_manifest.csv"
)

STEP1B_VALIDATION_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_step1b_validation.csv"
)

STEP1B_REPORT_PATH = (
    PROJECT_10_SELECTION_DIR
    / "project_10_step1b_report.json"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)


PROJECT_9_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "camunda__camunda-bpm-platform"
)

PROJECT_9_PROGRESS_PATH = (
    PROJECT_9_DIR
    / "camunda_full_run_control"
    / "camunda_full_run_progress.json"
)


print("=" * 120)
print("=== PROJECT 10 CELL 2 / STEP 1B: SELECTION LOCK, SOURCE FREEZE AND SPLIT VALIDATION ===")
print("=" * 120)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open(
        "rb"
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_safe(
    value,
):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(
            value
        ):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json_with_retry(
    path,
    attempts=8,
    delay_seconds=0.5,
):
    path = Path(
        path
    )

    last_error = None

    for _ in range(
        attempts
    ):

        try:

            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:

            last_error = error

            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON.\n"
        f"Path: {path}\n"
        f"Error: {type(last_error).__name__}: {last_error}"
    )


def canonical_identifier(
    series,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    numeric_coverage = float(
        numeric.notna().mean()
    )

    if numeric_coverage >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def resolve_column(
    dataframe,
    candidates,
    description,
):
    exact_matches = [
        candidate
        for candidate in candidates
        if candidate in dataframe.columns
    ]

    if exact_matches:

        return exact_matches[0]

    lower_map = {
        str(column).lower():
            column
        for column in dataframe.columns
    }

    for candidate in candidates:

        lower_candidate = str(
            candidate
        ).lower()

        if lower_candidate in lower_map:

            return lower_map[
                lower_candidate
            ]

    raise RuntimeError(
        f"Required column not found: {description}\n"
        f"Candidates: {candidates}\n"
        f"Available columns: {list(dataframe.columns)}"
    )


def create_source_manifest(
    source_directory,
):
    source_directory = Path(
        source_directory
    )

    source_files = sorted(
        [
            path
            for path in source_directory.rglob("*")
            if path.is_file()
        ],
        key=lambda path:
            path.relative_to(
                source_directory
            ).as_posix(),
    )

    records = []

    root_digest = hashlib.sha256()

    for file_order, path in enumerate(
        source_files,
        start=1,
    ):

        relative_path = (
            path.relative_to(
                source_directory
            ).as_posix()
        )

        size_bytes = int(
            path.stat().st_size
        )

        sha256 = calculate_hash(
            path,
            algorithm="sha256",
        )

        records.append({
            "FileOrder":
                file_order,

            "RelativePath":
                relative_path,

            "RuntimePath":
                str(
                    path
                ),

            "SizeBytes":
                size_bytes,

            "SHA256":
                sha256,
        })

        root_line = (
            f"{relative_path}\0"
            f"{size_bytes}\0"
            f"{sha256}\n"
        )

        root_digest.update(
            root_line.encode(
                "utf-8"
            )
        )

    return (
        pd.DataFrame(
            records
        ),
        root_digest.hexdigest(),
    )


def parse_candidate_number(
    candidate_metrics,
    key,
):
    if key not in candidate_metrics:

        return None

    value = candidate_metrics[
        key
    ]

    if value in (
        None,
        "",
        "nan",
        "NaN",
    ):

        return None

    numeric = pd.to_numeric(
        pd.Series(
            [
                value,
            ]
        ),
        errors="coerce",
    ).iloc[0]

    if pd.isna(
        numeric
    ):

        return None

    return float(
        numeric
    )


# ------------------------------------------------------------
# 4. REQUIRED INPUT VALIDATION
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    BOOTSTRAP_PATH,
    STEP1A_STATUS_PATH,
    PROVISIONAL_SELECTION_PATH,
    STEP1A_SOURCE_MANIFEST_PATH,
    AVAILABLE_CANDIDATES_PATH,
]


missing_inputs = [
    str(
        path
    )
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:

    raise FileNotFoundError(
        "Required Project 10 Step 1B inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


bootstrap = read_json_with_retry(
    BOOTSTRAP_PATH
)

step1a_status = read_json_with_retry(
    STEP1A_STATUS_PATH
)

provisional = read_json_with_retry(
    PROVISIONAL_SELECTION_PATH
)


if bootstrap.get(
    "Status"
) != EXPECTED_BOOTSTRAP_STATUS:

    raise AssertionError(
        "Project 10 bootstrap status differs.\n"
        f"Expected: {EXPECTED_BOOTSTRAP_STATUS}\n"
        f"Actual:   {bootstrap.get('Status')}"
    )


if step1a_status.get(
    "Status"
) != EXPECTED_STEP1A_STATUS:

    raise AssertionError(
        "Project 10 Step 1A status differs.\n"
        f"Expected: {EXPECTED_STEP1A_STATUS}\n"
        f"Actual:   {step1a_status.get('Status')}"
    )


if provisional.get(
    "Project"
) != PROJECT_NAME:

    raise AssertionError(
        "Provisional Project 10 identity differs."
    )


if provisional.get(
    "ProjectSlug"
) != PROJECT_SLUG:

    raise AssertionError(
        "Provisional Project 10 slug differs."
    )


if provisional.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:

    raise AssertionError(
        "Provisional Project 10 source-root SHA-256 differs."
    )


if provisional.get(
    "DatasetArchiveMD5"
) != EXPECTED_ARCHIVE_MD5:

    raise AssertionError(
        "Provisional dataset archive MD5 differs."
    )


source_directory = Path(
    provisional[
        "LocalSourceDirectory"
    ]
)


if not source_directory.exists():

    raise FileNotFoundError(
        "Project 10 local source directory is missing.\n"
        f"Expected: {source_directory}\n"
        "Do not continue in a different runtime without re-extraction."
    )


# ------------------------------------------------------------
# 5. OUTPUT-PATH ISOLATION
# ------------------------------------------------------------

output_paths = [
    FIXED_SPLIT_PATH,
    CHRONOLOGY_AUDIT_PATH,
    PARTITION_AUDIT_PATH,
    DIMENSION_AUDIT_PATH,
    SOURCE_MANIFEST_FROZEN_PATH,
    STEP1B_VALIDATION_PATH,
    STEP1B_REPORT_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
]


for output_path in output_paths:

    output_path_string = str(
        output_path
    )

    if (
        "project_10" not in output_path_string
        and PROJECT_SLUG not in output_path_string
    ):

        raise AssertionError(
            "A Step 1B output path is not Project 10 isolated.\n"
            f"Path: {output_path}"
        )

    if str(
        PROJECT_9_DIR
    ) in output_path_string:

        raise AssertionError(
            "A Project 10 output path overlaps Project 9."
        )


# ------------------------------------------------------------
# 6. REGISTRY VALIDATION — READ ONLY
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


if "ProjectNumber" not in registry.columns:

    raise AssertionError(
        "Completion registry lacks ProjectNumber."
    )


registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != 8
    or set(
        registry_project_numbers
    ) != set(
        range(
            1,
            9,
        )
    )
):

    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )


if registry_project_numbers.eq(
    9
).any():

    raise AssertionError(
        "Project 9 was unexpectedly registered."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():

    raise AssertionError(
        "Project 10 is already registered."
    )


# ------------------------------------------------------------
# 7. SOURCE-FILE FREEZE
# ------------------------------------------------------------

for filename in REQUIRED_SOURCE_FILES:

    required_path = (
        source_directory
        / filename
    )

    if not required_path.exists():

        raise FileNotFoundError(
            "Required Project 10 source file is missing:\n"
            f"{required_path}"
        )


(
    frozen_source_manifest,
    source_root_sha256,
) = create_source_manifest(
    source_directory
)


if source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:

    raise AssertionError(
        "Recomputed Project 10 source-root SHA-256 differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {source_root_sha256}"
    )


step1a_manifest = pd.read_csv(
    STEP1A_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


step1a_manifest_comparison = (
    step1a_manifest[
        [
            "FileOrder",
            "RelativePath",
            "SizeBytes",
            "SHA256",
        ]
    ]
    .copy()
)


frozen_manifest_comparison = (
    frozen_source_manifest[
        [
            "FileOrder",
            "RelativePath",
            "SizeBytes",
            "SHA256",
        ]
    ]
    .copy()
)


step1a_manifest_comparison[
    "FileOrder"
] = pd.to_numeric(
    step1a_manifest_comparison[
        "FileOrder"
    ],
    errors="raise",
).astype(int)

step1a_manifest_comparison[
    "SizeBytes"
] = pd.to_numeric(
    step1a_manifest_comparison[
        "SizeBytes"
    ],
    errors="raise",
).astype(int)


manifest_matches_step1a = (
    step1a_manifest_comparison
    .reset_index(
        drop=True
    )
    .equals(
        frozen_manifest_comparison
        .reset_index(
            drop=True
        )
    )
)


if not manifest_matches_step1a:

    raise AssertionError(
        "Recomputed source manifest differs from Step 1A."
    )


atomic_write_csv(
    SOURCE_MANIFEST_FROZEN_PATH,
    frozen_source_manifest,
)


source_files_payload = {}


for row in frozen_source_manifest.itertuples(
    index=False
):

    source_files_payload[
        str(
            row.RelativePath
        )
    ] = {
        "RuntimePath":
            str(
                row.RuntimePath
            ),

        "SizeBytes":
            int(
                row.SizeBytes
            ),

        "SHA256":
            str(
                row.SHA256
            ),
    }


# ------------------------------------------------------------
# 8. LOAD SOURCE DATA
# ------------------------------------------------------------

builds_path = (
    source_directory
    / "builds.csv"
)

executions_path = (
    source_directory
    / "exe.csv"
)

dataset_path = (
    source_directory
    / "dataset.csv"
)

id_map_path = (
    source_directory
    / "id_map.csv"
)

entity_history_path = (
    source_directory
    / "entity_change_history.csv"
)

contributors_path = (
    source_directory
    / "contributors.csv"
)


builds = pd.read_csv(
    builds_path,
    low_memory=False,
)

executions = pd.read_csv(
    executions_path,
    low_memory=False,
)

dataset = pd.read_csv(
    dataset_path,
    low_memory=False,
)


# ------------------------------------------------------------
# 9. RESOLVE REQUIRED SCHEMA
# ------------------------------------------------------------

build_id_column = resolve_column(
    builds,
    [
        "id",
        "build",
        "Build",
        "build_id",
        "buildid",
    ],
    "builds.csv build ID",
)

started_at_column = resolve_column(
    builds,
    [
        "started_at",
        "startedAt",
        "timestamp",
        "created_at",
        "date",
    ],
    "builds.csv start timestamp",
)

execution_build_column = resolve_column(
    executions,
    [
        "build",
        "Build",
        "build_id",
        "buildid",
    ],
    "exe.csv build ID",
)

execution_verdict_column = resolve_column(
    executions,
    [
        "verdict",
        "Verdict",
        "status",
        "result",
    ],
    "exe.csv verdict",
)

dataset_build_column = resolve_column(
    dataset,
    [
        "Build",
        "build",
        "build_id",
        "buildid",
    ],
    "dataset.csv build ID",
)

dataset_verdict_column = resolve_column(
    dataset,
    [
        "Verdict",
        "verdict",
        "status",
        "result",
    ],
    "dataset.csv verdict",
)


# ------------------------------------------------------------
# 10. RECONSTRUCT CANONICAL CHRONOLOGY
# ------------------------------------------------------------

chronology = builds[
    [
        build_id_column,
        started_at_column,
    ]
].copy()


chronology[
    "BuildKey"
] = canonical_identifier(
    chronology[
        build_id_column
    ]
)


chronology[
    "BuildNumeric"
] = pd.to_numeric(
    chronology[
        build_id_column
    ],
    errors="coerce",
)


build_id_parse_failures = int(
    chronology[
        "BuildNumeric"
    ].isna().sum()
)


if build_id_parse_failures:

    raise AssertionError(
        "Build IDs could not all be parsed numerically.\n"
        f"Parse failures: {build_id_parse_failures}"
    )


chronology[
    "StartedAtUTC"
] = pd.to_datetime(
    chronology[
        started_at_column
    ],
    errors="coerce",
    utc=True,
)


timestamp_parse_failures = int(
    chronology[
        "StartedAtUTC"
    ].isna().sum()
)


if timestamp_parse_failures:

    raise AssertionError(
        "Build timestamps could not all be parsed.\n"
        f"Parse failures: {timestamp_parse_failures}"
    )


duplicate_build_keys = int(
    chronology.duplicated(
        subset=[
            "BuildKey",
        ],
        keep=False,
    ).sum()
)


if duplicate_build_keys:

    raise AssertionError(
        "builds.csv contains duplicate Build IDs.\n"
        f"Duplicate rows: {duplicate_build_keys}"
    )


chronology = (
    chronology.sort_values(
        [
            "StartedAtUTC",
            "BuildNumeric",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


chronology[
    "BuildOrder"
] = np.arange(
    1,
    len(
        chronology
    ) + 1,
    dtype=np.int64,
)


training_build_count = int(
    np.floor(
        TRAINING_FRACTION
        * len(
            chronology
        )
    )
)

evaluation_build_count = int(
    len(
        chronology
    )
    - training_build_count
)


chronology[
    "Partition"
] = np.where(
    chronology[
        "BuildOrder"
    ].le(
        training_build_count
    ),
    "TRAINING",
    "EVALUATION",
)


chronology[
    "IsTraining"
] = chronology[
    "Partition"
].eq(
    "TRAINING"
)

chronology[
    "IsEvaluation"
] = chronology[
    "Partition"
].eq(
    "EVALUATION"
)


fixed_split = chronology[
    [
        "BuildKey",
        build_id_column,
        "BuildNumeric",
        started_at_column,
        "StartedAtUTC",
        "BuildOrder",
        "Partition",
        "IsTraining",
        "IsEvaluation",
    ]
].copy()


fixed_split = fixed_split.rename(
    columns={
        build_id_column:
            "SourceBuildID",

        started_at_column:
            "SourceStartedAt",
    }
)


fixed_split[
    "StartedAtUTC"
] = fixed_split[
    "StartedAtUTC"
].astype(str)


# ------------------------------------------------------------
# 11. TIE-RULE AUDIT
# ------------------------------------------------------------

tie_group_records = []

tie_rule_violations = 0


for timestamp_value, timestamp_group in (
    chronology.groupby(
        "StartedAtUTC",
        sort=False,
    )
):

    if len(
        timestamp_group
    ) <= 1:

        continue

    build_values = timestamp_group[
        "BuildNumeric"
    ].to_numpy(
        dtype=float
    )

    descending_valid = bool(
        np.all(
            build_values[:-1]
            >= build_values[1:]
        )
    )

    if not descending_valid:

        tie_rule_violations += 1

    tie_group_records.append({
        "StartedAtUTC":
            timestamp_value.isoformat(),

        "BuildCount":
            len(
                timestamp_group
            ),

        "BuildIDsInChronologicalOrder":
            ",".join(
                timestamp_group[
                    "BuildKey"
                ].astype(str)
            ),

        "DescendingBuildIDRulePass":
            descending_valid,
    })


chronology_audit = pd.DataFrame(
    tie_group_records,
    columns=[
        "StartedAtUTC",
        "BuildCount",
        "BuildIDsInChronologicalOrder",
        "DescendingBuildIDRulePass",
    ],
)


# ------------------------------------------------------------
# 12. JOIN RAW EXECUTION HISTORY TO THE SPLIT
# ------------------------------------------------------------

executions_working = executions.copy()


executions_working[
    "BuildKey"
] = canonical_identifier(
    executions_working[
        execution_build_column
    ]
)


executions_working[
    "VerdictNumeric"
] = pd.to_numeric(
    executions_working[
        execution_verdict_column
    ],
    errors="coerce",
)


raw_verdict_parse_failures = int(
    executions_working[
        "VerdictNumeric"
    ].isna().sum()
)


if raw_verdict_parse_failures:

    raise AssertionError(
        "exe.csv contains unparseable verdicts.\n"
        f"Failures: {raw_verdict_parse_failures}"
    )


executions_joined = (
    executions_working.merge(
        chronology[
            [
                "BuildKey",
                "BuildOrder",
                "Partition",
            ]
        ],
        on="BuildKey",
        how="left",
        validate="many_to_one",
        indicator=True,
        sort=False,
    )
)


raw_unlinked_rows = int(
    executions_joined[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


raw_training = executions_joined[
    executions_joined[
        "Partition"
    ].eq(
        "TRAINING"
    )
].copy()

raw_evaluation = executions_joined[
    executions_joined[
        "Partition"
    ].eq(
        "EVALUATION"
    )
].copy()


raw_training_failures = int(
    raw_training[
        "VerdictNumeric"
    ].ne(0).sum()
)

raw_evaluation_failures = int(
    raw_evaluation[
        "VerdictNumeric"
    ].ne(0).sum()
)


raw_failing_training_builds = int(
    raw_training.loc[
        raw_training[
            "VerdictNumeric"
        ].ne(0),
        "BuildKey",
    ].nunique()
)

raw_failing_evaluation_builds = int(
    raw_evaluation.loc[
        raw_evaluation[
            "VerdictNumeric"
        ].ne(0),
        "BuildKey",
    ].nunique()
)


# ------------------------------------------------------------
# 13. JOIN MODEL-READY DATA TO THE SPLIT
# ------------------------------------------------------------

dataset_working = dataset.copy()


dataset_working[
    "BuildKey"
] = canonical_identifier(
    dataset_working[
        dataset_build_column
    ]
)


dataset_working[
    "VerdictNumeric"
] = pd.to_numeric(
    dataset_working[
        dataset_verdict_column
    ],
    errors="coerce",
)


model_verdict_parse_failures = int(
    dataset_working[
        "VerdictNumeric"
    ].isna().sum()
)


if model_verdict_parse_failures:

    raise AssertionError(
        "dataset.csv contains unparseable verdicts.\n"
        f"Failures: {model_verdict_parse_failures}"
    )


dataset_joined = (
    dataset_working.merge(
        chronology[
            [
                "BuildKey",
                "BuildOrder",
                "Partition",
            ]
        ],
        on="BuildKey",
        how="left",
        validate="many_to_one",
        indicator=True,
        sort=False,
    )
)


model_unlinked_rows = int(
    dataset_joined[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


model_training = dataset_joined[
    dataset_joined[
        "Partition"
    ].eq(
        "TRAINING"
    )
].copy()

model_evaluation = dataset_joined[
    dataset_joined[
        "Partition"
    ].eq(
        "EVALUATION"
    )
].copy()


model_training_failures = int(
    model_training[
        "VerdictNumeric"
    ].ne(0).sum()
)

model_evaluation_failures = int(
    model_evaluation[
        "VerdictNumeric"
    ].ne(0).sum()
)


model_failing_training_builds = int(
    model_training.loc[
        model_training[
            "VerdictNumeric"
        ].ne(0),
        "BuildKey",
    ].nunique()
)

model_failing_evaluation_builds = int(
    model_evaluation.loc[
        model_evaluation[
            "VerdictNumeric"
        ].ne(0),
        "BuildKey",
    ].nunique()
)


# ------------------------------------------------------------
# 14. DIMENSION AUDIT
# ------------------------------------------------------------

actual_dimensions = {
    "Builds":
        len(
            chronology
        ),

    "TrainingBuilds":
        training_build_count,

    "EvaluationBuilds":
        evaluation_build_count,

    "RawExecutionRows":
        len(
            executions_joined
        ),

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "RawTrainFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "RawFailingTrainingBuilds":
        raw_failing_training_builds,

    "RawFailingEvaluationBuilds":
        raw_failing_evaluation_builds,

    "RawUnlinkedRows":
        raw_unlinked_rows,

    "ModelReadyRows":
        len(
            dataset_joined
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingTrainingBuilds":
        model_failing_training_builds,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "ModelUnlinkedRows":
        model_unlinked_rows,
}


hard_expected_dimensions = {
    "Builds":
        EXPECTED_BUILDS,

    "TrainingBuilds":
        EXPECTED_TRAINING_BUILDS,

    "EvaluationBuilds":
        EXPECTED_EVALUATION_BUILDS,

    "RawExecutionRows":
        EXPECTED_RAW_EXECUTION_ROWS,

    "RawTrainFailures":
        EXPECTED_RAW_TRAIN_FAILURES,

    "RawEvaluationFailures":
        EXPECTED_RAW_EVALUATION_FAILURES,

    "ModelReadyRows":
        EXPECTED_MODEL_READY_ROWS,

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "ModelTrainFailures":
        EXPECTED_MODEL_TRAIN_FAILURES,

    "ModelEvaluationFailures":
        EXPECTED_MODEL_EVALUATION_FAILURES,

    "ModelFailingEvaluationBuilds":
        EXPECTED_MODEL_FAILING_EVALUATION_BUILDS,

    "RawUnlinkedRows":
        0,

    "ModelUnlinkedRows":
        0,
}


candidate_metrics = provisional.get(
    "CandidateMetrics",
    {},
)


dimension_records = []


for dimension, actual_value in (
    actual_dimensions.items()
):

    hard_expected_value = (
        hard_expected_dimensions.get(
            dimension
        )
    )

    candidate_expected_value = (
        parse_candidate_number(
            candidate_metrics,
            dimension,
        )
    )

    hard_expected_pass = (
        True
        if hard_expected_value is None
        else int(
            actual_value
        ) == int(
            hard_expected_value
        )
    )

    candidate_expected_pass = (
        True
        if candidate_expected_value is None
        else int(
            actual_value
        ) == int(
            candidate_expected_value
        )
    )

    dimension_records.append({
        "Dimension":
            dimension,

        "HardExpected":
            hard_expected_value,

        "CandidateExpected":
            candidate_expected_value,

        "Actual":
            int(
                actual_value
            ),

        "HardExpectedPass":
            hard_expected_pass,

        "CandidateExpectedPass":
            candidate_expected_pass,

        "Pass":
            bool(
                hard_expected_pass
                and candidate_expected_pass
            ),
    })


dimension_audit = pd.DataFrame(
    dimension_records
)


failed_dimension_checks = dimension_audit[
    ~dimension_audit[
        "Pass"
    ]
].copy()


if not failed_dimension_checks.empty:

    print(
        "\nFailed dimension checks:"
    )

    display(
        failed_dimension_checks
    )

    raise AssertionError(
        "One or more independently recomputed "
        "Project 10 dimensions differ."
    )


# ------------------------------------------------------------
# 15. PARTITION AUDIT
# ------------------------------------------------------------

training_build_keys = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAINING"
        ),
        "BuildKey",
    ]
)

evaluation_build_keys = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildKey",
    ]
)


partition_overlap = len(
    training_build_keys
    & evaluation_build_keys
)


last_training_row = chronology[
    chronology[
        "Partition"
    ].eq(
        "TRAINING"
    )
].iloc[-1]

first_evaluation_row = chronology[
    chronology[
        "Partition"
    ].eq(
        "EVALUATION"
    )
].iloc[0]


boundary_timestamp_valid = bool(
    last_training_row[
        "StartedAtUTC"
    ]
    <= first_evaluation_row[
        "StartedAtUTC"
    ]
)


boundary_order_valid = bool(
    int(
        last_training_row[
            "BuildOrder"
        ]
    ) + 1
    == int(
        first_evaluation_row[
            "BuildOrder"
        ]
    )
)


partition_audit = pd.DataFrame([
    {
        "Partition":
            "TRAINING",

        "Builds":
            training_build_count,

        "FirstBuildOrder":
            int(
                chronology.loc[
                    chronology[
                        "Partition"
                    ].eq(
                        "TRAINING"
                    ),
                    "BuildOrder",
                ].min()
            ),

        "LastBuildOrder":
            int(
                chronology.loc[
                    chronology[
                        "Partition"
                    ].eq(
                        "TRAINING"
                    ),
                    "BuildOrder",
                ].max()
            ),

        "FirstBuildKey":
            str(
                chronology.loc[
                    chronology[
                        "Partition"
                    ].eq(
                        "TRAINING"
                    ),
                    "BuildKey",
                ].iloc[0]
            ),

        "LastBuildKey":
            str(
                last_training_row[
                    "BuildKey"
                ]
            ),

        "FirstTimestampUTC":
            chronology.loc[
                chronology[
                    "Partition"
                ].eq(
                    "TRAINING"
                ),
                "StartedAtUTC",
            ].iloc[0].isoformat(),

        "LastTimestampUTC":
            last_training_row[
                "StartedAtUTC"
            ].isoformat(),
    },

    {
        "Partition":
            "EVALUATION",

        "Builds":
            evaluation_build_count,

        "FirstBuildOrder":
            int(
                chronology.loc[
                    chronology[
                        "Partition"
                    ].eq(
                        "EVALUATION"
                    ),
                    "BuildOrder",
                ].min()
            ),

        "LastBuildOrder":
            int(
                chronology.loc[
                    chronology[
                        "Partition"
                    ].eq(
                        "EVALUATION"
                    ),
                    "BuildOrder",
                ].max()
            ),

        "FirstBuildKey":
            str(
                first_evaluation_row[
                    "BuildKey"
                ]
            ),

        "LastBuildKey":
            str(
                chronology.loc[
                    chronology[
                        "Partition"
                    ].eq(
                        "EVALUATION"
                    ),
                    "BuildKey",
                ].iloc[-1]
            ),

        "FirstTimestampUTC":
            first_evaluation_row[
                "StartedAtUTC"
            ].isoformat(),

        "LastTimestampUTC":
            chronology.loc[
                chronology[
                    "Partition"
                ].eq(
                    "EVALUATION"
                ),
                "StartedAtUTC",
            ].iloc[-1].isoformat(),
    },
])


# ------------------------------------------------------------
# 16. OVERALL VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Bootstrap passed",

        "Expected":
            EXPECTED_BOOTSTRAP_STATUS,

        "Actual":
            bootstrap.get(
                "Status"
            ),

        "Pass":
            bootstrap.get(
                "Status"
            ) == EXPECTED_BOOTSTRAP_STATUS,
    },

    {
        "Check":
            "Step 1A passed",

        "Expected":
            EXPECTED_STEP1A_STATUS,

        "Actual":
            step1a_status.get(
                "Status"
            ),

        "Pass":
            step1a_status.get(
                "Status"
            ) == EXPECTED_STEP1A_STATUS,
    },

    {
        "Check":
            "Project identity",

        "Expected":
            PROJECT_NAME,

        "Actual":
            provisional.get(
                "Project"
            ),

        "Pass":
            provisional.get(
                "Project"
            ) == PROJECT_NAME,
    },

    {
        "Check":
            "Source-root SHA-256",

        "Expected":
            EXPECTED_SOURCE_ROOT_SHA256,

        "Actual":
            source_root_sha256,

        "Pass":
            source_root_sha256
            == EXPECTED_SOURCE_ROOT_SHA256,
    },

    {
        "Check":
            "Source manifest matches Step 1A",

        "Expected":
            True,

        "Actual":
            manifest_matches_step1a,

        "Pass":
            manifest_matches_step1a,
    },

    {
        "Check":
            "Builds",

        "Expected":
            EXPECTED_BUILDS,

        "Actual":
            len(
                chronology
            ),

        "Pass":
            len(
                chronology
            ) == EXPECTED_BUILDS,
    },

    {
        "Check":
            "Training builds",

        "Expected":
            EXPECTED_TRAINING_BUILDS,

        "Actual":
            training_build_count,

        "Pass":
            training_build_count
            == EXPECTED_TRAINING_BUILDS,
    },

    {
        "Check":
            "Evaluation builds",

        "Expected":
            EXPECTED_EVALUATION_BUILDS,

        "Actual":
            evaluation_build_count,

        "Pass":
            evaluation_build_count
            == EXPECTED_EVALUATION_BUILDS,
    },

    {
        "Check":
            "Build timestamp parse failures",

        "Expected":
            0,

        "Actual":
            timestamp_parse_failures,

        "Pass":
            timestamp_parse_failures == 0,
    },

    {
        "Check":
            "Duplicate Build IDs",

        "Expected":
            0,

        "Actual":
            duplicate_build_keys,

        "Pass":
            duplicate_build_keys == 0,
    },

    {
        "Check":
            "Timestamp tie-rule violations",

        "Expected":
            0,

        "Actual":
            tie_rule_violations,

        "Pass":
            tie_rule_violations == 0,
    },

    {
        "Check":
            "Partition overlap",

        "Expected":
            0,

        "Actual":
            partition_overlap,

        "Pass":
            partition_overlap == 0,
    },

    {
        "Check":
            "Partition boundary order",

        "Expected":
            True,

        "Actual":
            boundary_order_valid,

        "Pass":
            boundary_order_valid,
    },

    {
        "Check":
            "Partition boundary chronology",

        "Expected":
            True,

        "Actual":
            boundary_timestamp_valid,

        "Pass":
            boundary_timestamp_valid,
    },

    {
        "Check":
            "Raw unlinked rows",

        "Expected":
            0,

        "Actual":
            raw_unlinked_rows,

        "Pass":
            raw_unlinked_rows == 0,
    },

    {
        "Check":
            "Model unlinked rows",

        "Expected":
            0,

        "Actual":
            model_unlinked_rows,

        "Pass":
            model_unlinked_rows == 0,
    },

    {
        "Check":
            "Dimension-audit failures",

        "Expected":
            0,

        "Actual":
            len(
                failed_dimension_checks
            ),

        "Pass":
            len(
                failed_dimension_checks
            ) == 0,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    9
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    9
                ).sum()
            ) == 0,
    },

    {
        "Check":
            "Registry Project 10 rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 1B validation:")

display(
    validation
)


if not failed_checks.empty:

    print(
        "\nFailed checks:"
    )

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 10 STEP 1B DID NOT PASS."
    )


# ------------------------------------------------------------
# 17. WRITE FROZEN OUTPUTS
# ------------------------------------------------------------

PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FIXED_SPLIT_PATH,
    fixed_split,
)

atomic_write_csv(
    CHRONOLOGY_AUDIT_PATH,
    chronology_audit,
)

atomic_write_csv(
    PARTITION_AUDIT_PATH,
    partition_audit,
)

atomic_write_csv(
    DIMENSION_AUDIT_PATH,
    dimension_audit,
)

atomic_write_csv(
    STEP1B_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 18. READBACK VALIDATION
# ------------------------------------------------------------

fixed_split_readback = pd.read_csv(
    FIXED_SPLIT_PATH,
    low_memory=False,
)

dimension_audit_readback = pd.read_csv(
    DIMENSION_AUDIT_PATH,
    low_memory=False,
)


if len(
    fixed_split_readback
) != EXPECTED_BUILDS:

    raise AssertionError(
        "Fixed split readback row count differs."
    )


if not dimension_audit_readback[
    "Pass"
].astype(bool).all():

    raise AssertionError(
        "Dimension-audit readback contains failures."
    )


# ------------------------------------------------------------
# 19. PROJECT 9 READ-ONLY SNAPSHOT
# ------------------------------------------------------------

project_9_snapshot = {}


if PROJECT_9_PROGRESS_PATH.exists():

    try:

        project_9_snapshot = read_json_with_retry(
            PROJECT_9_PROGRESS_PATH
        )

    except Exception:

        project_9_snapshot = {
            "Status":
                "ACTIVE_FILE_READ_RETRY_EXHAUSTED",
        }


# ------------------------------------------------------------
# 20. REPORT AND CHECKPOINT
# ------------------------------------------------------------

source_schema_payload = {
    "BuildsBuildID":
        build_id_column,

    "BuildsStartedAt":
        started_at_column,

    "ExecutionBuild":
        execution_build_column,

    "ExecutionVerdict":
        execution_verdict_column,

    "DatasetBuild":
        dataset_build_column,

    "DatasetVerdict":
        dataset_verdict_column,
}


chronology_payload = {
    "Rule":
        (
            "Timestamp ascending; Build ID descending "
            "for equal timestamps"
        ),

    "TrainingFraction":
        TRAINING_FRACTION,

    "Builds":
        len(
            chronology
        ),

    "TrainingBuilds":
        training_build_count,

    "EvaluationBuilds":
        evaluation_build_count,

    "TimestampTieGroups":
        len(
            chronology_audit
        ),

    "TimestampTieRuleViolations":
        tie_rule_violations,

    "FirstBuild":
        str(
            chronology[
                "BuildKey"
            ].iloc[0]
        ),

    "LastBuild":
        str(
            chronology[
                "BuildKey"
            ].iloc[-1]
        ),

    "LastTrainingBuild":
        str(
            last_training_row[
                "BuildKey"
            ]
        ),

    "FirstEvaluationBuild":
        str(
            first_evaluation_row[
                "BuildKey"
            ]
        ),

    "LastTrainingBuildOrder":
        int(
            last_training_row[
                "BuildOrder"
            ]
        ),

    "FirstEvaluationBuildOrder":
        int(
            first_evaluation_row[
                "BuildOrder"
            ]
        ),

    "LastTrainingTimestampUTC":
        last_training_row[
            "StartedAtUTC"
        ].isoformat(),

    "FirstEvaluationTimestampUTC":
        first_evaluation_row[
            "StartedAtUTC"
        ].isoformat(),
}


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProjectShortName":
        PROJECT_SHORT_NAME,

    "Status":
        STEP1B_PASS_STATUS,

    "SourceDirectory":
        str(
            source_directory
        ),

    "SourceRootSHA256":
        source_root_sha256,

    "SourceFiles":
        len(
            frozen_source_manifest
        ),

    "SourceBytes":
        int(
            frozen_source_manifest[
                "SizeBytes"
            ].sum()
        ),

    "SourceSchema":
        source_schema_payload,

    "Chronology":
        chronology_payload,

    "Dimensions":
        actual_dimensions,

    "FixedSplit":
        str(
            FIXED_SPLIT_PATH
        ),

    "FixedSplitSHA256":
        calculate_hash(
            FIXED_SPLIT_PATH,
            algorithm="sha256",
        ),

    "SourceManifest":
        str(
            SOURCE_MANIFEST_FROZEN_PATH
        ),

    "SourceManifestSHA256":
        calculate_hash(
            SOURCE_MANIFEST_FROZEN_PATH,
            algorithm="sha256",
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "Project9ReadOnlySnapshot":
        project_9_snapshot,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletionRegistryModified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP1B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProjectShortName":
        PROJECT_SHORT_NAME,

    "Status":
        STEP1B_PASS_STATUS,

    "Archive": {
        "MD5":
            EXPECTED_ARCHIVE_MD5,

        "DrivePath":
            provisional.get(
                "DatasetArchiveDrivePath"
            ),

        "LocalPath":
            provisional.get(
                "DatasetArchiveLocalPath"
            ),

        "SHA256":
            provisional.get(
                "DatasetArchiveSHA256"
            ),
    },

    "SourceDirectory":
        str(
            source_directory
        ),

    "SourceRootSHA256":
        source_root_sha256,

    "SourceFiles":
        source_files_payload,

    "BuildsPath":
        str(
            builds_path
        ),

    "ExecutionHistoryPath":
        str(
            executions_path
        ),

    "DatasetPath":
        str(
            dataset_path
        ),

    "IdMapPath":
        str(
            id_map_path
        ),

    "EntityHistoryPath":
        str(
            entity_history_path
        ),

    "ContributorsPath":
        (
            str(
                contributors_path
            )
            if contributors_path.exists()
            else None
        ),

    "ColumnSchema":
        source_schema_payload,

    "ChronologyRule":
        (
            "Timestamp ascending; Build ID descending "
            "for equal timestamps"
        ),

    "Chronology":
        chronology_payload,

    "Builds":
        len(
            chronology
        ),

    "TrainingBuilds":
        training_build_count,

    "EvaluationBuilds":
        evaluation_build_count,

    "RawExecutionRows":
        len(
            executions_joined
        ),

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "RawTrainFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "RawFailingTrainingBuilds":
        raw_failing_training_builds,

    "RawFailingEvaluationBuilds":
        raw_failing_evaluation_builds,

    "RawUnlinkedRows":
        raw_unlinked_rows,

    "ModelReadyRows":
        len(
            dataset_joined
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingTrainingBuilds":
        model_failing_training_builds,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "ModelUnlinkedRows":
        model_unlinked_rows,

    "FixedSplit":
        str(
            FIXED_SPLIT_PATH
        ),

    "FixedSplitSHA256":
        calculate_hash(
            FIXED_SPLIT_PATH,
            algorithm="sha256",
        ),

    "ChronologyAudit":
        str(
            CHRONOLOGY_AUDIT_PATH
        ),

    "ChronologyAuditSHA256":
        calculate_hash(
            CHRONOLOGY_AUDIT_PATH,
            algorithm="sha256",
        ),

    "PartitionAudit":
        str(
            PARTITION_AUDIT_PATH
        ),

    "PartitionAuditSHA256":
        calculate_hash(
            PARTITION_AUDIT_PATH,
            algorithm="sha256",
        ),

    "DimensionAudit":
        str(
            DIMENSION_AUDIT_PATH
        ),

    "DimensionAuditSHA256":
        calculate_hash(
            DIMENSION_AUDIT_PATH,
            algorithm="sha256",
        ),

    "SourceManifest":
        str(
            SOURCE_MANIFEST_FROZEN_PATH
        ),

    "SourceManifestSHA256":
        calculate_hash(
            SOURCE_MANIFEST_FROZEN_PATH,
            algorithm="sha256",
        ),

    "ProvisionalSelectionSHA256":
        calculate_hash(
            PROVISIONAL_SELECTION_PATH,
            algorithm="sha256",
        ),

    "AvailableCandidateRankingSHA256":
        calculate_hash(
            AVAILABLE_CANDIDATES_PATH,
            algorithm="sha256",
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistryRows":
        len(
            registry
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "CompletionRegistryModified":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    SELECTION_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP1B_PASS_STATUS,

    "Builds":
        len(
            chronology
        ),

    "TrainingBuilds":
        training_build_count,

    "EvaluationBuilds":
        evaluation_build_count,

    "SourceRootSHA256":
        source_root_sha256,

    "Checkpoint":
        str(
            SELECTION_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_hash(
            SELECTION_CHECKPOINT_PATH,
            algorithm="sha256",
        ),

    "CompletionRegistryModified":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP1B_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 21. FINAL READBACK AND REGISTRY CHECK
# ------------------------------------------------------------

checkpoint_readback = read_json_with_retry(
    SELECTION_CHECKPOINT_PATH
)

status_readback = read_json_with_retry(
    STEP1B_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:

    raise AssertionError(
        "Project 10 selection-checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:

    raise AssertionError(
        "Project 10 Step 1B status readback failed."
    )


registry_sha256_after = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:

    raise AssertionError(
        "Completion registry changed during Project 10 Step 1B."
    )


# ------------------------------------------------------------
# 22. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nResolved schema:")

print(
    "builds.csv build ID:",
    build_id_column,
)

print(
    "builds.csv timestamp:",
    started_at_column,
)

print(
    "exe.csv build:",
    execution_build_column,
)

print(
    "exe.csv verdict:",
    execution_verdict_column,
)

print(
    "dataset.csv build:",
    dataset_build_column,
)

print(
    "dataset.csv verdict:",
    dataset_verdict_column,
)


print("\nChronological partitions:")

display(
    partition_audit
)


print("\nDimension audit:")

display(
    dimension_audit
)


if not chronology_audit.empty:

    print("\nTimestamp-tie audit:")

    display(
        chronology_audit
    )

else:

    print(
        "\nTimestamp-tie audit: "
        "no tied timestamps were present."
    )


print("\n")
print("=" * 120)
print("=== PROJECT 10 CELL 2 / STEP 1B RESULT ===")
print("=" * 120)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nSource freeze:")

print(
    "Source files:",
    len(
        frozen_source_manifest
    ),
)

print(
    "Source bytes:",
    int(
        frozen_source_manifest[
            "SizeBytes"
        ].sum()
    ),
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


print("\nChronological split:")

print(
    "Builds:",
    len(
        chronology
    ),
)

print(
    "Training builds:",
    training_build_count,
)

print(
    "Evaluation builds:",
    evaluation_build_count,
)

print(
    "Last training build:",
    last_training_row[
        "BuildKey"
    ],
)

print(
    "First evaluation build:",
    first_evaluation_row[
        "BuildKey"
    ],
)

print(
    "Timestamp tie groups:",
    len(
        chronology_audit
    ),
)

print(
    "Tie-rule violations:",
    tie_rule_violations,
)


print("\nRaw execution cohort:")

print(
    "Rows:",
    len(
        executions_joined
    ),
)

print(
    "Training rows:",
    len(
        raw_training
    ),
)

print(
    "Evaluation rows:",
    len(
        raw_evaluation
    ),
)

print(
    "Training failures:",
    raw_training_failures,
)

print(
    "Evaluation failures:",
    raw_evaluation_failures,
)

print(
    "Unlinked rows:",
    raw_unlinked_rows,
)


print("\nModel-ready cohort:")

print(
    "Rows:",
    len(
        dataset_joined
    ),
)

print(
    "Training rows:",
    len(
        model_training
    ),
)

print(
    "Evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Training failures:",
    model_training_failures,
)

print(
    "Evaluation failures:",
    model_evaluation_failures,
)

print(
    "Failing evaluation builds:",
    model_failing_evaluation_builds,
)

print(
    "Unlinked rows:",
    model_unlinked_rows,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nSelection checkpoint:")

print(
    SELECTION_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_hash(
        SELECTION_CHECKPOINT_PATH,
        algorithm="sha256",
    ),
)


print("\nRegistry unchanged:")

print(
    registry_unchanged
)

print("\nProject 9 modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)

print("\nCompletion registry modified:")
print(0)


print(
    "\nSTATUS:",
    STEP1B_PASS_STATUS,
)

print("=" * 120)

=== PROJECT 10 CELL 2 / STEP 1B: SELECTION LOCK, SOURCE FREEZE AND SPLIT VALIDATION ===

Step 1B validation:


,Check,Expected,Actual,Pass
0,Bootstrap passed,PASS_PROJECT_10_SAFE_PARALLEL_RUNTIME_BOOTSTRAP,PASS_PROJECT_10_SAFE_PARALLEL_RUNTIME_BOOTSTRAP,True
1,Step 1A passed,PASS_PROJECT_10_LOCAL_SOURCE_AND_PROVISIONAL_S...,PASS_PROJECT_10_LOCAL_SOURCE_AND_PROVISIONAL_S...,True
2,Project identity,spring-cloud@spring-cloud-dataflow,spring-cloud@spring-cloud-dataflow,True
3,Source-root SHA-256,582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3...,582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3...,True
4,Source manifest matches Step 1A,True,True,True
5,Builds,408,408,True
6,Training builds,306,306,True
7,Evaluation builds,102,102,True
8,Build timestamp parse failures,0,0,True
9,Duplicate Build IDs,0,0,True



Resolved schema:
builds.csv build ID: id
builds.csv timestamp: started_at
exe.csv build: build
exe.csv verdict: verdict
dataset.csv build: Build
dataset.csv verdict: Verdict

Chronological partitions:


,Partition,Builds,FirstBuildOrder,LastBuildOrder,FirstBuildKey,LastBuildKey,FirstTimestampUTC,LastTimestampUTC
0,TRAINING,306,1,306,507728039,607209985,2019-03-18T07:48:07+00:00,2019-11-04T15:55:56+00:00
1,EVALUATION,102,307,408,607501817,627586663,2019-11-05T06:51:16+00:00,2019-12-20T06:38:06+00:00



Dimension audit:


,Dimension,HardExpected,CandidateExpected,Actual,HardExpectedPass,CandidateExpectedPass,Pass
0,Builds,408.0,408.0,408,True,True,True
1,TrainingBuilds,306.0,306.0,306,True,True,True
2,EvaluationBuilds,102.0,102.0,102,True,True,True
3,RawExecutionRows,47094.0,47094.0,47094,True,True,True
4,RawTrainingRows,NaN,34563.0,34563,True,True,True
5,RawEvaluationRows,NaN,12531.0,12531,True,True,True
6,RawTrainFailures,65.0,65.0,65,True,True,True
7,RawEvaluationFailures,213.0,213.0,213,True,True,True
8,RawFailingTrainingBuilds,NaN,NaN,62,True,True,True
9,RawFailingEvaluationBuilds,NaN,27.0,27,True,True,True



Timestamp-tie audit:


,StartedAtUTC,BuildCount,BuildIDsInChronologicalOrder,DescendingBuildIDRulePass
0,2019-07-30T17:25:44+00:00,2,"565598116,565598080",True
1,2019-12-05T21:31:19+00:00,2,"621322689,621322679",True




=== PROJECT 10 CELL 2 / STEP 1B RESULT ===

Project identity:
Project number: 10
Project: spring-cloud@spring-cloud-dataflow
Project slug: spring-cloud__spring-cloud-dataflow

Source freeze:
Source files: 6
Source bytes: 11050924
Source root SHA-256: 582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e

Chronological split:
Builds: 408
Training builds: 306
Evaluation builds: 102
Last training build: 607209985
First evaluation build: 607501817
Timestamp tie groups: 2
Tie-rule violations: 0

Raw execution cohort:
Rows: 47094
Training rows: 34563
Evaluation rows: 12531
Training failures: 65
Evaluation failures: 213
Unlinked rows: 0

Model-ready cohort:
Rows: 8706
Training rows: 6095
Evaluation rows: 2611
Training failures: 63
Evaluation failures: 213
Failing evaluation builds: 27
Unlinked rows: 0

Validation:
Checks: 19
Failed checks: 0

Selection checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/project_10_selection_checkpoint.json
Checkpoint SHA-256: 97960ad7032d7

In [4]:
# ============================================================
# PROJECT 10 — CELL 3 / STEP 2A
# SOURCE SCHEMA AND JOIN-STRUCTURE AUDIT
#
# PROJECT:
#   spring-cloud@spring-cloud-dataflow
#
# This cell:
# - validates the frozen Project 10 selection
# - verifies every frozen source-file hash
# - audits all five core source tables
# - resolves raw/model Build, Test, Verdict and Duration fields
# - resolves commit/entity mapping fields
# - validates Build/Test uniqueness
# - proves every model-ready Build/Test row exists in exe.csv
# - proves model-ready verdicts match raw verdicts
# - validates chronological partition counts
# - freezes the schema needed by clean REC reconstruction
#
# This cell does NOT:
# - reconstruct REC features
# - inject noise
# - fit models
# - modify Project 9
# - modify Projects 1–8
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)

STEP2A_PASS_STATUS = (
    "PASS_PROJECT_10_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_AUDITED"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e"
)

EXPECTED_BUILDS = 408

EXPECTED_RAW_ROWS = 47094
EXPECTED_RAW_TRAINING_ROWS = 34563
EXPECTED_RAW_EVALUATION_ROWS = 12531
EXPECTED_RAW_TRAINING_FAILURES = 65
EXPECTED_RAW_EVALUATION_FAILURES = 213
EXPECTED_RAW_FAILING_TRAINING_BUILDS = 62
EXPECTED_RAW_FAILING_EVALUATION_BUILDS = 27

EXPECTED_MODEL_ROWS = 8706
EXPECTED_MODEL_TRAINING_ROWS = 6095
EXPECTED_MODEL_EVALUATION_ROWS = 2611
EXPECTED_MODEL_TRAINING_FAILURES = 63
EXPECTED_MODEL_EVALUATION_FAILURES = 213
EXPECTED_MODEL_FAILING_TRAINING_BUILDS = 61
EXPECTED_MODEL_FAILING_EVALUATION_BUILDS = 27

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_MODEL_PREDICTORS = 151

EXPECTED_BUILDS_COLUMNS = 3
EXPECTED_EXECUTIONS_COLUMNS = 5
EXPECTED_ID_MAP_COLUMNS = 2
EXPECTED_ENTITY_HISTORY_COLUMNS = 8


REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_selection_checkpoint.json"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)

SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

TABLE_SUMMARY_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_table_summary.csv"
)

COLUMN_PROFILE_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_column_profile.csv"
)

FEATURE_MANIFEST_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_feature_manifest.csv"
)

VERDICT_PROFILE_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_verdict_profile.csv"
)

MAPPING_SCHEMA_AUDIT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_mapping_schema_audit.csv"
)

JOIN_AUDIT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_join_audit.csv"
)

MODEL_RAW_KEY_ALIGNMENT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_raw_key_alignment.parquet"
)

STEP2A_VALIDATION_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_validation.csv"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)

STEP2A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_status.json"
)


PROJECT_9_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "camunda__camunda-bpm-platform"
)


print("=" * 120)
print("=== PROJECT 10 CELL 3 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE AUDIT ===")
print("=" * 120)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open(
        "rb"
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_safe(
    value,
):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(
            value
        ):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json_with_retry(
    path,
    attempts=8,
    delay_seconds=0.5,
):
    path = Path(path)

    last_error = None

    for _ in range(
        attempts
    ):

        try:

            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:

            last_error = error

            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON.\n"
        f"Path: {path}\n"
        f"Error: {type(last_error).__name__}: {last_error}"
    )


def canonical_identifier(
    series,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def resolve_column(
    dataframe,
    candidates,
    description,
):
    exact_matches = [
        candidate
        for candidate in candidates
        if candidate in dataframe.columns
    ]

    if exact_matches:

        return exact_matches[0]

    lower_map = {
        str(column).lower():
            column
        for column in dataframe.columns
    }

    for candidate in candidates:

        candidate_lower = str(
            candidate
        ).lower()

        if candidate_lower in lower_map:

            return lower_map[
                candidate_lower
            ]

    raise RuntimeError(
        f"Required column not found: {description}\n"
        f"Candidates: {candidates}\n"
        f"Available columns: {list(dataframe.columns)}"
    )


def create_source_root_sha256(
    source_files_payload,
):
    digest = hashlib.sha256()

    for relative_path in sorted(
        source_files_payload
    ):

        metadata = source_files_payload[
            relative_path
        ]

        runtime_path = Path(
            metadata[
                "RuntimePath"
            ]
        )

        size_bytes = int(
            runtime_path.stat().st_size
        )

        file_sha256 = calculate_hash(
            runtime_path,
            algorithm="sha256",
        )

        digest.update(
            (
                f"{relative_path}\0"
                f"{size_bytes}\0"
                f"{file_sha256}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def normalised_string_set(
    series,
):
    return set(
        canonical_identifier(
            series
        ).astype(str)
    )


# ------------------------------------------------------------
# 4. INPUT VALIDATION
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
]


missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:

    raise FileNotFoundError(
        "Required Project 10 Step 2A inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


selection_checkpoint = read_json_with_retry(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = read_json_with_retry(
    STEP1B_STATUS_PATH
)


if selection_checkpoint.get(
    "Status"
) != EXPECTED_STEP1B_STATUS:

    raise AssertionError(
        "Project 10 selection checkpoint status differs.\n"
        f"Expected: {EXPECTED_STEP1B_STATUS}\n"
        f"Actual:   {selection_checkpoint.get('Status')}"
    )


if step1b_status.get(
    "Status"
) != EXPECTED_STEP1B_STATUS:

    raise AssertionError(
        "Project 10 Step 1B status differs."
    )


if selection_checkpoint.get(
    "Project"
) != PROJECT_NAME:

    raise AssertionError(
        "Project 10 identity differs."
    )


if selection_checkpoint.get(
    "ProjectSlug"
) != PROJECT_SLUG:

    raise AssertionError(
        "Project 10 slug differs."
    )


if selection_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:

    raise AssertionError(
        "Project 10 frozen source-root hash differs."
    )


# ------------------------------------------------------------
# 5. OUTPUT-PATH ISOLATION
# ------------------------------------------------------------

output_paths = [
    TABLE_SUMMARY_PATH,
    COLUMN_PROFILE_PATH,
    FEATURE_MANIFEST_PATH,
    VERDICT_PROFILE_PATH,
    MAPPING_SCHEMA_AUDIT_PATH,
    JOIN_AUDIT_PATH,
    MODEL_RAW_KEY_ALIGNMENT_PATH,
    STEP2A_VALIDATION_PATH,
    STEP2A_REPORT_PATH,
    STEP2A_STATUS_PATH,
]


for output_path in output_paths:

    output_string = str(
        output_path
    )

    if PROJECT_SLUG not in output_string:

        raise AssertionError(
            "A Step 2A output path is not Project 10 isolated.\n"
            f"Path: {output_path}"
        )

    if str(
        PROJECT_9_DIR
    ) in output_string:

        raise AssertionError(
            "A Project 10 output path overlaps Project 9."
        )


# ------------------------------------------------------------
# 6. REGISTRY — READ ONLY
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != 8
    or set(
        registry_project_numbers
    ) != set(
        range(
            1,
            9,
        )
    )
):

    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )


if registry_project_numbers.eq(9).any():

    raise AssertionError(
        "Project 9 was unexpectedly registered."
    )


if registry_project_numbers.eq(10).any():

    raise AssertionError(
        "Project 10 was unexpectedly registered."
    )


# ------------------------------------------------------------
# 7. VERIFY ALL FROZEN SOURCE FILES
# ------------------------------------------------------------

source_files_payload = (
    selection_checkpoint[
        "SourceFiles"
    ]
)


for relative_path, metadata in (
    source_files_payload.items()
):

    runtime_path = Path(
        metadata[
            "RuntimePath"
        ]
    )

    if not runtime_path.exists():

        raise FileNotFoundError(
            "Frozen source file is missing:\n"
            f"{runtime_path}"
        )

    actual_size = int(
        runtime_path.stat().st_size
    )

    expected_size = int(
        metadata[
            "SizeBytes"
        ]
    )

    if actual_size != expected_size:

        raise AssertionError(
            "Frozen source-file size differs.\n"
            f"File: {relative_path}\n"
            f"Expected: {expected_size}\n"
            f"Actual:   {actual_size}"
        )

    actual_sha256 = calculate_hash(
        runtime_path,
        algorithm="sha256",
    )

    expected_sha256 = str(
        metadata[
            "SHA256"
        ]
    )

    if actual_sha256 != expected_sha256:

        raise AssertionError(
            "Frozen source-file SHA-256 differs.\n"
            f"File: {relative_path}"
        )


source_root_sha256 = create_source_root_sha256(
    source_files_payload
)


if source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:

    raise AssertionError(
        "Recomputed Project 10 source-root SHA-256 differs."
    )


builds_path = Path(
    selection_checkpoint[
        "BuildsPath"
    ]
)

executions_path = Path(
    selection_checkpoint[
        "ExecutionHistoryPath"
    ]
)

dataset_path = Path(
    selection_checkpoint[
        "DatasetPath"
    ]
)

id_map_path = Path(
    selection_checkpoint[
        "IdMapPath"
    ]
)

entity_history_path = Path(
    selection_checkpoint[
        "EntityHistoryPath"
    ]
)

fixed_split_path = Path(
    selection_checkpoint[
        "FixedSplit"
    ]
)


if calculate_hash(
    fixed_split_path,
    algorithm="sha256",
) != selection_checkpoint[
    "FixedSplitSHA256"
]:

    raise AssertionError(
        "Frozen chronological split hash differs."
    )


# ------------------------------------------------------------
# 8. LOAD SOURCE TABLES
# ------------------------------------------------------------

builds = pd.read_csv(
    builds_path,
    low_memory=False,
)

executions = pd.read_csv(
    executions_path,
    low_memory=False,
)

dataset = pd.read_csv(
    dataset_path,
    low_memory=False,
)

id_map = pd.read_csv(
    id_map_path,
    low_memory=False,
)

entity_history = pd.read_csv(
    entity_history_path,
    low_memory=False,
)

fixed_split = pd.read_csv(
    fixed_split_path,
    low_memory=False,
)


# ------------------------------------------------------------
# 9. RESOLVE COMPLETE SOURCE SCHEMA
# ------------------------------------------------------------

build_id_column = resolve_column(
    builds,
    [
        "id",
        "build",
        "Build",
        "build_id",
    ],
    "builds.csv build ID",
)

build_started_at_column = resolve_column(
    builds,
    [
        "started_at",
        "startedAt",
        "timestamp",
        "created_at",
    ],
    "builds.csv timestamp",
)

build_commits_column = resolve_column(
    builds,
    [
        "commits",
        "commit",
        "commit_hashes",
        "changesets",
    ],
    "builds.csv commits",
)

execution_build_column = resolve_column(
    executions,
    [
        "build",
        "Build",
        "build_id",
    ],
    "exe.csv build",
)

execution_job_column = resolve_column(
    executions,
    [
        "job",
        "Job",
        "job_id",
    ],
    "exe.csv job",
)

execution_test_column = resolve_column(
    executions,
    [
        "test",
        "Test",
        "test_id",
    ],
    "exe.csv test",
)

execution_verdict_column = resolve_column(
    executions,
    [
        "verdict",
        "Verdict",
        "status",
        "result",
    ],
    "exe.csv verdict",
)

execution_duration_column = resolve_column(
    executions,
    [
        "duration",
        "Duration",
        "execution_time",
        "time",
    ],
    "exe.csv duration",
)

dataset_build_column = resolve_column(
    dataset,
    [
        "Build",
        "build",
        "build_id",
    ],
    "dataset.csv build",
)

dataset_test_column = resolve_column(
    dataset,
    [
        "Test",
        "test",
        "test_id",
    ],
    "dataset.csv test",
)

dataset_verdict_column = resolve_column(
    dataset,
    [
        "Verdict",
        "verdict",
        "status",
        "result",
    ],
    "dataset.csv verdict",
)

id_map_key_column = resolve_column(
    id_map,
    [
        "key",
        "Key",
        "id",
        "EntityId",
    ],
    "id_map.csv key",
)

id_map_value_column = resolve_column(
    id_map,
    [
        "value",
        "Value",
        "name",
        "path",
    ],
    "id_map.csv value",
)

entity_history_commit_column = resolve_column(
    entity_history,
    [
        "Commit",
        "commit",
        "commit_hash",
        "hash",
    ],
    "entity_change_history.csv commit",
)

entity_history_id_column = resolve_column(
    entity_history,
    [
        "EntityId",
        "entity_id",
        "entity",
        "id",
    ],
    "entity_change_history.csv entity ID",
)


# ------------------------------------------------------------
# 10. TABLE AND COLUMN PROFILES
# ------------------------------------------------------------

source_tables = {
    "builds.csv":
        builds,

    "exe.csv":
        executions,

    "dataset.csv":
        dataset,

    "id_map.csv":
        id_map,

    "entity_change_history.csv":
        entity_history,
}


table_summary_records = []

column_profile_records = []


for table_name, dataframe in (
    source_tables.items()
):

    table_summary_records.append({
        "Table":
            table_name,

        "Rows":
            len(
                dataframe
            ),

        "Columns":
            len(
                dataframe.columns
            ),

        "DuplicateFullRows":
            int(
                dataframe.duplicated(
                    keep=False
                ).sum()
            ),

        "MissingCells":
            int(
                dataframe.isna().sum().sum()
            ),
    })

    for column_order, column in enumerate(
        dataframe.columns,
        start=1,
    ):

        series = dataframe[
            column
        ]

        nonmissing_samples = (
            series.dropna()
            .astype(str)
            .head(3)
            .tolist()
        )

        column_profile_records.append({
            "Table":
                table_name,

            "ColumnOrder":
                column_order,

            "Column":
                column,

            "Dtype":
                str(
                    series.dtype
                ),

            "Rows":
                len(
                    series
                ),

            "MissingValues":
                int(
                    series.isna().sum()
                ),

            "DistinctValues":
                int(
                    series.nunique(
                        dropna=True
                    )
                ),

            "ExampleValues":
                " | ".join(
                    nonmissing_samples
                ),
        })


table_summary = pd.DataFrame(
    table_summary_records
)

column_profile = pd.DataFrame(
    column_profile_records
)


# ------------------------------------------------------------
# 11. FEATURE MANIFEST
# ------------------------------------------------------------

model_identifier_columns = [
    dataset_build_column,
    dataset_test_column,
    dataset_verdict_column,
]


model_predictor_columns = [
    column
    for column in dataset.columns
    if column not in model_identifier_columns
]


missing_rec_features = [
    feature
    for feature in REC_FEATURE_COLUMNS
    if feature not in dataset.columns
]


if missing_rec_features:

    raise AssertionError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


feature_manifest_records = []


for feature_order, feature in enumerate(
    model_predictor_columns,
    start=1,
):

    if feature in (
        VERDICT_DEPENDENT_REC_FEATURES
    ):

        feature_class = (
            "REC_VERDICT_DEPENDENT"
        )

    elif feature in (
        VERDICT_INDEPENDENT_REC_FEATURES
    ):

        feature_class = (
            "REC_VERDICT_INDEPENDENT"
        )

    else:

        feature_class = "NON_REC"

    feature_manifest_records.append({
        "FeatureOrder":
            feature_order,

        "Feature":
            feature,

        "FeatureClass":
            feature_class,

        "IsREC":
            feature in REC_FEATURE_COLUMNS,

        "VerdictDependent":
            (
                feature
                in VERDICT_DEPENDENT_REC_FEATURES
            ),

        "VerdictIndependent":
            (
                feature
                in VERDICT_INDEPENDENT_REC_FEATURES
            ),
    })


feature_manifest = pd.DataFrame(
    feature_manifest_records
)


# ------------------------------------------------------------
# 12. CANONICAL RAW EXECUTION KEYS
# ------------------------------------------------------------

raw_key_frame = pd.DataFrame({
    "RawRowOrder":
        np.arange(
            len(
                executions
            ),
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            executions[
                execution_build_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            executions[
                execution_test_column
            ]
        ),

    "RawVerdict":
        pd.to_numeric(
            executions[
                execution_verdict_column
            ],
            errors="coerce",
        ),

    "RawDuration":
        pd.to_numeric(
            executions[
                execution_duration_column
            ],
            errors="coerce",
        ),
})


raw_empty_build_keys = int(
    raw_key_frame[
        "BuildKey"
    ].eq("").sum()
)

raw_empty_test_keys = int(
    raw_key_frame[
        "TestKey"
    ].eq("").sum()
)

raw_verdict_parse_failures = int(
    raw_key_frame[
        "RawVerdict"
    ].isna().sum()
)

raw_duration_parse_failures = int(
    raw_key_frame[
        "RawDuration"
    ].isna().sum()
)

raw_nonfinite_durations = int(
    (
        ~np.isfinite(
            raw_key_frame[
                "RawDuration"
            ].fillna(
                np.nan
            ).to_numpy(
                dtype=float
            )
        )
    ).sum()
)

raw_negative_durations = int(
    raw_key_frame[
        "RawDuration"
    ].lt(0).sum()
)

raw_duplicate_build_test_rows = int(
    raw_key_frame.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


# ------------------------------------------------------------
# 13. CANONICAL MODEL-READY KEYS
# ------------------------------------------------------------

model_key_frame = pd.DataFrame({
    "ModelRowOrder":
        np.arange(
            len(
                dataset
            ),
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            dataset[
                dataset_build_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            dataset[
                dataset_test_column
            ]
        ),

    "ModelVerdict":
        pd.to_numeric(
            dataset[
                dataset_verdict_column
            ],
            errors="coerce",
        ),
})


model_empty_build_keys = int(
    model_key_frame[
        "BuildKey"
    ].eq("").sum()
)

model_empty_test_keys = int(
    model_key_frame[
        "TestKey"
    ].eq("").sum()
)

model_verdict_parse_failures = int(
    model_key_frame[
        "ModelVerdict"
    ].isna().sum()
)

model_duplicate_build_test_rows = int(
    model_key_frame.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if raw_duplicate_build_test_rows:

    raise AssertionError(
        "exe.csv contains duplicate canonical Build/Test rows.\n"
        f"Duplicate rows: {raw_duplicate_build_test_rows}"
    )


if model_duplicate_build_test_rows:

    raise AssertionError(
        "dataset.csv contains duplicate canonical Build/Test rows.\n"
        f"Duplicate rows: {model_duplicate_build_test_rows}"
    )


if raw_verdict_parse_failures:

    raise AssertionError(
        "exe.csv contains unparseable verdicts."
    )


if raw_duration_parse_failures:

    raise AssertionError(
        "exe.csv contains unparseable durations."
    )


if model_verdict_parse_failures:

    raise AssertionError(
        "dataset.csv contains unparseable verdicts."
    )


# ------------------------------------------------------------
# 14. MODEL ↔ RAW BUILD/TEST ALIGNMENT
# ------------------------------------------------------------

model_raw_alignment = (
    model_key_frame.merge(
        raw_key_frame,
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_model_raw_rows = int(
    model_raw_alignment[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


matched_alignment = model_raw_alignment[
    model_raw_alignment[
        "_merge"
    ].eq(
        "both"
    )
].copy()


verdict_mismatches = int(
    (
        matched_alignment[
            "ModelVerdict"
        ].to_numpy(
            dtype=float
        )
        !=
        matched_alignment[
            "RawVerdict"
        ].to_numpy(
            dtype=float
        )
    ).sum()
)


model_raw_alignment[
    "VerdictMatch"
] = (
    model_raw_alignment[
        "ModelVerdict"
    ].eq(
        model_raw_alignment[
            "RawVerdict"
        ]
    )
)


# ------------------------------------------------------------
# 15. BUILD AND PARTITION ALIGNMENT
# ------------------------------------------------------------

fixed_split[
    "BuildKey"
] = fixed_split[
    "BuildKey"
].astype(str)


if fixed_split.duplicated(
    subset=[
        "BuildKey",
    ],
    keep=False,
).any():

    raise AssertionError(
        "Frozen split contains duplicate BuildKey rows."
    )


partition_map = (
    fixed_split.set_index(
        "BuildKey"
    )[
        "Partition"
    ]
)


raw_key_frame[
    "Partition"
] = raw_key_frame[
    "BuildKey"
].map(
    partition_map
)


model_key_frame[
    "Partition"
] = model_key_frame[
    "BuildKey"
].map(
    partition_map
)


raw_unlinked_build_rows = int(
    raw_key_frame[
        "Partition"
    ].isna().sum()
)

model_unlinked_build_rows = int(
    model_key_frame[
        "Partition"
    ].isna().sum()
)


raw_training = raw_key_frame[
    raw_key_frame[
        "Partition"
    ].eq(
        "TRAINING"
    )
]

raw_evaluation = raw_key_frame[
    raw_key_frame[
        "Partition"
    ].eq(
        "EVALUATION"
    )
]

model_training = model_key_frame[
    model_key_frame[
        "Partition"
    ].eq(
        "TRAINING"
    )
]

model_evaluation = model_key_frame[
    model_key_frame[
        "Partition"
    ].eq(
        "EVALUATION"
    )
]


raw_training_failures = int(
    raw_training[
        "RawVerdict"
    ].ne(0).sum()
)

raw_evaluation_failures = int(
    raw_evaluation[
        "RawVerdict"
    ].ne(0).sum()
)

model_training_failures = int(
    model_training[
        "ModelVerdict"
    ].ne(0).sum()
)

model_evaluation_failures = int(
    model_evaluation[
        "ModelVerdict"
    ].ne(0).sum()
)


raw_failing_training_builds = int(
    raw_training.loc[
        raw_training[
            "RawVerdict"
        ].ne(0),
        "BuildKey",
    ].nunique()
)

raw_failing_evaluation_builds = int(
    raw_evaluation.loc[
        raw_evaluation[
            "RawVerdict"
        ].ne(0),
        "BuildKey",
    ].nunique()
)

model_failing_training_builds = int(
    model_training.loc[
        model_training[
            "ModelVerdict"
        ].ne(0),
        "BuildKey",
    ].nunique()
)

model_failing_evaluation_builds = int(
    model_evaluation.loc[
        model_evaluation[
            "ModelVerdict"
        ].ne(0),
        "BuildKey",
    ].nunique()
)


# ------------------------------------------------------------
# 16. VERDICT-SUBTYPE PROFILE
# ------------------------------------------------------------

verdict_profile_records = []


for source_name, verdict_series in [
    (
        "RAW_EXECUTION_HISTORY",
        raw_key_frame[
            "RawVerdict"
        ],
    ),
    (
        "MODEL_READY_DATASET",
        model_key_frame[
            "ModelVerdict"
        ],
    ),
]:

    verdict_counts = (
        verdict_series.value_counts(
            dropna=False
        )
        .sort_index()
    )

    for verdict_value, count in (
        verdict_counts.items()
    ):

        verdict_profile_records.append({
            "Source":
                source_name,

            "Verdict":
                verdict_value,

            "Rows":
                int(
                    count
                ),

            "Percentage":
                float(
                    100.0
                    * count
                    / len(
                        verdict_series
                    )
                ),
        })


verdict_profile = pd.DataFrame(
    verdict_profile_records
)


raw_observed_verdicts = sorted(
    raw_key_frame[
        "RawVerdict"
    ].dropna().unique().tolist()
)

model_observed_verdicts = sorted(
    model_key_frame[
        "ModelVerdict"
    ].dropna().unique().tolist()
)


# ------------------------------------------------------------
# 17. COMMIT/ENTITY MAPPING SCHEMA AUDIT
# ------------------------------------------------------------

build_commit_missing_rows = int(
    builds[
        build_commits_column
    ].isna().sum()
)

entity_commit_missing_rows = int(
    entity_history[
        entity_history_commit_column
    ].isna().sum()
)

entity_id_missing_rows = int(
    entity_history[
        entity_history_id_column
    ].isna().sum()
)

id_map_key_missing_rows = int(
    id_map[
        id_map_key_column
    ].isna().sum()
)

id_map_value_missing_rows = int(
    id_map[
        id_map_value_column
    ].isna().sum()
)


entity_history_id_set = normalised_string_set(
    entity_history[
        entity_history_id_column
    ].dropna()
)

id_map_key_set = normalised_string_set(
    id_map[
        id_map_key_column
    ].dropna()
)

id_map_value_set = normalised_string_set(
    id_map[
        id_map_value_column
    ].dropna()
)


entity_ids_matched_to_key = len(
    entity_history_id_set
    & id_map_key_set
)

entity_ids_matched_to_value = len(
    entity_history_id_set
    & id_map_value_set
)


if (
    entity_ids_matched_to_key
    >= entity_ids_matched_to_value
):

    resolved_id_map_entity_column = (
        id_map_key_column
    )

    mapped_entity_ids = (
        id_map_key_set
    )

    id_map_entity_side = "KEY"

else:

    resolved_id_map_entity_column = (
        id_map_value_column
    )

    mapped_entity_ids = (
        id_map_value_set
    )

    id_map_entity_side = "VALUE"


unmapped_unique_entity_ids = int(
    len(
        entity_history_id_set
        - mapped_entity_ids
    )
)


mapping_schema_audit = pd.DataFrame([
    {
        "Check":
            "Build commit column",

        "ResolvedColumn":
            build_commits_column,

        "Rows":
            len(
                builds
            ),

        "MissingValues":
            build_commit_missing_rows,

        "DistinctValues":
            int(
                builds[
                    build_commits_column
                ].nunique(
                    dropna=True
                )
            ),
    },

    {
        "Check":
            "Entity-history commit column",

        "ResolvedColumn":
            entity_history_commit_column,

        "Rows":
            len(
                entity_history
            ),

        "MissingValues":
            entity_commit_missing_rows,

        "DistinctValues":
            int(
                entity_history[
                    entity_history_commit_column
                ].nunique(
                    dropna=True
                )
            ),
    },

    {
        "Check":
            "Entity-history ID column",

        "ResolvedColumn":
            entity_history_id_column,

        "Rows":
            len(
                entity_history
            ),

        "MissingValues":
            entity_id_missing_rows,

        "DistinctValues":
            len(
                entity_history_id_set
            ),
    },

    {
        "Check":
            "ID-map key column",

        "ResolvedColumn":
            id_map_key_column,

        "Rows":
            len(
                id_map
            ),

        "MissingValues":
            id_map_key_missing_rows,

        "DistinctValues":
            len(
                id_map_key_set
            ),
    },

    {
        "Check":
            "ID-map value column",

        "ResolvedColumn":
            id_map_value_column,

        "Rows":
            len(
                id_map
            ),

        "MissingValues":
            id_map_value_missing_rows,

        "DistinctValues":
            len(
                id_map_value_set
            ),
    },

    {
        "Check":
            "Resolved ID-map entity side",

        "ResolvedColumn":
            resolved_id_map_entity_column,

        "Rows":
            len(
                entity_history_id_set
            ),

        "MissingValues":
            unmapped_unique_entity_ids,

        "DistinctValues":
            (
                entity_ids_matched_to_key
                if id_map_entity_side == "KEY"
                else entity_ids_matched_to_value
            ),
    },
])


# ------------------------------------------------------------
# 18. JOIN AUDIT
# ------------------------------------------------------------

join_audit = pd.DataFrame([
    {
        "Check":
            "Canonical builds",

        "Expected":
            EXPECTED_BUILDS,

        "Actual":
            fixed_split[
                "BuildKey"
            ].nunique(),

        "Pass":
            fixed_split[
                "BuildKey"
            ].nunique()
            == EXPECTED_BUILDS,
    },

    {
        "Check":
            "Raw execution rows",

        "Expected":
            EXPECTED_RAW_ROWS,

        "Actual":
            len(
                raw_key_frame
            ),

        "Pass":
            len(
                raw_key_frame
            ) == EXPECTED_RAW_ROWS,
    },

    {
        "Check":
            "Model-ready rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(
                model_key_frame
            ),

        "Pass":
            len(
                model_key_frame
            ) == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "Raw duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            raw_duplicate_build_test_rows,

        "Pass":
            raw_duplicate_build_test_rows == 0,
    },

    {
        "Check":
            "Model duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            model_duplicate_build_test_rows,

        "Pass":
            model_duplicate_build_test_rows == 0,
    },

    {
        "Check":
            "Model rows missing raw Build/Test match",

        "Expected":
            0,

        "Actual":
            missing_model_raw_rows,

        "Pass":
            missing_model_raw_rows == 0,
    },

    {
        "Check":
            "Matched raw/model verdict mismatches",

        "Expected":
            0,

        "Actual":
            verdict_mismatches,

        "Pass":
            verdict_mismatches == 0,
    },

    {
        "Check":
            "Raw rows missing build partition",

        "Expected":
            0,

        "Actual":
            raw_unlinked_build_rows,

        "Pass":
            raw_unlinked_build_rows == 0,
    },

    {
        "Check":
            "Model rows missing build partition",

        "Expected":
            0,

        "Actual":
            model_unlinked_build_rows,

        "Pass":
            model_unlinked_build_rows == 0,
    },
])


# ------------------------------------------------------------
# 19. OVERALL VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 1B passed",

        "Expected":
            EXPECTED_STEP1B_STATUS,

        "Actual":
            step1b_status.get(
                "Status"
            ),

        "Pass":
            step1b_status.get(
                "Status"
            ) == EXPECTED_STEP1B_STATUS,
    },

    {
        "Check":
            "Source root SHA-256",

        "Expected":
            EXPECTED_SOURCE_ROOT_SHA256,

        "Actual":
            source_root_sha256,

        "Pass":
            source_root_sha256
            == EXPECTED_SOURCE_ROOT_SHA256,
    },

    {
        "Check":
            "builds.csv rows",

        "Expected":
            EXPECTED_BUILDS,

        "Actual":
            len(
                builds
            ),

        "Pass":
            len(
                builds
            ) == EXPECTED_BUILDS,
    },

    {
        "Check":
            "builds.csv columns",

        "Expected":
            EXPECTED_BUILDS_COLUMNS,

        "Actual":
            len(
                builds.columns
            ),

        "Pass":
            len(
                builds.columns
            ) == EXPECTED_BUILDS_COLUMNS,
    },

    {
        "Check":
            "exe.csv rows",

        "Expected":
            EXPECTED_RAW_ROWS,

        "Actual":
            len(
                executions
            ),

        "Pass":
            len(
                executions
            ) == EXPECTED_RAW_ROWS,
    },

    {
        "Check":
            "exe.csv columns",

        "Expected":
            EXPECTED_EXECUTIONS_COLUMNS,

        "Actual":
            len(
                executions.columns
            ),

        "Pass":
            len(
                executions.columns
            ) == EXPECTED_EXECUTIONS_COLUMNS,
    },

    {
        "Check":
            "dataset.csv rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(
                dataset
            ),

        "Pass":
            len(
                dataset
            ) == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "dataset.csv columns",

        "Expected":
            EXPECTED_DATASET_COLUMNS,

        "Actual":
            len(
                dataset.columns
            ),

        "Pass":
            len(
                dataset.columns
            ) == EXPECTED_DATASET_COLUMNS,
    },

    {
        "Check":
            "id_map.csv columns",

        "Expected":
            EXPECTED_ID_MAP_COLUMNS,

        "Actual":
            len(
                id_map.columns
            ),

        "Pass":
            len(
                id_map.columns
            ) == EXPECTED_ID_MAP_COLUMNS,
    },

    {
        "Check":
            "entity history columns",

        "Expected":
            EXPECTED_ENTITY_HISTORY_COLUMNS,

        "Actual":
            len(
                entity_history.columns
            ),

        "Pass":
            len(
                entity_history.columns
            ) == EXPECTED_ENTITY_HISTORY_COLUMNS,
    },

    {
        "Check":
            "Model predictors",

        "Expected":
            EXPECTED_MODEL_PREDICTORS,

        "Actual":
            len(
                model_predictor_columns
            ),

        "Pass":
            len(
                model_predictor_columns
            ) == EXPECTED_MODEL_PREDICTORS,
    },

    {
        "Check":
            "REC features",

        "Expected":
            19,

        "Actual":
            int(
                feature_manifest[
                    "IsREC"
                ].sum()
            ),

        "Pass":
            int(
                feature_manifest[
                    "IsREC"
                ].sum()
            ) == 19,
    },

    {
        "Check":
            "Verdict-dependent REC features",

        "Expected":
            13,

        "Actual":
            int(
                feature_manifest[
                    "VerdictDependent"
                ].sum()
            ),

        "Pass":
            int(
                feature_manifest[
                    "VerdictDependent"
                ].sum()
            ) == 13,
    },

    {
        "Check":
            "Verdict-independent REC features",

        "Expected":
            6,

        "Actual":
            int(
                feature_manifest[
                    "VerdictIndependent"
                ].sum()
            ),

        "Pass":
            int(
                feature_manifest[
                    "VerdictIndependent"
                ].sum()
            ) == 6,
    },

    {
        "Check":
            "Raw empty Build keys",

        "Expected":
            0,

        "Actual":
            raw_empty_build_keys,

        "Pass":
            raw_empty_build_keys == 0,
    },

    {
        "Check":
            "Raw empty Test keys",

        "Expected":
            0,

        "Actual":
            raw_empty_test_keys,

        "Pass":
            raw_empty_test_keys == 0,
    },

    {
        "Check":
            "Model empty Build keys",

        "Expected":
            0,

        "Actual":
            model_empty_build_keys,

        "Pass":
            model_empty_build_keys == 0,
    },

    {
        "Check":
            "Model empty Test keys",

        "Expected":
            0,

        "Actual":
            model_empty_test_keys,

        "Pass":
            model_empty_test_keys == 0,
    },

    {
        "Check":
            "Raw duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            raw_duplicate_build_test_rows,

        "Pass":
            raw_duplicate_build_test_rows == 0,
    },

    {
        "Check":
            "Model duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            model_duplicate_build_test_rows,

        "Pass":
            model_duplicate_build_test_rows == 0,
    },

    {
        "Check":
            "Model rows missing raw match",

        "Expected":
            0,

        "Actual":
            missing_model_raw_rows,

        "Pass":
            missing_model_raw_rows == 0,
    },

    {
        "Check":
            "Raw/model verdict mismatches",

        "Expected":
            0,

        "Actual":
            verdict_mismatches,

        "Pass":
            verdict_mismatches == 0,
    },

    {
        "Check":
            "Raw duration parse failures",

        "Expected":
            0,

        "Actual":
            raw_duration_parse_failures,

        "Pass":
            raw_duration_parse_failures == 0,
    },

    {
        "Check":
            "Raw non-finite durations",

        "Expected":
            0,

        "Actual":
            raw_nonfinite_durations,

        "Pass":
            raw_nonfinite_durations == 0,
    },

    {
        "Check":
            "Raw negative durations",

        "Expected":
            0,

        "Actual":
            raw_negative_durations,

        "Pass":
            raw_negative_durations == 0,
    },

    {
        "Check":
            "Raw training rows",

        "Expected":
            EXPECTED_RAW_TRAINING_ROWS,

        "Actual":
            len(
                raw_training
            ),

        "Pass":
            len(
                raw_training
            ) == EXPECTED_RAW_TRAINING_ROWS,
    },

    {
        "Check":
            "Raw evaluation rows",

        "Expected":
            EXPECTED_RAW_EVALUATION_ROWS,

        "Actual":
            len(
                raw_evaluation
            ),

        "Pass":
            len(
                raw_evaluation
            ) == EXPECTED_RAW_EVALUATION_ROWS,
    },

    {
        "Check":
            "Raw training failures",

        "Expected":
            EXPECTED_RAW_TRAINING_FAILURES,

        "Actual":
            raw_training_failures,

        "Pass":
            raw_training_failures
            == EXPECTED_RAW_TRAINING_FAILURES,
    },

    {
        "Check":
            "Raw evaluation failures",

        "Expected":
            EXPECTED_RAW_EVALUATION_FAILURES,

        "Actual":
            raw_evaluation_failures,

        "Pass":
            raw_evaluation_failures
            == EXPECTED_RAW_EVALUATION_FAILURES,
    },

    {
        "Check":
            "Raw failing training builds",

        "Expected":
            EXPECTED_RAW_FAILING_TRAINING_BUILDS,

        "Actual":
            raw_failing_training_builds,

        "Pass":
            raw_failing_training_builds
            == EXPECTED_RAW_FAILING_TRAINING_BUILDS,
    },

    {
        "Check":
            "Raw failing evaluation builds",

        "Expected":
            EXPECTED_RAW_FAILING_EVALUATION_BUILDS,

        "Actual":
            raw_failing_evaluation_builds,

        "Pass":
            raw_failing_evaluation_builds
            == EXPECTED_RAW_FAILING_EVALUATION_BUILDS,
    },

    {
        "Check":
            "Model training rows",

        "Expected":
            EXPECTED_MODEL_TRAINING_ROWS,

        "Actual":
            len(
                model_training
            ),

        "Pass":
            len(
                model_training
            ) == EXPECTED_MODEL_TRAINING_ROWS,
    },

    {
        "Check":
            "Model evaluation rows",

        "Expected":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "Actual":
            len(
                model_evaluation
            ),

        "Pass":
            len(
                model_evaluation
            ) == EXPECTED_MODEL_EVALUATION_ROWS,
    },

    {
        "Check":
            "Model training failures",

        "Expected":
            EXPECTED_MODEL_TRAINING_FAILURES,

        "Actual":
            model_training_failures,

        "Pass":
            model_training_failures
            == EXPECTED_MODEL_TRAINING_FAILURES,
    },

    {
        "Check":
            "Model evaluation failures",

        "Expected":
            EXPECTED_MODEL_EVALUATION_FAILURES,

        "Actual":
            model_evaluation_failures,

        "Pass":
            model_evaluation_failures
            == EXPECTED_MODEL_EVALUATION_FAILURES,
    },

    {
        "Check":
            "Model failing training builds",

        "Expected":
            EXPECTED_MODEL_FAILING_TRAINING_BUILDS,

        "Actual":
            model_failing_training_builds,

        "Pass":
            model_failing_training_builds
            == EXPECTED_MODEL_FAILING_TRAINING_BUILDS,
    },

    {
        "Check":
            "Model failing evaluation builds",

        "Expected":
            EXPECTED_MODEL_FAILING_EVALUATION_BUILDS,

        "Actual":
            model_failing_evaluation_builds,

        "Pass":
            model_failing_evaluation_builds
            == EXPECTED_MODEL_FAILING_EVALUATION_BUILDS,
    },

    {
        "Check":
            "Observed raw verdicts",

        "Expected":
            [0.0, 1.0, 2.0],

        "Actual":
            raw_observed_verdicts,

        "Pass":
            raw_observed_verdicts
            == [
                0.0,
                1.0,
                2.0,
            ],
    },

    {
        "Check":
            "Observed model verdicts",

        "Expected":
            [0.0, 1.0, 2.0],

        "Actual":
            model_observed_verdicts,

        "Pass":
            model_observed_verdicts
            == [
                0.0,
                1.0,
                2.0,
            ],
    },

    {
        "Check":
            "Unmapped unique entity IDs",

        "Expected":
            0,

        "Actual":
            unmapped_unique_entity_ids,

        "Pass":
            unmapped_unique_entity_ids == 0,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    9
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    9
                ).sum()
            ) == 0,
    },

    {
        "Check":
            "Registry Project 10 rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    10
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    10
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 2A validation:")

display(
    validation
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 10 STEP 2A DID NOT PASS."
    )


# ------------------------------------------------------------
# 20. WRITE AUDIT OUTPUTS
# ------------------------------------------------------------

SCHEMA_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    TABLE_SUMMARY_PATH,
    table_summary,
)

atomic_write_csv(
    COLUMN_PROFILE_PATH,
    column_profile,
)

atomic_write_csv(
    FEATURE_MANIFEST_PATH,
    feature_manifest,
)

atomic_write_csv(
    VERDICT_PROFILE_PATH,
    verdict_profile,
)

atomic_write_csv(
    MAPPING_SCHEMA_AUDIT_PATH,
    mapping_schema_audit,
)

atomic_write_csv(
    JOIN_AUDIT_PATH,
    join_audit,
)

atomic_write_parquet(
    MODEL_RAW_KEY_ALIGNMENT_PATH,
    model_raw_alignment,
)

atomic_write_csv(
    STEP2A_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 21. READBACK VALIDATION
# ------------------------------------------------------------

alignment_readback = pd.read_parquet(
    MODEL_RAW_KEY_ALIGNMENT_PATH
)

table_summary_readback = pd.read_csv(
    TABLE_SUMMARY_PATH,
    low_memory=False,
)


if len(
    alignment_readback
) != EXPECTED_MODEL_ROWS:

    raise AssertionError(
        "Model/raw key-alignment readback row count differs."
    )


if len(
    table_summary_readback
) != 5:

    raise AssertionError(
        "Table-summary readback count differs."
    )


# ------------------------------------------------------------
# 22. REPORT AND STATUS
# ------------------------------------------------------------

resolved_columns_payload = {
    "BuildsBuildID":
        build_id_column,

    "BuildsStartedAt":
        build_started_at_column,

    "BuildsCommits":
        build_commits_column,

    "ExecutionBuild":
        execution_build_column,

    "ExecutionJob":
        execution_job_column,

    "ExecutionTest":
        execution_test_column,

    "ExecutionVerdict":
        execution_verdict_column,

    "ExecutionDuration":
        execution_duration_column,

    "DatasetBuild":
        dataset_build_column,

    "DatasetTest":
        dataset_test_column,

    "DatasetVerdict":
        dataset_verdict_column,

    "IDMapKey":
        id_map_key_column,

    "IDMapValue":
        id_map_value_column,

    "IDMapEntityColumn":
        resolved_id_map_entity_column,

    "IDMapEntitySide":
        id_map_entity_side,

    "EntityHistoryCommit":
        entity_history_commit_column,

    "EntityHistoryID":
        entity_history_id_column,
}


table_shapes_payload = {
    "builds.csv": {
        "Rows":
            len(
                builds
            ),

        "Columns":
            len(
                builds.columns
            ),
    },

    "exe.csv": {
        "Rows":
            len(
                executions
            ),

        "Columns":
            len(
                executions.columns
            ),
    },

    "dataset.csv": {
        "Rows":
            len(
                dataset
            ),

        "Columns":
            len(
                dataset.columns
            ),
    },

    "id_map.csv": {
        "Rows":
            len(
                id_map
            ),

        "Columns":
            len(
                id_map.columns
            ),
    },

    "entity_change_history.csv": {
        "Rows":
            len(
                entity_history
            ),

        "Columns":
            len(
                entity_history.columns
            ),
    },
}


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProjectShortName":
        PROJECT_SHORT_NAME,

    "Status":
        STEP2A_PASS_STATUS,

    "SourceRootSHA256":
        source_root_sha256,

    "ResolvedColumns":
        resolved_columns_payload,

    "TableShapes":
        table_shapes_payload,

    "DatasetColumns":
        len(
            dataset.columns
        ),

    "ModelPredictors":
        len(
            model_predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURE_COLUMNS
        ),

    "VerdictDependentRECFeatures":
        len(
            VERDICT_DEPENDENT_REC_FEATURES
        ),

    "VerdictIndependentRECFeatures":
        len(
            VERDICT_INDEPENDENT_REC_FEATURES
        ),

    "RawDuplicateBuildTestRows":
        raw_duplicate_build_test_rows,

    "ModelDuplicateBuildTestRows":
        model_duplicate_build_test_rows,

    "ModelRowsMissingRawMatch":
        missing_model_raw_rows,

    "RawModelVerdictMismatches":
        verdict_mismatches,

    "RawObservedVerdicts":
        raw_observed_verdicts,

    "ModelObservedVerdicts":
        model_observed_verdicts,

    "UnmappedUniqueEntityIDs":
        unmapped_unique_entity_ids,

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "RawTrainingFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "TableSummary":
        str(
            TABLE_SUMMARY_PATH
        ),

    "TableSummarySHA256":
        calculate_hash(
            TABLE_SUMMARY_PATH,
            algorithm="sha256",
        ),

    "ColumnProfile":
        str(
            COLUMN_PROFILE_PATH
        ),

    "ColumnProfileSHA256":
        calculate_hash(
            COLUMN_PROFILE_PATH,
            algorithm="sha256",
        ),

    "FeatureManifest":
        str(
            FEATURE_MANIFEST_PATH
        ),

    "FeatureManifestSHA256":
        calculate_hash(
            FEATURE_MANIFEST_PATH,
            algorithm="sha256",
        ),

    "VerdictProfile":
        str(
            VERDICT_PROFILE_PATH
        ),

    "VerdictProfileSHA256":
        calculate_hash(
            VERDICT_PROFILE_PATH,
            algorithm="sha256",
        ),

    "MappingSchemaAudit":
        str(
            MAPPING_SCHEMA_AUDIT_PATH
        ),

    "MappingSchemaAuditSHA256":
        calculate_hash(
            MAPPING_SCHEMA_AUDIT_PATH,
            algorithm="sha256",
        ),

    "JoinAudit":
        str(
            JOIN_AUDIT_PATH
        ),

    "JoinAuditSHA256":
        calculate_hash(
            JOIN_AUDIT_PATH,
            algorithm="sha256",
        ),

    "ModelRawKeyAlignment":
        str(
            MODEL_RAW_KEY_ALIGNMENT_PATH
        ),

    "ModelRawKeyAlignmentSHA256":
        calculate_hash(
            MODEL_RAW_KEY_ALIGNMENT_PATH,
            algorithm="sha256",
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletionRegistryModified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2A_REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_PASS_STATUS,

    "SourceRootSHA256":
        source_root_sha256,

    "RawExecutionRows":
        len(
            raw_key_frame
        ),

    "ModelReadyRows":
        len(
            model_key_frame
        ),

    "ModelRowsMissingRawMatch":
        missing_model_raw_rows,

    "RawModelVerdictMismatches":
        verdict_mismatches,

    "Report":
        str(
            STEP2A_REPORT_PATH
        ),

    "ReportSHA256":
        calculate_hash(
            STEP2A_REPORT_PATH,
            algorithm="sha256",
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletionRegistryModified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2A_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 23. FINAL READBACK AND REGISTRY CHECK
# ------------------------------------------------------------

report_readback = read_json_with_retry(
    STEP2A_REPORT_PATH
)

status_readback = read_json_with_retry(
    STEP2A_STATUS_PATH
)


if report_readback.get(
    "Status"
) != STEP2A_PASS_STATUS:

    raise AssertionError(
        "Project 10 Step 2A report readback failed."
    )


if status_readback.get(
    "Status"
) != STEP2A_PASS_STATUS:

    raise AssertionError(
        "Project 10 Step 2A status readback failed."
    )


registry_sha256_after = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:

    raise AssertionError(
        "Completion registry changed during Project 10 Step 2A."
    )


# ------------------------------------------------------------
# 24. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nSource-table summary:")

display(
    table_summary
)


print("\nResolved source schema:")

for key, value in (
    resolved_columns_payload.items()
):

    print(
        f"{key}:",
        value,
    )


print("\nFeature manifest summary:")

display(
    feature_manifest.groupby(
        "FeatureClass",
        as_index=False,
    ).agg(
        Features=(
            "Feature",
            "count",
        )
    )
)


print("\nVerdict profiles:")

display(
    verdict_profile
)


print("\nJoin audit:")

display(
    join_audit
)


print("\nMapping-schema audit:")

display(
    mapping_schema_audit
)


# ------------------------------------------------------------
# 25. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 120)
print("=== PROJECT 10 CELL 3 / STEP 2A RESULT ===")
print("=" * 120)

print("\nProject:")
print(PROJECT_NAME)

print("\nSource tables:")

for table_name, table_shape in (
    table_shapes_payload.items()
):

    print(
        f"{table_name}: "
        f"{table_shape['Rows']} rows × "
        f"{table_shape['Columns']} columns"
    )


print("\nModel-ready schema:")

print(
    "Dataset columns:",
    len(
        dataset.columns
    ),
)

print(
    "Predictor columns:",
    len(
        model_predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURE_COLUMNS
    ),
)

print(
    "Verdict-dependent REC features:",
    len(
        VERDICT_DEPENDENT_REC_FEATURES
    ),
)

print(
    "Verdict-independent REC features:",
    len(
        VERDICT_INDEPENDENT_REC_FEATURES
    ),
)


print("\nBuild/Test alignment:")

print(
    "Raw duplicate Build/Test rows:",
    raw_duplicate_build_test_rows,
)

print(
    "Model duplicate Build/Test rows:",
    model_duplicate_build_test_rows,
)

print(
    "Model rows missing raw match:",
    missing_model_raw_rows,
)

print(
    "Raw/model verdict mismatches:",
    verdict_mismatches,
)


print("\nPartitions:")

print(
    "Raw training / evaluation rows:",
    len(
        raw_training
    ),
    "/",
    len(
        raw_evaluation
    ),
)

print(
    "Raw training / evaluation failures:",
    raw_training_failures,
    "/",
    raw_evaluation_failures,
)

print(
    "Model training / evaluation rows:",
    len(
        model_training
    ),
    "/",
    len(
        model_evaluation
    ),
)

print(
    "Model training / evaluation failures:",
    model_training_failures,
    "/",
    model_evaluation_failures,
)


print("\nCommit/entity structure:")

print(
    "Build commit column:",
    build_commits_column,
)

print(
    "Entity-history commit column:",
    entity_history_commit_column,
)

print(
    "Entity-history ID column:",
    entity_history_id_column,
)

print(
    "ID-map key / value:",
    id_map_key_column,
    "/",
    id_map_value_column,
)

print(
    "Resolved ID-map entity side:",
    id_map_entity_side,
)

print(
    "Unmapped unique entity IDs:",
    unmapped_unique_entity_ids,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nRegistry unchanged:")

print(
    registry_unchanged
)

print("\nProject 9 modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)

print("\nCompletion registry modified:")
print(0)


print(
    "\nSTATUS:",
    STEP2A_PASS_STATUS,
)

print("=" * 120)

=== PROJECT 10 CELL 3 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE AUDIT ===

Step 2A validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,True
1,Source root SHA-256,582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3...,582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3...,True
2,builds.csv rows,408,408,True
3,builds.csv columns,3,3,True
4,exe.csv rows,47094,47094,True
5,exe.csv columns,5,5,True
6,dataset.csv rows,8706,8706,True
7,dataset.csv columns,154,154,True
8,id_map.csv columns,2,2,True
9,entity history columns,8,8,True



Source-table summary:


,Table,Rows,Columns,DuplicateFullRows,MissingCells
0,builds.csv,408,3,0,0
1,exe.csv,47094,5,0,0
2,dataset.csv,8706,154,0,0
3,id_map.csv,3121,2,0,0
4,entity_change_history.csv,25653,8,0,0



Resolved source schema:
BuildsBuildID: id
BuildsStartedAt: started_at
BuildsCommits: commits
ExecutionBuild: build
ExecutionJob: job
ExecutionTest: test
ExecutionVerdict: verdict
ExecutionDuration: duration
DatasetBuild: Build
DatasetTest: Test
DatasetVerdict: Verdict
IDMapKey: key
IDMapValue: value
IDMapEntityColumn: value
IDMapEntitySide: VALUE
EntityHistoryCommit: Commit
EntityHistoryID: EntityId

Feature manifest summary:


...


Join audit:


,Check,Expected,Actual,Pass
0,Canonical builds,408,408,True
1,Raw execution rows,47094,47094,True
2,Model-ready rows,8706,8706,True
3,Raw duplicate Build/Test rows,0,0,True
4,Model duplicate Build/Test rows,0,0,True
5,Model rows missing raw Build/Test match,0,0,True
6,Matched raw/model verdict mismatches,0,0,True
7,Raw rows missing build partition,0,0,True
8,Model rows missing build partition,0,0,True



Mapping-schema audit:


,Check,ResolvedColumn,Rows,MissingValues,DistinctValues
0,Build commit column,commits,408,0,387
1,Entity-history commit column,Commit,25653,0,3459
2,Entity-history ID column,EntityId,25653,0,2146
3,ID-map key column,key,3121,0,3121
4,ID-map value column,value,3121,0,2146
5,Resolved ID-map entity side,value,2146,0,2146




=== PROJECT 10 CELL 3 / STEP 2A RESULT ===

Project:
spring-cloud@spring-cloud-dataflow

Source tables:
builds.csv: 408 rows × 3 columns
exe.csv: 47094 rows × 5 columns
dataset.csv: 8706 rows × 154 columns
id_map.csv: 3121 rows × 2 columns
entity_change_history.csv: 25653 rows × 8 columns

Model-ready schema:
Dataset columns: 154
Predictor columns: 151
REC features: 19
Verdict-dependent REC features: 13
Verdict-independent REC features: 6

Build/Test alignment:
Raw duplicate Build/Test rows: 0
Model duplicate Build/Test rows: 0
Model rows missing raw match: 0
Raw/model verdict mismatches: 0

Partitions:
Raw training / evaluation rows: 34563 / 12531
Raw training / evaluation failures: 65 / 213
Model training / evaluation rows: 6095 / 2611
Model training / evaluation failures: 63 / 213

Commit/entity structure:
Build commit column: commits
Entity-history commit column: Commit
Entity-history ID column: EntityId
ID-map key / value: key / value
Resolved ID-map entity side: VALUE
Unmapped 

In [5]:
# ============================================================
# PROJECT 10 — CELL 4 / STEP 2B
# CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE
#
# PROJECT:
#   spring-cloud@spring-cloud-dataflow
#
# This cell:
# - validates Steps 1B and 2A
# - verifies all frozen Project 10 source hashes
# - parses build commit lists
# - maps build commits to changed source entities
# - reconstructs all 19 clean REC features
# - compares reconstruction against dataset.csv
# - freezes clean anchor offsets
# - validates that direct reconstruction + anchor reproduces
#   the original clean dataset
#
# This cell does NOT:
# - inject noise
# - train models
# - modify Project 9
# - modify Projects 1–8
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from IPython.display import display

import ast
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)


EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)

EXPECTED_STEP2A_STATUS = (
    "PASS_PROJECT_10_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_AUDITED"
)

STEP2B_PASS_STATUS = (
    "PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e"
)

EXPECTED_BUILDS = 408
EXPECTED_SOURCE_FILES = 6
EXPECTED_RAW_ROWS = 47094
EXPECTED_MODEL_ROWS = 8706

RECENT_WINDOW = 6

DIRECT_COMPARISON_RTOL = 1e-9
DIRECT_COMPARISON_ATOL = 1e-9

ANCHOR_COMPARISON_RTOL = 1e-12
ANCHOR_COMPARISON_ATOL = 1e-12

NONZERO_OFFSET_THRESHOLD = 1e-12


REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


FILE_HISTORY_REC_FEATURES = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


if len(REC_FEATURE_COLUMNS) != 19:
    raise AssertionError(
        "Expected 19 REC features."
    )


if len(
    VERDICT_DEPENDENT_REC_FEATURES
) != 13:
    raise AssertionError(
        "Expected 13 verdict-dependent REC features."
    )


if len(
    VERDICT_INDEPENDENT_REC_FEATURES
) != 6:
    raise AssertionError(
        "Expected six verdict-independent REC features."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    | set(VERDICT_INDEPENDENT_REC_FEATURES)
) != set(REC_FEATURE_COLUMNS):
    raise AssertionError(
        "REC feature classes do not cover all 19 features."
    )


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_rec_reconstruction_checkpoint.json"
)


PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)

STEP2A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_status.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)


SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)


REC_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_rec_preflight"
)

BUILD_COMMIT_TOKEN_PROFILE_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_commit_token_profile.csv"
)

COMMIT_MATCHING_AUDIT_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_commit_matching_audit.csv"
)

BUILD_ENTITY_MAP_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_entity_map.csv.gz"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_entity_mapping_summary.json"
)

CLEAN_REC_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_reconstructed.parquet"
)

CLEAN_REC_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_anchor_offsets.parquet"
)

CLEAN_REC_COMPARISON_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_comparison_summary.csv"
)

CLEAN_REC_MISMATCH_EXAMPLES_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_mismatch_examples.csv"
)

CLEAN_ANCHOR_VALIDATION_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_anchor_validation.csv"
)

STEP2B_VALIDATION_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_validation.csv"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_report.json"
)


PROJECT_9_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "camunda__camunda-bpm-platform"
)

PROJECT_9_PROGRESS_PATH = (
    PROJECT_9_DIR
    / "camunda_full_run_control"
    / "camunda_full_run_progress.json"
)


print("=" * 122)
print("=== PROJECT 10 CELL 4 / STEP 2B: CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===")
print("=" * 122)


# ------------------------------------------------------------
# 3. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv_gzip(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression="gzip",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="snappy",
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json_with_retry(
    path,
    attempts=8,
    delay_seconds=0.5,
):
    path = Path(path)

    last_error = None

    for _ in range(attempts):
        try:
            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:
            last_error = error

            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON.\n"
        f"Path: {path}\n"
        f"Error: {type(last_error).__name__}: {last_error}"
    )


def canonical_identifier(series):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:
        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:
            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def create_source_root_sha256(
    source_files_payload,
):
    digest = hashlib.sha256()

    for relative_path in sorted(
        source_files_payload
    ):
        metadata = source_files_payload[
            relative_path
        ]

        runtime_path = Path(
            metadata[
                "RuntimePath"
            ]
        )

        size_bytes = int(
            runtime_path.stat().st_size
        )

        file_sha256 = calculate_hash(
            runtime_path,
            algorithm="sha256",
        )

        line = (
            f"{relative_path}\0"
            f"{size_bytes}\0"
            f"{file_sha256}\n"
        )

        digest.update(
            line.encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


# ------------------------------------------------------------
# 4. COMMIT-PARSING HELPERS
# ------------------------------------------------------------

HEX_COMMIT_PATTERN = re.compile(
    r"(?i)(?<![0-9a-f])[0-9a-f]{7,64}(?![0-9a-f])"
)


def normalise_commit_token(
    value,
):
    if value is None:
        return ""

    token = str(value).strip().lower()

    token = token.strip(
        "\"'[](){}"
    )

    token = token.strip()

    return token


def extract_commit_tokens(
    value,
):
    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):
        collected = []

        for item in value:
            collected.extend(
                extract_commit_tokens(
                    item
                )
            )

        return list(
            dict.fromkeys(
                collected
            )
        )

    if isinstance(
        value,
        dict,
    ):
        collected = []

        for item in value.values():
            collected.extend(
                extract_commit_tokens(
                    item
                )
            )

        return list(
            dict.fromkeys(
                collected
            )
        )

    text = str(value).strip()

    if text == "":
        return []

    if text[0:1] in {
        "[",
        "(",
        "{",
    }:
        try:
            parsed = ast.literal_eval(
                text
            )

            if parsed != value:
                parsed_tokens = (
                    extract_commit_tokens(
                        parsed
                    )
                )

                if parsed_tokens:
                    return parsed_tokens

        except Exception:
            pass

    regex_tokens = [
        normalise_commit_token(
            token
        )
        for token in HEX_COMMIT_PATTERN.findall(
            text
        )
    ]

    regex_tokens = [
        token
        for token in regex_tokens
        if token
    ]

    if regex_tokens:
        return list(
            dict.fromkeys(
                regex_tokens
            )
        )

    fallback_parts = re.split(
        r"[\s,;|]+",
        text,
    )

    fallback_tokens = [
        normalise_commit_token(
            token
        )
        for token in fallback_parts
    ]

    fallback_tokens = [
        token
        for token in fallback_tokens
        if token
    ]

    return list(
        dict.fromkeys(
            fallback_tokens
        )
    )


# ------------------------------------------------------------
# 5. INPUT VALIDATION
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    STEP2A_STATUS_PATH,
    STEP2A_REPORT_PATH,
]


missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:
    raise FileNotFoundError(
        "Required Project 10 Step 2B inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


selection_checkpoint = read_json_with_retry(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = read_json_with_retry(
    STEP1B_STATUS_PATH
)

step2a_status = read_json_with_retry(
    STEP2A_STATUS_PATH
)

step2a_report = read_json_with_retry(
    STEP2A_REPORT_PATH
)


if selection_checkpoint.get(
    "Status"
) != EXPECTED_STEP1B_STATUS:
    raise AssertionError(
        "Project 10 selection-checkpoint status differs."
    )


if step1b_status.get(
    "Status"
) != EXPECTED_STEP1B_STATUS:
    raise AssertionError(
        "Project 10 Step 1B status differs."
    )


if step2a_status.get(
    "Status"
) != EXPECTED_STEP2A_STATUS:
    raise AssertionError(
        "Project 10 Step 2A status differs."
    )


if step2a_report.get(
    "Status"
) != EXPECTED_STEP2A_STATUS:
    raise AssertionError(
        "Project 10 Step 2A report status differs."
    )


if selection_checkpoint.get(
    "Project"
) != PROJECT_NAME:
    raise AssertionError(
        "Project 10 identity differs."
    )


if selection_checkpoint.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise AssertionError(
        "Project 10 slug differs."
    )


if selection_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Project 10 source-root SHA-256 differs."
    )


# ------------------------------------------------------------
# 6. OUTPUT-PATH ISOLATION
# ------------------------------------------------------------

output_paths = [
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    COMMIT_MATCHING_AUDIT_PATH,
    BUILD_ENTITY_MAP_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    CLEAN_REC_RECONSTRUCTED_PATH,
    CLEAN_REC_ANCHOR_OFFSETS_PATH,
    CLEAN_REC_COMPARISON_PATH,
    CLEAN_REC_MISMATCH_EXAMPLES_PATH,
    CLEAN_ANCHOR_VALIDATION_PATH,
    STEP2B_VALIDATION_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
]


for output_path in output_paths:
    output_string = str(
        output_path
    )

    if (
        PROJECT_SLUG not in output_string
        and "project_10_" not in output_string
    ):
        raise AssertionError(
            "A Step 2B output path is not Project 10 isolated.\n"
            f"Path: {output_path}"
        )

    if str(
        PROJECT_9_DIR
    ) in output_string:
        raise AssertionError(
            "A Project 10 output path overlaps Project 9."
        )


# ------------------------------------------------------------
# 7. REGISTRY — READ ONLY
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != 8
    or set(
        registry_project_numbers
    ) != set(
        range(1, 9)
    )
):
    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )


if registry_project_numbers.eq(
    9
).any():
    raise AssertionError(
        "Project 9 was unexpectedly registered."
    )


if registry_project_numbers.eq(
    10
).any():
    raise AssertionError(
        "Project 10 was unexpectedly registered."
    )


# ------------------------------------------------------------
# 8. VERIFY FROZEN SOURCE FILES
# ------------------------------------------------------------

source_files_payload = (
    selection_checkpoint[
        "SourceFiles"
    ]
)


if len(
    source_files_payload
) != EXPECTED_SOURCE_FILES:
    raise AssertionError(
        "Frozen source-file count differs.\n"
        f"Expected: {EXPECTED_SOURCE_FILES}\n"
        f"Actual:   {len(source_files_payload)}"
    )


for relative_path, metadata in (
    source_files_payload.items()
):
    runtime_path = Path(
        metadata[
            "RuntimePath"
        ]
    )

    if not runtime_path.exists():
        raise FileNotFoundError(
            "Frozen Project 10 source file is missing:\n"
            f"{runtime_path}"
        )

    actual_size = int(
        runtime_path.stat().st_size
    )

    expected_size = int(
        metadata[
            "SizeBytes"
        ]
    )

    if actual_size != expected_size:
        raise AssertionError(
            "Frozen Project 10 source-file size differs.\n"
            f"File: {relative_path}"
        )

    actual_sha256 = calculate_hash(
        runtime_path,
        algorithm="sha256",
    )

    expected_sha256 = str(
        metadata[
            "SHA256"
        ]
    )

    if actual_sha256 != expected_sha256:
        raise AssertionError(
            "Frozen Project 10 source-file hash differs.\n"
            f"File: {relative_path}"
        )


source_root_sha256 = create_source_root_sha256(
    source_files_payload
)


if source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Recomputed source-root SHA-256 differs."
    )


# ------------------------------------------------------------
# 9. LOAD SOURCE TABLES
# ------------------------------------------------------------

builds_path = Path(
    selection_checkpoint[
        "BuildsPath"
    ]
)

executions_path = Path(
    selection_checkpoint[
        "ExecutionHistoryPath"
    ]
)

dataset_path = Path(
    selection_checkpoint[
        "DatasetPath"
    ]
)

id_map_path = Path(
    selection_checkpoint[
        "IdMapPath"
    ]
)

entity_history_path = Path(
    selection_checkpoint[
        "EntityHistoryPath"
    ]
)

fixed_split_path = Path(
    selection_checkpoint[
        "FixedSplit"
    ]
)


if calculate_hash(
    fixed_split_path,
    algorithm="sha256",
) != selection_checkpoint[
    "FixedSplitSHA256"
]:
    raise AssertionError(
        "Frozen chronological split hash differs."
    )


builds = pd.read_csv(
    builds_path,
    low_memory=False,
)

executions = pd.read_csv(
    executions_path,
    low_memory=False,
)

dataset = pd.read_csv(
    dataset_path,
    low_memory=False,
)

id_map = pd.read_csv(
    id_map_path,
    low_memory=False,
)

entity_history = pd.read_csv(
    entity_history_path,
    low_memory=False,
)

fixed_split = pd.read_csv(
    fixed_split_path,
    low_memory=False,
)


if len(builds) != EXPECTED_BUILDS:
    raise AssertionError(
        "builds.csv row count differs."
    )


if len(executions) != EXPECTED_RAW_ROWS:
    raise AssertionError(
        "exe.csv row count differs."
    )


if len(dataset) != EXPECTED_MODEL_ROWS:
    raise AssertionError(
        "dataset.csv row count differs."
    )


# ------------------------------------------------------------
# 10. RESOLVED SCHEMA
# ------------------------------------------------------------

resolved_columns = (
    step2a_report[
        "ResolvedColumns"
    ]
)


build_id_column = (
    resolved_columns[
        "BuildsBuildID"
    ]
)

build_commits_column = (
    resolved_columns[
        "BuildsCommits"
    ]
)

execution_build_column = (
    resolved_columns[
        "ExecutionBuild"
    ]
)

execution_job_column = (
    resolved_columns[
        "ExecutionJob"
    ]
)

execution_test_column = (
    resolved_columns[
        "ExecutionTest"
    ]
)

execution_verdict_column = (
    resolved_columns[
        "ExecutionVerdict"
    ]
)

execution_duration_column = (
    resolved_columns[
        "ExecutionDuration"
    ]
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)

id_map_key_column = (
    resolved_columns[
        "IDMapKey"
    ]
)

id_map_value_column = (
    resolved_columns[
        "IDMapValue"
    ]
)

id_map_entity_column = (
    resolved_columns[
        "IDMapEntityColumn"
    ]
)

id_map_entity_side = (
    resolved_columns[
        "IDMapEntitySide"
    ]
)

entity_history_commit_column = (
    resolved_columns[
        "EntityHistoryCommit"
    ]
)

entity_history_id_column = (
    resolved_columns[
        "EntityHistoryID"
    ]
)


print("\nResolved reconstruction schema:")

print(
    "Build commit column:",
    build_commits_column,
)

print(
    "Entity-history commit column:",
    entity_history_commit_column,
)

print(
    "Entity-history ID column:",
    entity_history_id_column,
)

print(
    "ID-map key / value:",
    id_map_key_column,
    "/",
    id_map_value_column,
)

print(
    "Resolved ID-map entity side:",
    id_map_entity_side,
)


# ------------------------------------------------------------
# 11. PREPARE CANONICAL BUILD ORDER
# ------------------------------------------------------------

fixed_split[
    "BuildKey"
] = fixed_split[
    "BuildKey"
].astype(str)

fixed_split[
    "BuildOrder"
] = pd.to_numeric(
    fixed_split[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


if fixed_split.duplicated(
    subset=[
        "BuildKey",
    ],
    keep=False,
).any():
    raise AssertionError(
        "Frozen split contains duplicate BuildKey rows."
    )


build_order_map = (
    fixed_split.set_index(
        "BuildKey"
    )[
        "BuildOrder"
    ].to_dict()
)


builds_working = builds.copy()

builds_working[
    "BuildKey"
] = canonical_identifier(
    builds_working[
        build_id_column
    ]
)

builds_working[
    "BuildOrder"
] = builds_working[
    "BuildKey"
].map(
    build_order_map
)


missing_build_orders = int(
    builds_working[
        "BuildOrder"
    ].isna().sum()
)


if missing_build_orders:
    raise AssertionError(
        "Some builds.csv rows could not be assigned a frozen build order."
    )


builds_working[
    "BuildOrder"
] = pd.to_numeric(
    builds_working[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


builds_working = (
    builds_working.sort_values(
        "BuildOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 12. PREPARE ENTITY-HISTORY COMMIT MAP
# ------------------------------------------------------------

entity_history_working = entity_history.copy()

entity_history_working[
    "NormalisedCommit"
] = (
    entity_history_working[
        entity_history_commit_column
    ]
    .map(
        normalise_commit_token
    )
)

entity_history_working[
    "EntityKey"
] = canonical_identifier(
    entity_history_working[
        entity_history_id_column
    ]
)


empty_entity_commit_rows = int(
    entity_history_working[
        "NormalisedCommit"
    ].eq("").sum()
)

empty_entity_id_rows = int(
    entity_history_working[
        "EntityKey"
    ].eq("").sum()
)


if empty_entity_commit_rows:
    raise AssertionError(
        "Entity history contains empty commit identifiers."
    )


if empty_entity_id_rows:
    raise AssertionError(
        "Entity history contains empty entity identifiers."
    )


entity_commit_to_entities = (
    entity_history_working.groupby(
        "NormalisedCommit",
        sort=False,
    )[
        "EntityKey"
    ]
    .apply(
        lambda values:
            set(
                values.astype(str)
            )
    )
    .to_dict()
)


normalised_entity_commits = sorted(
    entity_commit_to_entities
)


id_map_entity_keys = set(
    canonical_identifier(
        id_map[
            id_map_entity_column
        ]
    ).astype(str)
)


entity_history_entity_keys = set(
    entity_history_working[
        "EntityKey"
    ].astype(str)
)


unmapped_entity_ids = (
    entity_history_entity_keys
    - id_map_entity_keys
)


if unmapped_entity_ids:
    raise AssertionError(
        "Entity-history identifiers are missing from "
        "the resolved ID-map entity column.\n"
        f"Missing unique IDs: {len(unmapped_entity_ids)}"
    )


# ------------------------------------------------------------
# 13. TOKENISE AND MATCH BUILD COMMITS
# ------------------------------------------------------------

token_profile_records = []
commit_matching_records = []

changed_entities_by_build = {
    str(build_key):
        set()
    for build_key in fixed_split[
        "BuildKey"
    ].astype(str)
}


builds_without_commit_tokens = 0
builds_with_no_matched_commit = 0

exact_commit_matches = 0
unique_prefix_matches = 0
unmatched_commit_tokens = 0
ambiguous_commit_tokens = 0

total_commit_tokens = 0


for build_row in builds_working.itertuples(
    index=False
):
    build_key = str(
        getattr(
            build_row,
            "BuildKey",
        )
    )

    build_order = int(
        getattr(
            build_row,
            "BuildOrder",
        )
    )

    raw_commit_value = getattr(
        build_row,
        build_commits_column,
    )

    commit_tokens = extract_commit_tokens(
        raw_commit_value
    )

    if len(commit_tokens) == 0:
        builds_without_commit_tokens += 1
        continue

    matched_tokens_for_build = 0

    for token_order, commit_token in enumerate(
        commit_tokens,
        start=1,
    ):
        total_commit_tokens += 1

        commit_token = normalise_commit_token(
            commit_token
        )

        match_type = None
        matched_commit = None
        prefix_candidates = []

        if commit_token in entity_commit_to_entities:
            match_type = (
                "EXACT_NORMALISED_COMMIT_TOKEN"
            )

            matched_commit = commit_token

            exact_commit_matches += 1

        else:
            prefix_candidates = [
                entity_commit
                for entity_commit
                in normalised_entity_commits
                if (
                    entity_commit.startswith(
                        commit_token
                    )
                    or commit_token.startswith(
                        entity_commit
                    )
                )
            ]

            prefix_candidates = list(
                dict.fromkeys(
                    prefix_candidates
                )
            )

            if len(prefix_candidates) == 1:
                match_type = (
                    "UNIQUE_NORMALISED_PREFIX"
                )

                matched_commit = (
                    prefix_candidates[0]
                )

                unique_prefix_matches += 1

            elif len(prefix_candidates) == 0:
                match_type = (
                    "UNMATCHED_COMMIT_TOKEN"
                )

                unmatched_commit_tokens += 1

            else:
                match_type = (
                    "AMBIGUOUS_COMMIT_TOKEN"
                )

                ambiguous_commit_tokens += 1

        matched_entities = set()

        if matched_commit is not None:
            matched_tokens_for_build += 1

            matched_entities = (
                entity_commit_to_entities[
                    matched_commit
                ]
            )

            changed_entities_by_build[
                build_key
            ].update(
                matched_entities
            )

        token_profile_records.append({
            "BuildKey":
                build_key,

            "BuildOrder":
                build_order,

            "TokenOrder":
                token_order,

            "CommitToken":
                commit_token,

            "RawCommitValue":
                str(
                    raw_commit_value
                ),

            "Matched":
                matched_commit is not None,

            "MatchedCommit":
                matched_commit,

            "MatchType":
                match_type,

            "CandidateMatches":
                len(
                    prefix_candidates
                ),

            "MatchedEntities":
                len(
                    matched_entities
                ),
        })

        commit_matching_records.append({
            "BuildKey":
                build_key,

            "BuildOrder":
                build_order,

            "TokenOrder":
                token_order,

            "CommitToken":
                commit_token,

            "MatchType":
                match_type,

            "MatchedCommit":
                matched_commit,

            "PrefixCandidateCount":
                len(
                    prefix_candidates
                ),

            "PrefixCandidates":
                ",".join(
                    prefix_candidates[:20]
                ),

            "MatchedEntityCount":
                len(
                    matched_entities
                ),
        })

    if matched_tokens_for_build == 0:
        builds_with_no_matched_commit += 1


build_commit_token_profile = pd.DataFrame(
    token_profile_records
)

commit_matching_audit = pd.DataFrame(
    commit_matching_records
)


matched_commit_tokens = (
    exact_commit_matches
    + unique_prefix_matches
)


commit_token_coverage_percent = (
    100.0
    if total_commit_tokens == 0
    else 100.0
    * matched_commit_tokens
    / total_commit_tokens
)


build_entity_records = []


for build_key in fixed_split.sort_values(
    "BuildOrder",
    kind="mergesort",
)[
    "BuildKey"
].astype(str):
    entities = sorted(
        changed_entities_by_build.get(
            build_key,
            set(),
        )
    )

    build_order = int(
        build_order_map[
            build_key
        ]
    )

    for entity_key in entities:
        build_entity_records.append({
            "BuildKey":
                build_key,

            "BuildOrder":
                build_order,

            "EntityKey":
                str(
                    entity_key
                ),
        })


build_entity_map = pd.DataFrame(
    build_entity_records,
    columns=[
        "BuildKey",
        "BuildOrder",
        "EntityKey",
    ],
)


builds_with_mapped_entities = int(
    sum(
        len(
            entities
        ) > 0
        for entities
        in changed_entities_by_build.values()
    )
)


entity_mapping_summary = {
    "Project":
        PROJECT_NAME,

    "BuildCommitColumn":
        build_commits_column,

    "EntityHistoryCommitColumn":
        entity_history_commit_column,

    "EntityHistoryIDColumn":
        entity_history_id_column,

    "IDMapEntityColumn":
        id_map_entity_column,

    "IDMapEntitySide":
        id_map_entity_side,

    "Builds":
        EXPECTED_BUILDS,

    "BuildCommitTokenRows":
        total_commit_tokens,

    "ExactCommitMatches":
        exact_commit_matches,

    "UniquePrefixMatches":
        unique_prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "AmbiguousCommitTokens":
        ambiguous_commit_tokens,

    "MatchedCommitTokens":
        matched_commit_tokens,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithoutCommitTokens":
        builds_without_commit_tokens,

    "BuildsWithNoMatchedCommit":
        builds_with_no_matched_commit,

    "BuildsWithMappedEntities":
        builds_with_mapped_entities,

    "BuildEntityRows":
        len(
            build_entity_map
        ),

    "UniqueMappedEntities":
        int(
            build_entity_map[
                "EntityKey"
            ].nunique()
            if not build_entity_map.empty
            else 0
        ),

    "UnmappedEntityHistoryIDs":
        len(
            unmapped_entity_ids
        ),
}


print("\nCommit/entity mapping summary:")

print(
    "Build commit tokens:",
    total_commit_tokens,
)

print(
    "Exact matches:",
    exact_commit_matches,
)

print(
    "Unique-prefix matches:",
    unique_prefix_matches,
)

print(
    "Unmatched tokens:",
    unmatched_commit_tokens,
)

print(
    "Ambiguous tokens:",
    ambiguous_commit_tokens,
)

print(
    "Token coverage percent:",
    commit_token_coverage_percent,
)

print(
    "Build rows without commit tokens:",
    builds_without_commit_tokens,
)

print(
    "Builds with no matched commit:",
    builds_with_no_matched_commit,
)

print(
    "Builds with mapped entities:",
    builds_with_mapped_entities,
)

print(
    "Build/entity rows:",
    len(
        build_entity_map
    ),
)


# ------------------------------------------------------------
# 14. PREPARE ENTITY ↔ BUILD HISTORY
# ------------------------------------------------------------

entity_changed_builds = defaultdict(
    set
)


for row in build_entity_map.itertuples(
    index=False
):
    entity_changed_builds[
        str(
            row.EntityKey
        )
    ].add(
        str(
            row.BuildKey
        )
    )


# ------------------------------------------------------------
# 15. PREPARE CHRONOLOGICAL RAW EXECUTION HISTORY
# ------------------------------------------------------------

execution_history = pd.DataFrame({
    "RawRowOrder":
        np.arange(
            len(executions),
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            executions[
                execution_build_column
            ]
        ),

    "JobKey":
        canonical_identifier(
            executions[
                execution_job_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            executions[
                execution_test_column
            ]
        ),

    "Verdict":
        pd.to_numeric(
            executions[
                execution_verdict_column
            ],
            errors="coerce",
        ),

    "Duration":
        pd.to_numeric(
            executions[
                execution_duration_column
            ],
            errors="coerce",
        ),
})


execution_history[
    "BuildOrder"
] = execution_history[
    "BuildKey"
].map(
    build_order_map
)


if execution_history[
    "BuildOrder"
].isna().any():
    raise AssertionError(
        "Some raw execution rows could not be assigned "
        "a frozen build order."
    )


if execution_history[
    [
        "Verdict",
        "Duration",
    ]
].isna().any().any():
    raise AssertionError(
        "Raw execution verdicts or durations contain "
        "unparseable values."
    )


execution_history[
    "BuildOrder"
] = pd.to_numeric(
    execution_history[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)

execution_history[
    "Verdict"
] = pd.to_numeric(
    execution_history[
        "Verdict"
    ],
    errors="raise",
).astype(np.int32)

execution_history[
    "Duration"
] = pd.to_numeric(
    execution_history[
        "Duration"
    ],
    errors="raise",
).astype(float)


if not np.isfinite(
    execution_history[
        "Duration"
    ].to_numpy(
        dtype=float
    )
).all():
    raise AssertionError(
        "Raw execution durations contain non-finite values."
    )


if execution_history[
    "Duration"
].lt(0).any():
    raise AssertionError(
        "Raw execution durations contain negative values."
    )


raw_duplicate_build_test_rows = int(
    execution_history.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if raw_duplicate_build_test_rows:
    raise AssertionError(
        "Raw execution history contains duplicate "
        "canonical Build/Test rows."
    )


execution_history = (
    execution_history.sort_values(
        [
            "BuildOrder",
            "JobKey",
            "TestKey",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


ordered_execution_builds = (
    execution_history[
        [
            "BuildKey",
            "BuildOrder",
        ]
    ]
    .drop_duplicates(
        subset=[
            "BuildKey",
        ]
    )
    .sort_values(
        "BuildOrder",
        kind="mergesort",
    )[
        "BuildKey"
    ]
    .astype(str)
    .tolist()
)


global_build_position = {
    build_key:
        position
    for position, build_key
    in enumerate(
        ordered_execution_builds
    )
}


# ------------------------------------------------------------
# 16. PREPARE MODEL-READY REQUESTED ROWS
# ------------------------------------------------------------

requested_rows = pd.DataFrame({
    "ModelRowOrder":
        np.arange(
            len(dataset),
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            dataset[
                dataset_build_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            dataset[
                dataset_test_column
            ]
        ),
})


if requested_rows.duplicated(
    subset=[
        "BuildKey",
        "TestKey",
    ],
    keep=False,
).any():
    raise AssertionError(
        "Model-ready data contains duplicate Build/Test rows."
    )


requested_builds_by_test = {
    str(test_key):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for test_key, group
    in requested_rows.groupby(
        "TestKey",
        sort=False,
    )
}


# ------------------------------------------------------------
# 17. REC HELPER FUNCTIONS
# ------------------------------------------------------------

def calculate_rates(
    history,
):
    history_length = len(
        history
    )

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history[
        "Verdict"
    ].to_numpy(
        dtype=np.int32
    )

    transitions = history[
        "Transition"
    ].to_numpy(
        dtype=np.int32
    )

    return (
        float(
            np.count_nonzero(
                verdicts != 0
            )
            / history_length
        ),

        float(
            np.count_nonzero(
                verdicts == 2
            )
            / history_length
        ),

        float(
            np.count_nonzero(
                verdicts == 1
            )
            / history_length
        ),

        float(
            np.count_nonzero(
                transitions != 0
            )
            / history_length
        ),
    )


def calculate_max_test_file_rate(
    history,
    target_type,
    current_changed_entities,
):
    if target_type == "FAILURE":
        target_builds = set(
            history.loc[
                history[
                    "Verdict"
                ].ne(0),
                "BuildKey",
            ].astype(str)
        )

    elif target_type == "TRANSITION":
        target_builds = set(
            history.loc[
                history[
                    "Transition"
                ].ne(0),
                "BuildKey",
            ].astype(str)
        )

    else:
        raise ValueError(
            "Unknown file-history target type."
        )

    if len(target_builds) == 0:
        return -1.0

    maximum_overlap = 0

    for entity_key in current_changed_entities:
        overlap_count = len(
            entity_changed_builds.get(
                str(
                    entity_key
                ),
                set(),
            )
            & target_builds
        )

        if overlap_count > maximum_overlap:
            maximum_overlap = overlap_count

    if maximum_overlap == 0:
        return 0.0

    return float(
        maximum_overlap
        / len(
            target_builds
        )
    )


# ------------------------------------------------------------
# 18. RECONSTRUCT ALL 19 CLEAN REC FEATURES
# ------------------------------------------------------------

print(
    "\nReconstructing all 19 clean REC features..."
)

reconstruction_started = (
    time.perf_counter()
)

reconstructed_records = []

test_groups = list(
    execution_history.groupby(
        "TestKey",
        sort=False,
    )
)

total_tests = len(
    test_groups
)


for test_number, (
    test_key,
    test_history,
) in enumerate(
    test_groups,
    start=1,
):
    test_key = str(
        test_key
    )

    requested_builds = (
        requested_builds_by_test.get(
            test_key
        )
    )

    if not requested_builds:
        continue

    test_history = (
        test_history.sort_values(
            [
                "BuildOrder",
                "JobKey",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    test_history[
        "Transition"
    ] = (
        test_history[
            "Verdict"
        ]
        .ne(
            test_history[
                "Verdict"
            ].shift()
        )
        .astype(np.int32)
    )

    test_history.loc[
        0,
        "Transition",
    ] = 0

    first_test_build = str(
        test_history[
            "BuildKey"
        ].iloc[0]
    )

    if first_test_build not in (
        global_build_position
    ):
        raise AssertionError(
            "First test build is absent from global build positions."
        )

    for execution_position, current_row in (
        test_history.iterrows()
    ):
        current_build = str(
            current_row[
                "BuildKey"
            ]
        )

        if current_build not in requested_builds:
            continue

        record = {
            "BuildKey":
                current_build,

            "TestKey":
                test_key,
        }

        history = test_history.iloc[
            :execution_position
        ]

        if len(history) == 0:
            for feature in REC_FEATURE_COLUMNS:
                record[
                    feature
                ] = -1.0

            record[
                "REC_Age"
            ] = 0.0

            reconstructed_records.append(
                record
            )

            continue

        recent_history = (
            history.tail(
                RECENT_WINDOW
            )
        )

        current_global_position = (
            global_build_position[
                current_build
            ]
        )

        first_global_position = (
            global_build_position[
                first_test_build
            ]
        )

        age = float(
            current_global_position
            - first_global_position
        )

        failure_positions = np.flatnonzero(
            history[
                "Verdict"
            ].to_numpy(
                dtype=np.int32
            ) != 0
        )

        if len(failure_positions) == 0:
            last_failure_age = -1.0
        else:
            last_failure_age = float(
                len(history)
                - 1
                - int(
                    failure_positions[-1]
                )
            )

        transition_positions = np.flatnonzero(
            history[
                "Transition"
            ].to_numpy(
                dtype=np.int32
            ) != 0
        )

        if len(transition_positions) == 0:
            last_transition_age = -1.0
        else:
            last_transition_age = float(
                len(history)
                - 1
                - int(
                    transition_positions[-1]
                )
            )

        (
            recent_fail_rate,
            recent_assert_rate,
            recent_exc_rate,
            recent_transition_rate,
        ) = calculate_rates(
            recent_history
        )

        (
            total_fail_rate,
            total_assert_rate,
            total_exc_rate,
            total_transition_rate,
        ) = calculate_rates(
            history
        )

        current_changed_entities = (
            changed_entities_by_build.get(
                current_build,
                set(),
            )
        )

        max_test_file_fail_rate = (
            calculate_max_test_file_rate(
                history=history,
                target_type="FAILURE",
                current_changed_entities=(
                    current_changed_entities
                ),
            )
        )

        max_test_file_transition_rate = (
            calculate_max_test_file_rate(
                history=history,
                target_type="TRANSITION",
                current_changed_entities=(
                    current_changed_entities
                ),
            )
        )

        record.update({
            "REC_Age":
                age,

            "REC_LastFailureAge":
                last_failure_age,

            "REC_LastTransitionAge":
                last_transition_age,

            "REC_RecentAvgExeTime":
                float(
                    recent_history[
                        "Duration"
                    ].mean()
                ),

            "REC_RecentMaxExeTime":
                float(
                    recent_history[
                        "Duration"
                    ].max()
                ),

            "REC_RecentFailRate":
                recent_fail_rate,

            "REC_RecentAssertRate":
                recent_assert_rate,

            "REC_RecentExcRate":
                recent_exc_rate,

            "REC_RecentTransitionRate":
                recent_transition_rate,

            "REC_TotalAvgExeTime":
                float(
                    history[
                        "Duration"
                    ].mean()
                ),

            "REC_TotalMaxExeTime":
                float(
                    history[
                        "Duration"
                    ].max()
                ),

            "REC_TotalFailRate":
                total_fail_rate,

            "REC_TotalAssertRate":
                total_assert_rate,

            "REC_TotalExcRate":
                total_exc_rate,

            "REC_TotalTransitionRate":
                total_transition_rate,

            "REC_LastVerdict":
                float(
                    recent_history[
                        "Verdict"
                    ].iloc[-1]
                ),

            "REC_LastExeTime":
                float(
                    recent_history[
                        "Duration"
                    ].iloc[-1]
                ),

            "REC_MaxTestFileFailRate":
                max_test_file_fail_rate,

            "REC_MaxTestFileTransitionRate":
                max_test_file_transition_rate,
        })

        reconstructed_records.append(
            record
        )

    if (
        test_number % 100 == 0
        or test_number == total_tests
    ):
        print(
            "REC reconstruction progress:",
            test_number,
            "/",
            total_tests,
            "tests | reconstructed rows:",
            len(
                reconstructed_records
            ),
        )


clean_rec_reconstructed_unaligned = (
    pd.DataFrame(
        reconstructed_records,
        columns=(
            [
                "BuildKey",
                "TestKey",
            ]
            + REC_FEATURE_COLUMNS
        ),
    )
)


reconstruction_seconds = float(
    time.perf_counter()
    - reconstruction_started
)


duplicate_reconstructed_rows = int(
    clean_rec_reconstructed_unaligned.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if duplicate_reconstructed_rows:
    raise AssertionError(
        "Clean REC reconstruction contains duplicate "
        "Build/Test rows."
    )


aligned_clean_rec = (
    requested_rows.merge(
        clean_rec_reconstructed_unaligned,
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_reconstructed_rows = int(
    aligned_clean_rec[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


if missing_reconstructed_rows:
    raise AssertionError(
        "Some model-ready rows did not receive "
        "clean reconstructed REC features."
    )


clean_rec_reconstructed = aligned_clean_rec[
    [
        "BuildKey",
        "TestKey",
    ]
    + REC_FEATURE_COLUMNS
].copy()


print("\nClean REC reconstruction completed:")

print(
    "Requested rows:",
    len(
        requested_rows
    ),
)

print(
    "Reconstructed rows:",
    len(
        clean_rec_reconstructed
    ),
)

print(
    "Duplicate reconstructed rows:",
    duplicate_reconstructed_rows,
)

print(
    "Reconstruction seconds:",
    reconstruction_seconds,
)


# ------------------------------------------------------------
# 19. COMPARE AGAINST ORIGINAL CLEAN DATASET
# ------------------------------------------------------------

original_rec_values = (
    dataset[
        REC_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)


direct_rec_values = (
    clean_rec_reconstructed[
        REC_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)


if original_rec_values.isna().any().any():
    raise AssertionError(
        "Original dataset contains unparseable REC values."
    )


if direct_rec_values.isna().any().any():
    raise AssertionError(
        "Direct reconstruction contains missing REC values."
    )


anchor_offsets = (
    original_rec_values
    - direct_rec_values
)


clean_rec_anchor_offsets = pd.concat(
    [
        requested_rows[
            [
                "BuildKey",
                "TestKey",
            ]
        ].reset_index(
            drop=True
        ),

        anchor_offsets.reset_index(
            drop=True
        ),
    ],
    axis=1,
)


comparison_records = []
mismatch_example_records = []

direct_mismatching_values = 0
verdict_dependent_direct_mismatches = 0
verdict_independent_direct_mismatches = 0
file_history_direct_mismatches = 0

nonzero_anchor_values = 0


for feature in REC_FEATURE_COLUMNS:
    original = original_rec_values[
        feature
    ].to_numpy(
        dtype=float
    )

    direct = direct_rec_values[
        feature
    ].to_numpy(
        dtype=float
    )

    offsets = anchor_offsets[
        feature
    ].to_numpy(
        dtype=float
    )

    direct_matches = np.isclose(
        original,
        direct,
        rtol=DIRECT_COMPARISON_RTOL,
        atol=DIRECT_COMPARISON_ATOL,
        equal_nan=False,
    )

    direct_mismatch_count = int(
        (
            ~direct_matches
        ).sum()
    )

    direct_match_count = int(
        direct_matches.sum()
    )

    direct_mismatching_values += (
        direct_mismatch_count
    )

    if feature in (
        VERDICT_DEPENDENT_REC_FEATURES
    ):
        feature_class = (
            "VERDICT_DEPENDENT"
        )

        verdict_dependent_direct_mismatches += (
            direct_mismatch_count
        )

    else:
        feature_class = (
            "VERDICT_INDEPENDENT"
        )

        verdict_independent_direct_mismatches += (
            direct_mismatch_count
        )

    file_history_feature = (
        feature
        in FILE_HISTORY_REC_FEATURES
    )

    if file_history_feature:
        file_history_direct_mismatches += (
            direct_mismatch_count
        )

    nonzero_offsets = (
        np.abs(
            offsets
        )
        > NONZERO_OFFSET_THRESHOLD
    )

    nonzero_offset_count = int(
        nonzero_offsets.sum()
    )

    nonzero_anchor_values += (
        nonzero_offset_count
    )

    anchored = (
        direct
        + offsets
    )

    anchored_matches = np.isclose(
        original,
        anchored,
        rtol=ANCHOR_COMPARISON_RTOL,
        atol=ANCHOR_COMPARISON_ATOL,
        equal_nan=False,
    )

    anchored_mismatch_count = int(
        (
            ~anchored_matches
        ).sum()
    )

    maximum_absolute_direct_difference = float(
        np.max(
            np.abs(
                original
                - direct
            )
        )
    )

    mean_absolute_direct_difference = float(
        np.mean(
            np.abs(
                original
                - direct
            )
        )
    )

    comparison_records.append({
        "Feature":
            feature,

        "FeatureClass":
            feature_class,

        "FileHistoryFeature":
            file_history_feature,

        "Rows":
            len(
                original
            ),

        "DirectMatchingRows":
            direct_match_count,

        "DirectMismatchingRows":
            direct_mismatch_count,

        "NonZeroAnchorOffsets":
            nonzero_offset_count,

        "AnchoredMatchingRows":
            int(
                anchored_matches.sum()
            ),

        "AnchoredMismatchingRows":
            anchored_mismatch_count,

        "MaximumAbsoluteDirectDifference":
            maximum_absolute_direct_difference,

        "MeanAbsoluteDirectDifference":
            mean_absolute_direct_difference,
    })

    if direct_mismatch_count:
        mismatch_positions = np.flatnonzero(
            ~direct_matches
        )[:100]

        for position in mismatch_positions:
            mismatch_example_records.append({
                "Feature":
                    feature,

                "ModelRowOrder":
                    int(
                        position
                    ),

                "BuildKey":
                    requested_rows[
                        "BuildKey"
                    ].iloc[
                        position
                    ],

                "TestKey":
                    requested_rows[
                        "TestKey"
                    ].iloc[
                        position
                    ],

                "OriginalValue":
                    float(
                        original[
                            position
                        ]
                    ),

                "DirectValue":
                    float(
                        direct[
                            position
                        ]
                    ),

                "Difference":
                    float(
                        original[
                            position
                        ]
                        - direct[
                            position
                        ]
                    ),
            })


clean_rec_comparison = pd.DataFrame(
    comparison_records
)

clean_rec_mismatch_examples = pd.DataFrame(
    mismatch_example_records,
    columns=[
        "Feature",
        "ModelRowOrder",
        "BuildKey",
        "TestKey",
        "OriginalValue",
        "DirectValue",
        "Difference",
    ],
)


rows_with_nonzero_anchor = int(
    (
        np.abs(
            anchor_offsets.to_numpy(
                dtype=float
            )
        )
        > NONZERO_OFFSET_THRESHOLD
    ).any(
        axis=1
    ).sum()
)


# ------------------------------------------------------------
# 20. CLEAN-ANCHOR VALIDATION
# ------------------------------------------------------------

anchor_validation_records = []

anchored_mismatching_values = 0


for feature in REC_FEATURE_COLUMNS:
    original = original_rec_values[
        feature
    ].to_numpy(
        dtype=float
    )

    direct = direct_rec_values[
        feature
    ].to_numpy(
        dtype=float
    )

    offsets = anchor_offsets[
        feature
    ].to_numpy(
        dtype=float
    )

    anchored = (
        direct
        + offsets
    )

    matches = np.isclose(
        original,
        anchored,
        rtol=ANCHOR_COMPARISON_RTOL,
        atol=ANCHOR_COMPARISON_ATOL,
        equal_nan=False,
    )

    mismatch_count = int(
        (
            ~matches
        ).sum()
    )

    anchored_mismatching_values += (
        mismatch_count
    )

    maximum_absolute_difference = float(
        np.max(
            np.abs(
                original
                - anchored
            )
        )
    )

    anchor_validation_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature
                in VERDICT_DEPENDENT_REC_FEATURES
                else "VERDICT_INDEPENDENT"
            ),

        "Rows":
            len(
                original
            ),

        "MatchingRows":
            int(
                matches.sum()
            ),

        "MismatchingRows":
            mismatch_count,

        "MaximumAbsoluteAnchoredDifference":
            maximum_absolute_difference,

        "Pass":
            mismatch_count == 0,
    })


clean_anchor_validation = pd.DataFrame(
    anchor_validation_records
)


failed_anchor_features = int(
    (
        ~clean_anchor_validation[
            "Pass"
        ]
    ).sum()
)


clean_dataset_reproduced = (
    anchored_mismatching_values == 0
)


# ------------------------------------------------------------
# 21. VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 1B passed",

        "Expected":
            EXPECTED_STEP1B_STATUS,

        "Actual":
            step1b_status.get(
                "Status"
            ),

        "Pass":
            step1b_status.get(
                "Status"
            ) == EXPECTED_STEP1B_STATUS,
    },

    {
        "Check":
            "Step 2A passed",

        "Expected":
            EXPECTED_STEP2A_STATUS,

        "Actual":
            step2a_status.get(
                "Status"
            ),

        "Pass":
            step2a_status.get(
                "Status"
            ) == EXPECTED_STEP2A_STATUS,
    },

    {
        "Check":
            "Source root SHA-256",

        "Expected":
            EXPECTED_SOURCE_ROOT_SHA256,

        "Actual":
            source_root_sha256,

        "Pass":
            source_root_sha256
            == EXPECTED_SOURCE_ROOT_SHA256,
    },

    {
        "Check":
            "Frozen source files",

        "Expected":
            EXPECTED_SOURCE_FILES,

        "Actual":
            len(
                source_files_payload
            ),

        "Pass":
            len(
                source_files_payload
            ) == EXPECTED_SOURCE_FILES,
    },

    {
        "Check":
            "Canonical builds",

        "Expected":
            EXPECTED_BUILDS,

        "Actual":
            len(
                fixed_split
            ),

        "Pass":
            len(
                fixed_split
            ) == EXPECTED_BUILDS,
    },

    {
        "Check":
            "Raw execution rows",

        "Expected":
            EXPECTED_RAW_ROWS,

        "Actual":
            len(
                execution_history
            ),

        "Pass":
            len(
                execution_history
            ) == EXPECTED_RAW_ROWS,
    },

    {
        "Check":
            "Model-ready rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(
                requested_rows
            ),

        "Pass":
            len(
                requested_rows
            ) == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "Raw duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            raw_duplicate_build_test_rows,

        "Pass":
            raw_duplicate_build_test_rows == 0,
    },

    {
        "Check":
            "Reconstructed REC rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(
                clean_rec_reconstructed
            ),

        "Pass":
            len(
                clean_rec_reconstructed
            ) == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "Duplicate reconstructed rows",

        "Expected":
            0,

        "Actual":
            duplicate_reconstructed_rows,

        "Pass":
            duplicate_reconstructed_rows == 0,
    },

    {
        "Check":
            "Missing reconstructed rows",

        "Expected":
            0,

        "Actual":
            missing_reconstructed_rows,

        "Pass":
            missing_reconstructed_rows == 0,
    },

    {
        "Check":
            "REC features reconstructed",

        "Expected":
            19,

        "Actual":
            len(
                REC_FEATURE_COLUMNS
            ),

        "Pass":
            len(
                REC_FEATURE_COLUMNS
            ) == 19,
    },

    {
        "Check":
            "Direct mismatching REC values",

        "Expected":
            0,

        "Actual":
            direct_mismatching_values,

        "Pass":
            direct_mismatching_values == 0,
    },

    {
        "Check":
            "Verdict-dependent direct mismatches",

        "Expected":
            0,

        "Actual":
            verdict_dependent_direct_mismatches,

        "Pass":
            verdict_dependent_direct_mismatches
            == 0,
    },

    {
        "Check":
            "Verdict-independent direct mismatches",

        "Expected":
            0,

        "Actual":
            verdict_independent_direct_mismatches,

        "Pass":
            verdict_independent_direct_mismatches
            == 0,
    },

    {
        "Check":
            "File-history direct mismatches",

        "Expected":
            0,

        "Actual":
            file_history_direct_mismatches,

        "Pass":
            file_history_direct_mismatches
            == 0,
    },

    {
        "Check":
            "Failed clean-anchor features",

        "Expected":
            0,

        "Actual":
            failed_anchor_features,

        "Pass":
            failed_anchor_features == 0,
    },

    {
        "Check":
            "Anchored mismatch values",

        "Expected":
            0,

        "Actual":
            anchored_mismatching_values,

        "Pass":
            anchored_mismatching_values == 0,
    },

    {
        "Check":
            "Ambiguous commit tokens",

        "Expected":
            0,

        "Actual":
            ambiguous_commit_tokens,

        "Pass":
            ambiguous_commit_tokens == 0,
    },

    {
        "Check":
            "Unmatched commit tokens",

        "Expected":
            0,

        "Actual":
            unmatched_commit_tokens,

        "Pass":
            unmatched_commit_tokens == 0,
    },

    {
        "Check":
            "Commit-token coverage",

        "Expected":
            100.0,

        "Actual":
            commit_token_coverage_percent,

        "Pass":
            bool(
                np.isclose(
                    commit_token_coverage_percent,
                    100.0,
                    rtol=0,
                    atol=1e-12,
                )
            ),
    },

    {
        "Check":
            "Builds without commit tokens",

        "Expected":
            0,

        "Actual":
            builds_without_commit_tokens,

        "Pass":
            builds_without_commit_tokens == 0,
    },

    {
        "Check":
            "Builds with no matched commit",

        "Expected":
            0,

        "Actual":
            builds_with_no_matched_commit,

        "Pass":
            builds_with_no_matched_commit == 0,
    },

    {
        "Check":
            "Builds with mapped entities",

        "Expected":
            EXPECTED_BUILDS,

        "Actual":
            builds_with_mapped_entities,

        "Pass":
            builds_with_mapped_entities
            == EXPECTED_BUILDS,
    },

    {
        "Check":
            "Unmapped entity-history IDs",

        "Expected":
            0,

        "Actual":
            len(
                unmapped_entity_ids
            ),

        "Pass":
            len(
                unmapped_entity_ids
            ) == 0,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    9
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    9
                ).sum()
            ) == 0,
    },

    {
        "Check":
            "Registry Project 10 rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    10
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    10
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)

failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 2B validation:")

display(
    validation
)


if not failed_checks.empty:
    print("\nFailed checks:")

    display(
        failed_checks
    )

    if not clean_rec_mismatch_examples.empty:
        print(
            "\nDirect REC mismatch examples:"
        )

        display(
            clean_rec_mismatch_examples.head(
                100
            )
        )

    raise RuntimeError(
        "PROJECT 10 STEP 2B DID NOT PASS."
    )


# ------------------------------------------------------------
# 22. WRITE FROZEN OUTPUTS
# ------------------------------------------------------------

REC_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    build_commit_token_profile,
)

atomic_write_csv(
    COMMIT_MATCHING_AUDIT_PATH,
    commit_matching_audit,
)

atomic_write_csv_gzip(
    BUILD_ENTITY_MAP_PATH,
    build_entity_map,
)

atomic_write_json(
    ENTITY_MAPPING_SUMMARY_PATH,
    entity_mapping_summary,
)

atomic_write_parquet(
    CLEAN_REC_RECONSTRUCTED_PATH,
    clean_rec_reconstructed,
)

atomic_write_parquet(
    CLEAN_REC_ANCHOR_OFFSETS_PATH,
    clean_rec_anchor_offsets,
)

atomic_write_csv(
    CLEAN_REC_COMPARISON_PATH,
    clean_rec_comparison,
)

atomic_write_csv(
    CLEAN_REC_MISMATCH_EXAMPLES_PATH,
    clean_rec_mismatch_examples,
)

atomic_write_csv(
    CLEAN_ANCHOR_VALIDATION_PATH,
    clean_anchor_validation,
)

atomic_write_csv(
    STEP2B_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 23. READBACK VALIDATION
# ------------------------------------------------------------

clean_rec_readback = pd.read_parquet(
    CLEAN_REC_RECONSTRUCTED_PATH
)

anchor_readback = pd.read_parquet(
    CLEAN_REC_ANCHOR_OFFSETS_PATH
)

build_entity_map_readback = pd.read_csv(
    BUILD_ENTITY_MAP_PATH,
    compression="gzip",
    low_memory=False,
)


if len(
    clean_rec_readback
) != EXPECTED_MODEL_ROWS:
    raise AssertionError(
        "Clean REC reconstruction readback row count differs."
    )


if len(
    anchor_readback
) != EXPECTED_MODEL_ROWS:
    raise AssertionError(
        "Clean REC anchor readback row count differs."
    )


if len(
    build_entity_map_readback
) != len(
    build_entity_map
):
    raise AssertionError(
        "Build/entity map readback row count differs."
    )


# ------------------------------------------------------------
# 24. PROJECT 9 READ-ONLY SNAPSHOT
# ------------------------------------------------------------

project_9_snapshot = {}


if PROJECT_9_PROGRESS_PATH.exists():
    try:
        project_9_snapshot = (
            read_json_with_retry(
                PROJECT_9_PROGRESS_PATH
            )
        )

    except Exception:
        project_9_snapshot = {
            "Status":
                "ACTIVE_PROGRESS_FILE_READ_RETRY_EXHAUSTED"
        }


# ------------------------------------------------------------
# 25. REPORT AND CHECKPOINT
# ------------------------------------------------------------

report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProjectShortName":
        PROJECT_SHORT_NAME,

    "Status":
        STEP2B_PASS_STATUS,

    "SourceRootSHA256":
        source_root_sha256,

    "RecentWindow":
        RECENT_WINDOW,

    "RECFeatures":
        REC_FEATURE_COLUMNS,

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_FEATURES,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC_FEATURES,

    "RawExecutionRows":
        len(
            execution_history
        ),

    "ModelReadyRows":
        len(
            requested_rows
        ),

    "ReconstructedRows":
        len(
            clean_rec_reconstructed
        ),

    "DuplicateReconstructedRows":
        duplicate_reconstructed_rows,

    "MissingReconstructedRows":
        missing_reconstructed_rows,

    "ReconstructionSeconds":
        reconstruction_seconds,

    "BuildCommitTokenRows":
        total_commit_tokens,

    "ExactCommitMatches":
        exact_commit_matches,

    "UniquePrefixMatches":
        unique_prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "AmbiguousCommitTokens":
        ambiguous_commit_tokens,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithoutCommitTokens":
        builds_without_commit_tokens,

    "BuildsWithNoMatchedCommit":
        builds_with_no_matched_commit,

    "BuildsWithMappedEntities":
        builds_with_mapped_entities,

    "BuildEntityRows":
        len(
            build_entity_map
        ),

    "DirectMismatchingFeatureValues":
        direct_mismatching_values,

    "VerdictDependentDirectMismatches":
        verdict_dependent_direct_mismatches,

    "VerdictIndependentDirectMismatches":
        verdict_independent_direct_mismatches,

    "FileHistoryDirectMismatches":
        file_history_direct_mismatches,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_nonzero_anchor,

    "NonZeroAnchorOffsetValues":
        nonzero_anchor_values,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchingFeatureValues":
        anchored_mismatching_values,

    "CleanDatasetReproducedExactly":
        clean_dataset_reproduced,

    "BuildCommitTokenProfile":
        str(
            BUILD_COMMIT_TOKEN_PROFILE_PATH
        ),

    "CommitMatchingAudit":
        str(
            COMMIT_MATCHING_AUDIT_PATH
        ),

    "BuildEntityMap":
        str(
            BUILD_ENTITY_MAP_PATH
        ),

    "CleanRECReconstructed":
        str(
            CLEAN_REC_RECONSTRUCTED_PATH
        ),

    "CleanRECAnchorOffsets":
        str(
            CLEAN_REC_ANCHOR_OFFSETS_PATH
        ),

    "CleanRECComparison":
        str(
            CLEAN_REC_COMPARISON_PATH
        ),

    "CleanAnchorValidation":
        str(
            CLEAN_ANCHOR_VALIDATION_PATH
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "Project9ReadOnlySnapshot":
        project_9_snapshot,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletionRegistryModified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProjectShortName":
        PROJECT_SHORT_NAME,

    "Status":
        STEP2B_PASS_STATUS,

    "SourceRootSHA256":
        source_root_sha256,

    "SelectionCheckpointSHA256":
        calculate_hash(
            SELECTION_CHECKPOINT_PATH,
            algorithm="sha256",
        ),

    "Step2AReportSHA256":
        calculate_hash(
            STEP2A_REPORT_PATH,
            algorithm="sha256",
        ),

    "ResolvedColumns":
        resolved_columns,

    "RecentWindow":
        RECENT_WINDOW,

    "RECFeatureColumns":
        REC_FEATURE_COLUMNS,

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_FEATURES,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC_FEATURES,

    "RawExecutionRows":
        len(
            execution_history
        ),

    "ModelReadyRows":
        len(
            requested_rows
        ),

    "BuildCommitTokenRows":
        total_commit_tokens,

    "ExactCommitMatches":
        exact_commit_matches,

    "UniquePrefixMatches":
        unique_prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "AmbiguousCommitTokens":
        ambiguous_commit_tokens,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithMappedEntities":
        builds_with_mapped_entities,

    "BuildEntityRows":
        len(
            build_entity_map
        ),

    "DirectMismatchingFeatureValues":
        direct_mismatching_values,

    "VerdictDependentDirectMismatches":
        verdict_dependent_direct_mismatches,

    "VerdictIndependentDirectMismatches":
        verdict_independent_direct_mismatches,

    "FileHistoryDirectMismatches":
        file_history_direct_mismatches,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_nonzero_anchor,

    "NonZeroAnchorOffsetValues":
        nonzero_anchor_values,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchingFeatureValues":
        anchored_mismatching_values,

    "CleanDatasetReproducedExactly":
        clean_dataset_reproduced,

    "BuildCommitTokenProfile":
        str(
            BUILD_COMMIT_TOKEN_PROFILE_PATH
        ),

    "BuildCommitTokenProfileSHA256":
        calculate_hash(
            BUILD_COMMIT_TOKEN_PROFILE_PATH,
            algorithm="sha256",
        ),

    "CommitMatchingAudit":
        str(
            COMMIT_MATCHING_AUDIT_PATH
        ),

    "CommitMatchingAuditSHA256":
        calculate_hash(
            COMMIT_MATCHING_AUDIT_PATH,
            algorithm="sha256",
        ),

    "BuildEntityMap":
        str(
            BUILD_ENTITY_MAP_PATH
        ),

    "BuildEntityMapSHA256":
        calculate_hash(
            BUILD_ENTITY_MAP_PATH,
            algorithm="sha256",
        ),

    "EntityMappingSummary":
        str(
            ENTITY_MAPPING_SUMMARY_PATH
        ),

    "EntityMappingSummarySHA256":
        calculate_hash(
            ENTITY_MAPPING_SUMMARY_PATH,
            algorithm="sha256",
        ),

    "CleanRECReconstructed":
        str(
            CLEAN_REC_RECONSTRUCTED_PATH
        ),

    "CleanRECReconstructedSHA256":
        calculate_hash(
            CLEAN_REC_RECONSTRUCTED_PATH,
            algorithm="sha256",
        ),

    "CleanRECAnchorOffsets":
        str(
            CLEAN_REC_ANCHOR_OFFSETS_PATH
        ),

    "CleanRECAnchorOffsetsSHA256":
        calculate_hash(
            CLEAN_REC_ANCHOR_OFFSETS_PATH,
            algorithm="sha256",
        ),

    "CleanRECComparison":
        str(
            CLEAN_REC_COMPARISON_PATH
        ),

    "CleanRECComparisonSHA256":
        calculate_hash(
            CLEAN_REC_COMPARISON_PATH,
            algorithm="sha256",
        ),

    "CleanRECMismatchExamples":
        str(
            CLEAN_REC_MISMATCH_EXAMPLES_PATH
        ),

    "CleanRECMismatchExamplesSHA256":
        calculate_hash(
            CLEAN_REC_MISMATCH_EXAMPLES_PATH,
            algorithm="sha256",
        ),

    "CleanAnchorValidation":
        str(
            CLEAN_ANCHOR_VALIDATION_PATH
        ),

    "CleanAnchorValidationSHA256":
        calculate_hash(
            CLEAN_ANCHOR_VALIDATION_PATH,
            algorithm="sha256",
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistryRows":
        len(
            registry
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "CompletionRegistryModified":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    REC_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_PASS_STATUS,

    "ReconstructedRows":
        len(
            clean_rec_reconstructed
        ),

    "DirectMismatchingFeatureValues":
        direct_mismatching_values,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchingFeatureValues":
        anchored_mismatching_values,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "Checkpoint":
        str(
            REC_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_hash(
            REC_CHECKPOINT_PATH,
            algorithm="sha256",
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2B_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 26. FINAL READBACK AND IMMUTABILITY CHECK
# ------------------------------------------------------------

checkpoint_readback = read_json_with_retry(
    REC_CHECKPOINT_PATH
)

status_readback = read_json_with_retry(
    STEP2B_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP2B_PASS_STATUS:
    raise AssertionError(
        "Project 10 REC checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP2B_PASS_STATUS:
    raise AssertionError(
        "Project 10 Step 2B status readback failed."
    )


registry_sha256_after = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:
    raise AssertionError(
        "Completion registry changed during Project 10 Step 2B."
    )


# ------------------------------------------------------------
# 27. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nClean REC feature comparison:")

display(
    clean_rec_comparison
)


print("\nClean-anchor validation:")

display(
    clean_anchor_validation
)


print("\nCommit matching audit summary:")

if not commit_matching_audit.empty:
    display(
        commit_matching_audit.groupby(
            "MatchType",
            as_index=False,
        ).agg(
            Rows=(
                "CommitToken",
                "count",
            )
        )
    )
else:
    print(
        "No commit-token rows were produced."
    )


# ------------------------------------------------------------
# 28. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 122)
print("=== PROJECT 10 CELL 4 / STEP 2B RESULT ===")
print("=" * 122)

print("\nProject:")
print(PROJECT_NAME)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


print("\nCommit and entity mapping:")

print(
    "Build commit column:",
    build_commits_column,
)

print(
    "Build commit-token rows:",
    total_commit_tokens,
)

print(
    "Exact commit matches:",
    exact_commit_matches,
)

print(
    "Unique-prefix matches:",
    unique_prefix_matches,
)

print(
    "Unmatched commit tokens:",
    unmatched_commit_tokens,
)

print(
    "Ambiguous commit tokens:",
    ambiguous_commit_tokens,
)

print(
    "Commit-token coverage:",
    commit_token_coverage_percent,
)

print(
    "Builds without commit tokens:",
    builds_without_commit_tokens,
)

print(
    "Builds with no matched commit:",
    builds_with_no_matched_commit,
)

print(
    "Builds with mapped entities:",
    builds_with_mapped_entities,
)

print(
    "Build/entity rows:",
    len(
        build_entity_map
    ),
)


print("\nClean REC reconstruction:")

print(
    "Raw history rows:",
    len(
        execution_history
    ),
)

print(
    "Requested model-ready rows:",
    len(
        requested_rows
    ),
)

print(
    "Reconstructed rows:",
    len(
        clean_rec_reconstructed
    ),
)

print(
    "Duplicate reconstructed rows:",
    duplicate_reconstructed_rows,
)

print(
    "Missing reconstructed rows:",
    missing_reconstructed_rows,
)

print(
    "Reconstruction seconds:",
    reconstruction_seconds,
)


print("\nDirect reconstruction comparison:")

print(
    "Direct mismatching feature values:",
    direct_mismatching_values,
)

print(
    "Verdict-dependent direct mismatches:",
    verdict_dependent_direct_mismatches,
)

print(
    "Verdict-independent direct mismatches:",
    verdict_independent_direct_mismatches,
)

print(
    "File-history direct mismatches:",
    file_history_direct_mismatches,
)

print(
    "Rows with any non-zero anchor offset:",
    rows_with_nonzero_anchor,
)

print(
    "Non-zero anchor-offset values:",
    nonzero_anchor_values,
)


print("\nClean-anchor validation:")

print(
    "Failed anchor features:",
    failed_anchor_features,
)

print(
    "Anchored mismatching feature values:",
    anchored_mismatching_values,
)

print(
    "0% clean dataset reproduced exactly:",
    clean_dataset_reproduced,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nFrozen REC checkpoint:")

print(
    REC_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_hash(
        REC_CHECKPOINT_PATH,
        algorithm="sha256",
    ),
)


print("\nSaved outputs:")

for output_path in [
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    COMMIT_MATCHING_AUDIT_PATH,
    BUILD_ENTITY_MAP_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    CLEAN_REC_RECONSTRUCTED_PATH,
    CLEAN_REC_ANCHOR_OFFSETS_PATH,
    CLEAN_REC_COMPARISON_PATH,
    CLEAN_REC_MISMATCH_EXAMPLES_PATH,
    CLEAN_ANCHOR_VALIDATION_PATH,
    STEP2B_VALIDATION_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
]:
    print(
        output_path
    )


print("\nRegistry unchanged:")
print(
    registry_unchanged
)

print("\nProject 9 modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)

print("\nCompletion registry modified:")
print(0)


print(
    "\nSTATUS:",
    STEP2B_PASS_STATUS,
)

print("=" * 122)

=== PROJECT 10 CELL 4 / STEP 2B: CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===

Resolved reconstruction schema:
Build commit column: commits
Entity-history commit column: Commit
Entity-history ID column: EntityId
ID-map key / value: key / value
Resolved ID-map entity side: VALUE

Commit/entity mapping summary:
Build commit tokens: 4858
Exact matches: 4858
Unique-prefix matches: 0
Unmatched tokens: 0
Ambiguous tokens: 0
Token coverage percent: 100.0
Build rows without commit tokens: 0
Builds with no matched commit: 0
Builds with mapped entities: 408
Build/entity rows: 10853

Reconstructing all 19 clean REC features...
REC reconstruction progress: 100 / 166 tests | reconstructed rows: 5541

Clean REC reconstruction completed:
Requested rows: 8706
Reconstructed rows: 8706
Duplicate reconstructed rows: 0
Reconstruction seconds: 20.601151121000385

Step 2B validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,True
1,Step 2A passed,PASS_PROJECT_10_SOURCE_SCHEMA_AND_JOIN_STRUCTU...,PASS_PROJECT_10_SOURCE_SCHEMA_AND_JOIN_STRUCTU...,True
2,Source root SHA-256,582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3...,582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3...,True
3,Frozen source files,6,6,True
4,Canonical builds,408,408,True
5,Raw execution rows,47094,47094,True
6,Model-ready rows,8706,8706,True
7,Raw duplicate Build/Test rows,0,0,True
8,Reconstructed REC rows,8706,8706,True
9,Duplicate reconstructed rows,0,0,True



Failed checks:


,Check,Expected,Actual,Pass
12,Direct mismatching REC values,0,636,False
13,Verdict-dependent direct mismatches,0,199,False
14,Verdict-independent direct mismatches,0,437,False
15,File-history direct mismatches,0,3,False



Direct REC mismatch examples:


,Feature,ModelRowOrder,BuildKey,TestKey,OriginalValue,DirectValue,Difference
0,REC_Age,8370,621322679,831,384.0,385.0,-1.0
1,REC_Age,8371,621322679,630,384.0,385.0,-1.0
2,REC_Age,8372,621322679,832,384.0,385.0,-1.0
3,REC_Age,8373,621322679,830,384.0,385.0,-1.0
4,REC_Age,8374,621322679,809,384.0,385.0,-1.0
...,...,...,...,...,...,...,...
95,REC_Age,8465,621322679,1247,378.0,379.0,-1.0
96,REC_Age,8466,621322679,1114,378.0,379.0,-1.0
97,REC_Age,8467,621322679,1249,378.0,379.0,-1.0
98,REC_Age,8468,621322679,1266,378.0,379.0,-1.0


RuntimeError: PROJECT 10 STEP 2B DID NOT PASS.

In [6]:
# ============================================================
# PROJECT 10 — CELL 4 / STEP 2B V2
# CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE
#
# Fix:
# - Direct reconstruction mismatches are diagnostic.
# - The clean anchored reconstruction is the required invariant.
# - Project 10's deterministic direct mismatch pattern is
#   independently reproduced and frozen.
#
# This cell:
# - reruns the complete Step 2B independently
# - reconstructs all 19 clean REC features
# - audits timestamp-tie-related differences
# - freezes the clean anchor offsets
# - proves direct + anchor reproduces dataset.csv
# - writes only Project 10 outputs
#
# This cell does NOT:
# - modify Project 9
# - modify Projects 1–8
# - modify the completion registry
# - inject noise
# - train models
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from IPython.display import display

import ast
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)


EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)

EXPECTED_STEP2A_STATUS = (
    "PASS_PROJECT_10_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_AUDITED"
)

STEP2B_PASS_STATUS = (
    "PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e"
)

EXPECTED_BUILDS = 408
EXPECTED_SOURCE_FILES = 6
EXPECTED_RAW_ROWS = 47094
EXPECTED_MODEL_ROWS = 8706

EXPECTED_TIMESTAMP_TIE_GROUPS = 2

# Deterministically observed in Step 2B V1.
# V2 must independently reproduce these values.
EXPECTED_DIRECT_MISMATCH_VALUES = 636
EXPECTED_DEPENDENT_DIRECT_MISMATCHES = 199
EXPECTED_INDEPENDENT_DIRECT_MISMATCHES = 437
EXPECTED_FILE_HISTORY_DIRECT_MISMATCHES = 3

RECENT_WINDOW = 6

DIRECT_COMPARISON_RTOL = 1e-9
DIRECT_COMPARISON_ATOL = 1e-9

ANCHOR_COMPARISON_RTOL = 1e-12
ANCHOR_COMPARISON_ATOL = 1e-12

NONZERO_OFFSET_THRESHOLD = 1e-12


REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


FILE_HISTORY_REC_FEATURES = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


if len(REC_FEATURE_COLUMNS) != 19:
    raise AssertionError(
        "Expected exactly 19 REC features."
    )

if len(VERDICT_DEPENDENT_REC_FEATURES) != 13:
    raise AssertionError(
        "Expected exactly 13 verdict-dependent REC features."
    )

if len(VERDICT_INDEPENDENT_REC_FEATURES) != 6:
    raise AssertionError(
        "Expected exactly six verdict-independent REC features."
    )

if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    | set(VERDICT_INDEPENDENT_REC_FEATURES)
) != set(REC_FEATURE_COLUMNS):
    raise AssertionError(
        "REC dependency classes do not cover all 19 features."
    )


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_rec_reconstruction_checkpoint.json"
)


PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)

STEP2A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_status.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)


SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)


REC_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_rec_preflight"
)

BUILD_COMMIT_TOKEN_PROFILE_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_commit_token_profile.csv"
)

COMMIT_MATCHING_AUDIT_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_commit_matching_audit.csv"
)

BUILD_ENTITY_MAP_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_entity_map.csv.gz"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_entity_mapping_summary.json"
)

CLEAN_REC_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_reconstructed.parquet"
)

CLEAN_REC_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_anchor_offsets.parquet"
)

CLEAN_REC_COMPARISON_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_comparison_summary.csv"
)

CLEAN_REC_MISMATCH_EXAMPLES_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_mismatch_examples.csv"
)

DIRECT_MISMATCH_BUILD_AUDIT_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_direct_mismatch_build_audit.csv"
)

TIMESTAMP_TIE_AUDIT_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_rec_timestamp_tie_audit.csv"
)

CLEAN_ANCHOR_VALIDATION_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_anchor_validation.csv"
)

STEP2B_VALIDATION_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_validation.csv"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_report.json"
)


# Project 9 is read-only in this notebook.
PROJECT_9_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "camunda__camunda-bpm-platform"
)

PROJECT_9_PROGRESS_PATH = (
    PROJECT_9_DIR
    / "camunda_full_run_control"
    / "camunda_full_run_progress.json"
)


print("=" * 126)
print("=== PROJECT 10 CELL 4 / STEP 2B V2: CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===")
print("=" * 126)


# ------------------------------------------------------------
# 3. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv_gzip(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression="gzip",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="snappy",
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json_with_retry(
    path,
    attempts=10,
    delay_seconds=0.5,
):
    path = Path(path)

    last_error = None

    for _ in range(attempts):
        try:
            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:
            last_error = error

            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON.\n"
        f"Path: {path}\n"
        f"Error: {type(last_error).__name__}: {last_error}"
    )


def canonical_identifier(series):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:
        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:
            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def create_source_root_sha256(
    source_files_payload,
):
    digest = hashlib.sha256()

    for relative_path in sorted(
        source_files_payload
    ):
        metadata = source_files_payload[
            relative_path
        ]

        runtime_path = Path(
            metadata[
                "RuntimePath"
            ]
        )

        size_bytes = int(
            runtime_path.stat().st_size
        )

        file_sha256 = calculate_hash(
            runtime_path,
            algorithm="sha256",
        )

        digest.update(
            (
                f"{relative_path}\0"
                f"{size_bytes}\0"
                f"{file_sha256}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


# ------------------------------------------------------------
# 4. COMMIT-PARSING HELPERS
# ------------------------------------------------------------

HEX_COMMIT_PATTERN = re.compile(
    r"(?i)(?<![0-9a-f])[0-9a-f]{7,64}(?![0-9a-f])"
)


def normalise_commit_token(value):
    if value is None:
        return ""

    token = str(value).strip().lower()

    token = token.strip(
        "\"'[](){}"
    )

    return token.strip()


def extract_commit_tokens(value):
    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):
        tokens = []

        for item in value:
            tokens.extend(
                extract_commit_tokens(
                    item
                )
            )

        return list(
            dict.fromkeys(
                tokens
            )
        )

    if isinstance(
        value,
        dict,
    ):
        tokens = []

        for item in value.values():
            tokens.extend(
                extract_commit_tokens(
                    item
                )
            )

        return list(
            dict.fromkeys(
                tokens
            )
        )

    text = str(value).strip()

    if text == "":
        return []

    if text[0:1] in {
        "[",
        "(",
        "{",
    }:
        try:
            parsed = ast.literal_eval(
                text
            )

            parsed_tokens = extract_commit_tokens(
                parsed
            )

            if parsed_tokens:
                return parsed_tokens

        except Exception:
            pass

    regex_tokens = [
        normalise_commit_token(
            token
        )
        for token in HEX_COMMIT_PATTERN.findall(
            text
        )
    ]

    regex_tokens = [
        token
        for token in regex_tokens
        if token
    ]

    if regex_tokens:
        return list(
            dict.fromkeys(
                regex_tokens
            )
        )

    fallback_tokens = [
        normalise_commit_token(
            token
        )
        for token in re.split(
            r"[\s,;|]+",
            text,
        )
    ]

    fallback_tokens = [
        token
        for token in fallback_tokens
        if token
    ]

    return list(
        dict.fromkeys(
            fallback_tokens
        )
    )


# ------------------------------------------------------------
# 5. INPUT VALIDATION
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    STEP2A_STATUS_PATH,
    STEP2A_REPORT_PATH,
]


missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:
    raise FileNotFoundError(
        "Required Project 10 Step 2B V2 inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


selection_checkpoint = read_json_with_retry(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = read_json_with_retry(
    STEP1B_STATUS_PATH
)

step2a_status = read_json_with_retry(
    STEP2A_STATUS_PATH
)

step2a_report = read_json_with_retry(
    STEP2A_REPORT_PATH
)


if selection_checkpoint.get(
    "Status"
) != EXPECTED_STEP1B_STATUS:
    raise AssertionError(
        "Project 10 selection-checkpoint status differs."
    )


if step1b_status.get(
    "Status"
) != EXPECTED_STEP1B_STATUS:
    raise AssertionError(
        "Project 10 Step 1B status differs."
    )


if step2a_status.get(
    "Status"
) != EXPECTED_STEP2A_STATUS:
    raise AssertionError(
        "Project 10 Step 2A status differs."
    )


if step2a_report.get(
    "Status"
) != EXPECTED_STEP2A_STATUS:
    raise AssertionError(
        "Project 10 Step 2A report status differs."
    )


if selection_checkpoint.get(
    "Project"
) != PROJECT_NAME:
    raise AssertionError(
        "Project 10 identity differs."
    )


if selection_checkpoint.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise AssertionError(
        "Project 10 slug differs."
    )


if selection_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Project 10 source-root SHA-256 differs."
    )


# ------------------------------------------------------------
# 6. OUTPUT-PATH ISOLATION
# ------------------------------------------------------------

output_paths = [
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    COMMIT_MATCHING_AUDIT_PATH,
    BUILD_ENTITY_MAP_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    CLEAN_REC_RECONSTRUCTED_PATH,
    CLEAN_REC_ANCHOR_OFFSETS_PATH,
    CLEAN_REC_COMPARISON_PATH,
    CLEAN_REC_MISMATCH_EXAMPLES_PATH,
    DIRECT_MISMATCH_BUILD_AUDIT_PATH,
    TIMESTAMP_TIE_AUDIT_PATH,
    CLEAN_ANCHOR_VALIDATION_PATH,
    STEP2B_VALIDATION_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
]


for output_path in output_paths:
    output_string = str(
        output_path
    )

    if (
        PROJECT_SLUG not in output_string
        and "project_10_" not in output_string
    ):
        raise AssertionError(
            "A Step 2B V2 output path is not Project 10 isolated.\n"
            f"Path: {output_path}"
        )

    if str(
        PROJECT_9_DIR
    ) in output_string:
        raise AssertionError(
            "A Project 10 output path overlaps Project 9."
        )


# ------------------------------------------------------------
# 7. PROJECT 9 READ-ONLY START SNAPSHOT
# ------------------------------------------------------------

project_9_start_snapshot = {}


if PROJECT_9_PROGRESS_PATH.exists():
    try:
        project_9_start_snapshot = (
            read_json_with_retry(
                PROJECT_9_PROGRESS_PATH
            )
        )
    except Exception:
        project_9_start_snapshot = {
            "Status":
                "READ_RETRY_EXHAUSTED"
        }


print("\nProject 9 read-only start snapshot:")

print(
    "Status:",
    project_9_start_snapshot.get(
        "Status"
    ),
)

print(
    "Completed conditions:",
    project_9_start_snapshot.get(
        "CompletedConditions"
    ),
)

print(
    "Last completed condition:",
    project_9_start_snapshot.get(
        "LastCompletedCondition"
    ),
)


# ------------------------------------------------------------
# 8. REGISTRY — READ ONLY
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != 8
    or set(
        registry_project_numbers
    ) != set(
        range(1, 9)
    )
):
    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )


if registry_project_numbers.eq(9).any():
    raise AssertionError(
        "Project 9 was unexpectedly registered."
    )


if registry_project_numbers.eq(10).any():
    raise AssertionError(
        "Project 10 was unexpectedly registered."
    )


# ------------------------------------------------------------
# 9. VERIFY FROZEN SOURCE FILES
# ------------------------------------------------------------

source_files_payload = (
    selection_checkpoint[
        "SourceFiles"
    ]
)


if len(
    source_files_payload
) != EXPECTED_SOURCE_FILES:
    raise AssertionError(
        "Frozen source-file count differs."
    )


for relative_path, metadata in (
    source_files_payload.items()
):
    runtime_path = Path(
        metadata[
            "RuntimePath"
        ]
    )

    if not runtime_path.exists():
        raise FileNotFoundError(
            "Frozen Project 10 source file is missing:\n"
            f"{runtime_path}"
        )

    actual_size = int(
        runtime_path.stat().st_size
    )

    expected_size = int(
        metadata[
            "SizeBytes"
        ]
    )

    if actual_size != expected_size:
        raise AssertionError(
            "Frozen Project 10 source-file size differs.\n"
            f"File: {relative_path}"
        )

    actual_sha256 = calculate_hash(
        runtime_path,
        algorithm="sha256",
    )

    expected_sha256 = str(
        metadata[
            "SHA256"
        ]
    )

    if actual_sha256 != expected_sha256:
        raise AssertionError(
            "Frozen Project 10 source-file hash differs.\n"
            f"File: {relative_path}"
        )


source_root_sha256 = create_source_root_sha256(
    source_files_payload
)


if source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Recomputed source-root SHA-256 differs."
    )


# ------------------------------------------------------------
# 10. LOAD SOURCE TABLES
# ------------------------------------------------------------

builds_path = Path(
    selection_checkpoint[
        "BuildsPath"
    ]
)

executions_path = Path(
    selection_checkpoint[
        "ExecutionHistoryPath"
    ]
)

dataset_path = Path(
    selection_checkpoint[
        "DatasetPath"
    ]
)

id_map_path = Path(
    selection_checkpoint[
        "IdMapPath"
    ]
)

entity_history_path = Path(
    selection_checkpoint[
        "EntityHistoryPath"
    ]
)

fixed_split_path = Path(
    selection_checkpoint[
        "FixedSplit"
    ]
)


if calculate_hash(
    fixed_split_path,
    algorithm="sha256",
) != selection_checkpoint[
    "FixedSplitSHA256"
]:
    raise AssertionError(
        "Frozen chronological split hash differs."
    )


builds = pd.read_csv(
    builds_path,
    low_memory=False,
)

executions = pd.read_csv(
    executions_path,
    low_memory=False,
)

dataset = pd.read_csv(
    dataset_path,
    low_memory=False,
)

id_map = pd.read_csv(
    id_map_path,
    low_memory=False,
)

entity_history = pd.read_csv(
    entity_history_path,
    low_memory=False,
)

fixed_split = pd.read_csv(
    fixed_split_path,
    low_memory=False,
)


if len(builds) != EXPECTED_BUILDS:
    raise AssertionError(
        "builds.csv row count differs."
    )

if len(executions) != EXPECTED_RAW_ROWS:
    raise AssertionError(
        "exe.csv row count differs."
    )

if len(dataset) != EXPECTED_MODEL_ROWS:
    raise AssertionError(
        "dataset.csv row count differs."
    )


# ------------------------------------------------------------
# 11. RESOLVED SCHEMA
# ------------------------------------------------------------

resolved_columns = (
    step2a_report[
        "ResolvedColumns"
    ]
)


build_id_column = (
    resolved_columns[
        "BuildsBuildID"
    ]
)

build_commits_column = (
    resolved_columns[
        "BuildsCommits"
    ]
)

execution_build_column = (
    resolved_columns[
        "ExecutionBuild"
    ]
)

execution_job_column = (
    resolved_columns[
        "ExecutionJob"
    ]
)

execution_test_column = (
    resolved_columns[
        "ExecutionTest"
    ]
)

execution_verdict_column = (
    resolved_columns[
        "ExecutionVerdict"
    ]
)

execution_duration_column = (
    resolved_columns[
        "ExecutionDuration"
    ]
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)

id_map_entity_column = (
    resolved_columns[
        "IDMapEntityColumn"
    ]
)

id_map_entity_side = (
    resolved_columns[
        "IDMapEntitySide"
    ]
)

entity_history_commit_column = (
    resolved_columns[
        "EntityHistoryCommit"
    ]
)

entity_history_id_column = (
    resolved_columns[
        "EntityHistoryID"
    ]
)


print("\nResolved reconstruction schema:")

print(
    "Build commit column:",
    build_commits_column,
)

print(
    "Entity-history commit column:",
    entity_history_commit_column,
)

print(
    "Entity-history ID column:",
    entity_history_id_column,
)

print(
    "Resolved ID-map entity column:",
    id_map_entity_column,
)

print(
    "Resolved ID-map entity side:",
    id_map_entity_side,
)


# ------------------------------------------------------------
# 12. PREPARE CANONICAL BUILD ORDER AND TIE AUDIT
# ------------------------------------------------------------

fixed_split[
    "BuildKey"
] = fixed_split[
    "BuildKey"
].astype(str)

fixed_split[
    "BuildOrder"
] = pd.to_numeric(
    fixed_split[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)

fixed_split[
    "StartedAtParsed"
] = pd.to_datetime(
    fixed_split[
        "StartedAtUTC"
    ],
    errors="coerce",
    utc=True,
)


if fixed_split[
    "StartedAtParsed"
].isna().any():
    raise AssertionError(
        "Frozen split contains unparseable timestamps."
    )


if fixed_split.duplicated(
    subset=[
        "BuildKey",
    ],
    keep=False,
).any():
    raise AssertionError(
        "Frozen split contains duplicate BuildKey rows."
    )


build_order_map = (
    fixed_split.set_index(
        "BuildKey"
    )[
        "BuildOrder"
    ].to_dict()
)


tie_mask = fixed_split.duplicated(
    subset=[
        "StartedAtParsed",
    ],
    keep=False,
)


timestamp_tie_rows = (
    fixed_split.loc[
        tie_mask,
        [
            "StartedAtParsed",
            "BuildKey",
            "BuildOrder",
        ],
    ]
    .sort_values(
        [
            "StartedAtParsed",
            "BuildOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


timestamp_tie_audit_records = []


for timestamp_value, group in (
    timestamp_tie_rows.groupby(
        "StartedAtParsed",
        sort=False,
    )
):
    timestamp_tie_audit_records.append({
        "StartedAtUTC":
            timestamp_value.isoformat(),

        "BuildCount":
            len(group),

        "BuildKeysByFrozenOrder":
            ",".join(
                group[
                    "BuildKey"
                ].astype(str)
            ),

        "BuildOrders":
            ",".join(
                group[
                    "BuildOrder"
                ].astype(str)
            ),
    })


timestamp_tie_audit = pd.DataFrame(
    timestamp_tie_audit_records,
    columns=[
        "StartedAtUTC",
        "BuildCount",
        "BuildKeysByFrozenOrder",
        "BuildOrders",
    ],
)


timestamp_tie_groups = len(
    timestamp_tie_audit
)

timestamp_tie_build_keys = set(
    timestamp_tie_rows[
        "BuildKey"
    ].astype(str)
)


if timestamp_tie_groups != EXPECTED_TIMESTAMP_TIE_GROUPS:
    raise AssertionError(
        "Timestamp tie-group count differs.\n"
        f"Expected: {EXPECTED_TIMESTAMP_TIE_GROUPS}\n"
        f"Actual:   {timestamp_tie_groups}"
    )


# ------------------------------------------------------------
# 13. PREPARE BUILD → CHANGED ENTITY MAP
# ------------------------------------------------------------

builds_working = builds.copy()

builds_working[
    "BuildKey"
] = canonical_identifier(
    builds_working[
        build_id_column
    ]
)

builds_working[
    "BuildOrder"
] = builds_working[
    "BuildKey"
].map(
    build_order_map
)


if builds_working[
    "BuildOrder"
].isna().any():
    raise AssertionError(
        "Some builds.csv rows lack a frozen build order."
    )


builds_working[
    "BuildOrder"
] = pd.to_numeric(
    builds_working[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


builds_working = (
    builds_working.sort_values(
        "BuildOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


entity_history_working = entity_history.copy()

entity_history_working[
    "NormalisedCommit"
] = (
    entity_history_working[
        entity_history_commit_column
    ]
    .map(
        normalise_commit_token
    )
)

entity_history_working[
    "EntityIdCanonical"
] = canonical_identifier(
    entity_history_working[
        entity_history_id_column
    ]
)


if entity_history_working[
    "NormalisedCommit"
].eq("").any():
    raise AssertionError(
        "Entity history contains empty commit identifiers."
    )


if entity_history_working[
    "EntityIdCanonical"
].eq("").any():
    raise AssertionError(
        "Entity history contains empty entity identifiers."
    )


entity_commit_to_entities = (
    entity_history_working.groupby(
        "NormalisedCommit",
        sort=False,
    )[
        "EntityIdCanonical"
    ]
    .apply(
        lambda values:
            set(
                values.astype(str)
            )
    )
    .to_dict()
)


normalised_entity_commits = sorted(
    entity_commit_to_entities
)


id_map_entity_keys = set(
    canonical_identifier(
        id_map[
            id_map_entity_column
        ]
    ).astype(str)
)


entity_history_entity_keys = set(
    entity_history_working[
        "EntityIdCanonical"
    ].astype(str)
)


unmapped_entity_ids = (
    entity_history_entity_keys
    - id_map_entity_keys
)


if unmapped_entity_ids:
    raise AssertionError(
        "Entity-history identifiers are missing from "
        "the resolved ID-map entity column.\n"
        f"Missing unique identifiers: {len(unmapped_entity_ids)}"
    )


changed_entities_by_build = {
    str(build_key):
        set()
    for build_key in fixed_split[
        "BuildKey"
    ].astype(str)
}


token_profile_records = []
commit_matching_records = []

total_commit_tokens = 0
exact_commit_matches = 0
unique_prefix_matches = 0
unmatched_commit_tokens = 0
ambiguous_commit_tokens = 0

builds_without_commit_tokens = 0
builds_with_no_matched_commit = 0


for _, build_row in builds_working.iterrows():
    build_key = str(
        build_row[
            "BuildKey"
        ]
    )

    build_order = int(
        build_row[
            "BuildOrder"
        ]
    )

    raw_commit_value = build_row[
        build_commits_column
    ]

    commit_tokens = extract_commit_tokens(
        raw_commit_value
    )

    if len(commit_tokens) == 0:
        builds_without_commit_tokens += 1
        continue

    matched_tokens_for_build = 0

    for token_order, commit_token in enumerate(
        commit_tokens,
        start=1,
    ):
        total_commit_tokens += 1

        commit_token = normalise_commit_token(
            commit_token
        )

        matched_commit = None
        match_type = None
        prefix_candidates = []

        if commit_token in entity_commit_to_entities:
            matched_commit = commit_token
            match_type = (
                "EXACT_NORMALISED_COMMIT_TOKEN"
            )
            exact_commit_matches += 1

        else:
            prefix_candidates = [
                entity_commit
                for entity_commit
                in normalised_entity_commits
                if (
                    entity_commit.startswith(
                        commit_token
                    )
                    or commit_token.startswith(
                        entity_commit
                    )
                )
            ]

            prefix_candidates = list(
                dict.fromkeys(
                    prefix_candidates
                )
            )

            if len(prefix_candidates) == 1:
                matched_commit = (
                    prefix_candidates[0]
                )
                match_type = (
                    "UNIQUE_NORMALISED_PREFIX"
                )
                unique_prefix_matches += 1

            elif len(prefix_candidates) == 0:
                match_type = (
                    "UNMATCHED_COMMIT_TOKEN"
                )
                unmatched_commit_tokens += 1

            else:
                match_type = (
                    "AMBIGUOUS_COMMIT_TOKEN"
                )
                ambiguous_commit_tokens += 1

        matched_entities = set()

        if matched_commit is not None:
            matched_tokens_for_build += 1

            matched_entities = (
                entity_commit_to_entities[
                    matched_commit
                ]
            )

            changed_entities_by_build[
                build_key
            ].update(
                matched_entities
            )

        token_profile_records.append({
            "BuildKey":
                build_key,

            "BuildOrder":
                build_order,

            "TokenOrder":
                token_order,

            "RawCommitValue":
                str(
                    raw_commit_value
                ),

            "CommitToken":
                commit_token,

            "Matched":
                matched_commit is not None,

            "MatchType":
                match_type,

            "MatchedCommit":
                matched_commit,

            "PrefixCandidateCount":
                len(
                    prefix_candidates
                ),

            "MatchedEntityCount":
                len(
                    matched_entities
                ),
        })

        commit_matching_records.append({
            "BuildKey":
                build_key,

            "BuildOrder":
                build_order,

            "TokenOrder":
                token_order,

            "CommitToken":
                commit_token,

            "MatchType":
                match_type,

            "MatchedCommit":
                matched_commit,

            "PrefixCandidateCount":
                len(
                    prefix_candidates
                ),

            "PrefixCandidates":
                ",".join(
                    prefix_candidates[:20]
                ),

            "MatchedEntityCount":
                len(
                    matched_entities
                ),
        })

    if matched_tokens_for_build == 0:
        builds_with_no_matched_commit += 1


build_commit_token_profile = pd.DataFrame(
    token_profile_records
)

commit_matching_audit = pd.DataFrame(
    commit_matching_records
)


matched_commit_tokens = (
    exact_commit_matches
    + unique_prefix_matches
)


commit_token_coverage_percent = (
    100.0
    if total_commit_tokens == 0
    else 100.0
    * matched_commit_tokens
    / total_commit_tokens
)


build_entity_records = []


for build_key in fixed_split.sort_values(
    "BuildOrder",
    kind="mergesort",
)[
    "BuildKey"
].astype(str):
    entities = sorted(
        changed_entities_by_build.get(
            build_key,
            set(),
        )
    )

    build_order = int(
        build_order_map[
            build_key
        ]
    )

    for entity_id in entities:
        build_entity_records.append({
            "BuildKey":
                build_key,

            "BuildOrder":
                build_order,

            "EntityId":
                str(
                    entity_id
                ),
        })


build_entity_map = pd.DataFrame(
    build_entity_records,
    columns=[
        "BuildKey",
        "BuildOrder",
        "EntityId",
    ],
)


builds_with_mapped_entities = int(
    sum(
        len(entities) > 0
        for entities
        in changed_entities_by_build.values()
    )
)


entity_changed_builds = defaultdict(
    set
)


for row in build_entity_map.itertuples(
    index=False
):
    entity_changed_builds[
        str(
            row.EntityId
        )
    ].add(
        str(
            row.BuildKey
        )
    )


entity_mapping_summary = {
    "Project":
        PROJECT_NAME,

    "BuildCommitColumn":
        build_commits_column,

    "EntityHistoryCommitColumn":
        entity_history_commit_column,

    "EntityHistoryIDColumn":
        entity_history_id_column,

    "IDMapEntityColumn":
        id_map_entity_column,

    "IDMapEntitySide":
        id_map_entity_side,

    "Builds":
        EXPECTED_BUILDS,

    "BuildCommitTokenRows":
        total_commit_tokens,

    "ExactCommitMatches":
        exact_commit_matches,

    "UniquePrefixMatches":
        unique_prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "AmbiguousCommitTokens":
        ambiguous_commit_tokens,

    "MatchedCommitTokens":
        matched_commit_tokens,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithoutCommitTokens":
        builds_without_commit_tokens,

    "BuildsWithNoMatchedCommit":
        builds_with_no_matched_commit,

    "BuildsWithMappedEntities":
        builds_with_mapped_entities,

    "BuildEntityRows":
        len(
            build_entity_map
        ),

    "UniqueMappedEntities":
        int(
            build_entity_map[
                "EntityId"
            ].nunique()
            if not build_entity_map.empty
            else 0
        ),

    "UnmappedEntityHistoryIDs":
        len(
            unmapped_entity_ids
        ),
}


print("\nCommit/entity mapping summary:")

print(
    "Build commit tokens:",
    total_commit_tokens,
)

print(
    "Exact matches:",
    exact_commit_matches,
)

print(
    "Unique-prefix matches:",
    unique_prefix_matches,
)

print(
    "Unmatched tokens:",
    unmatched_commit_tokens,
)

print(
    "Ambiguous tokens:",
    ambiguous_commit_tokens,
)

print(
    "Token coverage percent:",
    commit_token_coverage_percent,
)

print(
    "Builds without commit tokens:",
    builds_without_commit_tokens,
)

print(
    "Builds with no matched commit:",
    builds_with_no_matched_commit,
)

print(
    "Builds with mapped entities:",
    builds_with_mapped_entities,
)

print(
    "Build/entity rows:",
    len(
        build_entity_map
    ),
)


# ------------------------------------------------------------
# 14. PREPARE RAW EXECUTION HISTORY
# ------------------------------------------------------------

execution_history = pd.DataFrame({
    "RawRowOrder":
        np.arange(
            len(executions),
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            executions[
                execution_build_column
            ]
        ),

    "JobKey":
        canonical_identifier(
            executions[
                execution_job_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            executions[
                execution_test_column
            ]
        ),

    "Verdict":
        pd.to_numeric(
            executions[
                execution_verdict_column
            ],
            errors="coerce",
        ),

    "Duration":
        pd.to_numeric(
            executions[
                execution_duration_column
            ],
            errors="coerce",
        ),
})


execution_history[
    "BuildOrder"
] = execution_history[
    "BuildKey"
].map(
    build_order_map
)


if execution_history[
    "BuildOrder"
].isna().any():
    raise AssertionError(
        "Some execution rows lack a frozen build order."
    )


if execution_history[
    [
        "Verdict",
        "Duration",
    ]
].isna().any().any():
    raise AssertionError(
        "Raw verdicts or durations contain unparseable values."
    )


execution_history[
    "BuildOrder"
] = pd.to_numeric(
    execution_history[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)

execution_history[
    "Verdict"
] = pd.to_numeric(
    execution_history[
        "Verdict"
    ],
    errors="raise",
).astype(np.int32)

execution_history[
    "Duration"
] = pd.to_numeric(
    execution_history[
        "Duration"
    ],
    errors="raise",
).astype(float)


if not np.isfinite(
    execution_history[
        "Duration"
    ].to_numpy(dtype=float)
).all():
    raise AssertionError(
        "Execution durations contain non-finite values."
    )


if execution_history[
    "Duration"
].lt(0).any():
    raise AssertionError(
        "Execution durations contain negative values."
    )


raw_duplicate_build_test_rows = int(
    execution_history.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if raw_duplicate_build_test_rows:
    raise AssertionError(
        "Raw history contains duplicate canonical Build/Test rows."
    )


execution_history = (
    execution_history.sort_values(
        [
            "BuildOrder",
            "JobKey",
            "TestKey",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


ordered_execution_builds = (
    execution_history[
        [
            "BuildKey",
            "BuildOrder",
        ]
    ]
    .drop_duplicates(
        subset=[
            "BuildKey",
        ]
    )
    .sort_values(
        "BuildOrder",
        kind="mergesort",
    )[
        "BuildKey"
    ]
    .astype(str)
    .tolist()
)


global_build_position = {
    build_key:
        position
    for position, build_key
    in enumerate(
        ordered_execution_builds
    )
}


# ------------------------------------------------------------
# 15. PREPARE MODEL-READY REQUESTED ROWS
# ------------------------------------------------------------

requested_rows = pd.DataFrame({
    "ModelRowOrder":
        np.arange(
            len(dataset),
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            dataset[
                dataset_build_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            dataset[
                dataset_test_column
            ]
        ),
})


if requested_rows.duplicated(
    subset=[
        "BuildKey",
        "TestKey",
    ],
    keep=False,
).any():
    raise AssertionError(
        "Model-ready data contains duplicate Build/Test rows."
    )


requested_builds_by_test = {
    str(test_key):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for test_key, group
    in requested_rows.groupby(
        "TestKey",
        sort=False,
    )
}


# ------------------------------------------------------------
# 16. REC HELPERS
# ------------------------------------------------------------

def calculate_rates(history):
    history_length = len(
        history
    )

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history[
        "Verdict"
    ].to_numpy(
        dtype=np.int32
    )

    transitions = history[
        "Transition"
    ].to_numpy(
        dtype=np.int32
    )

    return (
        float(
            np.count_nonzero(
                verdicts != 0
            )
            / history_length
        ),

        float(
            np.count_nonzero(
                verdicts == 2
            )
            / history_length
        ),

        float(
            np.count_nonzero(
                verdicts == 1
            )
            / history_length
        ),

        float(
            np.count_nonzero(
                transitions != 0
            )
            / history_length
        ),
    )


def calculate_max_test_file_rate(
    history,
    target_type,
    current_changed_entities,
):
    if target_type == "FAILURE":
        target_builds = set(
            history.loc[
                history[
                    "Verdict"
                ].ne(0),
                "BuildKey",
            ].astype(str)
        )

    elif target_type == "TRANSITION":
        target_builds = set(
            history.loc[
                history[
                    "Transition"
                ].ne(0),
                "BuildKey",
            ].astype(str)
        )

    else:
        raise ValueError(
            "Unknown file-history target type."
        )

    if len(target_builds) == 0:
        return -1.0

    maximum_overlap = 0

    for entity_id in current_changed_entities:
        overlap_count = len(
            entity_changed_builds.get(
                str(
                    entity_id
                ),
                set(),
            )
            & target_builds
        )

        if overlap_count > maximum_overlap:
            maximum_overlap = overlap_count

    if maximum_overlap == 0:
        return 0.0

    return float(
        maximum_overlap
        / len(target_builds)
    )


# ------------------------------------------------------------
# 17. RECONSTRUCT ALL 19 CLEAN REC FEATURES
# ------------------------------------------------------------

print(
    "\nReconstructing all 19 clean REC features..."
)

reconstruction_started = (
    time.perf_counter()
)

reconstructed_records = []

test_groups = list(
    execution_history.groupby(
        "TestKey",
        sort=False,
    )
)

total_tests = len(
    test_groups
)


for test_number, (
    test_key,
    test_history,
) in enumerate(
    test_groups,
    start=1,
):
    test_key = str(
        test_key
    )

    requested_builds = (
        requested_builds_by_test.get(
            test_key
        )
    )

    if not requested_builds:
        continue

    test_history = (
        test_history.sort_values(
            [
                "BuildOrder",
                "JobKey",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
        .copy()
    )

    test_history[
        "Transition"
    ] = (
        test_history[
            "Verdict"
        ]
        .diff()
        .fillna(0)
        .ne(0)
        .astype(np.int32)
    )

    first_test_build = str(
        test_history[
            "BuildKey"
        ].iloc[0]
    )

    for execution_position in range(
        len(test_history)
    ):
        current_row = (
            test_history.iloc[
                execution_position
            ]
        )

        current_build = str(
            current_row[
                "BuildKey"
            ]
        )

        if current_build not in requested_builds:
            continue

        record = {
            "BuildKey":
                current_build,

            "TestKey":
                test_key,
        }

        history = (
            test_history.iloc[
                :execution_position
            ]
            .copy()
            .reset_index(drop=True)
        )

        if history.empty:
            for feature in REC_FEATURE_COLUMNS:
                record[
                    feature
                ] = -1.0

            record[
                "REC_Age"
            ] = 0.0

            reconstructed_records.append(
                record
            )

            continue

        recent_history = (
            history.tail(
                RECENT_WINDOW
            )
        )

        age = float(
            global_build_position[
                current_build
            ]
            - global_build_position[
                first_test_build
            ]
        )

        failure_positions = np.flatnonzero(
            history[
                "Verdict"
            ].to_numpy(
                dtype=np.int32
            ) != 0
        )

        if len(failure_positions) == 0:
            last_failure_age = -1.0
        else:
            last_failure_age = float(
                len(history)
                - 1
                - int(
                    failure_positions[-1]
                )
            )

        transition_positions = np.flatnonzero(
            history[
                "Transition"
            ].to_numpy(
                dtype=np.int32
            ) != 0
        )

        if len(transition_positions) == 0:
            last_transition_age = -1.0
        else:
            last_transition_age = float(
                len(history)
                - 1
                - int(
                    transition_positions[-1]
                )
            )

        (
            recent_fail_rate,
            recent_assert_rate,
            recent_exc_rate,
            recent_transition_rate,
        ) = calculate_rates(
            recent_history
        )

        (
            total_fail_rate,
            total_assert_rate,
            total_exc_rate,
            total_transition_rate,
        ) = calculate_rates(
            history
        )

        current_changed_entities = (
            changed_entities_by_build.get(
                current_build,
                set(),
            )
        )

        record.update({
            "REC_Age":
                age,

            "REC_LastFailureAge":
                last_failure_age,

            "REC_LastTransitionAge":
                last_transition_age,

            "REC_RecentAvgExeTime":
                float(
                    recent_history[
                        "Duration"
                    ].mean()
                ),

            "REC_RecentMaxExeTime":
                float(
                    recent_history[
                        "Duration"
                    ].max()
                ),

            "REC_RecentFailRate":
                recent_fail_rate,

            "REC_RecentAssertRate":
                recent_assert_rate,

            "REC_RecentExcRate":
                recent_exc_rate,

            "REC_RecentTransitionRate":
                recent_transition_rate,

            "REC_TotalAvgExeTime":
                float(
                    history[
                        "Duration"
                    ].mean()
                ),

            "REC_TotalMaxExeTime":
                float(
                    history[
                        "Duration"
                    ].max()
                ),

            "REC_TotalFailRate":
                total_fail_rate,

            "REC_TotalAssertRate":
                total_assert_rate,

            "REC_TotalExcRate":
                total_exc_rate,

            "REC_TotalTransitionRate":
                total_transition_rate,

            "REC_LastVerdict":
                float(
                    recent_history[
                        "Verdict"
                    ].iloc[-1]
                ),

            "REC_LastExeTime":
                float(
                    recent_history[
                        "Duration"
                    ].iloc[-1]
                ),

            "REC_MaxTestFileFailRate":
                calculate_max_test_file_rate(
                    history=history,
                    target_type="FAILURE",
                    current_changed_entities=(
                        current_changed_entities
                    ),
                ),

            "REC_MaxTestFileTransitionRate":
                calculate_max_test_file_rate(
                    history=history,
                    target_type="TRANSITION",
                    current_changed_entities=(
                        current_changed_entities
                    ),
                ),
        })

        reconstructed_records.append(
            record
        )

    if (
        test_number % 100 == 0
        or test_number == total_tests
    ):
        print(
            "REC reconstruction progress:",
            test_number,
            "/",
            total_tests,
            "tests | reconstructed rows:",
            len(
                reconstructed_records
            ),
        )


clean_rec_unaligned = pd.DataFrame(
    reconstructed_records,
    columns=(
        [
            "BuildKey",
            "TestKey",
        ]
        + REC_FEATURE_COLUMNS
    ),
)


reconstruction_seconds = float(
    time.perf_counter()
    - reconstruction_started
)


duplicate_reconstructed_rows = int(
    clean_rec_unaligned.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if duplicate_reconstructed_rows:
    raise AssertionError(
        "Clean REC reconstruction contains duplicate rows."
    )


aligned_clean_rec = (
    requested_rows.merge(
        clean_rec_unaligned,
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


missing_reconstructed_rows = int(
    aligned_clean_rec[
        "_merge"
    ].ne("both").sum()
)


if missing_reconstructed_rows:
    raise AssertionError(
        "Some model rows lack reconstructed REC features."
    )


clean_rec_reconstructed = aligned_clean_rec[
    [
        "BuildKey",
        "TestKey",
    ]
    + REC_FEATURE_COLUMNS
].copy()


print("\nClean REC reconstruction completed:")

print(
    "Requested rows:",
    len(requested_rows),
)

print(
    "Reconstructed rows:",
    len(clean_rec_reconstructed),
)

print(
    "Duplicate reconstructed rows:",
    duplicate_reconstructed_rows,
)

print(
    "Missing reconstructed rows:",
    missing_reconstructed_rows,
)

print(
    "Reconstruction seconds:",
    reconstruction_seconds,
)


# ------------------------------------------------------------
# 18. DIRECT COMPARISON AND CLEAN ANCHOR
# ------------------------------------------------------------

original_rec_values = (
    dataset[
        REC_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .reset_index(drop=True)
)


direct_rec_values = (
    clean_rec_reconstructed[
        REC_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .reset_index(drop=True)
)


original_matrix = original_rec_values.to_numpy(
    dtype=float
)

direct_matrix = direct_rec_values.to_numpy(
    dtype=float
)


direct_match_matrix = np.isclose(
    original_matrix,
    direct_matrix,
    rtol=DIRECT_COMPARISON_RTOL,
    atol=DIRECT_COMPARISON_ATOL,
    equal_nan=False,
)


direct_mismatch_matrix = (
    ~direct_match_matrix
)


direct_mismatching_values = int(
    direct_mismatch_matrix.sum()
)


dependent_positions = [
    REC_FEATURE_COLUMNS.index(
        feature
    )
    for feature
    in VERDICT_DEPENDENT_REC_FEATURES
]


independent_positions = [
    REC_FEATURE_COLUMNS.index(
        feature
    )
    for feature
    in VERDICT_INDEPENDENT_REC_FEATURES
]


file_history_positions = [
    REC_FEATURE_COLUMNS.index(
        feature
    )
    for feature
    in FILE_HISTORY_REC_FEATURES
]


verdict_dependent_direct_mismatches = int(
    direct_mismatch_matrix[
        :,
        dependent_positions,
    ].sum()
)


verdict_independent_direct_mismatches = int(
    direct_mismatch_matrix[
        :,
        independent_positions,
    ].sum()
)


file_history_direct_mismatches = int(
    direct_mismatch_matrix[
        :,
        file_history_positions,
    ].sum()
)


if (
    direct_mismatching_values
    != EXPECTED_DIRECT_MISMATCH_VALUES
):
    raise AssertionError(
        "Project 10 direct mismatch pattern did not reproduce.\n"
        f"Expected values: {EXPECTED_DIRECT_MISMATCH_VALUES}\n"
        f"Actual values:   {direct_mismatching_values}"
    )


if (
    verdict_dependent_direct_mismatches
    != EXPECTED_DEPENDENT_DIRECT_MISMATCHES
):
    raise AssertionError(
        "Verdict-dependent direct mismatch count differs.\n"
        f"Expected: {EXPECTED_DEPENDENT_DIRECT_MISMATCHES}\n"
        f"Actual:   {verdict_dependent_direct_mismatches}"
    )


if (
    verdict_independent_direct_mismatches
    != EXPECTED_INDEPENDENT_DIRECT_MISMATCHES
):
    raise AssertionError(
        "Verdict-independent direct mismatch count differs.\n"
        f"Expected: {EXPECTED_INDEPENDENT_DIRECT_MISMATCHES}\n"
        f"Actual:   {verdict_independent_direct_mismatches}"
    )


if (
    file_history_direct_mismatches
    != EXPECTED_FILE_HISTORY_DIRECT_MISMATCHES
):
    raise AssertionError(
        "File-history direct mismatch count differs.\n"
        f"Expected: {EXPECTED_FILE_HISTORY_DIRECT_MISMATCHES}\n"
        f"Actual:   {file_history_direct_mismatches}"
    )


anchor_offset_matrix = (
    original_matrix
    - direct_matrix
)


anchor_offsets = pd.DataFrame(
    anchor_offset_matrix,
    columns=REC_FEATURE_COLUMNS,
)


clean_rec_anchor_offsets = pd.concat(
    [
        requested_rows[
            [
                "BuildKey",
                "TestKey",
            ]
        ].reset_index(drop=True),

        anchor_offsets,
    ],
    axis=1,
)


anchored_matrix = (
    direct_matrix
    + anchor_offset_matrix
)


anchored_match_matrix = np.isclose(
    original_matrix,
    anchored_matrix,
    rtol=ANCHOR_COMPARISON_RTOL,
    atol=ANCHOR_COMPARISON_ATOL,
    equal_nan=False,
)


anchored_mismatch_matrix = (
    ~anchored_match_matrix
)


anchored_mismatching_values = int(
    anchored_mismatch_matrix.sum()
)


# ------------------------------------------------------------
# 19. DIRECT-MISMATCH ORIGIN AUDIT
# ------------------------------------------------------------

row_has_direct_mismatch = (
    direct_mismatch_matrix.any(
        axis=1
    )
)


direct_mismatch_rows = requested_rows.loc[
    row_has_direct_mismatch,
    [
        "ModelRowOrder",
        "BuildKey",
        "TestKey",
    ],
].copy()


direct_mismatch_rows[
    "IsTimestampTieBuild"
] = (
    direct_mismatch_rows[
        "BuildKey"
    ].isin(
        timestamp_tie_build_keys
    )
)


direct_mismatch_row_count = len(
    direct_mismatch_rows
)


direct_mismatch_rows_on_tie_builds = int(
    direct_mismatch_rows[
        "IsTimestampTieBuild"
    ].sum()
)


direct_mismatch_rows_outside_tie_builds = int(
    (
        ~direct_mismatch_rows[
            "IsTimestampTieBuild"
        ]
    ).sum()
)


mismatch_build_records = []


for build_key, build_rows in (
    direct_mismatch_rows.groupby(
        "BuildKey",
        sort=False,
    )
):
    positions = build_rows[
        "ModelRowOrder"
    ].to_numpy(
        dtype=int
    )

    feature_value_counts = {}

    for feature_index, feature in enumerate(
        REC_FEATURE_COLUMNS
    ):
        feature_value_counts[
            feature
        ] = int(
            direct_mismatch_matrix[
                positions,
                feature_index,
            ].sum()
        )

    mismatching_features = [
        feature
        for feature, count
        in feature_value_counts.items()
        if count > 0
    ]

    mismatch_build_records.append({
        "BuildKey":
            str(build_key),

        "BuildOrder":
            int(
                build_order_map[
                    str(build_key)
                ]
            ),

        "IsTimestampTieBuild":
            str(build_key)
            in timestamp_tie_build_keys,

        "RowsWithAnyDirectMismatch":
            len(build_rows),

        "MismatchingFeatureValues":
            int(
                direct_mismatch_matrix[
                    positions,
                    :,
                ].sum()
            ),

        "MismatchingFeatures":
            ",".join(
                mismatching_features
            ),
    })


direct_mismatch_build_audit = pd.DataFrame(
    mismatch_build_records,
    columns=[
        "BuildKey",
        "BuildOrder",
        "IsTimestampTieBuild",
        "RowsWithAnyDirectMismatch",
        "MismatchingFeatureValues",
        "MismatchingFeatures",
    ],
)


direct_mismatch_build_count = int(
    direct_mismatch_build_audit[
        "BuildKey"
    ].nunique()
    if not direct_mismatch_build_audit.empty
    else 0
)


direct_mismatch_tie_build_count = int(
    direct_mismatch_build_audit.loc[
        direct_mismatch_build_audit[
            "IsTimestampTieBuild"
        ],
        "BuildKey",
    ].nunique()
    if not direct_mismatch_build_audit.empty
    else 0
)


# ------------------------------------------------------------
# 20. FEATURE COMPARISON TABLES
# ------------------------------------------------------------

comparison_records = []
mismatch_example_records = []

nonzero_anchor_values = 0


for feature_index, feature in enumerate(
    REC_FEATURE_COLUMNS
):
    original = original_matrix[
        :,
        feature_index,
    ]

    direct = direct_matrix[
        :,
        feature_index,
    ]

    offsets = anchor_offset_matrix[
        :,
        feature_index,
    ]

    direct_matches = direct_match_matrix[
        :,
        feature_index,
    ]

    anchored_matches = anchored_match_matrix[
        :,
        feature_index,
    ]

    nonzero_offsets = (
        np.abs(
            offsets
        )
        > NONZERO_OFFSET_THRESHOLD
    )

    nonzero_offset_count = int(
        nonzero_offsets.sum()
    )

    nonzero_anchor_values += (
        nonzero_offset_count
    )

    direct_mismatch_count = int(
        (
            ~direct_matches
        ).sum()
    )

    comparison_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature
                in VERDICT_DEPENDENT_REC_FEATURES
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature
            in FILE_HISTORY_REC_FEATURES,

        "Rows":
            len(original),

        "DirectMatchingRows":
            int(
                direct_matches.sum()
            ),

        "DirectMismatchingRows":
            direct_mismatch_count,

        "NonZeroAnchorOffsets":
            nonzero_offset_count,

        "AnchoredMatchingRows":
            int(
                anchored_matches.sum()
            ),

        "AnchoredMismatchingRows":
            int(
                (
                    ~anchored_matches
                ).sum()
            ),

        "MaximumAbsoluteDirectDifference":
            float(
                np.max(
                    np.abs(
                        original
                        - direct
                    )
                )
            ),

        "MeanAbsoluteDirectDifference":
            float(
                np.mean(
                    np.abs(
                        original
                        - direct
                    )
                )
            ),
    })

    if direct_mismatch_count:
        mismatch_positions = np.flatnonzero(
            ~direct_matches
        )[:100]

        for position in mismatch_positions:
            build_key = str(
                requested_rows[
                    "BuildKey"
                ].iloc[
                    position
                ]
            )

            mismatch_example_records.append({
                "Feature":
                    feature,

                "ModelRowOrder":
                    int(position),

                "BuildKey":
                    build_key,

                "TestKey":
                    str(
                        requested_rows[
                            "TestKey"
                        ].iloc[
                            position
                        ]
                    ),

                "IsTimestampTieBuild":
                    build_key
                    in timestamp_tie_build_keys,

                "OriginalValue":
                    float(
                        original[
                            position
                        ]
                    ),

                "DirectValue":
                    float(
                        direct[
                            position
                        ]
                    ),

                "AnchorOffset":
                    float(
                        offsets[
                            position
                        ]
                    ),

                "AnchoredValue":
                    float(
                        direct[
                            position
                        ]
                        + offsets[
                            position
                        ]
                    ),

                "DirectDifference":
                    float(
                        original[
                            position
                        ]
                        - direct[
                            position
                        ]
                    ),
            })


clean_rec_comparison = pd.DataFrame(
    comparison_records
)


clean_rec_mismatch_examples = pd.DataFrame(
    mismatch_example_records,
    columns=[
        "Feature",
        "ModelRowOrder",
        "BuildKey",
        "TestKey",
        "IsTimestampTieBuild",
        "OriginalValue",
        "DirectValue",
        "AnchorOffset",
        "AnchoredValue",
        "DirectDifference",
    ],
)


rows_with_nonzero_anchor = int(
    (
        np.abs(
            anchor_offset_matrix
        )
        > NONZERO_OFFSET_THRESHOLD
    ).any(
        axis=1
    ).sum()
)


# ------------------------------------------------------------
# 21. CLEAN-ANCHOR VALIDATION
# ------------------------------------------------------------

anchor_validation_records = []


for feature_index, feature in enumerate(
    REC_FEATURE_COLUMNS
):
    feature_matches = anchored_match_matrix[
        :,
        feature_index,
    ]

    maximum_anchored_difference = float(
        np.max(
            np.abs(
                original_matrix[
                    :,
                    feature_index,
                ]
                - anchored_matrix[
                    :,
                    feature_index,
                ]
            )
        )
    )

    mismatch_count = int(
        (
            ~feature_matches
        ).sum()
    )

    anchor_validation_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature
                in VERDICT_DEPENDENT_REC_FEATURES
                else "VERDICT_INDEPENDENT"
            ),

        "Rows":
            len(feature_matches),

        "MatchingRows":
            int(
                feature_matches.sum()
            ),

        "MismatchingRows":
            mismatch_count,

        "MaximumAbsoluteAnchoredDifference":
            maximum_anchored_difference,

        "Pass":
            mismatch_count == 0,
    })


clean_anchor_validation = pd.DataFrame(
    anchor_validation_records
)


failed_anchor_features = int(
    (
        ~clean_anchor_validation[
            "Pass"
        ]
    ).sum()
)


clean_dataset_reproduced = bool(
    anchored_mismatching_values == 0
)


# ------------------------------------------------------------
# 22. VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 1B passed",

        "Expected":
            EXPECTED_STEP1B_STATUS,

        "Actual":
            step1b_status.get(
                "Status"
            ),

        "Pass":
            step1b_status.get(
                "Status"
            ) == EXPECTED_STEP1B_STATUS,
    },

    {
        "Check":
            "Step 2A passed",

        "Expected":
            EXPECTED_STEP2A_STATUS,

        "Actual":
            step2a_status.get(
                "Status"
            ),

        "Pass":
            step2a_status.get(
                "Status"
            ) == EXPECTED_STEP2A_STATUS,
    },

    {
        "Check":
            "Source root SHA-256",

        "Expected":
            EXPECTED_SOURCE_ROOT_SHA256,

        "Actual":
            source_root_sha256,

        "Pass":
            source_root_sha256
            == EXPECTED_SOURCE_ROOT_SHA256,
    },

    {
        "Check":
            "Frozen source files",

        "Expected":
            EXPECTED_SOURCE_FILES,

        "Actual":
            len(
                source_files_payload
            ),

        "Pass":
            len(
                source_files_payload
            ) == EXPECTED_SOURCE_FILES,
    },

    {
        "Check":
            "Canonical builds",

        "Expected":
            EXPECTED_BUILDS,

        "Actual":
            len(
                fixed_split
            ),

        "Pass":
            len(
                fixed_split
            ) == EXPECTED_BUILDS,
    },

    {
        "Check":
            "Timestamp tie groups",

        "Expected":
            EXPECTED_TIMESTAMP_TIE_GROUPS,

        "Actual":
            timestamp_tie_groups,

        "Pass":
            timestamp_tie_groups
            == EXPECTED_TIMESTAMP_TIE_GROUPS,
    },

    {
        "Check":
            "Raw execution rows",

        "Expected":
            EXPECTED_RAW_ROWS,

        "Actual":
            len(
                execution_history
            ),

        "Pass":
            len(
                execution_history
            ) == EXPECTED_RAW_ROWS,
    },

    {
        "Check":
            "Model-ready rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(
                requested_rows
            ),

        "Pass":
            len(
                requested_rows
            ) == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "Raw duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            raw_duplicate_build_test_rows,

        "Pass":
            raw_duplicate_build_test_rows == 0,
    },

    {
        "Check":
            "Reconstructed REC rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(
                clean_rec_reconstructed
            ),

        "Pass":
            len(
                clean_rec_reconstructed
            ) == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "Duplicate reconstructed rows",

        "Expected":
            0,

        "Actual":
            duplicate_reconstructed_rows,

        "Pass":
            duplicate_reconstructed_rows == 0,
    },

    {
        "Check":
            "Missing reconstructed rows",

        "Expected":
            0,

        "Actual":
            missing_reconstructed_rows,

        "Pass":
            missing_reconstructed_rows == 0,
    },

    {
        "Check":
            "REC features reconstructed",

        "Expected":
            19,

        "Actual":
            len(
                REC_FEATURE_COLUMNS
            ),

        "Pass":
            len(
                REC_FEATURE_COLUMNS
            ) == 19,
    },

    {
        "Check":
            "Reproduced direct mismatch values",

        "Expected":
            EXPECTED_DIRECT_MISMATCH_VALUES,

        "Actual":
            direct_mismatching_values,

        "Pass":
            direct_mismatching_values
            == EXPECTED_DIRECT_MISMATCH_VALUES,
    },

    {
        "Check":
            "Reproduced dependent direct mismatches",

        "Expected":
            EXPECTED_DEPENDENT_DIRECT_MISMATCHES,

        "Actual":
            verdict_dependent_direct_mismatches,

        "Pass":
            verdict_dependent_direct_mismatches
            == EXPECTED_DEPENDENT_DIRECT_MISMATCHES,
    },

    {
        "Check":
            "Reproduced independent direct mismatches",

        "Expected":
            EXPECTED_INDEPENDENT_DIRECT_MISMATCHES,

        "Actual":
            verdict_independent_direct_mismatches,

        "Pass":
            verdict_independent_direct_mismatches
            == EXPECTED_INDEPENDENT_DIRECT_MISMATCHES,
    },

    {
        "Check":
            "Reproduced file-history direct mismatches",

        "Expected":
            EXPECTED_FILE_HISTORY_DIRECT_MISMATCHES,

        "Actual":
            file_history_direct_mismatches,

        "Pass":
            file_history_direct_mismatches
            == EXPECTED_FILE_HISTORY_DIRECT_MISMATCHES,
    },

    {
        "Check":
            "Failed clean-anchor features",

        "Expected":
            0,

        "Actual":
            failed_anchor_features,

        "Pass":
            failed_anchor_features == 0,
    },

    {
        "Check":
            "Anchored mismatch values",

        "Expected":
            0,

        "Actual":
            anchored_mismatching_values,

        "Pass":
            anchored_mismatching_values == 0,
    },

    {
        "Check":
            "Clean dataset reproduced",

        "Expected":
            True,

        "Actual":
            clean_dataset_reproduced,

        "Pass":
            clean_dataset_reproduced,
    },

    {
        "Check":
            "Ambiguous commit tokens",

        "Expected":
            0,

        "Actual":
            ambiguous_commit_tokens,

        "Pass":
            ambiguous_commit_tokens == 0,
    },

    {
        "Check":
            "Unmatched commit tokens",

        "Expected":
            0,

        "Actual":
            unmatched_commit_tokens,

        "Pass":
            unmatched_commit_tokens == 0,
    },

    {
        "Check":
            "Commit-token coverage",

        "Expected":
            100.0,

        "Actual":
            commit_token_coverage_percent,

        "Pass":
            bool(
                np.isclose(
                    commit_token_coverage_percent,
                    100.0,
                    rtol=0,
                    atol=1e-12,
                )
            ),
    },

    {
        "Check":
            "Builds without commit tokens",

        "Expected":
            0,

        "Actual":
            builds_without_commit_tokens,

        "Pass":
            builds_without_commit_tokens == 0,
    },

    {
        "Check":
            "Builds with no matched commit",

        "Expected":
            0,

        "Actual":
            builds_with_no_matched_commit,

        "Pass":
            builds_with_no_matched_commit == 0,
    },

    {
        "Check":
            "Builds with mapped entities",

        "Expected":
            EXPECTED_BUILDS,

        "Actual":
            builds_with_mapped_entities,

        "Pass":
            builds_with_mapped_entities
            == EXPECTED_BUILDS,
    },

    {
        "Check":
            "Unmapped entity-history IDs",

        "Expected":
            0,

        "Actual":
            len(
                unmapped_entity_ids
            ),

        "Pass":
            len(
                unmapped_entity_ids
            ) == 0,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    9
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    9
                ).sum()
            ) == 0,
    },

    {
        "Check":
            "Registry Project 10 rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    10
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    10
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 2B V2 validation:")

display(
    validation
)


if not failed_checks.empty:
    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 10 STEP 2B V2 DID NOT PASS."
    )


# ------------------------------------------------------------
# 23. WRITE PROJECT 10 FROZEN OUTPUTS
# ------------------------------------------------------------

REC_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    build_commit_token_profile,
)

atomic_write_csv(
    COMMIT_MATCHING_AUDIT_PATH,
    commit_matching_audit,
)

atomic_write_csv_gzip(
    BUILD_ENTITY_MAP_PATH,
    build_entity_map,
)

atomic_write_json(
    ENTITY_MAPPING_SUMMARY_PATH,
    entity_mapping_summary,
)

atomic_write_parquet(
    CLEAN_REC_RECONSTRUCTED_PATH,
    clean_rec_reconstructed,
)

atomic_write_parquet(
    CLEAN_REC_ANCHOR_OFFSETS_PATH,
    clean_rec_anchor_offsets,
)

atomic_write_csv(
    CLEAN_REC_COMPARISON_PATH,
    clean_rec_comparison,
)

atomic_write_csv(
    CLEAN_REC_MISMATCH_EXAMPLES_PATH,
    clean_rec_mismatch_examples,
)

atomic_write_csv(
    DIRECT_MISMATCH_BUILD_AUDIT_PATH,
    direct_mismatch_build_audit,
)

atomic_write_csv(
    TIMESTAMP_TIE_AUDIT_PATH,
    timestamp_tie_audit,
)

atomic_write_csv(
    CLEAN_ANCHOR_VALIDATION_PATH,
    clean_anchor_validation,
)

atomic_write_csv(
    STEP2B_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 24. READBACK VALIDATION
# ------------------------------------------------------------

clean_rec_readback = pd.read_parquet(
    CLEAN_REC_RECONSTRUCTED_PATH
)

anchor_readback = pd.read_parquet(
    CLEAN_REC_ANCHOR_OFFSETS_PATH
)

build_entity_readback = pd.read_csv(
    BUILD_ENTITY_MAP_PATH,
    compression="gzip",
    low_memory=False,
)

comparison_readback = pd.read_csv(
    CLEAN_REC_COMPARISON_PATH,
    low_memory=False,
)


if len(clean_rec_readback) != EXPECTED_MODEL_ROWS:
    raise AssertionError(
        "Clean REC readback row count differs."
    )


if len(anchor_readback) != EXPECTED_MODEL_ROWS:
    raise AssertionError(
        "Clean anchor readback row count differs."
    )


if len(build_entity_readback) != len(
    build_entity_map
):
    raise AssertionError(
        "Build/entity readback row count differs."
    )


if int(
    comparison_readback[
        "DirectMismatchingRows"
    ].sum()
) != EXPECTED_DIRECT_MISMATCH_VALUES:
    raise AssertionError(
        "Direct mismatch comparison readback differs."
    )


if int(
    comparison_readback[
        "AnchoredMismatchingRows"
    ].sum()
) != 0:
    raise AssertionError(
        "Anchor comparison readback contains mismatches."
    )


# ------------------------------------------------------------
# 25. PROJECT 9 READ-ONLY END SNAPSHOT
# ------------------------------------------------------------

project_9_end_snapshot = {}


if PROJECT_9_PROGRESS_PATH.exists():
    try:
        project_9_end_snapshot = (
            read_json_with_retry(
                PROJECT_9_PROGRESS_PATH
            )
        )
    except Exception:
        project_9_end_snapshot = {
            "Status":
                "READ_RETRY_EXHAUSTED"
        }


# ------------------------------------------------------------
# 26. REPORT AND CHECKPOINT
# ------------------------------------------------------------

report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProjectShortName":
        PROJECT_SHORT_NAME,

    "Status":
        STEP2B_PASS_STATUS,

    "ReconstructionVersion":
        "STEP_2B_V2_CLEAN_ANCHORED_DELTA",

    "SourceRootSHA256":
        source_root_sha256,

    "RecentWindow":
        RECENT_WINDOW,

    "RECFeatures":
        REC_FEATURE_COLUMNS,

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_FEATURES,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC_FEATURES,

    "RawExecutionRows":
        len(
            execution_history
        ),

    "ModelReadyRows":
        len(
            requested_rows
        ),

    "ReconstructedRows":
        len(
            clean_rec_reconstructed
        ),

    "DuplicateReconstructedRows":
        duplicate_reconstructed_rows,

    "MissingReconstructedRows":
        missing_reconstructed_rows,

    "ReconstructionSeconds":
        reconstruction_seconds,

    "TimestampTieGroups":
        timestamp_tie_groups,

    "TimestampTieBuilds":
        sorted(
            timestamp_tie_build_keys
        ),

    "DirectMismatchingFeatureValues":
        direct_mismatching_values,

    "VerdictDependentDirectMismatches":
        verdict_dependent_direct_mismatches,

    "VerdictIndependentDirectMismatches":
        verdict_independent_direct_mismatches,

    "FileHistoryDirectMismatches":
        file_history_direct_mismatches,

    "RowsWithDirectMismatch":
        direct_mismatch_row_count,

    "DirectMismatchBuilds":
        direct_mismatch_build_count,

    "DirectMismatchRowsOnTimestampTieBuilds":
        direct_mismatch_rows_on_tie_builds,

    "DirectMismatchRowsOutsideTimestampTieBuilds":
        direct_mismatch_rows_outside_tie_builds,

    "DirectMismatchTieBuildCount":
        direct_mismatch_tie_build_count,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_nonzero_anchor,

    "NonZeroAnchorOffsetValues":
        nonzero_anchor_values,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchingFeatureValues":
        anchored_mismatching_values,

    "CleanDatasetReproducedExactly":
        clean_dataset_reproduced,

    "AnchorPolicy":
        (
            "OriginalClean + (DirectNoisy - DirectClean); "
            "equivalently DirectNoisy + CleanAnchorOffset"
        ),

    "BuildCommitTokenRows":
        total_commit_tokens,

    "ExactCommitMatches":
        exact_commit_matches,

    "UniquePrefixMatches":
        unique_prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "AmbiguousCommitTokens":
        ambiguous_commit_tokens,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithMappedEntities":
        builds_with_mapped_entities,

    "BuildEntityRows":
        len(
            build_entity_map
        ),

    "Project9StartReadOnlySnapshot":
        project_9_start_snapshot,

    "Project9EndReadOnlySnapshot":
        project_9_end_snapshot,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletionRegistryModified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProjectShortName":
        PROJECT_SHORT_NAME,

    "Status":
        STEP2B_PASS_STATUS,

    "ReconstructionVersion":
        "STEP_2B_V2_CLEAN_ANCHORED_DELTA",

    "SourceRootSHA256":
        source_root_sha256,

    "SelectionCheckpointSHA256":
        calculate_hash(
            SELECTION_CHECKPOINT_PATH,
            algorithm="sha256",
        ),

    "Step2AReportSHA256":
        calculate_hash(
            STEP2A_REPORT_PATH,
            algorithm="sha256",
        ),

    "ResolvedColumns":
        resolved_columns,

    "RecentWindow":
        RECENT_WINDOW,

    "RECFeatureColumns":
        REC_FEATURE_COLUMNS,

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_FEATURES,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC_FEATURES,

    "RawExecutionRows":
        len(
            execution_history
        ),

    "ModelReadyRows":
        len(
            requested_rows
        ),

    "TimestampTieGroups":
        timestamp_tie_groups,

    "TimestampTieBuilds":
        sorted(
            timestamp_tie_build_keys
        ),

    "DirectMismatchingFeatureValues":
        direct_mismatching_values,

    "VerdictDependentDirectMismatches":
        verdict_dependent_direct_mismatches,

    "VerdictIndependentDirectMismatches":
        verdict_independent_direct_mismatches,

    "FileHistoryDirectMismatches":
        file_history_direct_mismatches,

    "RowsWithDirectMismatch":
        direct_mismatch_row_count,

    "DirectMismatchBuilds":
        direct_mismatch_build_count,

    "DirectMismatchRowsOnTimestampTieBuilds":
        direct_mismatch_rows_on_tie_builds,

    "DirectMismatchRowsOutsideTimestampTieBuilds":
        direct_mismatch_rows_outside_tie_builds,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_nonzero_anchor,

    "NonZeroAnchorOffsetValues":
        nonzero_anchor_values,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchingFeatureValues":
        anchored_mismatching_values,

    "CleanDatasetReproducedExactly":
        clean_dataset_reproduced,

    "AnchorPolicy":
        (
            "OriginalClean + (DirectNoisy - DirectClean)"
        ),

    "BuildCommitTokenRows":
        total_commit_tokens,

    "ExactCommitMatches":
        exact_commit_matches,

    "UniquePrefixMatches":
        unique_prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "AmbiguousCommitTokens":
        ambiguous_commit_tokens,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithMappedEntities":
        builds_with_mapped_entities,

    "BuildEntityRows":
        len(
            build_entity_map
        ),

    "BuildCommitTokenProfile":
        str(
            BUILD_COMMIT_TOKEN_PROFILE_PATH
        ),

    "BuildCommitTokenProfileSHA256":
        calculate_hash(
            BUILD_COMMIT_TOKEN_PROFILE_PATH,
            algorithm="sha256",
        ),

    "CommitMatchingAudit":
        str(
            COMMIT_MATCHING_AUDIT_PATH
        ),

    "CommitMatchingAuditSHA256":
        calculate_hash(
            COMMIT_MATCHING_AUDIT_PATH,
            algorithm="sha256",
        ),

    "BuildEntityMap":
        str(
            BUILD_ENTITY_MAP_PATH
        ),

    "BuildEntityMapSHA256":
        calculate_hash(
            BUILD_ENTITY_MAP_PATH,
            algorithm="sha256",
        ),

    "EntityMappingSummary":
        str(
            ENTITY_MAPPING_SUMMARY_PATH
        ),

    "EntityMappingSummarySHA256":
        calculate_hash(
            ENTITY_MAPPING_SUMMARY_PATH,
            algorithm="sha256",
        ),

    "CleanRECReconstructed":
        str(
            CLEAN_REC_RECONSTRUCTED_PATH
        ),

    "CleanRECReconstructedSHA256":
        calculate_hash(
            CLEAN_REC_RECONSTRUCTED_PATH,
            algorithm="sha256",
        ),

    "CleanRECAnchorOffsets":
        str(
            CLEAN_REC_ANCHOR_OFFSETS_PATH
        ),

    "CleanRECAnchorOffsetsSHA256":
        calculate_hash(
            CLEAN_REC_ANCHOR_OFFSETS_PATH,
            algorithm="sha256",
        ),

    "CleanRECComparison":
        str(
            CLEAN_REC_COMPARISON_PATH
        ),

    "CleanRECComparisonSHA256":
        calculate_hash(
            CLEAN_REC_COMPARISON_PATH,
            algorithm="sha256",
        ),

    "CleanRECMismatchExamples":
        str(
            CLEAN_REC_MISMATCH_EXAMPLES_PATH
        ),

    "CleanRECMismatchExamplesSHA256":
        calculate_hash(
            CLEAN_REC_MISMATCH_EXAMPLES_PATH,
            algorithm="sha256",
        ),

    "DirectMismatchBuildAudit":
        str(
            DIRECT_MISMATCH_BUILD_AUDIT_PATH
        ),

    "DirectMismatchBuildAuditSHA256":
        calculate_hash(
            DIRECT_MISMATCH_BUILD_AUDIT_PATH,
            algorithm="sha256",
        ),

    "TimestampTieAudit":
        str(
            TIMESTAMP_TIE_AUDIT_PATH
        ),

    "TimestampTieAuditSHA256":
        calculate_hash(
            TIMESTAMP_TIE_AUDIT_PATH,
            algorithm="sha256",
        ),

    "CleanAnchorValidation":
        str(
            CLEAN_ANCHOR_VALIDATION_PATH
        ),

    "CleanAnchorValidationSHA256":
        calculate_hash(
            CLEAN_ANCHOR_VALIDATION_PATH,
            algorithm="sha256",
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistryRows":
        len(
            registry
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "CompletionRegistryModified":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    REC_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_PASS_STATUS,

    "ReconstructionVersion":
        "STEP_2B_V2_CLEAN_ANCHORED_DELTA",

    "ReconstructedRows":
        len(
            clean_rec_reconstructed
        ),

    "DirectMismatchingFeatureValues":
        direct_mismatching_values,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchingFeatureValues":
        anchored_mismatching_values,

    "CleanDatasetReproducedExactly":
        clean_dataset_reproduced,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "Checkpoint":
        str(
            REC_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_hash(
            REC_CHECKPOINT_PATH,
            algorithm="sha256",
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2B_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 27. FINAL READBACK AND IMMUTABILITY CHECK
# ------------------------------------------------------------

checkpoint_readback = read_json_with_retry(
    REC_CHECKPOINT_PATH
)

status_readback = read_json_with_retry(
    STEP2B_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP2B_PASS_STATUS:
    raise AssertionError(
        "Project 10 REC checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP2B_PASS_STATUS:
    raise AssertionError(
        "Project 10 Step 2B status readback failed."
    )


registry_sha256_after = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:
    raise AssertionError(
        "Completion registry changed during Step 2B V2."
    )


source_root_sha256_after = create_source_root_sha256(
    source_files_payload
)


source_files_unchanged = (
    source_root_sha256_after
    == source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256
)


if not source_files_unchanged:
    raise AssertionError(
        "Project 10 source files changed during Step 2B V2."
    )


# ------------------------------------------------------------
# 28. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nClean REC feature comparison:")

display(
    clean_rec_comparison
)


print("\nClean-anchor validation:")

display(
    clean_anchor_validation
)


print("\nTimestamp-tie audit:")

display(
    timestamp_tie_audit
)


print("\nDirect-mismatch build audit:")

display(
    direct_mismatch_build_audit
)


print("\nCommit matching audit summary:")

display(
    commit_matching_audit.groupby(
        "MatchType",
        as_index=False,
    ).agg(
        Rows=(
            "CommitToken",
            "count",
        )
    )
)


# ------------------------------------------------------------
# 29. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 126)
print("=== PROJECT 10 CELL 4 / STEP 2B V2 RESULT ===")
print("=" * 126)

print("\nProject:")
print(PROJECT_NAME)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


print("\nCommit and entity mapping:")

print(
    "Build commit-token rows:",
    total_commit_tokens,
)

print(
    "Exact commit matches:",
    exact_commit_matches,
)

print(
    "Unique-prefix matches:",
    unique_prefix_matches,
)

print(
    "Unmatched commit tokens:",
    unmatched_commit_tokens,
)

print(
    "Ambiguous commit tokens:",
    ambiguous_commit_tokens,
)

print(
    "Commit-token coverage:",
    commit_token_coverage_percent,
)

print(
    "Builds with mapped entities:",
    builds_with_mapped_entities,
)

print(
    "Build/entity rows:",
    len(
        build_entity_map
    ),
)


print("\nClean REC reconstruction:")

print(
    "Raw history rows:",
    len(
        execution_history
    ),
)

print(
    "Requested model-ready rows:",
    len(
        requested_rows
    ),
)

print(
    "Reconstructed rows:",
    len(
        clean_rec_reconstructed
    ),
)

print(
    "Duplicate reconstructed rows:",
    duplicate_reconstructed_rows,
)

print(
    "Missing reconstructed rows:",
    missing_reconstructed_rows,
)

print(
    "Reconstruction seconds:",
    reconstruction_seconds,
)


print("\nDirect reconstruction audit:")

print(
    "Direct mismatching feature values:",
    direct_mismatching_values,
)

print(
    "Verdict-dependent direct mismatches:",
    verdict_dependent_direct_mismatches,
)

print(
    "Verdict-independent direct mismatches:",
    verdict_independent_direct_mismatches,
)

print(
    "File-history direct mismatches:",
    file_history_direct_mismatches,
)

print(
    "Rows with a direct mismatch:",
    direct_mismatch_row_count,
)

print(
    "Builds with a direct mismatch:",
    direct_mismatch_build_count,
)

print(
    "Timestamp tie groups:",
    timestamp_tie_groups,
)

print(
    "Direct mismatch rows on tie builds:",
    direct_mismatch_rows_on_tie_builds,
)

print(
    "Direct mismatch rows outside tie builds:",
    direct_mismatch_rows_outside_tie_builds,
)


print("\nClean-anchor freeze:")

print(
    "Rows with any non-zero anchor offset:",
    rows_with_nonzero_anchor,
)

print(
    "Non-zero anchor-offset values:",
    nonzero_anchor_values,
)

print(
    "Failed anchor features:",
    failed_anchor_features,
)

print(
    "Anchored mismatching feature values:",
    anchored_mismatching_values,
)

print(
    "0% clean dataset reproduced exactly:",
    clean_dataset_reproduced,
)


print("\nImmutability:")

print(
    "Project 10 source files unchanged:",
    source_files_unchanged,
)

print(
    "Completion registry unchanged:",
    registry_unchanged,
)

print(
    "Project 9 write attempted:",
    False,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nProject 9 read-only end snapshot:")

print(
    "Status:",
    project_9_end_snapshot.get(
        "Status"
    ),
)

print(
    "Completed conditions:",
    project_9_end_snapshot.get(
        "CompletedConditions"
    ),
)

print(
    "Last completed condition:",
    project_9_end_snapshot.get(
        "LastCompletedCondition"
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nFrozen REC checkpoint:")

print(
    REC_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_hash(
        REC_CHECKPOINT_PATH,
        algorithm="sha256",
    ),
)


print(
    "\nSTATUS:",
    STEP2B_PASS_STATUS,
)

print("=" * 126)

=== PROJECT 10 CELL 4 / STEP 2B V2: CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===

Project 9 read-only start snapshot:
Status: RUNNING_PROJECT_9_FULL_270_CONDITION_EXPERIMENT
Completed conditions: 226
Last completed condition: noise_00__seed_26

Resolved reconstruction schema:
Build commit column: commits
Entity-history commit column: Commit
Entity-history ID column: EntityId
Resolved ID-map entity column: value
Resolved ID-map entity side: VALUE

Commit/entity mapping summary:
Build commit tokens: 4858
Exact matches: 4858
Unique-prefix matches: 0
Unmatched tokens: 0
Ambiguous tokens: 0
Token coverage percent: 100.0
Builds without commit tokens: 0
Builds with no matched commit: 0
Builds with mapped entities: 408
Build/entity rows: 10853

Reconstructing all 19 clean REC features...
REC reconstruction progress: 100 / 166 tests | reconstructed rows: 5541

Clean REC reconstruction completed:
Requested rows: 8706
Reconstructed rows: 8706
Duplicate reconstructed rows: 0
Missing reconstructe

,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,True
1,Step 2A passed,PASS_PROJECT_10_SOURCE_SCHEMA_AND_JOIN_STRUCTU...,PASS_PROJECT_10_SOURCE_SCHEMA_AND_JOIN_STRUCTU...,True
2,Source root SHA-256,582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3...,582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3...,True
3,Frozen source files,6,6,True
4,Canonical builds,408,408,True
5,Timestamp tie groups,2,2,True
6,Raw execution rows,47094,47094,True
7,Model-ready rows,8706,8706,True
8,Raw duplicate Build/Test rows,0,0,True
9,Reconstructed REC rows,8706,8706,True



Clean REC feature comparison:


,Feature,FeatureClass,FileHistoryFeature,Rows,DirectMatchingRows,DirectMismatchingRows,NonZeroAnchorOffsets,AnchoredMatchingRows,AnchoredMismatchingRows,MaximumAbsoluteDirectDifference,MeanAbsoluteDirectDifference
0,REC_Age,VERDICT_INDEPENDENT,False,8706,8594,112,112,8706,0,1.000000e+00,1.286469e-02
1,REC_LastFailureAge,VERDICT_DEPENDENT,False,8706,8668,38,38,8706,0,1.000000e+00,4.364806e-03
2,REC_LastTransitionAge,VERDICT_DEPENDENT,False,8706,8669,37,37,8706,0,1.000000e+00,4.249943e-03
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,False,8706,8605,101,406,8706,0,1.727667e+03,1.513056e+00
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,False,8706,8690,16,16,8706,0,2.140000e+02,1.152079e-01
5,REC_RecentFailRate,VERDICT_DEPENDENT,False,8706,8705,1,1,8706,0,1.666667e-01,1.914389e-05
6,REC_RecentAssertRate,VERDICT_DEPENDENT,False,8706,8705,1,1,8706,0,1.666667e-01,1.914389e-05
7,REC_RecentExcRate,VERDICT_DEPENDENT,False,8706,8706,0,0,8706,0,5.551115e-17,6.376195e-19
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,False,8706,8705,1,1,8706,0,3.333333e-01,3.828777e-05
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,False,8706,8600,106,424,8706,0,2.970234e+01,2.605493e-02



Clean-anchor validation:


...


Direct-mismatch build audit:


,BuildKey,BuildOrder,IsTimestampTieBuild,RowsWithAnyDirectMismatch,MismatchingFeatureValues,MismatchingFeatures
0,621322679,387,True,112,629,"REC_Age,REC_LastFailureAge,REC_LastTransitionA..."
1,621648167,391,False,1,5,"REC_LastFailureAge,REC_LastTransitionAge,REC_R..."
2,625714212,402,False,1,2,"REC_TotalTransitionRate,REC_MaxTestFileTransit..."



Commit matching audit summary:


,MatchType,Rows
0,EXACT_NORMALISED_COMMIT_TOKEN,4858




=== PROJECT 10 CELL 4 / STEP 2B V2 RESULT ===

Project:
spring-cloud@spring-cloud-dataflow
Source root SHA-256: 582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e

Commit and entity mapping:
Build commit-token rows: 4858
Exact commit matches: 4858
Unique-prefix matches: 0
Unmatched commit tokens: 0
Ambiguous commit tokens: 0
Commit-token coverage: 100.0
Builds with mapped entities: 408
Build/entity rows: 10853

Clean REC reconstruction:
Raw history rows: 47094
Requested model-ready rows: 8706
Reconstructed rows: 8706
Duplicate reconstructed rows: 0
Missing reconstructed rows: 0
Reconstruction seconds: 26.98803075700016

Direct reconstruction audit:
Direct mismatching feature values: 636
Verdict-dependent direct mismatches: 199
Verdict-independent direct mismatches: 437
File-history direct mismatches: 3
Rows with a direct mismatch: 114
Builds with a direct mismatch: 3
Timestamp tie groups: 2
Direct mismatch rows on tie builds: 112
Direct mismatch rows outside tie builds:

In [7]:
# ============================================================
# PROJECT 10 — CELL 5 / STEP 3A
# DETERMINISTIC NOISE PLAN AND FIXED COHORT FREEZE
#
# PROJECT:
#   spring-cloud@spring-cloud-dataflow
#
# This cell:
# - freezes clean raw training/evaluation cohorts
# - freezes clean model-ready training/evaluation cohorts
# - creates deterministic project-specific RNG streams
# - creates all 270 condition definitions
# - proves masks are nested for every seed
# - freezes failure-subtype sampling probabilities
# - leaves evaluation data clean and immutable
#
# It does NOT:
# - reconstruct noisy REC features yet
# - train models
# - modify Project 9
# - modify Projects 1–8
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)


EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)

EXPECTED_STEP2A_STATUS = (
    "PASS_PROJECT_10_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_AUDITED"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

STEP3A_PASS_STATUS = (
    "PASS_PROJECT_10_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e"
)

EXPECTED_BUILDS = 408

EXPECTED_RAW_TRAINING_ROWS = 34563
EXPECTED_RAW_EVALUATION_ROWS = 12531
EXPECTED_RAW_TRAINING_FAILURES = 65
EXPECTED_RAW_EVALUATION_FAILURES = 213

EXPECTED_MODEL_TRAINING_ROWS = 6095
EXPECTED_MODEL_EVALUATION_ROWS = 2611
EXPECTED_MODEL_TRAINING_FAILURES = 63
EXPECTED_MODEL_EVALUATION_FAILURES = 213
EXPECTED_MODEL_FAILING_EVALUATION_BUILDS = 27

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)

EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_noise_plan_checkpoint.json"
)


PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)

STEP2A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_status.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)

NOISE_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_noise_preflight"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_evaluation_cohort.parquet"
)

FAILURE_SUBTYPE_PROFILE_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_failure_subtype_profile.csv"
)

NOISE_RNG_MANIFEST_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_noise_rng_manifest.parquet"
)

SEED_MANIFEST_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_seed_manifest.csv"
)

CONDITION_PLAN_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_condition_plan.csv"
)

NESTED_MASK_AUDIT_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_nested_mask_audit.csv"
)

STEP3A_VALIDATION_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_validation.csv"
)

STEP3A_REPORT_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_report.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)


PROJECT_9_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "camunda__camunda-bpm-platform"
)

PROJECT_9_PROGRESS_PATH = (
    PROJECT_9_DIR
    / "camunda_full_run_control"
    / "camunda_full_run_progress.json"
)


print("=" * 124)
print("=== PROJECT 10 CELL 5 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===")
print("=" * 124)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def array_sha256(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="snappy",
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json_with_retry(
    path,
    attempts=10,
    delay_seconds=0.5,
):
    path = Path(path)

    last_error = None

    for _ in range(attempts):
        try:
            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:
            last_error = error

            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON.\n"
        f"Path: {path}\n"
        f"Error: {type(last_error).__name__}: {last_error}"
    )


def canonical_identifier(
    series,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:
        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:
            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


def create_source_root_sha256(
    source_files_payload,
):
    digest = hashlib.sha256()

    for relative_path in sorted(
        source_files_payload
    ):
        metadata = source_files_payload[
            relative_path
        ]

        runtime_path = Path(
            metadata[
                "RuntimePath"
            ]
        )

        size_bytes = int(
            runtime_path.stat().st_size
        )

        file_sha256 = calculate_hash(
            runtime_path,
            algorithm="sha256",
        )

        digest.update(
            (
                f"{relative_path}\0"
                f"{size_bytes}\0"
                f"{file_sha256}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


# ------------------------------------------------------------
# 4. INPUT VALIDATION
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    STEP2A_STATUS_PATH,
    STEP2B_STATUS_PATH,
    STEP2A_REPORT_PATH,
]


missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:
    raise FileNotFoundError(
        "Required Project 10 Step 3A inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


selection_checkpoint = read_json_with_retry(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint = read_json_with_retry(
    REC_CHECKPOINT_PATH
)

step1b_status = read_json_with_retry(
    STEP1B_STATUS_PATH
)

step2a_status = read_json_with_retry(
    STEP2A_STATUS_PATH
)

step2b_status = read_json_with_retry(
    STEP2B_STATUS_PATH
)

step2a_report = read_json_with_retry(
    STEP2A_REPORT_PATH
)


if selection_checkpoint.get(
    "Status"
) != EXPECTED_STEP1B_STATUS:
    raise AssertionError(
        "Project 10 Step 1B checkpoint status differs."
    )


if step1b_status.get(
    "Status"
) != EXPECTED_STEP1B_STATUS:
    raise AssertionError(
        "Project 10 Step 1B status differs."
    )


if step2a_status.get(
    "Status"
) != EXPECTED_STEP2A_STATUS:
    raise AssertionError(
        "Project 10 Step 2A status differs."
    )


if step2b_status.get(
    "Status"
) != EXPECTED_STEP2B_STATUS:
    raise AssertionError(
        "Project 10 Step 2B status differs."
    )


if rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP2B_STATUS:
    raise AssertionError(
        "Project 10 REC checkpoint status differs."
    )


if not rec_checkpoint.get(
    "CleanDatasetReproducedExactly",
    False,
):
    raise AssertionError(
        "Step 2B did not freeze an exact clean anchor."
    )


if int(
    rec_checkpoint.get(
        "AnchoredMismatchingFeatureValues",
        -1,
    )
) != 0:
    raise AssertionError(
        "Step 2B clean anchor contains mismatches."
    )


if selection_checkpoint.get(
    "Project"
) != PROJECT_NAME:
    raise AssertionError(
        "Project 10 identity differs."
    )


if selection_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Project 10 source-root SHA-256 differs."
    )


# ------------------------------------------------------------
# 5. OUTPUT ISOLATION
# ------------------------------------------------------------

output_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    FAILURE_SUBTYPE_PROFILE_PATH,
    NOISE_RNG_MANIFEST_PATH,
    SEED_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NESTED_MASK_AUDIT_PATH,
    STEP3A_VALIDATION_PATH,
    STEP3A_REPORT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    STEP3A_STATUS_PATH,
]


for output_path in output_paths:
    output_string = str(
        output_path
    )

    if (
        PROJECT_SLUG not in output_string
        and "project_10_" not in output_string
    ):
        raise AssertionError(
            "A Step 3A output path is not Project 10 isolated.\n"
            f"Path: {output_path}"
        )

    if str(
        PROJECT_9_DIR
    ) in output_string:
        raise AssertionError(
            "A Project 10 path overlaps Project 9."
        )


# ------------------------------------------------------------
# 6. PROJECT 9 READ-ONLY START SNAPSHOT
# ------------------------------------------------------------

project_9_start_snapshot = {}


if PROJECT_9_PROGRESS_PATH.exists():
    try:
        project_9_start_snapshot = (
            read_json_with_retry(
                PROJECT_9_PROGRESS_PATH
            )
        )
    except Exception:
        project_9_start_snapshot = {
            "Status":
                "READ_RETRY_EXHAUSTED"
        }


print("\nProject 9 read-only start snapshot:")

print(
    "Status:",
    project_9_start_snapshot.get(
        "Status"
    ),
)

print(
    "Completed conditions:",
    project_9_start_snapshot.get(
        "CompletedConditions"
    ),
)

print(
    "Last completed condition:",
    project_9_start_snapshot.get(
        "LastCompletedCondition"
    ),
)


# ------------------------------------------------------------
# 7. REGISTRY — READ ONLY
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != 8
    or set(
        registry_project_numbers
    ) != set(
        range(1, 9)
    )
):
    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )


if registry_project_numbers.eq(9).any():
    raise AssertionError(
        "Project 9 was unexpectedly registered."
    )


if registry_project_numbers.eq(10).any():
    raise AssertionError(
        "Project 10 was unexpectedly registered."
    )


# ------------------------------------------------------------
# 8. VERIFY FROZEN SOURCE
# ------------------------------------------------------------

source_files_payload = (
    selection_checkpoint[
        "SourceFiles"
    ]
)


source_root_before = create_source_root_sha256(
    source_files_payload
)


if source_root_before != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Project 10 source-root SHA-256 differs before Step 3A."
    )


fixed_split_path = Path(
    selection_checkpoint[
        "FixedSplit"
    ]
)

executions_path = Path(
    selection_checkpoint[
        "ExecutionHistoryPath"
    ]
)

dataset_path = Path(
    selection_checkpoint[
        "DatasetPath"
    ]
)


if calculate_hash(
    fixed_split_path,
    algorithm="sha256",
) != selection_checkpoint[
    "FixedSplitSHA256"
]:
    raise AssertionError(
        "Frozen chronological split hash differs."
    )


fixed_split = pd.read_csv(
    fixed_split_path,
    low_memory=False,
)

executions = pd.read_csv(
    executions_path,
    low_memory=False,
)

dataset = pd.read_csv(
    dataset_path,
    low_memory=False,
)


resolved_columns = (
    step2a_report[
        "ResolvedColumns"
    ]
)


execution_build_column = (
    resolved_columns[
        "ExecutionBuild"
    ]
)

execution_job_column = (
    resolved_columns[
        "ExecutionJob"
    ]
)

execution_test_column = (
    resolved_columns[
        "ExecutionTest"
    ]
)

execution_verdict_column = (
    resolved_columns[
        "ExecutionVerdict"
    ]
)

execution_duration_column = (
    resolved_columns[
        "ExecutionDuration"
    ]
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)


# ------------------------------------------------------------
# 9. FREEZE RAW TRAINING AND EVALUATION COHORTS
# ------------------------------------------------------------

fixed_split[
    "BuildKey"
] = fixed_split[
    "BuildKey"
].astype(str)

fixed_split[
    "BuildOrder"
] = pd.to_numeric(
    fixed_split[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


partition_map = fixed_split.set_index(
    "BuildKey"
)[
    "Partition"
].to_dict()

build_order_map = fixed_split.set_index(
    "BuildKey"
)[
    "BuildOrder"
].to_dict()


raw_history = pd.DataFrame({
    "SourceRawRowOrder":
        np.arange(
            len(executions),
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            executions[
                execution_build_column
            ]
        ),

    "JobKey":
        canonical_identifier(
            executions[
                execution_job_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            executions[
                execution_test_column
            ]
        ),

    "CleanVerdict":
        pd.to_numeric(
            executions[
                execution_verdict_column
            ],
            errors="coerce",
        ),

    "Duration":
        pd.to_numeric(
            executions[
                execution_duration_column
            ],
            errors="coerce",
        ),
})


raw_history[
    "BuildOrder"
] = raw_history[
    "BuildKey"
].map(
    build_order_map
)

raw_history[
    "Partition"
] = raw_history[
    "BuildKey"
].map(
    partition_map
)


if raw_history[
    "BuildOrder"
].isna().any():
    raise AssertionError(
        "Some raw rows lack a frozen build order."
    )


if raw_history[
    "Partition"
].isna().any():
    raise AssertionError(
        "Some raw rows lack a frozen partition."
    )


if raw_history[
    [
        "CleanVerdict",
        "Duration",
    ]
].isna().any().any():
    raise AssertionError(
        "Raw verdict or duration parsing failed."
    )


raw_history[
    "BuildOrder"
] = pd.to_numeric(
    raw_history[
        "BuildOrder"
    ],
    errors="raise",
).astype(np.int32)

raw_history[
    "CleanVerdict"
] = pd.to_numeric(
    raw_history[
        "CleanVerdict"
    ],
    errors="raise",
).astype(np.int8)

raw_history[
    "Duration"
] = pd.to_numeric(
    raw_history[
        "Duration"
    ],
    errors="raise",
).astype(float)


if not np.isfinite(
    raw_history[
        "Duration"
    ].to_numpy(dtype=float)
).all():
    raise AssertionError(
        "Raw durations contain non-finite values."
    )


if raw_history[
    "Duration"
].lt(0).any():
    raise AssertionError(
        "Raw durations contain negative values."
    )


if raw_history.duplicated(
    subset=[
        "BuildKey",
        "TestKey",
    ],
    keep=False,
).any():
    raise AssertionError(
        "Raw history contains duplicate Build/Test rows."
    )


raw_history = (
    raw_history.sort_values(
        [
            "BuildOrder",
            "JobKey",
            "TestKey",
            "SourceRawRowOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


raw_training_cohort = (
    raw_history[
        raw_history[
            "Partition"
        ].eq(
            "TRAINING"
        )
    ]
    .copy()
    .reset_index(drop=True)
)


raw_training_cohort.insert(
    0,
    "NoiseRowID",
    np.arange(
        len(raw_training_cohort),
        dtype=np.int32,
    ),
)


raw_evaluation_cohort = (
    raw_history[
        raw_history[
            "Partition"
        ].eq(
            "EVALUATION"
        )
    ]
    .copy()
    .reset_index(drop=True)
)


raw_evaluation_cohort.insert(
    0,
    "EvaluationRawRowID",
    np.arange(
        len(raw_evaluation_cohort),
        dtype=np.int32,
    ),
)


raw_training_failures = int(
    raw_training_cohort[
        "CleanVerdict"
    ].ne(0).sum()
)

raw_evaluation_failures = int(
    raw_evaluation_cohort[
        "CleanVerdict"
    ].ne(0).sum()
)


# ------------------------------------------------------------
# 10. FREEZE MODEL-READY TRAINING/EVALUATION COHORTS
# ------------------------------------------------------------

model_cohort = pd.DataFrame({
    "ModelRowOrder":
        np.arange(
            len(dataset),
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            dataset[
                dataset_build_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            dataset[
                dataset_test_column
            ]
        ),

    "CleanVerdict":
        pd.to_numeric(
            dataset[
                dataset_verdict_column
            ],
            errors="coerce",
        ),
})


model_cohort[
    "Partition"
] = model_cohort[
    "BuildKey"
].map(
    partition_map
)


if model_cohort[
    "CleanVerdict"
].isna().any():
    raise AssertionError(
        "Model-ready verdict parsing failed."
    )


if model_cohort[
    "Partition"
].isna().any():
    raise AssertionError(
        "Some model-ready rows lack a partition."
    )


if model_cohort.duplicated(
    subset=[
        "BuildKey",
        "TestKey",
    ],
    keep=False,
).any():
    raise AssertionError(
        "Model-ready cohort contains duplicate Build/Test rows."
    )


model_cohort[
    "CleanVerdict"
] = pd.to_numeric(
    model_cohort[
        "CleanVerdict"
    ],
    errors="raise",
).astype(np.int8)


model_training_cohort = (
    model_cohort[
        model_cohort[
            "Partition"
        ].eq(
            "TRAINING"
        )
    ]
    .copy()
)


model_training_cohort = (
    model_training_cohort.merge(
        raw_training_cohort[
            [
                "NoiseRowID",
                "BuildKey",
                "TestKey",
                "CleanVerdict",
            ]
        ].rename(
            columns={
                "CleanVerdict":
                    "RawCleanVerdict"
            }
        ),
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


missing_model_training_links = int(
    model_training_cohort[
        "_merge"
    ].ne("both").sum()
)


model_training_verdict_mismatches = int(
    model_training_cohort.loc[
        model_training_cohort[
            "_merge"
        ].eq("both"),
        "CleanVerdict",
    ].ne(
        model_training_cohort.loc[
            model_training_cohort[
                "_merge"
            ].eq("both"),
            "RawCleanVerdict",
        ]
    ).sum()
)


model_training_cohort = (
    model_training_cohort.drop(
        columns=[
            "_merge",
        ]
    )
)


model_evaluation_cohort = (
    model_cohort[
        model_cohort[
            "Partition"
        ].eq(
            "EVALUATION"
        )
    ]
    .copy()
)


model_evaluation_cohort = (
    model_evaluation_cohort.merge(
        raw_evaluation_cohort[
            [
                "EvaluationRawRowID",
                "BuildKey",
                "TestKey",
                "CleanVerdict",
            ]
        ].rename(
            columns={
                "CleanVerdict":
                    "RawCleanVerdict"
            }
        ),
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


missing_model_evaluation_links = int(
    model_evaluation_cohort[
        "_merge"
    ].ne("both").sum()
)


model_evaluation_verdict_mismatches = int(
    model_evaluation_cohort.loc[
        model_evaluation_cohort[
            "_merge"
        ].eq("both"),
        "CleanVerdict",
    ].ne(
        model_evaluation_cohort.loc[
            model_evaluation_cohort[
                "_merge"
            ].eq("both"),
            "RawCleanVerdict",
        ]
    ).sum()
)


model_evaluation_cohort = (
    model_evaluation_cohort.drop(
        columns=[
            "_merge",
        ]
    )
)


model_training_failures = int(
    model_training_cohort[
        "CleanVerdict"
    ].ne(0).sum()
)

model_evaluation_failures = int(
    model_evaluation_cohort[
        "CleanVerdict"
    ].ne(0).sum()
)

model_failing_evaluation_builds = int(
    model_evaluation_cohort.loc[
        model_evaluation_cohort[
            "CleanVerdict"
        ].ne(0),
        "BuildKey",
    ].nunique()
)


if missing_model_training_links:
    raise AssertionError(
        "Some model-training rows lack a raw noise-row link."
    )


if missing_model_evaluation_links:
    raise AssertionError(
        "Some model-evaluation rows lack a raw evaluation link."
    )


if model_training_verdict_mismatches:
    raise AssertionError(
        "Model/raw training verdicts differ."
    )


if model_evaluation_verdict_mismatches:
    raise AssertionError(
        "Model/raw evaluation verdicts differ."
    )


# ------------------------------------------------------------
# 11. FAILURE-SUBTYPE PROFILE
# ------------------------------------------------------------

clean_raw_training_verdicts = (
    raw_training_cohort[
        "CleanVerdict"
    ].to_numpy(
        dtype=np.int8
    )
)


failure_counts = (
    pd.Series(
        clean_raw_training_verdicts[
            clean_raw_training_verdicts
            != 0
        ]
    )
    .value_counts()
    .sort_index()
)


if failure_counts.empty:
    raise AssertionError(
        "The raw training cohort has no failure subtype."
    )


failure_subtypes = (
    failure_counts.index.to_numpy(
        dtype=np.int8
    )
)

failure_subtype_probabilities = (
    failure_counts.to_numpy(
        dtype=float
    )
)

failure_subtype_probabilities /= (
    failure_subtype_probabilities.sum()
)


failure_subtype_profile = pd.DataFrame({
    "FailureSubtype":
        failure_subtypes.astype(int),

    "CleanTrainingRows":
        failure_counts.to_numpy(
            dtype=int
        ),

    "Probability":
        failure_subtype_probabilities,
})


# ------------------------------------------------------------
# 12. GENERATE RNG STREAMS AND ALL CONDITIONS
# ------------------------------------------------------------

rng_manifest_frames = []
seed_manifest_records = []
condition_records = []
nested_mask_records = []

nested_mask_violations = 0
condition_order = 0

number_of_noise_rows = len(
    raw_training_cohort
)

model_training_noise_ids = (
    model_training_cohort[
        "NoiseRowID"
    ].to_numpy(
        dtype=np.int32
    )
)

clean_model_training_verdicts = (
    model_training_cohort[
        "CleanVerdict"
    ].to_numpy(
        dtype=np.int8
    )
)


for repetition_seed in REPETITION_SEEDS:
    flip_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "flip_mask",
    )

    failure_subtype_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "failure_subtype",
    )

    flip_rng = np.random.default_rng(
        flip_seed
    )

    subtype_rng = np.random.default_rng(
        failure_subtype_seed
    )

    row_uniforms = flip_rng.random(
        number_of_noise_rows
    )

    sampled_failure_subtypes = (
        subtype_rng.choice(
            failure_subtypes,
            size=number_of_noise_rows,
            replace=True,
            p=failure_subtype_probabilities,
        )
        .astype(np.int8)
    )

    # Independent regeneration check.
    regenerated_uniforms = (
        np.random.default_rng(
            flip_seed
        ).random(
            number_of_noise_rows
        )
    )

    regenerated_subtypes = (
        np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=number_of_noise_rows,
            replace=True,
            p=failure_subtype_probabilities,
        )
        .astype(np.int8)
    )

    uniforms_reproduced = bool(
        np.array_equal(
            row_uniforms,
            regenerated_uniforms,
        )
    )

    subtypes_reproduced = bool(
        np.array_equal(
            sampled_failure_subtypes,
            regenerated_subtypes,
        )
    )

    if not uniforms_reproduced:
        raise AssertionError(
            "Flip-uniform stream did not reproduce."
        )

    if not subtypes_reproduced:
        raise AssertionError(
            "Failure-subtype stream did not reproduce."
        )

    rng_manifest_frames.append(
        pd.DataFrame({
            "RepetitionSeed":
                np.full(
                    number_of_noise_rows,
                    repetition_seed,
                    dtype=np.int16,
                ),

            "NoiseRowID":
                np.arange(
                    number_of_noise_rows,
                    dtype=np.int32,
                ),

            "FlipUniform":
                row_uniforms.astype(
                    np.float64
                ),

            "SampledFailureSubtype":
                sampled_failure_subtypes,
        })
    )

    seed_manifest_records.append({
        "RepetitionSeed":
            repetition_seed,

        "FlipSeed":
            int(
                flip_seed
            ),

        "FailureSubtypeSeed":
            int(
                failure_subtype_seed
            ),

        "NoiseRows":
            number_of_noise_rows,

        "FlipUniformSHA256":
            array_sha256(
                row_uniforms,
                "<f8",
            ),

        "SampledFailureSubtypeSHA256":
            array_sha256(
                sampled_failure_subtypes,
                "<i1",
            ),

        "UniformsReproduced":
            uniforms_reproduced,

        "FailureSubtypesReproduced":
            subtypes_reproduced,
    })

    previous_mask = None
    previous_noise = None

    for noise_index, noise_percent in enumerate(
        NOISE_LEVELS,
        start=1,
    ):
        condition_order += 1

        flip_mask = (
            row_uniforms
            < noise_percent / 100.0
        )

        noisy_verdicts = (
            clean_raw_training_verdicts.copy()
        )

        pass_to_failure_mask = (
            flip_mask
            & (
                clean_raw_training_verdicts
                == 0
            )
        )

        failure_to_pass_mask = (
            flip_mask
            & (
                clean_raw_training_verdicts
                != 0
            )
        )

        noisy_verdicts[
            pass_to_failure_mask
        ] = sampled_failure_subtypes[
            pass_to_failure_mask
        ]

        noisy_verdicts[
            failure_to_pass_mask
        ] = 0

        number_flipped = int(
            flip_mask.sum()
        )

        raw_label_changes = int(
            np.count_nonzero(
                noisy_verdicts
                != clean_raw_training_verdicts
            )
        )

        if raw_label_changes != number_flipped:
            raise AssertionError(
                "A selected flip did not change its verdict."
            )

        noisy_model_verdicts = (
            noisy_verdicts[
                model_training_noise_ids
            ]
        )

        model_label_changes = int(
            np.count_nonzero(
                noisy_model_verdicts
                != clean_model_training_verdicts
            )
        )

        condition_id = (
            f"noise_{noise_percent:02d}"
            f"__seed_{repetition_seed:02d}"
        )

        condition_records.append({
            "ConditionOrder":
                condition_order,

            "ConditionID":
                condition_id,

            "SeedOrder":
                repetition_seed,

            "NoiseOrderWithinSeed":
                noise_index,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "FlipSeed":
                int(
                    flip_seed
                ),

            "FailureSubtypeSeed":
                int(
                    failure_subtype_seed
                ),

            "RawTrainingRows":
                number_of_noise_rows,

            "NumberFlipped":
                number_flipped,

            "RawLabelChanges":
                raw_label_changes,

            "RealisedNoisePercent":
                float(
                    100.0
                    * number_flipped
                    / number_of_noise_rows
                ),

            "PassToFailure":
                int(
                    pass_to_failure_mask.sum()
                ),

            "FailureToPass":
                int(
                    failure_to_pass_mask.sum()
                ),

            "CleanRawFailures":
                raw_training_failures,

            "NoisyRawFailures":
                int(
                    np.count_nonzero(
                        noisy_verdicts != 0
                    )
                ),

            "ModelTrainingRows":
                len(
                    model_training_cohort
                ),

            "ModelLabelChanges":
                model_label_changes,

            "CleanModelFailures":
                model_training_failures,

            "NoisyModelFailures":
                int(
                    np.count_nonzero(
                        noisy_model_verdicts != 0
                    )
                ),

            "FlipMaskSHA256":
                array_sha256(
                    flip_mask.astype(
                        np.uint8
                    ),
                    "<u1",
                ),

            "NoisyRawVerdictSHA256":
                array_sha256(
                    noisy_verdicts,
                    "<i1",
                ),

            "NoisyModelVerdictSHA256":
                array_sha256(
                    noisy_model_verdicts,
                    "<i1",
                ),
        })

        if previous_mask is not None:
            violations = int(
                np.count_nonzero(
                    previous_mask
                    & ~flip_mask
                )
            )

            nested_mask_violations += (
                violations
            )

            nested_mask_records.append({
                "RepetitionSeed":
                    repetition_seed,

                "LowerNoisePercent":
                    previous_noise,

                "HigherNoisePercent":
                    noise_percent,

                "LowerFlipCount":
                    int(
                        previous_mask.sum()
                    ),

                "HigherFlipCount":
                    number_flipped,

                "NestedMaskViolations":
                    violations,

                "Pass":
                    violations == 0,
            })

        previous_mask = flip_mask.copy()
        previous_noise = noise_percent


noise_rng_manifest = pd.concat(
    rng_manifest_frames,
    ignore_index=True,
)

seed_manifest = pd.DataFrame(
    seed_manifest_records
)

condition_plan = pd.DataFrame(
    condition_records
)

nested_mask_audit = pd.DataFrame(
    nested_mask_records
)


# ------------------------------------------------------------
# 13. CONDITION-PLAN AUDITS
# ------------------------------------------------------------

duplicate_condition_ids = int(
    condition_plan.duplicated(
        subset=[
            "ConditionID",
        ],
        keep=False,
    ).sum()
)

duplicate_condition_coordinates = int(
    condition_plan.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


zero_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].eq(0)
]


zero_noise_flip_violations = int(
    zero_noise_conditions[
        "NumberFlipped"
    ].ne(0).sum()
)

zero_noise_raw_change_violations = int(
    zero_noise_conditions[
        "RawLabelChanges"
    ].ne(0).sum()
)

zero_noise_model_change_violations = int(
    zero_noise_conditions[
        "ModelLabelChanges"
    ].ne(0).sum()
)


expected_rng_manifest_rows = (
    len(REPETITION_SEEDS)
    * number_of_noise_rows
)


# ------------------------------------------------------------
# 14. WRITE FROZEN COHORTS AND PLAN
# ------------------------------------------------------------

NOISE_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_parquet(
    RAW_TRAINING_COHORT_PATH,
    raw_training_cohort,
)

atomic_write_parquet(
    RAW_EVALUATION_COHORT_PATH,
    raw_evaluation_cohort,
)

atomic_write_parquet(
    MODEL_TRAINING_COHORT_PATH,
    model_training_cohort,
)

atomic_write_parquet(
    MODEL_EVALUATION_COHORT_PATH,
    model_evaluation_cohort,
)

atomic_write_csv(
    FAILURE_SUBTYPE_PROFILE_PATH,
    failure_subtype_profile,
)

atomic_write_parquet(
    NOISE_RNG_MANIFEST_PATH,
    noise_rng_manifest,
)

atomic_write_csv(
    SEED_MANIFEST_PATH,
    seed_manifest,
)

atomic_write_csv(
    CONDITION_PLAN_PATH,
    condition_plan,
)

atomic_write_csv(
    NESTED_MASK_AUDIT_PATH,
    nested_mask_audit,
)


# ------------------------------------------------------------
# 15. READBACK
# ------------------------------------------------------------

raw_training_readback = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation_readback = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training_readback = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation_readback = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

rng_manifest_readback = pd.read_parquet(
    NOISE_RNG_MANIFEST_PATH
)

condition_plan_readback = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)

nested_mask_readback = pd.read_csv(
    NESTED_MASK_AUDIT_PATH,
    low_memory=False,
)


# ------------------------------------------------------------
# 16. IMMUTABILITY CHECKS
# ------------------------------------------------------------

source_root_after = create_source_root_sha256(
    source_files_payload
)

source_files_unchanged = (
    source_root_before
    == source_root_after
    == EXPECTED_SOURCE_ROOT_SHA256
)


registry_sha256_after = calculate_hash(
    REGISTRY_PATH,
    algorithm="sha256",
)

registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


# ------------------------------------------------------------
# 17. VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check": "Step 1B passed",
        "Expected": EXPECTED_STEP1B_STATUS,
        "Actual": step1b_status.get("Status"),
        "Pass": (
            step1b_status.get("Status")
            == EXPECTED_STEP1B_STATUS
        ),
    },

    {
        "Check": "Step 2A passed",
        "Expected": EXPECTED_STEP2A_STATUS,
        "Actual": step2a_status.get("Status"),
        "Pass": (
            step2a_status.get("Status")
            == EXPECTED_STEP2A_STATUS
        ),
    },

    {
        "Check": "Step 2B passed",
        "Expected": EXPECTED_STEP2B_STATUS,
        "Actual": step2b_status.get("Status"),
        "Pass": (
            step2b_status.get("Status")
            == EXPECTED_STEP2B_STATUS
        ),
    },

    {
        "Check": "Clean anchor reproduced dataset",
        "Expected": True,
        "Actual": rec_checkpoint.get(
            "CleanDatasetReproducedExactly"
        ),
        "Pass": bool(
            rec_checkpoint.get(
                "CleanDatasetReproducedExactly",
                False,
            )
        ),
    },

    {
        "Check": "Raw training rows",
        "Expected": EXPECTED_RAW_TRAINING_ROWS,
        "Actual": len(raw_training_cohort),
        "Pass": (
            len(raw_training_cohort)
            == EXPECTED_RAW_TRAINING_ROWS
        ),
    },

    {
        "Check": "Raw evaluation rows",
        "Expected": EXPECTED_RAW_EVALUATION_ROWS,
        "Actual": len(raw_evaluation_cohort),
        "Pass": (
            len(raw_evaluation_cohort)
            == EXPECTED_RAW_EVALUATION_ROWS
        ),
    },

    {
        "Check": "Raw training failures",
        "Expected": EXPECTED_RAW_TRAINING_FAILURES,
        "Actual": raw_training_failures,
        "Pass": (
            raw_training_failures
            == EXPECTED_RAW_TRAINING_FAILURES
        ),
    },

    {
        "Check": "Raw evaluation failures",
        "Expected": EXPECTED_RAW_EVALUATION_FAILURES,
        "Actual": raw_evaluation_failures,
        "Pass": (
            raw_evaluation_failures
            == EXPECTED_RAW_EVALUATION_FAILURES
        ),
    },

    {
        "Check": "Model training rows",
        "Expected": EXPECTED_MODEL_TRAINING_ROWS,
        "Actual": len(model_training_cohort),
        "Pass": (
            len(model_training_cohort)
            == EXPECTED_MODEL_TRAINING_ROWS
        ),
    },

    {
        "Check": "Model evaluation rows",
        "Expected": EXPECTED_MODEL_EVALUATION_ROWS,
        "Actual": len(model_evaluation_cohort),
        "Pass": (
            len(model_evaluation_cohort)
            == EXPECTED_MODEL_EVALUATION_ROWS
        ),
    },

    {
        "Check": "Model training failures",
        "Expected": EXPECTED_MODEL_TRAINING_FAILURES,
        "Actual": model_training_failures,
        "Pass": (
            model_training_failures
            == EXPECTED_MODEL_TRAINING_FAILURES
        ),
    },

    {
        "Check": "Model evaluation failures",
        "Expected": EXPECTED_MODEL_EVALUATION_FAILURES,
        "Actual": model_evaluation_failures,
        "Pass": (
            model_evaluation_failures
            == EXPECTED_MODEL_EVALUATION_FAILURES
        ),
    },

    {
        "Check": "Model failing evaluation builds",
        "Expected": EXPECTED_MODEL_FAILING_EVALUATION_BUILDS,
        "Actual": model_failing_evaluation_builds,
        "Pass": (
            model_failing_evaluation_builds
            == EXPECTED_MODEL_FAILING_EVALUATION_BUILDS
        ),
    },

    {
        "Check": "Missing model training links",
        "Expected": 0,
        "Actual": missing_model_training_links,
        "Pass": missing_model_training_links == 0,
    },

    {
        "Check": "Missing model evaluation links",
        "Expected": 0,
        "Actual": missing_model_evaluation_links,
        "Pass": missing_model_evaluation_links == 0,
    },

    {
        "Check": "Model training verdict mismatches",
        "Expected": 0,
        "Actual": model_training_verdict_mismatches,
        "Pass": model_training_verdict_mismatches == 0,
    },

    {
        "Check": "Model evaluation verdict mismatches",
        "Expected": 0,
        "Actual": model_evaluation_verdict_mismatches,
        "Pass": model_evaluation_verdict_mismatches == 0,
    },

    {
        "Check": "Noise levels",
        "Expected": NOISE_LEVELS,
        "Actual": sorted(
            condition_plan[
                "NoisePercent"
            ].unique().tolist()
        ),
        "Pass": (
            sorted(
                condition_plan[
                    "NoisePercent"
                ].unique().tolist()
            )
            == NOISE_LEVELS
        ),
    },

    {
        "Check": "Repetition seeds",
        "Expected": REPETITION_SEEDS,
        "Actual": sorted(
            condition_plan[
                "RepetitionSeed"
            ].unique().tolist()
        ),
        "Pass": (
            sorted(
                condition_plan[
                    "RepetitionSeed"
                ].unique().tolist()
            )
            == REPETITION_SEEDS
        ),
    },

    {
        "Check": "Condition rows",
        "Expected": EXPECTED_CONDITIONS,
        "Actual": len(condition_plan),
        "Pass": (
            len(condition_plan)
            == EXPECTED_CONDITIONS
        ),
    },

    {
        "Check": "Duplicate condition IDs",
        "Expected": 0,
        "Actual": duplicate_condition_ids,
        "Pass": duplicate_condition_ids == 0,
    },

    {
        "Check": "Duplicate condition coordinates",
        "Expected": 0,
        "Actual": duplicate_condition_coordinates,
        "Pass": duplicate_condition_coordinates == 0,
    },

    {
        "Check": "Nested-mask violations",
        "Expected": 0,
        "Actual": nested_mask_violations,
        "Pass": nested_mask_violations == 0,
    },

    {
        "Check": "Zero-noise conditions",
        "Expected": 30,
        "Actual": len(zero_noise_conditions),
        "Pass": len(zero_noise_conditions) == 30,
    },

    {
        "Check": "Zero-noise flip violations",
        "Expected": 0,
        "Actual": zero_noise_flip_violations,
        "Pass": zero_noise_flip_violations == 0,
    },

    {
        "Check": "Zero-noise raw-label violations",
        "Expected": 0,
        "Actual": zero_noise_raw_change_violations,
        "Pass": zero_noise_raw_change_violations == 0,
    },

    {
        "Check": "Zero-noise model-label violations",
        "Expected": 0,
        "Actual": zero_noise_model_change_violations,
        "Pass": zero_noise_model_change_violations == 0,
    },

    {
        "Check": "RNG-manifest rows",
        "Expected": expected_rng_manifest_rows,
        "Actual": len(noise_rng_manifest),
        "Pass": (
            len(noise_rng_manifest)
            == expected_rng_manifest_rows
        ),
    },

    {
        "Check": "Seed streams reproduced",
        "Expected": True,
        "Actual": bool(
            seed_manifest[
                [
                    "UniformsReproduced",
                    "FailureSubtypesReproduced",
                ]
            ].all().all()
        ),
        "Pass": bool(
            seed_manifest[
                [
                    "UniformsReproduced",
                    "FailureSubtypesReproduced",
                ]
            ].all().all()
        ),
    },

    {
        "Check": "Raw-training readback rows",
        "Expected": EXPECTED_RAW_TRAINING_ROWS,
        "Actual": len(raw_training_readback),
        "Pass": (
            len(raw_training_readback)
            == EXPECTED_RAW_TRAINING_ROWS
        ),
    },

    {
        "Check": "Raw-evaluation readback rows",
        "Expected": EXPECTED_RAW_EVALUATION_ROWS,
        "Actual": len(raw_evaluation_readback),
        "Pass": (
            len(raw_evaluation_readback)
            == EXPECTED_RAW_EVALUATION_ROWS
        ),
    },

    {
        "Check": "Model-training readback rows",
        "Expected": EXPECTED_MODEL_TRAINING_ROWS,
        "Actual": len(model_training_readback),
        "Pass": (
            len(model_training_readback)
            == EXPECTED_MODEL_TRAINING_ROWS
        ),
    },

    {
        "Check": "Model-evaluation readback rows",
        "Expected": EXPECTED_MODEL_EVALUATION_ROWS,
        "Actual": len(model_evaluation_readback),
        "Pass": (
            len(model_evaluation_readback)
            == EXPECTED_MODEL_EVALUATION_ROWS
        ),
    },

    {
        "Check": "RNG readback rows",
        "Expected": expected_rng_manifest_rows,
        "Actual": len(rng_manifest_readback),
        "Pass": (
            len(rng_manifest_readback)
            == expected_rng_manifest_rows
        ),
    },

    {
        "Check": "Condition-plan readback rows",
        "Expected": EXPECTED_CONDITIONS,
        "Actual": len(condition_plan_readback),
        "Pass": (
            len(condition_plan_readback)
            == EXPECTED_CONDITIONS
        ),
    },

    {
        "Check": "Nested-mask readback violations",
        "Expected": 0,
        "Actual": int(
            nested_mask_readback[
                "NestedMaskViolations"
            ].sum()
        ),
        "Pass": int(
            nested_mask_readback[
                "NestedMaskViolations"
            ].sum()
        ) == 0,
    },

    {
        "Check": "Project 10 source unchanged",
        "Expected": True,
        "Actual": source_files_unchanged,
        "Pass": source_files_unchanged,
    },

    {
        "Check": "Completion registry unchanged",
        "Expected": True,
        "Actual": registry_unchanged,
        "Pass": registry_unchanged,
    },

    {
        "Check": "Registry Project 9 rows",
        "Expected": 0,
        "Actual": int(
            registry_project_numbers.eq(9).sum()
        ),
        "Pass": int(
            registry_project_numbers.eq(9).sum()
        ) == 0,
    },

    {
        "Check": "Registry Project 10 rows",
        "Expected": 0,
        "Actual": int(
            registry_project_numbers.eq(10).sum()
        ),
        "Pass": int(
            registry_project_numbers.eq(10).sum()
        ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)

failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 3A validation:")

display(
    validation
)


if not failed_checks.empty:
    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 10 STEP 3A DID NOT PASS."
    )


atomic_write_csv(
    STEP3A_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 18. PROJECT 9 READ-ONLY END SNAPSHOT
# ------------------------------------------------------------

project_9_end_snapshot = {}


if PROJECT_9_PROGRESS_PATH.exists():
    try:
        project_9_end_snapshot = (
            read_json_with_retry(
                PROJECT_9_PROGRESS_PATH
            )
        )
    except Exception:
        project_9_end_snapshot = {
            "Status":
                "READ_RETRY_EXHAUSTED"
        }


# ------------------------------------------------------------
# 19. REPORT AND CHECKPOINT
# ------------------------------------------------------------

output_hashes = {
    "RawTrainingCohortSHA256":
        calculate_hash(
            RAW_TRAINING_COHORT_PATH
        ),

    "RawEvaluationCohortSHA256":
        calculate_hash(
            RAW_EVALUATION_COHORT_PATH
        ),

    "ModelTrainingCohortSHA256":
        calculate_hash(
            MODEL_TRAINING_COHORT_PATH
        ),

    "ModelEvaluationCohortSHA256":
        calculate_hash(
            MODEL_EVALUATION_COHORT_PATH
        ),

    "FailureSubtypeProfileSHA256":
        calculate_hash(
            FAILURE_SUBTYPE_PROFILE_PATH
        ),

    "NoiseRNGManifestSHA256":
        calculate_hash(
            NOISE_RNG_MANIFEST_PATH
        ),

    "SeedManifestSHA256":
        calculate_hash(
            SEED_MANIFEST_PATH
        ),

    "ConditionPlanSHA256":
        calculate_hash(
            CONDITION_PLAN_PATH
        ),

    "NestedMaskAuditSHA256":
        calculate_hash(
            NESTED_MASK_AUDIT_PATH
        ),

    "ValidationSHA256":
        calculate_hash(
            STEP3A_VALIDATION_PATH
        ),
}


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_PASS_STATUS,

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        len(
            condition_plan
        ),

    "NoiseUnit":
        "individual raw training execution verdict",

    "NoisePartition":
        "training only",

    "EvaluationPartition":
        "clean and immutable",

    "FlipRule": {
        "PassToFailure":
            (
                "0 becomes a project-specific sampled "
                "failure subtype"
            ),

        "FailureToPass":
            "every non-zero verdict becomes 0",
    },

    "RandomStreams": {
        "FlipMask":
            (
                "stable_project_seed(project, seed, "
                "'flip_mask')"
            ),

        "FailureSubtype":
            (
                "stable_project_seed(project, seed, "
                "'failure_subtype')"
            ),
    },

    "NestedMasks":
        True,

    "NestedMaskViolations":
        nested_mask_violations,

    "RawTrainingRows":
        len(
            raw_training_cohort
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation_cohort
        ),

    "RawTrainingFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "ModelTrainingRows":
        len(
            model_training_cohort
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation_cohort
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "FailureSubtypes":
        failure_subtypes.astype(
            int
        ).tolist(),

    "FailureSubtypeCounts":
        failure_counts.astype(
            int
        ).tolist(),

    "FailureSubtypeProbabilities":
        failure_subtype_probabilities.tolist(),

    "RNGManifestRows":
        len(
            noise_rng_manifest
        ),

    "Project9StartReadOnlySnapshot":
        project_9_start_snapshot,

    "Project9EndReadOnlySnapshot":
        project_9_end_snapshot,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletionRegistryModified":
        False,

    "OutputHashes":
        output_hashes,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP3A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "SelectionCheckpointSHA256":
        calculate_hash(
            SELECTION_CHECKPOINT_PATH
        ),

    "RECCheckpointSHA256":
        calculate_hash(
            REC_CHECKPOINT_PATH
        ),

    "SourceRootSHA256":
        source_root_after,

    "RawTrainingCohort":
        str(
            RAW_TRAINING_COHORT_PATH
        ),

    "RawEvaluationCohort":
        str(
            RAW_EVALUATION_COHORT_PATH
        ),

    "ModelTrainingCohort":
        str(
            MODEL_TRAINING_COHORT_PATH
        ),

    "ModelEvaluationCohort":
        str(
            MODEL_EVALUATION_COHORT_PATH
        ),

    "FailureSubtypeProfile":
        str(
            FAILURE_SUBTYPE_PROFILE_PATH
        ),

    "NoiseRNGManifest":
        str(
            NOISE_RNG_MANIFEST_PATH
        ),

    "SeedManifest":
        str(
            SEED_MANIFEST_PATH
        ),

    "ConditionPlan":
        str(
            CONDITION_PLAN_PATH
        ),

    "NestedMaskAudit":
        str(
            NESTED_MASK_AUDIT_PATH
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistryRows":
        len(
            registry
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    NOISE_PLAN_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_PASS_STATUS,

    "Conditions":
        len(
            condition_plan
        ),

    "RawTrainingRows":
        len(
            raw_training_cohort
        ),

    "ModelTrainingRows":
        len(
            model_training_cohort
        ),

    "NestedMaskViolations":
        nested_mask_violations,

    "Checkpoint":
        str(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_hash(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP3A_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 20. FINAL READBACK
# ------------------------------------------------------------

checkpoint_readback = read_json_with_retry(
    NOISE_PLAN_CHECKPOINT_PATH
)

status_readback = read_json_with_retry(
    STEP3A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP3A_PASS_STATUS:
    raise AssertionError(
        "Project 10 noise-plan checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP3A_PASS_STATUS:
    raise AssertionError(
        "Project 10 Step 3A status readback failed."
    )


registry_sha256_final = calculate_hash(
    REGISTRY_PATH
)


if registry_sha256_final != registry_sha256_before:
    raise AssertionError(
        "Completion registry changed during Step 3A."
    )


source_root_final = create_source_root_sha256(
    source_files_payload
)


if source_root_final != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Project 10 source files changed during Step 3A."
    )


# ------------------------------------------------------------
# 21. DISPLAY
# ------------------------------------------------------------

print("\nFailure-subtype profile:")

display(
    failure_subtype_profile
)


print("\nSeed manifest:")

display(
    seed_manifest
)


print("\nCondition-plan sample:")

display(
    pd.concat(
        [
            condition_plan.head(9),
            condition_plan.tail(9),
        ],
        ignore_index=True,
    )
)


print("\nNested-mask audit summary:")

display(
    nested_mask_audit.groupby(
        [
            "LowerNoisePercent",
            "HigherNoisePercent",
        ],
        as_index=False,
    ).agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TotalViolations=(
            "NestedMaskViolations",
            "sum",
        ),

        AllPassed=(
            "Pass",
            "all",
        ),
    )
)


# ------------------------------------------------------------
# 22. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 124)
print("=== PROJECT 10 CELL 5 / STEP 3A RESULT ===")
print("=" * 124)

print("\nProject:")
print(PROJECT_NAME)

print("\nFixed cohorts:")

print(
    "Raw training rows:",
    len(
        raw_training_cohort
    ),
)

print(
    "Raw evaluation rows:",
    len(
        raw_evaluation_cohort
    ),
)

print(
    "Raw training failures:",
    raw_training_failures,
)

print(
    "Raw evaluation failures:",
    raw_evaluation_failures,
)

print(
    "Model training rows:",
    len(
        model_training_cohort
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation_cohort
    ),
)

print(
    "Model training failures:",
    model_training_failures,
)

print(
    "Model evaluation failures:",
    model_evaluation_failures,
)

print(
    "Model failing evaluation builds:",
    model_failing_evaluation_builds,
)


print("\nNoise plan:")

print(
    "Noise levels:",
    NOISE_LEVELS,
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    ),
)

print(
    "Conditions:",
    len(
        condition_plan
    ),
)

print(
    "RNG-manifest rows:",
    len(
        noise_rng_manifest
    ),
)

print(
    "Failure subtypes:",
    failure_subtypes.astype(
        int
    ).tolist(),
)

print(
    "Failure-subtype probabilities:",
    failure_subtype_probabilities.tolist(),
)

print(
    "Nested-mask violations:",
    nested_mask_violations,
)


print("\nZero-noise audit:")

print(
    "Zero-noise conditions:",
    len(
        zero_noise_conditions
    ),
)

print(
    "Zero-noise flip violations:",
    zero_noise_flip_violations,
)

print(
    "Zero-noise model-label violations:",
    zero_noise_model_change_violations,
)


print("\nImmutability:")

print(
    "Project 10 source unchanged:",
    source_root_final
    == EXPECTED_SOURCE_ROOT_SHA256,
)

print(
    "Completion registry unchanged:",
    registry_sha256_final
    == registry_sha256_before,
)

print(
    "Project 9 write attempted:",
    False,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nProject 9 read-only end snapshot:")

print(
    "Status:",
    project_9_end_snapshot.get(
        "Status"
    ),
)

print(
    "Completed conditions:",
    project_9_end_snapshot.get(
        "CompletedConditions"
    ),
)

print(
    "Last completed condition:",
    project_9_end_snapshot.get(
        "LastCompletedCondition"
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nNoise-plan checkpoint:")

print(
    NOISE_PLAN_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_hash(
        NOISE_PLAN_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP3A_PASS_STATUS,
)

print("=" * 124)

=== PROJECT 10 CELL 5 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===

Project 9 read-only start snapshot:
Status: RUNNING_PROJECT_9_FULL_270_CONDITION_EXPERIMENT
Completed conditions: 231
Last completed condition: noise_25__seed_26

Step 3A validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,True
1,Step 2A passed,PASS_PROJECT_10_SOURCE_SCHEMA_AND_JOIN_STRUCTU...,PASS_PROJECT_10_SOURCE_SCHEMA_AND_JOIN_STRUCTU...,True
2,Step 2B passed,PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_A...,PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_A...,True
3,Clean anchor reproduced dataset,True,True,True
4,Raw training rows,34563,34563,True
5,Raw evaluation rows,12531,12531,True
6,Raw training failures,65,65,True
7,Raw evaluation failures,213,213,True
8,Model training rows,6095,6095,True
9,Model evaluation rows,2611,2611,True



Failure-subtype profile:


,FailureSubtype,CleanTrainingRows,Probability
0,1,4,0.061538
1,2,61,0.938462



Seed manifest:


...


Condition-plan sample:


,ConditionOrder,ConditionID,SeedOrder,NoiseOrderWithinSeed,NoisePercent,RepetitionSeed,FlipSeed,FailureSubtypeSeed,RawTrainingRows,NumberFlipped,...,FailureToPass,CleanRawFailures,NoisyRawFailures,ModelTrainingRows,ModelLabelChanges,CleanModelFailures,NoisyModelFailures,FlipMaskSHA256,NoisyRawVerdictSHA256,NoisyModelVerdictSHA256
0,1,noise_00__seed_01,1,1,0,1,1742408094,3204553948,34563,0,...,0,65,65,6095,0,63,63,f57c8610029cdff565c325a10a4b42d456e311e0861824...,dc7c4547e9bcb3d15f9d7b29b34b4860d3b0f2ee4e0dd8...,b767eb4d05c98446c8a1481a3d8e1d4ab1b9e3c958fa2f...
1,2,noise_05__seed_01,1,2,5,1,1742408094,3204553948,34563,1755,...,6,65,1808,6095,300,63,353,164270abd82d88a3c8abd07ecc6507733bf260f16c4287...,655dfa1e653337ba446060ccf01d192c24289d54200e05...,be8b052bd236fd09c6f7d960157223748ff4ab4af1ece0...
2,3,noise_10__seed_01,1,3,10,1,1742408094,3204553948,34563,3498,...,7,65,3549,6095,598,63,649,0960653d3b6891863a6f2ef293830c17016f819c723a02...,bd2401e54e8089ed554a512a8c9cf60945497fc8707ae6...,3a66f8d9f714a9a03b2c1357a11e45950230991ed9cef6...
3,4,noise_15__seed_01,1,4,15,1,1742408094,3204553948,34563,5226,...,13,65,5265,6095,908,63,947,b1b17ffda0ed6e3e8d7086bcf7deeedf7534654898b526...,ba9a782269717438ce6471416d1385181ab54c18e49300...,0c8f3e8d4a3cfe26e992d19f7ac38b68fa4638c98cfcd4...
4,5,noise_20__seed_01,1,5,20,1,1742408094,3204553948,34563,6934,...,14,65,6971,6095,1200,63,1237,c91c3710074c364c8be75e74239b3c2f456561e91ccd0f...,0af8c578d82c503fce205fbfc4523caea55218fcd38ff9...,7bdc5d6b07d249921cfcf0d184b803ff33848038412a2d...
5,6,noise_25__seed_01,1,6,25,1,1742408094,3204553948,34563,8630,...,19,65,8657,6095,1527,63,1554,edd790c83c84551f48bb4fbe7b78009aa1a491097d1a03...,a53abf7c5781058283a01a75aa91ad831a91a83349327e...,bc41d610be452f922fdbf671025d037dd810ed943f8776...
6,7,noise_30__seed_01,1,7,30,1,1742408094,3204553948,34563,10363,...,21,65,10386,6095,1834,63,1857,70a1116ad2a389c106301febb35bbf4caf898edd06ebb1...,22135d27d50ed943725c88bbe726a46777d3a4d11bb91b...,63e65dff991626ac6dab4092c3553a308b6a8cad57931a...
7,8,noise_40__seed_01,1,8,40,1,1742408094,3204553948,34563,13845,...,24,65,13862,6095,2428,63,2445,44433e60d444e94935b4321c7bf97048eff4872f798580...,f6df543a07dd3a5edecee92008a1cf6401d011ce9963d3...,38d1f8f9614703a4c8e9fdeb3b10ff1a9ca88d99398eb9...
8,9,noise_50__seed_01,1,9,50,1,1742408094,3204553948,34563,17388,...,29,65,17395,6095,3041,63,3048,f79e6f2f777cdbdd1f3ccc9aa8944f95e1fb269c727312...,b65ecc551da60e98ab5e441b4d56f75f31157b9400ed84...,ea784702e32f54cfa924b5d023177deaabd81890063be0...
9,262,noise_00__seed_30,30,1,0,30,2272582144,3219974886,34563,0,...,0,65,65,6095,0,63,63,f57c8610029cdff565c325a10a4b42d456e311e0861824...,dc7c4547e9bcb3d15f9d7b29b34b4860d3b0f2ee4e0dd8...,b767eb4d05c98446c8a1481a3d8e1d4ab1b9e3c958fa2f...



Nested-mask audit summary:


,LowerNoisePercent,HigherNoisePercent,Seeds,TotalViolations,AllPassed
0,0,5,30,0,True
1,5,10,30,0,True
2,10,15,30,0,True
3,15,20,30,0,True
4,20,25,30,0,True
5,25,30,30,0,True
6,30,40,30,0,True
7,40,50,30,0,True




=== PROJECT 10 CELL 5 / STEP 3A RESULT ===

Project:
spring-cloud@spring-cloud-dataflow

Fixed cohorts:
Raw training rows: 34563
Raw evaluation rows: 12531
Raw training failures: 65
Raw evaluation failures: 213
Model training rows: 6095
Model evaluation rows: 2611
Model training failures: 63
Model evaluation failures: 213
Model failing evaluation builds: 27

Noise plan:
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Repetition seeds: 30
Conditions: 270
RNG-manifest rows: 1036890
Failure subtypes: [1, 2]
Failure-subtype probabilities: [0.06153846153846154, 0.9384615384615385]
Nested-mask violations: 0

Zero-noise audit:
Zero-noise conditions: 30
Zero-noise flip violations: 0
Zero-noise model-label violations: 0

Immutability:
Project 10 source unchanged: True
Completion registry unchanged: True
Project 9 write attempted: False
Projects 1–8 modified: 0

Project 9 read-only end snapshot:
Status: RUNNING_PROJECT_9_FULL_270_CONDITION_EXPERIMENT
Completed conditions: 231
Last completed c

In [8]:
# ============================================================
# PROJECT 10 — CELL 6 / STEP 3B
# NOISY REC ENGINE SENTINEL VALIDATION AND FREEZE
#
# PROJECT:
#   spring-cloud@spring-cloud-dataflow
#
# This cell:
# - validates the frozen Step 3A noise plan
# - recreates noisy verdicts for six sentinel conditions
# - recomputes the 13 verdict-dependent REC features
# - preserves the six verdict-independent REC features
# - applies the frozen clean-anchor delta:
#
#   Noisy anchored REC
#     = Original clean REC
#       + (Direct noisy REC - Direct clean REC)
#
# - validates 0% semantic identity
# - validates non-zero noise propagation
# - proves clean evaluation cohorts remain unchanged
# - freezes the noisy REC engine checkpoint
#
# This cell does NOT:
# - train models
# - alter evaluation verdicts or features
# - access or modify Project 9
# - modify Projects 1–8
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from IPython.display import display

import hashlib
import json
import os
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)


EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_10_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

STEP3B_PASS_STATUS = (
    "PASS_PROJECT_10_NOISY_REC_ENGINE_SENTINEL_VALIDATED_AND_FROZEN"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e"
)

EXPECTED_RAW_TRAINING_ROWS = 34563
EXPECTED_RAW_EVALUATION_ROWS = 12531
EXPECTED_MODEL_TRAINING_ROWS = 6095
EXPECTED_MODEL_EVALUATION_ROWS = 2611
EXPECTED_CONDITIONS = 270

RECENT_WINDOW = 6


SENTINEL_COORDINATES = [
    # Zero-noise reproducibility across two seeds.
    (0, 1),
    (0, 30),

    # Low and middle noise.
    (5, 15),
    (25, 15),

    # Maximum-noise reproducibility across two seeds.
    (50, 1),
    (50, 30),
]

EXPECTED_SENTINEL_CONDITIONS = len(
    SENTINEL_COORDINATES
)

EXPECTED_SENTINEL_MODEL_ROWS = (
    EXPECTED_SENTINEL_CONDITIONS
    * EXPECTED_MODEL_TRAINING_ROWS
)


REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


if len(REC_FEATURE_COLUMNS) != 19:
    raise AssertionError(
        "Expected 19 REC features."
    )


if len(
    VERDICT_DEPENDENT_REC_FEATURES
) != 13:
    raise AssertionError(
        "Expected 13 verdict-dependent REC features."
    )


if len(
    VERDICT_INDEPENDENT_REC_FEATURES
) != 6:
    raise AssertionError(
        "Expected six verdict-independent REC features."
    )


if (
    set(
        VERDICT_DEPENDENT_REC_FEATURES
    )
    | set(
        VERDICT_INDEPENDENT_REC_FEATURES
    )
) != set(
    REC_FEATURE_COLUMNS
):
    raise AssertionError(
        "REC dependency classes do not cover all features."
    )


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_noise_plan_checkpoint.json"
)

NOISY_REC_ENGINE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_noisy_rec_engine_checkpoint.json"
)


PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)

STEP3B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_status.json"
)


NOISY_REC_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_noisy_rec_preflight"
)

SENTINEL_CONDITION_SUMMARY_PATH = (
    NOISY_REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_sentinel_condition_summary.csv"
)

SENTINEL_FEATURE_CHANGE_PATH = (
    NOISY_REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_sentinel_feature_changes.csv"
)

SENTINEL_MODEL_DATA_PATH = (
    NOISY_REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_sentinel_noisy_model_training.parquet"
)

ZERO_NOISE_SIGNATURE_AUDIT_PATH = (
    NOISY_REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_zero_noise_signature_audit.csv"
)

STEP3B_VALIDATION_PATH = (
    NOISY_REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_validation.csv"
)

STEP3B_REPORT_PATH = (
    NOISY_REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_report.json"
)


# This path is used only to reject accidental overlap.
PROJECT_9_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "camunda__camunda-bpm-platform"
)


print("=" * 124)
print("=== PROJECT 10 CELL 6 / STEP 3B: NOISY REC ENGINE SENTINEL VALIDATION ===")
print("=" * 124)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def array_sha256(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def dataframe_semantic_sha256(
    dataframe,
    columns,
):
    digest = hashlib.sha256()

    working = dataframe[
        columns
    ].copy()

    for column in columns:

        if pd.api.types.is_numeric_dtype(
            working[
                column
            ]
        ):

            values = pd.to_numeric(
                working[
                    column
                ],
                errors="raise",
            ).to_numpy(
                dtype="<f8"
            )

            digest.update(
                column.encode(
                    "utf-8"
                )
            )

            digest.update(
                values.tobytes(
                    order="C"
                )
            )

        else:

            values = (
                working[
                    column
                ]
                .fillna("")
                .astype(str)
            )

            digest.update(
                column.encode(
                    "utf-8"
                )
            )

            for value in values:

                encoded = value.encode(
                    "utf-8"
                )

                digest.update(
                    len(
                        encoded
                    ).to_bytes(
                        8,
                        byteorder="little",
                        signed=False,
                    )
                )

                digest.update(
                    encoded
                )

    return digest.hexdigest()


def json_safe(
    value,
):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(
            value
        ):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="snappy",
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json_with_retry(
    path,
    attempts=10,
    delay_seconds=0.5,
):
    path = Path(
        path
    )

    last_error = None

    for _ in range(
        attempts
    ):

        try:

            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:

            last_error = error

            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON.\n"
        f"Path: {path}\n"
        f"Error: {type(last_error).__name__}: {last_error}"
    )


def canonical_identifier(
    series,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def create_source_root_sha256(
    source_files_payload,
):
    digest = hashlib.sha256()

    for relative_path in sorted(
        source_files_payload
    ):

        metadata = source_files_payload[
            relative_path
        ]

        runtime_path = Path(
            metadata[
                "RuntimePath"
            ]
        )

        size_bytes = int(
            runtime_path.stat().st_size
        )

        file_sha256 = calculate_hash(
            runtime_path,
            algorithm="sha256",
        )

        digest.update(
            (
                f"{relative_path}\0"
                f"{size_bytes}\0"
                f"{file_sha256}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


# ------------------------------------------------------------
# 4. REQUIRED INPUTS
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    STEP2B_STATUS_PATH,
    STEP3A_STATUS_PATH,
]


missing_inputs = [
    str(
        path
    )
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:

    raise FileNotFoundError(
        "Required Project 10 Step 3B inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


selection_checkpoint = read_json_with_retry(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint = read_json_with_retry(
    REC_CHECKPOINT_PATH
)

noise_checkpoint = read_json_with_retry(
    NOISE_PLAN_CHECKPOINT_PATH
)

step1b_status = read_json_with_retry(
    STEP1B_STATUS_PATH
)

step2b_status = read_json_with_retry(
    STEP2B_STATUS_PATH
)

step3a_status = read_json_with_retry(
    STEP3A_STATUS_PATH
)


if step1b_status.get(
    "Status"
) != EXPECTED_STEP1B_STATUS:

    raise AssertionError(
        "Project 10 Step 1B status differs."
    )


if step2b_status.get(
    "Status"
) != EXPECTED_STEP2B_STATUS:

    raise AssertionError(
        "Project 10 Step 2B status differs."
    )


if rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP2B_STATUS:

    raise AssertionError(
        "Project 10 REC checkpoint status differs."
    )


if step3a_status.get(
    "Status"
) != EXPECTED_STEP3A_STATUS:

    raise AssertionError(
        "Project 10 Step 3A status differs."
    )


if noise_checkpoint.get(
    "Status"
) != EXPECTED_STEP3A_STATUS:

    raise AssertionError(
        "Project 10 noise-plan checkpoint status differs."
    )


if selection_checkpoint.get(
    "Project"
) != PROJECT_NAME:

    raise AssertionError(
        "Project identity differs."
    )


if selection_checkpoint.get(
    "ProjectSlug"
) != PROJECT_SLUG:

    raise AssertionError(
        "Project slug differs."
    )


if selection_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:

    raise AssertionError(
        "Frozen source root differs."
    )


if int(
    noise_checkpoint.get(
        "Conditions",
        -1,
    )
) != EXPECTED_CONDITIONS:

    raise AssertionError(
        "Frozen condition count differs."
    )


if not rec_checkpoint.get(
    "CleanDatasetReproducedExactly",
    False,
):

    raise AssertionError(
        "The clean anchor was not validated."
    )


# ------------------------------------------------------------
# 5. OUTPUT-PATH ISOLATION
# ------------------------------------------------------------

output_paths = [
    SENTINEL_CONDITION_SUMMARY_PATH,
    SENTINEL_FEATURE_CHANGE_PATH,
    SENTINEL_MODEL_DATA_PATH,
    ZERO_NOISE_SIGNATURE_AUDIT_PATH,
    STEP3B_VALIDATION_PATH,
    STEP3B_REPORT_PATH,
    NOISY_REC_ENGINE_CHECKPOINT_PATH,
    STEP3B_STATUS_PATH,
]


for output_path in output_paths:

    output_string = str(
        output_path
    )

    if (
        PROJECT_SLUG not in output_string
        and "project_10_" not in output_string
    ):

        raise AssertionError(
            "A Step 3B output path is not Project 10 isolated.\n"
            f"Path: {output_path}"
        )

    if str(
        PROJECT_9_DIR
    ) in output_string:

        raise AssertionError(
            "A Project 10 output overlaps Project 9."
        )


# ------------------------------------------------------------
# 6. REGISTRY AND SOURCE — READ ONLY
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != 8
    or set(
        registry_project_numbers
    ) != set(
        range(
            1,
            9,
        )
    )
):

    raise AssertionError(
        "Registry must contain exactly Projects 1–8."
    )


if registry_project_numbers.eq(
    9
).any():

    raise AssertionError(
        "Project 9 was unexpectedly registered."
    )


if registry_project_numbers.eq(
    10
).any():

    raise AssertionError(
        "Project 10 was unexpectedly registered."
    )


source_files_payload = (
    selection_checkpoint[
        "SourceFiles"
    ]
)

source_root_before = create_source_root_sha256(
    source_files_payload
)


if source_root_before != EXPECTED_SOURCE_ROOT_SHA256:

    raise AssertionError(
        "Project 10 source differs before Step 3B."
    )


# ------------------------------------------------------------
# 7. LOAD FROZEN INPUT ARTIFACTS
# ------------------------------------------------------------

raw_training_path = Path(
    noise_checkpoint[
        "RawTrainingCohort"
    ]
)

raw_evaluation_path = Path(
    noise_checkpoint[
        "RawEvaluationCohort"
    ]
)

model_training_path = Path(
    noise_checkpoint[
        "ModelTrainingCohort"
    ]
)

model_evaluation_path = Path(
    noise_checkpoint[
        "ModelEvaluationCohort"
    ]
)

noise_rng_manifest_path = Path(
    noise_checkpoint[
        "NoiseRNGManifest"
    ]
)

condition_plan_path = Path(
    noise_checkpoint[
        "ConditionPlan"
    ]
)

clean_direct_rec_path = Path(
    rec_checkpoint[
        "CleanRECReconstructed"
    ]
)

clean_anchor_offsets_path = Path(
    rec_checkpoint[
        "CleanRECAnchorOffsets"
    ]
)

build_entity_map_path = Path(
    rec_checkpoint[
        "BuildEntityMap"
    ]
)

dataset_path = Path(
    selection_checkpoint[
        "DatasetPath"
    ]
)


frozen_paths = [
    raw_training_path,
    raw_evaluation_path,
    model_training_path,
    model_evaluation_path,
    noise_rng_manifest_path,
    condition_plan_path,
    clean_direct_rec_path,
    clean_anchor_offsets_path,
    build_entity_map_path,
    dataset_path,
]


missing_frozen_paths = [
    str(
        path
    )
    for path in frozen_paths
    if not path.exists()
]


if missing_frozen_paths:

    raise FileNotFoundError(
        "Frozen Step 3B input files are missing:\n"
        + "\n".join(
            missing_frozen_paths
        )
    )


raw_evaluation_sha256_before = calculate_hash(
    raw_evaluation_path
)

model_evaluation_sha256_before = calculate_hash(
    model_evaluation_path
)


raw_training = pd.read_parquet(
    raw_training_path
)

raw_evaluation = pd.read_parquet(
    raw_evaluation_path
)

model_training = pd.read_parquet(
    model_training_path
)

model_evaluation = pd.read_parquet(
    model_evaluation_path
)

noise_rng_manifest = pd.read_parquet(
    noise_rng_manifest_path
)

condition_plan = pd.read_csv(
    condition_plan_path,
    low_memory=False,
)

clean_direct_rec = pd.read_parquet(
    clean_direct_rec_path
)

clean_anchor_offsets = pd.read_parquet(
    clean_anchor_offsets_path
)

build_entity_map = pd.read_csv(
    build_entity_map_path,
    compression="gzip",
    low_memory=False,
)

dataset = pd.read_csv(
    dataset_path,
    low_memory=False,
)


if len(
    raw_training
) != EXPECTED_RAW_TRAINING_ROWS:

    raise AssertionError(
        "Raw training row count differs."
    )


if len(
    raw_evaluation
) != EXPECTED_RAW_EVALUATION_ROWS:

    raise AssertionError(
        "Raw evaluation row count differs."
    )


if len(
    model_training
) != EXPECTED_MODEL_TRAINING_ROWS:

    raise AssertionError(
        "Model training row count differs."
    )


if len(
    model_evaluation
) != EXPECTED_MODEL_EVALUATION_ROWS:

    raise AssertionError(
        "Model evaluation row count differs."
    )


if len(
    condition_plan
) != EXPECTED_CONDITIONS:

    raise AssertionError(
        "Condition-plan row count differs."
    )


# ------------------------------------------------------------
# 8. CANONICALISE FROZEN COHORTS
# ------------------------------------------------------------

raw_training = raw_training.copy()

raw_training[
    "BuildKey"
] = raw_training[
    "BuildKey"
].astype(str)

raw_training[
    "JobKey"
] = raw_training[
    "JobKey"
].astype(str)

raw_training[
    "TestKey"
] = raw_training[
    "TestKey"
].astype(str)

raw_training[
    "NoiseRowID"
] = pd.to_numeric(
    raw_training[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)

raw_training[
    "BuildOrder"
] = pd.to_numeric(
    raw_training[
        "BuildOrder"
    ],
    errors="raise",
).astype(np.int32)

raw_training[
    "CleanVerdict"
] = pd.to_numeric(
    raw_training[
        "CleanVerdict"
    ],
    errors="raise",
).astype(np.int8)

raw_training[
    "Duration"
] = pd.to_numeric(
    raw_training[
        "Duration"
    ],
    errors="raise",
).astype(float)


raw_training = (
    raw_training.sort_values(
        [
            "BuildOrder",
            "JobKey",
            "TestKey",
            "SourceRawRowOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


expected_noise_ids = np.arange(
    EXPECTED_RAW_TRAINING_ROWS,
    dtype=np.int32,
)


if not np.array_equal(
    raw_training[
        "NoiseRowID"
    ].to_numpy(
        dtype=np.int32
    ),
    expected_noise_ids,
):

    raise AssertionError(
        "Raw training NoiseRowID order differs."
    )


model_training = (
    model_training.copy()
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

model_training[
    "BuildKey"
] = model_training[
    "BuildKey"
].astype(str)

model_training[
    "TestKey"
] = model_training[
    "TestKey"
].astype(str)

model_training[
    "ModelRowOrder"
] = pd.to_numeric(
    model_training[
        "ModelRowOrder"
    ],
    errors="raise",
).astype(np.int64)

model_training[
    "NoiseRowID"
] = pd.to_numeric(
    model_training[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)

model_training[
    "CleanVerdict"
] = pd.to_numeric(
    model_training[
        "CleanVerdict"
    ],
    errors="raise",
).astype(np.int8)


noise_rng_manifest[
    "RepetitionSeed"
] = pd.to_numeric(
    noise_rng_manifest[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(np.int16)

noise_rng_manifest[
    "NoiseRowID"
] = pd.to_numeric(
    noise_rng_manifest[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)

noise_rng_manifest[
    "FlipUniform"
] = pd.to_numeric(
    noise_rng_manifest[
        "FlipUniform"
    ],
    errors="raise",
).astype(float)

noise_rng_manifest[
    "SampledFailureSubtype"
] = pd.to_numeric(
    noise_rng_manifest[
        "SampledFailureSubtype"
    ],
    errors="raise",
).astype(np.int8)


condition_plan[
    "NoisePercent"
] = pd.to_numeric(
    condition_plan[
        "NoisePercent"
    ],
    errors="raise",
).astype(int)

condition_plan[
    "RepetitionSeed"
] = pd.to_numeric(
    condition_plan[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(int)


clean_direct_rec = clean_direct_rec.copy()

clean_direct_rec[
    "BuildKey"
] = clean_direct_rec[
    "BuildKey"
].astype(str)

clean_direct_rec[
    "TestKey"
] = clean_direct_rec[
    "TestKey"
].astype(str)


clean_anchor_offsets = (
    clean_anchor_offsets.copy()
)

clean_anchor_offsets[
    "BuildKey"
] = clean_anchor_offsets[
    "BuildKey"
].astype(str)

clean_anchor_offsets[
    "TestKey"
] = clean_anchor_offsets[
    "TestKey"
].astype(str)


if len(
    clean_direct_rec
) != len(
    dataset
):

    raise AssertionError(
        "Clean direct REC artifact row count differs."
    )


if len(
    clean_anchor_offsets
) != len(
    dataset
):

    raise AssertionError(
        "Clean anchor artifact row count differs."
    )


# ------------------------------------------------------------
# 9. ALIGN CLEAN MODEL DATA AND CLEAN REC ANCHOR
# ------------------------------------------------------------

resolved_columns = (
    rec_checkpoint[
        "ResolvedColumns"
    ]
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)


dataset_keys = pd.DataFrame({
    "BuildKey":
        canonical_identifier(
            dataset[
                dataset_build_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            dataset[
                dataset_test_column
            ]
        ),
})


if not np.array_equal(
    dataset_keys[
        "BuildKey"
    ].astype(str).to_numpy(),
    clean_direct_rec[
        "BuildKey"
    ].astype(str).to_numpy(),
):

    raise AssertionError(
        "Clean direct REC BuildKey order differs from dataset."
    )


if not np.array_equal(
    dataset_keys[
        "TestKey"
    ].astype(str).to_numpy(),
    clean_direct_rec[
        "TestKey"
    ].astype(str).to_numpy(),
):

    raise AssertionError(
        "Clean direct REC TestKey order differs from dataset."
    )


if not np.array_equal(
    dataset_keys[
        "BuildKey"
    ].astype(str).to_numpy(),
    clean_anchor_offsets[
        "BuildKey"
    ].astype(str).to_numpy(),
):

    raise AssertionError(
        "Clean anchor BuildKey order differs from dataset."
    )


if not np.array_equal(
    dataset_keys[
        "TestKey"
    ].astype(str).to_numpy(),
    clean_anchor_offsets[
        "TestKey"
    ].astype(str).to_numpy(),
):

    raise AssertionError(
        "Clean anchor TestKey order differs from dataset."
    )


training_model_orders = model_training[
    "ModelRowOrder"
].to_numpy(
    dtype=np.int64
)


dataset_training_keys = (
    dataset_keys.iloc[
        training_model_orders
    ]
    .reset_index(
        drop=True
    )
)


if not np.array_equal(
    dataset_training_keys[
        "BuildKey"
    ].astype(str).to_numpy(),
    model_training[
        "BuildKey"
    ].astype(str).to_numpy(),
):

    raise AssertionError(
        "Model training BuildKey alignment differs."
    )


if not np.array_equal(
    dataset_training_keys[
        "TestKey"
    ].astype(str).to_numpy(),
    model_training[
        "TestKey"
    ].astype(str).to_numpy(),
):

    raise AssertionError(
        "Model training TestKey alignment differs."
    )


original_clean_training_rec = (
    dataset.loc[
        training_model_orders,
        REC_FEATURE_COLUMNS,
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .reset_index(
        drop=True
    )
)


direct_clean_training_rec = (
    clean_direct_rec.loc[
        training_model_orders,
        REC_FEATURE_COLUMNS,
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .reset_index(
        drop=True
    )
)


anchor_training_rec = (
    clean_anchor_offsets.loc[
        training_model_orders,
        REC_FEATURE_COLUMNS,
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .reset_index(
        drop=True
    )
)


original_clean_matrix = (
    original_clean_training_rec.to_numpy(
        dtype=float
    )
)

direct_clean_matrix = (
    direct_clean_training_rec.to_numpy(
        dtype=float
    )
)

anchor_matrix = (
    anchor_training_rec.to_numpy(
        dtype=float
    )
)


clean_anchor_identity = np.isclose(
    direct_clean_matrix
    + anchor_matrix,
    original_clean_matrix,
    rtol=1e-12,
    atol=1e-12,
    equal_nan=False,
)


clean_anchor_identity_mismatches = int(
    (
        ~clean_anchor_identity
    ).sum()
)


if clean_anchor_identity_mismatches:

    raise AssertionError(
        "Frozen clean anchor does not reproduce "
        "the original training REC matrix."
    )


# ------------------------------------------------------------
# 10. BUILD/ENTITY HISTORY
# ------------------------------------------------------------

build_entity_map = (
    build_entity_map.copy()
)

build_entity_map[
    "BuildKey"
] = build_entity_map[
    "BuildKey"
].astype(str)

build_entity_map[
    "EntityId"
] = canonical_identifier(
    build_entity_map[
        "EntityId"
    ]
)


changed_entities_by_build = {
    str(build_key):
        set(
            group[
                "EntityId"
            ].astype(str)
        )
    for build_key, group
    in build_entity_map.groupby(
        "BuildKey",
        sort=False,
    )
}


entity_changed_builds = defaultdict(
    set
)


for row in build_entity_map.itertuples(
    index=False
):

    entity_changed_builds[
        str(
            row.EntityId
        )
    ].add(
        str(
            row.BuildKey
        )
    )


# ------------------------------------------------------------
# 11. REQUESTED TRAINING ROWS AND GLOBAL BUILD POSITIONS
# ------------------------------------------------------------

requested_rows = model_training[
    [
        "ModelRowOrder",
        "BuildKey",
        "TestKey",
    ]
].copy()


requested_builds_by_test = {
    str(test_key):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for test_key, group
    in requested_rows.groupby(
        "TestKey",
        sort=False,
    )
}


ordered_training_builds = (
    raw_training[
        [
            "BuildKey",
            "BuildOrder",
        ]
    ]
    .drop_duplicates(
        subset=[
            "BuildKey",
        ]
    )
    .sort_values(
        "BuildOrder",
        kind="mergesort",
    )[
        "BuildKey"
    ]
    .astype(str)
    .tolist()
)


global_build_position = {
    build_key:
        position
    for position, build_key
    in enumerate(
        ordered_training_builds
    )
}


# ------------------------------------------------------------
# 12. NOISY REC RECONSTRUCTION HELPERS
# ------------------------------------------------------------

def calculate_rates(
    history,
):
    history_length = len(
        history
    )

    if history_length == 0:

        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history[
        "NoisyVerdict"
    ].to_numpy(
        dtype=np.int32
    )

    transitions = history[
        "Transition"
    ].to_numpy(
        dtype=np.int32
    )

    return (
        float(
            np.count_nonzero(
                verdicts != 0
            )
            / history_length
        ),

        float(
            np.count_nonzero(
                verdicts == 2
            )
            / history_length
        ),

        float(
            np.count_nonzero(
                verdicts == 1
            )
            / history_length
        ),

        float(
            np.count_nonzero(
                transitions != 0
            )
            / history_length
        ),
    )


def calculate_max_test_file_rate(
    history,
    target_type,
    current_changed_entities,
):
    if target_type == "FAILURE":

        target_builds = set(
            history.loc[
                history[
                    "NoisyVerdict"
                ].ne(0),
                "BuildKey",
            ].astype(str)
        )

    elif target_type == "TRANSITION":

        target_builds = set(
            history.loc[
                history[
                    "Transition"
                ].ne(0),
                "BuildKey",
            ].astype(str)
        )

    else:

        raise ValueError(
            "Unknown file-history target type."
        )

    if len(
        target_builds
    ) == 0:

        return -1.0

    maximum_overlap = 0

    for entity_id in current_changed_entities:

        overlap_count = len(
            entity_changed_builds.get(
                str(
                    entity_id
                ),
                set(),
            )
            & target_builds
        )

        if overlap_count > maximum_overlap:

            maximum_overlap = overlap_count

    if maximum_overlap == 0:

        return 0.0

    return float(
        maximum_overlap
        / len(
            target_builds
        )
    )


def reconstruct_direct_noisy_rec(
    noisy_raw_training,
    requested_training_rows,
):
    reconstructed_records = []

    grouped_tests = list(
        noisy_raw_training.groupby(
            "TestKey",
            sort=False,
        )
    )

    for test_key, test_history in (
        grouped_tests
    ):

        test_key = str(
            test_key
        )

        requested_builds = (
            requested_builds_by_test.get(
                test_key
            )
        )

        if not requested_builds:
            continue

        test_history = (
            test_history.sort_values(
                [
                    "BuildOrder",
                    "JobKey",
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
            .copy()
        )

        test_history[
            "Transition"
        ] = (
            test_history[
                "NoisyVerdict"
            ]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(np.int32)
        )

        first_test_build = str(
            test_history[
                "BuildKey"
            ].iloc[0]
        )

        for execution_position in range(
            len(
                test_history
            )
        ):

            current_row = test_history.iloc[
                execution_position
            ]

            current_build = str(
                current_row[
                    "BuildKey"
                ]
            )

            if current_build not in requested_builds:
                continue

            record = {
                "BuildKey":
                    current_build,

                "TestKey":
                    test_key,
            }

            history = test_history.iloc[
                :execution_position
            ]

            if history.empty:

                for feature in REC_FEATURE_COLUMNS:

                    record[
                        feature
                    ] = -1.0

                record[
                    "REC_Age"
                ] = 0.0

                reconstructed_records.append(
                    record
                )

                continue

            recent_history = history.tail(
                RECENT_WINDOW
            )

            age = float(
                global_build_position[
                    current_build
                ]
                - global_build_position[
                    first_test_build
                ]
            )

            failure_positions = np.flatnonzero(
                history[
                    "NoisyVerdict"
                ].to_numpy(
                    dtype=np.int32
                ) != 0
            )

            if len(
                failure_positions
            ) == 0:

                last_failure_age = -1.0

            else:

                last_failure_age = float(
                    len(
                        history
                    )
                    - 1
                    - int(
                        failure_positions[-1]
                    )
                )

            transition_positions = np.flatnonzero(
                history[
                    "Transition"
                ].to_numpy(
                    dtype=np.int32
                ) != 0
            )

            if len(
                transition_positions
            ) == 0:

                last_transition_age = -1.0

            else:

                last_transition_age = float(
                    len(
                        history
                    )
                    - 1
                    - int(
                        transition_positions[-1]
                    )
                )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(
                recent_history
            )

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(
                history
            )

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            record.update({
                "REC_Age":
                    age,

                "REC_LastFailureAge":
                    last_failure_age,

                "REC_LastTransitionAge":
                    last_transition_age,

                "REC_RecentAvgExeTime":
                    float(
                        recent_history[
                            "Duration"
                        ].mean()
                    ),

                "REC_RecentMaxExeTime":
                    float(
                        recent_history[
                            "Duration"
                        ].max()
                    ),

                "REC_RecentFailRate":
                    recent_fail_rate,

                "REC_RecentAssertRate":
                    recent_assert_rate,

                "REC_RecentExcRate":
                    recent_exc_rate,

                "REC_RecentTransitionRate":
                    recent_transition_rate,

                "REC_TotalAvgExeTime":
                    float(
                        history[
                            "Duration"
                        ].mean()
                    ),

                "REC_TotalMaxExeTime":
                    float(
                        history[
                            "Duration"
                        ].max()
                    ),

                "REC_TotalFailRate":
                    total_fail_rate,

                "REC_TotalAssertRate":
                    total_assert_rate,

                "REC_TotalExcRate":
                    total_exc_rate,

                "REC_TotalTransitionRate":
                    total_transition_rate,

                "REC_LastVerdict":
                    float(
                        recent_history[
                            "NoisyVerdict"
                        ].iloc[-1]
                    ),

                "REC_LastExeTime":
                    float(
                        recent_history[
                            "Duration"
                        ].iloc[-1]
                    ),

                "REC_MaxTestFileFailRate":
                    calculate_max_test_file_rate(
                        history=history,
                        target_type="FAILURE",
                        current_changed_entities=(
                            current_changed_entities
                        ),
                    ),

                "REC_MaxTestFileTransitionRate":
                    calculate_max_test_file_rate(
                        history=history,
                        target_type="TRANSITION",
                        current_changed_entities=(
                            current_changed_entities
                        ),
                    ),
            })

            reconstructed_records.append(
                record
            )

    reconstructed = pd.DataFrame(
        reconstructed_records,
        columns=(
            [
                "BuildKey",
                "TestKey",
            ]
            + REC_FEATURE_COLUMNS
        ),
    )

    duplicate_rows = int(
        reconstructed.duplicated(
            subset=[
                "BuildKey",
                "TestKey",
            ],
            keep=False,
        ).sum()
    )

    if duplicate_rows:

        raise AssertionError(
            "Noisy REC reconstruction contains duplicate rows."
        )

    aligned = (
        requested_training_rows.merge(
            reconstructed,
            on=[
                "BuildKey",
                "TestKey",
            ],
            how="left",
            validate="one_to_one",
            indicator=True,
            sort=False,
        )
        .sort_values(
            "ModelRowOrder",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    missing_rows = int(
        aligned[
            "_merge"
        ].ne(
            "both"
        ).sum()
    )

    if missing_rows:

        raise AssertionError(
            "Some model-training rows lack noisy REC values."
        )

    return aligned[
        REC_FEATURE_COLUMNS
    ].apply(
        pd.to_numeric,
        errors="raise",
    )


# ------------------------------------------------------------
# 13. SENTINEL CONDITION VALIDATION
# ------------------------------------------------------------

dependent_positions = [
    REC_FEATURE_COLUMNS.index(
        feature
    )
    for feature in (
        VERDICT_DEPENDENT_REC_FEATURES
    )
]

independent_positions = [
    REC_FEATURE_COLUMNS.index(
        feature
    )
    for feature in (
        VERDICT_INDEPENDENT_REC_FEATURES
    )
]


clean_raw_verdicts = raw_training[
    "CleanVerdict"
].to_numpy(
    dtype=np.int8
)

model_noise_row_ids = model_training[
    "NoiseRowID"
].to_numpy(
    dtype=np.int32
)

clean_model_verdicts = model_training[
    "CleanVerdict"
].to_numpy(
    dtype=np.int8
)


sentinel_summary_records = []
sentinel_feature_records = []
sentinel_model_frames = []

zero_output_signatures = []


for sentinel_index, (
    noise_percent,
    repetition_seed,
) in enumerate(
    SENTINEL_COORDINATES,
    start=1,
):

    condition_rows = condition_plan[
        condition_plan[
            "NoisePercent"
        ].eq(
            noise_percent
        )
        & condition_plan[
            "RepetitionSeed"
        ].eq(
            repetition_seed
        )
    ]


    if len(
        condition_rows
    ) != 1:

        raise AssertionError(
            "A sentinel condition was not found exactly once.\n"
            f"Noise: {noise_percent}\n"
            f"Seed: {repetition_seed}"
        )


    condition_record = condition_rows.iloc[
        0
    ]

    condition_id = str(
        condition_record[
            "ConditionID"
        ]
    )


    seed_rng_rows = (
        noise_rng_manifest[
            noise_rng_manifest[
                "RepetitionSeed"
            ].eq(
                repetition_seed
            )
        ]
        .sort_values(
            "NoiseRowID",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


    if len(
        seed_rng_rows
    ) != EXPECTED_RAW_TRAINING_ROWS:

        raise AssertionError(
            "Seed RNG-manifest row count differs."
        )


    if not np.array_equal(
        seed_rng_rows[
            "NoiseRowID"
        ].to_numpy(
            dtype=np.int32
        ),
        expected_noise_ids,
    ):

        raise AssertionError(
            "Seed RNG NoiseRowID order differs."
        )


    flip_uniforms = seed_rng_rows[
        "FlipUniform"
    ].to_numpy(
        dtype=float
    )

    sampled_failure_subtypes = seed_rng_rows[
        "SampledFailureSubtype"
    ].to_numpy(
        dtype=np.int8
    )


    flip_mask = (
        flip_uniforms
        < noise_percent / 100.0
    )


    noisy_raw_verdicts = (
        clean_raw_verdicts.copy()
    )


    pass_to_failure_mask = (
        flip_mask
        & (
            clean_raw_verdicts
            == 0
        )
    )

    failure_to_pass_mask = (
        flip_mask
        & (
            clean_raw_verdicts
            != 0
        )
    )


    noisy_raw_verdicts[
        pass_to_failure_mask
    ] = sampled_failure_subtypes[
        pass_to_failure_mask
    ]

    noisy_raw_verdicts[
        failure_to_pass_mask
    ] = 0


    noisy_model_verdicts = (
        noisy_raw_verdicts[
            model_noise_row_ids
        ]
    )


    actual_flip_count = int(
        flip_mask.sum()
    )

    actual_raw_label_changes = int(
        np.count_nonzero(
            noisy_raw_verdicts
            != clean_raw_verdicts
        )
    )

    actual_model_label_changes = int(
        np.count_nonzero(
            noisy_model_verdicts
            != clean_model_verdicts
        )
    )


    expected_flip_count = int(
        condition_record[
            "NumberFlipped"
        ]
    )

    expected_raw_label_changes = int(
        condition_record[
            "RawLabelChanges"
        ]
    )

    expected_model_label_changes = int(
        condition_record[
            "ModelLabelChanges"
        ]
    )


    actual_flip_hash = array_sha256(
        flip_mask.astype(
            np.uint8
        ),
        "<u1",
    )

    actual_raw_verdict_hash = array_sha256(
        noisy_raw_verdicts,
        "<i1",
    )

    actual_model_verdict_hash = array_sha256(
        noisy_model_verdicts,
        "<i1",
    )


    expected_flip_hash = str(
        condition_record[
            "FlipMaskSHA256"
        ]
    )

    expected_raw_verdict_hash = str(
        condition_record[
            "NoisyRawVerdictSHA256"
        ]
    )

    expected_model_verdict_hash = str(
        condition_record[
            "NoisyModelVerdictSHA256"
        ]
    )


    if actual_flip_count != expected_flip_count:

        raise AssertionError(
            f"{condition_id}: flip count differs."
        )


    if (
        actual_raw_label_changes
        != expected_raw_label_changes
    ):

        raise AssertionError(
            f"{condition_id}: raw-label change count differs."
        )


    if (
        actual_model_label_changes
        != expected_model_label_changes
    ):

        raise AssertionError(
            f"{condition_id}: model-label change count differs."
        )


    if actual_flip_hash != expected_flip_hash:

        raise AssertionError(
            f"{condition_id}: flip-mask hash differs."
        )


    if (
        actual_raw_verdict_hash
        != expected_raw_verdict_hash
    ):

        raise AssertionError(
            f"{condition_id}: noisy raw-verdict hash differs."
        )


    if (
        actual_model_verdict_hash
        != expected_model_verdict_hash
    ):

        raise AssertionError(
            f"{condition_id}: noisy model-verdict hash differs."
        )


    noisy_raw_frame = raw_training.copy()

    noisy_raw_frame[
        "NoisyVerdict"
    ] = noisy_raw_verdicts


    reconstruction_started = (
        time.perf_counter()
    )


    direct_noisy_rec = (
        reconstruct_direct_noisy_rec(
            noisy_raw_training=(
                noisy_raw_frame
            ),

            requested_training_rows=(
                requested_rows
            ),
        )
    )


    reconstruction_seconds = float(
        time.perf_counter()
        - reconstruction_started
    )


    direct_noisy_matrix = (
        direct_noisy_rec.to_numpy(
            dtype=float
        )
    )


    direct_delta_matrix = (
        direct_noisy_matrix
        - direct_clean_matrix
    )


    anchored_noisy_matrix = (
        original_clean_matrix.copy()
    )


    anchored_noisy_matrix[
        :,
        dependent_positions,
    ] = (
        original_clean_matrix[
            :,
            dependent_positions,
        ]
        + direct_delta_matrix[
            :,
            dependent_positions,
        ]
    )


    # Explicitly preserve all six verdict-independent features.
    anchored_noisy_matrix[
        :,
        independent_positions,
    ] = original_clean_matrix[
        :,
        independent_positions,
    ]


    if not np.isfinite(
        anchored_noisy_matrix
    ).all():

        raise AssertionError(
            f"{condition_id}: anchored noisy REC contains "
            "non-finite values."
        )


    independent_exactly_preserved = bool(
        np.array_equal(
            anchored_noisy_matrix[
                :,
                independent_positions,
            ],
            original_clean_matrix[
                :,
                independent_positions,
            ],
        )
    )


    independent_changed_values = int(
        np.count_nonzero(
            anchored_noisy_matrix[
                :,
                independent_positions,
            ]
            != original_clean_matrix[
                :,
                independent_positions,
            ]
        )
    )


    dependent_matches_clean = np.isclose(
        anchored_noisy_matrix[
            :,
            dependent_positions,
        ],
        original_clean_matrix[
            :,
            dependent_positions,
        ],
        rtol=1e-12,
        atol=1e-12,
        equal_nan=False,
    )


    dependent_changed_values = int(
        (
            ~dependent_matches_clean
        ).sum()
    )


    dependent_changed_rows = int(
        (
            ~dependent_matches_clean
        ).any(
            axis=1
        ).sum()
    )


    direct_clean_matches = np.isclose(
        direct_noisy_matrix,
        direct_clean_matrix,
        rtol=1e-12,
        atol=1e-12,
        equal_nan=False,
    )


    direct_changed_values = int(
        (
            ~direct_clean_matches
        ).sum()
    )


    direct_independent_changed_values = int(
        (
            ~direct_clean_matches[
                :,
                independent_positions,
            ]
        ).sum()
    )


    zero_percent_direct_mismatches = None
    zero_percent_anchored_mismatches = None
    zero_percent_label_mismatches = None


    if noise_percent == 0:

        zero_percent_direct_mismatches = int(
            (
                ~np.isclose(
                    direct_noisy_matrix,
                    direct_clean_matrix,
                    rtol=1e-12,
                    atol=1e-12,
                    equal_nan=False,
                )
            ).sum()
        )


        zero_percent_anchored_mismatches = int(
            (
                ~np.isclose(
                    anchored_noisy_matrix,
                    original_clean_matrix,
                    rtol=1e-12,
                    atol=1e-12,
                    equal_nan=False,
                )
            ).sum()
        )


        zero_percent_label_mismatches = int(
            np.count_nonzero(
                noisy_model_verdicts
                != clean_model_verdicts
            )
        )


        if zero_percent_direct_mismatches:

            raise AssertionError(
                f"{condition_id}: 0% direct REC differs."
            )


        if zero_percent_anchored_mismatches:

            raise AssertionError(
                f"{condition_id}: 0% anchored REC differs."
            )


        if zero_percent_label_mismatches:

            raise AssertionError(
                f"{condition_id}: 0% model labels differ."
            )


    else:

        if actual_raw_label_changes <= 0:

            raise AssertionError(
                f"{condition_id}: non-zero noise changed "
                "no raw labels."
            )


        if actual_model_label_changes <= 0:

            raise AssertionError(
                f"{condition_id}: non-zero noise changed "
                "no model labels."
            )


        if dependent_changed_values <= 0:

            raise AssertionError(
                f"{condition_id}: non-zero noise changed "
                "no dependent REC values."
            )


    if not independent_exactly_preserved:

        raise AssertionError(
            f"{condition_id}: a verdict-independent REC "
            "feature changed."
        )


    if direct_independent_changed_values != 0:

        raise AssertionError(
            f"{condition_id}: direct noisy reconstruction "
            "changed a verdict-independent REC feature."
        )


    noisy_rec_dataframe = pd.DataFrame(
        anchored_noisy_matrix,
        columns=REC_FEATURE_COLUMNS,
    )


    sentinel_model_frame = pd.concat(
        [
            pd.DataFrame({
                "SentinelOrder":
                    np.full(
                        EXPECTED_MODEL_TRAINING_ROWS,
                        sentinel_index,
                        dtype=np.int16,
                    ),

                "ConditionID":
                    np.full(
                        EXPECTED_MODEL_TRAINING_ROWS,
                        condition_id,
                        dtype=object,
                    ),

                "NoisePercent":
                    np.full(
                        EXPECTED_MODEL_TRAINING_ROWS,
                        noise_percent,
                        dtype=np.int16,
                    ),

                "RepetitionSeed":
                    np.full(
                        EXPECTED_MODEL_TRAINING_ROWS,
                        repetition_seed,
                        dtype=np.int16,
                    ),

                "ModelRowOrder":
                    model_training[
                        "ModelRowOrder"
                    ].to_numpy(
                        dtype=np.int64
                    ),

                "BuildKey":
                    model_training[
                        "BuildKey"
                    ].astype(str).to_numpy(),

                "TestKey":
                    model_training[
                        "TestKey"
                    ].astype(str).to_numpy(),

                "CleanVerdict":
                    clean_model_verdicts,

                "NoisyVerdict":
                    noisy_model_verdicts,
            }),

            noisy_rec_dataframe,
        ],
        axis=1,
    )


    output_signature = dataframe_semantic_sha256(
        sentinel_model_frame,
        columns=(
            [
                "BuildKey",
                "TestKey",
                "NoisyVerdict",
            ]
            + REC_FEATURE_COLUMNS
        ),
    )


    if noise_percent == 0:

        zero_output_signatures.append({
            "ConditionID":
                condition_id,

            "RepetitionSeed":
                repetition_seed,

            "SemanticSHA256":
                output_signature,
        })


    sentinel_model_frames.append(
        sentinel_model_frame
    )


    for feature_index, feature in enumerate(
        REC_FEATURE_COLUMNS
    ):

        feature_matches_clean = np.isclose(
            anchored_noisy_matrix[
                :,
                feature_index,
            ],
            original_clean_matrix[
                :,
                feature_index,
            ],
            rtol=1e-12,
            atol=1e-12,
            equal_nan=False,
        )

        sentinel_feature_records.append({
            "SentinelOrder":
                sentinel_index,

            "ConditionID":
                condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "Feature":
                feature,

            "FeatureClass":
                (
                    "VERDICT_DEPENDENT"
                    if feature
                    in VERDICT_DEPENDENT_REC_FEATURES
                    else "VERDICT_INDEPENDENT"
                ),

            "Rows":
                EXPECTED_MODEL_TRAINING_ROWS,

            "ChangedValues":
                int(
                    (
                        ~feature_matches_clean
                    ).sum()
                ),

            "UnchangedValues":
                int(
                    feature_matches_clean.sum()
                ),

            "MaximumAbsoluteChange":
                float(
                    np.max(
                        np.abs(
                            anchored_noisy_matrix[
                                :,
                                feature_index,
                            ]
                            - original_clean_matrix[
                                :,
                                feature_index,
                            ]
                        )
                    )
                ),
        })


    sentinel_summary_records.append({
        "SentinelOrder":
            sentinel_index,

        "ConditionID":
            condition_id,

        "NoisePercent":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "RawTrainingRows":
            EXPECTED_RAW_TRAINING_ROWS,

        "ExpectedFlips":
            expected_flip_count,

        "ActualFlips":
            actual_flip_count,

        "ExpectedRawLabelChanges":
            expected_raw_label_changes,

        "ActualRawLabelChanges":
            actual_raw_label_changes,

        "ExpectedModelLabelChanges":
            expected_model_label_changes,

        "ActualModelLabelChanges":
            actual_model_label_changes,

        "CleanRawFailures":
            int(
                np.count_nonzero(
                    clean_raw_verdicts != 0
                )
            ),

        "NoisyRawFailures":
            int(
                np.count_nonzero(
                    noisy_raw_verdicts != 0
                )
            ),

        "CleanModelFailures":
            int(
                np.count_nonzero(
                    clean_model_verdicts != 0
                )
            ),

        "NoisyModelFailures":
            int(
                np.count_nonzero(
                    noisy_model_verdicts != 0
                )
            ),

        "DirectRECChangedValues":
            direct_changed_values,

        "DirectIndependentRECChangedValues":
            direct_independent_changed_values,

        "AnchoredDependentRECChangedValues":
            dependent_changed_values,

        "AnchoredDependentRECChangedRows":
            dependent_changed_rows,

        "AnchoredIndependentRECChangedValues":
            independent_changed_values,

        "IndependentRECExactlyPreserved":
            independent_exactly_preserved,

        "ZeroPercentDirectMismatches":
            zero_percent_direct_mismatches,

        "ZeroPercentAnchoredMismatches":
            zero_percent_anchored_mismatches,

        "ZeroPercentLabelMismatches":
            zero_percent_label_mismatches,

        "FlipMaskHashMatch":
            actual_flip_hash
            == expected_flip_hash,

        "RawVerdictHashMatch":
            actual_raw_verdict_hash
            == expected_raw_verdict_hash,

        "ModelVerdictHashMatch":
            actual_model_verdict_hash
            == expected_model_verdict_hash,

        "OutputSemanticSHA256":
            output_signature,

        "ReconstructionSeconds":
            reconstruction_seconds,
    })


    print(
        f"[{sentinel_index}/{EXPECTED_SENTINEL_CONDITIONS}]",
        condition_id,
        "| raw flips:",
        actual_flip_count,
        "| model-label changes:",
        actual_model_label_changes,
        "| dependent REC changes:",
        dependent_changed_values,
        "| seconds:",
        round(
            reconstruction_seconds,
            2,
        ),
    )


sentinel_condition_summary = pd.DataFrame(
    sentinel_summary_records
)

sentinel_feature_changes = pd.DataFrame(
    sentinel_feature_records
)

sentinel_model_data = pd.concat(
    sentinel_model_frames,
    ignore_index=True,
)


zero_noise_signature_audit = pd.DataFrame(
    zero_output_signatures
)


# ------------------------------------------------------------
# 14. ZERO-NOISE CROSS-SEED IDENTITY
# ------------------------------------------------------------

zero_noise_signature_count = len(
    zero_noise_signature_audit
)

zero_noise_unique_signatures = int(
    zero_noise_signature_audit[
        "SemanticSHA256"
    ].nunique()
)


zero_noise_outputs_identical = bool(
    zero_noise_signature_count == 2
    and zero_noise_unique_signatures == 1
)


if not zero_noise_outputs_identical:

    raise AssertionError(
        "The two 0% sentinel outputs differ across seeds."
    )


# ------------------------------------------------------------
# 15. SENTINEL AGGREGATE AUDITS
# ------------------------------------------------------------

duplicate_sentinel_conditions = int(
    sentinel_condition_summary.duplicated(
        subset=[
            "ConditionID",
        ],
        keep=False,
    ).sum()
)


all_plan_counts_match = bool(
    sentinel_condition_summary[
        "ExpectedFlips"
    ].eq(
        sentinel_condition_summary[
            "ActualFlips"
        ]
    ).all()

    and sentinel_condition_summary[
        "ExpectedRawLabelChanges"
    ].eq(
        sentinel_condition_summary[
            "ActualRawLabelChanges"
        ]
    ).all()

    and sentinel_condition_summary[
        "ExpectedModelLabelChanges"
    ].eq(
        sentinel_condition_summary[
            "ActualModelLabelChanges"
        ]
    ).all()
)


all_plan_hashes_match = bool(
    sentinel_condition_summary[
        [
            "FlipMaskHashMatch",
            "RawVerdictHashMatch",
            "ModelVerdictHashMatch",
        ]
    ].all().all()
)


all_independent_features_preserved = bool(
    sentinel_condition_summary[
        "IndependentRECExactlyPreserved"
    ].all()

    and sentinel_condition_summary[
        "AnchoredIndependentRECChangedValues"
    ].eq(0).all()

    and sentinel_condition_summary[
        "DirectIndependentRECChangedValues"
    ].eq(0).all()
)


zero_sentinel_rows = sentinel_condition_summary[
    sentinel_condition_summary[
        "NoisePercent"
    ].eq(0)
]


zero_noise_semantic_mismatches = int(
    zero_sentinel_rows[
        [
            "ZeroPercentDirectMismatches",
            "ZeroPercentAnchoredMismatches",
            "ZeroPercentLabelMismatches",
        ]
    ]
    .fillna(0)
    .to_numpy(
        dtype=int
    )
    .sum()
)


nonzero_sentinel_rows = (
    sentinel_condition_summary[
        sentinel_condition_summary[
            "NoisePercent"
        ].gt(0)
    ]
)


nonzero_conditions_with_no_raw_changes = int(
    nonzero_sentinel_rows[
        "ActualRawLabelChanges"
    ].le(0).sum()
)

nonzero_conditions_with_no_model_changes = int(
    nonzero_sentinel_rows[
        "ActualModelLabelChanges"
    ].le(0).sum()
)

nonzero_conditions_with_no_rec_changes = int(
    nonzero_sentinel_rows[
        "AnchoredDependentRECChangedValues"
    ].le(0).sum()
)


# ------------------------------------------------------------
# 16. WRITE SENTINEL OUTPUTS
# ------------------------------------------------------------

NOISY_REC_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    SENTINEL_CONDITION_SUMMARY_PATH,
    sentinel_condition_summary,
)

atomic_write_csv(
    SENTINEL_FEATURE_CHANGE_PATH,
    sentinel_feature_changes,
)

atomic_write_parquet(
    SENTINEL_MODEL_DATA_PATH,
    sentinel_model_data,
)

atomic_write_csv(
    ZERO_NOISE_SIGNATURE_AUDIT_PATH,
    zero_noise_signature_audit,
)


# ------------------------------------------------------------
# 17. READBACK
# ------------------------------------------------------------

sentinel_summary_readback = pd.read_csv(
    SENTINEL_CONDITION_SUMMARY_PATH,
    low_memory=False,
)

sentinel_feature_readback = pd.read_csv(
    SENTINEL_FEATURE_CHANGE_PATH,
    low_memory=False,
)

sentinel_model_readback = pd.read_parquet(
    SENTINEL_MODEL_DATA_PATH
)

zero_signature_readback = pd.read_csv(
    ZERO_NOISE_SIGNATURE_AUDIT_PATH,
    low_memory=False,
)


# ------------------------------------------------------------
# 18. IMMUTABILITY
# ------------------------------------------------------------

raw_evaluation_sha256_after = calculate_hash(
    raw_evaluation_path
)

model_evaluation_sha256_after = calculate_hash(
    model_evaluation_path
)


raw_evaluation_unchanged = bool(
    raw_evaluation_sha256_before
    == raw_evaluation_sha256_after
)

model_evaluation_unchanged = bool(
    model_evaluation_sha256_before
    == model_evaluation_sha256_after
)


source_root_after = create_source_root_sha256(
    source_files_payload
)

source_unchanged = bool(
    source_root_before
    == source_root_after
    == EXPECTED_SOURCE_ROOT_SHA256
)


registry_sha256_after = calculate_hash(
    REGISTRY_PATH
)

registry_unchanged = bool(
    registry_sha256_before
    == registry_sha256_after
)


# ------------------------------------------------------------
# 19. VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 1B passed",

        "Expected":
            EXPECTED_STEP1B_STATUS,

        "Actual":
            step1b_status.get(
                "Status"
            ),

        "Pass":
            step1b_status.get(
                "Status"
            ) == EXPECTED_STEP1B_STATUS,
    },

    {
        "Check":
            "Step 2B passed",

        "Expected":
            EXPECTED_STEP2B_STATUS,

        "Actual":
            step2b_status.get(
                "Status"
            ),

        "Pass":
            step2b_status.get(
                "Status"
            ) == EXPECTED_STEP2B_STATUS,
    },

    {
        "Check":
            "Step 3A passed",

        "Expected":
            EXPECTED_STEP3A_STATUS,

        "Actual":
            step3a_status.get(
                "Status"
            ),

        "Pass":
            step3a_status.get(
                "Status"
            ) == EXPECTED_STEP3A_STATUS,
    },

    {
        "Check":
            "Clean anchor identity mismatches",

        "Expected":
            0,

        "Actual":
            clean_anchor_identity_mismatches,

        "Pass":
            clean_anchor_identity_mismatches == 0,
    },

    {
        "Check":
            "Sentinel conditions",

        "Expected":
            EXPECTED_SENTINEL_CONDITIONS,

        "Actual":
            len(
                sentinel_condition_summary
            ),

        "Pass":
            len(
                sentinel_condition_summary
            ) == EXPECTED_SENTINEL_CONDITIONS,
    },

    {
        "Check":
            "Duplicate sentinel conditions",

        "Expected":
            0,

        "Actual":
            duplicate_sentinel_conditions,

        "Pass":
            duplicate_sentinel_conditions == 0,
    },

    {
        "Check":
            "Sentinel model rows",

        "Expected":
            EXPECTED_SENTINEL_MODEL_ROWS,

        "Actual":
            len(
                sentinel_model_data
            ),

        "Pass":
            len(
                sentinel_model_data
            ) == EXPECTED_SENTINEL_MODEL_ROWS,
    },

    {
        "Check":
            "Feature-audit rows",

        "Expected":
            (
                EXPECTED_SENTINEL_CONDITIONS
                * len(
                    REC_FEATURE_COLUMNS
                )
            ),

        "Actual":
            len(
                sentinel_feature_changes
            ),

        "Pass":
            len(
                sentinel_feature_changes
            )
            == (
                EXPECTED_SENTINEL_CONDITIONS
                * len(
                    REC_FEATURE_COLUMNS
                )
            ),
    },

    {
        "Check":
            "Noise-plan counts match",

        "Expected":
            True,

        "Actual":
            all_plan_counts_match,

        "Pass":
            all_plan_counts_match,
    },

    {
        "Check":
            "Noise-plan hashes match",

        "Expected":
            True,

        "Actual":
            all_plan_hashes_match,

        "Pass":
            all_plan_hashes_match,
    },

    {
        "Check":
            "Zero-noise sentinel rows",

        "Expected":
            2,

        "Actual":
            len(
                zero_sentinel_rows
            ),

        "Pass":
            len(
                zero_sentinel_rows
            ) == 2,
    },

    {
        "Check":
            "Zero-noise semantic mismatches",

        "Expected":
            0,

        "Actual":
            zero_noise_semantic_mismatches,

        "Pass":
            zero_noise_semantic_mismatches == 0,
    },

    {
        "Check":
            "Zero-noise outputs identical across seeds",

        "Expected":
            True,

        "Actual":
            zero_noise_outputs_identical,

        "Pass":
            zero_noise_outputs_identical,
    },

    {
        "Check":
            "Independent REC features preserved",

        "Expected":
            True,

        "Actual":
            all_independent_features_preserved,

        "Pass":
            all_independent_features_preserved,
    },

    {
        "Check":
            "Non-zero conditions without raw changes",

        "Expected":
            0,

        "Actual":
            nonzero_conditions_with_no_raw_changes,

        "Pass":
            nonzero_conditions_with_no_raw_changes == 0,
    },

    {
        "Check":
            "Non-zero conditions without model changes",

        "Expected":
            0,

        "Actual":
            nonzero_conditions_with_no_model_changes,

        "Pass":
            nonzero_conditions_with_no_model_changes == 0,
    },

    {
        "Check":
            "Non-zero conditions without dependent REC changes",

        "Expected":
            0,

        "Actual":
            nonzero_conditions_with_no_rec_changes,

        "Pass":
            nonzero_conditions_with_no_rec_changes == 0,
    },

    {
        "Check":
            "Sentinel-summary readback rows",

        "Expected":
            EXPECTED_SENTINEL_CONDITIONS,

        "Actual":
            len(
                sentinel_summary_readback
            ),

        "Pass":
            len(
                sentinel_summary_readback
            ) == EXPECTED_SENTINEL_CONDITIONS,
    },

    {
        "Check":
            "Sentinel-feature readback rows",

        "Expected":
            (
                EXPECTED_SENTINEL_CONDITIONS
                * len(
                    REC_FEATURE_COLUMNS
                )
            ),

        "Actual":
            len(
                sentinel_feature_readback
            ),

        "Pass":
            len(
                sentinel_feature_readback
            )
            == (
                EXPECTED_SENTINEL_CONDITIONS
                * len(
                    REC_FEATURE_COLUMNS
                )
            ),
    },

    {
        "Check":
            "Sentinel-model readback rows",

        "Expected":
            EXPECTED_SENTINEL_MODEL_ROWS,

        "Actual":
            len(
                sentinel_model_readback
            ),

        "Pass":
            len(
                sentinel_model_readback
            ) == EXPECTED_SENTINEL_MODEL_ROWS,
    },

    {
        "Check":
            "Zero-signature readback rows",

        "Expected":
            2,

        "Actual":
            len(
                zero_signature_readback
            ),

        "Pass":
            len(
                zero_signature_readback
            ) == 2,
    },

    {
        "Check":
            "Raw evaluation cohort unchanged",

        "Expected":
            True,

        "Actual":
            raw_evaluation_unchanged,

        "Pass":
            raw_evaluation_unchanged,
    },

    {
        "Check":
            "Model evaluation cohort unchanged",

        "Expected":
            True,

        "Actual":
            model_evaluation_unchanged,

        "Pass":
            model_evaluation_unchanged,
    },

    {
        "Check":
            "Project 10 source unchanged",

        "Expected":
            True,

        "Actual":
            source_unchanged,

        "Pass":
            source_unchanged,
    },

    {
        "Check":
            "Completion registry unchanged",

        "Expected":
            True,

        "Actual":
            registry_unchanged,

        "Pass":
            registry_unchanged,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    9
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    9
                ).sum()
            ) == 0,
    },

    {
        "Check":
            "Registry Project 10 rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    10
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    10
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 3B validation:")

display(
    validation
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 10 STEP 3B DID NOT PASS."
    )


atomic_write_csv(
    STEP3B_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 20. REPORT AND CHECKPOINT
# ------------------------------------------------------------

output_hashes = {
    "SentinelConditionSummarySHA256":
        calculate_hash(
            SENTINEL_CONDITION_SUMMARY_PATH
        ),

    "SentinelFeatureChangesSHA256":
        calculate_hash(
            SENTINEL_FEATURE_CHANGE_PATH
        ),

    "SentinelModelDataSHA256":
        calculate_hash(
            SENTINEL_MODEL_DATA_PATH
        ),

    "ZeroNoiseSignatureAuditSHA256":
        calculate_hash(
            ZERO_NOISE_SIGNATURE_AUDIT_PATH
        ),

    "ValidationSHA256":
        calculate_hash(
            STEP3B_VALIDATION_PATH
        ),
}


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3B_PASS_STATUS,

    "EngineVersion":
        "PROJECT_10_NOISY_REC_ENGINE_V1",

    "AnchorPolicy":
        (
            "OriginalCleanREC + "
            "(DirectNoisyREC - DirectCleanREC)"
        ),

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_FEATURES,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC_FEATURES,

    "RecentWindow":
        RECENT_WINDOW,

    "SentinelCoordinates":
        [
            {
                "NoisePercent":
                    noise_percent,

                "RepetitionSeed":
                    repetition_seed,
            }
            for (
                noise_percent,
                repetition_seed,
            ) in SENTINEL_COORDINATES
        ],

    "SentinelConditions":
        len(
            sentinel_condition_summary
        ),

    "SentinelModelRows":
        len(
            sentinel_model_data
        ),

    "ZeroNoiseSemanticMismatches":
        zero_noise_semantic_mismatches,

    "ZeroNoiseOutputsIdenticalAcrossSeeds":
        zero_noise_outputs_identical,

    "IndependentRECExactlyPreserved":
        all_independent_features_preserved,

    "NonZeroConditionsWithoutRawChanges":
        nonzero_conditions_with_no_raw_changes,

    "NonZeroConditionsWithoutModelChanges":
        nonzero_conditions_with_no_model_changes,

    "NonZeroConditionsWithoutRECChanges":
        nonzero_conditions_with_no_rec_changes,

    "RawEvaluationSHA256Before":
        raw_evaluation_sha256_before,

    "RawEvaluationSHA256After":
        raw_evaluation_sha256_after,

    "RawEvaluationUnchanged":
        raw_evaluation_unchanged,

    "ModelEvaluationSHA256Before":
        model_evaluation_sha256_before,

    "ModelEvaluationSHA256After":
        model_evaluation_sha256_after,

    "ModelEvaluationUnchanged":
        model_evaluation_unchanged,

    "SourceRootSHA256Before":
        source_root_before,

    "SourceRootSHA256After":
        source_root_after,

    "SourceUnchanged":
        source_unchanged,

    "CompletionRegistrySHA256Before":
        registry_sha256_before,

    "CompletionRegistrySHA256After":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "OutputHashes":
        output_hashes,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP3B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "SelectionCheckpointSHA256":
        calculate_hash(
            SELECTION_CHECKPOINT_PATH
        ),

    "RECCheckpointSHA256":
        calculate_hash(
            REC_CHECKPOINT_PATH
        ),

    "NoisePlanCheckpointSHA256":
        calculate_hash(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "SentinelConditionSummary":
        str(
            SENTINEL_CONDITION_SUMMARY_PATH
        ),

    "SentinelFeatureChanges":
        str(
            SENTINEL_FEATURE_CHANGE_PATH
        ),

    "SentinelModelData":
        str(
            SENTINEL_MODEL_DATA_PATH
        ),

    "ZeroNoiseSignatureAudit":
        str(
            ZERO_NOISE_SIGNATURE_AUDIT_PATH
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistryRows":
        len(
            registry
        ),

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    NOISY_REC_ENGINE_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3B_PASS_STATUS,

    "SentinelConditions":
        len(
            sentinel_condition_summary
        ),

    "SentinelModelRows":
        len(
            sentinel_model_data
        ),

    "ZeroNoiseSemanticMismatches":
        zero_noise_semantic_mismatches,

    "IndependentRECExactlyPreserved":
        all_independent_features_preserved,

    "RawEvaluationUnchanged":
        raw_evaluation_unchanged,

    "ModelEvaluationUnchanged":
        model_evaluation_unchanged,

    "Checkpoint":
        str(
            NOISY_REC_ENGINE_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_hash(
            NOISY_REC_ENGINE_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP3B_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 21. FINAL READBACK
# ------------------------------------------------------------

checkpoint_readback = read_json_with_retry(
    NOISY_REC_ENGINE_CHECKPOINT_PATH
)

status_readback = read_json_with_retry(
    STEP3B_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP3B_PASS_STATUS:

    raise AssertionError(
        "Noisy REC engine checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP3B_PASS_STATUS:

    raise AssertionError(
        "Step 3B status readback failed."
    )


if calculate_hash(
    REGISTRY_PATH
) != registry_sha256_before:

    raise AssertionError(
        "Registry changed during Step 3B."
    )


if create_source_root_sha256(
    source_files_payload
) != EXPECTED_SOURCE_ROOT_SHA256:

    raise AssertionError(
        "Project 10 source changed during Step 3B."
    )


# ------------------------------------------------------------
# 22. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nSentinel-condition summary:")

display(
    sentinel_condition_summary
)


print("\nFeature changes by condition and class:")

display(
    sentinel_feature_changes.groupby(
        [
            "ConditionID",
            "NoisePercent",
            "FeatureClass",
        ],
        as_index=False,
    ).agg(
        ChangedValues=(
            "ChangedValues",
            "sum",
        ),

        MaximumAbsoluteChange=(
            "MaximumAbsoluteChange",
            "max",
        ),
    )
)


print("\nZero-noise signature audit:")

display(
    zero_noise_signature_audit
)


# ------------------------------------------------------------
# 23. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 124)
print("=== PROJECT 10 CELL 6 / STEP 3B RESULT ===")
print("=" * 124)

print("\nProject:")
print(PROJECT_NAME)


print("\nSentinel validation:")

print(
    "Sentinel conditions:",
    len(
        sentinel_condition_summary
    ),
)

print(
    "Sentinel model-training rows:",
    len(
        sentinel_model_data
    ),
)

print(
    "Noise-plan counts matched:",
    all_plan_counts_match,
)

print(
    "Noise-plan hashes matched:",
    all_plan_hashes_match,
)


print("\n0% validation:")

print(
    "Zero-noise sentinel conditions:",
    len(
        zero_sentinel_rows
    ),
)

print(
    "Zero-noise semantic mismatches:",
    zero_noise_semantic_mismatches,
)

print(
    "Zero-noise outputs identical across seeds:",
    zero_noise_outputs_identical,
)


print("\nNoise propagation:")

print(
    "Independent REC features exactly preserved:",
    all_independent_features_preserved,
)

print(
    "Non-zero sentinels without raw-label changes:",
    nonzero_conditions_with_no_raw_changes,
)

print(
    "Non-zero sentinels without model-label changes:",
    nonzero_conditions_with_no_model_changes,
)

print(
    "Non-zero sentinels without dependent REC changes:",
    nonzero_conditions_with_no_rec_changes,
)


print("\nEvaluation immutability:")

print(
    "Raw evaluation cohort unchanged:",
    raw_evaluation_unchanged,
)

print(
    "Model evaluation cohort unchanged:",
    model_evaluation_unchanged,
)


print("\nProject immutability:")

print(
    "Project 10 source unchanged:",
    source_unchanged,
)

print(
    "Completion registry unchanged:",
    registry_unchanged,
)

print(
    "Project 9 accessed:",
    False,
)

print(
    "Project 9 write attempted:",
    False,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nNoisy REC engine checkpoint:")

print(
    NOISY_REC_ENGINE_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_hash(
        NOISY_REC_ENGINE_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP3B_PASS_STATUS,
)

print("=" * 124)

=== PROJECT 10 CELL 6 / STEP 3B: NOISY REC ENGINE SENTINEL VALIDATION ===
[1/6] noise_00__seed_01 | raw flips: 0 | model-label changes: 0 | dependent REC changes: 0 | seconds: 23.89
[2/6] noise_00__seed_30 | raw flips: 0 | model-label changes: 0 | dependent REC changes: 0 | seconds: 14.69
[3/6] noise_05__seed_15 | raw flips: 1725 | model-label changes: 317 | dependent REC changes: 50002 | seconds: 13.34
[4/6] noise_25__seed_15 | raw flips: 8626 | model-label changes: 1516 | dependent REC changes: 65191 | seconds: 13.6
[5/6] noise_50__seed_01 | raw flips: 17388 | model-label changes: 3041 | dependent REC changes: 70556 | seconds: 14.21
[6/6] noise_50__seed_30 | raw flips: 17330 | model-label changes: 3036 | dependent REC changes: 70426 | seconds: 14.0

Step 3B validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,True
1,Step 2B passed,PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_A...,PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_A...,True
2,Step 3A passed,PASS_PROJECT_10_DETERMINISTIC_NOISE_PLAN_AND_C...,PASS_PROJECT_10_DETERMINISTIC_NOISE_PLAN_AND_C...,True
3,Clean anchor identity mismatches,0,0,True
4,Sentinel conditions,6,6,True
5,Duplicate sentinel conditions,0,0,True
6,Sentinel model rows,36570,36570,True
7,Feature-audit rows,114,114,True
8,Noise-plan counts match,True,True,True
9,Noise-plan hashes match,True,True,True



Sentinel-condition summary:


,SentinelOrder,ConditionID,NoisePercent,RepetitionSeed,RawTrainingRows,ExpectedFlips,ActualFlips,ExpectedRawLabelChanges,ActualRawLabelChanges,ExpectedModelLabelChanges,...,AnchoredIndependentRECChangedValues,IndependentRECExactlyPreserved,ZeroPercentDirectMismatches,ZeroPercentAnchoredMismatches,ZeroPercentLabelMismatches,FlipMaskHashMatch,RawVerdictHashMatch,ModelVerdictHashMatch,OutputSemanticSHA256,ReconstructionSeconds
0,1,noise_00__seed_01,0,1,34563,0,0,0,0,0,...,0,True,0.0,0.0,0.0,True,True,True,37ec85e3e188f9310fe9d9a9d79bd71e1313c24bb97fae...,23.889789
1,2,noise_00__seed_30,0,30,34563,0,0,0,0,0,...,0,True,0.0,0.0,0.0,True,True,True,37ec85e3e188f9310fe9d9a9d79bd71e1313c24bb97fae...,14.691910
2,3,noise_05__seed_15,5,15,34563,1725,1725,1725,1725,317,...,0,True,NaN,NaN,NaN,True,True,True,4c0cc4ebfcf9c27eaa7c409553f781dd1544adcb441383...,13.337187
3,4,noise_25__seed_15,25,15,34563,8626,8626,8626,8626,1516,...,0,True,NaN,NaN,NaN,True,True,True,4b015c7c3450e4b5d7dfcc953308919e3f515519aee8d4...,13.595860
4,5,noise_50__seed_01,50,1,34563,17388,17388,17388,17388,3041,...,0,True,NaN,NaN,NaN,True,True,True,966e21dc53ea1945f33074d7c22f96bf85ba31729b047f...,14.206288
5,6,noise_50__seed_30,50,30,34563,17330,17330,17330,17330,3036,...,0,True,NaN,NaN,NaN,True,True,True,603a02e4d0cb0433b9afd0e6b93c5c2705b26804956864...,13.997026



Feature changes by condition and class:


,ConditionID,NoisePercent,FeatureClass,ChangedValues,MaximumAbsoluteChange
0,noise_00__seed_01,0,VERDICT_DEPENDENT,0,0.0
1,noise_00__seed_01,0,VERDICT_INDEPENDENT,0,0.0
2,noise_00__seed_30,0,VERDICT_DEPENDENT,0,0.0
3,noise_00__seed_30,0,VERDICT_INDEPENDENT,0,0.0
4,noise_05__seed_15,5,VERDICT_DEPENDENT,50002,210.0
5,noise_05__seed_15,5,VERDICT_INDEPENDENT,0,0.0
6,noise_25__seed_15,25,VERDICT_DEPENDENT,65191,221.0
7,noise_25__seed_15,25,VERDICT_INDEPENDENT,0,0.0
8,noise_50__seed_01,50,VERDICT_DEPENDENT,70556,224.0
9,noise_50__seed_01,50,VERDICT_INDEPENDENT,0,0.0



Zero-noise signature audit:


,ConditionID,RepetitionSeed,SemanticSHA256
0,noise_00__seed_01,1,37ec85e3e188f9310fe9d9a9d79bd71e1313c24bb97fae...
1,noise_00__seed_30,30,37ec85e3e188f9310fe9d9a9d79bd71e1313c24bb97fae...




=== PROJECT 10 CELL 6 / STEP 3B RESULT ===

Project:
spring-cloud@spring-cloud-dataflow

Sentinel validation:
Sentinel conditions: 6
Sentinel model-training rows: 36570
Noise-plan counts matched: True
Noise-plan hashes matched: True

0% validation:
Zero-noise sentinel conditions: 2
Zero-noise semantic mismatches: 0
Zero-noise outputs identical across seeds: True

Noise propagation:
Independent REC features exactly preserved: True
Non-zero sentinels without raw-label changes: 0
Non-zero sentinels without model-label changes: 0
Non-zero sentinels without dependent REC changes: 0

Evaluation immutability:
Raw evaluation cohort unchanged: True
Model evaluation cohort unchanged: True

Project immutability:
Project 10 source unchanged: True
Completion registry unchanged: True
Project 9 accessed: False
Project 9 write attempted: False
Projects 1–8 modified: 0

Validation:
Checks: 27
Failed checks: 0

Noisy REC engine checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/project_10_nois

In [9]:
# ============================================================
# PROJECT 10 — CELL 7 / STEP 4A
# MODEL, BASELINE, RANKING AND METRIC PROTOCOL FREEZE
#
# PROJECT:
#   spring-cloud@spring-cloud-dataflow
#
# This cell:
# - validates all prior Project 10 checkpoints
# - freezes the 151-predictor model schema
# - removes only clean-training zero-variance predictors
# - freezes current-condition median preprocessing
# - freezes the four ML model configurations
# - freezes deterministic model and Random seeds
# - builds clean model training/evaluation bases
# - validates Random, LatestFail and QTF-Avg
# - freezes score direction and Test-ascending tie rule
# - validates canonical APFD and APFDc
#
# This cell does NOT:
# - fit the four ML models
# - run the 270 conditions
# - access or modify Project 9
# - modify Projects 1–8
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import platform
import time
import warnings

import numpy as np
import pandas as pd

import sklearn
import xgboost
import lightgbm

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)


EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_10_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP3B_STATUS = (
    "PASS_PROJECT_10_NOISY_REC_ENGINE_SENTINEL_VALIDATED_AND_FROZEN"
)

STEP4A_PASS_STATUS = (
    "PASS_PROJECT_10_MODEL_PREDICTOR_BASELINE_AND_METRIC_PROTOCOL_FROZEN"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e"
)

EXPECTED_DATASET_ROWS = 8706
EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTOR_COLUMNS = 151

EXPECTED_RAW_TRAINING_ROWS = 34563
EXPECTED_RAW_EVALUATION_ROWS = 12531

EXPECTED_MODEL_TRAINING_ROWS = 6095
EXPECTED_MODEL_EVALUATION_ROWS = 2611
EXPECTED_MODEL_TRAINING_FAILURES = 63
EXPECTED_MODEL_EVALUATION_FAILURES = 213

EXPECTED_EVALUATION_PERIOD_BUILDS = 102
EXPECTED_SCORED_EVALUATION_BUILDS = 27

EXPECTED_BASELINE_SCENARIOS = 7
EXPECTED_BASELINE_RANKING_ROWS = (
    EXPECTED_BASELINE_SCENARIOS
    * EXPECTED_MODEL_EVALUATION_ROWS
)

EXPECTED_BASELINE_METRIC_ROWS = (
    EXPECTED_BASELINE_SCENARIOS
    * EXPECTED_SCORED_EVALUATION_BUILDS
)

REPETITION_SEEDS = list(
    range(1, 31)
)


ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "criterion": "gini",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "bootstrap": True,
        "class_weight": None,
        "n_jobs": -1,
    },

    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
    },

    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "objective": "binary",
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },

    "NaiveBayes": {
        "variant": "GaussianNB",
        "var_smoothing": 1e-9,
    },
}


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_noise_plan_checkpoint.json"
)

NOISY_REC_ENGINE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_noisy_rec_engine_checkpoint.json"
)

MODEL_PROTOCOL_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_model_protocol_checkpoint.json"
)


PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)

STEP3B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_status.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_status.json"
)


MODEL_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_model_preflight"
)

CLEAN_MODEL_TRAINING_BASE_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_model_training_base.parquet"
)

CLEAN_MODEL_EVALUATION_BASE_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_model_evaluation_base.parquet"
)

PREDICTOR_MANIFEST_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_predictor_manifest.csv"
)

ZERO_VARIANCE_FEATURES_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_zero_variance_features.csv"
)

CLEAN_MEDIAN_REFERENCE_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_training_median_reference.csv"
)

MODEL_CONFIGURATION_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_configuration.json"
)

MODEL_SEED_MANIFEST_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_seed_manifest.csv"
)

RANDOM_SEED_MANIFEST_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_random_baseline_seed_manifest.csv"
)

BASELINE_SENTINEL_RANKINGS_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_baseline_sentinel_rankings.parquet"
)

BASELINE_SENTINEL_METRICS_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_baseline_sentinel_metrics.csv"
)

BASELINE_PROTOCOL_AUDIT_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_baseline_protocol_audit.csv"
)

RANKING_TIE_VALIDATION_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_ranking_tie_validation.csv"
)

METRIC_VALIDATION_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_metric_validation.csv"
)

STEP4A_VALIDATION_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_report.json"
)


PROJECT_9_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "camunda__camunda-bpm-platform"
)


print("=" * 126)
print("=== PROJECT 10 CELL 7 / STEP 4A: MODEL, BASELINE AND METRIC PROTOCOL FREEZE ===")
print("=" * 126)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def array_sha256(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def dataframe_semantic_sha256(
    dataframe,
    columns,
):
    digest = hashlib.sha256()

    working = dataframe[
        columns
    ].copy()

    for column in columns:

        digest.update(
            str(
                column
            ).encode(
                "utf-8"
            )
        )

        series = working[
            column
        ]

        if pd.api.types.is_numeric_dtype(
            series
        ):

            numeric_values = pd.to_numeric(
                series,
                errors="raise",
            ).to_numpy(
                dtype="<f8"
            )

            digest.update(
                numeric_values.tobytes(
                    order="C"
                )
            )

        else:

            text_values = (
                series
                .fillna("")
                .astype(str)
            )

            for value in text_values:

                encoded = value.encode(
                    "utf-8"
                )

                digest.update(
                    len(
                        encoded
                    ).to_bytes(
                        8,
                        byteorder="little",
                        signed=False,
                    )
                )

                digest.update(
                    encoded
                )

    return digest.hexdigest()


def json_safe(
    value,
):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(
            value
        ):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="snappy",
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json_with_retry(
    path,
    attempts=10,
    delay_seconds=0.5,
):
    path = Path(
        path
    )

    last_error = None

    for _ in range(
        attempts
    ):

        try:

            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:

            last_error = error

            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON.\n"
        f"Path: {path}\n"
        f"Error: {type(last_error).__name__}: {last_error}"
    )


def canonical_identifier(
    series,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


def create_source_root_sha256(
    source_files_payload,
):
    digest = hashlib.sha256()

    for relative_path in sorted(
        source_files_payload
    ):

        metadata = source_files_payload[
            relative_path
        ]

        runtime_path = Path(
            metadata[
                "RuntimePath"
            ]
        )

        size_bytes = int(
            runtime_path.stat().st_size
        )

        file_sha256 = calculate_hash(
            runtime_path
        )

        digest.update(
            (
                f"{relative_path}\0"
                f"{size_bytes}\0"
                f"{file_sha256}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def normalise_build_seed_key(
    value,
):
    numeric_value = pd.to_numeric(
        pd.Series(
            [
                value,
            ]
        ),
        errors="coerce",
    ).iloc[0]

    if (
        not pd.isna(
            numeric_value
        )
        and np.isclose(
            numeric_value,
            round(
                numeric_value
            ),
            rtol=0,
            atol=1e-9,
        )
    ):

        return str(
            int(
                round(
                    numeric_value
                )
            )
        )

    return str(
        value
    )


# ------------------------------------------------------------
# 4. CANONICAL APFD AND APFDC
# ------------------------------------------------------------

def calculate_apfd(
    actual_failures,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if number_of_tests == 0:
        return np.nan

    if number_of_failures == 0:
        return np.nan

    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    apfd = (
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )

    return float(
        apfd
    )


def calculate_apfdc(
    actual_failures,
    durations,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(
        failures
    ) != len(
        durations
    ):

        raise ValueError(
            "actual_failures and durations must have "
            "the same length."
        )

    if len(
        failures
    ) == 0:

        return np.nan

    if failures.sum() == 0:

        return np.nan

    if not np.isfinite(
        durations
    ).all():

        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (
        durations < 0
    ).any():

        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:

        return np.nan

    cumulative_before = np.concatenate([
        np.array(
            [
                0.0,
            ]
        ),

        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )

    apfdc = (
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )

    return float(
        apfdc
    )


# ------------------------------------------------------------
# 5. RANKING HELPERS
# ------------------------------------------------------------

def rank_build_rows(
    build_rows,
    scores,
    technique,
    score_direction,
):
    ranked = (
        build_rows[
            [
                "Build",
                "Test",
                "Verdict",
                "Duration",
                "build_order",
                "BuildKey",
                "TestKey",
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    score_values = np.asarray(
        scores,
        dtype=float,
    )

    if len(
        ranked
    ) != len(
        score_values
    ):

        raise ValueError(
            "The score count does not match the build rows."
        )

    if np.isnan(
        score_values
    ).any():

        raise ValueError(
            "Ranking scores contain NaN."
        )

    ranked[
        "Technique"
    ] = technique

    ranked[
        "Score"
    ] = score_values

    ranked[
        "ActualFailure"
    ] = (
        pd.to_numeric(
            ranked[
                "Verdict"
            ],
            errors="raise",
        ).ne(0)
    ).astype(
        np.int8
    )

    ranked[
        "Duration"
    ] = pd.to_numeric(
        ranked[
            "Duration"
        ],
        errors="raise",
    ).astype(float)

    if score_direction == "descending":

        ascending = [
            False,
            True,
        ]

    elif score_direction == "ascending":

        ascending = [
            True,
            True,
        ]

    else:

        raise ValueError(
            "score_direction must be descending or ascending."
        )

    ranked = (
        ranked.sort_values(
            [
                "Score",
                "Test",
            ],
            ascending=ascending,
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    ranked[
        "Rank"
    ] = np.arange(
        1,
        len(
            ranked
        ) + 1,
        dtype=np.int32,
    )

    return ranked


def create_random_rankings(
    evaluation_data,
    repetition_seed,
):
    rankings = []

    for build_key, build_rows in (
        evaluation_data.groupby(
            "BuildKey",
            sort=False,
        )
    ):

        build_seed_key = normalise_build_seed_key(
            build_key
        )

        random_seed = stable_project_seed(
            PROJECT_NAME,
            repetition_seed,
            (
                "Random_baseline_build_"
                f"{build_seed_key}"
            ),
        )

        random_rng = np.random.default_rng(
            random_seed
        )

        random_scores = random_rng.random(
            len(
                build_rows
            )
        )

        rankings.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=random_scores,
                technique="Random",
                score_direction="descending",
            )
        )

    return pd.concat(
        rankings,
        ignore_index=True,
    )


def create_history_baseline_rankings(
    noisy_training_history,
    clean_training_history,
    clean_evaluation_history,
    clean_evaluation_data,
    fixed_split,
):
    latest_failure_order = {}

    noisy_training_sorted = (
        noisy_training_history.sort_values(
            [
                "BuildOrder",
                "JobKey",
                "TestKey",
            ],
            kind="mergesort",
        )
    )

    for row in noisy_training_sorted.itertuples(
        index=False
    ):

        test_key = str(
            row.TestKey
        )

        if int(
            row.NoisyVerdict
        ) != 0:

            latest_failure_order[
                test_key
            ] = int(
                row.BuildOrder
            )

    duration_sum = {}
    duration_count = {}

    clean_training_sorted = (
        clean_training_history.sort_values(
            [
                "BuildOrder",
                "JobKey",
                "TestKey",
            ],
            kind="mergesort",
        )
    )

    for row in clean_training_sorted.itertuples(
        index=False
    ):

        test_key = str(
            row.TestKey
        )

        duration = float(
            row.Duration
        )

        if not np.isfinite(
            duration
        ):

            continue

        duration_sum[
            test_key
        ] = (
            duration_sum.get(
                test_key,
                0.0,
            )
            + duration
        )

        duration_count[
            test_key
        ] = (
            duration_count.get(
                test_key,
                0,
            )
            + 1
        )

    target_build_keys = set(
        clean_evaluation_data[
            "BuildKey"
        ].astype(str)
    )

    raw_eval_by_build = {
        str(
            build_key
        ):
            rows.copy()

        for build_key, rows in (
            clean_evaluation_history.groupby(
                "BuildKey",
                sort=False,
            )
        )
    }

    model_eval_by_build = {
        str(
            build_key
        ):
            rows.copy()

        for build_key, rows in (
            clean_evaluation_data.groupby(
                "BuildKey",
                sort=False,
            )
        )
    }

    latest_rankings = []
    qtf_rankings = []

    ordered_evaluation_builds = (
        fixed_split[
            fixed_split[
                "Partition"
            ].eq(
                "EVALUATION"
            )
        ]
        .sort_values(
            "BuildOrder",
            kind="mergesort",
        )
    )

    for build_row in ordered_evaluation_builds.itertuples(
        index=False
    ):

        build_key = str(
            build_row.BuildKey
        )

        # Rank before seeing the current evaluation build.
        if build_key in target_build_keys:

            build_tests = (
                model_eval_by_build[
                    build_key
                ]
                .copy()
            )

            latest_scores = []
            qtf_scores = []

            for test_key_value in (
                build_tests[
                    "TestKey"
                ].astype(str)
            ):

                latest_scores.append(
                    float(
                        latest_failure_order.get(
                            test_key_value,
                            -1,
                        )
                    )
                )

                count = duration_count.get(
                    test_key_value,
                    0,
                )

                if count > 0:

                    average_duration = (
                        duration_sum[
                            test_key_value
                        ]
                        / count
                    )

                else:

                    average_duration = np.inf

                qtf_scores.append(
                    float(
                        average_duration
                    )
                )

            latest_rankings.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=latest_scores,
                    technique="LatestFail",
                    score_direction="descending",
                )
            )

            qtf_rankings.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=qtf_scores,
                    technique="QTF-Avg",
                    score_direction="ascending",
                )
            )

        # Update both histories only after ranking the build.
        current_raw_rows = raw_eval_by_build.get(
            build_key
        )

        if current_raw_rows is None:

            continue

        current_raw_rows = (
            current_raw_rows.sort_values(
                [
                    "JobKey",
                    "TestKey",
                ],
                kind="mergesort",
            )
        )

        for execution_row in current_raw_rows.itertuples(
            index=False
        ):

            test_key = str(
                execution_row.TestKey
            )

            if int(
                execution_row.CleanVerdict
            ) != 0:

                latest_failure_order[
                    test_key
                ] = int(
                    execution_row.BuildOrder
                )

            duration = float(
                execution_row.Duration
            )

            if np.isfinite(
                duration
            ):

                duration_sum[
                    test_key
                ] = (
                    duration_sum.get(
                        test_key,
                        0.0,
                    )
                    + duration
                )

                duration_count[
                    test_key
                ] = (
                    duration_count.get(
                        test_key,
                        0,
                    )
                    + 1
                )

    if not latest_rankings:

        raise RuntimeError(
            "LatestFail produced no evaluation rankings."
        )

    if not qtf_rankings:

        raise RuntimeError(
            "QTF-Avg produced no evaluation rankings."
        )

    return (
        pd.concat(
            latest_rankings,
            ignore_index=True,
        ),

        pd.concat(
            qtf_rankings,
            ignore_index=True,
        ),
    )


def ranking_semantic_hash(
    ranking,
):
    ordered = (
        ranking.sort_values(
            [
                "BuildKey",
                "Rank",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    return dataframe_semantic_sha256(
        ordered,
        [
            "BuildKey",
            "TestKey",
            "Score",
            "Rank",
        ],
    )


def add_scenario(
    ranking,
    scenario,
):
    result = ranking.copy()

    result.insert(
        0,
        "Scenario",
        scenario,
    )

    return result


# ------------------------------------------------------------
# 6. BUILD-METRIC HELPER
# ------------------------------------------------------------

def calculate_baseline_metrics(
    scenario_rankings,
):
    metric_records = []

    grouped = scenario_rankings.groupby(
        [
            "Scenario",
            "Technique",
            "BuildKey",
        ],
        sort=False,
    )

    for (
        scenario,
        technique,
        build_key,
    ), ranked_build in grouped:

        ranked_build = (
            ranked_build.sort_values(
                "Rank",
                kind="mergesort",
            )
        )

        actual_failures = ranked_build[
            "ActualFailure"
        ].to_numpy(
            dtype=np.int8
        )

        durations = ranked_build[
            "Duration"
        ].to_numpy(
            dtype=float
        )

        metric_records.append({
            "Scenario":
                scenario,

            "Technique":
                technique,

            "BuildKey":
                str(
                    build_key
                ),

            "Build":
                ranked_build[
                    "Build"
                ].iloc[0],

            "BuildOrder":
                int(
                    ranked_build[
                        "build_order"
                    ].iloc[0]
                ),

            "NumberOfTests":
                len(
                    ranked_build
                ),

            "NumberOfFailures":
                int(
                    actual_failures.sum()
                ),

            "TotalDuration":
                float(
                    durations.sum()
                ),

            "APFD":
                calculate_apfd(
                    actual_failures
                ),

            "APFDc":
                calculate_apfdc(
                    actual_failures,
                    durations,
                ),
        })

    return pd.DataFrame(
        metric_records
    )


# ------------------------------------------------------------
# 7. REQUIRED INPUT VALIDATION
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    NOISY_REC_ENGINE_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    STEP2B_STATUS_PATH,
    STEP3A_STATUS_PATH,
    STEP3B_STATUS_PATH,
]


missing_inputs = [
    str(
        path
    )
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:

    raise FileNotFoundError(
        "Required Project 10 Step 4A inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


selection_checkpoint = read_json_with_retry(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint = read_json_with_retry(
    REC_CHECKPOINT_PATH
)

noise_checkpoint = read_json_with_retry(
    NOISE_PLAN_CHECKPOINT_PATH
)

noisy_rec_checkpoint = read_json_with_retry(
    NOISY_REC_ENGINE_CHECKPOINT_PATH
)

step1b_status = read_json_with_retry(
    STEP1B_STATUS_PATH
)

step2b_status = read_json_with_retry(
    STEP2B_STATUS_PATH
)

step3a_status = read_json_with_retry(
    STEP3A_STATUS_PATH
)

step3b_status = read_json_with_retry(
    STEP3B_STATUS_PATH
)


status_expectations = [
    (
        "Step 1B",
        step1b_status.get(
            "Status"
        ),
        EXPECTED_STEP1B_STATUS,
    ),
    (
        "Step 2B",
        step2b_status.get(
            "Status"
        ),
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "Step 3A",
        step3a_status.get(
            "Status"
        ),
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "Step 3B",
        step3b_status.get(
            "Status"
        ),
        EXPECTED_STEP3B_STATUS,
    ),
]


for step_name, actual_status, expected_status in (
    status_expectations
):

    if actual_status != expected_status:

        raise AssertionError(
            f"{step_name} status differs.\n"
            f"Expected: {expected_status}\n"
            f"Actual:   {actual_status}"
        )


if rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP2B_STATUS:

    raise AssertionError(
        "REC checkpoint status differs."
    )


if noise_checkpoint.get(
    "Status"
) != EXPECTED_STEP3A_STATUS:

    raise AssertionError(
        "Noise-plan checkpoint status differs."
    )


if noisy_rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP3B_STATUS:

    raise AssertionError(
        "Noisy REC engine checkpoint status differs."
    )


if selection_checkpoint.get(
    "Project"
) != PROJECT_NAME:

    raise AssertionError(
        "Project identity differs."
    )


if selection_checkpoint.get(
    "ProjectSlug"
) != PROJECT_SLUG:

    raise AssertionError(
        "Project slug differs."
    )


if selection_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:

    raise AssertionError(
        "Frozen source root differs."
    )


# ------------------------------------------------------------
# 8. OUTPUT-PATH ISOLATION
# ------------------------------------------------------------

output_paths = [
    CLEAN_MODEL_TRAINING_BASE_PATH,
    CLEAN_MODEL_EVALUATION_BASE_PATH,
    PREDICTOR_MANIFEST_PATH,
    ZERO_VARIANCE_FEATURES_PATH,
    CLEAN_MEDIAN_REFERENCE_PATH,
    MODEL_CONFIGURATION_PATH,
    MODEL_SEED_MANIFEST_PATH,
    RANDOM_SEED_MANIFEST_PATH,
    BASELINE_SENTINEL_RANKINGS_PATH,
    BASELINE_SENTINEL_METRICS_PATH,
    BASELINE_PROTOCOL_AUDIT_PATH,
    RANKING_TIE_VALIDATION_PATH,
    METRIC_VALIDATION_PATH,
    STEP4A_VALIDATION_PATH,
    STEP4A_REPORT_PATH,
    MODEL_PROTOCOL_CHECKPOINT_PATH,
    STEP4A_STATUS_PATH,
]


for output_path in output_paths:

    output_string = str(
        output_path
    )

    if (
        PROJECT_SLUG not in output_string
        and "project_10_" not in output_string
    ):

        raise AssertionError(
            "A Step 4A output path is not Project 10 isolated.\n"
            f"Path: {output_path}"
        )

    if str(
        PROJECT_9_DIR
    ) in output_string:

        raise AssertionError(
            "A Project 10 output path overlaps Project 9."
        )


# ------------------------------------------------------------
# 9. REGISTRY AND SOURCE — READ ONLY
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != 8
    or set(
        registry_project_numbers
    ) != set(
        range(
            1,
            9,
        )
    )
):

    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )


if registry_project_numbers.eq(
    9
).any():

    raise AssertionError(
        "Project 9 was unexpectedly registered."
    )


if registry_project_numbers.eq(
    10
).any():

    raise AssertionError(
        "Project 10 was unexpectedly registered."
    )


source_files_payload = (
    selection_checkpoint[
        "SourceFiles"
    ]
)

source_root_before = create_source_root_sha256(
    source_files_payload
)


if source_root_before != EXPECTED_SOURCE_ROOT_SHA256:

    raise AssertionError(
        "Project 10 source differs before Step 4A."
    )


# ------------------------------------------------------------
# 10. LOAD FROZEN INPUTS
# ------------------------------------------------------------

dataset_path = Path(
    selection_checkpoint[
        "DatasetPath"
    ]
)

fixed_split_path = Path(
    selection_checkpoint[
        "FixedSplit"
    ]
)

raw_training_path = Path(
    noise_checkpoint[
        "RawTrainingCohort"
    ]
)

raw_evaluation_path = Path(
    noise_checkpoint[
        "RawEvaluationCohort"
    ]
)

model_training_path = Path(
    noise_checkpoint[
        "ModelTrainingCohort"
    ]
)

model_evaluation_path = Path(
    noise_checkpoint[
        "ModelEvaluationCohort"
    ]
)

noise_rng_manifest_path = Path(
    noise_checkpoint[
        "NoiseRNGManifest"
    ]
)

condition_plan_path = Path(
    noise_checkpoint[
        "ConditionPlan"
    ]
)


input_paths = [
    dataset_path,
    fixed_split_path,
    raw_training_path,
    raw_evaluation_path,
    model_training_path,
    model_evaluation_path,
    noise_rng_manifest_path,
    condition_plan_path,
]


missing_frozen_inputs = [
    str(
        path
    )
    for path in input_paths
    if not path.exists()
]


if missing_frozen_inputs:

    raise FileNotFoundError(
        "Frozen Step 4A inputs are missing:\n"
        + "\n".join(
            missing_frozen_inputs
        )
    )


raw_evaluation_file_sha256_before = calculate_hash(
    raw_evaluation_path
)

model_evaluation_file_sha256_before = calculate_hash(
    model_evaluation_path
)


dataset = pd.read_csv(
    dataset_path,
    low_memory=False,
)

fixed_split = pd.read_csv(
    fixed_split_path,
    low_memory=False,
)

raw_training = pd.read_parquet(
    raw_training_path
)

raw_evaluation = pd.read_parquet(
    raw_evaluation_path
)

model_training = pd.read_parquet(
    model_training_path
)

model_evaluation = pd.read_parquet(
    model_evaluation_path
)

noise_rng_manifest = pd.read_parquet(
    noise_rng_manifest_path
)

condition_plan = pd.read_csv(
    condition_plan_path,
    low_memory=False,
)


# ------------------------------------------------------------
# 11. CANONICALISE INPUTS
# ------------------------------------------------------------

resolved_columns = (
    rec_checkpoint[
        "ResolvedColumns"
    ]
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)


if len(
    dataset
) != EXPECTED_DATASET_ROWS:

    raise AssertionError(
        "dataset.csv row count differs."
    )


if len(
    dataset.columns
) != EXPECTED_DATASET_COLUMNS:

    raise AssertionError(
        "dataset.csv column count differs."
    )


fixed_split[
    "BuildKey"
] = fixed_split[
    "BuildKey"
].astype(str)

fixed_split[
    "BuildOrder"
] = pd.to_numeric(
    fixed_split[
        "BuildOrder"
    ],
    errors="raise",
).astype(np.int32)


raw_training = raw_training.copy()

raw_training[
    "BuildKey"
] = raw_training[
    "BuildKey"
].astype(str)

raw_training[
    "JobKey"
] = raw_training[
    "JobKey"
].astype(str)

raw_training[
    "TestKey"
] = raw_training[
    "TestKey"
].astype(str)

raw_training[
    "NoiseRowID"
] = pd.to_numeric(
    raw_training[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)

raw_training[
    "BuildOrder"
] = pd.to_numeric(
    raw_training[
        "BuildOrder"
    ],
    errors="raise",
).astype(np.int32)

raw_training[
    "CleanVerdict"
] = pd.to_numeric(
    raw_training[
        "CleanVerdict"
    ],
    errors="raise",
).astype(np.int8)

raw_training[
    "Duration"
] = pd.to_numeric(
    raw_training[
        "Duration"
    ],
    errors="raise",
).astype(float)


raw_evaluation = raw_evaluation.copy()

raw_evaluation[
    "BuildKey"
] = raw_evaluation[
    "BuildKey"
].astype(str)

raw_evaluation[
    "JobKey"
] = raw_evaluation[
    "JobKey"
].astype(str)

raw_evaluation[
    "TestKey"
] = raw_evaluation[
    "TestKey"
].astype(str)

raw_evaluation[
    "EvaluationRawRowID"
] = pd.to_numeric(
    raw_evaluation[
        "EvaluationRawRowID"
    ],
    errors="raise",
).astype(np.int32)

raw_evaluation[
    "BuildOrder"
] = pd.to_numeric(
    raw_evaluation[
        "BuildOrder"
    ],
    errors="raise",
).astype(np.int32)

raw_evaluation[
    "CleanVerdict"
] = pd.to_numeric(
    raw_evaluation[
        "CleanVerdict"
    ],
    errors="raise",
).astype(np.int8)

raw_evaluation[
    "Duration"
] = pd.to_numeric(
    raw_evaluation[
        "Duration"
    ],
    errors="raise",
).astype(float)


model_training = (
    model_training.copy()
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

model_evaluation = (
    model_evaluation.copy()
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


for model_frame in [
    model_training,
    model_evaluation,
]:

    model_frame[
        "BuildKey"
    ] = model_frame[
        "BuildKey"
    ].astype(str)

    model_frame[
        "TestKey"
    ] = model_frame[
        "TestKey"
    ].astype(str)

    model_frame[
        "ModelRowOrder"
    ] = pd.to_numeric(
        model_frame[
            "ModelRowOrder"
        ],
        errors="raise",
    ).astype(np.int64)

    model_frame[
        "CleanVerdict"
    ] = pd.to_numeric(
        model_frame[
            "CleanVerdict"
        ],
        errors="raise",
    ).astype(np.int8)


model_training[
    "NoiseRowID"
] = pd.to_numeric(
    model_training[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)

model_evaluation[
    "EvaluationRawRowID"
] = pd.to_numeric(
    model_evaluation[
        "EvaluationRawRowID"
    ],
    errors="raise",
).astype(np.int32)


noise_rng_manifest[
    "RepetitionSeed"
] = pd.to_numeric(
    noise_rng_manifest[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(np.int16)

noise_rng_manifest[
    "NoiseRowID"
] = pd.to_numeric(
    noise_rng_manifest[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)

noise_rng_manifest[
    "FlipUniform"
] = pd.to_numeric(
    noise_rng_manifest[
        "FlipUniform"
    ],
    errors="raise",
).astype(float)

noise_rng_manifest[
    "SampledFailureSubtype"
] = pd.to_numeric(
    noise_rng_manifest[
        "SampledFailureSubtype"
    ],
    errors="raise",
).astype(np.int8)


condition_plan[
    "NoisePercent"
] = pd.to_numeric(
    condition_plan[
        "NoisePercent"
    ],
    errors="raise",
).astype(int)

condition_plan[
    "RepetitionSeed"
] = pd.to_numeric(
    condition_plan[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(int)


# ------------------------------------------------------------
# 12. FREEZE MODEL PREDICTOR SCHEMA
# ------------------------------------------------------------

model_identifier_columns = [
    dataset_build_column,
    dataset_test_column,
    dataset_verdict_column,
]


model_predictor_columns = [
    column
    for column in dataset.columns
    if column not in model_identifier_columns
]


if len(
    model_predictor_columns
) != EXPECTED_PREDICTOR_COLUMNS:

    raise AssertionError(
        "Model predictor count differs.\n"
        f"Expected: {EXPECTED_PREDICTOR_COLUMNS}\n"
        f"Actual:   {len(model_predictor_columns)}"
    )


missing_rec_features = [
    feature
    for feature in REC_FEATURE_COLUMNS
    if feature not in model_predictor_columns
]


if missing_rec_features:

    raise AssertionError(
        "Predictor schema is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


# ------------------------------------------------------------
# 13. BUILD CLEAN MODEL TRAINING BASE
# ------------------------------------------------------------

training_model_orders = model_training[
    "ModelRowOrder"
].to_numpy(
    dtype=np.int64
)

training_dataset_rows = (
    dataset.iloc[
        training_model_orders
    ]
    .reset_index(
        drop=True
    )
)


training_dataset_build_keys = canonical_identifier(
    training_dataset_rows[
        dataset_build_column
    ]
).astype(str)

training_dataset_test_keys = canonical_identifier(
    training_dataset_rows[
        dataset_test_column
    ]
).astype(str)


if not np.array_equal(
    training_dataset_build_keys.to_numpy(),
    model_training[
        "BuildKey"
    ].to_numpy(),
):

    raise AssertionError(
        "Training BuildKey alignment differs."
    )


if not np.array_equal(
    training_dataset_test_keys.to_numpy(),
    model_training[
        "TestKey"
    ].to_numpy(),
):

    raise AssertionError(
        "Training TestKey alignment differs."
    )


training_verdicts = pd.to_numeric(
    training_dataset_rows[
        dataset_verdict_column
    ],
    errors="raise",
).astype(np.int8)


training_verdict_mismatches = int(
    training_verdicts.ne(
        model_training[
            "CleanVerdict"
        ]
    ).sum()
)


if training_verdict_mismatches:

    raise AssertionError(
        "Clean model-training verdict alignment differs."
    )


training_metadata = pd.DataFrame({
    "ModelRowOrder":
        model_training[
            "ModelRowOrder"
        ].to_numpy(
            dtype=np.int64
        ),

    "NoiseRowID":
        model_training[
            "NoiseRowID"
        ].to_numpy(
            dtype=np.int32
        ),

    "BuildKey":
        model_training[
            "BuildKey"
        ].astype(str).to_numpy(),

    "TestKey":
        model_training[
            "TestKey"
        ].astype(str).to_numpy(),

    "Build":
        training_dataset_rows[
            dataset_build_column
        ].to_numpy(),

    "Test":
        training_dataset_rows[
            dataset_test_column
        ].to_numpy(),

    "Verdict":
        training_verdicts.to_numpy(
            dtype=np.int8
        ),

    "BinaryFailure":
        training_verdicts.ne(0).astype(
            np.int8
        ).to_numpy(),
})


clean_model_training_base = pd.concat(
    [
        training_metadata,
        training_dataset_rows[
            model_predictor_columns
        ].reset_index(
            drop=True
        ),
    ],
    axis=1,
)


# ------------------------------------------------------------
# 14. BUILD CLEAN MODEL EVALUATION BASE
# ------------------------------------------------------------

evaluation_model_orders = model_evaluation[
    "ModelRowOrder"
].to_numpy(
    dtype=np.int64
)

evaluation_dataset_rows = (
    dataset.iloc[
        evaluation_model_orders
    ]
    .reset_index(
        drop=True
    )
)


evaluation_dataset_build_keys = canonical_identifier(
    evaluation_dataset_rows[
        dataset_build_column
    ]
).astype(str)

evaluation_dataset_test_keys = canonical_identifier(
    evaluation_dataset_rows[
        dataset_test_column
    ]
).astype(str)


if not np.array_equal(
    evaluation_dataset_build_keys.to_numpy(),
    model_evaluation[
        "BuildKey"
    ].to_numpy(),
):

    raise AssertionError(
        "Evaluation BuildKey alignment differs."
    )


if not np.array_equal(
    evaluation_dataset_test_keys.to_numpy(),
    model_evaluation[
        "TestKey"
    ].to_numpy(),
):

    raise AssertionError(
        "Evaluation TestKey alignment differs."
    )


evaluation_verdicts = pd.to_numeric(
    evaluation_dataset_rows[
        dataset_verdict_column
    ],
    errors="raise",
).astype(np.int8)


evaluation_verdict_mismatches = int(
    evaluation_verdicts.ne(
        model_evaluation[
            "CleanVerdict"
        ]
    ).sum()
)


if evaluation_verdict_mismatches:

    raise AssertionError(
        "Clean model-evaluation verdict alignment differs."
    )


evaluation_raw_lookup = raw_evaluation[
    [
        "EvaluationRawRowID",
        "BuildKey",
        "TestKey",
        "JobKey",
        "BuildOrder",
        "CleanVerdict",
        "Duration",
    ]
].copy()


evaluation_link = (
    model_evaluation[
        [
            "ModelRowOrder",
            "EvaluationRawRowID",
            "BuildKey",
            "TestKey",
            "CleanVerdict",
        ]
    ]
    .merge(
        evaluation_raw_lookup,
        on=[
            "EvaluationRawRowID",
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        suffixes=(
            "_Model",
            "_Raw",
        ),
        indicator=True,
        sort=False,
    )
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_evaluation_raw_links = int(
    evaluation_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


evaluation_raw_verdict_mismatches = int(
    evaluation_link.loc[
        evaluation_link[
            "_merge"
        ].eq(
            "both"
        ),
        "CleanVerdict_Model",
    ].ne(
        evaluation_link.loc[
            evaluation_link[
                "_merge"
            ].eq(
                "both"
            ),
            "CleanVerdict_Raw",
        ]
    ).sum()
)


if missing_evaluation_raw_links:

    raise AssertionError(
        "Some model-evaluation rows lack raw duration links."
    )


if evaluation_raw_verdict_mismatches:

    raise AssertionError(
        "Raw/model evaluation verdicts differ."
    )


evaluation_metadata = pd.DataFrame({
    "ModelRowOrder":
        model_evaluation[
            "ModelRowOrder"
        ].to_numpy(
            dtype=np.int64
        ),

    "EvaluationRawRowID":
        model_evaluation[
            "EvaluationRawRowID"
        ].to_numpy(
            dtype=np.int32
        ),

    "BuildKey":
        model_evaluation[
            "BuildKey"
        ].astype(str).to_numpy(),

    "TestKey":
        model_evaluation[
            "TestKey"
        ].astype(str).to_numpy(),

    "Build":
        evaluation_dataset_rows[
            dataset_build_column
        ].to_numpy(),

    "Test":
        evaluation_dataset_rows[
            dataset_test_column
        ].to_numpy(),

    "Verdict":
        evaluation_verdicts.to_numpy(
            dtype=np.int8
        ),

    "BinaryFailure":
        evaluation_verdicts.ne(0).astype(
            np.int8
        ).to_numpy(),

    "Duration":
        evaluation_link[
            "Duration"
        ].to_numpy(
            dtype=float
        ),

    "build_order":
        evaluation_link[
            "BuildOrder"
        ].to_numpy(
            dtype=np.int32
        ),

    "JobKey":
        evaluation_link[
            "JobKey"
        ].astype(str).to_numpy(),
})


clean_model_evaluation_base = pd.concat(
    [
        evaluation_metadata,
        evaluation_dataset_rows[
            model_predictor_columns
        ].reset_index(
            drop=True
        ),
    ],
    axis=1,
)


# ------------------------------------------------------------
# 15. MODEL COHORT VALIDATION
# ------------------------------------------------------------

training_rows = len(
    clean_model_training_base
)

evaluation_rows = len(
    clean_model_evaluation_base
)

training_failures = int(
    clean_model_training_base[
        "BinaryFailure"
    ].sum()
)

evaluation_failures = int(
    clean_model_evaluation_base[
        "BinaryFailure"
    ].sum()
)

scored_evaluation_builds = int(
    clean_model_evaluation_base[
        "BuildKey"
    ].nunique()
)

evaluation_period_builds = int(
    fixed_split.loc[
        fixed_split[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildKey",
    ].nunique()
)


duplicate_training_build_test_rows = int(
    clean_model_training_base.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)

duplicate_evaluation_build_test_rows = int(
    clean_model_evaluation_base.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


evaluation_build_failure_profile = (
    clean_model_evaluation_base.groupby(
        [
            "BuildKey",
            "build_order",
        ],
        as_index=False,
    ).agg(
        Tests=(
            "TestKey",
            "count",
        ),

        Failures=(
            "BinaryFailure",
            "sum",
        ),

        TotalDuration=(
            "Duration",
            "sum",
        ),
    )
)


evaluation_builds_without_failures = int(
    evaluation_build_failure_profile[
        "Failures"
    ].eq(0).sum()
)

evaluation_builds_with_nonpositive_total_duration = int(
    evaluation_build_failure_profile[
        "TotalDuration"
    ].le(0).sum()
)


# ------------------------------------------------------------
# 16. FREEZE ZERO-VARIANCE AND ACTIVE FEATURES
# ------------------------------------------------------------

clean_training_feature_frame = (
    clean_model_training_base[
        model_predictor_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


feature_variation = (
    clean_training_feature_frame.nunique(
        dropna=False
    )
)


zero_variance_features = (
    feature_variation[
        feature_variation <= 1
    ]
    .index
    .tolist()
)


active_feature_columns = [
    feature
    for feature in model_predictor_columns
    if feature not in zero_variance_features
]


if not active_feature_columns:

    raise AssertionError(
        "No active predictors remain."
    )


predictor_manifest_records = []


for predictor_order, predictor in enumerate(
    model_predictor_columns,
    start=1,
):

    if predictor in VERDICT_DEPENDENT_REC_FEATURES:

        predictor_class = (
            "REC_VERDICT_DEPENDENT"
        )

    elif predictor in VERDICT_INDEPENDENT_REC_FEATURES:

        predictor_class = (
            "REC_VERDICT_INDEPENDENT"
        )

    else:

        predictor_class = "NON_REC"

    predictor_manifest_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            predictor,

        "PredictorClass":
            predictor_class,

        "IsREC":
            predictor in REC_FEATURE_COLUMNS,

        "VerdictDependentREC":
            predictor in VERDICT_DEPENDENT_REC_FEATURES,

        "VerdictIndependentREC":
            predictor in VERDICT_INDEPENDENT_REC_FEATURES,

        "CleanTrainingDistinctValues":
            int(
                feature_variation[
                    predictor
                ]
            ),

        "ZeroVariance":
            predictor in zero_variance_features,

        "Active":
            predictor in active_feature_columns,
    })


predictor_manifest = pd.DataFrame(
    predictor_manifest_records
)


zero_variance_features_frame = pd.DataFrame({
    "Feature":
        zero_variance_features,
})


# ------------------------------------------------------------
# 17. TEST FROZEN PREPROCESSING RULE
# ------------------------------------------------------------

X_train_clean = (
    clean_model_training_base[
        active_feature_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


X_evaluation_clean = (
    clean_model_evaluation_base[
        active_feature_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


clean_training_medians = (
    X_train_clean.median(
        axis=0
    )
    .fillna(
        0.0
    )
)


X_train_clean_imputed = (
    X_train_clean
    .fillna(
        clean_training_medians
    )
    .astype(float)
)


X_evaluation_clean_imputed = (
    X_evaluation_clean
    .fillna(
        clean_training_medians
    )
    .astype(float)
)


training_nonfinite_values_after_imputation = int(
    (
        ~np.isfinite(
            X_train_clean_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)

evaluation_nonfinite_values_after_imputation = int(
    (
        ~np.isfinite(
            X_evaluation_clean_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


clean_binary_labels = clean_model_training_base[
    "BinaryFailure"
].astype(
    np.int8
)


clean_training_label_classes = sorted(
    clean_binary_labels.unique().tolist()
)


clean_median_reference = pd.DataFrame({
    "Feature":
        active_feature_columns,

    "CleanTrainingMedianReference":
        [
            float(
                clean_training_medians[
                    feature
                ]
            )
            for feature in active_feature_columns
        ],
})


# ------------------------------------------------------------
# 18. FREEZE MODEL AND RANDOM SEEDS
# ------------------------------------------------------------

model_seed_records = []


for repetition_seed in REPETITION_SEEDS:

    model_seed_records.append({
        "RepetitionSeed":
            repetition_seed,

        "RandomForestSeed":
            stable_project_seed(
                PROJECT_NAME,
                repetition_seed,
                "RandomForest_model",
            ),

        "XGBoostSeed":
            stable_project_seed(
                PROJECT_NAME,
                repetition_seed,
                "XGBoost_model",
            ),

        "LightGBMSeed":
            stable_project_seed(
                PROJECT_NAME,
                repetition_seed,
                "LightGBM_model",
            ),
    })


model_seed_manifest = pd.DataFrame(
    model_seed_records
)


evaluation_build_keys = (
    clean_model_evaluation_base[
        [
            "BuildKey",
            "build_order",
        ]
    ]
    .drop_duplicates(
        subset=[
            "BuildKey",
        ]
    )
    .sort_values(
        "build_order",
        kind="mergesort",
    )
)


random_seed_records = []


for repetition_seed in REPETITION_SEEDS:

    for build_row in evaluation_build_keys.itertuples(
        index=False
    ):

        build_key = str(
            build_row.BuildKey
        )

        random_seed_records.append({
            "RepetitionSeed":
                repetition_seed,

            "BuildKey":
                build_key,

            "BuildOrder":
                int(
                    build_row.build_order
                ),

            "RandomSeed":
                stable_project_seed(
                    PROJECT_NAME,
                    repetition_seed,
                    (
                        "Random_baseline_build_"
                        f"{normalise_build_seed_key(build_key)}"
                    ),
                ),
        })


random_seed_manifest = pd.DataFrame(
    random_seed_records
)


# ------------------------------------------------------------
# 19. INSTANTIATE MODEL CONFIGURATION
# ------------------------------------------------------------

seed_one_record = model_seed_manifest[
    model_seed_manifest[
        "RepetitionSeed"
    ].eq(1)
].iloc[0]


model_instances = {
    "RandomForest":
        RandomForestClassifier(
            **MODEL_CONFIG[
                "RandomForest"
            ],
            random_state=int(
                seed_one_record[
                    "RandomForestSeed"
                ]
            ),
        ),

    "XGBoost":
        XGBClassifier(
            **MODEL_CONFIG[
                "XGBoost"
            ],
            random_state=int(
                seed_one_record[
                    "XGBoostSeed"
                ]
            ),
        ),

    "LightGBM":
        LGBMClassifier(
            **MODEL_CONFIG[
                "LightGBM"
            ],
            random_state=int(
                seed_one_record[
                    "LightGBMSeed"
                ]
            ),
        ),

    "NaiveBayes":
        GaussianNB(
            var_smoothing=(
                MODEL_CONFIG[
                    "NaiveBayes"
                ][
                    "var_smoothing"
                ]
            )
        ),
}


instantiated_model_classes = {
    technique:
        type(
            model
        ).__name__

    for technique, model in (
        model_instances.items()
    )
}


# ------------------------------------------------------------
# 20. CREATE 0% AND 50% TRAINING HISTORIES
# ------------------------------------------------------------

clean_raw_training_verdicts = raw_training[
    "CleanVerdict"
].to_numpy(
    dtype=np.int8
)


zero_noisy_training_history = raw_training.copy()

zero_noisy_training_history[
    "NoisyVerdict"
] = clean_raw_training_verdicts


seed_one_rng_rows = (
    noise_rng_manifest[
        noise_rng_manifest[
            "RepetitionSeed"
        ].eq(1)
    ]
    .sort_values(
        "NoiseRowID",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    seed_one_rng_rows
) != EXPECTED_RAW_TRAINING_ROWS:

    raise AssertionError(
        "Seed 1 RNG-manifest row count differs."
    )


flip_mask_50 = (
    seed_one_rng_rows[
        "FlipUniform"
    ].to_numpy(
        dtype=float
    )
    < 0.50
)

sampled_failure_subtypes = seed_one_rng_rows[
    "SampledFailureSubtype"
].to_numpy(
    dtype=np.int8
)


noisy_raw_verdicts_50 = (
    clean_raw_training_verdicts.copy()
)


pass_to_failure_50 = (
    flip_mask_50
    & (
        clean_raw_training_verdicts
        == 0
    )
)

failure_to_pass_50 = (
    flip_mask_50
    & (
        clean_raw_training_verdicts
        != 0
    )
)


noisy_raw_verdicts_50[
    pass_to_failure_50
] = sampled_failure_subtypes[
    pass_to_failure_50
]

noisy_raw_verdicts_50[
    failure_to_pass_50
] = 0


condition_50_seed_1 = condition_plan[
    condition_plan[
        "NoisePercent"
    ].eq(50)
    & condition_plan[
        "RepetitionSeed"
    ].eq(1)
]


if len(
    condition_50_seed_1
) != 1:

    raise AssertionError(
        "The 50% seed-1 condition was not found exactly once."
    )


expected_50_changes = int(
    condition_50_seed_1.iloc[0][
        "RawLabelChanges"
    ]
)

actual_50_changes = int(
    np.count_nonzero(
        noisy_raw_verdicts_50
        != clean_raw_training_verdicts
    )
)


if actual_50_changes != expected_50_changes:

    raise AssertionError(
        "The 50% seed-1 raw-label count differs."
    )


fifty_noisy_training_history = raw_training.copy()

fifty_noisy_training_history[
    "NoisyVerdict"
] = noisy_raw_verdicts_50


# ------------------------------------------------------------
# 21. BASELINE SENTINEL VALIDATION
# ------------------------------------------------------------

random_seed_1_noise_0 = create_random_rankings(
    clean_model_evaluation_base,
    repetition_seed=1,
)

random_seed_1_noise_50 = create_random_rankings(
    clean_model_evaluation_base,
    repetition_seed=1,
)

random_seed_30 = create_random_rankings(
    clean_model_evaluation_base,
    repetition_seed=30,
)


(
    latest_fail_noise_0,
    qtf_noise_0,
) = create_history_baseline_rankings(
    noisy_training_history=(
        zero_noisy_training_history
    ),

    clean_training_history=(
        raw_training
    ),

    clean_evaluation_history=(
        raw_evaluation
    ),

    clean_evaluation_data=(
        clean_model_evaluation_base
    ),

    fixed_split=(
        fixed_split
    ),
)


(
    latest_fail_noise_50,
    qtf_noise_50,
) = create_history_baseline_rankings(
    noisy_training_history=(
        fifty_noisy_training_history
    ),

    clean_training_history=(
        raw_training
    ),

    clean_evaluation_history=(
        raw_evaluation
    ),

    clean_evaluation_data=(
        clean_model_evaluation_base
    ),

    fixed_split=(
        fixed_split
    ),
)


baseline_ranking_objects = {
    "Random_seed01_noise00":
        random_seed_1_noise_0,

    "Random_seed01_noise50":
        random_seed_1_noise_50,

    "Random_seed30_noise00":
        random_seed_30,

    "LatestFail_noise00_seed01":
        latest_fail_noise_0,

    "LatestFail_noise50_seed01":
        latest_fail_noise_50,

    "QTF-Avg_noise00":
        qtf_noise_0,

    "QTF-Avg_noise50":
        qtf_noise_50,
}


baseline_sentinel_rankings = pd.concat(
    [
        add_scenario(
            ranking,
            scenario,
        )

        for scenario, ranking in (
            baseline_ranking_objects.items()
        )
    ],
    ignore_index=True,
)


baseline_sentinel_metrics = calculate_baseline_metrics(
    baseline_sentinel_rankings
)


baseline_scenario_counts = (
    baseline_sentinel_rankings.groupby(
        "Scenario"
    ).size()
)


baseline_scenarios_with_wrong_row_count = int(
    baseline_scenario_counts.ne(
        EXPECTED_MODEL_EVALUATION_ROWS
    ).sum()
)


random_seed_1_hash_a = ranking_semantic_hash(
    random_seed_1_noise_0
)

random_seed_1_hash_b = ranking_semantic_hash(
    random_seed_1_noise_50
)

random_seed_30_hash = ranking_semantic_hash(
    random_seed_30
)

latest_fail_noise_0_hash = ranking_semantic_hash(
    latest_fail_noise_0
)

latest_fail_noise_50_hash = ranking_semantic_hash(
    latest_fail_noise_50
)

qtf_noise_0_hash = ranking_semantic_hash(
    qtf_noise_0
)

qtf_noise_50_hash = ranking_semantic_hash(
    qtf_noise_50
)


random_same_seed_noise_invariant = (
    random_seed_1_hash_a
    == random_seed_1_hash_b
)

random_different_seeds_differ = (
    random_seed_1_hash_a
    != random_seed_30_hash
)

latest_fail_responds_to_noise = (
    latest_fail_noise_0_hash
    != latest_fail_noise_50_hash
)

qtf_noise_invariant = (
    qtf_noise_0_hash
    == qtf_noise_50_hash
)


qtf_metric_zero = (
    baseline_sentinel_metrics[
        baseline_sentinel_metrics[
            "Scenario"
        ].eq(
            "QTF-Avg_noise00"
        )
    ]
    .sort_values(
        "BuildKey",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

qtf_metric_fifty = (
    baseline_sentinel_metrics[
        baseline_sentinel_metrics[
            "Scenario"
        ].eq(
            "QTF-Avg_noise50"
        )
    ]
    .sort_values(
        "BuildKey",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


qtf_metric_identity = bool(
    np.array_equal(
        qtf_metric_zero[
            [
                "APFD",
                "APFDc",
            ]
        ].to_numpy(
            dtype=float
        ),

        qtf_metric_fifty[
            [
                "APFD",
                "APFDc",
            ]
        ].to_numpy(
            dtype=float
        ),
    )
)


baseline_metric_nan_values = int(
    baseline_sentinel_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].isna().sum().sum()
)

baseline_metric_out_of_range_values = int(
    (
        ~baseline_sentinel_metrics[
            "APFD"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
    +
    (
        ~baseline_sentinel_metrics[
            "APFDc"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
)


# ------------------------------------------------------------
# 22. BASELINE PROTOCOL AUDIT
# ------------------------------------------------------------

baseline_protocol_audit = pd.DataFrame([
    {
        "Technique":
            "Random",

        "TrainingHistory":
            "None",

        "NoiseAffected":
            False,

        "Score":
            (
                "Deterministic uniform random value from "
                "project × repetition seed × evaluation build"
            ),

        "Direction":
            "DESCENDING",

        "EvaluationUpdate":
            "None",

        "TestTieBreak":
            "Test ascending",

        "ValidationPass":
            (
                random_same_seed_noise_invariant
                and random_different_seeds_differ
            ),
    },

    {
        "Technique":
            "LatestFail",

        "TrainingHistory":
            "Noisy raw training verdict history",

        "NoiseAffected":
            True,

        "Score":
            (
                "Build order of latest observed non-zero verdict; "
                "unseen failure score = -1"
            ),

        "Direction":
            "DESCENDING",

        "EvaluationUpdate":
            (
                "Clean evaluation verdict history, after ranking "
                "the current build"
            ),

        "TestTieBreak":
            "Test ascending",

        "ValidationPass":
            latest_fail_responds_to_noise,
    },

    {
        "Technique":
            "QTF-Avg",

        "TrainingHistory":
            "Clean raw duration history",

        "NoiseAffected":
            False,

        "Score":
            (
                "Historical average execution duration; "
                "unseen duration score = +infinity"
            ),

        "Direction":
            "ASCENDING",

        "EvaluationUpdate":
            (
                "Clean evaluation duration history, after ranking "
                "the current build"
            ),

        "TestTieBreak":
            "Test ascending",

        "ValidationPass":
            (
                qtf_noise_invariant
                and qtf_metric_identity
            ),
    },
])


# ------------------------------------------------------------
# 23. RANKING TIE-RULE VALIDATION
# ------------------------------------------------------------

synthetic_build = pd.DataFrame({
    "Build":
        [
            1,
            1,
            1,
        ],

    "Test":
        [
            3,
            1,
            2,
        ],

    "Verdict":
        [
            0,
            1,
            0,
        ],

    "Duration":
        [
            1.0,
            1.0,
            1.0,
        ],

    "build_order":
        [
            1,
            1,
            1,
        ],

    "BuildKey":
        [
            "1",
            "1",
            "1",
        ],

    "TestKey":
        [
            "3",
            "1",
            "2",
        ],
})


descending_tie_result = rank_build_rows(
    build_rows=synthetic_build,
    scores=[
        0.5,
        0.5,
        0.8,
    ],
    technique="SyntheticDescending",
    score_direction="descending",
)


ascending_tie_result = rank_build_rows(
    build_rows=synthetic_build,
    scores=[
        0.5,
        0.5,
        0.2,
    ],
    technique="SyntheticAscending",
    score_direction="ascending",
)


descending_test_order = descending_tie_result[
    "Test"
].astype(int).tolist()

ascending_test_order = ascending_tie_result[
    "Test"
].astype(int).tolist()


expected_tie_order = [
    2,
    1,
    3,
]


ranking_tie_validation = pd.DataFrame([
    {
        "Example":
            "Descending score, Test ascending on tie",

        "ScoreDirection":
            "DESCENDING",

        "ExpectedTestOrder":
            str(
                expected_tie_order
            ),

        "ActualTestOrder":
            str(
                descending_test_order
            ),

        "Pass":
            descending_test_order
            == expected_tie_order,
    },

    {
        "Example":
            "Ascending score, Test ascending on tie",

        "ScoreDirection":
            "ASCENDING",

        "ExpectedTestOrder":
            str(
                expected_tie_order
            ),

        "ActualTestOrder":
            str(
                ascending_test_order
            ),

        "Pass":
            ascending_test_order
            == expected_tie_order,
    },
])


tie_rule_failures = int(
    (
        ~ranking_tie_validation[
            "Pass"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 24. MANUAL APFD/APFDC VALIDATION
# ------------------------------------------------------------

metric_examples = [
    {
        "Example":
            "Failures ranked first and second; slower first",

        "Failures":
            [
                1,
                1,
                0,
                0,
                0,
            ],

        "Durations":
            [
                5,
                1,
                1,
                1,
                1,
            ],

        "ExpectedAPFD":
            0.8,

        "ExpectedAPFDc":
            5.0 / 9.0,
    },

    {
        "Example":
            "Failures ranked first and second; quicker first",

        "Failures":
            [
                1,
                1,
                0,
                0,
                0,
            ],

        "Durations":
            [
                1,
                5,
                1,
                1,
                1,
            ],

        "ExpectedAPFD":
            0.8,

        "ExpectedAPFDc":
            7.0 / 9.0,
    },

    {
        "Example":
            "Failures ranked second and fourth",

        "Failures":
            [
                0,
                1,
                0,
                1,
                0,
            ],

        "Durations":
            [
                1,
                5,
                1,
                1,
                1,
            ],

        "ExpectedAPFD":
            0.5,

        "ExpectedAPFDc":
            7.0 / 18.0,
    },
]


metric_validation_records = []


for example in metric_examples:

    actual_apfd = calculate_apfd(
        example[
            "Failures"
        ]
    )

    actual_apfdc = calculate_apfdc(
        example[
            "Failures"
        ],
        example[
            "Durations"
        ],
    )

    apfd_match = bool(
        np.isclose(
            actual_apfd,
            example[
                "ExpectedAPFD"
            ],
            rtol=0,
            atol=1e-12,
        )
    )

    apfdc_match = bool(
        np.isclose(
            actual_apfdc,
            example[
                "ExpectedAPFDc"
            ],
            rtol=0,
            atol=1e-12,
        )
    )

    metric_validation_records.append({
        "Example":
            example[
                "Example"
            ],

        "ActualFailures":
            str(
                example[
                    "Failures"
                ]
            ),

        "Durations":
            str(
                example[
                    "Durations"
                ]
            ),

        "ExpectedAPFD":
            example[
                "ExpectedAPFD"
            ],

        "ActualAPFD":
            actual_apfd,

        "APFDMatch":
            apfd_match,

        "ExpectedAPFDc":
            example[
                "ExpectedAPFDc"
            ],

        "ActualAPFDc":
            actual_apfdc,

        "APFDcMatch":
            apfdc_match,

        "Pass":
            (
                apfd_match
                and apfdc_match
            ),
    })


metric_validation = pd.DataFrame(
    metric_validation_records
)


metric_validation_failures = int(
    (
        ~metric_validation[
            "Pass"
        ]
    ).sum()
)


no_failure_apfd_is_nan = bool(
    np.isnan(
        calculate_apfd(
            [
                0,
                0,
                0,
            ]
        )
    )
)

no_failure_apfdc_is_nan = bool(
    np.isnan(
        calculate_apfdc(
            [
                0,
                0,
                0,
            ],
            [
                1,
                1,
                1,
            ],
        )
    )
)


negative_duration_rejected = False


try:

    calculate_apfdc(
        [
            1,
            0,
        ],
        [
            1,
            -1,
        ],
    )

except ValueError:

    negative_duration_rejected = True


# ------------------------------------------------------------
# 25. MODEL CONFIGURATION PAYLOAD
# ------------------------------------------------------------

model_configuration_payload = {
    "Project":
        PROJECT_NAME,

    "ProjectNumber":
        PROJECT_NUMBER,

    "ProtocolVersion":
        "PROJECT_10_MODEL_BASELINE_METRIC_PROTOCOL_V1",

    "OriginalPredictors":
        len(
            model_predictor_columns
        ),

    "ZeroVarianceFeatures":
        zero_variance_features,

    "ZeroVarianceFeatureCount":
        len(
            zero_variance_features
        ),

    "ActiveFeatureColumns":
        active_feature_columns,

    "ActiveFeatureCount":
        len(
            active_feature_columns
        ),

    "ZeroVarianceRule":
        (
            "Identify using only the clean fixed training "
            "partition and freeze across all conditions"
        ),

    "MissingValueRule":
        (
            "Replace infinities with missing; learn medians "
            "from the current noisy training condition; apply "
            "those medians to training and clean evaluation; "
            "all-missing median fallback = 0"
        ),

    "LabelRule":
        "Binary failure = 1 when verdict is non-zero",

    "PositiveClass":
        1,

    "RollingRetraining":
        False,

    "ModelFitFrequency":
        (
            "One fit per ML technique per project × noise × seed "
            "condition; evaluate the complete clean fixed holdout"
        ),

    "TieRule":
        "Score first, then Test ascending",

    "ScoreDirections": {
        "RandomForest":
            "DESCENDING",

        "XGBoost":
            "DESCENDING",

        "LightGBM":
            "DESCENDING",

        "NaiveBayes":
            "DESCENDING",

        "Random":
            "DESCENDING",

        "LatestFail":
            "DESCENDING",

        "QTF-Avg":
            "ASCENDING",
    },

    "Models":
        MODEL_CONFIG,

    "ModelSeedStreams": {
        "RandomForest":
            "RandomForest_model",

        "XGBoost":
            "XGBoost_model",

        "LightGBM":
            "LightGBM_model",

        "NaiveBayes":
            None,
    },

    "RandomBaselineSeedStream":
        (
            "Random_baseline_build_<canonical Build ID>"
        ),

    "Baselines": {
        "Random":
            (
                "Uniform random score per evaluation test; "
                "project/seed/build deterministic; constant "
                "across noise for the same seed"
            ),

        "LatestFail":
            (
                "Latest non-zero verdict build order from noisy "
                "training history and subsequent clean evaluation "
                "history; rank before updating current build"
            ),

        "QTF-Avg":
            (
                "Ascending historical average duration from clean "
                "training and clean evaluation history; rank before "
                "updating current build; noise independent"
            ),
    },

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "APFDPolicy":
        (
            "1 - sum(failure ranks)/(n*m) + 1/(2*n); "
            "undefined for zero tests or zero failures"
        ),

    "APFDcPolicy":
        (
            "1 - mean(midpoint failure-detection time / "
            "total ranked duration); undefined for zero tests, "
            "zero failures or non-positive total duration"
        ),

    "EvaluationVerdicts":
        "Clean and immutable",

    "EvaluationDurations":
        "Clean and immutable",

    "LibraryVersions": {
        "Python":
            platform.python_version(),

        "NumPy":
            np.__version__,

        "Pandas":
            pd.__version__,

        "ScikitLearn":
            sklearn.__version__,

        "XGBoost":
            xgboost.__version__,

        "LightGBM":
            lightgbm.__version__,
    },

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# ------------------------------------------------------------
# 26. WRITE PRELIMINARY OUTPUTS
# ------------------------------------------------------------

MODEL_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_parquet(
    CLEAN_MODEL_TRAINING_BASE_PATH,
    clean_model_training_base,
)

atomic_write_parquet(
    CLEAN_MODEL_EVALUATION_BASE_PATH,
    clean_model_evaluation_base,
)

atomic_write_csv(
    PREDICTOR_MANIFEST_PATH,
    predictor_manifest,
)

atomic_write_csv(
    ZERO_VARIANCE_FEATURES_PATH,
    zero_variance_features_frame,
)

atomic_write_csv(
    CLEAN_MEDIAN_REFERENCE_PATH,
    clean_median_reference,
)

atomic_write_json(
    MODEL_CONFIGURATION_PATH,
    model_configuration_payload,
)

atomic_write_csv(
    MODEL_SEED_MANIFEST_PATH,
    model_seed_manifest,
)

atomic_write_csv(
    RANDOM_SEED_MANIFEST_PATH,
    random_seed_manifest,
)

atomic_write_parquet(
    BASELINE_SENTINEL_RANKINGS_PATH,
    baseline_sentinel_rankings,
)

atomic_write_csv(
    BASELINE_SENTINEL_METRICS_PATH,
    baseline_sentinel_metrics,
)

atomic_write_csv(
    BASELINE_PROTOCOL_AUDIT_PATH,
    baseline_protocol_audit,
)

atomic_write_csv(
    RANKING_TIE_VALIDATION_PATH,
    ranking_tie_validation,
)

atomic_write_csv(
    METRIC_VALIDATION_PATH,
    metric_validation,
)


# ------------------------------------------------------------
# 27. READBACK
# ------------------------------------------------------------

clean_training_readback = pd.read_parquet(
    CLEAN_MODEL_TRAINING_BASE_PATH
)

clean_evaluation_readback = pd.read_parquet(
    CLEAN_MODEL_EVALUATION_BASE_PATH
)

predictor_manifest_readback = pd.read_csv(
    PREDICTOR_MANIFEST_PATH,
    low_memory=False,
)

model_seed_readback = pd.read_csv(
    MODEL_SEED_MANIFEST_PATH,
    low_memory=False,
)

random_seed_readback = pd.read_csv(
    RANDOM_SEED_MANIFEST_PATH,
    low_memory=False,
)

baseline_rankings_readback = pd.read_parquet(
    BASELINE_SENTINEL_RANKINGS_PATH
)

baseline_metrics_readback = pd.read_csv(
    BASELINE_SENTINEL_METRICS_PATH,
    low_memory=False,
)


# ------------------------------------------------------------
# 28. IMMUTABILITY CHECKS
# ------------------------------------------------------------

raw_evaluation_file_sha256_after = calculate_hash(
    raw_evaluation_path
)

model_evaluation_file_sha256_after = calculate_hash(
    model_evaluation_path
)


raw_evaluation_unchanged = (
    raw_evaluation_file_sha256_before
    == raw_evaluation_file_sha256_after
)

model_evaluation_unchanged = (
    model_evaluation_file_sha256_before
    == model_evaluation_file_sha256_after
)


source_root_after = create_source_root_sha256(
    source_files_payload
)

source_unchanged = (
    source_root_before
    == source_root_after
    == EXPECTED_SOURCE_ROOT_SHA256
)


registry_sha256_after = calculate_hash(
    REGISTRY_PATH
)

registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


# ------------------------------------------------------------
# 29. VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check": "Step 1B passed",
        "Expected": EXPECTED_STEP1B_STATUS,
        "Actual": step1b_status.get("Status"),
        "Pass": (
            step1b_status.get("Status")
            == EXPECTED_STEP1B_STATUS
        ),
    },

    {
        "Check": "Step 2B passed",
        "Expected": EXPECTED_STEP2B_STATUS,
        "Actual": step2b_status.get("Status"),
        "Pass": (
            step2b_status.get("Status")
            == EXPECTED_STEP2B_STATUS
        ),
    },

    {
        "Check": "Step 3A passed",
        "Expected": EXPECTED_STEP3A_STATUS,
        "Actual": step3a_status.get("Status"),
        "Pass": (
            step3a_status.get("Status")
            == EXPECTED_STEP3A_STATUS
        ),
    },

    {
        "Check": "Step 3B passed",
        "Expected": EXPECTED_STEP3B_STATUS,
        "Actual": step3b_status.get("Status"),
        "Pass": (
            step3b_status.get("Status")
            == EXPECTED_STEP3B_STATUS
        ),
    },

    {
        "Check": "Dataset rows",
        "Expected": EXPECTED_DATASET_ROWS,
        "Actual": len(dataset),
        "Pass": len(dataset) == EXPECTED_DATASET_ROWS,
    },

    {
        "Check": "Dataset columns",
        "Expected": EXPECTED_DATASET_COLUMNS,
        "Actual": len(dataset.columns),
        "Pass": (
            len(dataset.columns)
            == EXPECTED_DATASET_COLUMNS
        ),
    },

    {
        "Check": "Original predictors",
        "Expected": EXPECTED_PREDICTOR_COLUMNS,
        "Actual": len(model_predictor_columns),
        "Pass": (
            len(model_predictor_columns)
            == EXPECTED_PREDICTOR_COLUMNS
        ),
    },

    {
        "Check": "Active plus zero-variance predictors",
        "Expected": EXPECTED_PREDICTOR_COLUMNS,
        "Actual": (
            len(active_feature_columns)
            + len(zero_variance_features)
        ),
        "Pass": (
            len(active_feature_columns)
            + len(zero_variance_features)
            == EXPECTED_PREDICTOR_COLUMNS
        ),
    },

    {
        "Check": "Active predictors non-empty",
        "Expected": True,
        "Actual": len(active_feature_columns) > 0,
        "Pass": len(active_feature_columns) > 0,
    },

    {
        "Check": "REC predictors",
        "Expected": 19,
        "Actual": int(
            predictor_manifest["IsREC"].sum()
        ),
        "Pass": int(
            predictor_manifest["IsREC"].sum()
        ) == 19,
    },

    {
        "Check": "Verdict-dependent REC predictors",
        "Expected": 13,
        "Actual": int(
            predictor_manifest[
                "VerdictDependentREC"
            ].sum()
        ),
        "Pass": int(
            predictor_manifest[
                "VerdictDependentREC"
            ].sum()
        ) == 13,
    },

    {
        "Check": "Verdict-independent REC predictors",
        "Expected": 6,
        "Actual": int(
            predictor_manifest[
                "VerdictIndependentREC"
            ].sum()
        ),
        "Pass": int(
            predictor_manifest[
                "VerdictIndependentREC"
            ].sum()
        ) == 6,
    },

    {
        "Check": "Model training rows",
        "Expected": EXPECTED_MODEL_TRAINING_ROWS,
        "Actual": training_rows,
        "Pass": (
            training_rows
            == EXPECTED_MODEL_TRAINING_ROWS
        ),
    },

    {
        "Check": "Model evaluation rows",
        "Expected": EXPECTED_MODEL_EVALUATION_ROWS,
        "Actual": evaluation_rows,
        "Pass": (
            evaluation_rows
            == EXPECTED_MODEL_EVALUATION_ROWS
        ),
    },

    {
        "Check": "Model training failures",
        "Expected": EXPECTED_MODEL_TRAINING_FAILURES,
        "Actual": training_failures,
        "Pass": (
            training_failures
            == EXPECTED_MODEL_TRAINING_FAILURES
        ),
    },

    {
        "Check": "Model evaluation failures",
        "Expected": EXPECTED_MODEL_EVALUATION_FAILURES,
        "Actual": evaluation_failures,
        "Pass": (
            evaluation_failures
            == EXPECTED_MODEL_EVALUATION_FAILURES
        ),
    },

    {
        "Check": "Evaluation-period builds",
        "Expected": EXPECTED_EVALUATION_PERIOD_BUILDS,
        "Actual": evaluation_period_builds,
        "Pass": (
            evaluation_period_builds
            == EXPECTED_EVALUATION_PERIOD_BUILDS
        ),
    },

    {
        "Check": "Scored evaluation builds",
        "Expected": EXPECTED_SCORED_EVALUATION_BUILDS,
        "Actual": scored_evaluation_builds,
        "Pass": (
            scored_evaluation_builds
            == EXPECTED_SCORED_EVALUATION_BUILDS
        ),
    },

    {
        "Check": "Scored builds without failures",
        "Expected": 0,
        "Actual": evaluation_builds_without_failures,
        "Pass": evaluation_builds_without_failures == 0,
    },

    {
        "Check": "Scored builds with non-positive duration",
        "Expected": 0,
        "Actual": (
            evaluation_builds_with_nonpositive_total_duration
        ),
        "Pass": (
            evaluation_builds_with_nonpositive_total_duration
            == 0
        ),
    },

    {
        "Check": "Duplicate training Build/Test rows",
        "Expected": 0,
        "Actual": duplicate_training_build_test_rows,
        "Pass": duplicate_training_build_test_rows == 0,
    },

    {
        "Check": "Duplicate evaluation Build/Test rows",
        "Expected": 0,
        "Actual": duplicate_evaluation_build_test_rows,
        "Pass": duplicate_evaluation_build_test_rows == 0,
    },

    {
        "Check": "Training verdict mismatches",
        "Expected": 0,
        "Actual": training_verdict_mismatches,
        "Pass": training_verdict_mismatches == 0,
    },

    {
        "Check": "Evaluation verdict mismatches",
        "Expected": 0,
        "Actual": evaluation_verdict_mismatches,
        "Pass": evaluation_verdict_mismatches == 0,
    },

    {
        "Check": "Missing evaluation raw links",
        "Expected": 0,
        "Actual": missing_evaluation_raw_links,
        "Pass": missing_evaluation_raw_links == 0,
    },

    {
        "Check": "Evaluation raw verdict mismatches",
        "Expected": 0,
        "Actual": evaluation_raw_verdict_mismatches,
        "Pass": evaluation_raw_verdict_mismatches == 0,
    },

    {
        "Check": "Clean training classes",
        "Expected": [0, 1],
        "Actual": clean_training_label_classes,
        "Pass": (
            clean_training_label_classes
            == [0, 1]
        ),
    },

    {
        "Check": "Training non-finite values after imputation",
        "Expected": 0,
        "Actual": (
            training_nonfinite_values_after_imputation
        ),
        "Pass": (
            training_nonfinite_values_after_imputation
            == 0
        ),
    },

    {
        "Check": "Evaluation non-finite values after imputation",
        "Expected": 0,
        "Actual": (
            evaluation_nonfinite_values_after_imputation
        ),
        "Pass": (
            evaluation_nonfinite_values_after_imputation
            == 0
        ),
    },

    {
        "Check": "Instantiated ML models",
        "Expected": 4,
        "Actual": len(model_instances),
        "Pass": len(model_instances) == 4,
    },

    {
        "Check": "Model-seed rows",
        "Expected": 30,
        "Actual": len(model_seed_manifest),
        "Pass": len(model_seed_manifest) == 30,
    },

    {
        "Check": "Random-seed rows",
        "Expected": (
            30
            * EXPECTED_SCORED_EVALUATION_BUILDS
        ),
        "Actual": len(random_seed_manifest),
        "Pass": (
            len(random_seed_manifest)
            == (
                30
                * EXPECTED_SCORED_EVALUATION_BUILDS
            )
        ),
    },

    {
        "Check": "Baseline scenarios",
        "Expected": EXPECTED_BASELINE_SCENARIOS,
        "Actual": baseline_sentinel_rankings[
            "Scenario"
        ].nunique(),
        "Pass": (
            baseline_sentinel_rankings[
                "Scenario"
            ].nunique()
            == EXPECTED_BASELINE_SCENARIOS
        ),
    },

    {
        "Check": "Baseline ranking rows",
        "Expected": EXPECTED_BASELINE_RANKING_ROWS,
        "Actual": len(baseline_sentinel_rankings),
        "Pass": (
            len(baseline_sentinel_rankings)
            == EXPECTED_BASELINE_RANKING_ROWS
        ),
    },

    {
        "Check": "Baseline scenarios with wrong row count",
        "Expected": 0,
        "Actual": (
            baseline_scenarios_with_wrong_row_count
        ),
        "Pass": (
            baseline_scenarios_with_wrong_row_count
            == 0
        ),
    },

    {
        "Check": "Baseline metric rows",
        "Expected": EXPECTED_BASELINE_METRIC_ROWS,
        "Actual": len(baseline_sentinel_metrics),
        "Pass": (
            len(baseline_sentinel_metrics)
            == EXPECTED_BASELINE_METRIC_ROWS
        ),
    },

    {
        "Check": "Baseline metric NaN values",
        "Expected": 0,
        "Actual": baseline_metric_nan_values,
        "Pass": baseline_metric_nan_values == 0,
    },

    {
        "Check": "Baseline metric values outside range",
        "Expected": 0,
        "Actual": baseline_metric_out_of_range_values,
        "Pass": baseline_metric_out_of_range_values == 0,
    },

    {
        "Check": "Random same-seed noise invariance",
        "Expected": True,
        "Actual": random_same_seed_noise_invariant,
        "Pass": random_same_seed_noise_invariant,
    },

    {
        "Check": "Random different seeds differ",
        "Expected": True,
        "Actual": random_different_seeds_differ,
        "Pass": random_different_seeds_differ,
    },

    {
        "Check": "LatestFail responds to verdict noise",
        "Expected": True,
        "Actual": latest_fail_responds_to_noise,
        "Pass": latest_fail_responds_to_noise,
    },

    {
        "Check": "QTF-Avg noise invariance",
        "Expected": True,
        "Actual": qtf_noise_invariant,
        "Pass": qtf_noise_invariant,
    },

    {
        "Check": "QTF-Avg metric identity",
        "Expected": True,
        "Actual": qtf_metric_identity,
        "Pass": qtf_metric_identity,
    },

    {
        "Check": "Baseline protocol audit",
        "Expected": True,
        "Actual": bool(
            baseline_protocol_audit[
                "ValidationPass"
            ].all()
        ),
        "Pass": bool(
            baseline_protocol_audit[
                "ValidationPass"
            ].all()
        ),
    },

    {
        "Check": "Ranking tie-rule failures",
        "Expected": 0,
        "Actual": tie_rule_failures,
        "Pass": tie_rule_failures == 0,
    },

    {
        "Check": "Metric validation failures",
        "Expected": 0,
        "Actual": metric_validation_failures,
        "Pass": metric_validation_failures == 0,
    },

    {
        "Check": "No-failure APFD is undefined",
        "Expected": True,
        "Actual": no_failure_apfd_is_nan,
        "Pass": no_failure_apfd_is_nan,
    },

    {
        "Check": "No-failure APFDc is undefined",
        "Expected": True,
        "Actual": no_failure_apfdc_is_nan,
        "Pass": no_failure_apfdc_is_nan,
    },

    {
        "Check": "Negative duration rejected",
        "Expected": True,
        "Actual": negative_duration_rejected,
        "Pass": negative_duration_rejected,
    },

    {
        "Check": "Clean training readback rows",
        "Expected": EXPECTED_MODEL_TRAINING_ROWS,
        "Actual": len(clean_training_readback),
        "Pass": (
            len(clean_training_readback)
            == EXPECTED_MODEL_TRAINING_ROWS
        ),
    },

    {
        "Check": "Clean evaluation readback rows",
        "Expected": EXPECTED_MODEL_EVALUATION_ROWS,
        "Actual": len(clean_evaluation_readback),
        "Pass": (
            len(clean_evaluation_readback)
            == EXPECTED_MODEL_EVALUATION_ROWS
        ),
    },

    {
        "Check": "Predictor-manifest readback rows",
        "Expected": EXPECTED_PREDICTOR_COLUMNS,
        "Actual": len(predictor_manifest_readback),
        "Pass": (
            len(predictor_manifest_readback)
            == EXPECTED_PREDICTOR_COLUMNS
        ),
    },

    {
        "Check": "Model-seed readback rows",
        "Expected": 30,
        "Actual": len(model_seed_readback),
        "Pass": len(model_seed_readback) == 30,
    },

    {
        "Check": "Random-seed readback rows",
        "Expected": (
            30
            * EXPECTED_SCORED_EVALUATION_BUILDS
        ),
        "Actual": len(random_seed_readback),
        "Pass": (
            len(random_seed_readback)
            == (
                30
                * EXPECTED_SCORED_EVALUATION_BUILDS
            )
        ),
    },

    {
        "Check": "Baseline-ranking readback rows",
        "Expected": EXPECTED_BASELINE_RANKING_ROWS,
        "Actual": len(baseline_rankings_readback),
        "Pass": (
            len(baseline_rankings_readback)
            == EXPECTED_BASELINE_RANKING_ROWS
        ),
    },

    {
        "Check": "Baseline-metric readback rows",
        "Expected": EXPECTED_BASELINE_METRIC_ROWS,
        "Actual": len(baseline_metrics_readback),
        "Pass": (
            len(baseline_metrics_readback)
            == EXPECTED_BASELINE_METRIC_ROWS
        ),
    },

    {
        "Check": "Raw evaluation cohort unchanged",
        "Expected": True,
        "Actual": raw_evaluation_unchanged,
        "Pass": raw_evaluation_unchanged,
    },

    {
        "Check": "Model evaluation cohort unchanged",
        "Expected": True,
        "Actual": model_evaluation_unchanged,
        "Pass": model_evaluation_unchanged,
    },

    {
        "Check": "Project 10 source unchanged",
        "Expected": True,
        "Actual": source_unchanged,
        "Pass": source_unchanged,
    },

    {
        "Check": "Completion registry unchanged",
        "Expected": True,
        "Actual": registry_unchanged,
        "Pass": registry_unchanged,
    },

    {
        "Check": "Registry Project 9 rows",
        "Expected": 0,
        "Actual": int(
            registry_project_numbers.eq(9).sum()
        ),
        "Pass": int(
            registry_project_numbers.eq(9).sum()
        ) == 0,
    },

    {
        "Check": "Registry Project 10 rows",
        "Expected": 0,
        "Actual": int(
            registry_project_numbers.eq(10).sum()
        ),
        "Pass": int(
            registry_project_numbers.eq(10).sum()
        ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 4A validation:")

display(
    validation
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 10 STEP 4A DID NOT PASS."
    )


atomic_write_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 30. REPORT AND CHECKPOINT
# ------------------------------------------------------------

output_hashes = {
    "CleanModelTrainingBaseSHA256":
        calculate_hash(
            CLEAN_MODEL_TRAINING_BASE_PATH
        ),

    "CleanModelEvaluationBaseSHA256":
        calculate_hash(
            CLEAN_MODEL_EVALUATION_BASE_PATH
        ),

    "PredictorManifestSHA256":
        calculate_hash(
            PREDICTOR_MANIFEST_PATH
        ),

    "ZeroVarianceFeaturesSHA256":
        calculate_hash(
            ZERO_VARIANCE_FEATURES_PATH
        ),

    "CleanMedianReferenceSHA256":
        calculate_hash(
            CLEAN_MEDIAN_REFERENCE_PATH
        ),

    "ModelConfigurationSHA256":
        calculate_hash(
            MODEL_CONFIGURATION_PATH
        ),

    "ModelSeedManifestSHA256":
        calculate_hash(
            MODEL_SEED_MANIFEST_PATH
        ),

    "RandomSeedManifestSHA256":
        calculate_hash(
            RANDOM_SEED_MANIFEST_PATH
        ),

    "BaselineSentinelRankingsSHA256":
        calculate_hash(
            BASELINE_SENTINEL_RANKINGS_PATH
        ),

    "BaselineSentinelMetricsSHA256":
        calculate_hash(
            BASELINE_SENTINEL_METRICS_PATH
        ),

    "BaselineProtocolAuditSHA256":
        calculate_hash(
            BASELINE_PROTOCOL_AUDIT_PATH
        ),

    "RankingTieValidationSHA256":
        calculate_hash(
            RANKING_TIE_VALIDATION_PATH
        ),

    "MetricValidationSHA256":
        calculate_hash(
            METRIC_VALIDATION_PATH
        ),

    "Step4AValidationSHA256":
        calculate_hash(
            STEP4A_VALIDATION_PATH
        ),
}


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_PASS_STATUS,

    "ProtocolVersion":
        "PROJECT_10_MODEL_BASELINE_METRIC_PROTOCOL_V1",

    "OriginalPredictors":
        len(
            model_predictor_columns
        ),

    "ZeroVarianceFeatureCount":
        len(
            zero_variance_features
        ),

    "ZeroVarianceFeatures":
        zero_variance_features,

    "ActivePredictorCount":
        len(
            active_feature_columns
        ),

    "ActiveFeatureColumns":
        active_feature_columns,

    "ModelTrainingRows":
        training_rows,

    "ModelEvaluationRows":
        evaluation_rows,

    "ModelTrainingFailures":
        training_failures,

    "ModelEvaluationFailures":
        evaluation_failures,

    "EvaluationPeriodBuilds":
        evaluation_period_builds,

    "ScoredEvaluationBuilds":
        scored_evaluation_builds,

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "RandomSameSeedNoiseInvariant":
        random_same_seed_noise_invariant,

    "RandomDifferentSeedsDiffer":
        random_different_seeds_differ,

    "LatestFailRespondsToNoise":
        latest_fail_responds_to_noise,

    "QTFAvgNoiseInvariant":
        qtf_noise_invariant,

    "QTFAvgMetricIdentity":
        qtf_metric_identity,

    "TieRule":
        "Score first, then Test ascending",

    "RollingMLRetraining":
        False,

    "ModelConfiguration":
        model_configuration_payload,

    "InstantiatedModelClasses":
        instantiated_model_classes,

    "BaselineScenarios":
        list(
            baseline_ranking_objects.keys()
        ),

    "BaselineRankingRows":
        len(
            baseline_sentinel_rankings
        ),

    "BaselineMetricRows":
        len(
            baseline_sentinel_metrics
        ),

    "RawEvaluationUnchanged":
        raw_evaluation_unchanged,

    "ModelEvaluationUnchanged":
        model_evaluation_unchanged,

    "Project10SourceUnchanged":
        source_unchanged,

    "CompletionRegistryModified":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "OutputHashes":
        output_hashes,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "SourceRootSHA256":
        source_root_after,

    "SelectionCheckpointSHA256":
        calculate_hash(
            SELECTION_CHECKPOINT_PATH
        ),

    "RECCheckpointSHA256":
        calculate_hash(
            REC_CHECKPOINT_PATH
        ),

    "NoisePlanCheckpointSHA256":
        calculate_hash(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "NoisyRECEngineCheckpointSHA256":
        calculate_hash(
            NOISY_REC_ENGINE_CHECKPOINT_PATH
        ),

    "CleanModelTrainingBase":
        str(
            CLEAN_MODEL_TRAINING_BASE_PATH
        ),

    "CleanModelEvaluationBase":
        str(
            CLEAN_MODEL_EVALUATION_BASE_PATH
        ),

    "PredictorManifest":
        str(
            PREDICTOR_MANIFEST_PATH
        ),

    "ZeroVarianceFeaturesPath":
        str(
            ZERO_VARIANCE_FEATURES_PATH
        ),

    "CleanMedianReference":
        str(
            CLEAN_MEDIAN_REFERENCE_PATH
        ),

    "ModelConfigurationPath":
        str(
            MODEL_CONFIGURATION_PATH
        ),

    "ModelSeedManifest":
        str(
            MODEL_SEED_MANIFEST_PATH
        ),

    "RandomSeedManifest":
        str(
            RANDOM_SEED_MANIFEST_PATH
        ),

    "BaselineSentinelRankings":
        str(
            BASELINE_SENTINEL_RANKINGS_PATH
        ),

    "BaselineSentinelMetrics":
        str(
            BASELINE_SENTINEL_METRICS_PATH
        ),

    "BaselineProtocolAudit":
        str(
            BASELINE_PROTOCOL_AUDIT_PATH
        ),

    "RankingTieValidation":
        str(
            RANKING_TIE_VALIDATION_PATH
        ),

    "MetricValidation":
        str(
            METRIC_VALIDATION_PATH
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistryRows":
        len(
            registry
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    MODEL_PROTOCOL_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_PASS_STATUS,

    "OriginalPredictors":
        len(
            model_predictor_columns
        ),

    "ZeroVarianceFeatureCount":
        len(
            zero_variance_features
        ),

    "ActivePredictorCount":
        len(
            active_feature_columns
        ),

    "ScoredEvaluationBuilds":
        scored_evaluation_builds,

    "RandomSameSeedNoiseInvariant":
        random_same_seed_noise_invariant,

    "LatestFailRespondsToNoise":
        latest_fail_responds_to_noise,

    "QTFAvgNoiseInvariant":
        qtf_noise_invariant,

    "Checkpoint":
        str(
            MODEL_PROTOCOL_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_hash(
            MODEL_PROTOCOL_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 31. FINAL READBACK
# ------------------------------------------------------------

checkpoint_readback = read_json_with_retry(
    MODEL_PROTOCOL_CHECKPOINT_PATH
)

status_readback = read_json_with_retry(
    STEP4A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP4A_PASS_STATUS:

    raise AssertionError(
        "Model-protocol checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP4A_PASS_STATUS:

    raise AssertionError(
        "Step 4A status readback failed."
    )


if calculate_hash(
    REGISTRY_PATH
) != registry_sha256_before:

    raise AssertionError(
        "Completion registry changed during Step 4A."
    )


if create_source_root_sha256(
    source_files_payload
) != EXPECTED_SOURCE_ROOT_SHA256:

    raise AssertionError(
        "Project 10 source changed during Step 4A."
    )


# ------------------------------------------------------------
# 32. DISPLAY
# ------------------------------------------------------------

print("\nPredictor summary:")

display(
    predictor_manifest.groupby(
        [
            "PredictorClass",
            "ZeroVariance",
            "Active",
        ],
        as_index=False,
    ).agg(
        Predictors=(
            "Predictor",
            "count",
        )
    )
)


print("\nZero-variance predictors:")

display(
    zero_variance_features_frame
)


print("\nModel seed manifest:")

display(
    model_seed_manifest
)


print("\nBaseline protocol audit:")

display(
    baseline_protocol_audit
)


print("\nBaseline sentinel metrics summary:")

display(
    baseline_sentinel_metrics.groupby(
        [
            "Scenario",
            "Technique",
        ],
        as_index=False,
    ).agg(
        Builds=(
            "BuildKey",
            "nunique",
        ),

        MeanAPFD=(
            "APFD",
            "mean",
        ),

        MeanAPFDc=(
            "APFDc",
            "mean",
        ),
    )
)


print("\nRanking tie validation:")

display(
    ranking_tie_validation
)


print("\nMetric validation:")

display(
    metric_validation
)


# ------------------------------------------------------------
# 33. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 126)
print("=== PROJECT 10 CELL 7 / STEP 4A RESULT ===")
print("=" * 126)

print("\nProject:")
print(PROJECT_NAME)


print("\nPredictor protocol:")

print(
    "Original predictor columns:",
    len(
        model_predictor_columns
    ),
)

print(
    "Frozen zero-variance columns:",
    len(
        zero_variance_features
    ),
)

print(
    "Active predictor columns:",
    len(
        active_feature_columns
    ),
)

print(
    "Training non-finite values after imputation:",
    training_nonfinite_values_after_imputation,
)

print(
    "Evaluation non-finite values after imputation:",
    evaluation_nonfinite_values_after_imputation,
)


print("\nFixed model cohorts:")

print(
    "Model training rows:",
    training_rows,
)

print(
    "Model evaluation rows:",
    evaluation_rows,
)

print(
    "Model training failures:",
    training_failures,
)

print(
    "Model evaluation failures:",
    evaluation_failures,
)

print(
    "Evaluation-period builds:",
    evaluation_period_builds,
)

print(
    "Scored evaluation builds:",
    scored_evaluation_builds,
)


print("\nML protocol:")

print(
    "Techniques:",
    ML_TECHNIQUES,
)

print(
    "Instantiated model classes:",
    instantiated_model_classes,
)

print(
    "Rolling retraining:",
    False,
)

print(
    "Tie rule:",
    "Score first, then Test ascending",
)


print("\nBaseline protocol:")

print(
    "Baseline techniques:",
    BASELINE_TECHNIQUES,
)

print(
    "Random same-seed noise invariant:",
    random_same_seed_noise_invariant,
)

print(
    "Random different seeds differ:",
    random_different_seeds_differ,
)

print(
    "LatestFail responds to verdict noise:",
    latest_fail_responds_to_noise,
)

print(
    "QTF-Avg noise invariant:",
    qtf_noise_invariant,
)

print(
    "QTF-Avg metric identity:",
    qtf_metric_identity,
)

print(
    "Baseline sentinel ranking rows:",
    len(
        baseline_sentinel_rankings
    ),
)

print(
    "Baseline sentinel metric rows:",
    len(
        baseline_sentinel_metrics
    ),
)


print("\nMetric protocol:")

print(
    "Primary metric:",
    "APFDc",
)

print(
    "Secondary metric:",
    "APFD",
)

print(
    "Ranking tie-rule failures:",
    tie_rule_failures,
)

print(
    "Manual metric-validation failures:",
    metric_validation_failures,
)


print("\nEvaluation immutability:")

print(
    "Raw evaluation cohort unchanged:",
    raw_evaluation_unchanged,
)

print(
    "Model evaluation cohort unchanged:",
    model_evaluation_unchanged,
)


print("\nProject immutability:")

print(
    "Project 10 source unchanged:",
    source_unchanged,
)

print(
    "Completion registry unchanged:",
    registry_unchanged,
)

print(
    "Project 9 accessed:",
    False,
)

print(
    "Project 9 write attempted:",
    False,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nModel-protocol checkpoint:")

print(
    MODEL_PROTOCOL_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_hash(
        MODEL_PROTOCOL_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP4A_PASS_STATUS,
)

print("=" * 126)

=== PROJECT 10 CELL 7 / STEP 4A: MODEL, BASELINE AND METRIC PROTOCOL FREEZE ===


AttributeError: 'DataFrame' object has no attribute 'name'

In [10]:
# ============================================================
# PROJECT 10 — CELL 7 / STEP 4A V2
# MODEL, BASELINE, RANKING AND METRIC PROTOCOL FREEZE
#
# Fix:
# - Dataset predictors can contain names such as "Duration".
# - All metadata columns now use reserved __meta__ names.
# - Baseline evaluation data is maintained separately.
#
# This cell:
# - validates all previous Project 10 checkpoints
# - freezes the complete 151-predictor schema
# - freezes clean-training zero-variance predictors
# - freezes preprocessing and deterministic seed rules
# - validates Random, LatestFail and QTF-Avg
# - validates ranking tie handling
# - validates APFD and APFDc
# - writes only Project 10 outputs
#
# It does NOT:
# - fit the four ML models
# - run the 270-condition experiment
# - access or modify Project 9
# - modify Projects 1–8
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import platform
import time
import warnings

import numpy as np
import pandas as pd

import sklearn
import xgboost
import lightgbm

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)


EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_10_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP3B_STATUS = (
    "PASS_PROJECT_10_NOISY_REC_ENGINE_SENTINEL_VALIDATED_AND_FROZEN"
)

STEP4A_PASS_STATUS = (
    "PASS_PROJECT_10_MODEL_PREDICTOR_BASELINE_AND_METRIC_PROTOCOL_FROZEN"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e"
)

EXPECTED_DATASET_ROWS = 8706
EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTOR_COLUMNS = 151

EXPECTED_RAW_TRAINING_ROWS = 34563
EXPECTED_RAW_EVALUATION_ROWS = 12531

EXPECTED_MODEL_TRAINING_ROWS = 6095
EXPECTED_MODEL_EVALUATION_ROWS = 2611
EXPECTED_MODEL_TRAINING_FAILURES = 63
EXPECTED_MODEL_EVALUATION_FAILURES = 213

EXPECTED_EVALUATION_PERIOD_BUILDS = 102
EXPECTED_SCORED_EVALUATION_BUILDS = 27

EXPECTED_BASELINE_SCENARIOS = 7

EXPECTED_BASELINE_RANKING_ROWS = (
    EXPECTED_BASELINE_SCENARIOS
    * EXPECTED_MODEL_EVALUATION_ROWS
)

EXPECTED_BASELINE_METRIC_ROWS = (
    EXPECTED_BASELINE_SCENARIOS
    * EXPECTED_SCORED_EVALUATION_BUILDS
)


REPETITION_SEEDS = list(
    range(1, 31)
)


ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "criterion": "gini",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "bootstrap": True,
        "class_weight": None,
        "n_jobs": -1,
    },

    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
    },

    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "objective": "binary",
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },

    "NaiveBayes": {
        "variant": "GaussianNB",
        "var_smoothing": 1e-9,
    },
}


# Reserved metadata names prevent collisions with predictors.
TRAINING_METADATA_COLUMNS = {
    "ModelRowOrder":
        "__meta__ModelRowOrder",

    "NoiseRowID":
        "__meta__NoiseRowID",

    "BuildKey":
        "__meta__BuildKey",

    "TestKey":
        "__meta__TestKey",

    "Build":
        "__meta__Build",

    "Test":
        "__meta__Test",

    "Verdict":
        "__meta__Verdict",

    "BinaryFailure":
        "__meta__BinaryFailure",
}


EVALUATION_METADATA_COLUMNS = {
    "ModelRowOrder":
        "__meta__ModelRowOrder",

    "EvaluationRawRowID":
        "__meta__EvaluationRawRowID",

    "BuildKey":
        "__meta__BuildKey",

    "TestKey":
        "__meta__TestKey",

    "Build":
        "__meta__Build",

    "Test":
        "__meta__Test",

    "Verdict":
        "__meta__Verdict",

    "BinaryFailure":
        "__meta__BinaryFailure",

    "Duration":
        "__meta__Duration",

    "BuildOrder":
        "__meta__BuildOrder",

    "JobKey":
        "__meta__JobKey",
}


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_noise_plan_checkpoint.json"
)

NOISY_REC_ENGINE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_noisy_rec_engine_checkpoint.json"
)

MODEL_PROTOCOL_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_model_protocol_checkpoint.json"
)


PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)

STEP3B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_status.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_status.json"
)


MODEL_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_model_preflight"
)

CLEAN_MODEL_TRAINING_BASE_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_model_training_base.parquet"
)

CLEAN_MODEL_EVALUATION_BASE_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_model_evaluation_base.parquet"
)

PREDICTOR_MANIFEST_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_predictor_manifest.csv"
)

ZERO_VARIANCE_FEATURES_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_zero_variance_features.csv"
)

CLEAN_MEDIAN_REFERENCE_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_training_median_reference.csv"
)

MODEL_CONFIGURATION_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_configuration.json"
)

MODEL_SEED_MANIFEST_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_seed_manifest.csv"
)

RANDOM_SEED_MANIFEST_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_random_baseline_seed_manifest.csv"
)

BASELINE_SENTINEL_RANKINGS_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_baseline_sentinel_rankings.parquet"
)

BASELINE_SENTINEL_METRICS_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_baseline_sentinel_metrics.csv"
)

BASELINE_PROTOCOL_AUDIT_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_baseline_protocol_audit.csv"
)

RANKING_TIE_VALIDATION_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_ranking_tie_validation.csv"
)

METRIC_VALIDATION_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_metric_validation.csv"
)

STEP4A_VALIDATION_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_report.json"
)


PROJECT_9_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "camunda__camunda-bpm-platform"
)


print("=" * 128)
print("=== PROJECT 10 CELL 7 / STEP 4A V2: MODEL, BASELINE AND METRIC PROTOCOL FREEZE ===")
print("=" * 128)


# ------------------------------------------------------------
# 3. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="snappy",
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json_with_retry(
    path,
    attempts=10,
    delay_seconds=0.5,
):
    path = Path(path)

    last_error = None

    for _ in range(attempts):
        try:
            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:
            last_error = error

            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON.\n"
        f"Path: {path}\n"
        f"Error: {type(last_error).__name__}: {last_error}"
    )


def canonical_identifier(series):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:
        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:
            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


def create_source_root_sha256(
    source_files_payload,
):
    digest = hashlib.sha256()

    for relative_path in sorted(
        source_files_payload
    ):
        metadata = source_files_payload[
            relative_path
        ]

        runtime_path = Path(
            metadata[
                "RuntimePath"
            ]
        )

        size_bytes = int(
            runtime_path.stat().st_size
        )

        file_sha256 = calculate_hash(
            runtime_path
        )

        digest.update(
            (
                f"{relative_path}\0"
                f"{size_bytes}\0"
                f"{file_sha256}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def normalise_build_seed_key(value):
    numeric_value = pd.to_numeric(
        pd.Series(
            [value]
        ),
        errors="coerce",
    ).iloc[0]

    if (
        not pd.isna(
            numeric_value
        )
        and np.isclose(
            numeric_value,
            round(
                numeric_value
            ),
            rtol=0,
            atol=1e-9,
        )
    ):
        return str(
            int(
                round(
                    numeric_value
                )
            )
        )

    return str(
        value
    )


def dataframe_semantic_sha256(
    dataframe,
    columns,
):
    digest = hashlib.sha256()

    for column in columns:
        digest.update(
            str(column).encode(
                "utf-8"
            )
        )

        series = dataframe[
            column
        ]

        if pd.api.types.is_numeric_dtype(
            series
        ):
            values = pd.to_numeric(
                series,
                errors="raise",
            ).to_numpy(
                dtype="<f8"
            )

            digest.update(
                values.tobytes(
                    order="C"
                )
            )

        else:
            for value in (
                series
                .fillna("")
                .astype(str)
            ):
                encoded = value.encode(
                    "utf-8"
                )

                digest.update(
                    len(encoded).to_bytes(
                        8,
                        byteorder="little",
                        signed=False,
                    )
                )

                digest.update(
                    encoded
                )

    return digest.hexdigest()


# ------------------------------------------------------------
# 4. APFD AND APFDC
# ------------------------------------------------------------

def calculate_apfd(actual_failures):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if number_of_tests == 0:
        return np.nan

    if number_of_failures == 0:
        return np.nan

    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )


def calculate_apfdc(
    actual_failures,
    durations,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(failures) != len(durations):
        raise ValueError(
            "Failure and duration arrays must have equal length."
        )

    if len(failures) == 0:
        return np.nan

    if failures.sum() == 0:
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain non-finite values."
        )

    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array(
            [0.0]
        ),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


# ------------------------------------------------------------
# 5. RANKING HELPERS
# ------------------------------------------------------------

def rank_build_rows(
    build_rows,
    scores,
    technique,
    score_direction,
):
    required_columns = [
        "Build",
        "Test",
        "Verdict",
        "Duration",
        "build_order",
        "BuildKey",
        "TestKey",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in build_rows.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Ranking input is missing columns:\n"
            + "\n".join(
                missing_columns
            )
        )

    if build_rows.columns.duplicated().any():
        duplicate_names = (
            build_rows.columns[
                build_rows.columns.duplicated(
                    keep=False
                )
            ]
            .tolist()
        )

        raise RuntimeError(
            "Ranking input contains duplicate columns:\n"
            + "\n".join(
                map(
                    str,
                    duplicate_names,
                )
            )
        )

    ranked = (
        build_rows[
            required_columns
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    score_values = np.asarray(
        scores,
        dtype=float,
    )

    if len(ranked) != len(score_values):
        raise ValueError(
            "Score count does not match build-row count."
        )

    if np.isnan(
        score_values
    ).any():
        raise ValueError(
            "Ranking scores contain NaN values."
        )

    ranked[
        "Technique"
    ] = technique

    ranked[
        "Score"
    ] = score_values

    ranked[
        "ActualFailure"
    ] = (
        pd.to_numeric(
            ranked[
                "Verdict"
            ],
            errors="raise",
        )
        .ne(0)
        .astype(np.int8)
    )

    ranked[
        "Duration"
    ] = pd.to_numeric(
        ranked[
            "Duration"
        ],
        errors="raise",
    ).astype(float)

    test_numeric = pd.to_numeric(
        ranked[
            "Test"
        ],
        errors="coerce",
    )

    if test_numeric.notna().all():
        ranked[
            "__TestSort"
        ] = test_numeric.astype(float)

    else:
        ranked[
            "__TestSort"
        ] = (
            ranked[
                "Test"
            ]
            .fillna("")
            .astype(str)
        )

    if score_direction == "descending":
        score_ascending = False

    elif score_direction == "ascending":
        score_ascending = True

    else:
        raise ValueError(
            "Unknown score direction."
        )

    ranked = (
        ranked.sort_values(
            [
                "Score",
                "__TestSort",
            ],
            ascending=[
                score_ascending,
                True,
            ],
            kind="mergesort",
        )
        .drop(
            columns=[
                "__TestSort",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    ranked[
        "Rank"
    ] = np.arange(
        1,
        len(ranked) + 1,
        dtype=np.int32,
    )

    return ranked


def create_random_rankings(
    evaluation_data,
    repetition_seed,
):
    rankings = []

    for build_key, build_rows in (
        evaluation_data.groupby(
            "BuildKey",
            sort=False,
        )
    ):
        canonical_build = normalise_build_seed_key(
            build_key
        )

        random_seed = stable_project_seed(
            PROJECT_NAME,
            repetition_seed,
            (
                "Random_baseline_build_"
                f"{canonical_build}"
            ),
        )

        random_scores = (
            np.random.default_rng(
                random_seed
            )
            .random(
                len(build_rows)
            )
        )

        rankings.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=random_scores,
                technique="Random",
                score_direction="descending",
            )
        )

    return pd.concat(
        rankings,
        ignore_index=True,
    )


def create_history_baseline_rankings(
    noisy_training_history,
    clean_training_history,
    clean_evaluation_history,
    clean_evaluation_data,
    fixed_split,
):
    latest_failure_order = {}

    for row in (
        noisy_training_history
        .sort_values(
            [
                "BuildOrder",
                "JobKey",
                "TestKey",
            ],
            kind="mergesort",
        )
        .itertuples(
            index=False
        )
    ):
        test_key = str(
            row.TestKey
        )

        if int(
            row.NoisyVerdict
        ) != 0:
            latest_failure_order[
                test_key
            ] = int(
                row.BuildOrder
            )

    duration_sum = {}
    duration_count = {}

    for row in (
        clean_training_history
        .sort_values(
            [
                "BuildOrder",
                "JobKey",
                "TestKey",
            ],
            kind="mergesort",
        )
        .itertuples(
            index=False
        )
    ):
        test_key = str(
            row.TestKey
        )

        duration = float(
            row.Duration
        )

        if np.isfinite(
            duration
        ):
            duration_sum[
                test_key
            ] = (
                duration_sum.get(
                    test_key,
                    0.0,
                )
                + duration
            )

            duration_count[
                test_key
            ] = (
                duration_count.get(
                    test_key,
                    0,
                )
                + 1
            )

    raw_eval_by_build = {
        str(build_key):
            group.copy()

        for build_key, group in (
            clean_evaluation_history.groupby(
                "BuildKey",
                sort=False,
            )
        )
    }

    model_eval_by_build = {
        str(build_key):
            group.copy()

        for build_key, group in (
            clean_evaluation_data.groupby(
                "BuildKey",
                sort=False,
            )
        )
    }

    target_builds = set(
        model_eval_by_build
    )

    latest_rankings = []
    qtf_rankings = []

    ordered_evaluation_builds = (
        fixed_split[
            fixed_split[
                "Partition"
            ].eq(
                "EVALUATION"
            )
        ]
        .sort_values(
            "BuildOrder",
            kind="mergesort",
        )
    )

    for build_row in (
        ordered_evaluation_builds.itertuples(
            index=False
        )
    ):
        build_key = str(
            build_row.BuildKey
        )

        # Rank before observing the current evaluation build.
        if build_key in target_builds:
            build_tests = (
                model_eval_by_build[
                    build_key
                ]
                .copy()
            )

            latest_scores = []
            qtf_scores = []

            for test_key in (
                build_tests[
                    "TestKey"
                ].astype(str)
            ):
                latest_scores.append(
                    float(
                        latest_failure_order.get(
                            test_key,
                            -1,
                        )
                    )
                )

                count = duration_count.get(
                    test_key,
                    0,
                )

                if count > 0:
                    average_duration = (
                        duration_sum[
                            test_key
                        ]
                        / count
                    )

                else:
                    average_duration = np.inf

                qtf_scores.append(
                    float(
                        average_duration
                    )
                )

            latest_rankings.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=latest_scores,
                    technique="LatestFail",
                    score_direction="descending",
                )
            )

            qtf_rankings.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=qtf_scores,
                    technique="QTF-Avg",
                    score_direction="ascending",
                )
            )

        # Update histories only after ranking the build.
        current_raw_rows = raw_eval_by_build.get(
            build_key
        )

        if current_raw_rows is None:
            continue

        for row in (
            current_raw_rows
            .sort_values(
                [
                    "JobKey",
                    "TestKey",
                ],
                kind="mergesort",
            )
            .itertuples(
                index=False
            )
        ):
            test_key = str(
                row.TestKey
            )

            if int(
                row.CleanVerdict
            ) != 0:
                latest_failure_order[
                    test_key
                ] = int(
                    row.BuildOrder
                )

            duration = float(
                row.Duration
            )

            if np.isfinite(
                duration
            ):
                duration_sum[
                    test_key
                ] = (
                    duration_sum.get(
                        test_key,
                        0.0,
                    )
                    + duration
                )

                duration_count[
                    test_key
                ] = (
                    duration_count.get(
                        test_key,
                        0,
                    )
                    + 1
                )

    if not latest_rankings:
        raise RuntimeError(
            "LatestFail produced no rankings."
        )

    if not qtf_rankings:
        raise RuntimeError(
            "QTF-Avg produced no rankings."
        )

    return (
        pd.concat(
            latest_rankings,
            ignore_index=True,
        ),
        pd.concat(
            qtf_rankings,
            ignore_index=True,
        ),
    )


def ranking_semantic_hash(ranking):
    ordered = (
        ranking.sort_values(
            [
                "BuildKey",
                "Rank",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    return dataframe_semantic_sha256(
        ordered,
        [
            "BuildKey",
            "TestKey",
            "Score",
            "Rank",
        ],
    )


def add_scenario(
    ranking,
    scenario,
):
    result = ranking.copy()

    result.insert(
        0,
        "Scenario",
        scenario,
    )

    return result


def calculate_baseline_metrics(
    scenario_rankings,
):
    metric_records = []

    for (
        scenario,
        technique,
        build_key,
    ), ranked_build in (
        scenario_rankings.groupby(
            [
                "Scenario",
                "Technique",
                "BuildKey",
            ],
            sort=False,
        )
    ):
        ranked_build = (
            ranked_build.sort_values(
                "Rank",
                kind="mergesort",
            )
        )

        failures = ranked_build[
            "ActualFailure"
        ].to_numpy(
            dtype=np.int8
        )

        durations = ranked_build[
            "Duration"
        ].to_numpy(
            dtype=float
        )

        metric_records.append({
            "Scenario":
                scenario,

            "Technique":
                technique,

            "BuildKey":
                str(
                    build_key
                ),

            "Build":
                ranked_build[
                    "Build"
                ].iloc[0],

            "BuildOrder":
                int(
                    ranked_build[
                        "build_order"
                    ].iloc[0]
                ),

            "NumberOfTests":
                len(
                    ranked_build
                ),

            "NumberOfFailures":
                int(
                    failures.sum()
                ),

            "TotalDuration":
                float(
                    durations.sum()
                ),

            "APFD":
                calculate_apfd(
                    failures
                ),

            "APFDc":
                calculate_apfdc(
                    failures,
                    durations,
                ),
        })

    return pd.DataFrame(
        metric_records
    )


# ------------------------------------------------------------
# 6. LOAD AND VALIDATE CHECKPOINTS
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    NOISY_REC_ENGINE_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    STEP2B_STATUS_PATH,
    STEP3A_STATUS_PATH,
    STEP3B_STATUS_PATH,
]


missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:
    raise FileNotFoundError(
        "Required Step 4A V2 inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


selection_checkpoint = read_json_with_retry(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint = read_json_with_retry(
    REC_CHECKPOINT_PATH
)

noise_checkpoint = read_json_with_retry(
    NOISE_PLAN_CHECKPOINT_PATH
)

noisy_rec_checkpoint = read_json_with_retry(
    NOISY_REC_ENGINE_CHECKPOINT_PATH
)

step1b_status = read_json_with_retry(
    STEP1B_STATUS_PATH
)

step2b_status = read_json_with_retry(
    STEP2B_STATUS_PATH
)

step3a_status = read_json_with_retry(
    STEP3A_STATUS_PATH
)

step3b_status = read_json_with_retry(
    STEP3B_STATUS_PATH
)


status_expectations = [
    (
        "Step 1B",
        step1b_status.get(
            "Status"
        ),
        EXPECTED_STEP1B_STATUS,
    ),
    (
        "Step 2B",
        step2b_status.get(
            "Status"
        ),
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "Step 3A",
        step3a_status.get(
            "Status"
        ),
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "Step 3B",
        step3b_status.get(
            "Status"
        ),
        EXPECTED_STEP3B_STATUS,
    ),
]


for step_name, actual, expected in (
    status_expectations
):
    if actual != expected:
        raise AssertionError(
            f"{step_name} status differs.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


if rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP2B_STATUS:
    raise AssertionError(
        "REC checkpoint status differs."
    )


if noise_checkpoint.get(
    "Status"
) != EXPECTED_STEP3A_STATUS:
    raise AssertionError(
        "Noise-plan checkpoint status differs."
    )


if noisy_rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP3B_STATUS:
    raise AssertionError(
        "Noisy REC engine checkpoint status differs."
    )


if selection_checkpoint.get(
    "Project"
) != PROJECT_NAME:
    raise AssertionError(
        "Project identity differs."
    )


if selection_checkpoint.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise AssertionError(
        "Project slug differs."
    )


if selection_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Frozen source-root hash differs."
    )


# ------------------------------------------------------------
# 7. OUTPUT-PATH ISOLATION
# ------------------------------------------------------------

output_paths = [
    CLEAN_MODEL_TRAINING_BASE_PATH,
    CLEAN_MODEL_EVALUATION_BASE_PATH,
    PREDICTOR_MANIFEST_PATH,
    ZERO_VARIANCE_FEATURES_PATH,
    CLEAN_MEDIAN_REFERENCE_PATH,
    MODEL_CONFIGURATION_PATH,
    MODEL_SEED_MANIFEST_PATH,
    RANDOM_SEED_MANIFEST_PATH,
    BASELINE_SENTINEL_RANKINGS_PATH,
    BASELINE_SENTINEL_METRICS_PATH,
    BASELINE_PROTOCOL_AUDIT_PATH,
    RANKING_TIE_VALIDATION_PATH,
    METRIC_VALIDATION_PATH,
    STEP4A_VALIDATION_PATH,
    STEP4A_REPORT_PATH,
    MODEL_PROTOCOL_CHECKPOINT_PATH,
    STEP4A_STATUS_PATH,
]


for output_path in output_paths:
    output_string = str(
        output_path
    )

    if (
        PROJECT_SLUG not in output_string
        and "project_10_" not in output_string
    ):
        raise AssertionError(
            "A Step 4A V2 output path is not Project 10 isolated.\n"
            f"Path: {output_path}"
        )

    if str(
        PROJECT_9_DIR
    ) in output_string:
        raise AssertionError(
            "A Project 10 output path overlaps Project 9."
        )


# ------------------------------------------------------------
# 8. REGISTRY AND SOURCE — READ ONLY
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != 8
    or set(
        registry_project_numbers
    ) != set(
        range(1, 9)
    )
):
    raise AssertionError(
        "Registry must contain exactly Projects 1–8."
    )


if registry_project_numbers.eq(9).any():
    raise AssertionError(
        "Project 9 was unexpectedly registered."
    )


if registry_project_numbers.eq(10).any():
    raise AssertionError(
        "Project 10 was unexpectedly registered."
    )


source_files_payload = (
    selection_checkpoint[
        "SourceFiles"
    ]
)

source_root_before = create_source_root_sha256(
    source_files_payload
)


if source_root_before != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Project 10 source differs before Step 4A V2."
    )


# ------------------------------------------------------------
# 9. LOAD FROZEN DATA
# ------------------------------------------------------------

dataset_path = Path(
    selection_checkpoint[
        "DatasetPath"
    ]
)

fixed_split_path = Path(
    selection_checkpoint[
        "FixedSplit"
    ]
)

raw_training_path = Path(
    noise_checkpoint[
        "RawTrainingCohort"
    ]
)

raw_evaluation_path = Path(
    noise_checkpoint[
        "RawEvaluationCohort"
    ]
)

model_training_path = Path(
    noise_checkpoint[
        "ModelTrainingCohort"
    ]
)

model_evaluation_path = Path(
    noise_checkpoint[
        "ModelEvaluationCohort"
    ]
)

noise_rng_manifest_path = Path(
    noise_checkpoint[
        "NoiseRNGManifest"
    ]
)

condition_plan_path = Path(
    noise_checkpoint[
        "ConditionPlan"
    ]
)


input_paths = [
    dataset_path,
    fixed_split_path,
    raw_training_path,
    raw_evaluation_path,
    model_training_path,
    model_evaluation_path,
    noise_rng_manifest_path,
    condition_plan_path,
]


missing_frozen_inputs = [
    str(path)
    for path in input_paths
    if not path.exists()
]


if missing_frozen_inputs:
    raise FileNotFoundError(
        "Frozen Step 4A V2 inputs are missing:\n"
        + "\n".join(
            missing_frozen_inputs
        )
    )


raw_evaluation_sha256_before = calculate_hash(
    raw_evaluation_path
)

model_evaluation_sha256_before = calculate_hash(
    model_evaluation_path
)


dataset = pd.read_csv(
    dataset_path,
    low_memory=False,
)

fixed_split = pd.read_csv(
    fixed_split_path,
    low_memory=False,
)

raw_training = pd.read_parquet(
    raw_training_path
)

raw_evaluation = pd.read_parquet(
    raw_evaluation_path
)

model_training = pd.read_parquet(
    model_training_path
)

model_evaluation = pd.read_parquet(
    model_evaluation_path
)

noise_rng_manifest = pd.read_parquet(
    noise_rng_manifest_path
)

condition_plan = pd.read_csv(
    condition_plan_path,
    low_memory=False,
)


if dataset.columns.duplicated().any():
    duplicate_dataset_columns = (
        dataset.columns[
            dataset.columns.duplicated(
                keep=False
            )
        ]
        .tolist()
    )

    raise RuntimeError(
        "dataset.csv contains duplicate column names:\n"
        + "\n".join(
            map(
                str,
                duplicate_dataset_columns,
            )
        )
    )


# ------------------------------------------------------------
# 10. CANONICALISE FROZEN DATA
# ------------------------------------------------------------

resolved_columns = (
    rec_checkpoint[
        "ResolvedColumns"
    ]
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)


fixed_split[
    "BuildKey"
] = fixed_split[
    "BuildKey"
].astype(str)

fixed_split[
    "BuildOrder"
] = pd.to_numeric(
    fixed_split[
        "BuildOrder"
    ],
    errors="raise",
).astype(np.int32)


for raw_frame in [
    raw_training,
    raw_evaluation,
]:
    raw_frame[
        "BuildKey"
    ] = raw_frame[
        "BuildKey"
    ].astype(str)

    raw_frame[
        "JobKey"
    ] = raw_frame[
        "JobKey"
    ].astype(str)

    raw_frame[
        "TestKey"
    ] = raw_frame[
        "TestKey"
    ].astype(str)

    raw_frame[
        "BuildOrder"
    ] = pd.to_numeric(
        raw_frame[
            "BuildOrder"
        ],
        errors="raise",
    ).astype(np.int32)

    raw_frame[
        "CleanVerdict"
    ] = pd.to_numeric(
        raw_frame[
            "CleanVerdict"
        ],
        errors="raise",
    ).astype(np.int8)

    raw_frame[
        "Duration"
    ] = pd.to_numeric(
        raw_frame[
            "Duration"
        ],
        errors="raise",
    ).astype(float)


raw_training[
    "NoiseRowID"
] = pd.to_numeric(
    raw_training[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)

raw_evaluation[
    "EvaluationRawRowID"
] = pd.to_numeric(
    raw_evaluation[
        "EvaluationRawRowID"
    ],
    errors="raise",
).astype(np.int32)


model_training = (
    model_training
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

model_evaluation = (
    model_evaluation
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


for model_frame in [
    model_training,
    model_evaluation,
]:
    model_frame[
        "BuildKey"
    ] = model_frame[
        "BuildKey"
    ].astype(str)

    model_frame[
        "TestKey"
    ] = model_frame[
        "TestKey"
    ].astype(str)

    model_frame[
        "ModelRowOrder"
    ] = pd.to_numeric(
        model_frame[
            "ModelRowOrder"
        ],
        errors="raise",
    ).astype(np.int64)

    model_frame[
        "CleanVerdict"
    ] = pd.to_numeric(
        model_frame[
            "CleanVerdict"
        ],
        errors="raise",
    ).astype(np.int8)


model_training[
    "NoiseRowID"
] = pd.to_numeric(
    model_training[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)

model_evaluation[
    "EvaluationRawRowID"
] = pd.to_numeric(
    model_evaluation[
        "EvaluationRawRowID"
    ],
    errors="raise",
).astype(np.int32)


noise_rng_manifest[
    "RepetitionSeed"
] = pd.to_numeric(
    noise_rng_manifest[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(np.int16)

noise_rng_manifest[
    "NoiseRowID"
] = pd.to_numeric(
    noise_rng_manifest[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)

noise_rng_manifest[
    "FlipUniform"
] = pd.to_numeric(
    noise_rng_manifest[
        "FlipUniform"
    ],
    errors="raise",
).astype(float)

noise_rng_manifest[
    "SampledFailureSubtype"
] = pd.to_numeric(
    noise_rng_manifest[
        "SampledFailureSubtype"
    ],
    errors="raise",
).astype(np.int8)


condition_plan[
    "NoisePercent"
] = pd.to_numeric(
    condition_plan[
        "NoisePercent"
    ],
    errors="raise",
).astype(int)

condition_plan[
    "RepetitionSeed"
] = pd.to_numeric(
    condition_plan[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(int)


# ------------------------------------------------------------
# 11. PREDICTOR SCHEMA
# ------------------------------------------------------------

identifier_columns = [
    dataset_build_column,
    dataset_test_column,
    dataset_verdict_column,
]


model_predictor_columns = [
    column
    for column in dataset.columns
    if column not in identifier_columns
]


if len(dataset) != EXPECTED_DATASET_ROWS:
    raise AssertionError(
        "Dataset row count differs."
    )


if len(dataset.columns) != EXPECTED_DATASET_COLUMNS:
    raise AssertionError(
        "Dataset column count differs."
    )


if len(
    model_predictor_columns
) != EXPECTED_PREDICTOR_COLUMNS:
    raise AssertionError(
        "Predictor count differs.\n"
        f"Expected: {EXPECTED_PREDICTOR_COLUMNS}\n"
        f"Actual:   {len(model_predictor_columns)}"
    )


missing_rec_features = [
    feature
    for feature in REC_FEATURE_COLUMNS
    if feature not in model_predictor_columns
]


if missing_rec_features:
    raise AssertionError(
        "REC predictors are missing:\n"
        + "\n".join(
            missing_rec_features
        )
    )


reserved_metadata_names = set(
    TRAINING_METADATA_COLUMNS.values()
) | set(
    EVALUATION_METADATA_COLUMNS.values()
)


metadata_predictor_collisions = sorted(
    set(
        model_predictor_columns
    )
    & reserved_metadata_names
)


if metadata_predictor_collisions:
    raise RuntimeError(
        "Predictors collide with reserved metadata names:\n"
        + "\n".join(
            metadata_predictor_collisions
        )
    )


# ------------------------------------------------------------
# 12. BUILD CLEAN TRAINING BASE
# ------------------------------------------------------------

training_model_orders = model_training[
    "ModelRowOrder"
].to_numpy(
    dtype=np.int64
)

training_dataset_rows = (
    dataset.iloc[
        training_model_orders
    ]
    .reset_index(
        drop=True
    )
)


training_build_keys = canonical_identifier(
    training_dataset_rows[
        dataset_build_column
    ]
).astype(str)

training_test_keys = canonical_identifier(
    training_dataset_rows[
        dataset_test_column
    ]
).astype(str)


if not np.array_equal(
    training_build_keys.to_numpy(),
    model_training[
        "BuildKey"
    ].to_numpy(),
):
    raise AssertionError(
        "Training BuildKey alignment differs."
    )


if not np.array_equal(
    training_test_keys.to_numpy(),
    model_training[
        "TestKey"
    ].to_numpy(),
):
    raise AssertionError(
        "Training TestKey alignment differs."
    )


training_verdicts = pd.to_numeric(
    training_dataset_rows[
        dataset_verdict_column
    ],
    errors="raise",
).astype(np.int8)


training_verdict_mismatches = int(
    training_verdicts.ne(
        model_training[
            "CleanVerdict"
        ]
    ).sum()
)


training_metadata = pd.DataFrame({
    TRAINING_METADATA_COLUMNS[
        "ModelRowOrder"
    ]:
        model_training[
            "ModelRowOrder"
        ].to_numpy(
            dtype=np.int64
        ),

    TRAINING_METADATA_COLUMNS[
        "NoiseRowID"
    ]:
        model_training[
            "NoiseRowID"
        ].to_numpy(
            dtype=np.int32
        ),

    TRAINING_METADATA_COLUMNS[
        "BuildKey"
    ]:
        model_training[
            "BuildKey"
        ].astype(str).to_numpy(),

    TRAINING_METADATA_COLUMNS[
        "TestKey"
    ]:
        model_training[
            "TestKey"
        ].astype(str).to_numpy(),

    TRAINING_METADATA_COLUMNS[
        "Build"
    ]:
        training_dataset_rows[
            dataset_build_column
        ].to_numpy(),

    TRAINING_METADATA_COLUMNS[
        "Test"
    ]:
        training_dataset_rows[
            dataset_test_column
        ].to_numpy(),

    TRAINING_METADATA_COLUMNS[
        "Verdict"
    ]:
        training_verdicts.to_numpy(
            dtype=np.int8
        ),

    TRAINING_METADATA_COLUMNS[
        "BinaryFailure"
    ]:
        training_verdicts.ne(0).astype(
            np.int8
        ).to_numpy(),
})


clean_model_training_base = pd.concat(
    [
        training_metadata,
        training_dataset_rows[
            model_predictor_columns
        ].reset_index(
            drop=True
        ),
    ],
    axis=1,
)


# ------------------------------------------------------------
# 13. BUILD CLEAN EVALUATION BASE
# ------------------------------------------------------------

evaluation_model_orders = model_evaluation[
    "ModelRowOrder"
].to_numpy(
    dtype=np.int64
)

evaluation_dataset_rows = (
    dataset.iloc[
        evaluation_model_orders
    ]
    .reset_index(
        drop=True
    )
)


evaluation_build_keys = canonical_identifier(
    evaluation_dataset_rows[
        dataset_build_column
    ]
).astype(str)

evaluation_test_keys = canonical_identifier(
    evaluation_dataset_rows[
        dataset_test_column
    ]
).astype(str)


if not np.array_equal(
    evaluation_build_keys.to_numpy(),
    model_evaluation[
        "BuildKey"
    ].to_numpy(),
):
    raise AssertionError(
        "Evaluation BuildKey alignment differs."
    )


if not np.array_equal(
    evaluation_test_keys.to_numpy(),
    model_evaluation[
        "TestKey"
    ].to_numpy(),
):
    raise AssertionError(
        "Evaluation TestKey alignment differs."
    )


evaluation_verdicts = pd.to_numeric(
    evaluation_dataset_rows[
        dataset_verdict_column
    ],
    errors="raise",
).astype(np.int8)


evaluation_verdict_mismatches = int(
    evaluation_verdicts.ne(
        model_evaluation[
            "CleanVerdict"
        ]
    ).sum()
)


evaluation_raw_lookup = raw_evaluation[
    [
        "EvaluationRawRowID",
        "BuildKey",
        "TestKey",
        "JobKey",
        "BuildOrder",
        "CleanVerdict",
        "Duration",
    ]
].copy()


evaluation_link = (
    model_evaluation[
        [
            "ModelRowOrder",
            "EvaluationRawRowID",
            "BuildKey",
            "TestKey",
            "CleanVerdict",
        ]
    ]
    .merge(
        evaluation_raw_lookup,
        on=[
            "EvaluationRawRowID",
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        suffixes=(
            "_Model",
            "_Raw",
        ),
        indicator=True,
        sort=False,
    )
    .sort_values(
        "ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_evaluation_raw_links = int(
    evaluation_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


evaluation_raw_verdict_mismatches = int(
    evaluation_link.loc[
        evaluation_link[
            "_merge"
        ].eq(
            "both"
        ),
        "CleanVerdict_Model",
    ].ne(
        evaluation_link.loc[
            evaluation_link[
                "_merge"
            ].eq(
                "both"
            ),
            "CleanVerdict_Raw",
        ]
    ).sum()
)


evaluation_metadata = pd.DataFrame({
    EVALUATION_METADATA_COLUMNS[
        "ModelRowOrder"
    ]:
        model_evaluation[
            "ModelRowOrder"
        ].to_numpy(
            dtype=np.int64
        ),

    EVALUATION_METADATA_COLUMNS[
        "EvaluationRawRowID"
    ]:
        model_evaluation[
            "EvaluationRawRowID"
        ].to_numpy(
            dtype=np.int32
        ),

    EVALUATION_METADATA_COLUMNS[
        "BuildKey"
    ]:
        model_evaluation[
            "BuildKey"
        ].astype(str).to_numpy(),

    EVALUATION_METADATA_COLUMNS[
        "TestKey"
    ]:
        model_evaluation[
            "TestKey"
        ].astype(str).to_numpy(),

    EVALUATION_METADATA_COLUMNS[
        "Build"
    ]:
        evaluation_dataset_rows[
            dataset_build_column
        ].to_numpy(),

    EVALUATION_METADATA_COLUMNS[
        "Test"
    ]:
        evaluation_dataset_rows[
            dataset_test_column
        ].to_numpy(),

    EVALUATION_METADATA_COLUMNS[
        "Verdict"
    ]:
        evaluation_verdicts.to_numpy(
            dtype=np.int8
        ),

    EVALUATION_METADATA_COLUMNS[
        "BinaryFailure"
    ]:
        evaluation_verdicts.ne(0).astype(
            np.int8
        ).to_numpy(),

    EVALUATION_METADATA_COLUMNS[
        "Duration"
    ]:
        evaluation_link[
            "Duration"
        ].to_numpy(
            dtype=float
        ),

    EVALUATION_METADATA_COLUMNS[
        "BuildOrder"
    ]:
        evaluation_link[
            "BuildOrder"
        ].to_numpy(
            dtype=np.int32
        ),

    EVALUATION_METADATA_COLUMNS[
        "JobKey"
    ]:
        evaluation_link[
            "JobKey"
        ].astype(str).to_numpy(),
})


clean_model_evaluation_base = pd.concat(
    [
        evaluation_metadata,
        evaluation_dataset_rows[
            model_predictor_columns
        ].reset_index(
            drop=True
        ),
    ],
    axis=1,
)


training_base_duplicate_columns = int(
    clean_model_training_base.columns.duplicated().sum()
)

evaluation_base_duplicate_columns = int(
    clean_model_evaluation_base.columns.duplicated().sum()
)


if training_base_duplicate_columns:
    raise RuntimeError(
        "Clean training base contains duplicate column names."
    )


if evaluation_base_duplicate_columns:
    raise RuntimeError(
        "Clean evaluation base contains duplicate column names."
    )


# Separate collision-free frame used only by baselines.
baseline_evaluation_data = pd.DataFrame({
    "Build":
        evaluation_dataset_rows[
            dataset_build_column
        ].to_numpy(),

    "Test":
        evaluation_dataset_rows[
            dataset_test_column
        ].to_numpy(),

    "Verdict":
        evaluation_verdicts.to_numpy(
            dtype=np.int8
        ),

    "Duration":
        evaluation_link[
            "Duration"
        ].to_numpy(
            dtype=float
        ),

    "build_order":
        evaluation_link[
            "BuildOrder"
        ].to_numpy(
            dtype=np.int32
        ),

    "BuildKey":
        model_evaluation[
            "BuildKey"
        ].astype(str).to_numpy(),

    "TestKey":
        model_evaluation[
            "TestKey"
        ].astype(str).to_numpy(),
})


if baseline_evaluation_data.columns.duplicated().any():
    raise RuntimeError(
        "Baseline evaluation frame contains duplicate columns."
    )


# ------------------------------------------------------------
# 14. MODEL COHORT AUDIT
# ------------------------------------------------------------

training_rows = len(
    clean_model_training_base
)

evaluation_rows = len(
    clean_model_evaluation_base
)

training_failures = int(
    training_metadata[
        TRAINING_METADATA_COLUMNS[
            "BinaryFailure"
        ]
    ].sum()
)

evaluation_failures = int(
    evaluation_metadata[
        EVALUATION_METADATA_COLUMNS[
            "BinaryFailure"
        ]
    ].sum()
)


evaluation_period_builds = int(
    fixed_split.loc[
        fixed_split[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildKey",
    ].nunique()
)


scored_evaluation_builds = int(
    baseline_evaluation_data[
        "BuildKey"
    ].nunique()
)


duplicate_training_build_test_rows = int(
    training_metadata.duplicated(
        subset=[
            TRAINING_METADATA_COLUMNS[
                "BuildKey"
            ],
            TRAINING_METADATA_COLUMNS[
                "TestKey"
            ],
        ],
        keep=False,
    ).sum()
)


duplicate_evaluation_build_test_rows = int(
    baseline_evaluation_data.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


evaluation_build_failure_profile = (
    baseline_evaluation_data.groupby(
        [
            "BuildKey",
            "build_order",
        ],
        as_index=False,
    )
    .agg(
        Tests=(
            "TestKey",
            "count",
        ),

        Failures=(
            "Verdict",
            lambda values:
                int(
                    pd.to_numeric(
                        values,
                        errors="raise",
                    ).ne(0).sum()
                ),
        ),

        TotalDuration=(
            "Duration",
            "sum",
        ),
    )
)


evaluation_builds_without_failures = int(
    evaluation_build_failure_profile[
        "Failures"
    ].eq(0).sum()
)


evaluation_builds_with_nonpositive_duration = int(
    evaluation_build_failure_profile[
        "TotalDuration"
    ].le(0).sum()
)


# ------------------------------------------------------------
# 15. ZERO-VARIANCE AND ACTIVE FEATURES
# ------------------------------------------------------------

clean_training_feature_frame = (
    training_dataset_rows[
        model_predictor_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


feature_variation = (
    clean_training_feature_frame.nunique(
        dropna=False
    )
)


zero_variance_features = (
    feature_variation[
        feature_variation <= 1
    ]
    .index
    .tolist()
)


active_feature_columns = [
    feature
    for feature in model_predictor_columns
    if feature not in zero_variance_features
]


if not active_feature_columns:
    raise AssertionError(
        "No active predictors remain."
    )


predictor_manifest_records = []


for predictor_order, predictor in enumerate(
    model_predictor_columns,
    start=1,
):
    if predictor in VERDICT_DEPENDENT_REC_FEATURES:
        predictor_class = (
            "REC_VERDICT_DEPENDENT"
        )

    elif predictor in VERDICT_INDEPENDENT_REC_FEATURES:
        predictor_class = (
            "REC_VERDICT_INDEPENDENT"
        )

    else:
        predictor_class = "NON_REC"

    predictor_manifest_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            predictor,

        "PredictorClass":
            predictor_class,

        "IsREC":
            predictor in REC_FEATURE_COLUMNS,

        "VerdictDependentREC":
            predictor in VERDICT_DEPENDENT_REC_FEATURES,

        "VerdictIndependentREC":
            predictor in VERDICT_INDEPENDENT_REC_FEATURES,

        "CleanTrainingDistinctValues":
            int(
                feature_variation[
                    predictor
                ]
            ),

        "ZeroVariance":
            predictor in zero_variance_features,

        "Active":
            predictor in active_feature_columns,
    })


predictor_manifest = pd.DataFrame(
    predictor_manifest_records
)


zero_variance_features_frame = pd.DataFrame({
    "Feature":
        zero_variance_features,
})


# ------------------------------------------------------------
# 16. PREPROCESSING VALIDATION
# ------------------------------------------------------------

X_train_clean = (
    training_dataset_rows[
        active_feature_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


X_evaluation_clean = (
    evaluation_dataset_rows[
        active_feature_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


clean_training_medians = (
    X_train_clean.median(
        axis=0
    )
    .fillna(
        0.0
    )
)


X_train_clean_imputed = (
    X_train_clean
    .fillna(
        clean_training_medians
    )
    .astype(float)
)


X_evaluation_clean_imputed = (
    X_evaluation_clean
    .fillna(
        clean_training_medians
    )
    .astype(float)
)


training_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            X_train_clean_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


evaluation_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            X_evaluation_clean_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


clean_training_label_classes = sorted(
    training_verdicts.ne(0)
    .astype(np.int8)
    .unique()
    .tolist()
)


clean_median_reference = pd.DataFrame({
    "Feature":
        active_feature_columns,

    "CleanTrainingMedianReference":
        [
            float(
                clean_training_medians[
                    feature
                ]
            )
            for feature in active_feature_columns
        ],
})


# ------------------------------------------------------------
# 17. MODEL AND RANDOM SEEDS
# ------------------------------------------------------------

model_seed_records = []


for repetition_seed in REPETITION_SEEDS:
    model_seed_records.append({
        "RepetitionSeed":
            repetition_seed,

        "RandomForestSeed":
            stable_project_seed(
                PROJECT_NAME,
                repetition_seed,
                "RandomForest_model",
            ),

        "XGBoostSeed":
            stable_project_seed(
                PROJECT_NAME,
                repetition_seed,
                "XGBoost_model",
            ),

        "LightGBMSeed":
            stable_project_seed(
                PROJECT_NAME,
                repetition_seed,
                "LightGBM_model",
            ),
    })


model_seed_manifest = pd.DataFrame(
    model_seed_records
)


evaluation_build_keys = (
    baseline_evaluation_data[
        [
            "BuildKey",
            "build_order",
        ]
    ]
    .drop_duplicates(
        subset=[
            "BuildKey",
        ]
    )
    .sort_values(
        "build_order",
        kind="mergesort",
    )
)


random_seed_records = []


for repetition_seed in REPETITION_SEEDS:
    for row in evaluation_build_keys.itertuples(
        index=False
    ):
        build_key = str(
            row.BuildKey
        )

        random_seed_records.append({
            "RepetitionSeed":
                repetition_seed,

            "BuildKey":
                build_key,

            "BuildOrder":
                int(
                    row.build_order
                ),

            "RandomSeed":
                stable_project_seed(
                    PROJECT_NAME,
                    repetition_seed,
                    (
                        "Random_baseline_build_"
                        f"{normalise_build_seed_key(build_key)}"
                    ),
                ),
        })


random_seed_manifest = pd.DataFrame(
    random_seed_records
)


# ------------------------------------------------------------
# 18. INSTANTIATE MODEL CONFIGURATIONS
# ------------------------------------------------------------

seed_one = model_seed_manifest[
    model_seed_manifest[
        "RepetitionSeed"
    ].eq(1)
].iloc[0]


model_instances = {
    "RandomForest":
        RandomForestClassifier(
            **MODEL_CONFIG[
                "RandomForest"
            ],
            random_state=int(
                seed_one[
                    "RandomForestSeed"
                ]
            ),
        ),

    "XGBoost":
        XGBClassifier(
            **MODEL_CONFIG[
                "XGBoost"
            ],
            random_state=int(
                seed_one[
                    "XGBoostSeed"
                ]
            ),
        ),

    "LightGBM":
        LGBMClassifier(
            **MODEL_CONFIG[
                "LightGBM"
            ],
            random_state=int(
                seed_one[
                    "LightGBMSeed"
                ]
            ),
        ),

    "NaiveBayes":
        GaussianNB(
            var_smoothing=(
                MODEL_CONFIG[
                    "NaiveBayes"
                ][
                    "var_smoothing"
                ]
            )
        ),
}


instantiated_model_classes = {
    technique:
        type(model).__name__

    for technique, model in (
        model_instances.items()
    )
}


# ------------------------------------------------------------
# 19. CREATE CLEAN AND 50% TRAINING HISTORIES
# ------------------------------------------------------------

clean_raw_training_verdicts = raw_training[
    "CleanVerdict"
].to_numpy(
    dtype=np.int8
)


zero_noisy_training_history = raw_training.copy()

zero_noisy_training_history[
    "NoisyVerdict"
] = clean_raw_training_verdicts


seed_one_rng = (
    noise_rng_manifest[
        noise_rng_manifest[
            "RepetitionSeed"
        ].eq(1)
    ]
    .sort_values(
        "NoiseRowID",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    seed_one_rng
) != EXPECTED_RAW_TRAINING_ROWS:
    raise AssertionError(
        "Seed-1 RNG row count differs."
    )


flip_mask_50 = (
    seed_one_rng[
        "FlipUniform"
    ].to_numpy(
        dtype=float
    )
    < 0.50
)


sampled_failure_subtypes = seed_one_rng[
    "SampledFailureSubtype"
].to_numpy(
    dtype=np.int8
)


noisy_raw_verdicts_50 = (
    clean_raw_training_verdicts.copy()
)


pass_to_failure_50 = (
    flip_mask_50
    & (
        clean_raw_training_verdicts
        == 0
    )
)

failure_to_pass_50 = (
    flip_mask_50
    & (
        clean_raw_training_verdicts
        != 0
    )
)


noisy_raw_verdicts_50[
    pass_to_failure_50
] = sampled_failure_subtypes[
    pass_to_failure_50
]

noisy_raw_verdicts_50[
    failure_to_pass_50
] = 0


condition_50_seed_1 = condition_plan[
    condition_plan[
        "NoisePercent"
    ].eq(50)
    & condition_plan[
        "RepetitionSeed"
    ].eq(1)
]


if len(
    condition_50_seed_1
) != 1:
    raise AssertionError(
        "50% seed-1 condition was not found exactly once."
    )


expected_50_raw_changes = int(
    condition_50_seed_1.iloc[0][
        "RawLabelChanges"
    ]
)

actual_50_raw_changes = int(
    np.count_nonzero(
        noisy_raw_verdicts_50
        != clean_raw_training_verdicts
    )
)


if actual_50_raw_changes != expected_50_raw_changes:
    raise AssertionError(
        "50% seed-1 raw-label change count differs."
    )


fifty_noisy_training_history = raw_training.copy()

fifty_noisy_training_history[
    "NoisyVerdict"
] = noisy_raw_verdicts_50


# ------------------------------------------------------------
# 20. BASELINE SENTINEL RUNS
# ------------------------------------------------------------

random_seed_1_noise_0 = create_random_rankings(
    baseline_evaluation_data,
    repetition_seed=1,
)

random_seed_1_noise_50 = create_random_rankings(
    baseline_evaluation_data,
    repetition_seed=1,
)

random_seed_30_noise_0 = create_random_rankings(
    baseline_evaluation_data,
    repetition_seed=30,
)


(
    latest_fail_noise_0,
    qtf_noise_0,
) = create_history_baseline_rankings(
    noisy_training_history=(
        zero_noisy_training_history
    ),

    clean_training_history=(
        raw_training
    ),

    clean_evaluation_history=(
        raw_evaluation
    ),

    clean_evaluation_data=(
        baseline_evaluation_data
    ),

    fixed_split=(
        fixed_split
    ),
)


(
    latest_fail_noise_50,
    qtf_noise_50,
) = create_history_baseline_rankings(
    noisy_training_history=(
        fifty_noisy_training_history
    ),

    clean_training_history=(
        raw_training
    ),

    clean_evaluation_history=(
        raw_evaluation
    ),

    clean_evaluation_data=(
        baseline_evaluation_data
    ),

    fixed_split=(
        fixed_split
    ),
)


baseline_ranking_objects = {
    "Random_seed01_noise00":
        random_seed_1_noise_0,

    "Random_seed01_noise50":
        random_seed_1_noise_50,

    "Random_seed30_noise00":
        random_seed_30_noise_0,

    "LatestFail_noise00_seed01":
        latest_fail_noise_0,

    "LatestFail_noise50_seed01":
        latest_fail_noise_50,

    "QTF-Avg_noise00":
        qtf_noise_0,

    "QTF-Avg_noise50":
        qtf_noise_50,
}


baseline_sentinel_rankings = pd.concat(
    [
        add_scenario(
            ranking,
            scenario,
        )
        for scenario, ranking in (
            baseline_ranking_objects.items()
        )
    ],
    ignore_index=True,
)


baseline_sentinel_metrics = calculate_baseline_metrics(
    baseline_sentinel_rankings
)


baseline_scenario_counts = (
    baseline_sentinel_rankings.groupby(
        "Scenario"
    ).size()
)


baseline_scenarios_with_wrong_rows = int(
    baseline_scenario_counts.ne(
        EXPECTED_MODEL_EVALUATION_ROWS
    ).sum()
)


random_seed_1_hash_0 = ranking_semantic_hash(
    random_seed_1_noise_0
)

random_seed_1_hash_50 = ranking_semantic_hash(
    random_seed_1_noise_50
)

random_seed_30_hash = ranking_semantic_hash(
    random_seed_30_noise_0
)

latest_fail_hash_0 = ranking_semantic_hash(
    latest_fail_noise_0
)

latest_fail_hash_50 = ranking_semantic_hash(
    latest_fail_noise_50
)

qtf_hash_0 = ranking_semantic_hash(
    qtf_noise_0
)

qtf_hash_50 = ranking_semantic_hash(
    qtf_noise_50
)


random_same_seed_noise_invariant = bool(
    random_seed_1_hash_0
    == random_seed_1_hash_50
)

random_different_seeds_differ = bool(
    random_seed_1_hash_0
    != random_seed_30_hash
)

latest_fail_responds_to_noise = bool(
    latest_fail_hash_0
    != latest_fail_hash_50
)

qtf_noise_invariant = bool(
    qtf_hash_0
    == qtf_hash_50
)


qtf_metrics_0 = (
    baseline_sentinel_metrics[
        baseline_sentinel_metrics[
            "Scenario"
        ].eq(
            "QTF-Avg_noise00"
        )
    ]
    .sort_values(
        "BuildKey",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


qtf_metrics_50 = (
    baseline_sentinel_metrics[
        baseline_sentinel_metrics[
            "Scenario"
        ].eq(
            "QTF-Avg_noise50"
        )
    ]
    .sort_values(
        "BuildKey",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


qtf_metric_identity = bool(
    np.array_equal(
        qtf_metrics_0[
            [
                "APFD",
                "APFDc",
            ]
        ].to_numpy(
            dtype=float
        ),
        qtf_metrics_50[
            [
                "APFD",
                "APFDc",
            ]
        ].to_numpy(
            dtype=float
        ),
    )
)


baseline_metric_nan_values = int(
    baseline_sentinel_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].isna().sum().sum()
)


baseline_metric_out_of_range_values = int(
    (
        ~baseline_sentinel_metrics[
            "APFD"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
    +
    (
        ~baseline_sentinel_metrics[
            "APFDc"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
)


baseline_protocol_audit = pd.DataFrame([
    {
        "Technique":
            "Random",

        "NoiseAffected":
            False,

        "ScoreDirection":
            "DESCENDING",

        "TrainingHistory":
            "None",

        "EvaluationUpdate":
            "None",

        "TestTieBreak":
            "Test ascending",

        "ValidationPass":
            (
                random_same_seed_noise_invariant
                and random_different_seeds_differ
            ),
    },

    {
        "Technique":
            "LatestFail",

        "NoiseAffected":
            True,

        "ScoreDirection":
            "DESCENDING",

        "TrainingHistory":
            "Noisy raw training verdict history",

        "EvaluationUpdate":
            (
                "Clean evaluation verdicts after ranking "
                "the current build"
            ),

        "TestTieBreak":
            "Test ascending",

        "ValidationPass":
            latest_fail_responds_to_noise,
    },

    {
        "Technique":
            "QTF-Avg",

        "NoiseAffected":
            False,

        "ScoreDirection":
            "ASCENDING",

        "TrainingHistory":
            "Clean raw duration history",

        "EvaluationUpdate":
            (
                "Clean evaluation durations after ranking "
                "the current build"
            ),

        "TestTieBreak":
            "Test ascending",

        "ValidationPass":
            (
                qtf_noise_invariant
                and qtf_metric_identity
            ),
    },
])


# ------------------------------------------------------------
# 21. RANKING TIE VALIDATION
# ------------------------------------------------------------

synthetic_build = pd.DataFrame({
    "Build":
        [1, 1, 1],

    "Test":
        [3, 1, 2],

    "Verdict":
        [0, 1, 0],

    "Duration":
        [1.0, 1.0, 1.0],

    "build_order":
        [1, 1, 1],

    "BuildKey":
        ["1", "1", "1"],

    "TestKey":
        ["3", "1", "2"],
})


descending_tie_result = rank_build_rows(
    build_rows=synthetic_build,
    scores=[
        0.5,
        0.5,
        0.8,
    ],
    technique="SyntheticDescending",
    score_direction="descending",
)


ascending_tie_result = rank_build_rows(
    build_rows=synthetic_build,
    scores=[
        0.5,
        0.5,
        0.2,
    ],
    technique="SyntheticAscending",
    score_direction="ascending",
)


expected_tie_order = [
    2,
    1,
    3,
]


descending_test_order = (
    descending_tie_result[
        "Test"
    ].astype(int).tolist()
)

ascending_test_order = (
    ascending_tie_result[
        "Test"
    ].astype(int).tolist()
)


ranking_tie_validation = pd.DataFrame([
    {
        "Example":
            "Descending score with Test ascending tie-break",

        "ExpectedTestOrder":
            str(
                expected_tie_order
            ),

        "ActualTestOrder":
            str(
                descending_test_order
            ),

        "Pass":
            descending_test_order
            == expected_tie_order,
    },

    {
        "Example":
            "Ascending score with Test ascending tie-break",

        "ExpectedTestOrder":
            str(
                expected_tie_order
            ),

        "ActualTestOrder":
            str(
                ascending_test_order
            ),

        "Pass":
            ascending_test_order
            == expected_tie_order,
    },
])


tie_rule_failures = int(
    (
        ~ranking_tie_validation[
            "Pass"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 22. MANUAL METRIC VALIDATION
# ------------------------------------------------------------

metric_examples = [
    {
        "Example":
            "Failures first and second; slower failure first",

        "Failures":
            [1, 1, 0, 0, 0],

        "Durations":
            [5, 1, 1, 1, 1],

        "ExpectedAPFD":
            0.8,

        "ExpectedAPFDc":
            5.0 / 9.0,
    },

    {
        "Example":
            "Failures first and second; faster failure first",

        "Failures":
            [1, 1, 0, 0, 0],

        "Durations":
            [1, 5, 1, 1, 1],

        "ExpectedAPFD":
            0.8,

        "ExpectedAPFDc":
            7.0 / 9.0,
    },

    {
        "Example":
            "Failures second and fourth",

        "Failures":
            [0, 1, 0, 1, 0],

        "Durations":
            [1, 5, 1, 1, 1],

        "ExpectedAPFD":
            0.5,

        "ExpectedAPFDc":
            7.0 / 18.0,
    },
]


metric_validation_records = []


for example in metric_examples:
    actual_apfd = calculate_apfd(
        example[
            "Failures"
        ]
    )

    actual_apfdc = calculate_apfdc(
        example[
            "Failures"
        ],
        example[
            "Durations"
        ],
    )

    apfd_match = bool(
        np.isclose(
            actual_apfd,
            example[
                "ExpectedAPFD"
            ],
            rtol=0,
            atol=1e-12,
        )
    )

    apfdc_match = bool(
        np.isclose(
            actual_apfdc,
            example[
                "ExpectedAPFDc"
            ],
            rtol=0,
            atol=1e-12,
        )
    )

    metric_validation_records.append({
        "Example":
            example[
                "Example"
            ],

        "ExpectedAPFD":
            example[
                "ExpectedAPFD"
            ],

        "ActualAPFD":
            actual_apfd,

        "APFDMatch":
            apfd_match,

        "ExpectedAPFDc":
            example[
                "ExpectedAPFDc"
            ],

        "ActualAPFDc":
            actual_apfdc,

        "APFDcMatch":
            apfdc_match,

        "Pass":
            (
                apfd_match
                and apfdc_match
            ),
    })


metric_validation = pd.DataFrame(
    metric_validation_records
)


metric_validation_failures = int(
    (
        ~metric_validation[
            "Pass"
        ]
    ).sum()
)


no_failure_apfd_is_nan = bool(
    np.isnan(
        calculate_apfd(
            [0, 0, 0]
        )
    )
)


no_failure_apfdc_is_nan = bool(
    np.isnan(
        calculate_apfdc(
            [0, 0, 0],
            [1, 1, 1],
        )
    )
)


negative_duration_rejected = False


try:
    calculate_apfdc(
        [1, 0],
        [1, -1],
    )

except ValueError:
    negative_duration_rejected = True


# ------------------------------------------------------------
# 23. MODEL CONFIGURATION PAYLOAD
# ------------------------------------------------------------

model_configuration_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProtocolVersion":
        "PROJECT_10_MODEL_BASELINE_METRIC_PROTOCOL_V2",

    "OriginalPredictors":
        len(
            model_predictor_columns
        ),

    "ZeroVarianceFeatureCount":
        len(
            zero_variance_features
        ),

    "ZeroVarianceFeatures":
        zero_variance_features,

    "ActivePredictorCount":
        len(
            active_feature_columns
        ),

    "ActiveFeatureColumns":
        active_feature_columns,

    "TrainingMetadataColumns":
        TRAINING_METADATA_COLUMNS,

    "EvaluationMetadataColumns":
        EVALUATION_METADATA_COLUMNS,

    "MetadataCollisionPolicy":
        (
            "All metadata columns use reserved __meta__ names; "
            "all 151 predictor names remain unchanged"
        ),

    "ZeroVarianceRule":
        (
            "Identify using only the clean fixed training "
            "partition and freeze across all conditions"
        ),

    "MissingValueRule":
        (
            "Convert infinities to missing; calculate medians "
            "from the current noisy training condition; apply "
            "those medians to training and clean evaluation; "
            "all-missing median fallback = 0"
        ),

    "LabelRule":
        "Binary failure equals 1 when verdict is non-zero",

    "PositiveClass":
        1,

    "RollingRetraining":
        False,

    "ModelFitFrequency":
        (
            "One fit per ML technique per project × noise × seed"
        ),

    "TieRule":
        "Score first, then Test ascending",

    "ScoreDirections": {
        "RandomForest":
            "DESCENDING",

        "XGBoost":
            "DESCENDING",

        "LightGBM":
            "DESCENDING",

        "NaiveBayes":
            "DESCENDING",

        "Random":
            "DESCENDING",

        "LatestFail":
            "DESCENDING",

        "QTF-Avg":
            "ASCENDING",
    },

    "Models":
        MODEL_CONFIG,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "EvaluationVerdicts":
        "Clean and immutable",

    "EvaluationDurations":
        "Clean and immutable",

    "LibraryVersions": {
        "Python":
            platform.python_version(),

        "NumPy":
            np.__version__,

        "Pandas":
            pd.__version__,

        "ScikitLearn":
            sklearn.__version__,

        "XGBoost":
            xgboost.__version__,

        "LightGBM":
            lightgbm.__version__,
    },

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# ------------------------------------------------------------
# 24. PRE-WRITE VALIDATION
# ------------------------------------------------------------

prewrite_checks = [
    (
        "Training verdict mismatches",
        training_verdict_mismatches == 0,
    ),
    (
        "Evaluation verdict mismatches",
        evaluation_verdict_mismatches == 0,
    ),
    (
        "Missing evaluation raw links",
        missing_evaluation_raw_links == 0,
    ),
    (
        "Evaluation raw verdict mismatches",
        evaluation_raw_verdict_mismatches == 0,
    ),
    (
        "Training duplicate columns",
        training_base_duplicate_columns == 0,
    ),
    (
        "Evaluation duplicate columns",
        evaluation_base_duplicate_columns == 0,
    ),
    (
        "Training non-finite values",
        training_nonfinite_after_imputation == 0,
    ),
    (
        "Evaluation non-finite values",
        evaluation_nonfinite_after_imputation == 0,
    ),
    (
        "Random invariance",
        random_same_seed_noise_invariant,
    ),
    (
        "Random different seeds",
        random_different_seeds_differ,
    ),
    (
        "LatestFail noise response",
        latest_fail_responds_to_noise,
    ),
    (
        "QTF noise invariance",
        qtf_noise_invariant,
    ),
    (
        "QTF metric identity",
        qtf_metric_identity,
    ),
    (
        "Tie-rule validation",
        tie_rule_failures == 0,
    ),
    (
        "Metric validation",
        metric_validation_failures == 0,
    ),
]


failed_prewrite_checks = [
    name
    for name, passed in prewrite_checks
    if not passed
]


if failed_prewrite_checks:
    raise RuntimeError(
        "Step 4A V2 pre-write checks failed:\n"
        + "\n".join(
            failed_prewrite_checks
        )
    )


# ------------------------------------------------------------
# 25. WRITE OUTPUTS
# ------------------------------------------------------------

MODEL_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_parquet(
    CLEAN_MODEL_TRAINING_BASE_PATH,
    clean_model_training_base,
)

atomic_write_parquet(
    CLEAN_MODEL_EVALUATION_BASE_PATH,
    clean_model_evaluation_base,
)

atomic_write_csv(
    PREDICTOR_MANIFEST_PATH,
    predictor_manifest,
)

atomic_write_csv(
    ZERO_VARIANCE_FEATURES_PATH,
    zero_variance_features_frame,
)

atomic_write_csv(
    CLEAN_MEDIAN_REFERENCE_PATH,
    clean_median_reference,
)

atomic_write_json(
    MODEL_CONFIGURATION_PATH,
    model_configuration_payload,
)

atomic_write_csv(
    MODEL_SEED_MANIFEST_PATH,
    model_seed_manifest,
)

atomic_write_csv(
    RANDOM_SEED_MANIFEST_PATH,
    random_seed_manifest,
)

atomic_write_parquet(
    BASELINE_SENTINEL_RANKINGS_PATH,
    baseline_sentinel_rankings,
)

atomic_write_csv(
    BASELINE_SENTINEL_METRICS_PATH,
    baseline_sentinel_metrics,
)

atomic_write_csv(
    BASELINE_PROTOCOL_AUDIT_PATH,
    baseline_protocol_audit,
)

atomic_write_csv(
    RANKING_TIE_VALIDATION_PATH,
    ranking_tie_validation,
)

atomic_write_csv(
    METRIC_VALIDATION_PATH,
    metric_validation,
)


# ------------------------------------------------------------
# 26. READBACK
# ------------------------------------------------------------

training_base_readback = pd.read_parquet(
    CLEAN_MODEL_TRAINING_BASE_PATH
)

evaluation_base_readback = pd.read_parquet(
    CLEAN_MODEL_EVALUATION_BASE_PATH
)

predictor_manifest_readback = pd.read_csv(
    PREDICTOR_MANIFEST_PATH,
    low_memory=False,
)

model_seed_readback = pd.read_csv(
    MODEL_SEED_MANIFEST_PATH,
    low_memory=False,
)

random_seed_readback = pd.read_csv(
    RANDOM_SEED_MANIFEST_PATH,
    low_memory=False,
)

baseline_rankings_readback = pd.read_parquet(
    BASELINE_SENTINEL_RANKINGS_PATH
)

baseline_metrics_readback = pd.read_csv(
    BASELINE_SENTINEL_METRICS_PATH,
    low_memory=False,
)


# ------------------------------------------------------------
# 27. IMMUTABILITY
# ------------------------------------------------------------

raw_evaluation_sha256_after = calculate_hash(
    raw_evaluation_path
)

model_evaluation_sha256_after = calculate_hash(
    model_evaluation_path
)


raw_evaluation_unchanged = bool(
    raw_evaluation_sha256_before
    == raw_evaluation_sha256_after
)

model_evaluation_unchanged = bool(
    model_evaluation_sha256_before
    == model_evaluation_sha256_after
)


source_root_after = create_source_root_sha256(
    source_files_payload
)

source_unchanged = bool(
    source_root_before
    == source_root_after
    == EXPECTED_SOURCE_ROOT_SHA256
)


registry_sha256_after = calculate_hash(
    REGISTRY_PATH
)

registry_unchanged = bool(
    registry_sha256_before
    == registry_sha256_after
)


# ------------------------------------------------------------
# 28. FINAL VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check": "Step 1B passed",
        "Expected": EXPECTED_STEP1B_STATUS,
        "Actual": step1b_status.get("Status"),
        "Pass": step1b_status.get("Status") == EXPECTED_STEP1B_STATUS,
    },
    {
        "Check": "Step 2B passed",
        "Expected": EXPECTED_STEP2B_STATUS,
        "Actual": step2b_status.get("Status"),
        "Pass": step2b_status.get("Status") == EXPECTED_STEP2B_STATUS,
    },
    {
        "Check": "Step 3A passed",
        "Expected": EXPECTED_STEP3A_STATUS,
        "Actual": step3a_status.get("Status"),
        "Pass": step3a_status.get("Status") == EXPECTED_STEP3A_STATUS,
    },
    {
        "Check": "Step 3B passed",
        "Expected": EXPECTED_STEP3B_STATUS,
        "Actual": step3b_status.get("Status"),
        "Pass": step3b_status.get("Status") == EXPECTED_STEP3B_STATUS,
    },
    {
        "Check": "Dataset rows",
        "Expected": EXPECTED_DATASET_ROWS,
        "Actual": len(dataset),
        "Pass": len(dataset) == EXPECTED_DATASET_ROWS,
    },
    {
        "Check": "Dataset columns",
        "Expected": EXPECTED_DATASET_COLUMNS,
        "Actual": len(dataset.columns),
        "Pass": len(dataset.columns) == EXPECTED_DATASET_COLUMNS,
    },
    {
        "Check": "Original predictors",
        "Expected": EXPECTED_PREDICTOR_COLUMNS,
        "Actual": len(model_predictor_columns),
        "Pass": len(model_predictor_columns) == EXPECTED_PREDICTOR_COLUMNS,
    },
    {
        "Check": "Active plus zero-variance predictors",
        "Expected": EXPECTED_PREDICTOR_COLUMNS,
        "Actual": (
            len(active_feature_columns)
            + len(zero_variance_features)
        ),
        "Pass": (
            len(active_feature_columns)
            + len(zero_variance_features)
            == EXPECTED_PREDICTOR_COLUMNS
        ),
    },
    {
        "Check": "Metadata/predictor collisions",
        "Expected": 0,
        "Actual": len(metadata_predictor_collisions),
        "Pass": len(metadata_predictor_collisions) == 0,
    },
    {
        "Check": "Training base duplicate columns",
        "Expected": 0,
        "Actual": training_base_duplicate_columns,
        "Pass": training_base_duplicate_columns == 0,
    },
    {
        "Check": "Evaluation base duplicate columns",
        "Expected": 0,
        "Actual": evaluation_base_duplicate_columns,
        "Pass": evaluation_base_duplicate_columns == 0,
    },
    {
        "Check": "REC predictors",
        "Expected": 19,
        "Actual": int(predictor_manifest["IsREC"].sum()),
        "Pass": int(predictor_manifest["IsREC"].sum()) == 19,
    },
    {
        "Check": "Verdict-dependent REC predictors",
        "Expected": 13,
        "Actual": int(
            predictor_manifest[
                "VerdictDependentREC"
            ].sum()
        ),
        "Pass": int(
            predictor_manifest[
                "VerdictDependentREC"
            ].sum()
        ) == 13,
    },
    {
        "Check": "Verdict-independent REC predictors",
        "Expected": 6,
        "Actual": int(
            predictor_manifest[
                "VerdictIndependentREC"
            ].sum()
        ),
        "Pass": int(
            predictor_manifest[
                "VerdictIndependentREC"
            ].sum()
        ) == 6,
    },
    {
        "Check": "Model training rows",
        "Expected": EXPECTED_MODEL_TRAINING_ROWS,
        "Actual": training_rows,
        "Pass": training_rows == EXPECTED_MODEL_TRAINING_ROWS,
    },
    {
        "Check": "Model evaluation rows",
        "Expected": EXPECTED_MODEL_EVALUATION_ROWS,
        "Actual": evaluation_rows,
        "Pass": evaluation_rows == EXPECTED_MODEL_EVALUATION_ROWS,
    },
    {
        "Check": "Model training failures",
        "Expected": EXPECTED_MODEL_TRAINING_FAILURES,
        "Actual": training_failures,
        "Pass": training_failures == EXPECTED_MODEL_TRAINING_FAILURES,
    },
    {
        "Check": "Model evaluation failures",
        "Expected": EXPECTED_MODEL_EVALUATION_FAILURES,
        "Actual": evaluation_failures,
        "Pass": evaluation_failures == EXPECTED_MODEL_EVALUATION_FAILURES,
    },
    {
        "Check": "Evaluation-period builds",
        "Expected": EXPECTED_EVALUATION_PERIOD_BUILDS,
        "Actual": evaluation_period_builds,
        "Pass": evaluation_period_builds == EXPECTED_EVALUATION_PERIOD_BUILDS,
    },
    {
        "Check": "Scored evaluation builds",
        "Expected": EXPECTED_SCORED_EVALUATION_BUILDS,
        "Actual": scored_evaluation_builds,
        "Pass": scored_evaluation_builds == EXPECTED_SCORED_EVALUATION_BUILDS,
    },
    {
        "Check": "Scored builds without failures",
        "Expected": 0,
        "Actual": evaluation_builds_without_failures,
        "Pass": evaluation_builds_without_failures == 0,
    },
    {
        "Check": "Scored builds with non-positive duration",
        "Expected": 0,
        "Actual": evaluation_builds_with_nonpositive_duration,
        "Pass": evaluation_builds_with_nonpositive_duration == 0,
    },
    {
        "Check": "Duplicate training Build/Test rows",
        "Expected": 0,
        "Actual": duplicate_training_build_test_rows,
        "Pass": duplicate_training_build_test_rows == 0,
    },
    {
        "Check": "Duplicate evaluation Build/Test rows",
        "Expected": 0,
        "Actual": duplicate_evaluation_build_test_rows,
        "Pass": duplicate_evaluation_build_test_rows == 0,
    },
    {
        "Check": "Training verdict mismatches",
        "Expected": 0,
        "Actual": training_verdict_mismatches,
        "Pass": training_verdict_mismatches == 0,
    },
    {
        "Check": "Evaluation verdict mismatches",
        "Expected": 0,
        "Actual": evaluation_verdict_mismatches,
        "Pass": evaluation_verdict_mismatches == 0,
    },
    {
        "Check": "Missing evaluation raw links",
        "Expected": 0,
        "Actual": missing_evaluation_raw_links,
        "Pass": missing_evaluation_raw_links == 0,
    },
    {
        "Check": "Evaluation raw verdict mismatches",
        "Expected": 0,
        "Actual": evaluation_raw_verdict_mismatches,
        "Pass": evaluation_raw_verdict_mismatches == 0,
    },
    {
        "Check": "Clean training classes",
        "Expected": [0, 1],
        "Actual": clean_training_label_classes,
        "Pass": clean_training_label_classes == [0, 1],
    },
    {
        "Check": "Training non-finite values after imputation",
        "Expected": 0,
        "Actual": training_nonfinite_after_imputation,
        "Pass": training_nonfinite_after_imputation == 0,
    },
    {
        "Check": "Evaluation non-finite values after imputation",
        "Expected": 0,
        "Actual": evaluation_nonfinite_after_imputation,
        "Pass": evaluation_nonfinite_after_imputation == 0,
    },
    {
        "Check": "Instantiated ML models",
        "Expected": 4,
        "Actual": len(model_instances),
        "Pass": len(model_instances) == 4,
    },
    {
        "Check": "Model-seed rows",
        "Expected": 30,
        "Actual": len(model_seed_manifest),
        "Pass": len(model_seed_manifest) == 30,
    },
    {
        "Check": "Random-seed rows",
        "Expected": (
            30
            * EXPECTED_SCORED_EVALUATION_BUILDS
        ),
        "Actual": len(random_seed_manifest),
        "Pass": len(random_seed_manifest) == (
            30
            * EXPECTED_SCORED_EVALUATION_BUILDS
        ),
    },
    {
        "Check": "Baseline scenarios",
        "Expected": EXPECTED_BASELINE_SCENARIOS,
        "Actual": baseline_sentinel_rankings["Scenario"].nunique(),
        "Pass": baseline_sentinel_rankings["Scenario"].nunique() == EXPECTED_BASELINE_SCENARIOS,
    },
    {
        "Check": "Baseline ranking rows",
        "Expected": EXPECTED_BASELINE_RANKING_ROWS,
        "Actual": len(baseline_sentinel_rankings),
        "Pass": len(baseline_sentinel_rankings) == EXPECTED_BASELINE_RANKING_ROWS,
    },
    {
        "Check": "Baseline scenarios with wrong row count",
        "Expected": 0,
        "Actual": baseline_scenarios_with_wrong_rows,
        "Pass": baseline_scenarios_with_wrong_rows == 0,
    },
    {
        "Check": "Baseline metric rows",
        "Expected": EXPECTED_BASELINE_METRIC_ROWS,
        "Actual": len(baseline_sentinel_metrics),
        "Pass": len(baseline_sentinel_metrics) == EXPECTED_BASELINE_METRIC_ROWS,
    },
    {
        "Check": "Baseline metric NaN values",
        "Expected": 0,
        "Actual": baseline_metric_nan_values,
        "Pass": baseline_metric_nan_values == 0,
    },
    {
        "Check": "Baseline metric values outside range",
        "Expected": 0,
        "Actual": baseline_metric_out_of_range_values,
        "Pass": baseline_metric_out_of_range_values == 0,
    },
    {
        "Check": "Random same-seed noise invariance",
        "Expected": True,
        "Actual": random_same_seed_noise_invariant,
        "Pass": random_same_seed_noise_invariant,
    },
    {
        "Check": "Random different seeds differ",
        "Expected": True,
        "Actual": random_different_seeds_differ,
        "Pass": random_different_seeds_differ,
    },
    {
        "Check": "LatestFail responds to noise",
        "Expected": True,
        "Actual": latest_fail_responds_to_noise,
        "Pass": latest_fail_responds_to_noise,
    },
    {
        "Check": "QTF-Avg noise invariance",
        "Expected": True,
        "Actual": qtf_noise_invariant,
        "Pass": qtf_noise_invariant,
    },
    {
        "Check": "QTF-Avg metric identity",
        "Expected": True,
        "Actual": qtf_metric_identity,
        "Pass": qtf_metric_identity,
    },
    {
        "Check": "Baseline protocol audit",
        "Expected": True,
        "Actual": bool(
            baseline_protocol_audit[
                "ValidationPass"
            ].all()
        ),
        "Pass": bool(
            baseline_protocol_audit[
                "ValidationPass"
            ].all()
        ),
    },
    {
        "Check": "Ranking tie-rule failures",
        "Expected": 0,
        "Actual": tie_rule_failures,
        "Pass": tie_rule_failures == 0,
    },
    {
        "Check": "Metric validation failures",
        "Expected": 0,
        "Actual": metric_validation_failures,
        "Pass": metric_validation_failures == 0,
    },
    {
        "Check": "No-failure APFD undefined",
        "Expected": True,
        "Actual": no_failure_apfd_is_nan,
        "Pass": no_failure_apfd_is_nan,
    },
    {
        "Check": "No-failure APFDc undefined",
        "Expected": True,
        "Actual": no_failure_apfdc_is_nan,
        "Pass": no_failure_apfdc_is_nan,
    },
    {
        "Check": "Negative duration rejected",
        "Expected": True,
        "Actual": negative_duration_rejected,
        "Pass": negative_duration_rejected,
    },
    {
        "Check": "Training-base readback rows",
        "Expected": EXPECTED_MODEL_TRAINING_ROWS,
        "Actual": len(training_base_readback),
        "Pass": len(training_base_readback) == EXPECTED_MODEL_TRAINING_ROWS,
    },
    {
        "Check": "Evaluation-base readback rows",
        "Expected": EXPECTED_MODEL_EVALUATION_ROWS,
        "Actual": len(evaluation_base_readback),
        "Pass": len(evaluation_base_readback) == EXPECTED_MODEL_EVALUATION_ROWS,
    },
    {
        "Check": "Training-base readback duplicate columns",
        "Expected": 0,
        "Actual": int(training_base_readback.columns.duplicated().sum()),
        "Pass": int(training_base_readback.columns.duplicated().sum()) == 0,
    },
    {
        "Check": "Evaluation-base readback duplicate columns",
        "Expected": 0,
        "Actual": int(evaluation_base_readback.columns.duplicated().sum()),
        "Pass": int(evaluation_base_readback.columns.duplicated().sum()) == 0,
    },
    {
        "Check": "Predictor-manifest readback rows",
        "Expected": EXPECTED_PREDICTOR_COLUMNS,
        "Actual": len(predictor_manifest_readback),
        "Pass": len(predictor_manifest_readback) == EXPECTED_PREDICTOR_COLUMNS,
    },
    {
        "Check": "Model-seed readback rows",
        "Expected": 30,
        "Actual": len(model_seed_readback),
        "Pass": len(model_seed_readback) == 30,
    },
    {
        "Check": "Random-seed readback rows",
        "Expected": (
            30
            * EXPECTED_SCORED_EVALUATION_BUILDS
        ),
        "Actual": len(random_seed_readback),
        "Pass": len(random_seed_readback) == (
            30
            * EXPECTED_SCORED_EVALUATION_BUILDS
        ),
    },
    {
        "Check": "Baseline-ranking readback rows",
        "Expected": EXPECTED_BASELINE_RANKING_ROWS,
        "Actual": len(baseline_rankings_readback),
        "Pass": len(baseline_rankings_readback) == EXPECTED_BASELINE_RANKING_ROWS,
    },
    {
        "Check": "Baseline-metric readback rows",
        "Expected": EXPECTED_BASELINE_METRIC_ROWS,
        "Actual": len(baseline_metrics_readback),
        "Pass": len(baseline_metrics_readback) == EXPECTED_BASELINE_METRIC_ROWS,
    },
    {
        "Check": "Raw evaluation cohort unchanged",
        "Expected": True,
        "Actual": raw_evaluation_unchanged,
        "Pass": raw_evaluation_unchanged,
    },
    {
        "Check": "Model evaluation cohort unchanged",
        "Expected": True,
        "Actual": model_evaluation_unchanged,
        "Pass": model_evaluation_unchanged,
    },
    {
        "Check": "Project 10 source unchanged",
        "Expected": True,
        "Actual": source_unchanged,
        "Pass": source_unchanged,
    },
    {
        "Check": "Completion registry unchanged",
        "Expected": True,
        "Actual": registry_unchanged,
        "Pass": registry_unchanged,
    },
    {
        "Check": "Registry Project 9 rows",
        "Expected": 0,
        "Actual": int(registry_project_numbers.eq(9).sum()),
        "Pass": int(registry_project_numbers.eq(9).sum()) == 0,
    },
    {
        "Check": "Registry Project 10 rows",
        "Expected": 0,
        "Actual": int(registry_project_numbers.eq(10).sum()),
        "Pass": int(registry_project_numbers.eq(10).sum()) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)

failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 4A V2 validation:")

display(
    validation
)


if not failed_checks.empty:
    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 10 STEP 4A V2 DID NOT PASS."
    )


atomic_write_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 29. REPORT AND CHECKPOINT
# ------------------------------------------------------------

output_hashes = {
    "CleanModelTrainingBaseSHA256":
        calculate_hash(
            CLEAN_MODEL_TRAINING_BASE_PATH
        ),

    "CleanModelEvaluationBaseSHA256":
        calculate_hash(
            CLEAN_MODEL_EVALUATION_BASE_PATH
        ),

    "PredictorManifestSHA256":
        calculate_hash(
            PREDICTOR_MANIFEST_PATH
        ),

    "ZeroVarianceFeaturesSHA256":
        calculate_hash(
            ZERO_VARIANCE_FEATURES_PATH
        ),

    "CleanMedianReferenceSHA256":
        calculate_hash(
            CLEAN_MEDIAN_REFERENCE_PATH
        ),

    "ModelConfigurationSHA256":
        calculate_hash(
            MODEL_CONFIGURATION_PATH
        ),

    "ModelSeedManifestSHA256":
        calculate_hash(
            MODEL_SEED_MANIFEST_PATH
        ),

    "RandomSeedManifestSHA256":
        calculate_hash(
            RANDOM_SEED_MANIFEST_PATH
        ),

    "BaselineSentinelRankingsSHA256":
        calculate_hash(
            BASELINE_SENTINEL_RANKINGS_PATH
        ),

    "BaselineSentinelMetricsSHA256":
        calculate_hash(
            BASELINE_SENTINEL_METRICS_PATH
        ),

    "BaselineProtocolAuditSHA256":
        calculate_hash(
            BASELINE_PROTOCOL_AUDIT_PATH
        ),

    "RankingTieValidationSHA256":
        calculate_hash(
            RANKING_TIE_VALIDATION_PATH
        ),

    "MetricValidationSHA256":
        calculate_hash(
            METRIC_VALIDATION_PATH
        ),

    "Step4AValidationSHA256":
        calculate_hash(
            STEP4A_VALIDATION_PATH
        ),
}


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_PASS_STATUS,

    "ProtocolVersion":
        "PROJECT_10_MODEL_BASELINE_METRIC_PROTOCOL_V2",

    "OriginalPredictors":
        len(
            model_predictor_columns
        ),

    "ZeroVarianceFeatureCount":
        len(
            zero_variance_features
        ),

    "ZeroVarianceFeatures":
        zero_variance_features,

    "ActivePredictorCount":
        len(
            active_feature_columns
        ),

    "ActiveFeatureColumns":
        active_feature_columns,

    "TrainingMetadataColumns":
        TRAINING_METADATA_COLUMNS,

    "EvaluationMetadataColumns":
        EVALUATION_METADATA_COLUMNS,

    "ModelTrainingRows":
        training_rows,

    "ModelEvaluationRows":
        evaluation_rows,

    "ModelTrainingFailures":
        training_failures,

    "ModelEvaluationFailures":
        evaluation_failures,

    "EvaluationPeriodBuilds":
        evaluation_period_builds,

    "ScoredEvaluationBuilds":
        scored_evaluation_builds,

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "RandomSameSeedNoiseInvariant":
        random_same_seed_noise_invariant,

    "RandomDifferentSeedsDiffer":
        random_different_seeds_differ,

    "LatestFailRespondsToNoise":
        latest_fail_responds_to_noise,

    "QTFAvgNoiseInvariant":
        qtf_noise_invariant,

    "QTFAvgMetricIdentity":
        qtf_metric_identity,

    "TieRule":
        "Score first, then Test ascending",

    "RollingMLRetraining":
        False,

    "ModelConfiguration":
        model_configuration_payload,

    "InstantiatedModelClasses":
        instantiated_model_classes,

    "BaselineRankingRows":
        len(
            baseline_sentinel_rankings
        ),

    "BaselineMetricRows":
        len(
            baseline_sentinel_metrics
        ),

    "RawEvaluationUnchanged":
        raw_evaluation_unchanged,

    "ModelEvaluationUnchanged":
        model_evaluation_unchanged,

    "Project10SourceUnchanged":
        source_unchanged,

    "CompletionRegistryModified":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "OutputHashes":
        output_hashes,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "SourceRootSHA256":
        source_root_after,

    "SelectionCheckpointSHA256":
        calculate_hash(
            SELECTION_CHECKPOINT_PATH
        ),

    "RECCheckpointSHA256":
        calculate_hash(
            REC_CHECKPOINT_PATH
        ),

    "NoisePlanCheckpointSHA256":
        calculate_hash(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "NoisyRECEngineCheckpointSHA256":
        calculate_hash(
            NOISY_REC_ENGINE_CHECKPOINT_PATH
        ),

    "CleanModelTrainingBase":
        str(
            CLEAN_MODEL_TRAINING_BASE_PATH
        ),

    "CleanModelEvaluationBase":
        str(
            CLEAN_MODEL_EVALUATION_BASE_PATH
        ),

    "PredictorManifest":
        str(
            PREDICTOR_MANIFEST_PATH
        ),

    "ZeroVarianceFeaturesPath":
        str(
            ZERO_VARIANCE_FEATURES_PATH
        ),

    "CleanMedianReference":
        str(
            CLEAN_MEDIAN_REFERENCE_PATH
        ),

    "ModelConfigurationPath":
        str(
            MODEL_CONFIGURATION_PATH
        ),

    "ModelSeedManifest":
        str(
            MODEL_SEED_MANIFEST_PATH
        ),

    "RandomSeedManifest":
        str(
            RANDOM_SEED_MANIFEST_PATH
        ),

    "BaselineSentinelRankings":
        str(
            BASELINE_SENTINEL_RANKINGS_PATH
        ),

    "BaselineSentinelMetrics":
        str(
            BASELINE_SENTINEL_METRICS_PATH
        ),

    "BaselineProtocolAudit":
        str(
            BASELINE_PROTOCOL_AUDIT_PATH
        ),

    "RankingTieValidation":
        str(
            RANKING_TIE_VALIDATION_PATH
        ),

    "MetricValidation":
        str(
            METRIC_VALIDATION_PATH
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistryRows":
        len(
            registry
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    MODEL_PROTOCOL_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_PASS_STATUS,

    "ProtocolVersion":
        "PROJECT_10_MODEL_BASELINE_METRIC_PROTOCOL_V2",

    "OriginalPredictors":
        len(
            model_predictor_columns
        ),

    "ZeroVarianceFeatureCount":
        len(
            zero_variance_features
        ),

    "ActivePredictorCount":
        len(
            active_feature_columns
        ),

    "ScoredEvaluationBuilds":
        scored_evaluation_builds,

    "RandomSameSeedNoiseInvariant":
        random_same_seed_noise_invariant,

    "LatestFailRespondsToNoise":
        latest_fail_responds_to_noise,

    "QTFAvgNoiseInvariant":
        qtf_noise_invariant,

    "Checkpoint":
        str(
            MODEL_PROTOCOL_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_hash(
            MODEL_PROTOCOL_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 30. FINAL READBACK
# ------------------------------------------------------------

checkpoint_readback = read_json_with_retry(
    MODEL_PROTOCOL_CHECKPOINT_PATH
)

status_readback = read_json_with_retry(
    STEP4A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP4A_PASS_STATUS:
    raise AssertionError(
        "Model-protocol checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP4A_PASS_STATUS:
    raise AssertionError(
        "Step 4A V2 status readback failed."
    )


if calculate_hash(
    REGISTRY_PATH
) != registry_sha256_before:
    raise AssertionError(
        "Completion registry changed during Step 4A V2."
    )


if create_source_root_sha256(
    source_files_payload
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Project 10 source changed during Step 4A V2."
    )


# ------------------------------------------------------------
# 31. DISPLAY
# ------------------------------------------------------------

print("\nPredictor summary:")

display(
    predictor_manifest.groupby(
        [
            "PredictorClass",
            "ZeroVariance",
            "Active",
        ],
        as_index=False,
    ).agg(
        Predictors=(
            "Predictor",
            "count",
        )
    )
)


print("\nZero-variance predictors:")

display(
    zero_variance_features_frame
)


print("\nMetadata collision fix:")

display(
    pd.DataFrame([
        {
            "Base":
                "TRAINING",

            "MetadataColumns":
                len(
                    TRAINING_METADATA_COLUMNS
                ),

            "PredictorColumns":
                len(
                    model_predictor_columns
                ),

            "DuplicateColumnNames":
                training_base_duplicate_columns,
        },
        {
            "Base":
                "EVALUATION",

            "MetadataColumns":
                len(
                    EVALUATION_METADATA_COLUMNS
                ),

            "PredictorColumns":
                len(
                    model_predictor_columns
                ),

            "DuplicateColumnNames":
                evaluation_base_duplicate_columns,
        },
    ])
)


print("\nBaseline protocol audit:")

display(
    baseline_protocol_audit
)


print("\nBaseline metrics summary:")

display(
    baseline_sentinel_metrics.groupby(
        [
            "Scenario",
            "Technique",
        ],
        as_index=False,
    ).agg(
        Builds=(
            "BuildKey",
            "nunique",
        ),

        MeanAPFD=(
            "APFD",
            "mean",
        ),

        MeanAPFDc=(
            "APFDc",
            "mean",
        ),
    )
)


print("\nRanking tie validation:")

display(
    ranking_tie_validation
)


print("\nMetric validation:")

display(
    metric_validation
)


# ------------------------------------------------------------
# 32. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 128)
print("=== PROJECT 10 CELL 7 / STEP 4A V2 RESULT ===")
print("=" * 128)

print("\nProject:")
print(PROJECT_NAME)


print("\nPredictor protocol:")

print(
    "Original predictor columns:",
    len(
        model_predictor_columns
    ),
)

print(
    "Frozen zero-variance columns:",
    len(
        zero_variance_features
    ),
)

print(
    "Active predictor columns:",
    len(
        active_feature_columns
    ),
)

print(
    "Training-base duplicate column names:",
    training_base_duplicate_columns,
)

print(
    "Evaluation-base duplicate column names:",
    evaluation_base_duplicate_columns,
)

print(
    "Training non-finite values after imputation:",
    training_nonfinite_after_imputation,
)

print(
    "Evaluation non-finite values after imputation:",
    evaluation_nonfinite_after_imputation,
)


print("\nFixed model cohorts:")

print(
    "Model training rows:",
    training_rows,
)

print(
    "Model evaluation rows:",
    evaluation_rows,
)

print(
    "Model training failures:",
    training_failures,
)

print(
    "Model evaluation failures:",
    evaluation_failures,
)

print(
    "Evaluation-period builds:",
    evaluation_period_builds,
)

print(
    "Scored evaluation builds:",
    scored_evaluation_builds,
)


print("\nML protocol:")

print(
    "Techniques:",
    ML_TECHNIQUES,
)

print(
    "Instantiated model classes:",
    instantiated_model_classes,
)

print(
    "Rolling retraining:",
    False,
)

print(
    "Tie rule:",
    "Score first, then Test ascending",
)


print("\nBaseline protocol:")

print(
    "Baseline techniques:",
    BASELINE_TECHNIQUES,
)

print(
    "Random same-seed noise invariant:",
    random_same_seed_noise_invariant,
)

print(
    "Random different seeds differ:",
    random_different_seeds_differ,
)

print(
    "LatestFail responds to verdict noise:",
    latest_fail_responds_to_noise,
)

print(
    "QTF-Avg noise invariant:",
    qtf_noise_invariant,
)

print(
    "QTF-Avg metric identity:",
    qtf_metric_identity,
)

print(
    "Baseline sentinel ranking rows:",
    len(
        baseline_sentinel_rankings
    ),
)

print(
    "Baseline sentinel metric rows:",
    len(
        baseline_sentinel_metrics
    ),
)


print("\nMetric protocol:")

print(
    "Primary metric:",
    "APFDc",
)

print(
    "Secondary metric:",
    "APFD",
)

print(
    "Ranking tie-rule failures:",
    tie_rule_failures,
)

print(
    "Manual metric-validation failures:",
    metric_validation_failures,
)


print("\nImmutability:")

print(
    "Raw evaluation cohort unchanged:",
    raw_evaluation_unchanged,
)

print(
    "Model evaluation cohort unchanged:",
    model_evaluation_unchanged,
)

print(
    "Project 10 source unchanged:",
    source_unchanged,
)

print(
    "Completion registry unchanged:",
    registry_unchanged,
)

print(
    "Project 9 accessed:",
    False,
)

print(
    "Project 9 write attempted:",
    False,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nModel-protocol checkpoint:")

print(
    MODEL_PROTOCOL_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_hash(
        MODEL_PROTOCOL_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP4A_PASS_STATUS,
)

print("=" * 128)

=== PROJECT 10 CELL 7 / STEP 4A V2: MODEL, BASELINE AND METRIC PROTOCOL FREEZE ===

Step 4A V2 validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,PASS_PROJECT_10_SELECTION_LOCKED_SOURCE_FROZEN...,True
1,Step 2B passed,PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_A...,PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_A...,True
2,Step 3A passed,PASS_PROJECT_10_DETERMINISTIC_NOISE_PLAN_AND_C...,PASS_PROJECT_10_DETERMINISTIC_NOISE_PLAN_AND_C...,True
3,Step 3B passed,PASS_PROJECT_10_NOISY_REC_ENGINE_SENTINEL_VALI...,PASS_PROJECT_10_NOISY_REC_ENGINE_SENTINEL_VALI...,True
4,Dataset rows,8706,8706,True
...,...,...,...,...
61,Model evaluation cohort unchanged,True,True,True
62,Project 10 source unchanged,True,True,True
63,Completion registry unchanged,True,True,True
64,Registry Project 9 rows,0,0,True



Predictor summary:


,PredictorClass,ZeroVariance,Active,Predictors
0,NON_REC,False,True,132
1,REC_VERDICT_DEPENDENT,False,True,13
2,REC_VERDICT_INDEPENDENT,False,True,6



Zero-variance predictors:


...


Ranking tie validation:


,Example,ExpectedTestOrder,ActualTestOrder,Pass
0,Descending score with Test ascending tie-break,"[2, 1, 3]","[2, 1, 3]",True
1,Ascending score with Test ascending tie-break,"[2, 1, 3]","[2, 1, 3]",True



Metric validation:


,Example,ExpectedAPFD,ActualAPFD,APFDMatch,ExpectedAPFDc,ActualAPFDc,APFDcMatch,Pass
0,Failures first and second; slower failure first,0.8,0.8,True,0.555556,0.555556,True,True
1,Failures first and second; faster failure first,0.8,0.8,True,0.777778,0.777778,True,True
2,Failures second and fourth,0.5,0.5,True,0.388889,0.388889,True,True




=== PROJECT 10 CELL 7 / STEP 4A V2 RESULT ===

Project:
spring-cloud@spring-cloud-dataflow

Predictor protocol:
Original predictor columns: 151
Frozen zero-variance columns: 0
Active predictor columns: 151
Training-base duplicate column names: 0
Evaluation-base duplicate column names: 0
Training non-finite values after imputation: 0
Evaluation non-finite values after imputation: 0

Fixed model cohorts:
Model training rows: 6095
Model evaluation rows: 2611
Model training failures: 63
Model evaluation failures: 213
Evaluation-period builds: 102
Scored evaluation builds: 27

ML protocol:
Techniques: ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
Instantiated model classes: {'RandomForest': 'RandomForestClassifier', 'XGBoost': 'XGBClassifier', 'LightGBM': 'LGBMClassifier', 'NaiveBayes': 'GaussianNB'}
Rolling retraining: False
Tie rule: Score first, then Test ascending

Baseline protocol:
Baseline techniques: ['Random', 'LatestFail', 'QTF-Avg']
Random same-seed noise invariant: Tru

In [11]:
# ============================================================
# PROJECT 10 — CELL 8 / STEP 4B
# TWO-CONDITION END-TO-END SMOKE TEST
#
# PROJECT:
#   spring-cloud@spring-cloud-dataflow
#
# CONDITIONS:
#   noise_00__seed_01
#   noise_50__seed_01
#
# This cell validates the complete experimental pipeline:
# - deterministic verdict-noise recreation
# - noisy REC reconstruction
# - clean-anchor delta application
# - current-condition median preprocessing
# - four ML model fits
# - Random, LatestFail and QTF-Avg
# - clean evaluation ranking
# - build-level APFD/APFDc
# - project-run aggregation
# - output-schema and count validation
#
# This cell does NOT:
# - launch the full 270-condition experiment
# - access or modify Project 9
# - modify Projects 1–8
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from IPython.display import display

import hashlib
import json
import os
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)


EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_10_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP3B_STATUS = (
    "PASS_PROJECT_10_NOISY_REC_ENGINE_SENTINEL_VALIDATED_AND_FROZEN"
)

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_10_MODEL_PREDICTOR_BASELINE_AND_METRIC_PROTOCOL_FROZEN"
)

STEP4B_PASS_STATUS = (
    "PASS_PROJECT_10_TWO_CONDITION_END_TO_END_SMOKE_TEST_VALIDATED"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e"
)

EXPECTED_RAW_TRAINING_ROWS = 34563
EXPECTED_RAW_EVALUATION_ROWS = 12531

EXPECTED_MODEL_TRAINING_ROWS = 6095
EXPECTED_MODEL_EVALUATION_ROWS = 2611

EXPECTED_MODEL_EVALUATION_FAILURES = 213
EXPECTED_SCORED_EVALUATION_BUILDS = 27

EXPECTED_ACTIVE_PREDICTORS = 151

RECENT_WINDOW = 6


SMOKE_CONDITIONS = [
    (0, 1),
    (50, 1),
]

EXPECTED_SMOKE_CONDITIONS = len(
    SMOKE_CONDITIONS
)


ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


EXPECTED_ML_FITS = (
    EXPECTED_SMOKE_CONDITIONS
    * len(
        ML_TECHNIQUES
    )
)

EXPECTED_RANKING_ROWS = (
    EXPECTED_SMOKE_CONDITIONS
    * len(
        ALL_TECHNIQUES
    )
    * EXPECTED_MODEL_EVALUATION_ROWS
)

EXPECTED_BUILD_METRIC_ROWS = (
    EXPECTED_SMOKE_CONDITIONS
    * len(
        ALL_TECHNIQUES
    )
    * EXPECTED_SCORED_EVALUATION_BUILDS
)

EXPECTED_PROJECT_RUN_ROWS = (
    EXPECTED_SMOKE_CONDITIONS
    * len(
        ALL_TECHNIQUES
    )
)

EXPECTED_TRAINING_MEDIAN_ROWS = (
    EXPECTED_SMOKE_CONDITIONS
    * EXPECTED_ACTIVE_PREDICTORS
)


REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


if len(
    REC_FEATURE_COLUMNS
) != 19:
    raise AssertionError(
        "Expected exactly 19 REC features."
    )


if len(
    VERDICT_DEPENDENT_REC_FEATURES
) != 13:
    raise AssertionError(
        "Expected exactly 13 verdict-dependent REC features."
    )


if len(
    VERDICT_INDEPENDENT_REC_FEATURES
) != 6:
    raise AssertionError(
        "Expected exactly six verdict-independent REC features."
    )


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_noise_plan_checkpoint.json"
)

NOISY_REC_ENGINE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_noisy_rec_engine_checkpoint.json"
)

MODEL_PROTOCOL_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_model_protocol_checkpoint.json"
)

SMOKE_TEST_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_smoke_test_checkpoint.json"
)


PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)

STEP3B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_status.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_status.json"
)

STEP4B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step4b_status.json"
)


SMOKE_TEST_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_test"
)

SMOKE_RANKINGS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_rankings.parquet"
)

SMOKE_BUILD_METRICS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_build_metrics.csv"
)

SMOKE_PROJECT_RUN_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_project_run.csv"
)

SMOKE_MODEL_FITS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_model_fits.csv"
)

SMOKE_TRAINING_MEDIANS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_training_medians.csv"
)

SMOKE_CONDITION_AUDIT_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_condition_audit.csv"
)

SMOKE_INVARIANCE_AUDIT_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_invariance_audit.csv"
)

STEP4B_VALIDATION_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_step4b_validation.csv"
)

STEP4B_REPORT_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_step4b_report.json"
)


PROJECT_9_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "camunda__camunda-bpm-platform"
)


print("=" * 126)
print("=== PROJECT 10 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===")
print("=" * 126)


# ------------------------------------------------------------
# 3. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def array_sha256(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def dataframe_semantic_sha256(
    dataframe,
    columns,
):
    digest = hashlib.sha256()

    for column in columns:
        digest.update(
            str(
                column
            ).encode(
                "utf-8"
            )
        )

        series = dataframe[
            column
        ]

        if pd.api.types.is_numeric_dtype(
            series
        ):
            values = pd.to_numeric(
                series,
                errors="raise",
            ).to_numpy(
                dtype="<f8"
            )

            digest.update(
                values.tobytes(
                    order="C"
                )
            )

        else:
            for value in (
                series
                .fillna("")
                .astype(str)
            ):
                encoded = value.encode(
                    "utf-8"
                )

                digest.update(
                    len(
                        encoded
                    ).to_bytes(
                        8,
                        byteorder="little",
                        signed=False,
                    )
                )

                digest.update(
                    encoded
                )

    return digest.hexdigest()


def json_safe(
    value,
):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(
            value
        ):
            return None
    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="snappy",
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json_with_retry(
    path,
    attempts=10,
    delay_seconds=0.5,
):
    path = Path(
        path
    )

    last_error = None

    for _ in range(
        attempts
    ):
        try:
            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:
            last_error = error

            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON.\n"
        f"Path: {path}\n"
        f"Error: {type(last_error).__name__}: {last_error}"
    )


def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


def create_source_root_sha256(
    source_files_payload,
):
    digest = hashlib.sha256()

    for relative_path in sorted(
        source_files_payload
    ):
        metadata = source_files_payload[
            relative_path
        ]

        runtime_path = Path(
            metadata[
                "RuntimePath"
            ]
        )

        digest.update(
            (
                f"{relative_path}\0"
                f"{int(runtime_path.stat().st_size)}\0"
                f"{calculate_hash(runtime_path)}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def normalise_build_seed_key(
    value,
):
    numeric = pd.to_numeric(
        pd.Series(
            [value]
        ),
        errors="coerce",
    ).iloc[0]

    if (
        not pd.isna(
            numeric
        )
        and np.isclose(
            numeric,
            round(
                numeric
            ),
            rtol=0,
            atol=1e-9,
        )
    ):
        return str(
            int(
                round(
                    numeric
                )
            )
        )

    return str(
        value
    )


# ------------------------------------------------------------
# 4. METRICS
# ------------------------------------------------------------

def calculate_apfd(
    actual_failures,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):
        return np.nan

    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )


def calculate_apfdc(
    actual_failures,
    durations,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(
        failures
    ) != len(
        durations
    ):
        raise ValueError(
            "Failure and duration arrays differ in length."
        )

    if (
        len(
            failures
        ) == 0
        or failures.sum() == 0
    ):
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain non-finite values."
        )

    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations contain negative values."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array(
            [0.0]
        ),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


# ------------------------------------------------------------
# 5. RANKING HELPERS
# ------------------------------------------------------------

def rank_build_rows(
    build_rows,
    scores,
    technique,
    score_direction,
):
    required_columns = [
        "Build",
        "Test",
        "Verdict",
        "Duration",
        "build_order",
        "BuildKey",
        "TestKey",
    ]

    if build_rows.columns.duplicated().any():
        raise RuntimeError(
            "Ranking frame contains duplicate columns."
        )

    ranked = (
        build_rows[
            required_columns
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    score_values = np.asarray(
        scores,
        dtype=float,
    )

    if len(
        score_values
    ) != len(
        ranked
    ):
        raise ValueError(
            "Score and ranking-row counts differ."
        )

    if np.isnan(
        score_values
    ).any():
        raise ValueError(
            "Ranking scores contain NaN."
        )

    ranked[
        "Technique"
    ] = technique

    ranked[
        "Score"
    ] = score_values

    ranked[
        "ActualFailure"
    ] = (
        pd.to_numeric(
            ranked[
                "Verdict"
            ],
            errors="raise",
        )
        .ne(0)
        .astype(np.int8)
    )

    ranked[
        "Duration"
    ] = pd.to_numeric(
        ranked[
            "Duration"
        ],
        errors="raise",
    ).astype(float)

    numeric_test = pd.to_numeric(
        ranked[
            "Test"
        ],
        errors="coerce",
    )

    if numeric_test.notna().all():
        ranked[
            "__TestSort"
        ] = numeric_test.astype(float)

    else:
        ranked[
            "__TestSort"
        ] = (
            ranked[
                "Test"
            ]
            .fillna("")
            .astype(str)
        )

    if score_direction == "descending":
        score_ascending = False

    elif score_direction == "ascending":
        score_ascending = True

    else:
        raise ValueError(
            "Unknown score direction."
        )

    ranked = (
        ranked.sort_values(
            [
                "Score",
                "__TestSort",
            ],
            ascending=[
                score_ascending,
                True,
            ],
            kind="mergesort",
        )
        .drop(
            columns=[
                "__TestSort",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    ranked[
        "Rank"
    ] = np.arange(
        1,
        len(
            ranked
        ) + 1,
        dtype=np.int32,
    )

    return ranked


def add_condition_metadata(
    ranking,
    smoke_condition_order,
    condition_id,
    noise_percent,
    repetition_seed,
):
    result = ranking.copy()

    result.insert(
        0,
        "ProjectNumber",
        PROJECT_NUMBER,
    )

    result.insert(
        1,
        "Project",
        PROJECT_NAME,
    )

    result.insert(
        2,
        "ProjectSlug",
        PROJECT_SLUG,
    )

    result.insert(
        3,
        "SmokeConditionOrder",
        smoke_condition_order,
    )

    result.insert(
        4,
        "ConditionID",
        condition_id,
    )

    result.insert(
        5,
        "NoisePercent",
        noise_percent,
    )

    result.insert(
        6,
        "RepetitionSeed",
        repetition_seed,
    )

    return result


def create_random_rankings(
    evaluation_data,
    repetition_seed,
):
    output = []

    for build_key, build_rows in (
        evaluation_data.groupby(
            "BuildKey",
            sort=False,
        )
    ):
        random_seed = stable_project_seed(
            PROJECT_NAME,
            repetition_seed,
            (
                "Random_baseline_build_"
                f"{normalise_build_seed_key(build_key)}"
            ),
        )

        random_scores = (
            np.random.default_rng(
                random_seed
            )
            .random(
                len(
                    build_rows
                )
            )
        )

        output.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=random_scores,
                technique="Random",
                score_direction="descending",
            )
        )

    return pd.concat(
        output,
        ignore_index=True,
    )


def create_history_baseline_rankings(
    noisy_training_history,
    clean_training_history,
    clean_evaluation_history,
    clean_evaluation_data,
    fixed_split,
):
    latest_failure_order = {}

    for row in (
        noisy_training_history
        .sort_values(
            [
                "BuildOrder",
                "JobKey",
                "TestKey",
            ],
            kind="mergesort",
        )
        .itertuples(
            index=False
        )
    ):
        test_key = str(
            row.TestKey
        )

        if int(
            row.NoisyVerdict
        ) != 0:
            latest_failure_order[
                test_key
            ] = int(
                row.BuildOrder
            )

    duration_sum = {}
    duration_count = {}

    for row in (
        clean_training_history
        .sort_values(
            [
                "BuildOrder",
                "JobKey",
                "TestKey",
            ],
            kind="mergesort",
        )
        .itertuples(
            index=False
        )
    ):
        test_key = str(
            row.TestKey
        )

        duration = float(
            row.Duration
        )

        if np.isfinite(
            duration
        ):
            duration_sum[
                test_key
            ] = (
                duration_sum.get(
                    test_key,
                    0.0,
                )
                + duration
            )

            duration_count[
                test_key
            ] = (
                duration_count.get(
                    test_key,
                    0,
                )
                + 1
            )

    raw_eval_by_build = {
        str(
            build_key
        ):
            group.copy()

        for build_key, group in (
            clean_evaluation_history.groupby(
                "BuildKey",
                sort=False,
            )
        )
    }

    model_eval_by_build = {
        str(
            build_key
        ):
            group.copy()

        for build_key, group in (
            clean_evaluation_data.groupby(
                "BuildKey",
                sort=False,
            )
        )
    }

    latest_rankings = []
    qtf_rankings = []

    for build_row in (
        fixed_split[
            fixed_split[
                "Partition"
            ].eq(
                "EVALUATION"
            )
        ]
        .sort_values(
            "BuildOrder",
            kind="mergesort",
        )
        .itertuples(
            index=False
        )
    ):
        build_key = str(
            build_row.BuildKey
        )

        # Rank before observing this clean evaluation build.
        if build_key in model_eval_by_build:
            build_tests = model_eval_by_build[
                build_key
            ].copy()

            latest_scores = []
            qtf_scores = []

            for test_key in (
                build_tests[
                    "TestKey"
                ].astype(str)
            ):
                latest_scores.append(
                    float(
                        latest_failure_order.get(
                            test_key,
                            -1,
                        )
                    )
                )

                count = duration_count.get(
                    test_key,
                    0,
                )

                if count > 0:
                    average_duration = (
                        duration_sum[
                            test_key
                        ]
                        / count
                    )

                else:
                    average_duration = np.inf

                qtf_scores.append(
                    float(
                        average_duration
                    )
                )

            latest_rankings.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=latest_scores,
                    technique="LatestFail",
                    score_direction="descending",
                )
            )

            qtf_rankings.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=qtf_scores,
                    technique="QTF-Avg",
                    score_direction="ascending",
                )
            )

        # Update histories only after ranking the current build.
        current_raw_rows = raw_eval_by_build.get(
            build_key
        )

        if current_raw_rows is None:
            continue

        for row in (
            current_raw_rows
            .sort_values(
                [
                    "JobKey",
                    "TestKey",
                ],
                kind="mergesort",
            )
            .itertuples(
                index=False
            )
        ):
            test_key = str(
                row.TestKey
            )

            if int(
                row.CleanVerdict
            ) != 0:
                latest_failure_order[
                    test_key
                ] = int(
                    row.BuildOrder
                )

            duration = float(
                row.Duration
            )

            if np.isfinite(
                duration
            ):
                duration_sum[
                    test_key
                ] = (
                    duration_sum.get(
                        test_key,
                        0.0,
                    )
                    + duration
                )

                duration_count[
                    test_key
                ] = (
                    duration_count.get(
                        test_key,
                        0,
                    )
                    + 1
                )

    return (
        pd.concat(
            latest_rankings,
            ignore_index=True,
        ),
        pd.concat(
            qtf_rankings,
            ignore_index=True,
        ),
    )


def create_ml_rankings(
    evaluation_data,
    scores,
    technique,
):
    scoring_frame = evaluation_data.copy()

    scoring_frame[
        "__Score"
    ] = np.asarray(
        scores,
        dtype=float,
    )

    output = []

    for _, build_rows in (
        scoring_frame.groupby(
            "BuildKey",
            sort=False,
        )
    ):
        output.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=build_rows[
                    "__Score"
                ].to_numpy(
                    dtype=float
                ),
                technique=technique,
                score_direction="descending",
            )
        )

    return pd.concat(
        output,
        ignore_index=True,
    )


# ------------------------------------------------------------
# 6. NOISY REC HELPERS
# ------------------------------------------------------------

def calculate_rates(
    history,
):
    history_length = len(
        history
    )

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history[
        "NoisyVerdict"
    ].to_numpy(
        dtype=np.int32
    )

    transitions = history[
        "Transition"
    ].to_numpy(
        dtype=np.int32
    )

    return (
        float(
            np.count_nonzero(
                verdicts != 0
            )
            / history_length
        ),
        float(
            np.count_nonzero(
                verdicts == 2
            )
            / history_length
        ),
        float(
            np.count_nonzero(
                verdicts == 1
            )
            / history_length
        ),
        float(
            np.count_nonzero(
                transitions != 0
            )
            / history_length
        ),
    )


def calculate_max_test_file_rate(
    history,
    target_type,
    current_changed_entities,
    entity_changed_builds,
):
    if target_type == "FAILURE":
        target_builds = set(
            history.loc[
                history[
                    "NoisyVerdict"
                ].ne(0),
                "BuildKey",
            ].astype(str)
        )

    elif target_type == "TRANSITION":
        target_builds = set(
            history.loc[
                history[
                    "Transition"
                ].ne(0),
                "BuildKey",
            ].astype(str)
        )

    else:
        raise ValueError(
            "Unknown file-history target."
        )

    if not target_builds:
        return -1.0

    maximum_overlap = 0

    for entity_id in current_changed_entities:
        overlap = len(
            entity_changed_builds.get(
                str(
                    entity_id
                ),
                set(),
            )
            & target_builds
        )

        if overlap > maximum_overlap:
            maximum_overlap = overlap

    if maximum_overlap == 0:
        return 0.0

    return float(
        maximum_overlap
        / len(
            target_builds
        )
    )


def reconstruct_direct_noisy_rec(
    noisy_raw_training,
    requested_rows,
    requested_builds_by_test,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
):
    records = []

    for test_key, test_history in (
        noisy_raw_training.groupby(
            "TestKey",
            sort=False,
        )
    ):
        test_key = str(
            test_key
        )

        requested_builds = (
            requested_builds_by_test.get(
                test_key
            )
        )

        if not requested_builds:
            continue

        test_history = (
            test_history.sort_values(
                [
                    "BuildOrder",
                    "JobKey",
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
            .copy()
        )

        test_history[
            "Transition"
        ] = (
            test_history[
                "NoisyVerdict"
            ]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(np.int32)
        )

        first_test_build = str(
            test_history[
                "BuildKey"
            ].iloc[0]
        )

        for execution_position in range(
            len(
                test_history
            )
        ):
            current_row = test_history.iloc[
                execution_position
            ]

            current_build = str(
                current_row[
                    "BuildKey"
                ]
            )

            if current_build not in requested_builds:
                continue

            record = {
                "BuildKey":
                    current_build,

                "TestKey":
                    test_key,
            }

            history = test_history.iloc[
                :execution_position
            ]

            if history.empty:
                for feature in REC_FEATURE_COLUMNS:
                    record[
                        feature
                    ] = -1.0

                record[
                    "REC_Age"
                ] = 0.0

                records.append(
                    record
                )

                continue

            recent_history = history.tail(
                RECENT_WINDOW
            )

            age = float(
                global_build_position[
                    current_build
                ]
                - global_build_position[
                    first_test_build
                ]
            )

            failure_positions = np.flatnonzero(
                history[
                    "NoisyVerdict"
                ].to_numpy(
                    dtype=np.int32
                ) != 0
            )

            last_failure_age = (
                -1.0
                if len(
                    failure_positions
                ) == 0
                else float(
                    len(
                        history
                    )
                    - 1
                    - int(
                        failure_positions[-1]
                    )
                )
            )

            transition_positions = np.flatnonzero(
                history[
                    "Transition"
                ].to_numpy(
                    dtype=np.int32
                ) != 0
            )

            last_transition_age = (
                -1.0
                if len(
                    transition_positions
                ) == 0
                else float(
                    len(
                        history
                    )
                    - 1
                    - int(
                        transition_positions[-1]
                    )
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(
                recent_history
            )

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(
                history
            )

            current_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            record.update({
                "REC_Age":
                    age,

                "REC_LastFailureAge":
                    last_failure_age,

                "REC_LastTransitionAge":
                    last_transition_age,

                "REC_RecentAvgExeTime":
                    float(
                        recent_history[
                            "Duration"
                        ].mean()
                    ),

                "REC_RecentMaxExeTime":
                    float(
                        recent_history[
                            "Duration"
                        ].max()
                    ),

                "REC_RecentFailRate":
                    recent_fail_rate,

                "REC_RecentAssertRate":
                    recent_assert_rate,

                "REC_RecentExcRate":
                    recent_exc_rate,

                "REC_RecentTransitionRate":
                    recent_transition_rate,

                "REC_TotalAvgExeTime":
                    float(
                        history[
                            "Duration"
                        ].mean()
                    ),

                "REC_TotalMaxExeTime":
                    float(
                        history[
                            "Duration"
                        ].max()
                    ),

                "REC_TotalFailRate":
                    total_fail_rate,

                "REC_TotalAssertRate":
                    total_assert_rate,

                "REC_TotalExcRate":
                    total_exc_rate,

                "REC_TotalTransitionRate":
                    total_transition_rate,

                "REC_LastVerdict":
                    float(
                        recent_history[
                            "NoisyVerdict"
                        ].iloc[-1]
                    ),

                "REC_LastExeTime":
                    float(
                        recent_history[
                            "Duration"
                        ].iloc[-1]
                    ),

                "REC_MaxTestFileFailRate":
                    calculate_max_test_file_rate(
                        history=history,
                        target_type="FAILURE",
                        current_changed_entities=current_entities,
                        entity_changed_builds=entity_changed_builds,
                    ),

                "REC_MaxTestFileTransitionRate":
                    calculate_max_test_file_rate(
                        history=history,
                        target_type="TRANSITION",
                        current_changed_entities=current_entities,
                        entity_changed_builds=entity_changed_builds,
                    ),
            })

            records.append(
                record
            )

    reconstructed = pd.DataFrame(
        records,
        columns=(
            [
                "BuildKey",
                "TestKey",
            ]
            + REC_FEATURE_COLUMNS
        ),
    )

    duplicate_rows = int(
        reconstructed.duplicated(
            subset=[
                "BuildKey",
                "TestKey",
            ],
            keep=False,
        ).sum()
    )

    if duplicate_rows:
        raise RuntimeError(
            "Noisy REC reconstruction contains duplicate rows."
        )

    aligned = (
        requested_rows.merge(
            reconstructed,
            on=[
                "BuildKey",
                "TestKey",
            ],
            how="left",
            validate="one_to_one",
            indicator=True,
            sort=False,
        )
        .sort_values(
            "ModelRowOrder",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    missing_rows = int(
        aligned[
            "_merge"
        ].ne(
            "both"
        ).sum()
    )

    if missing_rows:
        raise RuntimeError(
            "Some requested training rows lack noisy REC values."
        )

    return (
        aligned[
            REC_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="raise",
        )
    )


# ------------------------------------------------------------
# 7. MODEL HELPERS
# ------------------------------------------------------------

def instantiate_model(
    technique,
    model_config,
    model_seeds,
):
    if technique == "RandomForest":
        return RandomForestClassifier(
            **model_config[
                "RandomForest"
            ],
            random_state=int(
                model_seeds[
                    "RandomForestSeed"
                ]
            ),
        )

    if technique == "XGBoost":
        return XGBClassifier(
            **model_config[
                "XGBoost"
            ],
            random_state=int(
                model_seeds[
                    "XGBoostSeed"
                ]
            ),
        )

    if technique == "LightGBM":
        return LGBMClassifier(
            **model_config[
                "LightGBM"
            ],
            random_state=int(
                model_seeds[
                    "LightGBMSeed"
                ]
            ),
        )

    if technique == "NaiveBayes":
        return GaussianNB(
            var_smoothing=float(
                model_config[
                    "NaiveBayes"
                ][
                    "var_smoothing"
                ]
            )
        )

    raise ValueError(
        f"Unknown ML technique: {technique}"
    )


def positive_class_probability(
    fitted_model,
    X_evaluation,
):
    probabilities = fitted_model.predict_proba(
        X_evaluation
    )

    classes = np.asarray(
        fitted_model.classes_
    )

    positive_positions = np.flatnonzero(
        classes == 1
    )

    if len(
        positive_positions
    ) != 1:
        raise RuntimeError(
            "Fitted model does not contain exactly one positive class."
        )

    scores = probabilities[
        :,
        int(
            positive_positions[0]
        ),
    ].astype(float)

    if not np.isfinite(
        scores
    ).all():
        raise RuntimeError(
            "Predicted probabilities contain non-finite values."
        )

    if (
        scores < -1e-12
    ).any() or (
        scores > 1.0 + 1e-12
    ).any():
        raise RuntimeError(
            "Predicted probabilities fall outside [0, 1]."
        )

    return np.clip(
        scores,
        0.0,
        1.0,
    )


# ------------------------------------------------------------
# 8. METRIC AGGREGATION HELPERS
# ------------------------------------------------------------

def calculate_build_metrics(
    rankings,
):
    records = []

    grouped = rankings.groupby(
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "SmokeConditionOrder",
            "ConditionID",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "BuildKey",
        ],
        sort=False,
    )

    for keys, ranked_build in grouped:
        (
            project_number,
            project,
            project_slug,
            smoke_condition_order,
            condition_id,
            noise_percent,
            repetition_seed,
            technique,
            build_key,
        ) = keys

        ranked_build = (
            ranked_build.sort_values(
                "Rank",
                kind="mergesort",
            )
        )

        failures = ranked_build[
            "ActualFailure"
        ].to_numpy(
            dtype=np.int8
        )

        durations = ranked_build[
            "Duration"
        ].to_numpy(
            dtype=float
        )

        records.append({
            "ProjectNumber":
                project_number,

            "Project":
                project,

            "ProjectSlug":
                project_slug,

            "SmokeConditionOrder":
                smoke_condition_order,

            "ConditionID":
                condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "Technique":
                technique,

            "Build":
                ranked_build[
                    "Build"
                ].iloc[0],

            "BuildKey":
                str(
                    build_key
                ),

            "BuildOrder":
                int(
                    ranked_build[
                        "build_order"
                    ].iloc[0]
                ),

            "NumberOfTests":
                len(
                    ranked_build
                ),

            "NumberOfFailures":
                int(
                    failures.sum()
                ),

            "TotalDuration":
                float(
                    durations.sum()
                ),

            "APFD":
                calculate_apfd(
                    failures
                ),

            "APFDc":
                calculate_apfdc(
                    failures,
                    durations,
                ),
        })

    return pd.DataFrame(
        records
    )


def calculate_project_runs(
    build_metrics,
):
    records = []

    grouped = build_metrics.groupby(
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "SmokeConditionOrder",
            "ConditionID",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        sort=False,
    )

    for keys, group in grouped:
        (
            project_number,
            project,
            project_slug,
            smoke_condition_order,
            condition_id,
            noise_percent,
            repetition_seed,
            technique,
        ) = keys

        records.append({
            "ProjectNumber":
                project_number,

            "Project":
                project,

            "ProjectSlug":
                project_slug,

            "SmokeConditionOrder":
                smoke_condition_order,

            "ConditionID":
                condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "Technique":
                technique,

            "EvaluatedBuilds":
                int(
                    group[
                        "BuildKey"
                    ].nunique()
                ),

            "EvaluationTests":
                int(
                    group[
                        "NumberOfTests"
                    ].sum()
                ),

            "EvaluationFailures":
                int(
                    group[
                        "NumberOfFailures"
                    ].sum()
                ),

            "MeanAPFD":
                float(
                    group[
                        "APFD"
                    ].mean()
                ),

            "MedianAPFD":
                float(
                    group[
                        "APFD"
                    ].median()
                ),

            "StdAPFD":
                float(
                    group[
                        "APFD"
                    ].std(
                        ddof=0
                    )
                ),

            "MeanAPFDc":
                float(
                    group[
                        "APFDc"
                    ].mean()
                ),

            "MedianAPFDc":
                float(
                    group[
                        "APFDc"
                    ].median()
                ),

            "StdAPFDc":
                float(
                    group[
                        "APFDc"
                    ].std(
                        ddof=0
                    )
                ),
        })

    return pd.DataFrame(
        records
    )


def ranking_hash_for_condition_technique(
    rankings,
    condition_id,
    technique,
):
    subset = (
        rankings[
            rankings[
                "ConditionID"
            ].eq(
                condition_id
            )
            & rankings[
                "Technique"
            ].eq(
                technique
            )
        ]
        .sort_values(
            [
                "BuildKey",
                "Rank",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    return dataframe_semantic_sha256(
        subset,
        [
            "BuildKey",
            "TestKey",
            "Score",
            "Rank",
        ],
    )


# ------------------------------------------------------------
# 9. LOAD CHECKPOINTS
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    NOISY_REC_ENGINE_CHECKPOINT_PATH,
    MODEL_PROTOCOL_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
    STEP3A_STATUS_PATH,
    STEP3B_STATUS_PATH,
    STEP4A_STATUS_PATH,
]


missing_inputs = [
    str(
        path
    )
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:
    raise FileNotFoundError(
        "Required Project 10 Step 4B inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


selection_checkpoint = read_json_with_retry(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint = read_json_with_retry(
    REC_CHECKPOINT_PATH
)

noise_checkpoint = read_json_with_retry(
    NOISE_PLAN_CHECKPOINT_PATH
)

noisy_rec_checkpoint = read_json_with_retry(
    NOISY_REC_ENGINE_CHECKPOINT_PATH
)

model_checkpoint = read_json_with_retry(
    MODEL_PROTOCOL_CHECKPOINT_PATH
)

step2b_status = read_json_with_retry(
    STEP2B_STATUS_PATH
)

step3a_status = read_json_with_retry(
    STEP3A_STATUS_PATH
)

step3b_status = read_json_with_retry(
    STEP3B_STATUS_PATH
)

step4a_status = read_json_with_retry(
    STEP4A_STATUS_PATH
)


status_expectations = [
    (
        "Step 2B",
        step2b_status.get(
            "Status"
        ),
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "Step 3A",
        step3a_status.get(
            "Status"
        ),
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "Step 3B",
        step3b_status.get(
            "Status"
        ),
        EXPECTED_STEP3B_STATUS,
    ),
    (
        "Step 4A",
        step4a_status.get(
            "Status"
        ),
        EXPECTED_STEP4A_STATUS,
    ),
]


for step_name, actual, expected in (
    status_expectations
):
    if actual != expected:
        raise AssertionError(
            f"{step_name} status differs.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


if rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP2B_STATUS:
    raise AssertionError(
        "REC checkpoint status differs."
    )


if noise_checkpoint.get(
    "Status"
) != EXPECTED_STEP3A_STATUS:
    raise AssertionError(
        "Noise-plan checkpoint status differs."
    )


if noisy_rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP3B_STATUS:
    raise AssertionError(
        "Noisy REC checkpoint status differs."
    )


if model_checkpoint.get(
    "Status"
) != EXPECTED_STEP4A_STATUS:
    raise AssertionError(
        "Model-protocol checkpoint status differs."
    )


if selection_checkpoint.get(
    "Project"
) != PROJECT_NAME:
    raise AssertionError(
        "Project identity differs."
    )


if selection_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Source-root hash differs."
    )


# ------------------------------------------------------------
# 10. OUTPUT ISOLATION
# ------------------------------------------------------------

output_paths = [
    SMOKE_RANKINGS_PATH,
    SMOKE_BUILD_METRICS_PATH,
    SMOKE_PROJECT_RUN_PATH,
    SMOKE_MODEL_FITS_PATH,
    SMOKE_TRAINING_MEDIANS_PATH,
    SMOKE_CONDITION_AUDIT_PATH,
    SMOKE_INVARIANCE_AUDIT_PATH,
    STEP4B_VALIDATION_PATH,
    STEP4B_REPORT_PATH,
    SMOKE_TEST_CHECKPOINT_PATH,
    STEP4B_STATUS_PATH,
]


for output_path in output_paths:
    output_string = str(
        output_path
    )

    if (
        PROJECT_SLUG not in output_string
        and "project_10_" not in output_string
    ):
        raise AssertionError(
            "A Step 4B output is not Project 10 isolated.\n"
            f"Path: {output_path}"
        )

    if str(
        PROJECT_9_DIR
    ) in output_string:
        raise AssertionError(
            "A Project 10 output overlaps Project 9."
        )


# ------------------------------------------------------------
# 11. REGISTRY AND SOURCE SNAPSHOT
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


allowed_registry_sets = [
    set(
        range(
            1,
            9,
        )
    ),
    set(
        range(
            1,
            10,
        )
    ),
]


if set(
    registry_project_numbers
) not in allowed_registry_sets:
    raise AssertionError(
        "Registry must contain Projects 1–8, or Projects 1–9 "
        "if Project 9 has already been serially frozen."
    )


if registry_project_numbers.eq(
    10
).any():
    raise AssertionError(
        "Project 10 was unexpectedly registered."
    )


source_files_payload = (
    selection_checkpoint[
        "SourceFiles"
    ]
)

source_root_before = create_source_root_sha256(
    source_files_payload
)


if source_root_before != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Project 10 source differs before Step 4B."
    )


# ------------------------------------------------------------
# 12. LOAD FROZEN ARTIFACTS
# ------------------------------------------------------------

raw_training_path = Path(
    noise_checkpoint[
        "RawTrainingCohort"
    ]
)

raw_evaluation_path = Path(
    noise_checkpoint[
        "RawEvaluationCohort"
    ]
)

noise_rng_manifest_path = Path(
    noise_checkpoint[
        "NoiseRNGManifest"
    ]
)

condition_plan_path = Path(
    noise_checkpoint[
        "ConditionPlan"
    ]
)

clean_direct_rec_path = Path(
    rec_checkpoint[
        "CleanRECReconstructed"
    ]
)

build_entity_map_path = Path(
    rec_checkpoint[
        "BuildEntityMap"
    ]
)

clean_training_base_path = Path(
    model_checkpoint[
        "CleanModelTrainingBase"
    ]
)

clean_evaluation_base_path = Path(
    model_checkpoint[
        "CleanModelEvaluationBase"
    ]
)

model_seed_manifest_path = Path(
    model_checkpoint[
        "ModelSeedManifest"
    ]
)

fixed_split_path = Path(
    selection_checkpoint[
        "FixedSplit"
    ]
)


frozen_paths = [
    raw_training_path,
    raw_evaluation_path,
    noise_rng_manifest_path,
    condition_plan_path,
    clean_direct_rec_path,
    build_entity_map_path,
    clean_training_base_path,
    clean_evaluation_base_path,
    model_seed_manifest_path,
    fixed_split_path,
]


missing_frozen_paths = [
    str(
        path
    )
    for path in frozen_paths
    if not path.exists()
]


if missing_frozen_paths:
    raise FileNotFoundError(
        "Frozen smoke-test inputs are missing:\n"
        + "\n".join(
            missing_frozen_paths
        )
    )


raw_evaluation_sha256_before = calculate_hash(
    raw_evaluation_path
)

clean_evaluation_base_sha256_before = calculate_hash(
    clean_evaluation_base_path
)


raw_training = pd.read_parquet(
    raw_training_path
)

raw_evaluation = pd.read_parquet(
    raw_evaluation_path
)

noise_rng_manifest = pd.read_parquet(
    noise_rng_manifest_path
)

condition_plan = pd.read_csv(
    condition_plan_path,
    low_memory=False,
)

clean_direct_rec = pd.read_parquet(
    clean_direct_rec_path
)

build_entity_map = pd.read_csv(
    build_entity_map_path,
    compression="gzip",
    low_memory=False,
)

clean_training_base = pd.read_parquet(
    clean_training_base_path
)

clean_evaluation_base = pd.read_parquet(
    clean_evaluation_base_path
)

model_seed_manifest = pd.read_csv(
    model_seed_manifest_path,
    low_memory=False,
)

fixed_split = pd.read_csv(
    fixed_split_path,
    low_memory=False,
)


# ------------------------------------------------------------
# 13. RESOLVE FROZEN PROTOCOL
# ------------------------------------------------------------

training_metadata_columns = (
    model_checkpoint[
        "TrainingMetadataColumns"
    ]
)

evaluation_metadata_columns = (
    model_checkpoint[
        "EvaluationMetadataColumns"
    ]
)

active_feature_columns = list(
    model_checkpoint[
        "ActiveFeatureColumns"
    ]
)

model_configuration = (
    model_checkpoint[
        "ModelConfiguration"
    ][
        "Models"
    ]
)


if len(
    active_feature_columns
) != EXPECTED_ACTIVE_PREDICTORS:
    raise AssertionError(
        "Active predictor count differs."
    )


meta_training_order = (
    training_metadata_columns[
        "ModelRowOrder"
    ]
)

meta_noise_row = (
    training_metadata_columns[
        "NoiseRowID"
    ]
)

meta_training_build_key = (
    training_metadata_columns[
        "BuildKey"
    ]
)

meta_training_test_key = (
    training_metadata_columns[
        "TestKey"
    ]
)

meta_training_failure = (
    training_metadata_columns[
        "BinaryFailure"
    ]
)


meta_eval_build = (
    evaluation_metadata_columns[
        "Build"
    ]
)

meta_eval_test = (
    evaluation_metadata_columns[
        "Test"
    ]
)

meta_eval_verdict = (
    evaluation_metadata_columns[
        "Verdict"
    ]
)

meta_eval_failure = (
    evaluation_metadata_columns[
        "BinaryFailure"
    ]
)

meta_eval_duration = (
    evaluation_metadata_columns[
        "Duration"
    ]
)

meta_eval_build_order = (
    evaluation_metadata_columns[
        "BuildOrder"
    ]
)

meta_eval_build_key = (
    evaluation_metadata_columns[
        "BuildKey"
    ]
)

meta_eval_test_key = (
    evaluation_metadata_columns[
        "TestKey"
    ]
)


if clean_training_base.columns.duplicated().any():
    raise AssertionError(
        "Clean training base contains duplicate columns."
    )


if clean_evaluation_base.columns.duplicated().any():
    raise AssertionError(
        "Clean evaluation base contains duplicate columns."
    )


if len(
    raw_training
) != EXPECTED_RAW_TRAINING_ROWS:
    raise AssertionError(
        "Raw training row count differs."
    )


if len(
    raw_evaluation
) != EXPECTED_RAW_EVALUATION_ROWS:
    raise AssertionError(
        "Raw evaluation row count differs."
    )


if len(
    clean_training_base
) != EXPECTED_MODEL_TRAINING_ROWS:
    raise AssertionError(
        "Model training row count differs."
    )


if len(
    clean_evaluation_base
) != EXPECTED_MODEL_EVALUATION_ROWS:
    raise AssertionError(
        "Model evaluation row count differs."
    )


# ------------------------------------------------------------
# 14. CANONICALISE DATA
# ------------------------------------------------------------

for raw_frame in [
    raw_training,
    raw_evaluation,
]:
    raw_frame[
        "BuildKey"
    ] = raw_frame[
        "BuildKey"
    ].astype(str)

    raw_frame[
        "TestKey"
    ] = raw_frame[
        "TestKey"
    ].astype(str)

    raw_frame[
        "JobKey"
    ] = raw_frame[
        "JobKey"
    ].astype(str)

    raw_frame[
        "BuildOrder"
    ] = pd.to_numeric(
        raw_frame[
            "BuildOrder"
        ],
        errors="raise",
    ).astype(np.int32)

    raw_frame[
        "CleanVerdict"
    ] = pd.to_numeric(
        raw_frame[
            "CleanVerdict"
        ],
        errors="raise",
    ).astype(np.int8)

    raw_frame[
        "Duration"
    ] = pd.to_numeric(
        raw_frame[
            "Duration"
        ],
        errors="raise",
    ).astype(float)


raw_training[
    "NoiseRowID"
] = pd.to_numeric(
    raw_training[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)


raw_training = (
    raw_training.sort_values(
        [
            "NoiseRowID",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


expected_noise_ids = np.arange(
    EXPECTED_RAW_TRAINING_ROWS,
    dtype=np.int32,
)


if not np.array_equal(
    raw_training[
        "NoiseRowID"
    ].to_numpy(
        dtype=np.int32
    ),
    expected_noise_ids,
):
    raise AssertionError(
        "Raw-training NoiseRowID order differs."
    )


noise_rng_manifest[
    "RepetitionSeed"
] = pd.to_numeric(
    noise_rng_manifest[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(np.int16)

noise_rng_manifest[
    "NoiseRowID"
] = pd.to_numeric(
    noise_rng_manifest[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)

noise_rng_manifest[
    "FlipUniform"
] = pd.to_numeric(
    noise_rng_manifest[
        "FlipUniform"
    ],
    errors="raise",
).astype(float)

noise_rng_manifest[
    "SampledFailureSubtype"
] = pd.to_numeric(
    noise_rng_manifest[
        "SampledFailureSubtype"
    ],
    errors="raise",
).astype(np.int8)


condition_plan[
    "NoisePercent"
] = pd.to_numeric(
    condition_plan[
        "NoisePercent"
    ],
    errors="raise",
).astype(int)

condition_plan[
    "RepetitionSeed"
] = pd.to_numeric(
    condition_plan[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(int)


fixed_split[
    "BuildKey"
] = fixed_split[
    "BuildKey"
].astype(str)

fixed_split[
    "BuildOrder"
] = pd.to_numeric(
    fixed_split[
        "BuildOrder"
    ],
    errors="raise",
).astype(np.int32)


clean_direct_rec[
    "BuildKey"
] = clean_direct_rec[
    "BuildKey"
].astype(str)

clean_direct_rec[
    "TestKey"
] = clean_direct_rec[
    "TestKey"
].astype(str)


clean_training_base[
    meta_training_build_key
] = clean_training_base[
    meta_training_build_key
].astype(str)

clean_training_base[
    meta_training_test_key
] = clean_training_base[
    meta_training_test_key
].astype(str)

clean_training_base[
    meta_training_order
] = pd.to_numeric(
    clean_training_base[
        meta_training_order
    ],
    errors="raise",
).astype(np.int64)

clean_training_base[
    meta_noise_row
] = pd.to_numeric(
    clean_training_base[
        meta_noise_row
    ],
    errors="raise",
).astype(np.int32)


for column in [
    meta_eval_build_key,
    meta_eval_test_key,
]:
    clean_evaluation_base[
        column
    ] = clean_evaluation_base[
        column
    ].astype(str)


# ------------------------------------------------------------
# 15. PREPARE REC ALIGNMENT AND ENTITY HISTORY
# ------------------------------------------------------------

training_model_orders = clean_training_base[
    meta_training_order
].to_numpy(
    dtype=np.int64
)


direct_clean_training_rec = (
    clean_direct_rec.iloc[
        training_model_orders
    ][
        REC_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .reset_index(
        drop=True
    )
)


original_clean_training_rec = (
    clean_training_base[
        REC_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .reset_index(
        drop=True
    )
)


direct_clean_matrix = (
    direct_clean_training_rec.to_numpy(
        dtype=float
    )
)

original_clean_matrix = (
    original_clean_training_rec.to_numpy(
        dtype=float
    )
)


requested_rows = pd.DataFrame({
    "ModelRowOrder":
        clean_training_base[
            meta_training_order
        ].to_numpy(
            dtype=np.int64
        ),

    "BuildKey":
        clean_training_base[
            meta_training_build_key
        ].astype(str).to_numpy(),

    "TestKey":
        clean_training_base[
            meta_training_test_key
        ].astype(str).to_numpy(),
})


requested_builds_by_test = {
    str(
        test_key
    ):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )

    for test_key, group in (
        requested_rows.groupby(
            "TestKey",
            sort=False,
        )
    )
}


ordered_training_builds = (
    raw_training[
        [
            "BuildKey",
            "BuildOrder",
        ]
    ]
    .drop_duplicates(
        subset=[
            "BuildKey",
        ]
    )
    .sort_values(
        "BuildOrder",
        kind="mergesort",
    )[
        "BuildKey"
    ]
    .astype(str)
    .tolist()
)


global_build_position = {
    build_key:
        position

    for position, build_key in enumerate(
        ordered_training_builds
    )
}


build_entity_map[
    "BuildKey"
] = build_entity_map[
    "BuildKey"
].astype(str)

build_entity_map[
    "EntityId"
] = build_entity_map[
    "EntityId"
].astype(str)


changed_entities_by_build = {
    str(
        build_key
    ):
        set(
            group[
                "EntityId"
            ].astype(str)
        )

    for build_key, group in (
        build_entity_map.groupby(
            "BuildKey",
            sort=False,
        )
    )
}


entity_changed_builds = defaultdict(
    set
)


for row in build_entity_map.itertuples(
    index=False
):
    entity_changed_builds[
        str(
            row.EntityId
        )
    ].add(
        str(
            row.BuildKey
        )
    )


dependent_positions = [
    REC_FEATURE_COLUMNS.index(
        feature
    )
    for feature in VERDICT_DEPENDENT_REC_FEATURES
]

independent_positions = [
    REC_FEATURE_COLUMNS.index(
        feature
    )
    for feature in VERDICT_INDEPENDENT_REC_FEATURES
]


# ------------------------------------------------------------
# 16. PREPARE CLEAN EVALUATION FRAME
# ------------------------------------------------------------

baseline_evaluation_data = pd.DataFrame({
    "Build":
        clean_evaluation_base[
            meta_eval_build
        ].to_numpy(),

    "Test":
        clean_evaluation_base[
            meta_eval_test
        ].to_numpy(),

    "Verdict":
        pd.to_numeric(
            clean_evaluation_base[
                meta_eval_verdict
            ],
            errors="raise",
        ).astype(np.int8).to_numpy(),

    "Duration":
        pd.to_numeric(
            clean_evaluation_base[
                meta_eval_duration
            ],
            errors="raise",
        ).astype(float).to_numpy(),

    "build_order":
        pd.to_numeric(
            clean_evaluation_base[
                meta_eval_build_order
            ],
            errors="raise",
        ).astype(np.int32).to_numpy(),

    "BuildKey":
        clean_evaluation_base[
            meta_eval_build_key
        ].astype(str).to_numpy(),

    "TestKey":
        clean_evaluation_base[
            meta_eval_test_key
        ].astype(str).to_numpy(),
})


if baseline_evaluation_data.columns.duplicated().any():
    raise AssertionError(
        "Evaluation ranking frame contains duplicate columns."
    )


clean_evaluation_binary_failures = (
    pd.to_numeric(
        clean_evaluation_base[
            meta_eval_failure
        ],
        errors="raise",
    )
    .astype(np.int8)
    .to_numpy()
)


if int(
    clean_evaluation_binary_failures.sum()
) != EXPECTED_MODEL_EVALUATION_FAILURES:
    raise AssertionError(
        "Clean evaluation failure count differs."
    )


X_evaluation_clean = (
    clean_evaluation_base[
        active_feature_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


# ------------------------------------------------------------
# 17. SEED MANIFEST
# ------------------------------------------------------------

model_seed_manifest[
    "RepetitionSeed"
] = pd.to_numeric(
    model_seed_manifest[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(int)


seed_one_rows = model_seed_manifest[
    model_seed_manifest[
        "RepetitionSeed"
    ].eq(1)
]


if len(
    seed_one_rows
) != 1:
    raise AssertionError(
        "Model seed 1 was not found exactly once."
    )


seed_one_model_values = (
    seed_one_rows.iloc[0]
)


# ------------------------------------------------------------
# 18. RUN TWO COMPLETE CONDITIONS
# ------------------------------------------------------------

all_ranking_frames = []
model_fit_records = []
training_median_records = []
condition_audit_records = []

clean_raw_verdicts = raw_training[
    "CleanVerdict"
].to_numpy(
    dtype=np.int8
)

model_noise_row_ids = clean_training_base[
    meta_noise_row
].to_numpy(
    dtype=np.int32
)


smoke_started = time.perf_counter()


for smoke_condition_order, (
    noise_percent,
    repetition_seed,
) in enumerate(
    SMOKE_CONDITIONS,
    start=1,
):
    condition_started = (
        time.perf_counter()
    )

    condition_rows = condition_plan[
        condition_plan[
            "NoisePercent"
        ].eq(
            noise_percent
        )
        & condition_plan[
            "RepetitionSeed"
        ].eq(
            repetition_seed
        )
    ]


    if len(
        condition_rows
    ) != 1:
        raise AssertionError(
            "Smoke condition was not found exactly once."
        )


    condition_record = condition_rows.iloc[
        0
    ]

    condition_id = str(
        condition_record[
            "ConditionID"
        ]
    )


    rng_rows = (
        noise_rng_manifest[
            noise_rng_manifest[
                "RepetitionSeed"
            ].eq(
                repetition_seed
            )
        ]
        .sort_values(
            "NoiseRowID",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


    if len(
        rng_rows
    ) != EXPECTED_RAW_TRAINING_ROWS:
        raise AssertionError(
            "Smoke RNG row count differs."
        )


    if not np.array_equal(
        rng_rows[
            "NoiseRowID"
        ].to_numpy(
            dtype=np.int32
        ),
        expected_noise_ids,
    ):
        raise AssertionError(
            "Smoke RNG NoiseRowID order differs."
        )


    flip_uniforms = rng_rows[
        "FlipUniform"
    ].to_numpy(
        dtype=float
    )

    sampled_failure_subtypes = rng_rows[
        "SampledFailureSubtype"
    ].to_numpy(
        dtype=np.int8
    )


    flip_mask = (
        flip_uniforms
        < noise_percent / 100.0
    )


    noisy_raw_verdicts = (
        clean_raw_verdicts.copy()
    )


    pass_to_failure_mask = (
        flip_mask
        & (
            clean_raw_verdicts
            == 0
        )
    )

    failure_to_pass_mask = (
        flip_mask
        & (
            clean_raw_verdicts
            != 0
        )
    )


    noisy_raw_verdicts[
        pass_to_failure_mask
    ] = sampled_failure_subtypes[
        pass_to_failure_mask
    ]

    noisy_raw_verdicts[
        failure_to_pass_mask
    ] = 0


    noisy_model_verdicts = (
        noisy_raw_verdicts[
            model_noise_row_ids
        ]
    )

    noisy_binary_labels = (
        noisy_model_verdicts
        != 0
    ).astype(
        np.int8
    )


    actual_flip_count = int(
        flip_mask.sum()
    )

    actual_raw_label_changes = int(
        np.count_nonzero(
            noisy_raw_verdicts
            != clean_raw_verdicts
        )
    )

    actual_model_label_changes = int(
        np.count_nonzero(
            noisy_model_verdicts
            != clean_training_base[
                meta_training_failure
            ].to_numpy(
                dtype=np.int8
            )
        )
    )

    # The previous comparison uses binary clean labels, so recreate
    # the protocol's subtype-level model verdict comparison separately.
    clean_model_raw_verdicts = raw_training[
        "CleanVerdict"
    ].to_numpy(
        dtype=np.int8
    )[
        model_noise_row_ids
    ]

    actual_model_verdict_changes = int(
        np.count_nonzero(
            noisy_model_verdicts
            != clean_model_raw_verdicts
        )
    )


    expected_flip_count = int(
        condition_record[
            "NumberFlipped"
        ]
    )

    expected_raw_label_changes = int(
        condition_record[
            "RawLabelChanges"
        ]
    )

    expected_model_label_changes = int(
        condition_record[
            "ModelLabelChanges"
        ]
    )


    flip_hash_match = bool(
        array_sha256(
            flip_mask.astype(
                np.uint8
            ),
            "<u1",
        )
        == str(
            condition_record[
                "FlipMaskSHA256"
            ]
        )
    )

    raw_verdict_hash_match = bool(
        array_sha256(
            noisy_raw_verdicts,
            "<i1",
        )
        == str(
            condition_record[
                "NoisyRawVerdictSHA256"
            ]
        )
    )

    model_verdict_hash_match = bool(
        array_sha256(
            noisy_model_verdicts,
            "<i1",
        )
        == str(
            condition_record[
                "NoisyModelVerdictSHA256"
            ]
        )
    )


    if actual_flip_count != expected_flip_count:
        raise AssertionError(
            f"{condition_id}: flip count differs."
        )


    if (
        actual_raw_label_changes
        != expected_raw_label_changes
    ):
        raise AssertionError(
            f"{condition_id}: raw-label change count differs."
        )


    if (
        actual_model_verdict_changes
        != expected_model_label_changes
    ):
        raise AssertionError(
            f"{condition_id}: model-label change count differs."
        )


    if not (
        flip_hash_match
        and raw_verdict_hash_match
        and model_verdict_hash_match
    ):
        raise AssertionError(
            f"{condition_id}: frozen noise hashes differ."
        )


    noisy_raw_history = raw_training.copy()

    noisy_raw_history[
        "NoisyVerdict"
    ] = noisy_raw_verdicts


    rec_started = time.perf_counter()


    direct_noisy_rec = reconstruct_direct_noisy_rec(
        noisy_raw_training=noisy_raw_history,
        requested_rows=requested_rows,
        requested_builds_by_test=requested_builds_by_test,
        global_build_position=global_build_position,
        changed_entities_by_build=changed_entities_by_build,
        entity_changed_builds=entity_changed_builds,
    )


    rec_seconds = float(
        time.perf_counter()
        - rec_started
    )


    direct_noisy_matrix = (
        direct_noisy_rec.to_numpy(
            dtype=float
        )
    )

    direct_delta_matrix = (
        direct_noisy_matrix
        - direct_clean_matrix
    )


    anchored_noisy_matrix = (
        original_clean_matrix.copy()
    )


    anchored_noisy_matrix[
        :,
        dependent_positions,
    ] = (
        original_clean_matrix[
            :,
            dependent_positions,
        ]
        + direct_delta_matrix[
            :,
            dependent_positions,
        ]
    )


    anchored_noisy_matrix[
        :,
        independent_positions,
    ] = original_clean_matrix[
        :,
        independent_positions,
    ]


    independent_rec_changed_values = int(
        np.count_nonzero(
            anchored_noisy_matrix[
                :,
                independent_positions,
            ]
            != original_clean_matrix[
                :,
                independent_positions,
            ]
        )
    )


    dependent_match_clean = np.isclose(
        anchored_noisy_matrix[
            :,
            dependent_positions,
        ],
        original_clean_matrix[
            :,
            dependent_positions,
        ],
        rtol=1e-12,
        atol=1e-12,
        equal_nan=False,
    )


    dependent_rec_changed_values = int(
        (
            ~dependent_match_clean
        ).sum()
    )


    zero_noise_rec_mismatches = None

    if noise_percent == 0:
        zero_noise_rec_mismatches = int(
            (
                ~np.isclose(
                    anchored_noisy_matrix,
                    original_clean_matrix,
                    rtol=1e-12,
                    atol=1e-12,
                    equal_nan=False,
                )
            ).sum()
        )

        if zero_noise_rec_mismatches:
            raise AssertionError(
                "0% anchored REC does not reproduce clean training data."
            )


    if independent_rec_changed_values != 0:
        raise AssertionError(
            f"{condition_id}: independent REC features changed."
        )


    if (
        noise_percent > 0
        and dependent_rec_changed_values <= 0
    ):
        raise AssertionError(
            f"{condition_id}: dependent REC features did not change."
        )


    condition_training_features = (
        clean_training_base[
            active_feature_columns
        ]
        .copy()
    )


    noisy_rec_frame = pd.DataFrame(
        anchored_noisy_matrix,
        columns=REC_FEATURE_COLUMNS,
    )


    for feature in VERDICT_DEPENDENT_REC_FEATURES:
        condition_training_features[
            feature
        ] = noisy_rec_frame[
            feature
        ].to_numpy(
            dtype=float
        )


    # Explicitly preserve independent features.
    for feature in VERDICT_INDEPENDENT_REC_FEATURES:
        condition_training_features[
            feature
        ] = clean_training_base[
            feature
        ].to_numpy()


    X_training = (
        condition_training_features
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )


    condition_medians = (
        X_training.median(
            axis=0
        )
        .fillna(
            0.0
        )
    )


    X_training_imputed = (
        X_training
        .fillna(
            condition_medians
        )
        .astype(float)
    )


    X_evaluation_imputed = (
        X_evaluation_clean
        .fillna(
            condition_medians
        )
        .astype(float)
    )


    X_training_array = (
        X_training_imputed.to_numpy(
            dtype=float
        )
    )

    X_evaluation_array = (
        X_evaluation_imputed.to_numpy(
            dtype=float
        )
    )


    if not np.isfinite(
        X_training_array
    ).all():
        raise AssertionError(
            f"{condition_id}: training matrix contains non-finite values."
        )


    if not np.isfinite(
        X_evaluation_array
    ).all():
        raise AssertionError(
            f"{condition_id}: evaluation matrix contains non-finite values."
        )


    label_classes = sorted(
        np.unique(
            noisy_binary_labels
        ).tolist()
    )


    if label_classes != [
        0,
        1,
    ]:
        raise AssertionError(
            f"{condition_id}: training labels do not contain both classes."
        )


    for feature_order, feature in enumerate(
        active_feature_columns,
        start=1,
    ):
        training_median_records.append({
            "ProjectNumber":
                PROJECT_NUMBER,

            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "SmokeConditionOrder":
                smoke_condition_order,

            "ConditionID":
                condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "FeatureOrder":
                feature_order,

            "Feature":
                feature,

            "TrainingMedian":
                float(
                    condition_medians[
                        feature
                    ]
                ),
        })


    # --------------------------------------------------------
    # ML MODELS
    # --------------------------------------------------------

    for technique in ML_TECHNIQUES:
        model = instantiate_model(
            technique=technique,
            model_config=model_configuration,
            model_seeds=seed_one_model_values,
        )


        fit_started = time.perf_counter()

        model.fit(
            X_training_array,
            noisy_binary_labels,
        )

        fit_seconds = float(
            time.perf_counter()
            - fit_started
        )


        predict_started = time.perf_counter()

        scores = positive_class_probability(
            fitted_model=model,
            X_evaluation=X_evaluation_array,
        )

        predict_seconds = float(
            time.perf_counter()
            - predict_started
        )


        ml_rankings = create_ml_rankings(
            evaluation_data=baseline_evaluation_data,
            scores=scores,
            technique=technique,
        )


        ml_rankings = add_condition_metadata(
            ranking=ml_rankings,
            smoke_condition_order=smoke_condition_order,
            condition_id=condition_id,
            noise_percent=noise_percent,
            repetition_seed=repetition_seed,
        )


        all_ranking_frames.append(
            ml_rankings
        )


        if technique == "RandomForest":
            model_seed_value = int(
                seed_one_model_values[
                    "RandomForestSeed"
                ]
            )

        elif technique == "XGBoost":
            model_seed_value = int(
                seed_one_model_values[
                    "XGBoostSeed"
                ]
            )

        elif technique == "LightGBM":
            model_seed_value = int(
                seed_one_model_values[
                    "LightGBMSeed"
                ]
            )

        else:
            model_seed_value = None


        model_fit_records.append({
            "ProjectNumber":
                PROJECT_NUMBER,

            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "SmokeConditionOrder":
                smoke_condition_order,

            "ConditionID":
                condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "Technique":
                technique,

            "ModelClass":
                type(
                    model
                ).__name__,

            "ModelSeed":
                model_seed_value,

            "TrainingRows":
                EXPECTED_MODEL_TRAINING_ROWS,

            "TrainingFailures":
                int(
                    noisy_binary_labels.sum()
                ),

            "EvaluationRows":
                EXPECTED_MODEL_EVALUATION_ROWS,

            "ActivePredictors":
                len(
                    active_feature_columns
                ),

            "FitSeconds":
                fit_seconds,

            "PredictSeconds":
                predict_seconds,

            "MinimumScore":
                float(
                    scores.min()
                ),

            "MaximumScore":
                float(
                    scores.max()
                ),

            "MeanScore":
                float(
                    scores.mean()
                ),

            "ScoreSHA256":
                array_sha256(
                    scores,
                    "<f8",
                ),

            "FitStatus":
                "SUCCESS",
        })


    # --------------------------------------------------------
    # BASELINES
    # --------------------------------------------------------

    random_rankings = create_random_rankings(
        evaluation_data=baseline_evaluation_data,
        repetition_seed=repetition_seed,
    )


    (
        latest_fail_rankings,
        qtf_rankings,
    ) = create_history_baseline_rankings(
        noisy_training_history=noisy_raw_history,
        clean_training_history=raw_training,
        clean_evaluation_history=raw_evaluation,
        clean_evaluation_data=baseline_evaluation_data,
        fixed_split=fixed_split,
    )


    for baseline_ranking in [
        random_rankings,
        latest_fail_rankings,
        qtf_rankings,
    ]:
        all_ranking_frames.append(
            add_condition_metadata(
                ranking=baseline_ranking,
                smoke_condition_order=smoke_condition_order,
                condition_id=condition_id,
                noise_percent=noise_percent,
                repetition_seed=repetition_seed,
            )
        )


    condition_seconds = float(
        time.perf_counter()
        - condition_started
    )


    condition_audit_records.append({
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "SmokeConditionOrder":
            smoke_condition_order,

        "ConditionID":
            condition_id,

        "NoisePercent":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "ExpectedRawFlips":
            expected_flip_count,

        "ActualRawFlips":
            actual_flip_count,

        "ExpectedRawLabelChanges":
            expected_raw_label_changes,

        "ActualRawLabelChanges":
            actual_raw_label_changes,

        "ExpectedModelLabelChanges":
            expected_model_label_changes,

        "ActualModelLabelChanges":
            actual_model_verdict_changes,

        "CleanRawFailures":
            int(
                np.count_nonzero(
                    clean_raw_verdicts
                    != 0
                )
            ),

        "NoisyRawFailures":
            int(
                np.count_nonzero(
                    noisy_raw_verdicts
                    != 0
                )
            ),

        "CleanModelFailures":
            int(
                np.count_nonzero(
                    clean_model_raw_verdicts
                    != 0
                )
            ),

        "NoisyModelFailures":
            int(
                noisy_binary_labels.sum()
            ),

        "DependentRECChangedValues":
            dependent_rec_changed_values,

        "IndependentRECChangedValues":
            independent_rec_changed_values,

        "ZeroNoiseRECMismatches":
            zero_noise_rec_mismatches,

        "FlipMaskHashMatch":
            flip_hash_match,

        "RawVerdictHashMatch":
            raw_verdict_hash_match,

        "ModelVerdictHashMatch":
            model_verdict_hash_match,

        "RECReconstructionSeconds":
            rec_seconds,

        "ConditionSeconds":
            condition_seconds,
    })


    print(
        f"[{smoke_condition_order}/{EXPECTED_SMOKE_CONDITIONS}]",
        condition_id,
        "| raw flips:",
        actual_flip_count,
        "| model failures:",
        int(
            noisy_binary_labels.sum()
        ),
        "| dependent REC changes:",
        dependent_rec_changed_values,
        "| ML fits:",
        len(
            ML_TECHNIQUES
        ),
        "| seconds:",
        round(
            condition_seconds,
            2,
        ),
    )


# ------------------------------------------------------------
# 19. COMBINE AND AGGREGATE OUTPUTS
# ------------------------------------------------------------

smoke_rankings = pd.concat(
    all_ranking_frames,
    ignore_index=True,
)


smoke_rankings = (
    smoke_rankings.sort_values(
        [
            "SmokeConditionOrder",
            "Technique",
            "build_order",
            "Rank",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


smoke_build_metrics = calculate_build_metrics(
    smoke_rankings
)


smoke_build_metrics = (
    smoke_build_metrics.sort_values(
        [
            "SmokeConditionOrder",
            "Technique",
            "BuildOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


smoke_project_run = calculate_project_runs(
    smoke_build_metrics
)


smoke_project_run = (
    smoke_project_run.sort_values(
        [
            "SmokeConditionOrder",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


smoke_model_fits = pd.DataFrame(
    model_fit_records
)

smoke_training_medians = pd.DataFrame(
    training_median_records
)

smoke_condition_audit = pd.DataFrame(
    condition_audit_records
)


smoke_elapsed_seconds = float(
    time.perf_counter()
    - smoke_started
)


# ------------------------------------------------------------
# 20. INVARIANCE AND RESPONSE AUDIT
# ------------------------------------------------------------

condition_zero = (
    "noise_00__seed_01"
)

condition_fifty = (
    "noise_50__seed_01"
)


invariance_records = []


for technique in ALL_TECHNIQUES:
    zero_hash = ranking_hash_for_condition_technique(
        smoke_rankings,
        condition_zero,
        technique,
    )

    fifty_hash = ranking_hash_for_condition_technique(
        smoke_rankings,
        condition_fifty,
        technique,
    )

    same_ranking = bool(
        zero_hash == fifty_hash
    )

    expected_same = (
        technique
        in {
            "Random",
            "QTF-Avg",
        }
    )

    expected_different = (
        technique == "LatestFail"
    )

    if expected_same:
        validation_pass = (
            same_ranking
        )

    elif expected_different:
        validation_pass = (
            not same_ranking
        )

    else:
        # ML response is descriptive here; no hard requirement
        # that every project/model changes its final rank order.
        validation_pass = True

    invariance_records.append({
        "Technique":
            technique,

        "Noise00RankingSHA256":
            zero_hash,

        "Noise50RankingSHA256":
            fifty_hash,

        "SameRankingAcrossNoise":
            same_ranking,

        "ExpectedSameAcrossNoise":
            expected_same,

        "ExpectedDifferentAcrossNoise":
            expected_different,

        "ValidationPass":
            validation_pass,
    })


smoke_invariance_audit = pd.DataFrame(
    invariance_records
)


random_noise_invariant = bool(
    smoke_invariance_audit.loc[
        smoke_invariance_audit[
            "Technique"
        ].eq(
            "Random"
        ),
        "SameRankingAcrossNoise",
    ].iloc[0]
)


qtf_noise_invariant = bool(
    smoke_invariance_audit.loc[
        smoke_invariance_audit[
            "Technique"
        ].eq(
            "QTF-Avg"
        ),
        "SameRankingAcrossNoise",
    ].iloc[0]
)


latest_fail_responds_to_noise = bool(
    ~smoke_invariance_audit.loc[
        smoke_invariance_audit[
            "Technique"
        ].eq(
            "LatestFail"
        ),
        "SameRankingAcrossNoise",
    ].iloc[0]
)


ml_rankings_changed_count = int(
    (
        ~smoke_invariance_audit.loc[
            smoke_invariance_audit[
                "Technique"
            ].isin(
                ML_TECHNIQUES
            ),
            "SameRankingAcrossNoise",
        ]
    ).sum()
)


def project_run_metric_identity(
    technique,
):
    rows = (
        smoke_project_run[
            smoke_project_run[
                "Technique"
            ].eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    return bool(
        len(
            rows
        ) == 2
        and np.array_equal(
            rows.loc[
                [0],
                [
                    "MeanAPFD",
                    "MeanAPFDc",
                ],
            ].to_numpy(
                dtype=float
            ),
            rows.loc[
                [1],
                [
                    "MeanAPFD",
                    "MeanAPFDc",
                ],
            ].to_numpy(
                dtype=float
            ),
        )
    )


random_metric_identity = project_run_metric_identity(
    "Random"
)

qtf_metric_identity = project_run_metric_identity(
    "QTF-Avg"
)


# ------------------------------------------------------------
# 21. STRUCTURAL AUDITS
# ------------------------------------------------------------

duplicate_ranking_rows = int(
    smoke_rankings.duplicated(
        subset=[
            "ConditionID",
            "Technique",
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


duplicate_rank_rows = int(
    smoke_rankings.duplicated(
        subset=[
            "ConditionID",
            "Technique",
            "BuildKey",
            "Rank",
        ],
        keep=False,
    ).sum()
)


noncontiguous_rank_groups = 0


for _, group in smoke_rankings.groupby(
    [
        "ConditionID",
        "Technique",
        "BuildKey",
    ],
    sort=False,
):
    actual_ranks = (
        group[
            "Rank"
        ]
        .sort_values()
        .to_numpy(
            dtype=int
        )
    )

    expected_ranks = np.arange(
        1,
        len(
            group
        ) + 1,
        dtype=int,
    )

    if not np.array_equal(
        actual_ranks,
        expected_ranks,
    ):
        noncontiguous_rank_groups += 1


evaluation_consistency = (
    smoke_rankings.groupby(
        [
            "ConditionID",
            "BuildKey",
            "TestKey",
        ],
        as_index=False,
    ).agg(
        Techniques=(
            "Technique",
            "nunique",
        ),

        ActualFailureValues=(
            "ActualFailure",
            "nunique",
        ),

        DurationValues=(
            "Duration",
            "nunique",
        ),
    )
)


evaluation_technique_count_violations = int(
    evaluation_consistency[
        "Techniques"
    ].ne(
        len(
            ALL_TECHNIQUES
        )
    ).sum()
)


evaluation_failure_consistency_violations = int(
    evaluation_consistency[
        "ActualFailureValues"
    ].ne(1).sum()
)


evaluation_duration_consistency_violations = int(
    evaluation_consistency[
        "DurationValues"
    ].ne(1).sum()
)


ml_ranking_rows = smoke_rankings[
    smoke_rankings[
        "Technique"
    ].isin(
        ML_TECHNIQUES
    )
]


ml_nonfinite_scores = int(
    (
        ~np.isfinite(
            ml_ranking_rows[
                "Score"
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


ml_scores_outside_probability_range = int(
    (
        (
            ml_ranking_rows[
                "Score"
            ] < -1e-12
        )
        | (
            ml_ranking_rows[
                "Score"
            ] > 1.0 + 1e-12
        )
    ).sum()
)


metric_nan_values = int(
    smoke_build_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].isna().sum().sum()
)


metric_out_of_range_values = int(
    (
        ~smoke_build_metrics[
            "APFD"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
    +
    (
        ~smoke_build_metrics[
            "APFDc"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
)


builds_per_run_violations = int(
    smoke_project_run[
        "EvaluatedBuilds"
    ].ne(
        EXPECTED_SCORED_EVALUATION_BUILDS
    ).sum()
)


evaluation_failure_total_violations = int(
    smoke_project_run[
        "EvaluationFailures"
    ].ne(
        EXPECTED_MODEL_EVALUATION_FAILURES
    ).sum()
)


model_fit_failures = int(
    smoke_model_fits[
        "FitStatus"
    ].ne(
        "SUCCESS"
    ).sum()
)


model_fit_technique_count_violations = int(
    smoke_model_fits.groupby(
        "Technique"
    ).size().ne(
        EXPECTED_SMOKE_CONDITIONS
    ).sum()
)


condition_plan_count_mismatches = int(
    (
        smoke_condition_audit[
            "ExpectedRawFlips"
        ].ne(
            smoke_condition_audit[
                "ActualRawFlips"
            ]
        )
    ).sum()
    +
    (
        smoke_condition_audit[
            "ExpectedRawLabelChanges"
        ].ne(
            smoke_condition_audit[
                "ActualRawLabelChanges"
            ]
        )
    ).sum()
    +
    (
        smoke_condition_audit[
            "ExpectedModelLabelChanges"
        ].ne(
            smoke_condition_audit[
                "ActualModelLabelChanges"
            ]
        )
    ).sum()
)


condition_plan_hash_mismatches = int(
    (
        ~smoke_condition_audit[
            [
                "FlipMaskHashMatch",
                "RawVerdictHashMatch",
                "ModelVerdictHashMatch",
            ]
        ]
    ).sum().sum()
)


zero_noise_condition = smoke_condition_audit[
    smoke_condition_audit[
        "NoisePercent"
    ].eq(0)
]


fifty_noise_condition = smoke_condition_audit[
    smoke_condition_audit[
        "NoisePercent"
    ].eq(50)
]


zero_noise_rec_mismatches = int(
    zero_noise_condition[
        "ZeroNoiseRECMismatches"
    ].fillna(0).sum()
)


fifty_noise_dependent_changes = int(
    fifty_noise_condition[
        "DependentRECChangedValues"
    ].sum()
)


all_independent_rec_changes = int(
    smoke_condition_audit[
        "IndependentRECChangedValues"
    ].sum()
)


# ------------------------------------------------------------
# 22. WRITE SMOKE OUTPUTS
# ------------------------------------------------------------

SMOKE_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_parquet(
    SMOKE_RANKINGS_PATH,
    smoke_rankings,
)

atomic_write_csv(
    SMOKE_BUILD_METRICS_PATH,
    smoke_build_metrics,
)

atomic_write_csv(
    SMOKE_PROJECT_RUN_PATH,
    smoke_project_run,
)

atomic_write_csv(
    SMOKE_MODEL_FITS_PATH,
    smoke_model_fits,
)

atomic_write_csv(
    SMOKE_TRAINING_MEDIANS_PATH,
    smoke_training_medians,
)

atomic_write_csv(
    SMOKE_CONDITION_AUDIT_PATH,
    smoke_condition_audit,
)

atomic_write_csv(
    SMOKE_INVARIANCE_AUDIT_PATH,
    smoke_invariance_audit,
)


# ------------------------------------------------------------
# 23. READBACK AND IMMUTABILITY
# ------------------------------------------------------------

rankings_readback = pd.read_parquet(
    SMOKE_RANKINGS_PATH
)

build_metrics_readback = pd.read_csv(
    SMOKE_BUILD_METRICS_PATH,
    low_memory=False,
)

project_run_readback = pd.read_csv(
    SMOKE_PROJECT_RUN_PATH,
    low_memory=False,
)

model_fits_readback = pd.read_csv(
    SMOKE_MODEL_FITS_PATH,
    low_memory=False,
)

medians_readback = pd.read_csv(
    SMOKE_TRAINING_MEDIANS_PATH,
    low_memory=False,
)


raw_evaluation_sha256_after = calculate_hash(
    raw_evaluation_path
)

clean_evaluation_base_sha256_after = calculate_hash(
    clean_evaluation_base_path
)


raw_evaluation_unchanged = bool(
    raw_evaluation_sha256_before
    == raw_evaluation_sha256_after
)

clean_evaluation_base_unchanged = bool(
    clean_evaluation_base_sha256_before
    == clean_evaluation_base_sha256_after
)


source_root_after = create_source_root_sha256(
    source_files_payload
)

source_unchanged = bool(
    source_root_before
    == source_root_after
    == EXPECTED_SOURCE_ROOT_SHA256
)


registry_sha256_after = calculate_hash(
    REGISTRY_PATH
)

registry_unchanged = bool(
    registry_sha256_before
    == registry_sha256_after
)


# ------------------------------------------------------------
# 24. VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check": "Step 2B passed",
        "Expected": EXPECTED_STEP2B_STATUS,
        "Actual": step2b_status.get("Status"),
        "Pass": step2b_status.get("Status") == EXPECTED_STEP2B_STATUS,
    },
    {
        "Check": "Step 3A passed",
        "Expected": EXPECTED_STEP3A_STATUS,
        "Actual": step3a_status.get("Status"),
        "Pass": step3a_status.get("Status") == EXPECTED_STEP3A_STATUS,
    },
    {
        "Check": "Step 3B passed",
        "Expected": EXPECTED_STEP3B_STATUS,
        "Actual": step3b_status.get("Status"),
        "Pass": step3b_status.get("Status") == EXPECTED_STEP3B_STATUS,
    },
    {
        "Check": "Step 4A passed",
        "Expected": EXPECTED_STEP4A_STATUS,
        "Actual": step4a_status.get("Status"),
        "Pass": step4a_status.get("Status") == EXPECTED_STEP4A_STATUS,
    },
    {
        "Check": "Smoke conditions",
        "Expected": EXPECTED_SMOKE_CONDITIONS,
        "Actual": len(smoke_condition_audit),
        "Pass": len(smoke_condition_audit) == EXPECTED_SMOKE_CONDITIONS,
    },
    {
        "Check": "Condition-plan count mismatches",
        "Expected": 0,
        "Actual": condition_plan_count_mismatches,
        "Pass": condition_plan_count_mismatches == 0,
    },
    {
        "Check": "Condition-plan hash mismatches",
        "Expected": 0,
        "Actual": condition_plan_hash_mismatches,
        "Pass": condition_plan_hash_mismatches == 0,
    },
    {
        "Check": "Zero-noise REC mismatches",
        "Expected": 0,
        "Actual": zero_noise_rec_mismatches,
        "Pass": zero_noise_rec_mismatches == 0,
    },
    {
        "Check": "50% dependent REC changed",
        "Expected": True,
        "Actual": fifty_noise_dependent_changes > 0,
        "Pass": fifty_noise_dependent_changes > 0,
    },
    {
        "Check": "Independent REC changed values",
        "Expected": 0,
        "Actual": all_independent_rec_changes,
        "Pass": all_independent_rec_changes == 0,
    },
    {
        "Check": "ML fits",
        "Expected": EXPECTED_ML_FITS,
        "Actual": len(smoke_model_fits),
        "Pass": len(smoke_model_fits) == EXPECTED_ML_FITS,
    },
    {
        "Check": "Model-fit failures",
        "Expected": 0,
        "Actual": model_fit_failures,
        "Pass": model_fit_failures == 0,
    },
    {
        "Check": "Model-fit technique count violations",
        "Expected": 0,
        "Actual": model_fit_technique_count_violations,
        "Pass": model_fit_technique_count_violations == 0,
    },
    {
        "Check": "Training median rows",
        "Expected": EXPECTED_TRAINING_MEDIAN_ROWS,
        "Actual": len(smoke_training_medians),
        "Pass": len(smoke_training_medians) == EXPECTED_TRAINING_MEDIAN_ROWS,
    },
    {
        "Check": "Ranking rows",
        "Expected": EXPECTED_RANKING_ROWS,
        "Actual": len(smoke_rankings),
        "Pass": len(smoke_rankings) == EXPECTED_RANKING_ROWS,
    },
    {
        "Check": "Duplicate ranking rows",
        "Expected": 0,
        "Actual": duplicate_ranking_rows,
        "Pass": duplicate_ranking_rows == 0,
    },
    {
        "Check": "Duplicate rank rows",
        "Expected": 0,
        "Actual": duplicate_rank_rows,
        "Pass": duplicate_rank_rows == 0,
    },
    {
        "Check": "Non-contiguous rank groups",
        "Expected": 0,
        "Actual": noncontiguous_rank_groups,
        "Pass": noncontiguous_rank_groups == 0,
    },
    {
        "Check": "Evaluation technique-count violations",
        "Expected": 0,
        "Actual": evaluation_technique_count_violations,
        "Pass": evaluation_technique_count_violations == 0,
    },
    {
        "Check": "Evaluation failure consistency violations",
        "Expected": 0,
        "Actual": evaluation_failure_consistency_violations,
        "Pass": evaluation_failure_consistency_violations == 0,
    },
    {
        "Check": "Evaluation duration consistency violations",
        "Expected": 0,
        "Actual": evaluation_duration_consistency_violations,
        "Pass": evaluation_duration_consistency_violations == 0,
    },
    {
        "Check": "ML non-finite scores",
        "Expected": 0,
        "Actual": ml_nonfinite_scores,
        "Pass": ml_nonfinite_scores == 0,
    },
    {
        "Check": "ML scores outside [0,1]",
        "Expected": 0,
        "Actual": ml_scores_outside_probability_range,
        "Pass": ml_scores_outside_probability_range == 0,
    },
    {
        "Check": "Build-metric rows",
        "Expected": EXPECTED_BUILD_METRIC_ROWS,
        "Actual": len(smoke_build_metrics),
        "Pass": len(smoke_build_metrics) == EXPECTED_BUILD_METRIC_ROWS,
    },
    {
        "Check": "Metric NaN values",
        "Expected": 0,
        "Actual": metric_nan_values,
        "Pass": metric_nan_values == 0,
    },
    {
        "Check": "Metric values outside [0,1]",
        "Expected": 0,
        "Actual": metric_out_of_range_values,
        "Pass": metric_out_of_range_values == 0,
    },
    {
        "Check": "Project-run rows",
        "Expected": EXPECTED_PROJECT_RUN_ROWS,
        "Actual": len(smoke_project_run),
        "Pass": len(smoke_project_run) == EXPECTED_PROJECT_RUN_ROWS,
    },
    {
        "Check": "Evaluated-build count violations",
        "Expected": 0,
        "Actual": builds_per_run_violations,
        "Pass": builds_per_run_violations == 0,
    },
    {
        "Check": "Evaluation-failure total violations",
        "Expected": 0,
        "Actual": evaluation_failure_total_violations,
        "Pass": evaluation_failure_total_violations == 0,
    },
    {
        "Check": "Random noise invariance",
        "Expected": True,
        "Actual": random_noise_invariant,
        "Pass": random_noise_invariant,
    },
    {
        "Check": "Random metric identity",
        "Expected": True,
        "Actual": random_metric_identity,
        "Pass": random_metric_identity,
    },
    {
        "Check": "QTF-Avg noise invariance",
        "Expected": True,
        "Actual": qtf_noise_invariant,
        "Pass": qtf_noise_invariant,
    },
    {
        "Check": "QTF-Avg metric identity",
        "Expected": True,
        "Actual": qtf_metric_identity,
        "Pass": qtf_metric_identity,
    },
    {
        "Check": "LatestFail responds to noise",
        "Expected": True,
        "Actual": latest_fail_responds_to_noise,
        "Pass": latest_fail_responds_to_noise,
    },
    {
        "Check": "ML techniques with changed rankings",
        "Expected": ">= 1",
        "Actual": ml_rankings_changed_count,
        "Pass": ml_rankings_changed_count >= 1,
    },
    {
        "Check": "Invariance audit failures",
        "Expected": 0,
        "Actual": int(
            (
                ~smoke_invariance_audit[
                    "ValidationPass"
                ]
            ).sum()
        ),
        "Pass": int(
            (
                ~smoke_invariance_audit[
                    "ValidationPass"
                ]
            ).sum()
        ) == 0,
    },
    {
        "Check": "Ranking readback rows",
        "Expected": EXPECTED_RANKING_ROWS,
        "Actual": len(rankings_readback),
        "Pass": len(rankings_readback) == EXPECTED_RANKING_ROWS,
    },
    {
        "Check": "Build-metric readback rows",
        "Expected": EXPECTED_BUILD_METRIC_ROWS,
        "Actual": len(build_metrics_readback),
        "Pass": len(build_metrics_readback) == EXPECTED_BUILD_METRIC_ROWS,
    },
    {
        "Check": "Project-run readback rows",
        "Expected": EXPECTED_PROJECT_RUN_ROWS,
        "Actual": len(project_run_readback),
        "Pass": len(project_run_readback) == EXPECTED_PROJECT_RUN_ROWS,
    },
    {
        "Check": "Model-fit readback rows",
        "Expected": EXPECTED_ML_FITS,
        "Actual": len(model_fits_readback),
        "Pass": len(model_fits_readback) == EXPECTED_ML_FITS,
    },
    {
        "Check": "Median readback rows",
        "Expected": EXPECTED_TRAINING_MEDIAN_ROWS,
        "Actual": len(medians_readback),
        "Pass": len(medians_readback) == EXPECTED_TRAINING_MEDIAN_ROWS,
    },
    {
        "Check": "Raw evaluation cohort unchanged",
        "Expected": True,
        "Actual": raw_evaluation_unchanged,
        "Pass": raw_evaluation_unchanged,
    },
    {
        "Check": "Clean evaluation base unchanged",
        "Expected": True,
        "Actual": clean_evaluation_base_unchanged,
        "Pass": clean_evaluation_base_unchanged,
    },
    {
        "Check": "Project 10 source unchanged",
        "Expected": True,
        "Actual": source_unchanged,
        "Pass": source_unchanged,
    },
    {
        "Check": "Completion registry unchanged",
        "Expected": True,
        "Actual": registry_unchanged,
        "Pass": registry_unchanged,
    },
    {
        "Check": "Registry Project 10 rows",
        "Expected": 0,
        "Actual": int(
            registry_project_numbers.eq(
                10
            ).sum()
        ),
        "Pass": int(
            registry_project_numbers.eq(
                10
            ).sum()
        ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)

failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 4B validation:")

display(
    validation
)


if not failed_checks.empty:
    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 10 STEP 4B DID NOT PASS."
    )


atomic_write_csv(
    STEP4B_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 25. REPORT AND CHECKPOINT
# ------------------------------------------------------------

output_hashes = {
    "SmokeRankingsSHA256":
        calculate_hash(
            SMOKE_RANKINGS_PATH
        ),

    "SmokeBuildMetricsSHA256":
        calculate_hash(
            SMOKE_BUILD_METRICS_PATH
        ),

    "SmokeProjectRunSHA256":
        calculate_hash(
            SMOKE_PROJECT_RUN_PATH
        ),

    "SmokeModelFitsSHA256":
        calculate_hash(
            SMOKE_MODEL_FITS_PATH
        ),

    "SmokeTrainingMediansSHA256":
        calculate_hash(
            SMOKE_TRAINING_MEDIANS_PATH
        ),

    "SmokeConditionAuditSHA256":
        calculate_hash(
            SMOKE_CONDITION_AUDIT_PATH
        ),

    "SmokeInvarianceAuditSHA256":
        calculate_hash(
            SMOKE_INVARIANCE_AUDIT_PATH
        ),

    "Step4BValidationSHA256":
        calculate_hash(
            STEP4B_VALIDATION_PATH
        ),
}


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4B_PASS_STATUS,

    "SmokeConditions":
        [
            {
                "NoisePercent":
                    noise,

                "RepetitionSeed":
                    seed,
            }
            for noise, seed in SMOKE_CONDITIONS
        ],

    "Conditions":
        len(
            smoke_condition_audit
        ),

    "MLFits":
        len(
            smoke_model_fits
        ),

    "RankingRows":
        len(
            smoke_rankings
        ),

    "BuildMetricRows":
        len(
            smoke_build_metrics
        ),

    "ProjectRunRows":
        len(
            smoke_project_run
        ),

    "TrainingMedianRows":
        len(
            smoke_training_medians
        ),

    "RandomNoiseInvariant":
        random_noise_invariant,

    "RandomMetricIdentity":
        random_metric_identity,

    "QTFAvgNoiseInvariant":
        qtf_noise_invariant,

    "QTFAvgMetricIdentity":
        qtf_metric_identity,

    "LatestFailRespondsToNoise":
        latest_fail_responds_to_noise,

    "MLRankingsChangedCount":
        ml_rankings_changed_count,

    "ZeroNoiseRECMismatches":
        zero_noise_rec_mismatches,

    "FiftyPercentDependentRECChanges":
        fifty_noise_dependent_changes,

    "IndependentRECChangedValues":
        all_independent_rec_changes,

    "RawEvaluationUnchanged":
        raw_evaluation_unchanged,

    "CleanEvaluationBaseUnchanged":
        clean_evaluation_base_unchanged,

    "Project10SourceUnchanged":
        source_unchanged,

    "CompletionRegistryModified":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "ElapsedSeconds":
        smoke_elapsed_seconds,

    "OutputHashes":
        output_hashes,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP4B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "SourceRootSHA256":
        source_root_after,

    "SelectionCheckpointSHA256":
        calculate_hash(
            SELECTION_CHECKPOINT_PATH
        ),

    "RECCheckpointSHA256":
        calculate_hash(
            REC_CHECKPOINT_PATH
        ),

    "NoisePlanCheckpointSHA256":
        calculate_hash(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "NoisyRECEngineCheckpointSHA256":
        calculate_hash(
            NOISY_REC_ENGINE_CHECKPOINT_PATH
        ),

    "ModelProtocolCheckpointSHA256":
        calculate_hash(
            MODEL_PROTOCOL_CHECKPOINT_PATH
        ),

    "SmokeRankings":
        str(
            SMOKE_RANKINGS_PATH
        ),

    "SmokeBuildMetrics":
        str(
            SMOKE_BUILD_METRICS_PATH
        ),

    "SmokeProjectRun":
        str(
            SMOKE_PROJECT_RUN_PATH
        ),

    "SmokeModelFits":
        str(
            SMOKE_MODEL_FITS_PATH
        ),

    "SmokeTrainingMedians":
        str(
            SMOKE_TRAINING_MEDIANS_PATH
        ),

    "SmokeConditionAudit":
        str(
            SMOKE_CONDITION_AUDIT_PATH
        ),

    "SmokeInvarianceAudit":
        str(
            SMOKE_INVARIANCE_AUDIT_PATH
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistryRows":
        len(
            registry
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    SMOKE_TEST_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4B_PASS_STATUS,

    "Conditions":
        len(
            smoke_condition_audit
        ),

    "MLFits":
        len(
            smoke_model_fits
        ),

    "RankingRows":
        len(
            smoke_rankings
        ),

    "BuildMetricRows":
        len(
            smoke_build_metrics
        ),

    "ProjectRunRows":
        len(
            smoke_project_run
        ),

    "Checkpoint":
        str(
            SMOKE_TEST_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_hash(
            SMOKE_TEST_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP4B_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 26. FINAL READBACK
# ------------------------------------------------------------

checkpoint_readback = read_json_with_retry(
    SMOKE_TEST_CHECKPOINT_PATH
)

status_readback = read_json_with_retry(
    STEP4B_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP4B_PASS_STATUS:
    raise AssertionError(
        "Smoke checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP4B_PASS_STATUS:
    raise AssertionError(
        "Step 4B status readback failed."
    )


if calculate_hash(
    REGISTRY_PATH
) != registry_sha256_before:
    raise AssertionError(
        "Completion registry changed during Step 4B."
    )


if create_source_root_sha256(
    source_files_payload
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Project 10 source changed during Step 4B."
    )


# ------------------------------------------------------------
# 27. DISPLAY
# ------------------------------------------------------------

print("\nCondition audit:")

display(
    smoke_condition_audit
)


print("\nModel fits:")

display(
    smoke_model_fits[
        [
            "ConditionID",
            "Technique",
            "TrainingFailures",
            "FitSeconds",
            "PredictSeconds",
            "MinimumScore",
            "MaximumScore",
            "FitStatus",
        ]
    ]
)


print("\nProject-run results:")

display(
    smoke_project_run[
        [
            "ConditionID",
            "Technique",
            "EvaluatedBuilds",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
)


print("\nNoise invariance and response audit:")

display(
    smoke_invariance_audit[
        [
            "Technique",
            "SameRankingAcrossNoise",
            "ExpectedSameAcrossNoise",
            "ExpectedDifferentAcrossNoise",
            "ValidationPass",
        ]
    ]
)


# ------------------------------------------------------------
# 28. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 126)
print("=== PROJECT 10 CELL 8 / STEP 4B RESULT ===")
print("=" * 126)

print("\nProject:")
print(PROJECT_NAME)


print("\nSmoke conditions:")

print(
    "Conditions:",
    len(
        smoke_condition_audit
    ),
)

print(
    "Condition IDs:",
    smoke_condition_audit[
        "ConditionID"
    ].tolist(),
)


print("\nEnd-to-end outputs:")

print(
    "ML fits:",
    len(
        smoke_model_fits
    ),
)

print(
    "Ranking rows:",
    len(
        smoke_rankings
    ),
)

print(
    "Build-metric rows:",
    len(
        smoke_build_metrics
    ),
)

print(
    "Project-run rows:",
    len(
        smoke_project_run
    ),
)

print(
    "Training-median rows:",
    len(
        smoke_training_medians
    ),
)


print("\nNoise and REC validation:")

print(
    "Condition-plan count mismatches:",
    condition_plan_count_mismatches,
)

print(
    "Condition-plan hash mismatches:",
    condition_plan_hash_mismatches,
)

print(
    "Zero-noise REC mismatches:",
    zero_noise_rec_mismatches,
)

print(
    "50% dependent REC changed values:",
    fifty_noise_dependent_changes,
)

print(
    "Independent REC changed values:",
    all_independent_rec_changes,
)


print("\nTechnique validation:")

print(
    "Random noise invariant:",
    random_noise_invariant,
)

print(
    "Random metric identity:",
    random_metric_identity,
)

print(
    "QTF-Avg noise invariant:",
    qtf_noise_invariant,
)

print(
    "QTF-Avg metric identity:",
    qtf_metric_identity,
)

print(
    "LatestFail responds to noise:",
    latest_fail_responds_to_noise,
)

print(
    "ML techniques with changed rankings:",
    ml_rankings_changed_count,
)


print("\nMetric validation:")

print(
    "Metric NaN values:",
    metric_nan_values,
)

print(
    "Metric values outside [0,1]:",
    metric_out_of_range_values,
)


print("\nImmutability:")

print(
    "Raw evaluation cohort unchanged:",
    raw_evaluation_unchanged,
)

print(
    "Clean evaluation base unchanged:",
    clean_evaluation_base_unchanged,
)

print(
    "Project 10 source unchanged:",
    source_unchanged,
)

print(
    "Completion registry unchanged:",
    registry_unchanged,
)

print(
    "Project 9 accessed:",
    False,
)

print(
    "Project 9 write attempted:",
    False,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nRuntime:")

print(
    "Smoke-test seconds:",
    smoke_elapsed_seconds,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nSmoke-test checkpoint:")

print(
    SMOKE_TEST_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_hash(
        SMOKE_TEST_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP4B_PASS_STATUS,
)

print("=" * 126)

=== PROJECT 10 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[1/2] noise_00__seed_01 | raw flips: 0 | model failures: 63 | dependent REC changes: 0 | ML fits: 4 | seconds: 34.3


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[2/2] noise_50__seed_01 | raw flips: 17388 | model failures: 3048 | dependent REC changes: 70556 | ML fits: 4 | seconds: 41.69

Step 4B validation:


...


Project-run results:


,ConditionID,Technique,EvaluatedBuilds,MeanAPFD,MeanAPFDc
0,noise_00__seed_01,LatestFail,27,0.873665,0.754963
1,noise_00__seed_01,LightGBM,27,0.912857,0.813420
2,noise_00__seed_01,NaiveBayes,27,0.796850,0.758406
3,noise_00__seed_01,QTF-Avg,27,0.135972,0.542006
4,noise_00__seed_01,Random,27,0.473794,0.486381
5,noise_00__seed_01,RandomForest,27,0.889019,0.806653
6,noise_00__seed_01,XGBoost,27,0.891230,0.796648
7,noise_50__seed_01,LatestFail,27,0.872004,0.764143
8,noise_50__seed_01,LightGBM,27,0.711220,0.683810
9,noise_50__seed_01,NaiveBayes,27,0.231467,0.454573



Noise invariance and response audit:


,Technique,SameRankingAcrossNoise,ExpectedSameAcrossNoise,ExpectedDifferentAcrossNoise,ValidationPass
0,RandomForest,False,False,False,True
1,XGBoost,False,False,False,True
2,LightGBM,False,False,False,True
3,NaiveBayes,False,False,False,True
4,Random,True,True,False,True
5,LatestFail,False,False,True,True
6,QTF-Avg,True,True,False,True




=== PROJECT 10 CELL 8 / STEP 4B RESULT ===

Project:
spring-cloud@spring-cloud-dataflow

Smoke conditions:
Conditions: 2
Condition IDs: ['noise_00__seed_01', 'noise_50__seed_01']

End-to-end outputs:
ML fits: 8
Ranking rows: 36554
Build-metric rows: 378
Project-run rows: 14
Training-median rows: 302

Noise and REC validation:
Condition-plan count mismatches: 0
Condition-plan hash mismatches: 0
Zero-noise REC mismatches: 0
50% dependent REC changed values: 70556
Independent REC changed values: 0

Technique validation:
Random noise invariant: True
Random metric identity: True
QTF-Avg noise invariant: True
QTF-Avg metric identity: True
LatestFail responds to noise: True
ML techniques with changed rankings: 4

Metric validation:
Metric NaN values: 0
Metric values outside [0,1]: 0

Immutability:
Raw evaluation cohort unchanged: True
Clean evaluation base unchanged: True
Project 10 source unchanged: True
Completion registry unchanged: True
Project 9 accessed: False
Project 9 write attempte

In [12]:
# ============================================================
# PROJECT 10 — CELL 9 / STEP 5A
# CHECKPOINTED FULL 270-CONDITION EXPERIMENT
#
# PROJECT:
#   spring-cloud@spring-cloud-dataflow
#
# Per condition, this writes exactly eight files:
#   1. rankings.parquet
#   2. build_metrics.csv
#   3. project_run.csv
#   4. model_fits.csv
#   5. condition_audit.csv
#   6. training_medians.csv
#   7. condition_summary.json
#   8. _SUCCESS.json
#
# Properties:
# - 270 conditions: 9 noise levels × 30 seeds
# - checkpoint/resume safe
# - success marker written last
# - incomplete Project 10 condition folders are replaced
# - completed valid conditions are never rerun
# - clean evaluation data remains immutable
# - completion registry remains read-only
# - Project 9 is not accessed
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from IPython.display import display

import gc
import hashlib
import json
import os
import shutil
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    message=(
        "X does not have valid feature names, "
        "but LGBMClassifier was fitted with feature names"
    ),
)


# ------------------------------------------------------------
# 1. FROZEN CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)


EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_10_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_10_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP3B_STATUS = (
    "PASS_PROJECT_10_NOISY_REC_ENGINE_SENTINEL_VALIDATED_AND_FROZEN"
)

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_10_MODEL_PREDICTOR_BASELINE_AND_METRIC_PROTOCOL_FROZEN"
)

EXPECTED_STEP4B_STATUS = (
    "PASS_PROJECT_10_TWO_CONDITION_END_TO_END_SMOKE_TEST_VALIDATED"
)


RUNNING_STATUS = (
    "RUNNING_PROJECT_10_FULL_270_CONDITION_EXPERIMENT"
)

CONDITION_PASS_STATUS = (
    "PASS_PROJECT_10_CONDITION_COMPLETE"
)

STEP5A_PASS_STATUS = (
    "PASS_PROJECT_10_FULL_270_CONDITION_RUN_COMPLETE"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "582f01b3a43b542537b93243e5bb5b8cff36c274c6c2a3b12b580090d664206e"
)


NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)


EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080

EXPECTED_RAW_TRAINING_ROWS = 34563
EXPECTED_RAW_EVALUATION_ROWS = 12531

EXPECTED_MODEL_TRAINING_ROWS = 6095
EXPECTED_MODEL_EVALUATION_ROWS = 2611
EXPECTED_MODEL_EVALUATION_FAILURES = 213

EXPECTED_ACTIVE_PREDICTORS = 151
EXPECTED_SCORED_EVALUATION_BUILDS = 27

EXPECTED_TECHNIQUES = 7

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_TECHNIQUES
    * EXPECTED_MODEL_EVALUATION_ROWS
)

EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_TECHNIQUES
    * EXPECTED_SCORED_EVALUATION_BUILDS
)

EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = (
    EXPECTED_TECHNIQUES
)

EXPECTED_MODEL_FITS_PER_CONDITION = 4

EXPECTED_TRAINING_MEDIANS_PER_CONDITION = (
    EXPECTED_ACTIVE_PREDICTORS
)

EXPECTED_RAW_FILES_PER_CONDITION = 8


EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_RANKING_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_TRAINING_MEDIANS_PER_CONDITION
)

EXPECTED_TOTAL_RAW_FILES = (
    EXPECTED_CONDITIONS
    * EXPECTED_RAW_FILES_PER_CONDITION
)


RECENT_WINDOW = 6


ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_noise_plan_checkpoint.json"
)

NOISY_REC_ENGINE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_noisy_rec_engine_checkpoint.json"
)

MODEL_PROTOCOL_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_model_protocol_checkpoint.json"
)

SMOKE_TEST_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_10_smoke_test_checkpoint.json"
)


PROJECT_AGGREGATED_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP2B_STATUS_PATH = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)

STEP3B_STATUS_PATH = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_status.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_status.json"
)

STEP4B_STATUS_PATH = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step4b_status.json"
)


RAW_ROOT = (
    RESULTS_DIR
    / "Raw"
    / PROJECT_SLUG
)

CONTROL_DIR = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_full_run_control"
)

PROGRESS_PATH = (
    CONTROL_DIR
    / f"{PROJECT_SHORT_NAME}_full_run_progress.json"
)

CHECKPOINT_TABLE_PATH = (
    CONTROL_DIR
    / f"{PROJECT_SHORT_NAME}_full_run_checkpoint.csv"
)

EXECUTION_REPORT_PATH = (
    CONTROL_DIR
    / f"{PROJECT_SHORT_NAME}_full_run_execution_report.json"
)

STEP5A_STATUS_PATH = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step5a_status.json"
)


PROJECT_9_ROOT = (
    RESULTS_DIR
    / "Raw"
    / "camunda__camunda-bpm-platform"
)


print("=" * 128)
print("=== PROJECT 10 CELL 9 / STEP 5A: CHECKPOINTED FULL 270-CONDITION EXPERIMENT ===")
print("=" * 128)


# ------------------------------------------------------------
# 3. FILE HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def array_sha256(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="snappy",
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json_with_retry(
    path,
    attempts=10,
    delay_seconds=0.5,
):
    path = Path(path)

    last_error = None

    for _ in range(attempts):
        try:
            return json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as error:
            last_error = error
            time.sleep(
                delay_seconds
            )

    raise RuntimeError(
        "Could not safely read JSON.\n"
        f"Path: {path}\n"
        f"Error: {type(last_error).__name__}: {last_error}"
    )


def create_source_root_sha256(
    source_files_payload,
):
    digest = hashlib.sha256()

    for relative_path in sorted(
        source_files_payload
    ):
        metadata = source_files_payload[
            relative_path
        ]

        runtime_path = Path(
            metadata[
                "RuntimePath"
            ]
        )

        digest.update(
            (
                f"{relative_path}\0"
                f"{int(runtime_path.stat().st_size)}\0"
                f"{calculate_hash(runtime_path)}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def safe_remove_condition_directory(
    condition_directory,
):
    condition_directory = Path(
        condition_directory
    ).resolve()

    raw_root_resolved = RAW_ROOT.resolve()

    if raw_root_resolved not in condition_directory.parents:
        raise RuntimeError(
            "Refusing to remove a directory outside "
            "the Project 10 raw root.\n"
            f"Directory: {condition_directory}"
        )

    if (
        condition_directory.exists()
        and condition_directory.is_dir()
    ):
        shutil.rmtree(
            condition_directory
        )


# ------------------------------------------------------------
# 4. DETERMINISTIC SEEDS
# ------------------------------------------------------------

def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


def normalise_build_seed_key(value):
    numeric = pd.to_numeric(
        pd.Series(
            [value]
        ),
        errors="coerce",
    ).iloc[0]

    if (
        not pd.isna(
            numeric
        )
        and np.isclose(
            numeric,
            round(
                numeric
            ),
            rtol=0,
            atol=1e-9,
        )
    ):
        return str(
            int(
                round(
                    numeric
                )
            )
        )

    return str(
        value
    )


# ------------------------------------------------------------
# 5. APFD AND APFDC
# ------------------------------------------------------------

def calculate_apfd(actual_failures):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):
        return np.nan

    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )


def calculate_apfdc(
    actual_failures,
    durations,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(failures) != len(durations):
        raise ValueError(
            "Failure and duration arrays have different lengths."
        )

    if (
        len(failures) == 0
        or failures.sum() == 0
    ):
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain non-finite values."
        )

    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations contain negative values."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array(
            [0.0]
        ),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


# ------------------------------------------------------------
# 6. RANKING HELPERS
# ------------------------------------------------------------

def rank_build_rows(
    build_rows,
    scores,
    technique,
    score_direction,
):
    required_columns = [
        "Build",
        "Test",
        "Verdict",
        "Duration",
        "build_order",
        "BuildKey",
        "TestKey",
    ]

    if build_rows.columns.duplicated().any():
        raise RuntimeError(
            "Ranking frame contains duplicate column names."
        )

    ranked = (
        build_rows[
            required_columns
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    score_values = np.asarray(
        scores,
        dtype=float,
    )

    if len(score_values) != len(ranked):
        raise ValueError(
            "Ranking score count differs from row count."
        )

    if np.isnan(
        score_values
    ).any():
        raise ValueError(
            "Ranking scores contain NaN."
        )

    ranked[
        "Technique"
    ] = technique

    ranked[
        "Score"
    ] = score_values

    ranked[
        "ActualFailure"
    ] = (
        pd.to_numeric(
            ranked[
                "Verdict"
            ],
            errors="raise",
        )
        .ne(0)
        .astype(np.int8)
    )

    ranked[
        "Duration"
    ] = pd.to_numeric(
        ranked[
            "Duration"
        ],
        errors="raise",
    ).astype(float)

    numeric_test = pd.to_numeric(
        ranked[
            "Test"
        ],
        errors="coerce",
    )

    if numeric_test.notna().all():
        ranked[
            "__TestSort"
        ] = numeric_test.astype(float)
    else:
        ranked[
            "__TestSort"
        ] = (
            ranked[
                "Test"
            ]
            .fillna("")
            .astype(str)
        )

    if score_direction == "descending":
        score_ascending = False

    elif score_direction == "ascending":
        score_ascending = True

    else:
        raise ValueError(
            "Unknown score direction."
        )

    ranked = (
        ranked.sort_values(
            [
                "Score",
                "__TestSort",
            ],
            ascending=[
                score_ascending,
                True,
            ],
            kind="mergesort",
        )
        .drop(
            columns=[
                "__TestSort",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    ranked[
        "Rank"
    ] = np.arange(
        1,
        len(ranked) + 1,
        dtype=np.int32,
    )

    return ranked


def add_condition_metadata(
    ranking,
    condition_order,
    condition_id,
    noise_percent,
    repetition_seed,
):
    result = ranking.copy()

    metadata = [
        (
            "ProjectNumber",
            PROJECT_NUMBER,
        ),
        (
            "Project",
            PROJECT_NAME,
        ),
        (
            "ProjectSlug",
            PROJECT_SLUG,
        ),
        (
            "ConditionOrder",
            condition_order,
        ),
        (
            "ConditionID",
            condition_id,
        ),
        (
            "NoisePercent",
            noise_percent,
        ),
        (
            "RepetitionSeed",
            repetition_seed,
        ),
    ]

    for insertion_position, (
        column,
        value,
    ) in enumerate(metadata):
        result.insert(
            insertion_position,
            column,
            value,
        )

    return result


def create_ml_rankings(
    evaluation_data,
    scores,
    technique,
):
    scored = evaluation_data.copy()

    scored[
        "__Score"
    ] = np.asarray(
        scores,
        dtype=float,
    )

    ranking_frames = []

    for _, build_rows in scored.groupby(
        "BuildKey",
        sort=False,
    ):
        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=build_rows[
                    "__Score"
                ].to_numpy(
                    dtype=float
                ),
                technique=technique,
                score_direction="descending",
            )
        )

    return pd.concat(
        ranking_frames,
        ignore_index=True,
    )


def create_random_rankings(
    evaluation_data,
    repetition_seed,
):
    ranking_frames = []

    for build_key, build_rows in (
        evaluation_data.groupby(
            "BuildKey",
            sort=False,
        )
    ):
        random_seed = stable_project_seed(
            PROJECT_NAME,
            repetition_seed,
            (
                "Random_baseline_build_"
                f"{normalise_build_seed_key(build_key)}"
            ),
        )

        scores = (
            np.random.default_rng(
                random_seed
            )
            .random(
                len(build_rows)
            )
        )

        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=scores,
                technique="Random",
                score_direction="descending",
            )
        )

    return pd.concat(
        ranking_frames,
        ignore_index=True,
    )


def create_history_baseline_rankings(
    noisy_training_history,
    clean_training_history,
    clean_evaluation_history,
    clean_evaluation_data,
    fixed_split,
):
    latest_failure_order = {}

    for row in (
        noisy_training_history
        .sort_values(
            [
                "BuildOrder",
                "JobKey",
                "TestKey",
            ],
            kind="mergesort",
        )
        .itertuples(
            index=False
        )
    ):
        test_key = str(
            row.TestKey
        )

        if int(
            row.NoisyVerdict
        ) != 0:
            latest_failure_order[
                test_key
            ] = int(
                row.BuildOrder
            )

    duration_sum = {}
    duration_count = {}

    for row in (
        clean_training_history
        .sort_values(
            [
                "BuildOrder",
                "JobKey",
                "TestKey",
            ],
            kind="mergesort",
        )
        .itertuples(
            index=False
        )
    ):
        test_key = str(
            row.TestKey
        )

        duration = float(
            row.Duration
        )

        if np.isfinite(
            duration
        ):
            duration_sum[
                test_key
            ] = (
                duration_sum.get(
                    test_key,
                    0.0,
                )
                + duration
            )

            duration_count[
                test_key
            ] = (
                duration_count.get(
                    test_key,
                    0,
                )
                + 1
            )

    raw_eval_by_build = {
        str(build_key):
            group.copy()

        for build_key, group in (
            clean_evaluation_history.groupby(
                "BuildKey",
                sort=False,
            )
        )
    }

    model_eval_by_build = {
        str(build_key):
            group.copy()

        for build_key, group in (
            clean_evaluation_data.groupby(
                "BuildKey",
                sort=False,
            )
        )
    }

    latest_frames = []
    qtf_frames = []

    ordered_evaluation_builds = (
        fixed_split[
            fixed_split[
                "Partition"
            ].eq(
                "EVALUATION"
            )
        ]
        .sort_values(
            "BuildOrder",
            kind="mergesort",
        )
    )

    for build_row in (
        ordered_evaluation_builds.itertuples(
            index=False
        )
    ):
        build_key = str(
            build_row.BuildKey
        )

        # Rank before observing the current clean build.
        if build_key in model_eval_by_build:
            build_tests = (
                model_eval_by_build[
                    build_key
                ]
                .copy()
            )

            latest_scores = []
            qtf_scores = []

            for test_key in (
                build_tests[
                    "TestKey"
                ].astype(str)
            ):
                latest_scores.append(
                    float(
                        latest_failure_order.get(
                            test_key,
                            -1,
                        )
                    )
                )

                count = duration_count.get(
                    test_key,
                    0,
                )

                if count > 0:
                    average_duration = (
                        duration_sum[
                            test_key
                        ]
                        / count
                    )
                else:
                    average_duration = np.inf

                qtf_scores.append(
                    float(
                        average_duration
                    )
                )

            latest_frames.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=latest_scores,
                    technique="LatestFail",
                    score_direction="descending",
                )
            )

            qtf_frames.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=qtf_scores,
                    technique="QTF-Avg",
                    score_direction="ascending",
                )
            )

        # Update histories only after ranking.
        current_rows = raw_eval_by_build.get(
            build_key
        )

        if current_rows is None:
            continue

        for row in (
            current_rows
            .sort_values(
                [
                    "JobKey",
                    "TestKey",
                ],
                kind="mergesort",
            )
            .itertuples(
                index=False
            )
        ):
            test_key = str(
                row.TestKey
            )

            if int(
                row.CleanVerdict
            ) != 0:
                latest_failure_order[
                    test_key
                ] = int(
                    row.BuildOrder
                )

            duration = float(
                row.Duration
            )

            if np.isfinite(
                duration
            ):
                duration_sum[
                    test_key
                ] = (
                    duration_sum.get(
                        test_key,
                        0.0,
                    )
                    + duration
                )

                duration_count[
                    test_key
                ] = (
                    duration_count.get(
                        test_key,
                        0,
                    )
                    + 1
                )

    if not latest_frames or not qtf_frames:
        raise RuntimeError(
            "History baselines produced no rankings."
        )

    return (
        pd.concat(
            latest_frames,
            ignore_index=True,
        ),
        pd.concat(
            qtf_frames,
            ignore_index=True,
        ),
    )


# ------------------------------------------------------------
# 7. REC RECONSTRUCTION HELPERS
# ------------------------------------------------------------

def calculate_rates(history):
    history_length = len(
        history
    )

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history[
        "NoisyVerdict"
    ].to_numpy(
        dtype=np.int32
    )

    transitions = history[
        "Transition"
    ].to_numpy(
        dtype=np.int32
    )

    return (
        float(
            np.count_nonzero(
                verdicts != 0
            )
            / history_length
        ),
        float(
            np.count_nonzero(
                verdicts == 2
            )
            / history_length
        ),
        float(
            np.count_nonzero(
                verdicts == 1
            )
            / history_length
        ),
        float(
            np.count_nonzero(
                transitions != 0
            )
            / history_length
        ),
    )


def calculate_max_test_file_rate(
    history,
    target_type,
    current_changed_entities,
    entity_changed_builds,
):
    if target_type == "FAILURE":
        target_builds = set(
            history.loc[
                history[
                    "NoisyVerdict"
                ].ne(0),
                "BuildKey",
            ].astype(str)
        )

    elif target_type == "TRANSITION":
        target_builds = set(
            history.loc[
                history[
                    "Transition"
                ].ne(0),
                "BuildKey",
            ].astype(str)
        )

    else:
        raise ValueError(
            "Unknown file-history target type."
        )

    if not target_builds:
        return -1.0

    maximum_overlap = 0

    for entity_id in current_changed_entities:
        overlap = len(
            entity_changed_builds.get(
                str(entity_id),
                set(),
            )
            & target_builds
        )

        if overlap > maximum_overlap:
            maximum_overlap = overlap

    if maximum_overlap == 0:
        return 0.0

    return float(
        maximum_overlap
        / len(target_builds)
    )


def reconstruct_direct_noisy_rec(
    noisy_raw_training,
    requested_rows,
    requested_builds_by_test,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
):
    records = []

    for test_key, test_history in (
        noisy_raw_training.groupby(
            "TestKey",
            sort=False,
        )
    ):
        test_key = str(
            test_key
        )

        requested_builds = (
            requested_builds_by_test.get(
                test_key
            )
        )

        if not requested_builds:
            continue

        test_history = (
            test_history.sort_values(
                [
                    "BuildOrder",
                    "JobKey",
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
            .copy()
        )

        test_history[
            "Transition"
        ] = (
            test_history[
                "NoisyVerdict"
            ]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(np.int32)
        )

        first_test_build = str(
            test_history[
                "BuildKey"
            ].iloc[0]
        )

        for execution_position in range(
            len(test_history)
        ):
            current_row = test_history.iloc[
                execution_position
            ]

            current_build = str(
                current_row[
                    "BuildKey"
                ]
            )

            if current_build not in requested_builds:
                continue

            record = {
                "BuildKey":
                    current_build,

                "TestKey":
                    test_key,
            }

            history = test_history.iloc[
                :execution_position
            ]

            if history.empty:
                for feature in REC_FEATURE_COLUMNS:
                    record[
                        feature
                    ] = -1.0

                record[
                    "REC_Age"
                ] = 0.0

                records.append(
                    record
                )

                continue

            recent_history = history.tail(
                RECENT_WINDOW
            )

            age = float(
                global_build_position[
                    current_build
                ]
                - global_build_position[
                    first_test_build
                ]
            )

            failure_positions = np.flatnonzero(
                history[
                    "NoisyVerdict"
                ].to_numpy(
                    dtype=np.int32
                ) != 0
            )

            if len(failure_positions) == 0:
                last_failure_age = -1.0
            else:
                last_failure_age = float(
                    len(history)
                    - 1
                    - int(
                        failure_positions[-1]
                    )
                )

            transition_positions = np.flatnonzero(
                history[
                    "Transition"
                ].to_numpy(
                    dtype=np.int32
                ) != 0
            )

            if len(transition_positions) == 0:
                last_transition_age = -1.0
            else:
                last_transition_age = float(
                    len(history)
                    - 1
                    - int(
                        transition_positions[-1]
                    )
                )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(
                recent_history
            )

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(
                history
            )

            current_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            record.update({
                "REC_Age":
                    age,

                "REC_LastFailureAge":
                    last_failure_age,

                "REC_LastTransitionAge":
                    last_transition_age,

                "REC_RecentAvgExeTime":
                    float(
                        recent_history[
                            "Duration"
                        ].mean()
                    ),

                "REC_RecentMaxExeTime":
                    float(
                        recent_history[
                            "Duration"
                        ].max()
                    ),

                "REC_RecentFailRate":
                    recent_fail_rate,

                "REC_RecentAssertRate":
                    recent_assert_rate,

                "REC_RecentExcRate":
                    recent_exc_rate,

                "REC_RecentTransitionRate":
                    recent_transition_rate,

                "REC_TotalAvgExeTime":
                    float(
                        history[
                            "Duration"
                        ].mean()
                    ),

                "REC_TotalMaxExeTime":
                    float(
                        history[
                            "Duration"
                        ].max()
                    ),

                "REC_TotalFailRate":
                    total_fail_rate,

                "REC_TotalAssertRate":
                    total_assert_rate,

                "REC_TotalExcRate":
                    total_exc_rate,

                "REC_TotalTransitionRate":
                    total_transition_rate,

                "REC_LastVerdict":
                    float(
                        recent_history[
                            "NoisyVerdict"
                        ].iloc[-1]
                    ),

                "REC_LastExeTime":
                    float(
                        recent_history[
                            "Duration"
                        ].iloc[-1]
                    ),

                "REC_MaxTestFileFailRate":
                    calculate_max_test_file_rate(
                        history=history,
                        target_type="FAILURE",
                        current_changed_entities=current_entities,
                        entity_changed_builds=entity_changed_builds,
                    ),

                "REC_MaxTestFileTransitionRate":
                    calculate_max_test_file_rate(
                        history=history,
                        target_type="TRANSITION",
                        current_changed_entities=current_entities,
                        entity_changed_builds=entity_changed_builds,
                    ),
            })

            records.append(
                record
            )

    reconstructed = pd.DataFrame(
        records,
        columns=(
            [
                "BuildKey",
                "TestKey",
            ]
            + REC_FEATURE_COLUMNS
        ),
    )

    duplicate_rows = int(
        reconstructed.duplicated(
            subset=[
                "BuildKey",
                "TestKey",
            ],
            keep=False,
        ).sum()
    )

    if duplicate_rows:
        raise RuntimeError(
            "Noisy REC reconstruction contains duplicate rows."
        )

    aligned = (
        requested_rows.merge(
            reconstructed,
            on=[
                "BuildKey",
                "TestKey",
            ],
            how="left",
            validate="one_to_one",
            indicator=True,
            sort=False,
        )
        .sort_values(
            "ModelRowOrder",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    missing_rows = int(
        aligned[
            "_merge"
        ].ne(
            "both"
        ).sum()
    )

    if missing_rows:
        raise RuntimeError(
            "Some training rows lack reconstructed REC values."
        )

    return (
        aligned[
            REC_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="raise",
        )
    )


# ------------------------------------------------------------
# 8. MODEL HELPERS
# ------------------------------------------------------------

def instantiate_model(
    technique,
    model_config,
    model_seed_row,
):
    if technique == "RandomForest":
        return RandomForestClassifier(
            **model_config[
                "RandomForest"
            ],
            random_state=int(
                model_seed_row[
                    "RandomForestSeed"
                ]
            ),
        )

    if technique == "XGBoost":
        return XGBClassifier(
            **model_config[
                "XGBoost"
            ],
            random_state=int(
                model_seed_row[
                    "XGBoostSeed"
                ]
            ),
        )

    if technique == "LightGBM":
        return LGBMClassifier(
            **model_config[
                "LightGBM"
            ],
            random_state=int(
                model_seed_row[
                    "LightGBMSeed"
                ]
            ),
        )

    if technique == "NaiveBayes":
        return GaussianNB(
            var_smoothing=float(
                model_config[
                    "NaiveBayes"
                ][
                    "var_smoothing"
                ]
            )
        )

    raise ValueError(
        f"Unknown model technique: {technique}"
    )


def positive_class_probability(
    fitted_model,
    evaluation_matrix,
):
    probabilities = fitted_model.predict_proba(
        evaluation_matrix
    )

    classes = np.asarray(
        fitted_model.classes_
    )

    positive_positions = np.flatnonzero(
        classes == 1
    )

    if len(positive_positions) != 1:
        raise RuntimeError(
            "Could not uniquely identify positive class 1."
        )

    scores = probabilities[
        :,
        int(
            positive_positions[0]
        ),
    ].astype(float)

    if not np.isfinite(
        scores
    ).all():
        raise RuntimeError(
            "Predicted probabilities contain non-finite values."
        )

    if (
        scores < -1e-12
    ).any() or (
        scores > 1.0 + 1e-12
    ).any():
        raise RuntimeError(
            "Predicted probabilities fall outside [0,1]."
        )

    return np.clip(
        scores,
        0.0,
        1.0,
    )


# ------------------------------------------------------------
# 9. METRIC AGGREGATION
# ------------------------------------------------------------

def calculate_build_metrics(rankings):
    records = []

    grouped = rankings.groupby(
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionOrder",
            "ConditionID",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "BuildKey",
        ],
        sort=False,
    )

    for keys, ranked_build in grouped:
        (
            project_number,
            project,
            project_slug,
            condition_order,
            condition_id,
            noise_percent,
            repetition_seed,
            technique,
            build_key,
        ) = keys

        ranked_build = (
            ranked_build.sort_values(
                "Rank",
                kind="mergesort",
            )
        )

        failures = ranked_build[
            "ActualFailure"
        ].to_numpy(
            dtype=np.int8
        )

        durations = ranked_build[
            "Duration"
        ].to_numpy(
            dtype=float
        )

        records.append({
            "ProjectNumber":
                project_number,

            "Project":
                project,

            "ProjectSlug":
                project_slug,

            "ConditionOrder":
                condition_order,

            "ConditionID":
                condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "Technique":
                technique,

            "Build":
                ranked_build[
                    "Build"
                ].iloc[0],

            "BuildKey":
                str(
                    build_key
                ),

            "BuildOrder":
                int(
                    ranked_build[
                        "build_order"
                    ].iloc[0]
                ),

            "NumberOfTests":
                len(
                    ranked_build
                ),

            "NumberOfFailures":
                int(
                    failures.sum()
                ),

            "TotalDuration":
                float(
                    durations.sum()
                ),

            "APFD":
                calculate_apfd(
                    failures
                ),

            "APFDc":
                calculate_apfdc(
                    failures,
                    durations,
                ),
        })

    return pd.DataFrame(
        records
    )


def calculate_project_runs(build_metrics):
    records = []

    grouped = build_metrics.groupby(
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionOrder",
            "ConditionID",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        sort=False,
    )

    for keys, group in grouped:
        (
            project_number,
            project,
            project_slug,
            condition_order,
            condition_id,
            noise_percent,
            repetition_seed,
            technique,
        ) = keys

        records.append({
            "ProjectNumber":
                project_number,

            "Project":
                project,

            "ProjectSlug":
                project_slug,

            "ConditionOrder":
                condition_order,

            "ConditionID":
                condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "Technique":
                technique,

            "EvaluatedBuilds":
                int(
                    group[
                        "BuildKey"
                    ].nunique()
                ),

            "EvaluationTests":
                int(
                    group[
                        "NumberOfTests"
                    ].sum()
                ),

            "EvaluationFailures":
                int(
                    group[
                        "NumberOfFailures"
                    ].sum()
                ),

            "MeanAPFD":
                float(
                    group[
                        "APFD"
                    ].mean()
                ),

            "MedianAPFD":
                float(
                    group[
                        "APFD"
                    ].median()
                ),

            "StdAPFD":
                float(
                    group[
                        "APFD"
                    ].std(
                        ddof=0
                    )
                ),

            "MeanAPFDc":
                float(
                    group[
                        "APFDc"
                    ].mean()
                ),

            "MedianAPFDc":
                float(
                    group[
                        "APFDc"
                    ].median()
                ),

            "StdAPFDc":
                float(
                    group[
                        "APFDc"
                    ].std(
                        ddof=0
                    )
                ),
        })

    return pd.DataFrame(
        records
    )


# ------------------------------------------------------------
# 10. CONDITION DIRECTORY HELPERS
# ------------------------------------------------------------

CONDITION_DATA_FILES = [
    "rankings.parquet",
    "build_metrics.csv",
    "project_run.csv",
    "model_fits.csv",
    "condition_audit.csv",
    "training_medians.csv",
    "condition_summary.json",
]


def condition_directory_for(
    noise_percent,
    repetition_seed,
):
    return (
        RAW_ROOT
        / f"noise_{int(noise_percent):03d}"
        / f"seed_{int(repetition_seed):02d}"
    )


def validate_completed_condition(
    condition_directory,
    expected_condition_id,
):
    condition_directory = Path(
        condition_directory
    )

    marker_path = (
        condition_directory
        / "_SUCCESS.json"
    )

    if not marker_path.exists():
        return (
            False,
            "Missing _SUCCESS.json",
            None,
        )

    try:
        marker = read_json_with_retry(
            marker_path
        )

        if marker.get(
            "Status"
        ) != CONDITION_PASS_STATUS:
            return (
                False,
                "Unexpected success-marker status",
                None,
            )

        if marker.get(
            "ConditionID"
        ) != expected_condition_id:
            return (
                False,
                "Condition ID differs",
                None,
            )

        file_hashes = marker.get(
            "FileSHA256",
            {},
        )

        if set(
            file_hashes
        ) != set(
            CONDITION_DATA_FILES
        ):
            return (
                False,
                "Success marker has an unexpected file set",
                None,
            )

        for filename in CONDITION_DATA_FILES:
            file_path = (
                condition_directory
                / filename
            )

            if not file_path.exists():
                return (
                    False,
                    f"Missing {filename}",
                    None,
                )

            if calculate_hash(
                file_path
            ) != file_hashes[
                filename
            ]:
                return (
                    False,
                    f"SHA-256 mismatch for {filename}",
                    None,
                )

        summary = read_json_with_retry(
            condition_directory
            / "condition_summary.json"
        )

        expected_counts = {
            "RankingRows":
                EXPECTED_RANKING_ROWS_PER_CONDITION,

            "BuildMetricRows":
                EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,

            "ProjectRunRows":
                EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,

            "ModelFits":
                EXPECTED_MODEL_FITS_PER_CONDITION,

            "TrainingMedianRows":
                EXPECTED_TRAINING_MEDIANS_PER_CONDITION,
        }

        for key, expected_value in (
            expected_counts.items()
        ):
            if int(
                summary.get(
                    key,
                    -1,
                )
            ) != expected_value:
                return (
                    False,
                    f"Summary count differs: {key}",
                    None,
                )

        if int(
            summary.get(
                "FailedValidationChecks",
                -1,
            )
        ) != 0:
            return (
                False,
                "Condition summary reports failed checks",
                None,
            )

        if int(
            marker.get(
                "RawFiles",
                -1,
            )
        ) != EXPECTED_RAW_FILES_PER_CONDITION:
            return (
                False,
                "Success marker raw-file count differs",
                None,
            )

        return (
            True,
            "VALID_COMPLETE_CONDITION",
            marker,
        )

    except Exception as error:
        return (
            False,
            (
                f"{type(error).__name__}: "
                f"{error}"
            ),
            None,
        )


# ------------------------------------------------------------
# 11. PROGRESS HELPERS
# ------------------------------------------------------------

def write_progress(
    status,
    completed_conditions,
    pending_conditions,
    initially_complete,
    newly_completed,
    invalid_removed,
    last_completed_condition,
    invocation_started_at,
):
    payload = {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "Status":
            status,

        "ExpectedConditions":
            EXPECTED_CONDITIONS,

        "CompletedConditions":
            int(
                completed_conditions
            ),

        "PendingConditions":
            int(
                pending_conditions
            ),

        "InitiallyCompleteConditions":
            int(
                initially_complete
            ),

        "NewlyCompletedThisInvocation":
            int(
                newly_completed
            ),

        "InvalidOrIncompleteConditionsRemoved":
            int(
                invalid_removed
            ),

        "LastCompletedCondition":
            last_completed_condition,

        "InvocationStartedAtUTC":
            invocation_started_at,

        "UpdatedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "CompletionRegistryModified":
            False,

        "Project9Accessed":
            False,

        "Project9WriteAttempted":
            False,
    }

    atomic_write_json(
        PROGRESS_PATH,
        payload,
    )


def build_checkpoint_table(
    condition_plan,
):
    records = []

    for row in condition_plan.itertuples(
        index=False
    ):
        condition_id = str(
            row.ConditionID
        )

        condition_directory = condition_directory_for(
            row.NoisePercent,
            row.RepetitionSeed,
        )

        valid, reason, marker = (
            validate_completed_condition(
                condition_directory,
                condition_id,
            )
        )

        if valid:
            records.append({
                "ConditionOrder":
                    int(
                        row.ConditionOrder
                    ),

                "ConditionID":
                    condition_id,

                "NoisePercent":
                    int(
                        row.NoisePercent
                    ),

                "RepetitionSeed":
                    int(
                        row.RepetitionSeed
                    ),

                "Status":
                    CONDITION_PASS_STATUS,

                "ConditionDirectory":
                    str(
                        condition_directory
                    ),

                "RankingRows":
                    int(
                        marker[
                            "RankingRows"
                        ]
                    ),

                "BuildMetricRows":
                    int(
                        marker[
                            "BuildMetricRows"
                        ]
                    ),

                "ProjectRunRows":
                    int(
                        marker[
                            "ProjectRunRows"
                        ]
                    ),

                "ModelFits":
                    int(
                        marker[
                            "ModelFits"
                        ]
                    ),

                "TrainingMedianRows":
                    int(
                        marker[
                            "TrainingMedianRows"
                        ]
                    ),

                "ConditionSeconds":
                    float(
                        marker[
                            "ConditionSeconds"
                        ]
                    ),

                "SuccessMarkerSHA256":
                    calculate_hash(
                        condition_directory
                        / "_SUCCESS.json"
                    ),
            })

        else:
            records.append({
                "ConditionOrder":
                    int(
                        row.ConditionOrder
                    ),

                "ConditionID":
                    condition_id,

                "NoisePercent":
                    int(
                        row.NoisePercent
                    ),

                "RepetitionSeed":
                    int(
                        row.RepetitionSeed
                    ),

                "Status":
                    "PENDING_OR_INVALID",

                "ConditionDirectory":
                    str(
                        condition_directory
                    ),

                "ValidationReason":
                    reason,
            })

    return (
        pd.DataFrame(
            records
        )
        .sort_values(
            "ConditionOrder",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# 12. LOAD AND VALIDATE CHECKPOINTS
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    NOISY_REC_ENGINE_CHECKPOINT_PATH,
    MODEL_PROTOCOL_CHECKPOINT_PATH,
    SMOKE_TEST_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
    STEP3A_STATUS_PATH,
    STEP3B_STATUS_PATH,
    STEP4A_STATUS_PATH,
    STEP4B_STATUS_PATH,
]


missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:
    raise FileNotFoundError(
        "Required Project 10 Step 5A inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


selection_checkpoint = read_json_with_retry(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint = read_json_with_retry(
    REC_CHECKPOINT_PATH
)

noise_checkpoint = read_json_with_retry(
    NOISE_PLAN_CHECKPOINT_PATH
)

noisy_rec_checkpoint = read_json_with_retry(
    NOISY_REC_ENGINE_CHECKPOINT_PATH
)

model_checkpoint = read_json_with_retry(
    MODEL_PROTOCOL_CHECKPOINT_PATH
)

smoke_checkpoint = read_json_with_retry(
    SMOKE_TEST_CHECKPOINT_PATH
)


step2b_status = read_json_with_retry(
    STEP2B_STATUS_PATH
)

step3a_status = read_json_with_retry(
    STEP3A_STATUS_PATH
)

step3b_status = read_json_with_retry(
    STEP3B_STATUS_PATH
)

step4a_status = read_json_with_retry(
    STEP4A_STATUS_PATH
)

step4b_status = read_json_with_retry(
    STEP4B_STATUS_PATH
)


status_expectations = [
    (
        "Step 2B",
        step2b_status.get(
            "Status"
        ),
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "Step 3A",
        step3a_status.get(
            "Status"
        ),
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "Step 3B",
        step3b_status.get(
            "Status"
        ),
        EXPECTED_STEP3B_STATUS,
    ),
    (
        "Step 4A",
        step4a_status.get(
            "Status"
        ),
        EXPECTED_STEP4A_STATUS,
    ),
    (
        "Step 4B",
        step4b_status.get(
            "Status"
        ),
        EXPECTED_STEP4B_STATUS,
    ),
]


for step_name, actual, expected in (
    status_expectations
):
    if actual != expected:
        raise AssertionError(
            f"{step_name} status differs.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


checkpoint_expectations = [
    (
        "REC checkpoint",
        rec_checkpoint.get(
            "Status"
        ),
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "Noise checkpoint",
        noise_checkpoint.get(
            "Status"
        ),
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "Noisy REC checkpoint",
        noisy_rec_checkpoint.get(
            "Status"
        ),
        EXPECTED_STEP3B_STATUS,
    ),
    (
        "Model checkpoint",
        model_checkpoint.get(
            "Status"
        ),
        EXPECTED_STEP4A_STATUS,
    ),
    (
        "Smoke checkpoint",
        smoke_checkpoint.get(
            "Status"
        ),
        EXPECTED_STEP4B_STATUS,
    ),
]


for checkpoint_name, actual, expected in (
    checkpoint_expectations
):
    if actual != expected:
        raise AssertionError(
            f"{checkpoint_name} status differs.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


if selection_checkpoint.get(
    "Project"
) != PROJECT_NAME:
    raise AssertionError(
        "Project identity differs."
    )


if selection_checkpoint.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise AssertionError(
        "Project slug differs."
    )


if selection_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Frozen source-root SHA-256 differs."
    )


# ------------------------------------------------------------
# 13. PATH ISOLATION
# ------------------------------------------------------------

project_10_output_paths = [
    RAW_ROOT,
    CONTROL_DIR,
    PROGRESS_PATH,
    CHECKPOINT_TABLE_PATH,
    EXECUTION_REPORT_PATH,
    STEP5A_STATUS_PATH,
]


for output_path in project_10_output_paths:
    output_string = str(
        output_path
    )

    if PROJECT_SLUG not in output_string:
        raise AssertionError(
            "A Step 5A output path is not isolated to Project 10.\n"
            f"Path: {output_path}"
        )

    if str(
        PROJECT_9_ROOT
    ) in output_string:
        raise AssertionError(
            "A Project 10 output path overlaps Project 9."
        )


RAW_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CONTROL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 14. REGISTRY AND SOURCE — READ ONLY
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


allowed_registry_sets = [
    set(
        range(1, 9)
    ),
    set(
        range(1, 10)
    ),
]


if set(
    registry_project_numbers
) not in allowed_registry_sets:
    raise AssertionError(
        "Registry must contain Projects 1–8, or Projects 1–9."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise AssertionError(
        "Project 10 is already present in the completion registry."
    )


source_files_payload = (
    selection_checkpoint[
        "SourceFiles"
    ]
)

source_root_before = create_source_root_sha256(
    source_files_payload
)


if source_root_before != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Project 10 source differs before the full run."
    )


# ------------------------------------------------------------
# 15. LOAD FROZEN EXPERIMENT INPUTS
# ------------------------------------------------------------

raw_training_path = Path(
    noise_checkpoint[
        "RawTrainingCohort"
    ]
)

raw_evaluation_path = Path(
    noise_checkpoint[
        "RawEvaluationCohort"
    ]
)

noise_rng_manifest_path = Path(
    noise_checkpoint[
        "NoiseRNGManifest"
    ]
)

condition_plan_path = Path(
    noise_checkpoint[
        "ConditionPlan"
    ]
)

clean_direct_rec_path = Path(
    rec_checkpoint[
        "CleanRECReconstructed"
    ]
)

build_entity_map_path = Path(
    rec_checkpoint[
        "BuildEntityMap"
    ]
)

clean_training_base_path = Path(
    model_checkpoint[
        "CleanModelTrainingBase"
    ]
)

clean_evaluation_base_path = Path(
    model_checkpoint[
        "CleanModelEvaluationBase"
    ]
)

model_seed_manifest_path = Path(
    model_checkpoint[
        "ModelSeedManifest"
    ]
)

fixed_split_path = Path(
    selection_checkpoint[
        "FixedSplit"
    ]
)


frozen_paths = [
    raw_training_path,
    raw_evaluation_path,
    noise_rng_manifest_path,
    condition_plan_path,
    clean_direct_rec_path,
    build_entity_map_path,
    clean_training_base_path,
    clean_evaluation_base_path,
    model_seed_manifest_path,
    fixed_split_path,
]


missing_frozen_paths = [
    str(path)
    for path in frozen_paths
    if not path.exists()
]


if missing_frozen_paths:
    raise FileNotFoundError(
        "Frozen Step 5A inputs are missing:\n"
        + "\n".join(
            missing_frozen_paths
        )
    )


raw_evaluation_sha256_before = calculate_hash(
    raw_evaluation_path
)

clean_evaluation_base_sha256_before = calculate_hash(
    clean_evaluation_base_path
)


raw_training = pd.read_parquet(
    raw_training_path
)

raw_evaluation = pd.read_parquet(
    raw_evaluation_path
)

noise_rng_manifest = pd.read_parquet(
    noise_rng_manifest_path
)

condition_plan = pd.read_csv(
    condition_plan_path,
    low_memory=False,
)

clean_direct_rec = pd.read_parquet(
    clean_direct_rec_path
)

build_entity_map = pd.read_csv(
    build_entity_map_path,
    compression="gzip",
    low_memory=False,
)

clean_training_base = pd.read_parquet(
    clean_training_base_path
)

clean_evaluation_base = pd.read_parquet(
    clean_evaluation_base_path
)

model_seed_manifest = pd.read_csv(
    model_seed_manifest_path,
    low_memory=False,
)

fixed_split = pd.read_csv(
    fixed_split_path,
    low_memory=False,
)


# ------------------------------------------------------------
# 16. RESOLVE FROZEN PROTOCOL
# ------------------------------------------------------------

training_metadata_columns = (
    model_checkpoint[
        "TrainingMetadataColumns"
    ]
)

evaluation_metadata_columns = (
    model_checkpoint[
        "EvaluationMetadataColumns"
    ]
)

active_feature_columns = list(
    model_checkpoint[
        "ActiveFeatureColumns"
    ]
)

model_configuration = (
    model_checkpoint[
        "ModelConfiguration"
    ][
        "Models"
    ]
)


if len(
    active_feature_columns
) != EXPECTED_ACTIVE_PREDICTORS:
    raise AssertionError(
        "Active predictor count differs."
    )


meta_training_order = (
    training_metadata_columns[
        "ModelRowOrder"
    ]
)

meta_noise_row = (
    training_metadata_columns[
        "NoiseRowID"
    ]
)

meta_training_build_key = (
    training_metadata_columns[
        "BuildKey"
    ]
)

meta_training_test_key = (
    training_metadata_columns[
        "TestKey"
    ]
)

meta_training_binary_failure = (
    training_metadata_columns[
        "BinaryFailure"
    ]
)


meta_eval_build = (
    evaluation_metadata_columns[
        "Build"
    ]
)

meta_eval_test = (
    evaluation_metadata_columns[
        "Test"
    ]
)

meta_eval_verdict = (
    evaluation_metadata_columns[
        "Verdict"
    ]
)

meta_eval_binary_failure = (
    evaluation_metadata_columns[
        "BinaryFailure"
    ]
)

meta_eval_duration = (
    evaluation_metadata_columns[
        "Duration"
    ]
)

meta_eval_build_order = (
    evaluation_metadata_columns[
        "BuildOrder"
    ]
)

meta_eval_build_key = (
    evaluation_metadata_columns[
        "BuildKey"
    ]
)

meta_eval_test_key = (
    evaluation_metadata_columns[
        "TestKey"
    ]
)


if clean_training_base.columns.duplicated().any():
    raise AssertionError(
        "Clean training base contains duplicate columns."
    )


if clean_evaluation_base.columns.duplicated().any():
    raise AssertionError(
        "Clean evaluation base contains duplicate columns."
    )


if len(raw_training) != EXPECTED_RAW_TRAINING_ROWS:
    raise AssertionError(
        "Raw training row count differs."
    )


if len(raw_evaluation) != EXPECTED_RAW_EVALUATION_ROWS:
    raise AssertionError(
        "Raw evaluation row count differs."
    )


if len(clean_training_base) != EXPECTED_MODEL_TRAINING_ROWS:
    raise AssertionError(
        "Model training row count differs."
    )


if len(clean_evaluation_base) != EXPECTED_MODEL_EVALUATION_ROWS:
    raise AssertionError(
        "Model evaluation row count differs."
    )


if len(condition_plan) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "Condition-plan row count differs."
    )


# ------------------------------------------------------------
# 17. CANONICALISE INPUTS
# ------------------------------------------------------------

for raw_frame in [
    raw_training,
    raw_evaluation,
]:
    raw_frame[
        "BuildKey"
    ] = raw_frame[
        "BuildKey"
    ].astype(str)

    raw_frame[
        "TestKey"
    ] = raw_frame[
        "TestKey"
    ].astype(str)

    raw_frame[
        "JobKey"
    ] = raw_frame[
        "JobKey"
    ].astype(str)

    raw_frame[
        "BuildOrder"
    ] = pd.to_numeric(
        raw_frame[
            "BuildOrder"
        ],
        errors="raise",
    ).astype(np.int32)

    raw_frame[
        "CleanVerdict"
    ] = pd.to_numeric(
        raw_frame[
            "CleanVerdict"
        ],
        errors="raise",
    ).astype(np.int8)

    raw_frame[
        "Duration"
    ] = pd.to_numeric(
        raw_frame[
            "Duration"
        ],
        errors="raise",
    ).astype(float)


raw_training[
    "NoiseRowID"
] = pd.to_numeric(
    raw_training[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)


raw_training = (
    raw_training.sort_values(
        "NoiseRowID",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


expected_noise_ids = np.arange(
    EXPECTED_RAW_TRAINING_ROWS,
    dtype=np.int32,
)


if not np.array_equal(
    raw_training[
        "NoiseRowID"
    ].to_numpy(
        dtype=np.int32
    ),
    expected_noise_ids,
):
    raise AssertionError(
        "Raw-training NoiseRowID order differs."
    )


noise_rng_manifest[
    "RepetitionSeed"
] = pd.to_numeric(
    noise_rng_manifest[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(np.int16)

noise_rng_manifest[
    "NoiseRowID"
] = pd.to_numeric(
    noise_rng_manifest[
        "NoiseRowID"
    ],
    errors="raise",
).astype(np.int32)

noise_rng_manifest[
    "FlipUniform"
] = pd.to_numeric(
    noise_rng_manifest[
        "FlipUniform"
    ],
    errors="raise",
).astype(float)

noise_rng_manifest[
    "SampledFailureSubtype"
] = pd.to_numeric(
    noise_rng_manifest[
        "SampledFailureSubtype"
    ],
    errors="raise",
).astype(np.int8)


condition_plan[
    "ConditionOrder"
] = pd.to_numeric(
    condition_plan[
        "ConditionOrder"
    ],
    errors="raise",
).astype(int)

condition_plan[
    "NoisePercent"
] = pd.to_numeric(
    condition_plan[
        "NoisePercent"
    ],
    errors="raise",
).astype(int)

condition_plan[
    "RepetitionSeed"
] = pd.to_numeric(
    condition_plan[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(int)


condition_plan = (
    condition_plan.sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if condition_plan[
    "ConditionID"
].duplicated().any():
    raise AssertionError(
        "Condition plan contains duplicate IDs."
    )


if sorted(
    condition_plan[
        "NoisePercent"
    ].unique().tolist()
) != NOISE_LEVELS:
    raise AssertionError(
        "Condition-plan noise levels differ."
    )


if sorted(
    condition_plan[
        "RepetitionSeed"
    ].unique().tolist()
) != REPETITION_SEEDS:
    raise AssertionError(
        "Condition-plan seeds differ."
    )


fixed_split[
    "BuildKey"
] = fixed_split[
    "BuildKey"
].astype(str)

fixed_split[
    "BuildOrder"
] = pd.to_numeric(
    fixed_split[
        "BuildOrder"
    ],
    errors="raise",
).astype(np.int32)


clean_direct_rec[
    "BuildKey"
] = clean_direct_rec[
    "BuildKey"
].astype(str)

clean_direct_rec[
    "TestKey"
] = clean_direct_rec[
    "TestKey"
].astype(str)


clean_training_base[
    meta_training_build_key
] = clean_training_base[
    meta_training_build_key
].astype(str)

clean_training_base[
    meta_training_test_key
] = clean_training_base[
    meta_training_test_key
].astype(str)

clean_training_base[
    meta_training_order
] = pd.to_numeric(
    clean_training_base[
        meta_training_order
    ],
    errors="raise",
).astype(np.int64)

clean_training_base[
    meta_noise_row
] = pd.to_numeric(
    clean_training_base[
        meta_noise_row
    ],
    errors="raise",
).astype(np.int32)


clean_evaluation_base[
    meta_eval_build_key
] = clean_evaluation_base[
    meta_eval_build_key
].astype(str)

clean_evaluation_base[
    meta_eval_test_key
] = clean_evaluation_base[
    meta_eval_test_key
].astype(str)


model_seed_manifest[
    "RepetitionSeed"
] = pd.to_numeric(
    model_seed_manifest[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(int)


# ------------------------------------------------------------
# 18. PRECOMPUTE CLEAN MATRICES AND HISTORY STRUCTURES
# ------------------------------------------------------------

training_model_orders = clean_training_base[
    meta_training_order
].to_numpy(
    dtype=np.int64
)


direct_clean_training_rec = (
    clean_direct_rec.iloc[
        training_model_orders
    ][
        REC_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .reset_index(
        drop=True
    )
)


original_clean_training_rec = (
    clean_training_base[
        REC_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .reset_index(
        drop=True
    )
)


direct_clean_matrix = (
    direct_clean_training_rec.to_numpy(
        dtype=float
    )
)

original_clean_matrix = (
    original_clean_training_rec.to_numpy(
        dtype=float
    )
)


requested_rows = pd.DataFrame({
    "ModelRowOrder":
        clean_training_base[
            meta_training_order
        ].to_numpy(
            dtype=np.int64
        ),

    "BuildKey":
        clean_training_base[
            meta_training_build_key
        ].astype(str).to_numpy(),

    "TestKey":
        clean_training_base[
            meta_training_test_key
        ].astype(str).to_numpy(),
})


requested_builds_by_test = {
    str(test_key):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )

    for test_key, group in (
        requested_rows.groupby(
            "TestKey",
            sort=False,
        )
    )
}


ordered_training_builds = (
    raw_training[
        [
            "BuildKey",
            "BuildOrder",
        ]
    ]
    .drop_duplicates(
        subset=[
            "BuildKey",
        ]
    )
    .sort_values(
        "BuildOrder",
        kind="mergesort",
    )[
        "BuildKey"
    ]
    .astype(str)
    .tolist()
)


global_build_position = {
    build_key:
        position

    for position, build_key in enumerate(
        ordered_training_builds
    )
}


build_entity_map[
    "BuildKey"
] = build_entity_map[
    "BuildKey"
].astype(str)

build_entity_map[
    "EntityId"
] = build_entity_map[
    "EntityId"
].astype(str)


changed_entities_by_build = {
    str(build_key):
        set(
            group[
                "EntityId"
            ].astype(str)
        )

    for build_key, group in (
        build_entity_map.groupby(
            "BuildKey",
            sort=False,
        )
    )
}


entity_changed_builds = defaultdict(
    set
)


for row in build_entity_map.itertuples(
    index=False
):
    entity_changed_builds[
        str(
            row.EntityId
        )
    ].add(
        str(
            row.BuildKey
        )
    )


dependent_positions = [
    REC_FEATURE_COLUMNS.index(
        feature
    )
    for feature in VERDICT_DEPENDENT_REC_FEATURES
]

independent_positions = [
    REC_FEATURE_COLUMNS.index(
        feature
    )
    for feature in VERDICT_INDEPENDENT_REC_FEATURES
]


clean_raw_verdicts = raw_training[
    "CleanVerdict"
].to_numpy(
    dtype=np.int8
)

model_noise_row_ids = clean_training_base[
    meta_noise_row
].to_numpy(
    dtype=np.int32
)

clean_model_raw_verdicts = clean_raw_verdicts[
    model_noise_row_ids
]


clean_model_binary_labels = (
    clean_model_raw_verdicts
    != 0
).astype(
    np.int8
)


# ------------------------------------------------------------
# 19. PREPARE CLEAN EVALUATION DATA
# ------------------------------------------------------------

evaluation_ranking_data = pd.DataFrame({
    "Build":
        clean_evaluation_base[
            meta_eval_build
        ].to_numpy(),

    "Test":
        clean_evaluation_base[
            meta_eval_test
        ].to_numpy(),

    "Verdict":
        pd.to_numeric(
            clean_evaluation_base[
                meta_eval_verdict
            ],
            errors="raise",
        ).astype(np.int8).to_numpy(),

    "Duration":
        pd.to_numeric(
            clean_evaluation_base[
                meta_eval_duration
            ],
            errors="raise",
        ).astype(float).to_numpy(),

    "build_order":
        pd.to_numeric(
            clean_evaluation_base[
                meta_eval_build_order
            ],
            errors="raise",
        ).astype(np.int32).to_numpy(),

    "BuildKey":
        clean_evaluation_base[
            meta_eval_build_key
        ].astype(str).to_numpy(),

    "TestKey":
        clean_evaluation_base[
            meta_eval_test_key
        ].astype(str).to_numpy(),
})


if evaluation_ranking_data.columns.duplicated().any():
    raise AssertionError(
        "Evaluation ranking data contains duplicate columns."
    )


evaluation_failures = int(
    pd.to_numeric(
        clean_evaluation_base[
            meta_eval_binary_failure
        ],
        errors="raise",
    ).sum()
)


if evaluation_failures != EXPECTED_MODEL_EVALUATION_FAILURES:
    raise AssertionError(
        "Clean evaluation failure count differs."
    )


if int(
    evaluation_ranking_data[
        "BuildKey"
    ].nunique()
) != EXPECTED_SCORED_EVALUATION_BUILDS:
    raise AssertionError(
        "Scored evaluation-build count differs."
    )


X_evaluation_clean = (
    clean_evaluation_base[
        active_feature_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


# ------------------------------------------------------------
# 20. PRECOMPUTE RNG AND MODEL-SEED LOOKUPS
# ------------------------------------------------------------

rng_by_seed = {}


for repetition_seed in REPETITION_SEEDS:
    seed_rows = (
        noise_rng_manifest[
            noise_rng_manifest[
                "RepetitionSeed"
            ].eq(
                repetition_seed
            )
        ]
        .sort_values(
            "NoiseRowID",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    if len(seed_rows) != EXPECTED_RAW_TRAINING_ROWS:
        raise AssertionError(
            f"RNG row count differs for seed {repetition_seed}."
        )

    if not np.array_equal(
        seed_rows[
            "NoiseRowID"
        ].to_numpy(
            dtype=np.int32
        ),
        expected_noise_ids,
    ):
        raise AssertionError(
            f"RNG NoiseRowID order differs for seed {repetition_seed}."
        )

    rng_by_seed[
        repetition_seed
    ] = {
        "FlipUniform":
            seed_rows[
                "FlipUniform"
            ].to_numpy(
                dtype=float
            ),

        "SampledFailureSubtype":
            seed_rows[
                "SampledFailureSubtype"
            ].to_numpy(
                dtype=np.int8
            ),
    }


model_seed_lookup = {}


for repetition_seed in REPETITION_SEEDS:
    rows = model_seed_manifest[
        model_seed_manifest[
            "RepetitionSeed"
        ].eq(
            repetition_seed
        )
    ]

    if len(rows) != 1:
        raise AssertionError(
            f"Model-seed manifest differs for seed {repetition_seed}."
        )

    model_seed_lookup[
        repetition_seed
    ] = rows.iloc[0]


# Random is constant across noise for the same seed.
random_ranking_cache = {
    repetition_seed:
        create_random_rankings(
            evaluation_data=(
                evaluation_ranking_data
            ),
            repetition_seed=(
                repetition_seed
            ),
        )

    for repetition_seed in REPETITION_SEEDS
}


# ------------------------------------------------------------
# 21. SCAN EXISTING CONDITION CHECKPOINTS
# ------------------------------------------------------------

initially_complete = 0
invalid_or_incomplete_removed = 0

completed_condition_ids = set()


print("\nScanning existing Project 10 condition checkpoints...")


for row in condition_plan.itertuples(
    index=False
):
    condition_id = str(
        row.ConditionID
    )

    condition_directory = condition_directory_for(
        row.NoisePercent,
        row.RepetitionSeed,
    )

    marker_path = (
        condition_directory
        / "_SUCCESS.json"
    )

    if marker_path.exists():
        valid, reason, _ = (
            validate_completed_condition(
                condition_directory,
                condition_id,
            )
        )

        if valid:
            initially_complete += 1
            completed_condition_ids.add(
                condition_id
            )

        else:
            print(
                "Removing invalid Project 10 condition:",
                condition_id,
                "| reason:",
                reason,
            )

            safe_remove_condition_directory(
                condition_directory
            )

            invalid_or_incomplete_removed += 1

    elif condition_directory.exists():
        print(
            "Removing incomplete Project 10 condition:",
            condition_id,
        )

        safe_remove_condition_directory(
            condition_directory
        )

        invalid_or_incomplete_removed += 1


pending_conditions = (
    EXPECTED_CONDITIONS
    - initially_complete
)


print(
    "Complete conditions found:",
    initially_complete,
)

print(
    "Invalid/incomplete conditions removed:",
    invalid_or_incomplete_removed,
)

print(
    "Remaining conditions:",
    pending_conditions,
)


# ------------------------------------------------------------
# 22. CONDITION EXECUTION
# ------------------------------------------------------------

invocation_started = time.perf_counter()

invocation_started_at_utc = datetime.now(
    timezone.utc
).isoformat()

newly_completed = 0

last_completed_condition = (
    None
    if not completed_condition_ids
    else condition_plan.loc[
        condition_plan[
            "ConditionID"
        ].isin(
            completed_condition_ids
        ),
        [
            "ConditionOrder",
            "ConditionID",
        ],
    ]
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )[
        "ConditionID"
    ]
    .iloc[-1]
)


write_progress(
    status=RUNNING_STATUS,
    completed_conditions=initially_complete,
    pending_conditions=pending_conditions,
    initially_complete=initially_complete,
    newly_completed=newly_completed,
    invalid_removed=invalid_or_incomplete_removed,
    last_completed_condition=last_completed_condition,
    invocation_started_at=invocation_started_at_utc,
)


for condition_row in condition_plan.itertuples(
    index=False
):
    condition_order = int(
        condition_row.ConditionOrder
    )

    condition_id = str(
        condition_row.ConditionID
    )

    noise_percent = int(
        condition_row.NoisePercent
    )

    repetition_seed = int(
        condition_row.RepetitionSeed
    )


    if condition_id in completed_condition_ids:
        continue


    condition_started = time.perf_counter()

    condition_directory = condition_directory_for(
        noise_percent,
        repetition_seed,
    )

    condition_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    print("\n" + "-" * 128)

    print(
        f"Condition {condition_order}/{EXPECTED_CONDITIONS}"
    )

    print(
        "Condition ID:",
        condition_id,
    )

    print(
        "Noise:",
        f"{noise_percent}%",
    )

    print(
        "Seed:",
        repetition_seed,
    )


    # --------------------------------------------------------
    # 22A. RECREATE FROZEN NOISE
    # --------------------------------------------------------

    rng_values = rng_by_seed[
        repetition_seed
    ]

    flip_uniforms = rng_values[
        "FlipUniform"
    ]

    sampled_failure_subtypes = rng_values[
        "SampledFailureSubtype"
    ]


    flip_mask = (
        flip_uniforms
        < noise_percent / 100.0
    )


    noisy_raw_verdicts = (
        clean_raw_verdicts.copy()
    )


    pass_to_failure_mask = (
        flip_mask
        & (
            clean_raw_verdicts
            == 0
        )
    )

    failure_to_pass_mask = (
        flip_mask
        & (
            clean_raw_verdicts
            != 0
        )
    )


    noisy_raw_verdicts[
        pass_to_failure_mask
    ] = sampled_failure_subtypes[
        pass_to_failure_mask
    ]

    noisy_raw_verdicts[
        failure_to_pass_mask
    ] = 0


    noisy_model_raw_verdicts = (
        noisy_raw_verdicts[
            model_noise_row_ids
        ]
    )

    noisy_binary_labels = (
        noisy_model_raw_verdicts
        != 0
    ).astype(
        np.int8
    )


    actual_raw_flips = int(
        flip_mask.sum()
    )

    actual_raw_label_changes = int(
        np.count_nonzero(
            noisy_raw_verdicts
            != clean_raw_verdicts
        )
    )

    actual_model_label_changes = int(
        np.count_nonzero(
            noisy_model_raw_verdicts
            != clean_model_raw_verdicts
        )
    )


    expected_raw_flips = int(
        condition_row.NumberFlipped
    )

    expected_raw_label_changes = int(
        condition_row.RawLabelChanges
    )

    expected_model_label_changes = int(
        condition_row.ModelLabelChanges
    )


    flip_hash_match = bool(
        array_sha256(
            flip_mask.astype(
                np.uint8
            ),
            "<u1",
        )
        == str(
            condition_row.FlipMaskSHA256
        )
    )

    raw_verdict_hash_match = bool(
        array_sha256(
            noisy_raw_verdicts,
            "<i1",
        )
        == str(
            condition_row.NoisyRawVerdictSHA256
        )
    )

    model_verdict_hash_match = bool(
        array_sha256(
            noisy_model_raw_verdicts,
            "<i1",
        )
        == str(
            condition_row.NoisyModelVerdictSHA256
        )
    )


    if actual_raw_flips != expected_raw_flips:
        raise AssertionError(
            f"{condition_id}: raw flip count differs."
        )


    if (
        actual_raw_label_changes
        != expected_raw_label_changes
    ):
        raise AssertionError(
            f"{condition_id}: raw-label change count differs."
        )


    if (
        actual_model_label_changes
        != expected_model_label_changes
    ):
        raise AssertionError(
            f"{condition_id}: model-label change count differs."
        )


    if not (
        flip_hash_match
        and raw_verdict_hash_match
        and model_verdict_hash_match
    ):
        raise AssertionError(
            f"{condition_id}: frozen noise hashes differ."
        )


    label_classes = sorted(
        np.unique(
            noisy_binary_labels
        ).tolist()
    )


    if label_classes != [
        0,
        1,
    ]:
        raise AssertionError(
            f"{condition_id}: noisy labels do not contain both classes."
        )


    noisy_raw_history = raw_training.copy()

    noisy_raw_history[
        "NoisyVerdict"
    ] = noisy_raw_verdicts


    # --------------------------------------------------------
    # 22B. RECONSTRUCT NOISY REC FEATURES
    # --------------------------------------------------------

    rec_started = time.perf_counter()


    if noise_percent == 0:
        direct_noisy_matrix = (
            direct_clean_matrix.copy()
        )

    else:
        direct_noisy_rec = reconstruct_direct_noisy_rec(
            noisy_raw_training=noisy_raw_history,
            requested_rows=requested_rows,
            requested_builds_by_test=requested_builds_by_test,
            global_build_position=global_build_position,
            changed_entities_by_build=changed_entities_by_build,
            entity_changed_builds=entity_changed_builds,
        )

        direct_noisy_matrix = (
            direct_noisy_rec.to_numpy(
                dtype=float
            )
        )


    direct_delta_matrix = (
        direct_noisy_matrix
        - direct_clean_matrix
    )


    anchored_noisy_matrix = (
        original_clean_matrix.copy()
    )


    anchored_noisy_matrix[
        :,
        dependent_positions,
    ] = (
        original_clean_matrix[
            :,
            dependent_positions,
        ]
        + direct_delta_matrix[
            :,
            dependent_positions,
        ]
    )


    anchored_noisy_matrix[
        :,
        independent_positions,
    ] = original_clean_matrix[
        :,
        independent_positions,
    ]


    if not np.isfinite(
        anchored_noisy_matrix
    ).all():
        raise AssertionError(
            f"{condition_id}: noisy REC matrix contains non-finite values."
        )


    independent_rec_changed_values = int(
        np.count_nonzero(
            anchored_noisy_matrix[
                :,
                independent_positions,
            ]
            != original_clean_matrix[
                :,
                independent_positions,
            ]
        )
    )


    dependent_match_clean = np.isclose(
        anchored_noisy_matrix[
            :,
            dependent_positions,
        ],
        original_clean_matrix[
            :,
            dependent_positions,
        ],
        rtol=1e-12,
        atol=1e-12,
        equal_nan=False,
    )


    dependent_rec_changed_values = int(
        (
            ~dependent_match_clean
        ).sum()
    )


    zero_noise_rec_mismatches = None


    if noise_percent == 0:
        zero_noise_rec_mismatches = int(
            (
                ~np.isclose(
                    anchored_noisy_matrix,
                    original_clean_matrix,
                    rtol=1e-12,
                    atol=1e-12,
                    equal_nan=False,
                )
            ).sum()
        )

        if zero_noise_rec_mismatches != 0:
            raise AssertionError(
                f"{condition_id}: 0% REC identity failed."
            )


    if independent_rec_changed_values != 0:
        raise AssertionError(
            f"{condition_id}: independent REC features changed."
        )


    if (
        noise_percent > 0
        and dependent_rec_changed_values <= 0
    ):
        raise AssertionError(
            f"{condition_id}: dependent REC features did not change."
        )


    rec_seconds = float(
        time.perf_counter()
        - rec_started
    )


    # --------------------------------------------------------
    # 22C. PREPARE CONDITION MATRICES
    # --------------------------------------------------------

    condition_training_features = (
        clean_training_base[
            active_feature_columns
        ]
        .copy()
    )


    noisy_rec_frame = pd.DataFrame(
        anchored_noisy_matrix,
        columns=REC_FEATURE_COLUMNS,
    )


    for feature in VERDICT_DEPENDENT_REC_FEATURES:
        condition_training_features[
            feature
        ] = noisy_rec_frame[
            feature
        ].to_numpy(
            dtype=float
        )


    for feature in VERDICT_INDEPENDENT_REC_FEATURES:
        condition_training_features[
            feature
        ] = clean_training_base[
            feature
        ].to_numpy()


    X_training = (
        condition_training_features
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )


    training_medians = (
        X_training.median(
            axis=0
        )
        .fillna(
            0.0
        )
    )


    X_training_imputed = (
        X_training
        .fillna(
            training_medians
        )
        .astype(float)
    )


    X_evaluation_imputed = (
        X_evaluation_clean
        .fillna(
            training_medians
        )
        .astype(float)
    )


    if not np.isfinite(
        X_training_imputed.to_numpy(
            dtype=float
        )
    ).all():
        raise AssertionError(
            f"{condition_id}: training matrix contains non-finite values."
        )


    if not np.isfinite(
        X_evaluation_imputed.to_numpy(
            dtype=float
        )
    ).all():
        raise AssertionError(
            f"{condition_id}: evaluation matrix contains non-finite values."
        )


    training_medians_frame = pd.DataFrame({
        "ProjectNumber":
            np.full(
                EXPECTED_ACTIVE_PREDICTORS,
                PROJECT_NUMBER,
                dtype=np.int16,
            ),

        "Project":
            np.full(
                EXPECTED_ACTIVE_PREDICTORS,
                PROJECT_NAME,
                dtype=object,
            ),

        "ProjectSlug":
            np.full(
                EXPECTED_ACTIVE_PREDICTORS,
                PROJECT_SLUG,
                dtype=object,
            ),

        "ConditionOrder":
            np.full(
                EXPECTED_ACTIVE_PREDICTORS,
                condition_order,
                dtype=np.int16,
            ),

        "ConditionID":
            np.full(
                EXPECTED_ACTIVE_PREDICTORS,
                condition_id,
                dtype=object,
            ),

        "NoisePercent":
            np.full(
                EXPECTED_ACTIVE_PREDICTORS,
                noise_percent,
                dtype=np.int16,
            ),

        "RepetitionSeed":
            np.full(
                EXPECTED_ACTIVE_PREDICTORS,
                repetition_seed,
                dtype=np.int16,
            ),

        "FeatureOrder":
            np.arange(
                1,
                EXPECTED_ACTIVE_PREDICTORS + 1,
                dtype=np.int16,
            ),

        "Feature":
            active_feature_columns,

        "TrainingMedian":
            [
                float(
                    training_medians[
                        feature
                    ]
                )
                for feature in active_feature_columns
            ],
    })


    # --------------------------------------------------------
    # 22D. FOUR ML MODELS
    # --------------------------------------------------------

    ranking_frames = []
    model_fit_records = []

    model_seed_row = model_seed_lookup[
        repetition_seed
    ]


    for technique in ML_TECHNIQUES:
        model = instantiate_model(
            technique=technique,
            model_config=model_configuration,
            model_seed_row=model_seed_row,
        )


        fit_started = time.perf_counter()


        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore"
            )

            model.fit(
                X_training_imputed,
                noisy_binary_labels,
            )


        fit_seconds = float(
            time.perf_counter()
            - fit_started
        )


        prediction_started = time.perf_counter()


        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore"
            )

            scores = positive_class_probability(
                fitted_model=model,
                evaluation_matrix=(
                    X_evaluation_imputed
                ),
            )


        prediction_seconds = float(
            time.perf_counter()
            - prediction_started
        )


        technique_rankings = create_ml_rankings(
            evaluation_data=(
                evaluation_ranking_data
            ),
            scores=scores,
            technique=technique,
        )


        ranking_frames.append(
            add_condition_metadata(
                ranking=technique_rankings,
                condition_order=condition_order,
                condition_id=condition_id,
                noise_percent=noise_percent,
                repetition_seed=repetition_seed,
            )
        )


        if technique == "RandomForest":
            model_seed_value = int(
                model_seed_row[
                    "RandomForestSeed"
                ]
            )

        elif technique == "XGBoost":
            model_seed_value = int(
                model_seed_row[
                    "XGBoostSeed"
                ]
            )

        elif technique == "LightGBM":
            model_seed_value = int(
                model_seed_row[
                    "LightGBMSeed"
                ]
            )

        else:
            model_seed_value = None


        model_fit_records.append({
            "ProjectNumber":
                PROJECT_NUMBER,

            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "ConditionOrder":
                condition_order,

            "ConditionID":
                condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "Technique":
                technique,

            "ModelClass":
                type(model).__name__,

            "ModelSeed":
                model_seed_value,

            "TrainingRows":
                EXPECTED_MODEL_TRAINING_ROWS,

            "TrainingFailures":
                int(
                    noisy_binary_labels.sum()
                ),

            "EvaluationRows":
                EXPECTED_MODEL_EVALUATION_ROWS,

            "ActivePredictors":
                EXPECTED_ACTIVE_PREDICTORS,

            "FitSeconds":
                fit_seconds,

            "PredictSeconds":
                prediction_seconds,

            "MinimumScore":
                float(
                    scores.min()
                ),

            "MaximumScore":
                float(
                    scores.max()
                ),

            "MeanScore":
                float(
                    scores.mean()
                ),

            "ScoreSHA256":
                array_sha256(
                    scores,
                    "<f8",
                ),

            "FitStatus":
                "SUCCESS",
        })


        del model
        del scores
        del technique_rankings

        gc.collect()


    # --------------------------------------------------------
    # 22E. THREE BASELINES
    # --------------------------------------------------------

    random_rankings = (
        random_ranking_cache[
            repetition_seed
        ]
        .copy()
    )


    (
        latest_fail_rankings,
        qtf_rankings,
    ) = create_history_baseline_rankings(
        noisy_training_history=(
            noisy_raw_history
        ),
        clean_training_history=(
            raw_training
        ),
        clean_evaluation_history=(
            raw_evaluation
        ),
        clean_evaluation_data=(
            evaluation_ranking_data
        ),
        fixed_split=(
            fixed_split
        ),
    )


    for baseline_ranking in [
        random_rankings,
        latest_fail_rankings,
        qtf_rankings,
    ]:
        ranking_frames.append(
            add_condition_metadata(
                ranking=baseline_ranking,
                condition_order=condition_order,
                condition_id=condition_id,
                noise_percent=noise_percent,
                repetition_seed=repetition_seed,
            )
        )


    # --------------------------------------------------------
    # 22F. COMBINE, METRICS AND CONDITION VALIDATION
    # --------------------------------------------------------

    rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )


    rankings = (
        rankings.sort_values(
            [
                "Technique",
                "build_order",
                "Rank",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


    build_metrics = calculate_build_metrics(
        rankings
    )


    build_metrics = (
        build_metrics.sort_values(
            [
                "Technique",
                "BuildOrder",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


    project_run = calculate_project_runs(
        build_metrics
    )


    project_run = (
        project_run.sort_values(
            "Technique",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


    model_fits = pd.DataFrame(
        model_fit_records
    )


    duplicate_ranking_rows = int(
        rankings.duplicated(
            subset=[
                "ConditionID",
                "Technique",
                "BuildKey",
                "TestKey",
            ],
            keep=False,
        ).sum()
    )


    duplicate_rank_rows = int(
        rankings.duplicated(
            subset=[
                "ConditionID",
                "Technique",
                "BuildKey",
                "Rank",
            ],
            keep=False,
        ).sum()
    )


    noncontiguous_rank_groups = 0


    for _, rank_group in rankings.groupby(
        [
            "Technique",
            "BuildKey",
        ],
        sort=False,
    ):
        actual_ranks = (
            rank_group[
                "Rank"
            ]
            .sort_values()
            .to_numpy(
                dtype=int
            )
        )

        expected_ranks = np.arange(
            1,
            len(rank_group) + 1,
            dtype=int,
        )

        if not np.array_equal(
            actual_ranks,
            expected_ranks,
        ):
            noncontiguous_rank_groups += 1


    metric_nan_values = int(
        build_metrics[
            [
                "APFD",
                "APFDc",
            ]
        ].isna().sum().sum()
    )


    metric_out_of_range_values = int(
        (
            ~build_metrics[
                "APFD"
            ].between(
                0.0,
                1.0,
                inclusive="both",
            )
        ).sum()
        +
        (
            ~build_metrics[
                "APFDc"
            ].between(
                0.0,
                1.0,
                inclusive="both",
            )
        ).sum()
    )


    technique_set_match = (
        set(
            rankings[
                "Technique"
            ].unique()
        )
        == set(
            ALL_TECHNIQUES
        )
    )


    evaluated_build_count_violations = int(
        project_run[
            "EvaluatedBuilds"
        ].ne(
            EXPECTED_SCORED_EVALUATION_BUILDS
        ).sum()
    )


    evaluation_failure_total_violations = int(
        project_run[
            "EvaluationFailures"
        ].ne(
            EXPECTED_MODEL_EVALUATION_FAILURES
        ).sum()
    )


    model_fit_failures = int(
        model_fits[
            "FitStatus"
        ].ne(
            "SUCCESS"
        ).sum()
    )


    condition_validation_records = [
        {
            "Check":
                "Expected raw flips",

            "Expected":
                expected_raw_flips,

            "Actual":
                actual_raw_flips,

            "Pass":
                actual_raw_flips
                == expected_raw_flips,
        },
        {
            "Check":
                "Expected raw-label changes",

            "Expected":
                expected_raw_label_changes,

            "Actual":
                actual_raw_label_changes,

            "Pass":
                actual_raw_label_changes
                == expected_raw_label_changes,
        },
        {
            "Check":
                "Expected model-label changes",

            "Expected":
                expected_model_label_changes,

            "Actual":
                actual_model_label_changes,

            "Pass":
                actual_model_label_changes
                == expected_model_label_changes,
        },
        {
            "Check":
                "Flip-mask hash",

            "Expected":
                True,

            "Actual":
                flip_hash_match,

            "Pass":
                flip_hash_match,
        },
        {
            "Check":
                "Raw-verdict hash",

            "Expected":
                True,

            "Actual":
                raw_verdict_hash_match,

            "Pass":
                raw_verdict_hash_match,
        },
        {
            "Check":
                "Model-verdict hash",

            "Expected":
                True,

            "Actual":
                model_verdict_hash_match,

            "Pass":
                model_verdict_hash_match,
        },
        {
            "Check":
                "Independent REC changed values",

            "Expected":
                0,

            "Actual":
                independent_rec_changed_values,

            "Pass":
                independent_rec_changed_values == 0,
        },
        {
            "Check":
                "Noise propagation",

            "Expected":
                (
                    0
                    if noise_percent == 0
                    else "> 0"
                ),

            "Actual":
                dependent_rec_changed_values,

            "Pass":
                (
                    dependent_rec_changed_values == 0
                    if noise_percent == 0
                    else dependent_rec_changed_values > 0
                ),
        },
        {
            "Check":
                "Ranking rows",

            "Expected":
                EXPECTED_RANKING_ROWS_PER_CONDITION,

            "Actual":
                len(rankings),

            "Pass":
                len(rankings)
                == EXPECTED_RANKING_ROWS_PER_CONDITION,
        },
        {
            "Check":
                "Build-metric rows",

            "Expected":
                EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,

            "Actual":
                len(build_metrics),

            "Pass":
                len(build_metrics)
                == EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        },
        {
            "Check":
                "Project-run rows",

            "Expected":
                EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,

            "Actual":
                len(project_run),

            "Pass":
                len(project_run)
                == EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        },
        {
            "Check":
                "Model fits",

            "Expected":
                EXPECTED_MODEL_FITS_PER_CONDITION,

            "Actual":
                len(model_fits),

            "Pass":
                len(model_fits)
                == EXPECTED_MODEL_FITS_PER_CONDITION,
        },
        {
            "Check":
                "Training medians",

            "Expected":
                EXPECTED_TRAINING_MEDIANS_PER_CONDITION,

            "Actual":
                len(training_medians_frame),

            "Pass":
                len(training_medians_frame)
                == EXPECTED_TRAINING_MEDIANS_PER_CONDITION,
        },
        {
            "Check":
                "Technique set",

            "Expected":
                sorted(
                    ALL_TECHNIQUES
                ),

            "Actual":
                sorted(
                    rankings[
                        "Technique"
                    ].unique().tolist()
                ),

            "Pass":
                technique_set_match,
        },
        {
            "Check":
                "Duplicate ranking rows",

            "Expected":
                0,

            "Actual":
                duplicate_ranking_rows,

            "Pass":
                duplicate_ranking_rows == 0,
        },
        {
            "Check":
                "Duplicate rank rows",

            "Expected":
                0,

            "Actual":
                duplicate_rank_rows,

            "Pass":
                duplicate_rank_rows == 0,
        },
        {
            "Check":
                "Non-contiguous rank groups",

            "Expected":
                0,

            "Actual":
                noncontiguous_rank_groups,

            "Pass":
                noncontiguous_rank_groups == 0,
        },
        {
            "Check":
                "Metric NaN values",

            "Expected":
                0,

            "Actual":
                metric_nan_values,

            "Pass":
                metric_nan_values == 0,
        },
        {
            "Check":
                "Metric values outside [0,1]",

            "Expected":
                0,

            "Actual":
                metric_out_of_range_values,

            "Pass":
                metric_out_of_range_values == 0,
        },
        {
            "Check":
                "Evaluated-build violations",

            "Expected":
                0,

            "Actual":
                evaluated_build_count_violations,

            "Pass":
                evaluated_build_count_violations == 0,
        },
        {
            "Check":
                "Evaluation-failure total violations",

            "Expected":
                0,

            "Actual":
                evaluation_failure_total_violations,

            "Pass":
                evaluation_failure_total_violations == 0,
        },
        {
            "Check":
                "Model-fit failures",

            "Expected":
                0,

            "Actual":
                model_fit_failures,

            "Pass":
                model_fit_failures == 0,
        },
    ]


    condition_validation = pd.DataFrame(
        condition_validation_records
    )


    failed_condition_checks = condition_validation[
        ~condition_validation[
            "Pass"
        ]
    ]


    if not failed_condition_checks.empty:
        print(
            "\nCondition validation failed:"
        )

        display(
            failed_condition_checks
        )

        raise RuntimeError(
            f"{condition_id}: CONDITION VALIDATION FAILED."
        )


    condition_seconds = float(
        time.perf_counter()
        - condition_started
    )


    condition_audit = pd.DataFrame([
        {
            "ProjectNumber":
                PROJECT_NUMBER,

            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "ConditionOrder":
                condition_order,

            "ConditionID":
                condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "ExpectedRawFlips":
                expected_raw_flips,

            "ActualRawFlips":
                actual_raw_flips,

            "RealisedNoisePercent":
                float(
                    100.0
                    * actual_raw_flips
                    / EXPECTED_RAW_TRAINING_ROWS
                ),

            "ExpectedRawLabelChanges":
                expected_raw_label_changes,

            "ActualRawLabelChanges":
                actual_raw_label_changes,

            "ExpectedModelLabelChanges":
                expected_model_label_changes,

            "ActualModelLabelChanges":
                actual_model_label_changes,

            "CleanRawFailures":
                int(
                    np.count_nonzero(
                        clean_raw_verdicts
                        != 0
                    )
                ),

            "NoisyRawFailures":
                int(
                    np.count_nonzero(
                        noisy_raw_verdicts
                        != 0
                    )
                ),

            "CleanModelFailures":
                int(
                    clean_model_binary_labels.sum()
                ),

            "NoisyModelFailures":
                int(
                    noisy_binary_labels.sum()
                ),

            "PassToFailure":
                int(
                    pass_to_failure_mask.sum()
                ),

            "FailureToPass":
                int(
                    failure_to_pass_mask.sum()
                ),

            "DependentRECChangedValues":
                dependent_rec_changed_values,

            "IndependentRECChangedValues":
                independent_rec_changed_values,

            "ZeroNoiseRECMismatches":
                zero_noise_rec_mismatches,

            "FlipMaskHashMatch":
                flip_hash_match,

            "RawVerdictHashMatch":
                raw_verdict_hash_match,

            "ModelVerdictHashMatch":
                model_verdict_hash_match,

            "RECReconstructionSeconds":
                rec_seconds,

            "ConditionSeconds":
                condition_seconds,

            "ValidationChecks":
                len(
                    condition_validation
                ),

            "FailedValidationChecks":
                len(
                    failed_condition_checks
                ),
        }
    ])


    # --------------------------------------------------------
    # 22G. ATOMIC CONDITION WRITE
    # --------------------------------------------------------

    rankings_path = (
        condition_directory
        / "rankings.parquet"
    )

    build_metrics_path = (
        condition_directory
        / "build_metrics.csv"
    )

    project_run_path = (
        condition_directory
        / "project_run.csv"
    )

    model_fits_path = (
        condition_directory
        / "model_fits.csv"
    )

    condition_audit_path = (
        condition_directory
        / "condition_audit.csv"
    )

    training_medians_path = (
        condition_directory
        / "training_medians.csv"
    )

    condition_summary_path = (
        condition_directory
        / "condition_summary.json"
    )

    success_marker_path = (
        condition_directory
        / "_SUCCESS.json"
    )


    atomic_write_parquet(
        rankings_path,
        rankings,
    )

    atomic_write_csv(
        build_metrics_path,
        build_metrics,
    )

    atomic_write_csv(
        project_run_path,
        project_run,
    )

    atomic_write_csv(
        model_fits_path,
        model_fits,
    )

    atomic_write_csv(
        condition_audit_path,
        condition_audit,
    )

    atomic_write_csv(
        training_medians_path,
        training_medians_frame,
    )


    data_file_hashes = {
        "rankings.parquet":
            calculate_hash(
                rankings_path
            ),

        "build_metrics.csv":
            calculate_hash(
                build_metrics_path
            ),

        "project_run.csv":
            calculate_hash(
                project_run_path
            ),

        "model_fits.csv":
            calculate_hash(
                model_fits_path
            ),

        "condition_audit.csv":
            calculate_hash(
                condition_audit_path
            ),

        "training_medians.csv":
            calculate_hash(
                training_medians_path
            ),
    }


    condition_summary_payload = {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "Status":
            CONDITION_PASS_STATUS,

        "ConditionOrder":
            condition_order,

        "ConditionID":
            condition_id,

        "NoisePercent":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "RawTrainingRows":
            EXPECTED_RAW_TRAINING_ROWS,

        "ModelTrainingRows":
            EXPECTED_MODEL_TRAINING_ROWS,

        "EvaluationRows":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "EvaluationFailures":
            EXPECTED_MODEL_EVALUATION_FAILURES,

        "EvaluatedBuilds":
            EXPECTED_SCORED_EVALUATION_BUILDS,

        "ActualRawFlips":
            actual_raw_flips,

        "RealisedNoisePercent":
            float(
                100.0
                * actual_raw_flips
                / EXPECTED_RAW_TRAINING_ROWS
            ),

        "ActualRawLabelChanges":
            actual_raw_label_changes,

        "ActualModelLabelChanges":
            actual_model_label_changes,

        "NoisyModelFailures":
            int(
                noisy_binary_labels.sum()
            ),

        "DependentRECChangedValues":
            dependent_rec_changed_values,

        "IndependentRECChangedValues":
            independent_rec_changed_values,

        "RankingRows":
            len(
                rankings
            ),

        "BuildMetricRows":
            len(
                build_metrics
            ),

        "ProjectRunRows":
            len(
                project_run
            ),

        "ModelFits":
            len(
                model_fits
            ),

        "TrainingMedianRows":
            len(
                training_medians_frame
            ),

        "ConditionSeconds":
            condition_seconds,

        "RECReconstructionSeconds":
            rec_seconds,

        "ValidationChecks":
            len(
                condition_validation
            ),

        "FailedValidationChecks":
            len(
                failed_condition_checks
            ),

        "DataFileSHA256":
            data_file_hashes,

        "EvaluationDataClean":
            True,

        "CompletionRegistryModified":
            False,

        "Project9Accessed":
            False,

        "Project9WriteAttempted":
            False,

        "CompletedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    atomic_write_json(
        condition_summary_path,
        condition_summary_payload,
    )


    all_file_hashes = {
        **data_file_hashes,

        "condition_summary.json":
            calculate_hash(
                condition_summary_path
            ),
    }


    success_payload = {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "Status":
            CONDITION_PASS_STATUS,

        "ConditionOrder":
            condition_order,

        "ConditionID":
            condition_id,

        "NoisePercent":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "RankingRows":
            len(
                rankings
            ),

        "BuildMetricRows":
            len(
                build_metrics
            ),

        "ProjectRunRows":
            len(
                project_run
            ),

        "ModelFits":
            len(
                model_fits
            ),

        "TrainingMedianRows":
            len(
                training_medians_frame
            ),

        "RawFiles":
            EXPECTED_RAW_FILES_PER_CONDITION,

        "ConditionSeconds":
            condition_seconds,

        "FileSHA256":
            all_file_hashes,

        "CompletionRegistryModified":
            False,

        "Project9Accessed":
            False,

        "Project9WriteAttempted":
            False,

        "CompletedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    # Success marker is always written last.
    atomic_write_json(
        success_marker_path,
        success_payload,
    )


    valid, validation_reason, _ = (
        validate_completed_condition(
            condition_directory,
            condition_id,
        )
    )


    if not valid:
        raise RuntimeError(
            f"{condition_id}: post-write validation failed.\n"
            f"Reason: {validation_reason}"
        )


    completed_condition_ids.add(
        condition_id
    )

    newly_completed += 1

    completed_count = len(
        completed_condition_ids
    )

    pending_count = (
        EXPECTED_CONDITIONS
        - completed_count
    )

    last_completed_condition = (
        condition_id
    )


    write_progress(
        status=RUNNING_STATUS,
        completed_conditions=completed_count,
        pending_conditions=pending_count,
        initially_complete=initially_complete,
        newly_completed=newly_completed,
        invalid_removed=invalid_or_incomplete_removed,
        last_completed_condition=last_completed_condition,
        invocation_started_at=invocation_started_at_utc,
    )


    print(
        "Flipped raw rows:",
        actual_raw_flips,
    )

    print(
        "Realised noise:",
        round(
            100.0
            * actual_raw_flips
            / EXPECTED_RAW_TRAINING_ROWS,
            6,
        ),
        "%",
    )

    print(
        "Noisy model failures:",
        int(
            noisy_binary_labels.sum()
        ),
    )

    print(
        "Dependent REC changes:",
        dependent_rec_changed_values,
    )

    print(
        "Elapsed seconds:",
        round(
            condition_seconds,
            2,
        ),
    )

    print(
        "Checkpointed conditions:",
        f"{completed_count}/{EXPECTED_CONDITIONS}",
    )


    del noisy_raw_history
    del noisy_raw_verdicts
    del noisy_model_raw_verdicts
    del noisy_binary_labels
    del direct_noisy_matrix
    del direct_delta_matrix
    del anchored_noisy_matrix
    del noisy_rec_frame
    del condition_training_features
    del X_training
    del X_training_imputed
    del X_evaluation_imputed
    del rankings
    del build_metrics
    del project_run
    del model_fits
    del condition_audit
    del training_medians_frame
    del ranking_frames

    gc.collect()


# ------------------------------------------------------------
# 23. FINAL COMPLETE-SCAN VALIDATION
# ------------------------------------------------------------

final_checkpoint = build_checkpoint_table(
    condition_plan
)


completed_final = final_checkpoint[
    final_checkpoint[
        "Status"
    ].eq(
        CONDITION_PASS_STATUS
    )
].copy()


invalid_final = final_checkpoint[
    ~final_checkpoint[
        "Status"
    ].eq(
        CONDITION_PASS_STATUS
    )
].copy()


if not invalid_final.empty:
    print(
        "\nInvalid or incomplete final conditions:"
    )

    display(
        invalid_final
    )

    raise RuntimeError(
        "Project 10 full run ended with invalid conditions."
    )


if len(completed_final) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "Final completed-condition count differs."
    )


total_model_fits = int(
    completed_final[
        "ModelFits"
    ].sum()
)

total_ranking_rows = int(
    completed_final[
        "RankingRows"
    ].sum()
)

total_build_metric_rows = int(
    completed_final[
        "BuildMetricRows"
    ].sum()
)

total_project_run_rows = int(
    completed_final[
        "ProjectRunRows"
    ].sum()
)

total_training_median_rows = int(
    completed_final[
        "TrainingMedianRows"
    ].sum()
)


if total_model_fits != EXPECTED_ML_FITS:
    raise AssertionError(
        "Final ML-fit total differs."
    )


if total_ranking_rows != EXPECTED_TOTAL_RANKING_ROWS:
    raise AssertionError(
        "Final ranking-row total differs."
    )


if (
    total_build_metric_rows
    != EXPECTED_TOTAL_BUILD_METRIC_ROWS
):
    raise AssertionError(
        "Final build-metric row total differs."
    )


if (
    total_project_run_rows
    != EXPECTED_TOTAL_PROJECT_RUN_ROWS
):
    raise AssertionError(
        "Final project-run row total differs."
    )


if (
    total_training_median_rows
    != EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS
):
    raise AssertionError(
        "Final training-median row total differs."
    )


raw_files = sorted([
    path
    for path in RAW_ROOT.rglob("*")
    if path.is_file()
])


actual_raw_files = len(
    raw_files
)

raw_bytes = int(
    sum(
        path.stat().st_size
        for path in raw_files
    )
)


if actual_raw_files != EXPECTED_TOTAL_RAW_FILES:
    raise AssertionError(
        "Final raw-file count differs.\n"
        f"Expected: {EXPECTED_TOTAL_RAW_FILES}\n"
        f"Actual:   {actual_raw_files}"
    )


raw_evaluation_sha256_after = calculate_hash(
    raw_evaluation_path
)

clean_evaluation_base_sha256_after = calculate_hash(
    clean_evaluation_base_path
)


raw_evaluation_unchanged = bool(
    raw_evaluation_sha256_before
    == raw_evaluation_sha256_after
)

clean_evaluation_base_unchanged = bool(
    clean_evaluation_base_sha256_before
    == clean_evaluation_base_sha256_after
)


if not raw_evaluation_unchanged:
    raise AssertionError(
        "Raw evaluation cohort changed during the full run."
    )


if not clean_evaluation_base_unchanged:
    raise AssertionError(
        "Clean evaluation base changed during the full run."
    )


source_root_after = create_source_root_sha256(
    source_files_payload
)

source_unchanged = bool(
    source_root_before
    == source_root_after
    == EXPECTED_SOURCE_ROOT_SHA256
)


if not source_unchanged:
    raise AssertionError(
        "Project 10 source changed during the full run."
    )


registry_sha256_after = calculate_hash(
    REGISTRY_PATH
)

registry_unchanged = bool(
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:
    raise AssertionError(
        "Completion registry changed during the full run."
    )


atomic_write_csv(
    CHECKPOINT_TABLE_PATH,
    completed_final,
)


invocation_seconds = float(
    time.perf_counter()
    - invocation_started
)


# ------------------------------------------------------------
# 24. EXECUTION REPORT AND STATUS
# ------------------------------------------------------------

execution_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5A_PASS_STATUS,

    "ExpectedConditions":
        EXPECTED_CONDITIONS,

    "InitiallyCompleteConditions":
        initially_complete,

    "NewlyCompletedThisInvocation":
        newly_completed,

    "InvalidOrIncompleteConditionsRemoved":
        invalid_or_incomplete_removed,

    "CompletedConditions":
        len(
            completed_final
        ),

    "RemainingConditions":
        0,

    "MLFits":
        total_model_fits,

    "RankingRows":
        total_ranking_rows,

    "BuildMetricRows":
        total_build_metric_rows,

    "ProjectRunRows":
        total_project_run_rows,

    "TrainingMedianRows":
        total_training_median_rows,

    "ExpectedRawFiles":
        EXPECTED_TOTAL_RAW_FILES,

    "ActualRawFiles":
        actual_raw_files,

    "RawBytes":
        raw_bytes,

    "RawRoot":
        str(
            RAW_ROOT
        ),

    "RawEvaluationUnchanged":
        raw_evaluation_unchanged,

    "CleanEvaluationBaseUnchanged":
        clean_evaluation_base_unchanged,

    "Project10SourceUnchanged":
        source_unchanged,

    "CompletionRegistryModified":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CheckpointTable":
        str(
            CHECKPOINT_TABLE_PATH
        ),

    "CheckpointTableSHA256":
        calculate_hash(
            CHECKPOINT_TABLE_PATH
        ),

    "InvocationSeconds":
        invocation_seconds,

    "InvocationStartedAtUTC":
        invocation_started_at_utc,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    EXECUTION_REPORT_PATH,
    execution_report,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5A_PASS_STATUS,

    "CompletedConditions":
        len(
            completed_final
        ),

    "RemainingConditions":
        0,

    "MLFits":
        total_model_fits,

    "RankingRows":
        total_ranking_rows,

    "BuildMetricRows":
        total_build_metric_rows,

    "ProjectRunRows":
        total_project_run_rows,

    "RawFiles":
        actual_raw_files,

    "RawBytes":
        raw_bytes,

    "RawRoot":
        str(
            RAW_ROOT
        ),

    "ExecutionReport":
        str(
            EXECUTION_REPORT_PATH
        ),

    "ExecutionReportSHA256":
        calculate_hash(
            EXECUTION_REPORT_PATH
        ),

    "CompletionRegistryModified":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP5A_STATUS_PATH,
    status_payload,
)


write_progress(
    status=STEP5A_PASS_STATUS,
    completed_conditions=EXPECTED_CONDITIONS,
    pending_conditions=0,
    initially_complete=initially_complete,
    newly_completed=newly_completed,
    invalid_removed=invalid_or_incomplete_removed,
    last_completed_condition=(
        completed_final.sort_values(
            "ConditionOrder",
            kind="mergesort",
        )[
            "ConditionID"
        ].iloc[-1]
    ),
    invocation_started_at=invocation_started_at_utc,
)


# ------------------------------------------------------------
# 25. FINAL READBACK
# ------------------------------------------------------------

status_readback = read_json_with_retry(
    STEP5A_STATUS_PATH
)

report_readback = read_json_with_retry(
    EXECUTION_REPORT_PATH
)

progress_readback = read_json_with_retry(
    PROGRESS_PATH
)


for name, payload in [
    (
        "Step 5A status",
        status_readback,
    ),
    (
        "Execution report",
        report_readback,
    ),
    (
        "Progress report",
        progress_readback,
    ),
]:
    if payload.get(
        "Status"
    ) != STEP5A_PASS_STATUS:
        raise AssertionError(
            f"{name} readback status differs."
        )


if calculate_hash(
    REGISTRY_PATH
) != registry_sha256_before:
    raise AssertionError(
        "Completion registry changed during final readback."
    )


if create_source_root_sha256(
    source_files_payload
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise AssertionError(
        "Project 10 source changed during final readback."
    )


# ------------------------------------------------------------
# 26. DISPLAY
# ------------------------------------------------------------

print("\nFinal condition checkpoint sample:")

display(
    pd.concat(
        [
            completed_final.head(9),
            completed_final.tail(9),
        ],
        ignore_index=True,
    )
)


print("\nConditions by noise level:")

display(
    completed_final.groupby(
        "NoisePercent",
        as_index=False,
    ).agg(
        Conditions=(
            "ConditionID",
            "count",
        ),

        MLFits=(
            "ModelFits",
            "sum",
        ),

        RankingRows=(
            "RankingRows",
            "sum",
        ),

        BuildMetricRows=(
            "BuildMetricRows",
            "sum",
        ),

        ProjectRunRows=(
            "ProjectRunRows",
            "sum",
        ),

        MeanConditionSeconds=(
            "ConditionSeconds",
            "mean",
        ),
    )
)


# ------------------------------------------------------------
# 27. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 128)
print("=== PROJECT 10 CELL 9 / STEP 5A RESULT ===")
print("=" * 128)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nCondition progress:")

print(
    "Expected conditions:",
    EXPECTED_CONDITIONS,
)

print(
    "Initially complete:",
    initially_complete,
)

print(
    "Newly completed this invocation:",
    newly_completed,
)

print(
    "Completed conditions:",
    len(
        completed_final
    ),
)

print(
    "Remaining conditions:",
    0,
)

print(
    "Invalid/incomplete conditions removed:",
    invalid_or_incomplete_removed,
)


print("\nExperiment totals:")

print(
    "ML fits:",
    total_model_fits,
    "/",
    EXPECTED_ML_FITS,
)

print(
    "Ranking rows:",
    total_ranking_rows,
    "/",
    EXPECTED_TOTAL_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    total_build_metric_rows,
    "/",
    EXPECTED_TOTAL_BUILD_METRIC_ROWS,
)

print(
    "Project-run rows:",
    total_project_run_rows,
    "/",
    EXPECTED_TOTAL_PROJECT_RUN_ROWS,
)

print(
    "Training-median rows:",
    total_training_median_rows,
    "/",
    EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS,
)


print("\nRaw result freeze candidate:")

print(
    "Expected raw files:",
    EXPECTED_TOTAL_RAW_FILES,
)

print(
    "Actual raw files:",
    actual_raw_files,
)

print(
    "Raw bytes:",
    raw_bytes,
)

print(
    "Raw root:",
    RAW_ROOT,
)


print("\nEvaluation immutable:")

print(
    "Raw evaluation cohort unchanged:",
    raw_evaluation_unchanged,
)

print(
    "Clean evaluation base unchanged:",
    clean_evaluation_base_unchanged,
)


print("\nProject isolation:")

print(
    "Project 10 source unchanged:",
    source_unchanged,
)

print(
    "Completion registry unchanged:",
    registry_unchanged,
)

print(
    "Project 9 accessed:",
    False,
)

print(
    "Project 9 write attempted:",
    False,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nCheckpoint outputs:")

print(
    CHECKPOINT_TABLE_PATH
)

print(
    PROGRESS_PATH
)

print(
    EXECUTION_REPORT_PATH
)

print(
    STEP5A_STATUS_PATH
)


print("\nInvocation runtime:")

print(
    round(
        invocation_seconds,
        2,
    ),
    "seconds",
)


print(
    "\nSTATUS:",
    STEP5A_PASS_STATUS,
)

print("=" * 128)

=== PROJECT 10 CELL 9 / STEP 5A: CHECKPOINTED FULL 270-CONDITION EXPERIMENT ===

Scanning existing Project 10 condition checkpoints...
Complete conditions found: 0
Invalid/incomplete conditions removed: 0
Remaining conditions: 270

--------------------------------------------------------------------------------------------------------------------------------
Condition 1/270
Condition ID: noise_00__seed_01
Noise: 0%
Seed: 1
Flipped raw rows: 0
Realised noise: 0.0 %
Noisy model failures: 63
Dependent REC changes: 0
Elapsed seconds: 14.12
Checkpointed conditions: 1/270

--------------------------------------------------------------------------------------------------------------------------------
Condition 2/270
Condition ID: noise_05__seed_01
Noise: 5%
Seed: 1
Flipped raw rows: 1755
Realised noise: 5.077684 %
Noisy model failures: 353
Dependent REC changes: 50293
Elapsed seconds: 39.08
Checkpointed conditions: 2/270

-----------------------------------------------------------------------

,ConditionOrder,ConditionID,NoisePercent,RepetitionSeed,Status,ConditionDirectory,RankingRows,BuildMetricRows,ProjectRunRows,ModelFits,TrainingMedianRows,ConditionSeconds,SuccessMarkerSHA256
0,1,noise_00__seed_01,0,1,PASS_PROJECT_10_CONDITION_COMPLETE,/content/drive/MyDrive/Thesis_Experiment/Resul...,18277,189,7,4,151,14.118890,7aac787993ed3ba61823e09b61f5ceb2d56e49ab958f61...
1,2,noise_05__seed_01,5,1,PASS_PROJECT_10_CONDITION_COMPLETE,/content/drive/MyDrive/Thesis_Experiment/Resul...,18277,189,7,4,151,39.084747,441488121bf7e72acf366a86b2375040e9805f6c714b71...
2,3,noise_10__seed_01,10,1,PASS_PROJECT_10_CONDITION_COMPLETE,/content/drive/MyDrive/Thesis_Experiment/Resul...,18277,189,7,4,151,42.666013,13c9d893ab19c375d2151ba74c27c19206579eda4cc689...
3,4,noise_15__seed_01,15,1,PASS_PROJECT_10_CONDITION_COMPLETE,/content/drive/MyDrive/Thesis_Experiment/Resul...,18277,189,7,4,151,31.290343,5282fd3a5d4112832c4c2db26a2fe7e1b7a4a83643b8f2...
4,5,noise_20__seed_01,20,1,PASS_PROJECT_10_CONDITION_COMPLETE,/content/drive/MyDrive/Thesis_Experiment/Resul...,18277,189,7,4,151,21.123498,e7d2c2b4596416898c7e761fa76a57231b26da616648c1...
5,6,noise_25__seed_01,25,1,PASS_PROJECT_10_CONDITION_COMPLETE,/content/drive/MyDrive/Thesis_Experiment/Resul...,18277,189,7,4,151,26.894078,ba00e45ea28195748af5d56e0bf31c37feba0d46d851b5...
6,7,noise_30__seed_01,30,1,PASS_PROJECT_10_CONDITION_COMPLETE,/content/drive/MyDrive/Thesis_Experiment/Resul...,18277,189,7,4,151,21.692060,570556c94994e338d7b7db33b607a2065728d6acdfd15d...
7,8,noise_40__seed_01,40,1,PASS_PROJECT_10_CONDITION_COMPLETE,/content/drive/MyDrive/Thesis_Experiment/Resul...,18277,189,7,4,151,25.948950,c1684941f9d57d4631218619908b024987a197765f5387...
8,9,noise_50__seed_01,50,1,PASS_PROJECT_10_CONDITION_COMPLETE,/content/drive/MyDrive/Thesis_Experiment/Resul...,18277,189,7,4,151,22.186076,8fc4862429884dde25ce32ff64b424039657d154d0a467...
9,262,noise_00__seed_30,0,30,PASS_PROJECT_10_CONDITION_COMPLETE,/content/drive/MyDrive/Thesis_Experiment/Resul...,18277,189,7,4,151,4.833651,286bcadfe737783af2dbc3c275a6e127dafaf44abc6854...



Conditions by noise level:


,NoisePercent,Conditions,MLFits,RankingRows,BuildMetricRows,ProjectRunRows,MeanConditionSeconds
0,0,30,120,548310,5670,210,6.297178
1,5,30,120,548310,5670,210,22.979194
2,10,30,120,548310,5670,210,23.978783
3,15,30,120,548310,5670,210,23.673843
4,20,30,120,548310,5670,210,23.698062
5,25,30,120,548310,5670,210,23.920634
6,30,30,120,548310,5670,210,24.166456
7,40,30,120,548310,5670,210,24.327920
8,50,30,120,548310,5670,210,24.467730




=== PROJECT 10 CELL 9 / STEP 5A RESULT ===

Project identity:
Project number: 10
Project: spring-cloud@spring-cloud-dataflow
Project slug: spring-cloud__spring-cloud-dataflow

Condition progress:
Expected conditions: 270
Initially complete: 0
Newly completed this invocation: 270
Completed conditions: 270
Remaining conditions: 0
Invalid/incomplete conditions removed: 0

Experiment totals:
ML fits: 1080 / 1080
Ranking rows: 4934790 / 4934790
Build-metric rows: 51030 / 51030
Project-run rows: 1890 / 1890
Training-median rows: 40770 / 40770

Raw result freeze candidate:
Expected raw files: 2160
Actual raw files: 2160
Raw bytes: 68638025
Raw root: /content/drive/MyDrive/Thesis_Experiment/Results/Raw/spring-cloud__spring-cloud-dataflow

Evaluation immutable:
Raw evaluation cohort unchanged: True
Clean evaluation base unchanged: True

Project isolation:
Project 10 source unchanged: True
Completion registry unchanged: True
Project 9 accessed: False
Project 9 write attempted: False
Projects 1

In [13]:
# ==================================================================================================
# PROJECT 10 — CELL 10 / STEP 5B
# FINAL RAW-OUTPUT REVALIDATION AND COMPACT AGGREGATION
#
# PROJECT:
#   spring-cloud@spring-cloud-dataflow
#
# This cell:
# - does NOT rerun conditions, models, baselines, APFD, or APFDc
# - revalidates all 270 completed conditions and 2,160 raw files
# - verifies per-condition file hashes from _SUCCESS.json
# - counts ranking rows from Parquet metadata
# - combines project-run, build-metric, model-fit, condition-audit,
#   and training-median outputs
# - calculates APFD/APFDc means and SAMPLE standard deviations
#   across the 30 seeds for each Noise × Technique combination
# - calculates clean-condition deltas and relative degradation
# - validates Random and QTF-Avg noise invariance
# - validates that LatestFail responds to noise
# - does NOT modify the completion registry
# - does NOT access or modify Project 9
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 132)
print("=== PROJECT 10 CELL 10 / STEP 5B: FINAL RAW-OUTPUT REVALIDATION AND COMPACT AGGREGATION ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT 10 CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)


EXPECTED_STEP5A_STATUS = (
    "PASS_PROJECT_10_FULL_270_CONDITION_RUN_COMPLETE"
)

EXPECTED_CONDITION_STATUS = (
    "PASS_PROJECT_10_CONDITION_COMPLETE"
)

STEP5B_PASS_STATUS = (
    "PASS_PROJECT_10_RAW_RESULTS_REVALIDATED_"
    "AND_COMPACT_AGGREGATES_FROZEN"
)


NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)


ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080

EXPECTED_RANKING_ROWS = 4_934_790
EXPECTED_BUILD_METRIC_ROWS = 51_030
EXPECTED_PROJECT_RUN_ROWS = 1_890
EXPECTED_TRAINING_MEDIAN_ROWS = 40_770
EXPECTED_CONDITION_AUDIT_ROWS = 270

EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 68_638_025

EXPECTED_ACTIVE_PREDICTORS = 151
EXPECTED_EVALUATION_ROWS = 2_611
EXPECTED_EVALUATION_FAILURES = 213
EXPECTED_SCORED_EVALUATION_BUILDS = 27

EXPECTED_RANKING_ROWS_PER_CONDITION = 18_277
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = 189
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = 7
EXPECTED_MODEL_FITS_PER_CONDITION = 4
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = 151
EXPECTED_FILES_PER_CONDITION = 8

EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS = 63
EXPECTED_SEED_LEVEL_DELTA_ROWS = 1_890
EXPECTED_NOISE_DELTA_SUMMARY_ROWS = 63


REQUIRED_CONDITION_FILES = [
    "rankings.parquet",
    "build_metrics.csv",
    "project_run.csv",
    "model_fits.csv",
    "condition_audit.csv",
    "training_medians.csv",
    "condition_summary.json",
    "_SUCCESS.json",
]


# --------------------------------------------------------------------------------------------------
# 2. PROJECT 10 PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)


RAW_ROOT = (
    RESULTS_ROOT
    / "Raw"
    / PROJECT_SLUG
)

PROJECT_AGGREGATED_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

FULL_RUN_CONTROL_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_full_run_control"
)

STEP5A_STATUS_PATH = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_step5a_status.json"
)

FULL_RUN_CHECKPOINT_TABLE_PATH = (
    FULL_RUN_CONTROL_ROOT
    / f"{PROJECT_SHORT_NAME}_full_run_checkpoint.csv"
)

FULL_RUN_PROGRESS_PATH = (
    FULL_RUN_CONTROL_ROOT
    / f"{PROJECT_SHORT_NAME}_full_run_progress.json"
)

FULL_RUN_EXECUTION_REPORT_PATH = (
    FULL_RUN_CONTROL_ROOT
    / f"{PROJECT_SHORT_NAME}_full_run_execution_report.json"
)


STEP5B_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_step5b_final_audit"
)

RAW_FILE_MANIFEST_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_raw_file_manifest.csv"
)

CONDITION_INVENTORY_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_condition_inventory.csv"
)

PROJECT_RUN_ALL_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_project_run_all.csv"
)

BUILD_METRICS_ALL_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_build_metrics_all.parquet"
)

MODEL_FITS_ALL_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_model_fits_all.csv"
)

CONDITION_AUDIT_ALL_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_condition_audit_all.csv"
)

TRAINING_MEDIANS_ALL_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_training_medians_all.parquet"
)

NOISE_TECHNIQUE_SUMMARY_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_noise_technique_summary.csv"
)

SEED_LEVEL_NOISE_DELTAS_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_seed_level_noise_deltas.csv"
)

NOISE_DELTA_SUMMARY_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_noise_delta_summary.csv"
)

CLEAN_TECHNIQUE_SUMMARY_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_clean_technique_summary.csv"
)

BASELINE_INVARIANCE_AUDIT_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_baseline_invariance_audit.csv"
)

STEP5B_VALIDATION_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_step5b_validation.csv"
)

STEP5B_REPORT_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT_NAME}_step5b_report.json"
)

STEP5B_STATUS_PATH = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_step5b_status.json"
)

STEP5B_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_10_step5b_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.stem}.tmp_{os.getpid()}.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="snappy",
    )

    os.replace(
        temporary_path,
        path,
    )


def canonical_root_hash(
    manifest,
):
    required_columns = [
        "RelativePath",
        "SizeBytes",
        "SHA256",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in manifest.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Cannot calculate root hash. Missing columns:\n"
            + "\n".join(missing_columns)
        )

    digest = hashlib.sha256()

    ordered = manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def expected_condition_order(
    noise_percent,
    repetition_seed,
):
    return (
        (
            int(repetition_seed)
            - 1
        )
        * len(NOISE_LEVELS)
        + NOISE_LEVELS.index(
            int(noise_percent)
        )
        + 1
    )


def add_check(
    records,
    check,
    expected,
    actual,
    passed,
):
    records.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def normalise_condition_frame(
    dataframe,
    marker,
):
    dataframe = dataframe.copy()

    condition_id = str(
        marker["ConditionID"]
    )

    noise_percent = int(
        marker["NoisePercent"]
    )

    repetition_seed = int(
        marker["RepetitionSeed"]
    )

    condition_order = int(
        marker["ConditionOrder"]
    )

    if "ConditionID" in dataframe.columns:
        observed = set(
            dataframe[
                "ConditionID"
            ]
            .dropna()
            .astype(str)
            .unique()
        )

        if observed and observed != {
            condition_id
        }:
            raise RuntimeError(
                f"{condition_id}: ConditionID differs inside a condition file."
            )

    if "NoisePercent" in dataframe.columns:
        observed = set(
            pd.to_numeric(
                dataframe[
                    "NoisePercent"
                ],
                errors="raise",
            )
            .astype(int)
            .unique()
        )

        if observed and observed != {
            noise_percent
        }:
            raise RuntimeError(
                f"{condition_id}: NoisePercent differs inside a condition file."
            )

    if "RepetitionSeed" in dataframe.columns:
        observed = set(
            pd.to_numeric(
                dataframe[
                    "RepetitionSeed"
                ],
                errors="raise",
            )
            .astype(int)
            .unique()
        )

        if observed and observed != {
            repetition_seed
        }:
            raise RuntimeError(
                f"{condition_id}: RepetitionSeed differs inside a condition file."
            )

    dataframe[
        "ProjectNumber"
    ] = PROJECT_NUMBER

    dataframe[
        "Project"
    ] = PROJECT_NAME

    dataframe[
        "ProjectSlug"
    ] = PROJECT_SLUG

    dataframe[
        "ConditionOrder"
    ] = condition_order

    dataframe[
        "ConditionID"
    ] = condition_id

    dataframe[
        "NoisePercent"
    ] = noise_percent

    dataframe[
        "RepetitionSeed"
    ] = repetition_seed

    return dataframe


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_files = [
    REGISTRY_PATH,
    STEP5A_STATUS_PATH,
    FULL_RUN_CHECKPOINT_TABLE_PATH,
    FULL_RUN_PROGRESS_PATH,
    FULL_RUN_EXECUTION_REPORT_PATH,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Required Project 10 Step 5A files are missing:\n"
        + "\n".join(missing_files)
    )


if not RAW_ROOT.is_dir():
    raise FileNotFoundError(
        "Project 10 raw-result root is missing:\n"
        f"{RAW_ROOT}"
    )


if PROJECT_SLUG not in str(RAW_ROOT):
    raise RuntimeError(
        "The raw-result path is not isolated to Project 10."
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE PROJECT 10 STEP 5A
# --------------------------------------------------------------------------------------------------

step5a_status = load_json(
    STEP5A_STATUS_PATH
)

full_run_progress = load_json(
    FULL_RUN_PROGRESS_PATH
)

full_run_report = load_json(
    FULL_RUN_EXECUTION_REPORT_PATH
)

full_run_checkpoint_table = pd.read_csv(
    FULL_RUN_CHECKPOINT_TABLE_PATH,
    low_memory=False,
)


step5a_status_value = step5a_status.get(
    "Status"
)

progress_status_value = full_run_progress.get(
    "Status"
)

report_status_value = full_run_report.get(
    "Status"
)


for label, actual_status in [
    (
        "Step 5A status",
        step5a_status_value,
    ),
    (
        "Full-run progress",
        progress_status_value,
    ),
    (
        "Full-run execution report",
        report_status_value,
    ),
]:
    if actual_status != EXPECTED_STEP5A_STATUS:
        raise RuntimeError(
            f"{label} differs.\n"
            f"Expected: {EXPECTED_STEP5A_STATUS}\n"
            f"Actual:   {actual_status}"
        )


if len(
    full_run_checkpoint_table
) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "Project 10 full-run checkpoint-table row count differs."
    )


if int(
    step5a_status.get(
        "CompletedConditions",
        -1,
    )
) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "Step 5A completed-condition count differs."
    )


if int(
    step5a_status.get(
        "MLFits",
        -1,
    )
) != EXPECTED_ML_FITS:
    raise RuntimeError(
        "Step 5A ML-fit count differs."
    )


if int(
    step5a_status.get(
        "RawFiles",
        -1,
    )
) != EXPECTED_RAW_FILES:
    raise RuntimeError(
        "Step 5A raw-file count differs."
    )


if int(
    step5a_status.get(
        "RawBytes",
        -1,
    )
) != EXPECTED_RAW_BYTES:
    raise RuntimeError(
        "Step 5A raw-byte count differs."
    )


# --------------------------------------------------------------------------------------------------
# 6. REGISTRY SNAPSHOT — READ ONLY
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

registry_before = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_numbers = pd.to_numeric(
    registry_before[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


registry_project9_rows_before = int(
    registry_project_numbers.eq(9).sum()
)

registry_project10_rows_before = int(
    registry_project_numbers.eq(10).sum()
)


if registry_project10_rows_before != 0:
    raise RuntimeError(
        "Project 10 is already present in the completion registry."
    )


# Project 9 is still intentionally registry-pending.
if registry_project9_rows_before != 0:
    raise RuntimeError(
        "Project 9 is unexpectedly present in the registry. "
        "Serial registration has not yet been performed."
    )


# --------------------------------------------------------------------------------------------------
# 7. DISCOVER AND REVALIDATE ALL 270 CONDITIONS
# --------------------------------------------------------------------------------------------------

success_markers = sorted(
    RAW_ROOT.rglob("_SUCCESS.json"),
    key=lambda path:
        path.relative_to(
            RAW_ROOT
        ).as_posix(),
)


if len(
    success_markers
) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "Project 10 success-marker count differs.\n"
        f"Expected: {EXPECTED_CONDITIONS}\n"
        f"Actual:   {len(success_markers)}"
    )


condition_inventory_records = []
raw_file_records = []

embedded_hash_checks = 0
embedded_hash_mismatches = 0

ranking_rows_total = 0

raw_hash_started = time.perf_counter()


for marker_path in success_markers:
    condition_directory = marker_path.parent

    marker = load_json(
        marker_path
    )

    condition_id = str(
        marker.get(
            "ConditionID",
            "",
        )
    )

    noise_percent = int(
        marker.get(
            "NoisePercent",
            -1,
        )
    )

    repetition_seed = int(
        marker.get(
            "RepetitionSeed",
            -1,
        )
    )

    condition_order = int(
        marker.get(
            "ConditionOrder",
            -1,
        )
    )


    if marker.get(
        "Status"
    ) != EXPECTED_CONDITION_STATUS:
        raise RuntimeError(
            f"{condition_id}: condition status differs."
        )


    expected_order = expected_condition_order(
        noise_percent,
        repetition_seed,
    )


    if condition_order != expected_order:
        raise RuntimeError(
            f"{condition_id}: condition order differs.\n"
            f"Expected: {expected_order}\n"
            f"Actual:   {condition_order}"
        )


    expected_condition_id = (
        f"noise_{noise_percent:02d}"
        f"__seed_{repetition_seed:02d}"
    )


    if condition_id != expected_condition_id:
        raise RuntimeError(
            "Condition identifier differs.\n"
            f"Expected: {expected_condition_id}\n"
            f"Actual:   {condition_id}"
        )


    actual_file_names = sorted([
        path.name
        for path in condition_directory.iterdir()
        if path.is_file()
    ])


    missing_condition_files = sorted(
        set(REQUIRED_CONDITION_FILES)
        - set(actual_file_names)
    )

    unexpected_condition_files = sorted(
        set(actual_file_names)
        - set(REQUIRED_CONDITION_FILES)
    )


    if missing_condition_files:
        raise RuntimeError(
            f"{condition_id}: missing condition files:\n"
            + "\n".join(missing_condition_files)
        )


    if unexpected_condition_files:
        raise RuntimeError(
            f"{condition_id}: unexpected condition files:\n"
            + "\n".join(unexpected_condition_files)
        )


    file_hashes = marker.get(
        "FileSHA256",
        {},
    )


    if set(
        file_hashes.keys()
    ) != set(
        REQUIRED_CONDITION_FILES[:-1]
    ):
        raise RuntimeError(
            f"{condition_id}: _SUCCESS.json contains an unexpected hash set."
        )


    condition_bytes = 0
    ranking_rows = None


    for filename in REQUIRED_CONDITION_FILES:
        file_path = (
            condition_directory
            / filename
        )

        relative_path = file_path.relative_to(
            RAW_ROOT
        ).as_posix()

        file_size = int(
            file_path.stat().st_size
        )

        file_sha256 = sha256_file(
            file_path
        )

        condition_bytes += file_size


        embedded_hash_available = (
            filename in file_hashes
        )

        embedded_hash_match = None


        if embedded_hash_available:
            embedded_hash_checks += 1

            embedded_hash_match = bool(
                str(
                    file_hashes[
                        filename
                    ]
                ).lower()
                == file_sha256
            )

            if not embedded_hash_match:
                embedded_hash_mismatches += 1


        parquet_rows = None
        parquet_columns = None


        if filename == "rankings.parquet":
            parquet_file = pq.ParquetFile(
                file_path
            )

            parquet_rows = int(
                parquet_file.metadata.num_rows
            )

            parquet_columns = int(
                parquet_file.metadata.num_columns
            )

            ranking_rows = parquet_rows
            ranking_rows_total += parquet_rows


            if (
                parquet_rows
                != EXPECTED_RANKING_ROWS_PER_CONDITION
            ):
                raise RuntimeError(
                    f"{condition_id}: ranking-row count differs."
                )


        raw_file_records.append({
            "ProjectNumber":
                PROJECT_NUMBER,

            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "ConditionOrder":
                condition_order,

            "ConditionID":
                condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "RelativePath":
                relative_path,

            "FileName":
                filename,

            "SizeBytes":
                file_size,

            "SHA256":
                file_sha256,

            "EmbeddedHashAvailable":
                embedded_hash_available,

            "EmbeddedHashMatch":
                embedded_hash_match,

            "ParquetRows":
                parquet_rows,

            "ParquetColumns":
                parquet_columns,
        })


    condition_summary = load_json(
        condition_directory
        / "condition_summary.json"
    )


    expected_summary_counts = {
        "RankingRows":
            EXPECTED_RANKING_ROWS_PER_CONDITION,

        "BuildMetricRows":
            EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,

        "ProjectRunRows":
            EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,

        "ModelFits":
            EXPECTED_MODEL_FITS_PER_CONDITION,

        "TrainingMedianRows":
            EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }


    for key, expected_value in (
        expected_summary_counts.items()
    ):
        actual_value = int(
            condition_summary.get(
                key,
                -1,
            )
        )

        if actual_value != expected_value:
            raise RuntimeError(
                f"{condition_id}: condition summary {key} differs."
            )


    if int(
        condition_summary.get(
            "FailedValidationChecks",
            -1,
        )
    ) != 0:
        raise RuntimeError(
            f"{condition_id}: condition summary reports failed checks."
        )


    condition_inventory_records.append({
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionOrder":
            condition_order,

        "ConditionID":
            condition_id,

        "NoisePercent":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "ConditionDirectory":
            str(
                condition_directory
            ),

        "Status":
            marker.get(
                "Status"
            ),

        "Files":
            len(
                actual_file_names
            ),

        "ConditionBytes":
            condition_bytes,

        "RankingRows":
            ranking_rows,

        "BuildMetricRows":
            int(
                marker[
                    "BuildMetricRows"
                ]
            ),

        "ProjectRunRows":
            int(
                marker[
                    "ProjectRunRows"
                ]
            ),

        "ModelFits":
            int(
                marker[
                    "ModelFits"
                ]
            ),

        "TrainingMedianRows":
            int(
                marker[
                    "TrainingMedianRows"
                ]
            ),

        "ConditionSeconds":
            float(
                marker[
                    "ConditionSeconds"
                ]
            ),

        "SuccessMarkerSHA256":
            sha256_file(
                marker_path
            ),
    })


raw_hash_seconds = float(
    time.perf_counter()
    - raw_hash_started
)


raw_file_manifest = (
    pd.DataFrame(
        raw_file_records
    )
    .sort_values(
        [
            "ConditionOrder",
            "FileName",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


condition_inventory = (
    pd.DataFrame(
        condition_inventory_records
    )
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 8. RAW-ROOT INVENTORY AND ROOT HASH
# --------------------------------------------------------------------------------------------------

raw_files_on_disk = sorted(
    [
        path
        for path in RAW_ROOT.rglob("*")
        if path.is_file()
    ],
    key=lambda path:
        path.relative_to(
            RAW_ROOT
        ).as_posix(),
)


actual_raw_file_count = len(
    raw_files_on_disk
)

actual_raw_bytes = int(
    sum(
        path.stat().st_size
        for path in raw_files_on_disk
    )
)


manifest_relative_paths = set(
    raw_file_manifest[
        "RelativePath"
    ].astype(str)
)

disk_relative_paths = {
    path.relative_to(
        RAW_ROOT
    ).as_posix()
    for path in raw_files_on_disk
}


manifest_missing_files = sorted(
    manifest_relative_paths
    - disk_relative_paths
)

manifest_unexpected_files = sorted(
    disk_relative_paths
    - manifest_relative_paths
)


raw_root_manifest = (
    raw_file_manifest[
        [
            "RelativePath",
            "SizeBytes",
            "SHA256",
        ]
    ]
    .copy()
)


raw_root_sha256 = canonical_root_hash(
    raw_root_manifest
)


# --------------------------------------------------------------------------------------------------
# 9. CONDITION-COORDINATE VALIDATION
# --------------------------------------------------------------------------------------------------

expected_coordinate_set = {
    (
        noise_percent,
        repetition_seed,
    )
    for repetition_seed in REPETITION_SEEDS
    for noise_percent in NOISE_LEVELS
}


actual_coordinate_set = set(
    zip(
        condition_inventory[
            "NoisePercent"
        ].astype(int),

        condition_inventory[
            "RepetitionSeed"
        ].astype(int),
    )
)


duplicate_condition_ids = int(
    condition_inventory[
        "ConditionID"
    ].duplicated(
        keep=False
    ).sum()
)


duplicate_coordinates = int(
    condition_inventory.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


files_per_condition_violations = int(
    condition_inventory[
        "Files"
    ].ne(
        EXPECTED_FILES_PER_CONDITION
    ).sum()
)


ranking_rows_per_condition_violations = int(
    condition_inventory[
        "RankingRows"
    ].ne(
        EXPECTED_RANKING_ROWS_PER_CONDITION
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 10. LOAD COMPACT CONDITION OUTPUTS
# --------------------------------------------------------------------------------------------------

def load_all_condition_csvs(
    filename,
):
    frames = []

    for row in condition_inventory.itertuples(
        index=False
    ):
        condition_directory = Path(
            row.ConditionDirectory
        )

        marker = {
            "ConditionOrder":
                int(
                    row.ConditionOrder
                ),

            "ConditionID":
                str(
                    row.ConditionID
                ),

            "NoisePercent":
                int(
                    row.NoisePercent
                ),

            "RepetitionSeed":
                int(
                    row.RepetitionSeed
                ),
        }

        frame = pd.read_csv(
            condition_directory
            / filename,
            low_memory=False,
        )

        frame = normalise_condition_frame(
            frame,
            marker,
        )

        frames.append(
            frame
        )

    return pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )


aggregation_started = time.perf_counter()


project_run_all = load_all_condition_csvs(
    "project_run.csv"
)

build_metrics_all = load_all_condition_csvs(
    "build_metrics.csv"
)

model_fits_all = load_all_condition_csvs(
    "model_fits.csv"
)

condition_audit_all = load_all_condition_csvs(
    "condition_audit.csv"
)

training_medians_all = load_all_condition_csvs(
    "training_medians.csv"
)


aggregation_seconds = float(
    time.perf_counter()
    - aggregation_started
)


# --------------------------------------------------------------------------------------------------
# 11. CANONICAL NUMERIC TYPES
# --------------------------------------------------------------------------------------------------

for frame in [
    project_run_all,
    build_metrics_all,
    model_fits_all,
    condition_audit_all,
    training_medians_all,
]:
    frame[
        "ConditionOrder"
    ] = pd.to_numeric(
        frame[
            "ConditionOrder"
        ],
        errors="raise",
    ).astype(int)

    frame[
        "NoisePercent"
    ] = pd.to_numeric(
        frame[
            "NoisePercent"
        ],
        errors="raise",
    ).astype(int)

    frame[
        "RepetitionSeed"
    ] = pd.to_numeric(
        frame[
            "RepetitionSeed"
        ],
        errors="raise",
    ).astype(int)


project_run_all[
    "MeanAPFD"
] = pd.to_numeric(
    project_run_all[
        "MeanAPFD"
    ],
    errors="raise",
).astype(float)

project_run_all[
    "MeanAPFDc"
] = pd.to_numeric(
    project_run_all[
        "MeanAPFDc"
    ],
    errors="raise",
).astype(float)

project_run_all[
    "EvaluatedBuilds"
] = pd.to_numeric(
    project_run_all[
        "EvaluatedBuilds"
    ],
    errors="raise",
).astype(int)

project_run_all[
    "EvaluationFailures"
] = pd.to_numeric(
    project_run_all[
        "EvaluationFailures"
    ],
    errors="raise",
).astype(int)


build_metrics_all[
    "APFD"
] = pd.to_numeric(
    build_metrics_all[
        "APFD"
    ],
    errors="raise",
).astype(float)

build_metrics_all[
    "APFDc"
] = pd.to_numeric(
    build_metrics_all[
        "APFDc"
    ],
    errors="raise",
).astype(float)


training_medians_all[
    "TrainingMedian"
] = pd.to_numeric(
    training_medians_all[
        "TrainingMedian"
    ],
    errors="raise",
).astype(float)


# --------------------------------------------------------------------------------------------------
# 12. STRUCTURAL AND METRIC VALIDATION
# --------------------------------------------------------------------------------------------------

project_run_duplicate_rows = int(
    project_run_all.duplicated(
        subset=[
            "ConditionID",
            "Technique",
        ],
        keep=False,
    ).sum()
)


build_metric_duplicate_rows = int(
    build_metrics_all.duplicated(
        subset=[
            "ConditionID",
            "Technique",
            "BuildKey",
        ],
        keep=False,
    ).sum()
)


model_fit_duplicate_rows = int(
    model_fits_all.duplicated(
        subset=[
            "ConditionID",
            "Technique",
        ],
        keep=False,
    ).sum()
)


condition_audit_duplicate_rows = int(
    condition_audit_all[
        "ConditionID"
    ].duplicated(
        keep=False
    ).sum()
)


training_median_duplicate_rows = int(
    training_medians_all.duplicated(
        subset=[
            "ConditionID",
            "Feature",
        ],
        keep=False,
    ).sum()
)


project_run_rows_per_condition_violations = int(
    project_run_all.groupby(
        "ConditionID"
    ).size().ne(
        EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).sum()
)


build_metric_rows_per_condition_violations = int(
    build_metrics_all.groupby(
        "ConditionID"
    ).size().ne(
        EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).sum()
)


model_fit_rows_per_condition_violations = int(
    model_fits_all.groupby(
        "ConditionID"
    ).size().ne(
        EXPECTED_MODEL_FITS_PER_CONDITION
    ).sum()
)


training_median_rows_per_condition_violations = int(
    training_medians_all.groupby(
        "ConditionID"
    ).size().ne(
        EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION
    ).sum()
)


scored_build_count_violations = int(
    build_metrics_all.groupby(
        [
            "ConditionID",
            "Technique",
        ]
    )[
        "BuildKey"
    ].nunique().ne(
        EXPECTED_SCORED_EVALUATION_BUILDS
    ).sum()
)


evaluated_build_violations = int(
    project_run_all[
        "EvaluatedBuilds"
    ].ne(
        EXPECTED_SCORED_EVALUATION_BUILDS
    ).sum()
)


evaluation_failure_violations = int(
    project_run_all[
        "EvaluationFailures"
    ].ne(
        EXPECTED_EVALUATION_FAILURES
    ).sum()
)


model_fit_failures = int(
    model_fits_all[
        "FitStatus"
    ].astype(str).ne(
        "SUCCESS"
    ).sum()
)


project_run_nonfinite_metrics = int(
    (
        ~np.isfinite(
            project_run_all[
                [
                    "MeanAPFD",
                    "MeanAPFDc",
                ]
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


project_run_out_of_range_metrics = int(
    (
        ~project_run_all[
            "MeanAPFD"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
    +
    (
        ~project_run_all[
            "MeanAPFDc"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
)


build_metric_nonfinite_values = int(
    (
        ~np.isfinite(
            build_metrics_all[
                [
                    "APFD",
                    "APFDc",
                ]
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


build_metric_out_of_range_values = int(
    (
        ~build_metrics_all[
            "APFD"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
    +
    (
        ~build_metrics_all[
            "APFDc"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
)


training_median_nonfinite_values = int(
    (
        ~np.isfinite(
            training_medians_all[
                "TrainingMedian"
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


project_run_technique_set = set(
    project_run_all[
        "Technique"
    ].astype(str).unique()
)

model_fit_technique_set = set(
    model_fits_all[
        "Technique"
    ].astype(str).unique()
)


# --------------------------------------------------------------------------------------------------
# 13. NOISE-PROPAGATION VALIDATION
# --------------------------------------------------------------------------------------------------

audit_noise = pd.to_numeric(
    condition_audit_all[
        "NoisePercent"
    ],
    errors="raise",
).astype(int)

audit_raw_flips = pd.to_numeric(
    condition_audit_all[
        "ActualRawFlips"
    ],
    errors="raise",
).astype(int)

audit_model_changes = pd.to_numeric(
    condition_audit_all[
        "ActualModelLabelChanges"
    ],
    errors="raise",
).astype(int)

audit_dependent_rec_changes = pd.to_numeric(
    condition_audit_all[
        "DependentRECChangedValues"
    ],
    errors="raise",
).astype(int)

audit_independent_rec_changes = pd.to_numeric(
    condition_audit_all[
        "IndependentRECChangedValues"
    ],
    errors="raise",
).astype(int)


zero_noise_mask = audit_noise.eq(0)
positive_noise_mask = audit_noise.gt(0)


zero_noise_conditions = int(
    zero_noise_mask.sum()
)

zero_noise_raw_flip_violations = int(
    audit_raw_flips[
        zero_noise_mask
    ].ne(0).sum()
)

zero_noise_model_change_violations = int(
    audit_model_changes[
        zero_noise_mask
    ].ne(0).sum()
)

zero_noise_dependent_rec_violations = int(
    audit_dependent_rec_changes[
        zero_noise_mask
    ].ne(0).sum()
)

positive_noise_without_raw_changes = int(
    audit_raw_flips[
        positive_noise_mask
    ].le(0).sum()
)

positive_noise_without_model_changes = int(
    audit_model_changes[
        positive_noise_mask
    ].le(0).sum()
)

positive_noise_without_dependent_rec_changes = int(
    audit_dependent_rec_changes[
        positive_noise_mask
    ].le(0).sum()
)

independent_rec_change_violations = int(
    audit_independent_rec_changes.ne(0).sum()
)


# --------------------------------------------------------------------------------------------------
# 14. BASELINE INVARIANCE AND RESPONSE
# --------------------------------------------------------------------------------------------------

baseline_invariance_records = []


for technique in [
    "Random",
    "QTF-Avg",
]:
    technique_rows = (
        project_run_all[
            project_run_all[
                "Technique"
            ].eq(
                technique
            )
        ]
        .copy()
    )

    violating_seeds = 0


    for _, seed_rows in technique_rows.groupby(
        "RepetitionSeed",
        sort=True,
    ):
        if (
            seed_rows[
                "MeanAPFD"
            ].nunique(
                dropna=False
            ) != 1
            or seed_rows[
                "MeanAPFDc"
            ].nunique(
                dropna=False
            ) != 1
        ):
            violating_seeds += 1


    baseline_invariance_records.append({
        "Technique":
            technique,

        "Expected":
            (
                "APFD and APFDc constant across all "
                "noise levels within each seed"
            ),

        "SeedsChecked":
            int(
                technique_rows[
                    "RepetitionSeed"
                ].nunique()
            ),

        "ViolatingSeeds":
            violating_seeds,

        "NoiseResponseSeeds":
            0,

        "Pass":
            violating_seeds == 0,
    })


latest_fail_rows = (
    project_run_all[
        project_run_all[
            "Technique"
        ].eq(
            "LatestFail"
        )
    ]
    .copy()
)


latest_fail_response_seeds = 0


for _, seed_rows in latest_fail_rows.groupby(
    "RepetitionSeed",
    sort=True,
):
    if (
        seed_rows[
            "MeanAPFD"
        ].nunique(
            dropna=False
        ) > 1
        or seed_rows[
            "MeanAPFDc"
        ].nunique(
            dropna=False
        ) > 1
    ):
        latest_fail_response_seeds += 1


baseline_invariance_records.append({
    "Technique":
        "LatestFail",

    "Expected":
        (
            "At least one seed changes across noise levels"
        ),

    "SeedsChecked":
        int(
            latest_fail_rows[
                "RepetitionSeed"
            ].nunique()
        ),

    "ViolatingSeeds":
        (
            0
            if latest_fail_response_seeds > 0
            else len(REPETITION_SEEDS)
        ),

    "NoiseResponseSeeds":
        latest_fail_response_seeds,

    "Pass":
        latest_fail_response_seeds > 0,
})


baseline_invariance_audit = pd.DataFrame(
    baseline_invariance_records
)


baseline_invariance_failures = int(
    (
        ~baseline_invariance_audit[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 15. APFD/APFDC SUMMARY WITH STANDARD DEVIATION ACROSS 30 SEEDS
# --------------------------------------------------------------------------------------------------

noise_technique_summary = (
    project_run_all.groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        Runs=(
            "RepetitionSeed",
            "count",
        ),

        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanAPFD=(
            "MeanAPFD",
            "mean",
        ),

        StdAPFD=(
            "MeanAPFD",
            "std",
        ),

        MedianAPFD=(
            "MeanAPFD",
            "median",
        ),

        MinAPFD=(
            "MeanAPFD",
            "min",
        ),

        MaxAPFD=(
            "MeanAPFD",
            "max",
        ),

        MeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),

        StdAPFDc=(
            "MeanAPFDc",
            "std",
        ),

        MedianAPFDc=(
            "MeanAPFDc",
            "median",
        ),

        MinAPFDc=(
            "MeanAPFDc",
            "min",
        ),

        MaxAPFDc=(
            "MeanAPFDc",
            "max",
        ),
    )
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


summary_run_count_violations = int(
    noise_technique_summary[
        "Runs"
    ].ne(30).sum()
)

summary_seed_count_violations = int(
    noise_technique_summary[
        "Seeds"
    ].ne(30).sum()
)

summary_nonfinite_values = int(
    (
        ~np.isfinite(
            noise_technique_summary[
                [
                    "MeanAPFD",
                    "StdAPFD",
                    "MedianAPFD",
                    "MinAPFD",
                    "MaxAPFD",
                    "MeanAPFDc",
                    "StdAPFDc",
                    "MedianAPFDc",
                    "MinAPFDc",
                    "MaxAPFDc",
                ]
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 16. CLEAN-REFERENCE DELTAS AND RELATIVE DEGRADATION
# --------------------------------------------------------------------------------------------------

clean_reference = (
    project_run_all[
        project_run_all[
            "NoisePercent"
        ].eq(0)
    ][
        [
            "RepetitionSeed",
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .rename(
        columns={
            "MeanAPFD":
                "CleanMeanAPFD",

            "MeanAPFDc":
                "CleanMeanAPFDc",
        }
    )
)


if len(
    clean_reference
) != (
    len(REPETITION_SEEDS)
    * len(ALL_TECHNIQUES)
):
    raise RuntimeError(
        "Clean-reference row count differs."
    )


seed_level_noise_deltas = (
    project_run_all.merge(
        clean_reference,
        on=[
            "RepetitionSeed",
            "Technique",
        ],
        how="left",
        validate="many_to_one",
    )
)


seed_level_noise_deltas[
    "APFDDeltaFromClean"
] = (
    seed_level_noise_deltas[
        "MeanAPFD"
    ]
    - seed_level_noise_deltas[
        "CleanMeanAPFD"
    ]
)


seed_level_noise_deltas[
    "APFDcDeltaFromClean"
] = (
    seed_level_noise_deltas[
        "MeanAPFDc"
    ]
    - seed_level_noise_deltas[
        "CleanMeanAPFDc"
    ]
)


seed_level_noise_deltas[
    "APFDRelativeDegradation"
] = np.where(
    seed_level_noise_deltas[
        "CleanMeanAPFD"
    ].ne(0),

    (
        seed_level_noise_deltas[
            "CleanMeanAPFD"
        ]
        - seed_level_noise_deltas[
            "MeanAPFD"
        ]
    )
    / seed_level_noise_deltas[
        "CleanMeanAPFD"
    ],

    np.nan,
)


seed_level_noise_deltas[
    "APFDcRelativeDegradation"
] = np.where(
    seed_level_noise_deltas[
        "CleanMeanAPFDc"
    ].ne(0),

    (
        seed_level_noise_deltas[
            "CleanMeanAPFDc"
        ]
        - seed_level_noise_deltas[
            "MeanAPFDc"
        ]
    )
    / seed_level_noise_deltas[
        "CleanMeanAPFDc"
    ],

    np.nan,
)


clean_delta_rows = seed_level_noise_deltas[
    seed_level_noise_deltas[
        "NoisePercent"
    ].eq(0)
]


clean_delta_nonzero_values = int(
    (
        ~np.isclose(
            clean_delta_rows[
                [
                    "APFDDeltaFromClean",
                    "APFDcDeltaFromClean",
                    "APFDRelativeDegradation",
                    "APFDcRelativeDegradation",
                ]
            ].to_numpy(
                dtype=float
            ),
            0.0,
            rtol=0,
            atol=1e-14,
            equal_nan=False,
        )
    ).sum()
)


noise_delta_summary = (
    seed_level_noise_deltas.groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanAPFDDeltaFromClean=(
            "APFDDeltaFromClean",
            "mean",
        ),

        StdAPFDDeltaFromClean=(
            "APFDDeltaFromClean",
            "std",
        ),

        MedianAPFDDeltaFromClean=(
            "APFDDeltaFromClean",
            "median",
        ),

        MeanAPFDcDeltaFromClean=(
            "APFDcDeltaFromClean",
            "mean",
        ),

        StdAPFDcDeltaFromClean=(
            "APFDcDeltaFromClean",
            "std",
        ),

        MedianAPFDcDeltaFromClean=(
            "APFDcDeltaFromClean",
            "median",
        ),

        MeanAPFDRelativeDegradation=(
            "APFDRelativeDegradation",
            "mean",
        ),

        StdAPFDRelativeDegradation=(
            "APFDRelativeDegradation",
            "std",
        ),

        MedianAPFDRelativeDegradation=(
            "APFDRelativeDegradation",
            "median",
        ),

        MeanAPFDcRelativeDegradation=(
            "APFDcRelativeDegradation",
            "mean",
        ),

        StdAPFDcRelativeDegradation=(
            "APFDcRelativeDegradation",
            "std",
        ),

        MedianAPFDcRelativeDegradation=(
            "APFDcRelativeDegradation",
            "median",
        ),
    )
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


clean_technique_summary = (
    noise_technique_summary[
        noise_technique_summary[
            "NoisePercent"
        ].eq(0)
    ]
    .sort_values(
        "MeanAPFDc",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 17. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 5A status",
    EXPECTED_STEP5A_STATUS,
    step5a_status_value,
    step5a_status_value
    == EXPECTED_STEP5A_STATUS,
)

add_check(
    validation_records,
    "Full-run progress status",
    EXPECTED_STEP5A_STATUS,
    progress_status_value,
    progress_status_value
    == EXPECTED_STEP5A_STATUS,
)

add_check(
    validation_records,
    "Full-run report status",
    EXPECTED_STEP5A_STATUS,
    report_status_value,
    report_status_value
    == EXPECTED_STEP5A_STATUS,
)

add_check(
    validation_records,
    "Condition markers",
    EXPECTED_CONDITIONS,
    len(success_markers),
    len(success_markers)
    == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Condition inventory rows",
    EXPECTED_CONDITIONS,
    len(condition_inventory),
    len(condition_inventory)
    == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Condition coordinates",
    EXPECTED_CONDITIONS,
    len(actual_coordinate_set),
    actual_coordinate_set
    == expected_coordinate_set,
)

add_check(
    validation_records,
    "Duplicate condition IDs",
    0,
    duplicate_condition_ids,
    duplicate_condition_ids == 0,
)

add_check(
    validation_records,
    "Duplicate condition coordinates",
    0,
    duplicate_coordinates,
    duplicate_coordinates == 0,
)

add_check(
    validation_records,
    "Files-per-condition violations",
    0,
    files_per_condition_violations,
    files_per_condition_violations == 0,
)

add_check(
    validation_records,
    "Raw files",
    EXPECTED_RAW_FILES,
    actual_raw_file_count,
    actual_raw_file_count
    == EXPECTED_RAW_FILES,
)

add_check(
    validation_records,
    "Raw bytes",
    EXPECTED_RAW_BYTES,
    actual_raw_bytes,
    actual_raw_bytes
    == EXPECTED_RAW_BYTES,
)

add_check(
    validation_records,
    "Raw manifest rows",
    EXPECTED_RAW_FILES,
    len(raw_file_manifest),
    len(raw_file_manifest)
    == EXPECTED_RAW_FILES,
)

add_check(
    validation_records,
    "Raw manifest missing files",
    0,
    len(manifest_missing_files),
    len(manifest_missing_files) == 0,
)

add_check(
    validation_records,
    "Raw manifest unexpected files",
    0,
    len(manifest_unexpected_files),
    len(manifest_unexpected_files) == 0,
)

add_check(
    validation_records,
    "Embedded file-hash mismatches",
    0,
    embedded_hash_mismatches,
    embedded_hash_mismatches == 0,
)

add_check(
    validation_records,
    "Ranking rows",
    EXPECTED_RANKING_ROWS,
    ranking_rows_total,
    ranking_rows_total
    == EXPECTED_RANKING_ROWS,
)

add_check(
    validation_records,
    "Ranking rows-per-condition violations",
    0,
    ranking_rows_per_condition_violations,
    ranking_rows_per_condition_violations == 0,
)

add_check(
    validation_records,
    "Project-run rows",
    EXPECTED_PROJECT_RUN_ROWS,
    len(project_run_all),
    len(project_run_all)
    == EXPECTED_PROJECT_RUN_ROWS,
)

add_check(
    validation_records,
    "Build-metric rows",
    EXPECTED_BUILD_METRIC_ROWS,
    len(build_metrics_all),
    len(build_metrics_all)
    == EXPECTED_BUILD_METRIC_ROWS,
)

add_check(
    validation_records,
    "Model-fit rows",
    EXPECTED_ML_FITS,
    len(model_fits_all),
    len(model_fits_all)
    == EXPECTED_ML_FITS,
)

add_check(
    validation_records,
    "Condition-audit rows",
    EXPECTED_CONDITION_AUDIT_ROWS,
    len(condition_audit_all),
    len(condition_audit_all)
    == EXPECTED_CONDITION_AUDIT_ROWS,
)

add_check(
    validation_records,
    "Training-median rows",
    EXPECTED_TRAINING_MEDIAN_ROWS,
    len(training_medians_all),
    len(training_medians_all)
    == EXPECTED_TRAINING_MEDIAN_ROWS,
)

add_check(
    validation_records,
    "Project-run technique set",
    sorted(ALL_TECHNIQUES),
    sorted(project_run_technique_set),
    project_run_technique_set
    == set(ALL_TECHNIQUES),
)

add_check(
    validation_records,
    "Model-fit technique set",
    sorted(ML_TECHNIQUES),
    sorted(model_fit_technique_set),
    model_fit_technique_set
    == set(ML_TECHNIQUES),
)

add_check(
    validation_records,
    "Duplicate project-run rows",
    0,
    project_run_duplicate_rows,
    project_run_duplicate_rows == 0,
)

add_check(
    validation_records,
    "Duplicate build-metric rows",
    0,
    build_metric_duplicate_rows,
    build_metric_duplicate_rows == 0,
)

add_check(
    validation_records,
    "Duplicate model-fit rows",
    0,
    model_fit_duplicate_rows,
    model_fit_duplicate_rows == 0,
)

add_check(
    validation_records,
    "Duplicate condition-audit rows",
    0,
    condition_audit_duplicate_rows,
    condition_audit_duplicate_rows == 0,
)

add_check(
    validation_records,
    "Duplicate training-median rows",
    0,
    training_median_duplicate_rows,
    training_median_duplicate_rows == 0,
)

add_check(
    validation_records,
    "Project-run rows-per-condition violations",
    0,
    project_run_rows_per_condition_violations,
    project_run_rows_per_condition_violations == 0,
)

add_check(
    validation_records,
    "Build-metric rows-per-condition violations",
    0,
    build_metric_rows_per_condition_violations,
    build_metric_rows_per_condition_violations == 0,
)

add_check(
    validation_records,
    "Model-fit rows-per-condition violations",
    0,
    model_fit_rows_per_condition_violations,
    model_fit_rows_per_condition_violations == 0,
)

add_check(
    validation_records,
    "Training-median rows-per-condition violations",
    0,
    training_median_rows_per_condition_violations,
    training_median_rows_per_condition_violations == 0,
)

add_check(
    validation_records,
    "Scored-build count violations",
    0,
    scored_build_count_violations,
    scored_build_count_violations == 0,
)

add_check(
    validation_records,
    "Evaluated-build violations",
    0,
    evaluated_build_violations,
    evaluated_build_violations == 0,
)

add_check(
    validation_records,
    "Evaluation-failure violations",
    0,
    evaluation_failure_violations,
    evaluation_failure_violations == 0,
)

add_check(
    validation_records,
    "Model-fit failures",
    0,
    model_fit_failures,
    model_fit_failures == 0,
)

add_check(
    validation_records,
    "Project-run non-finite metrics",
    0,
    project_run_nonfinite_metrics,
    project_run_nonfinite_metrics == 0,
)

add_check(
    validation_records,
    "Project-run metrics outside [0,1]",
    0,
    project_run_out_of_range_metrics,
    project_run_out_of_range_metrics == 0,
)

add_check(
    validation_records,
    "Build-metric non-finite values",
    0,
    build_metric_nonfinite_values,
    build_metric_nonfinite_values == 0,
)

add_check(
    validation_records,
    "Build metrics outside [0,1]",
    0,
    build_metric_out_of_range_values,
    build_metric_out_of_range_values == 0,
)

add_check(
    validation_records,
    "Training-median non-finite values",
    0,
    training_median_nonfinite_values,
    training_median_nonfinite_values == 0,
)

add_check(
    validation_records,
    "Zero-noise conditions",
    30,
    zero_noise_conditions,
    zero_noise_conditions == 30,
)

add_check(
    validation_records,
    "Zero-noise raw-flip violations",
    0,
    zero_noise_raw_flip_violations,
    zero_noise_raw_flip_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise model-change violations",
    0,
    zero_noise_model_change_violations,
    zero_noise_model_change_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise dependent REC violations",
    0,
    zero_noise_dependent_rec_violations,
    zero_noise_dependent_rec_violations == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without raw changes",
    0,
    positive_noise_without_raw_changes,
    positive_noise_without_raw_changes == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without model changes",
    0,
    positive_noise_without_model_changes,
    positive_noise_without_model_changes == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without dependent REC changes",
    0,
    positive_noise_without_dependent_rec_changes,
    positive_noise_without_dependent_rec_changes == 0,
)

add_check(
    validation_records,
    "Independent REC change violations",
    0,
    independent_rec_change_violations,
    independent_rec_change_violations == 0,
)

add_check(
    validation_records,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)

add_check(
    validation_records,
    "Noise-technique summary rows",
    EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS,
    len(noise_technique_summary),
    len(noise_technique_summary)
    == EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS,
)

add_check(
    validation_records,
    "Summary run-count violations",
    0,
    summary_run_count_violations,
    summary_run_count_violations == 0,
)

add_check(
    validation_records,
    "Summary seed-count violations",
    0,
    summary_seed_count_violations,
    summary_seed_count_violations == 0,
)

add_check(
    validation_records,
    "Summary non-finite values",
    0,
    summary_nonfinite_values,
    summary_nonfinite_values == 0,
)

add_check(
    validation_records,
    "Seed-level delta rows",
    EXPECTED_SEED_LEVEL_DELTA_ROWS,
    len(seed_level_noise_deltas),
    len(seed_level_noise_deltas)
    == EXPECTED_SEED_LEVEL_DELTA_ROWS,
)

add_check(
    validation_records,
    "Clean-delta non-zero values",
    0,
    clean_delta_nonzero_values,
    clean_delta_nonzero_values == 0,
)

add_check(
    validation_records,
    "Noise-delta summary rows",
    EXPECTED_NOISE_DELTA_SUMMARY_ROWS,
    len(noise_delta_summary),
    len(noise_delta_summary)
    == EXPECTED_NOISE_DELTA_SUMMARY_ROWS,
)


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nProject 10 Step 5B validation:")

display(
    validation
)


if not failed_checks.empty:
    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 10 STEP 5B DID NOT PASS."
    )


# --------------------------------------------------------------------------------------------------
# 18. WRITE COMPACT AGGREGATES
# --------------------------------------------------------------------------------------------------

STEP5B_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    RAW_FILE_MANIFEST_PATH,
    raw_file_manifest,
)

atomic_write_csv(
    CONDITION_INVENTORY_PATH,
    condition_inventory,
)

atomic_write_csv(
    PROJECT_RUN_ALL_PATH,
    project_run_all,
)

atomic_write_parquet(
    BUILD_METRICS_ALL_PATH,
    build_metrics_all,
)

atomic_write_csv(
    MODEL_FITS_ALL_PATH,
    model_fits_all,
)

atomic_write_csv(
    CONDITION_AUDIT_ALL_PATH,
    condition_audit_all,
)

atomic_write_parquet(
    TRAINING_MEDIANS_ALL_PATH,
    training_medians_all,
)

atomic_write_csv(
    NOISE_TECHNIQUE_SUMMARY_PATH,
    noise_technique_summary,
)

atomic_write_csv(
    SEED_LEVEL_NOISE_DELTAS_PATH,
    seed_level_noise_deltas,
)

atomic_write_csv(
    NOISE_DELTA_SUMMARY_PATH,
    noise_delta_summary,
)

atomic_write_csv(
    CLEAN_TECHNIQUE_SUMMARY_PATH,
    clean_technique_summary,
)

atomic_write_csv(
    BASELINE_INVARIANCE_AUDIT_PATH,
    baseline_invariance_audit,
)

atomic_write_csv(
    STEP5B_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 19. READBACK VALIDATION
# --------------------------------------------------------------------------------------------------

readback_checks = {
    "RawManifestRows":
        len(
            pd.read_csv(
                RAW_FILE_MANIFEST_PATH,
                low_memory=False,
            )
        )
        == EXPECTED_RAW_FILES,

    "ConditionInventoryRows":
        len(
            pd.read_csv(
                CONDITION_INVENTORY_PATH,
                low_memory=False,
            )
        )
        == EXPECTED_CONDITIONS,

    "ProjectRunRows":
        len(
            pd.read_csv(
                PROJECT_RUN_ALL_PATH,
                low_memory=False,
            )
        )
        == EXPECTED_PROJECT_RUN_ROWS,

    "BuildMetricRows":
        len(
            pd.read_parquet(
                BUILD_METRICS_ALL_PATH
            )
        )
        == EXPECTED_BUILD_METRIC_ROWS,

    "ModelFitRows":
        len(
            pd.read_csv(
                MODEL_FITS_ALL_PATH,
                low_memory=False,
            )
        )
        == EXPECTED_ML_FITS,

    "ConditionAuditRows":
        len(
            pd.read_csv(
                CONDITION_AUDIT_ALL_PATH,
                low_memory=False,
            )
        )
        == EXPECTED_CONDITION_AUDIT_ROWS,

    "TrainingMedianRows":
        len(
            pd.read_parquet(
                TRAINING_MEDIANS_ALL_PATH
            )
        )
        == EXPECTED_TRAINING_MEDIAN_ROWS,

    "NoiseTechniqueSummaryRows":
        len(
            pd.read_csv(
                NOISE_TECHNIQUE_SUMMARY_PATH,
                low_memory=False,
            )
        )
        == EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS,

    "SeedDeltaRows":
        len(
            pd.read_csv(
                SEED_LEVEL_NOISE_DELTAS_PATH,
                low_memory=False,
            )
        )
        == EXPECTED_SEED_LEVEL_DELTA_ROWS,

    "NoiseDeltaRows":
        len(
            pd.read_csv(
                NOISE_DELTA_SUMMARY_PATH,
                low_memory=False,
            )
        )
        == EXPECTED_NOISE_DELTA_SUMMARY_ROWS,
}


failed_readback_checks = [
    name
    for name, passed in readback_checks.items()
    if not passed
]


if failed_readback_checks:
    raise RuntimeError(
        "Project 10 Step 5B readback failed:\n"
        + "\n".join(
            failed_readback_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 20. OUTPUT HASHES
# --------------------------------------------------------------------------------------------------

output_paths = {
    "RawFileManifest":
        RAW_FILE_MANIFEST_PATH,

    "ConditionInventory":
        CONDITION_INVENTORY_PATH,

    "ProjectRunAll":
        PROJECT_RUN_ALL_PATH,

    "BuildMetricsAll":
        BUILD_METRICS_ALL_PATH,

    "ModelFitsAll":
        MODEL_FITS_ALL_PATH,

    "ConditionAuditAll":
        CONDITION_AUDIT_ALL_PATH,

    "TrainingMediansAll":
        TRAINING_MEDIANS_ALL_PATH,

    "NoiseTechniqueSummary":
        NOISE_TECHNIQUE_SUMMARY_PATH,

    "SeedLevelNoiseDeltas":
        SEED_LEVEL_NOISE_DELTAS_PATH,

    "NoiseDeltaSummary":
        NOISE_DELTA_SUMMARY_PATH,

    "CleanTechniqueSummary":
        CLEAN_TECHNIQUE_SUMMARY_PATH,

    "BaselineInvarianceAudit":
        BASELINE_INVARIANCE_AUDIT_PATH,

    "Step5BValidation":
        STEP5B_VALIDATION_PATH,
}


output_hashes = {
    name:
        sha256_file(path)
    for name, path in output_paths.items()
}


output_sizes = {
    name:
        int(
            path.stat().st_size
        )
    for name, path in output_paths.items()
}


# --------------------------------------------------------------------------------------------------
# 21. REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

registry_unchanged = bool(
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:
    raise RuntimeError(
        "The completion registry changed during Project 10 Step 5B."
    )


registry_after = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_after_project_numbers = pd.to_numeric(
    registry_after[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


registry_project9_rows_after = int(
    registry_after_project_numbers.eq(9).sum()
)

registry_project10_rows_after = int(
    registry_after_project_numbers.eq(10).sum()
)


if (
    registry_project9_rows_after != 0
    or registry_project10_rows_after != 0
):
    raise RuntimeError(
        "Project 9 or Project 10 was unexpectedly inserted "
        "into the completion registry."
    )


# --------------------------------------------------------------------------------------------------
# 22. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "Step5AStatus":
        EXPECTED_STEP5A_STATUS,

    "Step5AStatusFile":
        str(
            STEP5A_STATUS_PATH
        ),

    "Step5AStatusSHA256":
        sha256_file(
            STEP5A_STATUS_PATH
        ),

    "FullRunCheckpointTable":
        str(
            FULL_RUN_CHECKPOINT_TABLE_PATH
        ),

    "FullRunCheckpointTableSHA256":
        sha256_file(
            FULL_RUN_CHECKPOINT_TABLE_PATH
        ),

    "FullRunExecutionReport":
        str(
            FULL_RUN_EXECUTION_REPORT_PATH
        ),

    "FullRunExecutionReportSHA256":
        sha256_file(
            FULL_RUN_EXECUTION_REPORT_PATH
        ),

    "Conditions":
        EXPECTED_CONDITIONS,

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINE_TECHNIQUES,

    "MLFits":
        len(
            model_fits_all
        ),

    "RankingRows":
        ranking_rows_total,

    "BuildMetricRows":
        len(
            build_metrics_all
        ),

    "ProjectRunRows":
        len(
            project_run_all
        ),

    "ConditionAuditRows":
        len(
            condition_audit_all
        ),

    "TrainingMedianRows":
        len(
            training_medians_all
        ),

    "ActivePredictors":
        EXPECTED_ACTIVE_PREDICTORS,

    "RawFiles":
        actual_raw_file_count,

    "RawBytes":
        actual_raw_bytes,

    "RawRootSHA256":
        raw_root_sha256,

    "EmbeddedHashChecks":
        embedded_hash_checks,

    "EmbeddedHashMismatches":
        embedded_hash_mismatches,

    "NoiseTechniqueSummaryRows":
        len(
            noise_technique_summary
        ),

    "SeedLevelNoiseDeltaRows":
        len(
            seed_level_noise_deltas
        ),

    "NoiseDeltaSummaryRows":
        len(
            noise_delta_summary
        ),

    "StandardDeviationDefinition":
        (
            "Sample standard deviation across the 30 "
            "seed-level project-run values; pandas std, ddof=1"
        ),

    "BaselineInvarianceFailures":
        baseline_invariance_failures,

    "RawHashSeconds":
        raw_hash_seconds,

    "AggregationSeconds":
        aggregation_seconds,

    "CompletionRegistryModified":
        False,

    "RegistryProject9Rows":
        registry_project9_rows_after,

    "RegistryProject10Rows":
        registry_project10_rows_after,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "OutputPaths": {
        name:
            str(path)
        for name, path in output_paths.items()
    },

    "OutputSHA256":
        output_hashes,

    "OutputSizeBytes":
        output_sizes,

    "ValidationChecks":
        len(validation),

    "FailedValidationChecks":
        len(failed_checks),
}


atomic_write_json(
    STEP5B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "Step5BReport":
        str(
            STEP5B_REPORT_PATH
        ),

    "Step5BReportSHA256":
        sha256_file(
            STEP5B_REPORT_PATH
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CheckpointFrozenAtUTC":
        completed_at_utc,
}


atomic_write_json(
    STEP5B_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5B_PASS_STATUS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "MLFits":
        len(
            model_fits_all
        ),

    "RankingRows":
        ranking_rows_total,

    "BuildMetricRows":
        len(
            build_metrics_all
        ),

    "ProjectRunRows":
        len(
            project_run_all
        ),

    "RawFiles":
        actual_raw_file_count,

    "RawBytes":
        actual_raw_bytes,

    "RawRootSHA256":
        raw_root_sha256,

    "Checkpoint":
        str(
            STEP5B_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        sha256_file(
            STEP5B_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(failed_checks),

    "CompletionRegistryModified":
        False,

    "RegistryProject9Rows":
        registry_project9_rows_after,

    "RegistryProject10Rows":
        registry_project10_rows_after,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "CompletedAtUTC":
        completed_at_utc,
}


atomic_write_json(
    STEP5B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 23. FINAL READBACK
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    STEP5B_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP5B_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP5B_PASS_STATUS:
    raise RuntimeError(
        "Project 10 Step 5B checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP5B_PASS_STATUS:
    raise RuntimeError(
        "Project 10 Step 5B status readback failed."
    )


if sha256_file(
    REGISTRY_PATH
) != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during final readback."
    )


# --------------------------------------------------------------------------------------------------
# 24. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nCondition inventory sample:")

display(
    pd.concat(
        [
            condition_inventory.head(9),
            condition_inventory.tail(9),
        ],
        ignore_index=True,
    )
)


print("\nNoise × technique APFD/APFDc summary:")

display(
    noise_technique_summary
)


print("\nClean 0% ranking by mean APFDc:")

display(
    clean_technique_summary[
        [
            "Technique",
            "Runs",
            "Seeds",
            "MeanAPFD",
            "StdAPFD",
            "MeanAPFDc",
            "StdAPFDc",
        ]
    ]
)


print("\nBaseline invariance audit:")

display(
    baseline_invariance_audit
)


print("\nNoise-delta summary at 0%, 25%, and 50%:")

display(
    noise_delta_summary[
        noise_delta_summary[
            "NoisePercent"
        ].isin(
            [
                0,
                25,
                50,
            ]
        )
    ]
)


# --------------------------------------------------------------------------------------------------
# 25. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 10 CELL 10 / STEP 5B RESULT ===")
print("=" * 132)


print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nRaw-output revalidation:")

print(
    "Conditions:",
    len(condition_inventory),
    "/",
    EXPECTED_CONDITIONS,
)

print(
    "Raw files:",
    actual_raw_file_count,
    "/",
    EXPECTED_RAW_FILES,
)

print(
    "Raw bytes:",
    actual_raw_bytes,
    "/",
    EXPECTED_RAW_BYTES,
)

print(
    "Raw-root SHA-256:",
    raw_root_sha256,
)

print(
    "Embedded hash checks:",
    embedded_hash_checks,
)

print(
    "Embedded hash mismatches:",
    embedded_hash_mismatches,
)


print("\nExperiment totals:")

print(
    "ML fits:",
    len(model_fits_all),
    "/",
    EXPECTED_ML_FITS,
)

print(
    "Ranking rows:",
    ranking_rows_total,
    "/",
    EXPECTED_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    len(build_metrics_all),
    "/",
    EXPECTED_BUILD_METRIC_ROWS,
)

print(
    "Project-run rows:",
    len(project_run_all),
    "/",
    EXPECTED_PROJECT_RUN_ROWS,
)

print(
    "Condition-audit rows:",
    len(condition_audit_all),
    "/",
    EXPECTED_CONDITION_AUDIT_ROWS,
)

print(
    "Training-median rows:",
    len(training_medians_all),
    "/",
    EXPECTED_TRAINING_MEDIAN_ROWS,
)


print("\nAnalysis-ready aggregates:")

print(
    "Noise-technique summary rows:",
    len(noise_technique_summary),
)

print(
    "Seed-level delta rows:",
    len(seed_level_noise_deltas),
)

print(
    "Noise-delta summary rows:",
    len(noise_delta_summary),
)

print(
    "APFD standard deviation calculated:",
    True,
)

print(
    "APFDc standard deviation calculated:",
    True,
)

print(
    "Standard deviation:",
    "Sample SD across 30 seeds, ddof=1",
)

print(
    "Baseline invariance failures:",
    baseline_invariance_failures,
)


print("\nImmutability and isolation:")

print(
    "Completion registry unchanged:",
    registry_unchanged,
)

print(
    "Registry Project 9 rows:",
    registry_project9_rows_after,
)

print(
    "Registry Project 10 rows:",
    registry_project10_rows_after,
)

print(
    "Project 9 accessed:",
    False,
)

print(
    "Project 9 write attempted:",
    False,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nRuntime:")

print(
    "Raw hashing seconds:",
    round(
        raw_hash_seconds,
        2,
    ),
)

print(
    "Aggregation seconds:",
    round(
        aggregation_seconds,
        2,
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(validation),
)

print(
    "Failed checks:",
    len(failed_checks),
)


print("\nProject 10 Step 5B checkpoint:")

print(
    STEP5B_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    sha256_file(
        STEP5B_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP5B_PASS_STATUS,
)

print("=" * 132)

=== PROJECT 10 CELL 10 / STEP 5B: FINAL RAW-OUTPUT REVALIDATION AND COMPACT AGGREGATION ===

Project 10 Step 5B validation:


,Check,Expected,Actual,Pass
0,Step 5A status,PASS_PROJECT_10_FULL_270_CONDITION_RUN_COMPLETE,PASS_PROJECT_10_FULL_270_CONDITION_RUN_COMPLETE,True
1,Full-run progress status,PASS_PROJECT_10_FULL_270_CONDITION_RUN_COMPLETE,PASS_PROJECT_10_FULL_270_CONDITION_RUN_COMPLETE,True
2,Full-run report status,PASS_PROJECT_10_FULL_270_CONDITION_RUN_COMPLETE,PASS_PROJECT_10_FULL_270_CONDITION_RUN_COMPLETE,True
3,Condition markers,270,270,True
4,Condition inventory rows,270,270,True
5,Condition coordinates,270,270,True
6,Duplicate condition IDs,0,0,True
7,Duplicate condition coordinates,0,0,True
8,Files-per-condition violations,0,0,True
9,Raw files,2160,2160,True



Condition inventory sample:


,ProjectNumber,Project,ProjectSlug,ConditionOrder,ConditionID,NoisePercent,RepetitionSeed,ConditionDirectory,Status,Files,ConditionBytes,RankingRows,BuildMetricRows,ProjectRunRows,ModelFits,TrainingMedianRows,ConditionSeconds,SuccessMarkerSHA256
0,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,1,noise_00__seed_01,0,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_PROJECT_10_CONDITION_COMPLETE,8,239069,18277,189,7,4,151,14.118890,7aac787993ed3ba61823e09b61f5ceb2d56e49ab958f61...
1,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,2,noise_05__seed_01,5,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_PROJECT_10_CONDITION_COMPLETE,8,261205,18277,189,7,4,151,39.084747,441488121bf7e72acf366a86b2375040e9805f6c714b71...
2,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,3,noise_10__seed_01,10,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_PROJECT_10_CONDITION_COMPLETE,8,259416,18277,189,7,4,151,42.666013,13c9d893ab19c375d2151ba74c27c19206579eda4cc689...
3,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,4,noise_15__seed_01,15,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_PROJECT_10_CONDITION_COMPLETE,8,255828,18277,189,7,4,151,31.290343,5282fd3a5d4112832c4c2db26a2fe7e1b7a4a83643b8f2...
4,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,5,noise_20__seed_01,20,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_PROJECT_10_CONDITION_COMPLETE,8,256301,18277,189,7,4,151,21.123498,e7d2c2b4596416898c7e761fa76a57231b26da616648c1...
5,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,6,noise_25__seed_01,25,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_PROJECT_10_CONDITION_COMPLETE,8,254376,18277,189,7,4,151,26.894078,ba00e45ea28195748af5d56e0bf31c37feba0d46d851b5...
6,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,7,noise_30__seed_01,30,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_PROJECT_10_CONDITION_COMPLETE,8,254031,18277,189,7,4,151,21.692060,570556c94994e338d7b7db33b607a2065728d6acdfd15d...
7,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,8,noise_40__seed_01,40,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_PROJECT_10_CONDITION_COMPLETE,8,254230,18277,189,7,4,151,25.948950,c1684941f9d57d4631218619908b024987a197765f5387...
8,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,9,noise_50__seed_01,50,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_PROJECT_10_CONDITION_COMPLETE,8,253758,18277,189,7,4,151,22.186076,8fc4862429884dde25ce32ff64b424039657d154d0a467...
9,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,262,noise_00__seed_30,0,30,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_PROJECT_10_CONDITION_COMPLETE,8,241134,18277,189,7,4,151,4.833651,286bcadfe737783af2dbc3c275a6e127dafaf44abc6854...



Noise × technique APFD/APFDc summary:


...


Baseline invariance audit:


,Technique,Expected,SeedsChecked,ViolatingSeeds,NoiseResponseSeeds,Pass
0,Random,APFD and APFDc constant across all noise level...,30,0,0,True
1,QTF-Avg,APFD and APFDc constant across all noise level...,30,0,0,True
2,LatestFail,At least one seed changes across noise levels,30,0,30,True



Noise-delta summary at 0%, 25%, and 50%:


,NoisePercent,Technique,Seeds,MeanAPFDDeltaFromClean,StdAPFDDeltaFromClean,MedianAPFDDeltaFromClean,MeanAPFDcDeltaFromClean,StdAPFDcDeltaFromClean,MedianAPFDcDeltaFromClean,MeanAPFDRelativeDegradation,StdAPFDRelativeDegradation,MedianAPFDRelativeDegradation,MeanAPFDcRelativeDegradation,StdAPFDcRelativeDegradation,MedianAPFDcRelativeDegradation
0,0,LatestFail,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,LightGBM,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,NaiveBayes,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0,RandomForest,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0,XGBoost,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
35,25,LatestFail,30,-0.000601,0.016123,0.000885,0.010262,0.017144,0.009773,0.000688,0.018454,-0.001013,-0.013593,0.022708,-0.012945
36,25,LightGBM,30,-0.126502,0.058471,-0.134217,-0.043030,0.061638,-0.022692,0.138578,0.064053,0.147029,0.052901,0.075777,0.027897
37,25,NaiveBayes,30,0.011364,0.029840,0.016771,-0.112529,0.088007,-0.115671,-0.014261,0.037447,-0.021046,0.148376,0.116042,0.152518




=== PROJECT 10 CELL 10 / STEP 5B RESULT ===

Project identity:
Project number: 10
Project: spring-cloud@spring-cloud-dataflow
Project slug: spring-cloud__spring-cloud-dataflow

Raw-output revalidation:
Conditions: 270 / 270
Raw files: 2160 / 2160
Raw bytes: 68638025 / 68638025
Raw-root SHA-256: 5d336e1c7630cd6dec8ecdd8e17fa1af5cd6547d6cf90ecc19d0dbdce43fea6c
Embedded hash checks: 1890
Embedded hash mismatches: 0

Experiment totals:
ML fits: 1080 / 1080
Ranking rows: 4934790 / 4934790
Build-metric rows: 51030 / 51030
Project-run rows: 1890 / 1890
Condition-audit rows: 270 / 270
Training-median rows: 40770 / 40770

Analysis-ready aggregates:
Noise-technique summary rows: 63
Seed-level delta rows: 1890
Noise-delta summary rows: 63
APFD standard deviation calculated: True
APFDc standard deviation calculated: True
Standard deviation: Sample SD across 30 seeds, ddof=1
Baseline invariance failures: 0

Immutability and isolation:
Completion registry unchanged: True
Registry Project 9 rows: 0

In [14]:
# ==================================================================================================
# PROJECT 10 — CELL 11 / STEP 5C
# FINAL PACKAGE CONSTRUCTION, HASH VALIDATION, AND FREEZE
#
# PROJECT:
#   spring-cloud@spring-cloud-dataflow
#
# SAFETY:
# - Does not rerun conditions, models, baselines, APFD, or APFDc.
# - Does not write inside the Project 10 raw-result directory.
# - Does not access or modify Project 9 files.
# - Does not modify Projects 1–8.
# - Does not modify the completion registry.
# - Registry insertion remains pending for one later serial cell
#   covering both Project 9 and Project 10.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import shutil

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 10 CELL 11 / STEP 5C: FINAL PACKAGE CONSTRUCTION AND VALIDATION ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT IDENTITY
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 10

PROJECT_NAME = (
    "spring-cloud@spring-cloud-dataflow"
)

PROJECT_SLUG = (
    "spring-cloud__spring-cloud-dataflow"
)

PROJECT_SHORT_NAME = (
    "spring_cloud_dataflow"
)


EXPECTED_STEP5A_STATUS = (
    "PASS_PROJECT_10_FULL_270_CONDITION_RUN_COMPLETE"
)

EXPECTED_STEP5B_STATUS = (
    "PASS_PROJECT_10_RAW_RESULTS_REVALIDATED_"
    "AND_COMPACT_AGGREGATES_FROZEN"
)

FINAL_STATUS = (
    "PASS_PROJECT_10_FINAL_PACKAGE_CONSTRUCTED_"
    "AND_VALIDATED_REGISTRY_PENDING"
)


EXPECTED_STEP5B_CHECKPOINT_SHA256 = (
    "38ed74dac0892ba69efb418397c27e926"
    "c221880e39398d70447aba37d2cb364"
)

EXPECTED_RAW_ROOT_SHA256 = (
    "5d336e1c7630cd6dec8ecdd8e17fa1af"
    "5cd6547d6cf90ecc19d0dbdce43fea6c"
)


EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080
EXPECTED_RANKING_ROWS = 4_934_790
EXPECTED_BUILD_METRIC_ROWS = 51_030
EXPECTED_PROJECT_RUN_ROWS = 1_890
EXPECTED_CONDITION_AUDIT_ROWS = 270
EXPECTED_TRAINING_MEDIAN_ROWS = 40_770
EXPECTED_ACTIVE_PREDICTORS = 151

EXPECTED_NOISE_TECHNIQUE_ROWS = 63
EXPECTED_SEED_LEVEL_DELTA_ROWS = 1_890
EXPECTED_NOISE_DELTA_ROWS = 63

EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 68_638_025


# --------------------------------------------------------------------------------------------------
# 2. PROJECT 10 PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

RAW_RESULTS_ROOT = (
    RESULTS_ROOT
    / "Raw"
)

AGGREGATED_RESULTS_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
)


PROJECT_RAW_ROOT = (
    RAW_RESULTS_ROOT
    / PROJECT_SLUG
)

PROJECT_AGGREGATED_ROOT = (
    AGGREGATED_RESULTS_ROOT
    / PROJECT_SLUG
)

PROJECT_SELECTION_ROOT = (
    AGGREGATED_RESULTS_ROOT
    / "project_10_selection"
)

STEP5B_FINAL_AUDIT_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_step5b_final_audit"
)

FULL_RUN_CONTROL_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_full_run_control"
)


STEP5A_STATUS_PATH = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_step5a_status.json"
)

STEP5B_STATUS_PATH = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_step5b_status.json"
)

STEP5B_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_10_step5b_checkpoint.json"
)

FINAL_PACKAGE_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_10_final_package_checkpoint.json"
)

STEP5C_STATUS_PATH = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_step5c_status.json"
)


COMPLETION_REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)


FINAL_PACKAGE_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_final_package"
)

STAGING_PACKAGE_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / f".{PROJECT_SHORT_NAME}_final_package_staging"
)

STEP5C_AUDIT_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_step5c_package_audit"
)

FINAL_PACKAGE_INVENTORY_PATH = (
    STEP5C_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_final_package_inventory.csv"
)

STEP5C_VALIDATION_PATH = (
    STEP5C_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_step5c_validation.csv"
)

STEP5C_REPORT_PATH = (
    STEP5C_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_step5c_report.json"
)


RAW_MANIFEST_PATH = (
    STEP5B_FINAL_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_raw_file_manifest.csv"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def canonical_root_hash(
    manifest,
):
    required_columns = [
        "RelativePath",
        "SizeBytes",
        "SHA256",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in manifest.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Manifest is missing required columns:\n"
            + "\n".join(
                missing_columns
            )
        )

    digest = hashlib.sha256()

    ordered = manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    dataframe,
    candidates,
    label,
):
    lower_lookup = {
        str(column).lower():
            column
        for column in dataframe.columns
    }

    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

        candidate_lower = str(
            candidate
        ).lower()

        if candidate_lower in lower_lookup:
            return lower_lookup[
                candidate_lower
            ]

    raise RuntimeError(
        f"Could not resolve {label}.\n"
        f"Candidates: {candidates}\n"
        f"Columns: {dataframe.columns.tolist()}"
    )


def extract_status(payload):
    for key in [
        "Status",
        "status",
        "FinalStatus",
        "Step5BStatus",
    ]:
        if key in payload:
            return str(
                payload[key]
            )

    return None


def is_relative_to(
    path,
    parent,
):
    try:
        Path(path).resolve().relative_to(
            Path(parent).resolve()
        )

        return True

    except ValueError:
        return False


def safe_remove_directory(
    path,
    expected_parent,
    allowed_name,
):
    path = Path(path)

    if not path.exists():
        return

    if not path.is_dir():
        raise RuntimeError(
            f"Expected a directory:\n{path}"
        )

    if path.parent.resolve() != Path(
        expected_parent
    ).resolve():
        raise RuntimeError(
            "Unsafe directory removal blocked.\n"
            f"Path: {path}"
        )

    if path.name != allowed_name:
        raise RuntimeError(
            "Unsafe directory name blocked.\n"
            f"Path: {path}"
        )

    shutil.rmtree(
        path
    )


def build_tree_manifest(root):
    root = Path(root)

    records = []

    for file_path in sorted(
        [
            path
            for path in root.rglob("*")
            if path.is_file()
        ],
        key=lambda path:
            path.relative_to(
                root
            ).as_posix(),
    ):
        records.append({
            "RelativePath":
                file_path.relative_to(
                    root
                ).as_posix(),

            "SizeBytes":
                int(
                    file_path.stat().st_size
                ),

            "SHA256":
                sha256_file(
                    file_path
                ),
        })

    return pd.DataFrame(
        records
    )


def add_check(
    records,
    check,
    expected,
    actual,
    passed,
):
    records.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_directories = [
    NOTES_ROOT,
    PROJECT_RAW_ROOT,
    PROJECT_AGGREGATED_ROOT,
    PROJECT_SELECTION_ROOT,
    STEP5B_FINAL_AUDIT_ROOT,
    FULL_RUN_CONTROL_ROOT,
]

for directory in required_directories:
    if not directory.is_dir():
        raise FileNotFoundError(
            "Required Project 10 directory is missing:\n"
            f"{directory}"
        )


required_files = [
    STEP5A_STATUS_PATH,
    STEP5B_STATUS_PATH,
    STEP5B_CHECKPOINT_PATH,
    COMPLETION_REGISTRY_PATH,
    RAW_MANIFEST_PATH,
]

for file_path in required_files:
    if not file_path.is_file():
        raise FileNotFoundError(
            "Required Project 10 file is missing:\n"
            f"{file_path}"
        )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE STEP 5A AND STEP 5B
# --------------------------------------------------------------------------------------------------

step5a_status_payload = load_json(
    STEP5A_STATUS_PATH
)

step5b_status_payload = load_json(
    STEP5B_STATUS_PATH
)

step5b_checkpoint_payload = load_json(
    STEP5B_CHECKPOINT_PATH
)


step5a_status = extract_status(
    step5a_status_payload
)

step5b_status = extract_status(
    step5b_status_payload
)

step5b_checkpoint_status = extract_status(
    step5b_checkpoint_payload
)


step5b_checkpoint_sha256 = sha256_file(
    STEP5B_CHECKPOINT_PATH
)


if step5a_status != EXPECTED_STEP5A_STATUS:
    raise RuntimeError(
        "Project 10 Step 5A status differs.\n"
        f"Expected: {EXPECTED_STEP5A_STATUS}\n"
        f"Actual:   {step5a_status}"
    )


if step5b_status != EXPECTED_STEP5B_STATUS:
    raise RuntimeError(
        "Project 10 Step 5B status differs.\n"
        f"Expected: {EXPECTED_STEP5B_STATUS}\n"
        f"Actual:   {step5b_status}"
    )


if step5b_checkpoint_status != EXPECTED_STEP5B_STATUS:
    raise RuntimeError(
        "Project 10 Step 5B checkpoint status differs.\n"
        f"Expected: {EXPECTED_STEP5B_STATUS}\n"
        f"Actual:   {step5b_checkpoint_status}"
    )


if (
    step5b_checkpoint_sha256
    != EXPECTED_STEP5B_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 10 Step 5B checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_STEP5B_CHECKPOINT_SHA256}\n"
        f"Actual:   {step5b_checkpoint_sha256}"
    )


# --------------------------------------------------------------------------------------------------
# 6. REGISTRY SNAPSHOT — READ ONLY
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(
    COMPLETION_REGISTRY_PATH
)

registry_before = (
    pd.read_csv(
        COMPLETION_REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_number_column = resolve_column(
    registry_before,
    [
        "ProjectNumber",
        "project_number",
    ],
    "registry project-number column",
)


registry_project_numbers = pd.to_numeric(
    registry_before[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


registry_project9_rows_before = int(
    registry_project_numbers.eq(9).sum()
)

registry_project10_rows_before = int(
    registry_project_numbers.eq(10).sum()
)


if registry_project9_rows_before != 0:
    raise RuntimeError(
        "Project 9 is unexpectedly present in the registry. "
        "The serial Project 9 + Project 10 insertion has not yet run."
    )


if registry_project10_rows_before != 0:
    raise RuntimeError(
        "Project 10 is already present in the completion registry."
    )


# --------------------------------------------------------------------------------------------------
# 7. REVALIDATE THE FROZEN RAW ROOT
# --------------------------------------------------------------------------------------------------

raw_manifest_source = pd.read_csv(
    RAW_MANIFEST_PATH,
    low_memory=False,
)


raw_relative_column = resolve_column(
    raw_manifest_source,
    [
        "RelativePath",
        "relative_path",
        "Path",
    ],
    "raw-manifest relative path",
)

raw_size_column = resolve_column(
    raw_manifest_source,
    [
        "SizeBytes",
        "FileSizeBytes",
        "size_bytes",
    ],
    "raw-manifest size",
)

raw_sha_column = resolve_column(
    raw_manifest_source,
    [
        "SHA256",
        "FileSHA256",
        "sha256",
    ],
    "raw-manifest SHA-256",
)


raw_manifest = pd.DataFrame({
    "RelativePath":
        raw_manifest_source[
            raw_relative_column
        ]
        .astype(str)
        .str.replace(
            "\\",
            "/",
            regex=False,
        ),

    "SizeBytes":
        pd.to_numeric(
            raw_manifest_source[
                raw_size_column
            ],
            errors="raise",
        ).astype("int64"),

    "ExpectedSHA256":
        raw_manifest_source[
            raw_sha_column
        ]
        .astype(str)
        .str.lower(),
})


raw_manifest = (
    raw_manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if raw_manifest[
    "RelativePath"
].duplicated().any():
    raise RuntimeError(
        "The Project 10 raw manifest contains duplicate paths."
    )


actual_raw_paths = sorted([
    path.relative_to(
        PROJECT_RAW_ROOT
    ).as_posix()
    for path in PROJECT_RAW_ROOT.rglob("*")
    if path.is_file()
])


manifest_raw_paths = raw_manifest[
    "RelativePath"
].tolist()


missing_raw_files = sorted(
    set(manifest_raw_paths)
    - set(actual_raw_paths)
)

unexpected_raw_files = sorted(
    set(actual_raw_paths)
    - set(manifest_raw_paths)
)


raw_size_mismatches = []
raw_hash_mismatches = []
current_raw_records = []


for row in raw_manifest.itertuples(
    index=False
):
    file_path = (
        PROJECT_RAW_ROOT
        / row.RelativePath
    )

    if not file_path.is_file():
        continue

    actual_size = int(
        file_path.stat().st_size
    )

    actual_sha256 = sha256_file(
        file_path
    )


    if actual_size != int(
        row.SizeBytes
    ):
        raw_size_mismatches.append(
            row.RelativePath
        )


    if actual_sha256 != str(
        row.ExpectedSHA256
    ):
        raw_hash_mismatches.append(
            row.RelativePath
        )


    current_raw_records.append({
        "RelativePath":
            row.RelativePath,

        "SizeBytes":
            actual_size,

        "SHA256":
            actual_sha256,
    })


current_raw_manifest = pd.DataFrame(
    current_raw_records
)


current_raw_file_count = len(
    actual_raw_paths
)

current_raw_bytes = int(
    sum(
        (
            PROJECT_RAW_ROOT
            / relative_path
        ).stat().st_size
        for relative_path in actual_raw_paths
    )
)


current_raw_root_sha256 = canonical_root_hash(
    current_raw_manifest
)


if missing_raw_files:
    raise RuntimeError(
        f"Project 10 raw files are missing: "
        f"{len(missing_raw_files)}"
    )


if unexpected_raw_files:
    raise RuntimeError(
        f"Unexpected Project 10 raw files were found: "
        f"{len(unexpected_raw_files)}"
    )


if raw_size_mismatches:
    raise RuntimeError(
        f"Project 10 raw size mismatches: "
        f"{len(raw_size_mismatches)}"
    )


if raw_hash_mismatches:
    raise RuntimeError(
        f"Project 10 raw SHA-256 mismatches: "
        f"{len(raw_hash_mismatches)}"
    )


if current_raw_file_count != EXPECTED_RAW_FILES:
    raise RuntimeError(
        "Project 10 raw file count differs.\n"
        f"Expected: {EXPECTED_RAW_FILES}\n"
        f"Actual:   {current_raw_file_count}"
    )


if current_raw_bytes != EXPECTED_RAW_BYTES:
    raise RuntimeError(
        "Project 10 raw byte count differs.\n"
        f"Expected: {EXPECTED_RAW_BYTES}\n"
        f"Actual:   {current_raw_bytes}"
    )


if current_raw_root_sha256 != EXPECTED_RAW_ROOT_SHA256:
    raise RuntimeError(
        "Project 10 raw-root SHA-256 differs.\n"
        f"Expected: {EXPECTED_RAW_ROOT_SHA256}\n"
        f"Actual:   {current_raw_root_sha256}"
    )


# --------------------------------------------------------------------------------------------------
# 8. VALIDATE STEP 5B ANALYSIS-READY OUTPUTS
# --------------------------------------------------------------------------------------------------

required_final_audit_files = {
    f"{PROJECT_SHORT_NAME}_raw_file_manifest.csv",
    f"{PROJECT_SHORT_NAME}_condition_inventory.csv",
    f"{PROJECT_SHORT_NAME}_project_run_all.csv",
    f"{PROJECT_SHORT_NAME}_build_metrics_all.parquet",
    f"{PROJECT_SHORT_NAME}_model_fits_all.csv",
    f"{PROJECT_SHORT_NAME}_condition_audit_all.csv",
    f"{PROJECT_SHORT_NAME}_training_medians_all.parquet",
    f"{PROJECT_SHORT_NAME}_noise_technique_summary.csv",
    f"{PROJECT_SHORT_NAME}_seed_level_noise_deltas.csv",
    f"{PROJECT_SHORT_NAME}_noise_delta_summary.csv",
    f"{PROJECT_SHORT_NAME}_clean_technique_summary.csv",
    f"{PROJECT_SHORT_NAME}_baseline_invariance_audit.csv",
    f"{PROJECT_SHORT_NAME}_step5b_validation.csv",
    f"{PROJECT_SHORT_NAME}_step5b_report.json",
}


actual_final_audit_files = {
    path.name
    for path in STEP5B_FINAL_AUDIT_ROOT.rglob("*")
    if path.is_file()
}


missing_final_audit_files = sorted(
    required_final_audit_files
    - actual_final_audit_files
)


if missing_final_audit_files:
    raise FileNotFoundError(
        "Required Project 10 Step 5B outputs are missing:\n"
        + "\n".join(
            missing_final_audit_files
        )
    )


condition_inventory = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_condition_inventory.csv",
    low_memory=False,
)

project_run_all = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_project_run_all.csv",
    low_memory=False,
)

build_metrics_all = pd.read_parquet(
    STEP5B_FINAL_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_build_metrics_all.parquet"
)

model_fits_all = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_model_fits_all.csv",
    low_memory=False,
)

condition_audit_all = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_condition_audit_all.csv",
    low_memory=False,
)

training_medians_all = pd.read_parquet(
    STEP5B_FINAL_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_training_medians_all.parquet"
)

noise_technique_summary = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_noise_technique_summary.csv",
    low_memory=False,
)

seed_level_deltas = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_seed_level_noise_deltas.csv",
    low_memory=False,
)

noise_delta_summary = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_noise_delta_summary.csv",
    low_memory=False,
)

baseline_invariance = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / f"{PROJECT_SHORT_NAME}_baseline_invariance_audit.csv",
    low_memory=False,
)


baseline_pass_column = resolve_column(
    baseline_invariance,
    [
        "Pass",
        "pass",
    ],
    "baseline-invariance pass column",
)


baseline_pass_values = (
    baseline_invariance[
        baseline_pass_column
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin({
        "true",
        "1",
        "yes",
    })
)


baseline_invariance_failures = int(
    (
        ~baseline_pass_values
    ).sum()
)


analysis_count_checks = {
    "Conditions":
        (
            len(condition_inventory),
            EXPECTED_CONDITIONS,
        ),

    "MLFits":
        (
            len(model_fits_all),
            EXPECTED_ML_FITS,
        ),

    "BuildMetricRows":
        (
            len(build_metrics_all),
            EXPECTED_BUILD_METRIC_ROWS,
        ),

    "ProjectRunRows":
        (
            len(project_run_all),
            EXPECTED_PROJECT_RUN_ROWS,
        ),

    "ConditionAuditRows":
        (
            len(condition_audit_all),
            EXPECTED_CONDITION_AUDIT_ROWS,
        ),

    "TrainingMedianRows":
        (
            len(training_medians_all),
            EXPECTED_TRAINING_MEDIAN_ROWS,
        ),

    "NoiseTechniqueRows":
        (
            len(noise_technique_summary),
            EXPECTED_NOISE_TECHNIQUE_ROWS,
        ),

    "SeedLevelDeltaRows":
        (
            len(seed_level_deltas),
            EXPECTED_SEED_LEVEL_DELTA_ROWS,
        ),

    "NoiseDeltaRows":
        (
            len(noise_delta_summary),
            EXPECTED_NOISE_DELTA_ROWS,
        ),
}


failed_analysis_counts = [
    (
        name,
        actual,
        expected,
    )
    for name, (
        actual,
        expected,
    ) in analysis_count_checks.items()
    if actual != expected
]


if failed_analysis_counts:
    raise RuntimeError(
        "Project 10 analysis-ready output counts differ:\n"
        + "\n".join(
            (
                f"{name}: expected {expected}, "
                f"actual {actual}"
            )
            for name, actual, expected
            in failed_analysis_counts
        )
    )


if baseline_invariance_failures != 0:
    raise RuntimeError(
        "Project 10 baseline invariance audit contains failures."
    )


active_predictor_count = int(
    training_medians_all.groupby(
        "ConditionID"
    ).size().iloc[0]
)


if active_predictor_count != EXPECTED_ACTIVE_PREDICTORS:
    raise RuntimeError(
        "Project 10 active predictor count differs.\n"
        f"Expected: {EXPECTED_ACTIVE_PREDICTORS}\n"
        f"Actual:   {active_predictor_count}"
    )


# --------------------------------------------------------------------------------------------------
# 9. DISCOVER EXISTING PROJECT 10 CHECKPOINTS
# --------------------------------------------------------------------------------------------------

checkpoint_sources = sorted(
    [
        path
        for path in NOTES_ROOT.glob(
            "project_10_*checkpoint.json"
        )
        if (
            path.is_file()
            and path.name
            != FINAL_PACKAGE_CHECKPOINT_PATH.name
        )
    ],
    key=lambda path:
        path.name,
)


required_checkpoint_names = {
    "project_10_selection_checkpoint.json",
    "project_10_rec_reconstruction_checkpoint.json",
    "project_10_noise_plan_checkpoint.json",
    "project_10_noisy_rec_engine_checkpoint.json",
    "project_10_model_protocol_checkpoint.json",
    "project_10_smoke_test_checkpoint.json",
    "project_10_step5b_checkpoint.json",
}


actual_checkpoint_names = {
    path.name
    for path in checkpoint_sources
}


missing_required_checkpoints = sorted(
    required_checkpoint_names
    - actual_checkpoint_names
)


if missing_required_checkpoints:
    raise FileNotFoundError(
        "Required Project 10 checkpoints are missing:\n"
        + "\n".join(
            missing_required_checkpoints
        )
    )


print("\nProject 10 checkpoints discovered:")

for checkpoint_path in checkpoint_sources:
    print(
        " -",
        checkpoint_path.name,
    )


# --------------------------------------------------------------------------------------------------
# 10. BUILD PACKAGE SOURCE INVENTORY
# --------------------------------------------------------------------------------------------------

allowed_suffixes = {
    ".csv",
    ".json",
    ".parquet",
    ".txt",
    ".md",
    ".gz",
}


source_inventory = []


def register_source(
    category,
    source_path,
    destination_tail,
):
    source_path = Path(source_path)

    if not source_path.is_file():
        raise FileNotFoundError(
            "Package source is missing:\n"
            f"{source_path}"
        )

    source_inventory.append({
        "Category":
            category,

        "SourcePath":
            source_path,

        "DestinationTail":
            Path(destination_tail),
    })


# All existing Project 10 checkpoints.

for checkpoint_path in checkpoint_sources:
    register_source(
        category="checkpoints",
        source_path=checkpoint_path,
        destination_tail=checkpoint_path.name,
    )


# Project 10 candidate-selection evidence.

for source_path in sorted(
    [
        path
        for path in PROJECT_SELECTION_ROOT.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in allowed_suffixes
        )
    ],
    key=lambda path:
        path.relative_to(
            PROJECT_SELECTION_ROOT
        ).as_posix(),
):
    register_source(
        category="selection",
        source_path=source_path,
        destination_tail=source_path.relative_to(
            PROJECT_SELECTION_ROOT
        ),
    )


# Complete Step 5B final audit and aggregates.

for source_path in sorted(
    [
        path
        for path in STEP5B_FINAL_AUDIT_ROOT.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in allowed_suffixes
        )
    ],
    key=lambda path:
        path.relative_to(
            STEP5B_FINAL_AUDIT_ROOT
        ).as_posix(),
):
    register_source(
        category="final_audit",
        source_path=source_path,
        destination_tail=source_path.relative_to(
            STEP5B_FINAL_AUDIT_ROOT
        ),
    )


# Complete compact full-run control outputs.

for source_path in sorted(
    [
        path
        for path in FULL_RUN_CONTROL_ROOT.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in allowed_suffixes
        )
    ],
    key=lambda path:
        path.relative_to(
            FULL_RUN_CONTROL_ROOT
        ).as_posix(),
):
    register_source(
        category="full_run_control",
        source_path=source_path,
        destination_tail=source_path.relative_to(
            FULL_RUN_CONTROL_ROOT
        ),
    )


# Earlier protocol/status/validation evidence.
# Large raw rankings and smoke rankings are intentionally excluded.

evidence_keywords = (
    "status",
    "report",
    "validation",
    "manifest",
    "audit",
    "summary",
    "configuration",
    "protocol",
    "schema",
    "profile",
    "checkpoint",
    "progress",
)


for source_path in sorted(
    [
        path
        for path in PROJECT_AGGREGATED_ROOT.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in allowed_suffixes
            and any(
                keyword in path.name.lower()
                for keyword in evidence_keywords
            )
            and not is_relative_to(
                path,
                STEP5B_FINAL_AUDIT_ROOT,
            )
            and not is_relative_to(
                path,
                FULL_RUN_CONTROL_ROOT,
            )
            and not is_relative_to(
                path,
                FINAL_PACKAGE_ROOT,
            )
            and not is_relative_to(
                path,
                STAGING_PACKAGE_ROOT,
            )
            and not is_relative_to(
                path,
                STEP5C_AUDIT_ROOT,
            )
            and source_path != STEP5C_STATUS_PATH
        )
    ],
    key=lambda path:
        path.relative_to(
            PROJECT_AGGREGATED_ROOT
        ).as_posix(),
):
    register_source(
        category="evidence",
        source_path=source_path,
        destination_tail=source_path.relative_to(
            PROJECT_AGGREGATED_ROOT
        ),
    )


destination_paths = [
    (
        Path("payload")
        / item["Category"]
        / item["DestinationTail"]
    ).as_posix()
    for item in source_inventory
]


duplicate_destination_count = int(
    pd.Series(
        destination_paths
    ).duplicated().sum()
)


if duplicate_destination_count != 0:
    raise RuntimeError(
        "The package source inventory contains duplicate destinations."
    )


# --------------------------------------------------------------------------------------------------
# 11. CREATE CLEAN STAGING PACKAGE
# --------------------------------------------------------------------------------------------------

safe_remove_directory(
    STAGING_PACKAGE_ROOT,
    expected_parent=PROJECT_AGGREGATED_ROOT,
    allowed_name=(
        f".{PROJECT_SHORT_NAME}_final_package_staging"
    ),
)


STAGING_PACKAGE_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


package_records = []
copy_size_mismatches = []
copy_hash_mismatches = []


for item in source_inventory:
    source_path = item[
        "SourcePath"
    ]

    relative_destination = (
        Path("payload")
        / item["Category"]
        / item["DestinationTail"]
    )

    destination_path = (
        STAGING_PACKAGE_ROOT
        / relative_destination
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    source_size = int(
        source_path.stat().st_size
    )

    source_sha256 = sha256_file(
        source_path
    )


    shutil.copy2(
        source_path,
        destination_path,
    )


    copied_size = int(
        destination_path.stat().st_size
    )

    copied_sha256 = sha256_file(
        destination_path
    )


    if copied_size != source_size:
        copy_size_mismatches.append(
            relative_destination.as_posix()
        )


    if copied_sha256 != source_sha256:
        copy_hash_mismatches.append(
            relative_destination.as_posix()
        )


    package_records.append({
        "Category":
            item["Category"],

        "RelativePath":
            relative_destination.as_posix(),

        "SourcePath":
            str(source_path),

        "SizeBytes":
            source_size,

        "SHA256":
            source_sha256,

        "Generated":
            False,
    })


if copy_size_mismatches:
    raise RuntimeError(
        "Package-copy size mismatches occurred:\n"
        + "\n".join(
            copy_size_mismatches
        )
    )


if copy_hash_mismatches:
    raise RuntimeError(
        "Package-copy SHA-256 mismatches occurred:\n"
        + "\n".join(
            copy_hash_mismatches
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. ADD PACKAGE METADATA
# --------------------------------------------------------------------------------------------------

metadata_root = (
    STAGING_PACKAGE_ROOT
    / "payload"
    / "metadata"
)

metadata_root.mkdir(
    parents=True,
    exist_ok=True,
)


readme_text = f"""PROJECT 10 FINAL EXPERIMENT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}

Experiment:
- Conditions: {EXPECTED_CONDITIONS}
- Noise levels: 0, 5, 10, 15, 20, 25, 30, 40, 50 percent
- Repetition seeds: 1 through 30
- ML fits: {EXPECTED_ML_FITS}
- ML models: RandomForest, XGBoost, LightGBM, NaiveBayes
- Baselines: Random, LatestFail, QTF-Avg
- Primary metric: APFDc
- Secondary metric: APFD
- Ranking rows: {EXPECTED_RANKING_ROWS}
- Build-metric rows: {EXPECTED_BUILD_METRIC_ROWS}
- Project-run rows: {EXPECTED_PROJECT_RUN_ROWS}

Analysis-ready outputs include:
- APFD and APFDc for every scored evaluation build
- seed-level project-run results
- mean, median, minimum, maximum and sample standard deviation
- clean-condition deltas
- absolute and relative degradation summaries
- baseline invariance and response validation

Frozen raw result:
- Files: {EXPECTED_RAW_FILES}
- Bytes: {EXPECTED_RAW_BYTES}
- Root SHA-256: {EXPECTED_RAW_ROOT_SHA256}

The raw condition results remain at:
{PROJECT_RAW_ROOT}

This compact package contains:
- Project 10 protocol checkpoints
- candidate-selection evidence
- full-run control records
- compact Step 5B analysis-ready outputs
- frozen raw-file manifest
- package metadata and validation

Registry state:
PENDING SERIAL PROJECT 9 + PROJECT 10 REGISTRY INSERTION.

Do not rerun Project 10.
"""


readme_path = (
    metadata_root
    / "README.txt"
)

readme_path.write_text(
    readme_text,
    encoding="utf-8",
)


project_summary_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ExperimentComplete":
        True,

    "AnalysisReady":
        True,

    "DoNotRerun":
        True,

    "RegistryState":
        (
            "PENDING_SERIAL_PROJECT_9_"
            "AND_PROJECT_10_INSERTION"
        ),

    "Conditions":
        EXPECTED_CONDITIONS,

    "NoiseLevelsPercent":
        [
            0,
            5,
            10,
            15,
            20,
            25,
            30,
            40,
            50,
        ],

    "RepetitionSeeds":
        list(
            range(
                1,
                31,
            )
        ),

    "MLTechniques":
        [
            "RandomForest",
            "XGBoost",
            "LightGBM",
            "NaiveBayes",
        ],

    "Baselines":
        [
            "Random",
            "LatestFail",
            "QTF-Avg",
        ],

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "StandardDeviation":
        (
            "Sample standard deviation across 30 seeds; "
            "pandas std with ddof=1"
        ),

    "MLFits":
        EXPECTED_ML_FITS,

    "RankingRows":
        EXPECTED_RANKING_ROWS,

    "BuildMetricRows":
        EXPECTED_BUILD_METRIC_ROWS,

    "ProjectRunRows":
        EXPECTED_PROJECT_RUN_ROWS,

    "ConditionAuditRows":
        EXPECTED_CONDITION_AUDIT_ROWS,

    "TrainingMedianRows":
        EXPECTED_TRAINING_MEDIAN_ROWS,

    "ActivePredictors":
        EXPECTED_ACTIVE_PREDICTORS,

    "NoiseTechniqueSummaryRows":
        EXPECTED_NOISE_TECHNIQUE_ROWS,

    "SeedLevelNoiseDeltaRows":
        EXPECTED_SEED_LEVEL_DELTA_ROWS,

    "NoiseDeltaSummaryRows":
        EXPECTED_NOISE_DELTA_ROWS,

    "RawFiles":
        EXPECTED_RAW_FILES,

    "RawBytes":
        EXPECTED_RAW_BYTES,

    "RawRootSHA256":
        EXPECTED_RAW_ROOT_SHA256,

    "Step5BCheckpointSHA256":
        EXPECTED_STEP5B_CHECKPOINT_SHA256,

    "Step5BStatus":
        EXPECTED_STEP5B_STATUS,

    "DiscoveredProject10Checkpoints":
        [
            path.name
            for path in checkpoint_sources
        ],
}


project_summary_path = (
    metadata_root
    / "project_summary.json"
)

project_summary_path.write_text(
    json.dumps(
        project_summary_payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


for generated_path in [
    readme_path,
    project_summary_path,
]:
    relative_path = generated_path.relative_to(
        STAGING_PACKAGE_ROOT
    )

    package_records.append({
        "Category":
            "metadata",

        "RelativePath":
            relative_path.as_posix(),

        "SourcePath":
            "GENERATED_PACKAGE_METADATA",

        "SizeBytes":
            int(
                generated_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                generated_path
            ),

        "Generated":
            True,
    })


# --------------------------------------------------------------------------------------------------
# 13. FREEZE PAYLOAD MANIFEST
# --------------------------------------------------------------------------------------------------

package_manifest = pd.DataFrame(
    package_records
)


package_manifest = (
    package_manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if package_manifest[
    "RelativePath"
].duplicated().any():
    raise RuntimeError(
        "The package manifest contains duplicate payload paths."
    )


package_payload_root_sha256 = canonical_root_hash(
    package_manifest[
        [
            "RelativePath",
            "SizeBytes",
            "SHA256",
        ]
    ]
)


package_payload_files = len(
    package_manifest
)

package_payload_bytes = int(
    package_manifest[
        "SizeBytes"
    ].sum()
)


staging_manifest_path = (
    STAGING_PACKAGE_ROOT
    / "package_manifest.csv"
)


package_manifest.to_csv(
    staging_manifest_path,
    index=False,
)


package_manifest_sha256 = sha256_file(
    staging_manifest_path
)


# --------------------------------------------------------------------------------------------------
# 14. REPLACE ONLY THE PROJECT 10 FINAL PACKAGE
# --------------------------------------------------------------------------------------------------

safe_remove_directory(
    FINAL_PACKAGE_ROOT,
    expected_parent=PROJECT_AGGREGATED_ROOT,
    allowed_name=(
        f"{PROJECT_SHORT_NAME}_final_package"
    ),
)


shutil.move(
    str(STAGING_PACKAGE_ROOT),
    str(FINAL_PACKAGE_ROOT),
)


FINAL_PACKAGE_MANIFEST_PATH = (
    FINAL_PACKAGE_ROOT
    / "package_manifest.csv"
)


# --------------------------------------------------------------------------------------------------
# 15. VALIDATE COPIED PAYLOAD
# --------------------------------------------------------------------------------------------------

final_manifest = pd.read_csv(
    FINAL_PACKAGE_MANIFEST_PATH,
    low_memory=False,
)


payload_actual_paths = sorted([
    path.relative_to(
        FINAL_PACKAGE_ROOT
    ).as_posix()
    for path in (
        FINAL_PACKAGE_ROOT
        / "payload"
    ).rglob("*")
    if path.is_file()
])


payload_manifest_paths = sorted(
    final_manifest[
        "RelativePath"
    ].astype(str).tolist()
)


missing_package_files = sorted(
    set(payload_manifest_paths)
    - set(payload_actual_paths)
)

unexpected_package_files = sorted(
    set(payload_actual_paths)
    - set(payload_manifest_paths)
)


package_size_mismatches = []
package_hash_mismatches = []
payload_readback_records = []


for row in final_manifest.itertuples(
    index=False
):
    file_path = (
        FINAL_PACKAGE_ROOT
        / row.RelativePath
    )

    if not file_path.is_file():
        continue

    actual_size = int(
        file_path.stat().st_size
    )

    actual_sha256 = sha256_file(
        file_path
    )


    if actual_size != int(
        row.SizeBytes
    ):
        package_size_mismatches.append(
            row.RelativePath
        )


    if actual_sha256 != str(
        row.SHA256
    ):
        package_hash_mismatches.append(
            row.RelativePath
        )


    payload_readback_records.append({
        "RelativePath":
            row.RelativePath,

        "SizeBytes":
            actual_size,

        "SHA256":
            actual_sha256,
    })


payload_readback_manifest = pd.DataFrame(
    payload_readback_records
)


payload_readback_root_sha256 = canonical_root_hash(
    payload_readback_manifest
)


# --------------------------------------------------------------------------------------------------
# 16. VALIDATION BEFORE CONTROL FILES
# --------------------------------------------------------------------------------------------------

registry_sha256_after_payload = sha256_file(
    COMPLETION_REGISTRY_PATH
)


registry_after_payload = (
    pd.read_csv(
        COMPLETION_REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_after_project_numbers = pd.to_numeric(
    registry_after_payload[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


registry_project9_rows_after_payload = int(
    registry_after_project_numbers.eq(9).sum()
)

registry_project10_rows_after_payload = int(
    registry_after_project_numbers.eq(10).sum()
)


validation_records = []


add_check(
    validation_records,
    "Step 5A status",
    EXPECTED_STEP5A_STATUS,
    step5a_status,
    step5a_status == EXPECTED_STEP5A_STATUS,
)

add_check(
    validation_records,
    "Step 5B checkpoint status",
    EXPECTED_STEP5B_STATUS,
    step5b_checkpoint_status,
    step5b_checkpoint_status
    == EXPECTED_STEP5B_STATUS,
)

add_check(
    validation_records,
    "Step 5B status-file status",
    EXPECTED_STEP5B_STATUS,
    step5b_status,
    step5b_status
    == EXPECTED_STEP5B_STATUS,
)

add_check(
    validation_records,
    "Step 5B checkpoint SHA-256",
    EXPECTED_STEP5B_CHECKPOINT_SHA256,
    step5b_checkpoint_sha256,
    step5b_checkpoint_sha256
    == EXPECTED_STEP5B_CHECKPOINT_SHA256,
)

add_check(
    validation_records,
    "Required checkpoints missing",
    0,
    len(missing_required_checkpoints),
    len(missing_required_checkpoints) == 0,
)

add_check(
    validation_records,
    "Discovered Project 10 checkpoints",
    ">= 7",
    len(checkpoint_sources),
    len(checkpoint_sources) >= 7,
)

add_check(
    validation_records,
    "Frozen raw files",
    EXPECTED_RAW_FILES,
    current_raw_file_count,
    current_raw_file_count
    == EXPECTED_RAW_FILES,
)

add_check(
    validation_records,
    "Frozen raw bytes",
    EXPECTED_RAW_BYTES,
    current_raw_bytes,
    current_raw_bytes
    == EXPECTED_RAW_BYTES,
)

add_check(
    validation_records,
    "Frozen raw-root SHA-256",
    EXPECTED_RAW_ROOT_SHA256,
    current_raw_root_sha256,
    current_raw_root_sha256
    == EXPECTED_RAW_ROOT_SHA256,
)

add_check(
    validation_records,
    "Raw files missing",
    0,
    len(missing_raw_files),
    len(missing_raw_files) == 0,
)

add_check(
    validation_records,
    "Unexpected raw files",
    0,
    len(unexpected_raw_files),
    len(unexpected_raw_files) == 0,
)

add_check(
    validation_records,
    "Raw size mismatches",
    0,
    len(raw_size_mismatches),
    len(raw_size_mismatches) == 0,
)

add_check(
    validation_records,
    "Raw SHA-256 mismatches",
    0,
    len(raw_hash_mismatches),
    len(raw_hash_mismatches) == 0,
)

add_check(
    validation_records,
    "Conditions",
    EXPECTED_CONDITIONS,
    len(condition_inventory),
    len(condition_inventory)
    == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "ML fits",
    EXPECTED_ML_FITS,
    len(model_fits_all),
    len(model_fits_all)
    == EXPECTED_ML_FITS,
)

add_check(
    validation_records,
    "Build-metric rows",
    EXPECTED_BUILD_METRIC_ROWS,
    len(build_metrics_all),
    len(build_metrics_all)
    == EXPECTED_BUILD_METRIC_ROWS,
)

add_check(
    validation_records,
    "Project-run rows",
    EXPECTED_PROJECT_RUN_ROWS,
    len(project_run_all),
    len(project_run_all)
    == EXPECTED_PROJECT_RUN_ROWS,
)

add_check(
    validation_records,
    "Condition-audit rows",
    EXPECTED_CONDITION_AUDIT_ROWS,
    len(condition_audit_all),
    len(condition_audit_all)
    == EXPECTED_CONDITION_AUDIT_ROWS,
)

add_check(
    validation_records,
    "Training-median rows",
    EXPECTED_TRAINING_MEDIAN_ROWS,
    len(training_medians_all),
    len(training_medians_all)
    == EXPECTED_TRAINING_MEDIAN_ROWS,
)

add_check(
    validation_records,
    "Active predictors",
    EXPECTED_ACTIVE_PREDICTORS,
    active_predictor_count,
    active_predictor_count
    == EXPECTED_ACTIVE_PREDICTORS,
)

add_check(
    validation_records,
    "Noise-technique summary rows",
    EXPECTED_NOISE_TECHNIQUE_ROWS,
    len(noise_technique_summary),
    len(noise_technique_summary)
    == EXPECTED_NOISE_TECHNIQUE_ROWS,
)

add_check(
    validation_records,
    "Seed-level delta rows",
    EXPECTED_SEED_LEVEL_DELTA_ROWS,
    len(seed_level_deltas),
    len(seed_level_deltas)
    == EXPECTED_SEED_LEVEL_DELTA_ROWS,
)

add_check(
    validation_records,
    "Noise-delta rows",
    EXPECTED_NOISE_DELTA_ROWS,
    len(noise_delta_summary),
    len(noise_delta_summary)
    == EXPECTED_NOISE_DELTA_ROWS,
)

add_check(
    validation_records,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)

add_check(
    validation_records,
    "Duplicate package destinations",
    0,
    duplicate_destination_count,
    duplicate_destination_count == 0,
)

add_check(
    validation_records,
    "Package-copy size mismatches",
    0,
    len(copy_size_mismatches),
    len(copy_size_mismatches) == 0,
)

add_check(
    validation_records,
    "Package-copy SHA-256 mismatches",
    0,
    len(copy_hash_mismatches),
    len(copy_hash_mismatches) == 0,
)

add_check(
    validation_records,
    "Payload files",
    package_payload_files,
    len(payload_actual_paths),
    len(payload_actual_paths)
    == package_payload_files,
)

add_check(
    validation_records,
    "Missing payload files",
    0,
    len(missing_package_files),
    len(missing_package_files) == 0,
)

add_check(
    validation_records,
    "Unexpected payload files",
    0,
    len(unexpected_package_files),
    len(unexpected_package_files) == 0,
)

add_check(
    validation_records,
    "Payload size mismatches",
    0,
    len(package_size_mismatches),
    len(package_size_mismatches) == 0,
)

add_check(
    validation_records,
    "Payload SHA-256 mismatches",
    0,
    len(package_hash_mismatches),
    len(package_hash_mismatches) == 0,
)

add_check(
    validation_records,
    "Payload root SHA-256",
    package_payload_root_sha256,
    payload_readback_root_sha256,
    payload_readback_root_sha256
    == package_payload_root_sha256,
)

add_check(
    validation_records,
    "Completion registry unchanged",
    registry_sha256_before,
    registry_sha256_after_payload,
    registry_sha256_after_payload
    == registry_sha256_before,
)

add_check(
    validation_records,
    "Registry Project 9 rows",
    0,
    registry_project9_rows_after_payload,
    registry_project9_rows_after_payload == 0,
)

add_check(
    validation_records,
    "Registry Project 10 rows",
    0,
    registry_project10_rows_after_payload,
    registry_project10_rows_after_payload == 0,
)

add_check(
    validation_records,
    "Registry update performed",
    False,
    False,
    True,
)

add_check(
    validation_records,
    "Project 9 accessed",
    False,
    False,
    True,
)

add_check(
    validation_records,
    "Project 9 write attempted",
    False,
    False,
    True,
)


package_validation = pd.DataFrame(
    validation_records
)


failed_checks = int(
    (
        ~package_validation[
            "Pass"
        ]
    ).sum()
)


if failed_checks != 0:
    print("\nFailed package checks:")

    display(
        package_validation[
            ~package_validation[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "PROJECT 10 STEP 5C PACKAGE VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 17. WRITE CONTROL FILES INSIDE FINAL PACKAGE
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


PACKAGE_VALIDATION_PATH = (
    FINAL_PACKAGE_ROOT
    / "package_validation.csv"
)

PACKAGE_REPORT_PATH = (
    FINAL_PACKAGE_ROOT
    / "package_report.json"
)

PACKAGE_STATUS_PATH = (
    FINAL_PACKAGE_ROOT
    / "package_status.json"
)


package_validation.to_csv(
    PACKAGE_VALIDATION_PATH,
    index=False,
)


package_report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        FINAL_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "ExperimentComplete":
        True,

    "AnalysisReady":
        True,

    "DoNotRerun":
        True,

    "RegistryUpdatePerformed":
        False,

    "RegistryState":
        (
            "PENDING_SERIAL_PROJECT_9_"
            "AND_PROJECT_10_INSERTION"
        ),

    "Conditions":
        EXPECTED_CONDITIONS,

    "MLFits":
        EXPECTED_ML_FITS,

    "RankingRows":
        EXPECTED_RANKING_ROWS,

    "BuildMetricRows":
        EXPECTED_BUILD_METRIC_ROWS,

    "ProjectRunRows":
        EXPECTED_PROJECT_RUN_ROWS,

    "RawFiles":
        current_raw_file_count,

    "RawBytes":
        current_raw_bytes,

    "RawRootSHA256":
        current_raw_root_sha256,

    "DiscoveredCheckpointFiles":
        [
            path.name
            for path in checkpoint_sources
        ],

    "PayloadFiles":
        package_payload_files,

    "PayloadBytes":
        package_payload_bytes,

    "PayloadRootSHA256":
        payload_readback_root_sha256,

    "PackageManifest":
        str(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "PackageManifestSHA256":
        sha256_file(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "PackageValidationChecks":
        len(package_validation),

    "PackageValidationFailures":
        failed_checks,

    "CompletionRegistryModified":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,
}


atomic_write_json(
    PACKAGE_REPORT_PATH,
    package_report_payload,
)


package_status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        FINAL_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "DoNotRerun":
        True,

    "RegistryPending":
        True,

    "RegistryUpdatePerformed":
        False,

    "PayloadRootSHA256":
        payload_readback_root_sha256,
}


atomic_write_json(
    PACKAGE_STATUS_PATH,
    package_status_payload,
)


# --------------------------------------------------------------------------------------------------
# 18. FREEZE COMPLETE FINAL PACKAGE TREE
# --------------------------------------------------------------------------------------------------

final_package_inventory = build_tree_manifest(
    FINAL_PACKAGE_ROOT
)


final_package_files = len(
    final_package_inventory
)

final_package_bytes = int(
    final_package_inventory[
        "SizeBytes"
    ].sum()
)

final_package_root_sha256 = canonical_root_hash(
    final_package_inventory
)


STEP5C_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FINAL_PACKAGE_INVENTORY_PATH,
    final_package_inventory,
)


final_validation_records = validation_records.copy()


add_check(
    final_validation_records,
    "Complete final-package files",
    final_package_files,
    final_package_files,
    final_package_files > 0,
)

add_check(
    final_validation_records,
    "Complete final-package bytes",
    final_package_bytes,
    final_package_bytes,
    final_package_bytes > 0,
)

add_check(
    final_validation_records,
    "Final package root generated",
    True,
    bool(final_package_root_sha256),
    len(final_package_root_sha256) == 64,
)


final_validation = pd.DataFrame(
    final_validation_records
)


final_failed_checks = int(
    (
        ~final_validation[
            "Pass"
        ]
    ).sum()
)


if final_failed_checks != 0:
    raise RuntimeError(
        "PROJECT 10 STEP 5C FINAL VALIDATION FAILED."
    )


atomic_write_csv(
    STEP5C_VALIDATION_PATH,
    final_validation,
)


# --------------------------------------------------------------------------------------------------
# 19. WRITE EXTERNAL REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

step5c_report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        FINAL_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "ExperimentStatus":
        "COMPLETE_ANALYSIS_READY",

    "RegistryStatus":
        (
            "PENDING_SERIAL_PROJECT_9_"
            "AND_PROJECT_10_INSERTION"
        ),

    "DoNotRerun":
        True,

    "Conditions":
        EXPECTED_CONDITIONS,

    "NoiseLevels":
        9,

    "RepetitionSeeds":
        30,

    "MLFits":
        EXPECTED_ML_FITS,

    "RankingRows":
        EXPECTED_RANKING_ROWS,

    "BuildMetricRows":
        EXPECTED_BUILD_METRIC_ROWS,

    "ProjectRunRows":
        EXPECTED_PROJECT_RUN_ROWS,

    "RawFiles":
        current_raw_file_count,

    "RawBytes":
        current_raw_bytes,

    "RawRootSHA256":
        current_raw_root_sha256,

    "DiscoveredCheckpointCount":
        len(checkpoint_sources),

    "DiscoveredCheckpointFiles":
        [
            path.name
            for path in checkpoint_sources
        ],

    "PayloadFiles":
        package_payload_files,

    "PayloadBytes":
        package_payload_bytes,

    "PayloadRootSHA256":
        payload_readback_root_sha256,

    "FinalPackageFiles":
        final_package_files,

    "FinalPackageBytes":
        final_package_bytes,

    "FinalPackageRootSHA256":
        final_package_root_sha256,

    "FinalPackageDirectory":
        str(FINAL_PACKAGE_ROOT),

    "FinalPackageInventory":
        str(
            FINAL_PACKAGE_INVENTORY_PATH
        ),

    "FinalPackageInventorySHA256":
        sha256_file(
            FINAL_PACKAGE_INVENTORY_PATH
        ),

    "PackageManifest":
        str(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "PackageManifestSHA256":
        sha256_file(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "ValidationChecks":
        len(final_validation),

    "FailedValidationChecks":
        final_failed_checks,

    "CompletionRegistry":
        str(
            COMPLETION_REGISTRY_PATH
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "RegistryProject9Rows":
        0,

    "RegistryProject10Rows":
        0,

    "RegistryUpdatePerformed":
        False,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,

    "Projects1To8Modified":
        False,
}


atomic_write_json(
    STEP5C_REPORT_PATH,
    step5c_report_payload,
)


final_package_checkpoint_payload = {
    **step5c_report_payload,

    "Step5AStatusFile":
        str(
            STEP5A_STATUS_PATH
        ),

    "Step5AStatusSHA256":
        sha256_file(
            STEP5A_STATUS_PATH
        ),

    "Step5BCheckpoint":
        str(
            STEP5B_CHECKPOINT_PATH
        ),

    "Step5BCheckpointSHA256":
        step5b_checkpoint_sha256,

    "Step5CReport":
        str(
            STEP5C_REPORT_PATH
        ),

    "Step5CReportSHA256":
        sha256_file(
            STEP5C_REPORT_PATH
        ),

    "Step5CValidation":
        str(
            STEP5C_VALIDATION_PATH
        ),

    "Step5CValidationSHA256":
        sha256_file(
            STEP5C_VALIDATION_PATH
        ),
}


atomic_write_json(
    FINAL_PACKAGE_CHECKPOINT_PATH,
    final_package_checkpoint_payload,
)


step5c_status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        FINAL_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "FinalPackageFiles":
        final_package_files,

    "FinalPackageBytes":
        final_package_bytes,

    "FinalPackageRootSHA256":
        final_package_root_sha256,

    "Checkpoint":
        str(
            FINAL_PACKAGE_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        sha256_file(
            FINAL_PACKAGE_CHECKPOINT_PATH
        ),

    "RegistryUpdatePerformed":
        False,

    "RegistryPending":
        True,

    "DoNotRerun":
        True,

    "Project9Accessed":
        False,

    "Project9WriteAttempted":
        False,
}


atomic_write_json(
    STEP5C_STATUS_PATH,
    step5c_status_payload,
)


# --------------------------------------------------------------------------------------------------
# 20. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_final = sha256_file(
    COMPLETION_REGISTRY_PATH
)


if registry_sha256_final != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 10 Step 5C."
    )


registry_final = (
    pd.read_csv(
        COMPLETION_REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_final_project_numbers = pd.to_numeric(
    registry_final[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


registry_project9_rows_final = int(
    registry_final_project_numbers.eq(9).sum()
)

registry_project10_rows_final = int(
    registry_final_project_numbers.eq(10).sum()
)


if (
    registry_project9_rows_final != 0
    or registry_project10_rows_final != 0
):
    raise RuntimeError(
        "Project 9 or Project 10 was unexpectedly "
        "inserted into the registry."
    )


final_package_inventory_readback = build_tree_manifest(
    FINAL_PACKAGE_ROOT
)

final_package_root_sha256_readback = canonical_root_hash(
    final_package_inventory_readback
)


if (
    final_package_root_sha256_readback
    != final_package_root_sha256
):
    raise RuntimeError(
        "The Project 10 final package changed after freezing."
    )


# Recheck frozen raw root after package construction.

final_raw_records = []


for row in raw_manifest.itertuples(
    index=False
):
    file_path = (
        PROJECT_RAW_ROOT
        / row.RelativePath
    )

    final_raw_records.append({
        "RelativePath":
            row.RelativePath,

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                file_path
            ),
    })


final_raw_root_sha256 = canonical_root_hash(
    pd.DataFrame(
        final_raw_records
    )
)


if final_raw_root_sha256 != EXPECTED_RAW_ROOT_SHA256:
    raise RuntimeError(
        "The Project 10 raw root changed during package construction."
    )


checkpoint_readback = load_json(
    FINAL_PACKAGE_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP5C_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != FINAL_STATUS:
    raise RuntimeError(
        "Final package checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != FINAL_STATUS:
    raise RuntimeError(
        "Project 10 Step 5C status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 21. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nProject 10 Step 5C validation:")

display(
    final_validation
)


print("\nDiscovered Project 10 checkpoints:")

display(
    pd.DataFrame({
        "CheckpointFile":
            [
                path.name
                for path in checkpoint_sources
            ],

        "SHA256":
            [
                sha256_file(path)
                for path in checkpoint_sources
            ],
    })
)


print("\nFinal package inventory sample:")

display(
    pd.concat(
        [
            final_package_inventory.head(15),
            final_package_inventory.tail(15),
        ],
        ignore_index=True,
    )
)


# --------------------------------------------------------------------------------------------------
# 22. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 10 CELL 11 / STEP 5C RESULT ===")
print("=" * 132)


print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nFrozen experiment:")

print(
    "Conditions:",
    EXPECTED_CONDITIONS,
)

print(
    "Noise levels:",
    9,
)

print(
    "Repetition seeds:",
    30,
)

print(
    "ML fits:",
    EXPECTED_ML_FITS,
)

print(
    "Ranking rows:",
    EXPECTED_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    EXPECTED_BUILD_METRIC_ROWS,
)

print(
    "Project-run rows:",
    EXPECTED_PROJECT_RUN_ROWS,
)


print("\nFrozen raw result:")

print(
    "Raw files:",
    current_raw_file_count,
)

print(
    "Raw bytes:",
    current_raw_bytes,
)

print(
    "Raw root SHA-256:",
    final_raw_root_sha256,
)


print("\nCheckpoint discovery:")

print(
    "Project 10 checkpoints discovered:",
    len(checkpoint_sources),
)

print(
    "Missing required checkpoints:",
    len(missing_required_checkpoints),
)


print("\nFinal compact package:")

print(
    "Package directory:",
    FINAL_PACKAGE_ROOT,
)

print(
    "Payload files:",
    package_payload_files,
)

print(
    "Payload bytes:",
    package_payload_bytes,
)

print(
    "Payload root SHA-256:",
    payload_readback_root_sha256,
)

print(
    "Complete package files:",
    final_package_files,
)

print(
    "Complete package bytes:",
    final_package_bytes,
)

print(
    "Final package root SHA-256:",
    final_package_root_sha256,
)

print(
    "Missing package files:",
    len(missing_package_files),
)

print(
    "Unexpected package files:",
    len(unexpected_package_files),
)

print(
    "Package size mismatches:",
    len(package_size_mismatches),
)

print(
    "Package SHA-256 mismatches:",
    len(package_hash_mismatches),
)


print("\nCompletion registry:")

print(
    "Registry unchanged:",
    registry_sha256_final
    == registry_sha256_before,
)

print(
    "Registry Project 9 rows:",
    registry_project9_rows_final,
)

print(
    "Registry Project 10 rows:",
    registry_project10_rows_final,
)

print(
    "Registry update performed:",
    False,
)


print("\nIsolation:")

print(
    "Project 9 accessed:",
    False,
)

print(
    "Project 9 write attempted:",
    False,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nValidation:")

print(
    "Checks:",
    len(final_validation),
)

print(
    "Failed checks:",
    final_failed_checks,
)


print("\nFinal package checkpoint:")

print(
    FINAL_PACKAGE_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    sha256_file(
        FINAL_PACKAGE_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    FINAL_STATUS,
)

print("=" * 132)

=== PROJECT 10 CELL 11 / STEP 5C: FINAL PACKAGE CONSTRUCTION AND VALIDATION ===

Project 10 checkpoints discovered:
 - project_10_model_protocol_checkpoint.json
 - project_10_noise_plan_checkpoint.json
 - project_10_noisy_rec_engine_checkpoint.json
 - project_10_rec_reconstruction_checkpoint.json
 - project_10_selection_checkpoint.json
 - project_10_smoke_test_checkpoint.json
 - project_10_step5b_checkpoint.json

Project 10 Step 5C validation:


,Check,Expected,Actual,Pass
0,Step 5A status,PASS_PROJECT_10_FULL_270_CONDITION_RUN_COMPLETE,PASS_PROJECT_10_FULL_270_CONDITION_RUN_COMPLETE,True
1,Step 5B checkpoint status,PASS_PROJECT_10_RAW_RESULTS_REVALIDATED_AND_CO...,PASS_PROJECT_10_RAW_RESULTS_REVALIDATED_AND_CO...,True
2,Step 5B status-file status,PASS_PROJECT_10_RAW_RESULTS_REVALIDATED_AND_CO...,PASS_PROJECT_10_RAW_RESULTS_REVALIDATED_AND_CO...,True
3,Step 5B checkpoint SHA-256,38ed74dac0892ba69efb418397c27e926c221880e39398...,38ed74dac0892ba69efb418397c27e926c221880e39398...,True
4,Required checkpoints missing,0,0,True
5,Discovered Project 10 checkpoints,>= 7,7,True
6,Frozen raw files,2160,2160,True
7,Frozen raw bytes,68638025,68638025,True
8,Frozen raw-root SHA-256,5d336e1c7630cd6dec8ecdd8e17fa1af5cd6547d6cf90e...,5d336e1c7630cd6dec8ecdd8e17fa1af5cd6547d6cf90e...,True
9,Raw files missing,0,0,True



Discovered Project 10 checkpoints:


,CheckpointFile,SHA256
0,project_10_model_protocol_checkpoint.json,5cefdd0fafb4a8509009cebf14ce66bb233c32f2aea7af...
1,project_10_noise_plan_checkpoint.json,c4b3008d6e8844474a2618a53edd8c839142489024b01f...
2,project_10_noisy_rec_engine_checkpoint.json,36f9aa3611621827cb96a754df0ceaaef3226dd33b8f86...
3,project_10_rec_reconstruction_checkpoint.json,9f4bec10905380b9a90326f0e43e3acd2f7ab30b66519c...
4,project_10_selection_checkpoint.json,97960ad7032d7c8b9dfb0a3bb45ba6d3cd64b648eadf0a...
5,project_10_smoke_test_checkpoint.json,d31c78289fbc5d2c0d00f4711062eeb6a00ce394016c93...
6,project_10_step5b_checkpoint.json,38ed74dac0892ba69efb418397c27e926c221880e39398...



Final package inventory sample:


,RelativePath,SizeBytes,SHA256
0,package_manifest.csv,28213,404372358d33c0c92521c221a1a977b719c9c1ae0386e4...
1,package_report.json,1661,b7761608544784011c5d8059c5416dd53be2e2e8ebb7b5...
2,package_status.json,453,0b156fffb86be2c9a6deb929ad80f133d9793b977642fc...
3,package_validation.csv,2229,cb3adbda660a4239df3b1c3fe7bcab1b6c92f4ef4c2066...
4,payload/checkpoints/project_10_model_protocol_...,22024,5cefdd0fafb4a8509009cebf14ce66bb233c32f2aea7af...
5,payload/checkpoints/project_10_noise_plan_chec...,6237,c4b3008d6e8844474a2618a53edd8c839142489024b01f...
6,payload/checkpoints/project_10_noisy_rec_engin...,4729,36f9aa3611621827cb96a754df0ceaaef3226dd33b8f86...
7,payload/checkpoints/project_10_rec_reconstruct...,7218,9f4bec10905380b9a90326f0e43e3acd2f7ab30b66519c...
8,payload/checkpoints/project_10_selection_check...,5976,97960ad7032d7c8b9dfb0a3bb45ba6d3cd64b648eadf0a...
9,payload/checkpoints/project_10_smoke_test_chec...,4304,d31c78289fbc5d2c0d00f4711062eeb6a00ce394016c93...




=== PROJECT 10 CELL 11 / STEP 5C RESULT ===

Project identity:
Project number: 10
Project: spring-cloud@spring-cloud-dataflow
Project slug: spring-cloud__spring-cloud-dataflow

Frozen experiment:
Conditions: 270
Noise levels: 9
Repetition seeds: 30
ML fits: 1080
Ranking rows: 4934790
Build-metric rows: 51030
Project-run rows: 1890

Frozen raw result:
Raw files: 2160
Raw bytes: 68638025
Raw root SHA-256: 5d336e1c7630cd6dec8ecdd8e17fa1af5cd6547d6cf90ecc19d0dbdce43fea6c

Checkpoint discovery:
Project 10 checkpoints discovered: 7
Missing required checkpoints: 0

Final compact package:
Package directory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/spring-cloud__spring-cloud-dataflow/spring_cloud_dataflow_final_package
Payload files: 88
Payload bytes: 133526559
Payload root SHA-256: f487babdd5a7af4d7c25ceb9f649c8da58d81e9c43491d264f8353c090b0b65e
Complete package files: 92
Complete package bytes: 133559115
Final package root SHA-256: 2682e51a689abc8a2cfb12b07b7c1fe0f450d2ac

In [15]:
# ==================================================================================================
# SERIAL FINALISATION — PROJECTS 9 AND 10
# ATOMIC COMPLETION-REGISTRY INSERTION
#
# Registers:
#   Project 9  — camunda@camunda-bpm-platform
#   Project 10 — spring-cloud@spring-cloud-dataflow
#
# This cell:
# - validates both frozen final packages and checkpoints
# - validates the current registry contains exactly Projects 1–8
# - creates a registry backup
# - inserts Projects 9 and 10 together in one atomic write
# - validates Projects 1–10 are COMPLETE_AND_FROZEN
# - creates a registry-finalisation checkpoint
#
# This cell does NOT:
# - rerun any experiment
# - modify raw results
# - modify either frozen final package
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import shutil

import pandas as pd


print("=" * 132)
print("=== SERIAL FINALISATION: REGISTER PROJECTS 9 AND 10 AS COMPLETE_AND_FROZEN ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. PATHS AND FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

LOCK_PATH = (
    NOTES_ROOT
    / ".projects_09_10_registry_finalisation.lock"
)

FINALISATION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "projects_09_10_registry_finalisation_checkpoint.json"
)

FINALISATION_STATUS_PATH = (
    NOTES_ROOT
    / "projects_09_10_registry_finalisation_status.json"
)


EXPECTED_PRE_INSERTION_REGISTRY_SHA256 = (
    "a442446f3ca6207b31213fc422fb0c883"
    "c96608a8e6a3907da969e326808bee7"
)

COMPLETE_STATUS = (
    "COMPLETE_AND_FROZEN"
)

FINAL_STATUS = (
    "PASS_PROJECTS_09_10_REGISTERED_COMPLETE_AND_FROZEN"
)


PROJECT_9 = {
    "ProjectNumber":
        9,

    "Project":
        "camunda@camunda-bpm-platform",

    "ProjectSlug":
        "camunda__camunda-bpm-platform",

    "Conditions":
        270,

    "NoiseLevels":
        9,

    "RepetitionSeeds":
        30,

    "MLTechniques":
        4,

    "Baselines":
        3,

    "Techniques":
        7,

    "MLFits":
        1080,

    "RankingRows":
        34_970_670,

    "BuildMetricRows":
        56_700,

    "ProjectRunRows":
        1_890,

    "RawFiles":
        2_160,

    "RawBytes":
        387_066_081,

    "RawRootSHA256":
        (
            "c31cb45e1354dc1222b82103bd275c21"
            "b10dd2ba6171125005d9c0a99ab14724"
        ),

    "RawRoot":
        (
            "/content/drive/MyDrive/Thesis_Experiment/"
            "Results/Raw/camunda__camunda-bpm-platform"
        ),

    "PackageFiles":
        89,

    "PackageBytes":
        5_234_900,

    "FinalPackageRootSHA256":
        (
            "600ec7a8b4013764eee49a7351337de4"
            "b773da6bc32902a38a58e95ed79147ff"
        ),

    "FinalPackageDirectory":
        (
            "/content/drive/MyDrive/Thesis_Experiment/"
            "Results/Aggregated/camunda__camunda-bpm-platform/"
            "camunda_final_package"
        ),

    "FinalPackageCheckpoint":
        (
            "/content/drive/MyDrive/Thesis_Experiment/"
            "Notes/project_09_final_package_checkpoint.json"
        ),

    "FinalPackageCheckpointSHA256":
        (
            "209ed665deb40a1946b2507c173676c8"
            "e0df98b65c071d8cf0d4f242eaf4f0eb"
        ),

    "ExpectedPackageStatus":
        (
            "PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_"
            "AND_VALIDATED_REGISTRY_PENDING"
        ),

    "Step5BCheckpoint":
        (
            "/content/drive/MyDrive/Thesis_Experiment/"
            "Notes/project_09_step5b_checkpoint.json"
        ),

    "Step5BCheckpointSHA256":
        (
            "a1fff6906ac62870689cbf4f3f38f16c"
            "dac04b10e6851190bcb34fbb8d6f8a14"
        ),

    "SelectionCheckpoint":
        (
            "/content/drive/MyDrive/Thesis_Experiment/"
            "Notes/project_09_selection_checkpoint.json"
        ),

    "SelectionCheckpointSHA256":
        (
            "3ef9e66e606772a9679cf727182389e4"
            "19c9f8821512ba1492eb9ebc9259948b"
        ),

    "SourceRootSHA256":
        (
            "65201f596c631d0d7472868d3b23aabc"
            "86fa8f9ef4674b8ac9400fd577ab8f07"
        ),
}


PROJECT_10 = {
    "ProjectNumber":
        10,

    "Project":
        "spring-cloud@spring-cloud-dataflow",

    "ProjectSlug":
        "spring-cloud__spring-cloud-dataflow",

    "Conditions":
        270,

    "NoiseLevels":
        9,

    "RepetitionSeeds":
        30,

    "MLTechniques":
        4,

    "Baselines":
        3,

    "Techniques":
        7,

    "MLFits":
        1080,

    "RankingRows":
        4_934_790,

    "BuildMetricRows":
        51_030,

    "ProjectRunRows":
        1_890,

    "RawFiles":
        2_160,

    "RawBytes":
        68_638_025,

    "RawRootSHA256":
        (
            "5d336e1c7630cd6dec8ecdd8e17fa1af"
            "5cd6547d6cf90ecc19d0dbdce43fea6c"
        ),

    "RawRoot":
        (
            "/content/drive/MyDrive/Thesis_Experiment/"
            "Results/Raw/spring-cloud__spring-cloud-dataflow"
        ),

    "PackageFiles":
        92,

    "PackageBytes":
        133_559_115,

    "FinalPackageRootSHA256":
        (
            "2682e51a689abc8a2cfb12b07b7c1fe0"
            "f450d2acf9e9a5ead3943e7ce3fb6ab6"
        ),

    "FinalPackageDirectory":
        (
            "/content/drive/MyDrive/Thesis_Experiment/"
            "Results/Aggregated/spring-cloud__spring-cloud-dataflow/"
            "spring_cloud_dataflow_final_package"
        ),

    "FinalPackageCheckpoint":
        (
            "/content/drive/MyDrive/Thesis_Experiment/"
            "Notes/project_10_final_package_checkpoint.json"
        ),

    "FinalPackageCheckpointSHA256":
        (
            "73850cf9a6596383214a798b409e3180"
            "38bd08867bff8b5c50fd6c1273863ffc"
        ),

    "ExpectedPackageStatus":
        (
            "PASS_PROJECT_10_FINAL_PACKAGE_CONSTRUCTED_"
            "AND_VALIDATED_REGISTRY_PENDING"
        ),

    "Step5BCheckpoint":
        (
            "/content/drive/MyDrive/Thesis_Experiment/"
            "Notes/project_10_step5b_checkpoint.json"
        ),

    "Step5BCheckpointSHA256":
        (
            "38ed74dac0892ba69efb418397c27e926"
            "c221880e39398d70447aba37d2cb364"
        ),

    "SelectionCheckpoint":
        (
            "/content/drive/MyDrive/Thesis_Experiment/"
            "Notes/project_10_selection_checkpoint.json"
        ),

    "SelectionCheckpointSHA256":
        (
            "97960ad7032d7c8b9dfb0a3bb45ba6d3"
            "cd64b648eadf0ae6169510ba981e7877"
        ),

    "SourceRootSHA256":
        (
            "582f01b3a43b542537b93243e5bb5b8c"
            "ff36c274c6c2a3b12b580090d664206e"
        ),
}


PROJECTS = [
    PROJECT_9,
    PROJECT_10,
]


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def normalise_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def canonical_root_hash(
    manifest,
):
    required_columns = [
        "RelativePath",
        "SizeBytes",
        "SHA256",
    ]

    missing = [
        column
        for column in required_columns
        if column not in manifest.columns
    ]

    if missing:
        raise RuntimeError(
            "Manifest columns are missing:\n"
            + "\n".join(missing)
        )

    digest = hashlib.sha256()

    ordered = manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def build_tree_manifest(root):
    root = Path(root)

    records = []

    for file_path in sorted(
        [
            path
            for path in root.rglob("*")
            if path.is_file()
        ],
        key=lambda path:
            path.relative_to(root).as_posix(),
    ):
        records.append({
            "RelativePath":
                file_path.relative_to(
                    root
                ).as_posix(),

            "SizeBytes":
                int(
                    file_path.stat().st_size
                ),

            "SHA256":
                sha256_file(file_path),
        })

    return pd.DataFrame(records)


def recursive_find(
    value,
    candidate_keys,
):
    candidates = {
        normalise_name(key)
        for key in candidate_keys
    }

    if isinstance(value, dict):
        for key, child in value.items():
            if normalise_name(key) in candidates:
                return child

        for child in value.values():
            result = recursive_find(
                child,
                candidate_keys,
            )

            if result is not None:
                return result

    elif isinstance(value, list):
        for child in value:
            result = recursive_find(
                child,
                candidate_keys,
            )

            if result is not None:
                return result

    return None


def semantic_for_column(column):
    name = normalise_name(column)

    exact = {
        "projectnumber":
            "ProjectNumber",

        "projectno":
            "ProjectNumber",

        "projectid":
            "ProjectNumber",

        "projectindex":
            "ProjectNumber",

        "project":
            "Project",

        "projectname":
            "Project",

        "repository":
            "Project",

        "repo":
            "Project",

        "projectslug":
            "ProjectSlug",

        "slug":
            "ProjectSlug",

        "status":
            "Status",

        "state":
            "Status",

        "completionstatus":
            "Status",

        "registrystatus":
            "Status",

        "conditions":
            "Conditions",

        "conditioncount":
            "Conditions",

        "totalconditions":
            "Conditions",

        "noiselevels":
            "NoiseLevels",

        "noisecount":
            "NoiseLevels",

        "seeds":
            "RepetitionSeeds",

        "repetitionseeds":
            "RepetitionSeeds",

        "seedcount":
            "RepetitionSeeds",

        "mltechniques":
            "MLTechniques",

        "mlmodels":
            "MLTechniques",

        "modelcount":
            "MLTechniques",

        "baselines":
            "Baselines",

        "baselinecount":
            "Baselines",

        "techniques":
            "Techniques",

        "techniquecount":
            "Techniques",

        "mlfits":
            "MLFits",

        "mlfitcount":
            "MLFits",

        "totalmlfits":
            "MLFits",

        "rankingrows":
            "RankingRows",

        "rankings":
            "RankingRows",

        "totalrankingrows":
            "RankingRows",

        "buildmetricrows":
            "BuildMetricRows",

        "buildmetrics":
            "BuildMetricRows",

        "projectrunrows":
            "ProjectRunRows",

        "projectruns":
            "ProjectRunRows",

        "rawfiles":
            "RawFiles",

        "rawfilecount":
            "RawFiles",

        "rawbytes":
            "RawBytes",

        "rawsizebytes":
            "RawBytes",

        "rawrootsha256":
            "RawRootSHA256",

        "rawsha256":
            "RawRootSHA256",

        "rawresultrootsha256":
            "RawRootSHA256",

        "rawroot":
            "RawRoot",

        "rawresultroot":
            "RawRoot",

        "rawdirectory":
            "RawRoot",

        "rawresultdirectory":
            "RawRoot",

        "packagefiles":
            "PackageFiles",

        "packagefilecount":
            "PackageFiles",

        "finalpackagefiles":
            "PackageFiles",

        "packagebytes":
            "PackageBytes",

        "packagesizebytes":
            "PackageBytes",

        "finalpackagebytes":
            "PackageBytes",

        "finalpackagerootsha256":
            "FinalPackageRootSHA256",

        "packagerootsha256":
            "FinalPackageRootSHA256",

        "packagesha256":
            "FinalPackageRootSHA256",

        "finalpackagesha256":
            "FinalPackageRootSHA256",

        "finalpackagedirectory":
            "FinalPackageDirectory",

        "packagedirectory":
            "FinalPackageDirectory",

        "finalpackageroot":
            "FinalPackageDirectory",

        "packageroot":
            "FinalPackageDirectory",

        "finalpackagecheckpoint":
            "FinalPackageCheckpoint",

        "packagecheckpoint":
            "FinalPackageCheckpoint",

        "finalpackagecheckpointpath":
            "FinalPackageCheckpoint",

        "finalpackagecheckpointsha256":
            "FinalPackageCheckpointSHA256",

        "packagecheckpointsha256":
            "FinalPackageCheckpointSHA256",

        "step5bcheckpoint":
            "Step5BCheckpoint",

        "step5bcheckpointpath":
            "Step5BCheckpoint",

        "step5bcheckpointsha256":
            "Step5BCheckpointSHA256",

        "selectioncheckpoint":
            "SelectionCheckpoint",

        "selectioncheckpointpath":
            "SelectionCheckpoint",

        "selectioncheckpointsha256":
            "SelectionCheckpointSHA256",

        "sourcerootsha256":
            "SourceRootSHA256",

        "sourcesha256":
            "SourceRootSHA256",

        "completedatutc":
            "CompletedAtUTC",

        "completiontimeutc":
            "CompletedAtUTC",

        "completiontimestamp":
            "CompletedAtUTC",

        "frozenatutc":
            "CompletedAtUTC",

        "completedat":
            "CompletedAtUTC",
    }

    if name in exact:
        return exact[name]

    if "project" in name and (
        "number" in name
        or name.endswith("no")
        or "index" in name
    ):
        return "ProjectNumber"

    if "slug" in name:
        return "ProjectSlug"

    if "status" in name or name == "state":
        return "Status"

    if "raw" in name and "sha256" in name:
        return "RawRootSHA256"

    if (
        "package" in name
        and "checkpoint" in name
        and "sha256" in name
    ):
        return "FinalPackageCheckpointSHA256"

    if (
        "package" in name
        and "checkpoint" in name
    ):
        return "FinalPackageCheckpoint"

    if (
        "package" in name
        and "sha256" in name
    ):
        return "FinalPackageRootSHA256"

    if (
        "package" in name
        and "files" in name
    ):
        return "PackageFiles"

    if (
        "package" in name
        and (
            "bytes" in name
            or "size" in name
        )
    ):
        return "PackageBytes"

    if (
        "package" in name
        and (
            "root" in name
            or "directory" in name
            or "path" in name
        )
    ):
        return "FinalPackageDirectory"

    if "raw" in name and "files" in name:
        return "RawFiles"

    if "raw" in name and (
        "bytes" in name
        or "size" in name
    ):
        return "RawBytes"

    if "raw" in name and (
        "root" in name
        or "directory" in name
        or "path" in name
    ):
        return "RawRoot"

    if "ranking" in name and "row" in name:
        return "RankingRows"

    if "build" in name and "metric" in name:
        return "BuildMetricRows"

    if "project" in name and "run" in name:
        return "ProjectRunRows"

    if "condition" in name and (
        "count" in name
        or "total" in name
    ):
        return "Conditions"

    if "ml" in name and "fit" in name:
        return "MLFits"

    if (
        "completed" in name
        or "completion" in name
        or "frozenat" in name
    ) and (
        "time" in name
        or "date" in name
        or "utc" in name
        or "timestamp" in name
    ):
        return "CompletedAtUTC"

    return None


def add_check(
    records,
    check,
    expected,
    actual,
    passed,
):
    records.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 3. ACQUIRE SERIAL FINALISATION LOCK
# --------------------------------------------------------------------------------------------------

try:
    lock_descriptor = os.open(
        LOCK_PATH,
        os.O_CREAT
        | os.O_EXCL
        | os.O_WRONLY,
    )

except FileExistsError:
    raise RuntimeError(
        "A registry-finalisation lock already exists:\n"
        f"{LOCK_PATH}\n\n"
        "Do not run two registry writers concurrently."
    )


try:
    os.write(
        lock_descriptor,
        json.dumps({
            "PID":
                os.getpid(),

            "CreatedAtUTC":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }).encode("utf-8"),
    )

finally:
    os.close(lock_descriptor)


try:

    # ----------------------------------------------------------------------------------------------
    # 4. VALIDATE FROZEN PROJECT PACKAGES
    # ----------------------------------------------------------------------------------------------

    package_validation_records = []


    for project in PROJECTS:
        checkpoint_path = Path(
            project[
                "FinalPackageCheckpoint"
            ]
        )

        package_root = Path(
            project[
                "FinalPackageDirectory"
            ]
        )

        raw_root = Path(
            project[
                "RawRoot"
            ]
        )

        step5b_checkpoint_path = Path(
            project[
                "Step5BCheckpoint"
            ]
        )

        selection_checkpoint_path = Path(
            project[
                "SelectionCheckpoint"
            ]
        )


        required_paths = [
            checkpoint_path,
            package_root,
            raw_root,
            step5b_checkpoint_path,
            selection_checkpoint_path,
        ]


        missing_paths = [
            str(path)
            for path in required_paths
            if not path.exists()
        ]


        if missing_paths:
            raise FileNotFoundError(
                f"Project {project['ProjectNumber']} "
                "required paths are missing:\n"
                + "\n".join(missing_paths)
            )


        checkpoint_sha256 = sha256_file(
            checkpoint_path
        )

        step5b_checkpoint_sha256 = sha256_file(
            step5b_checkpoint_path
        )

        selection_checkpoint_sha256 = sha256_file(
            selection_checkpoint_path
        )


        checkpoint_payload = load_json(
            checkpoint_path
        )


        checkpoint_status = recursive_find(
            checkpoint_payload,
            [
                "Status",
            ],
        )

        checkpoint_raw_sha256 = recursive_find(
            checkpoint_payload,
            [
                "RawRootSHA256",
                "FrozenRawRootSHA256",
            ],
        )

        checkpoint_package_sha256 = recursive_find(
            checkpoint_payload,
            [
                "FinalPackageRootSHA256",
            ],
        )

        checkpoint_package_files = recursive_find(
            checkpoint_payload,
            [
                "FinalPackageFiles",
            ],
        )

        checkpoint_package_bytes = recursive_find(
            checkpoint_payload,
            [
                "FinalPackageBytes",
            ],
        )

        completed_at_utc = recursive_find(
            checkpoint_payload,
            [
                "CompletedAtUTC",
            ],
        )


        package_manifest = build_tree_manifest(
            package_root
        )

        package_files = len(
            package_manifest
        )

        package_bytes = int(
            package_manifest[
                "SizeBytes"
            ].sum()
        )

        package_root_sha256 = canonical_root_hash(
            package_manifest
        )


        project[
            "CompletedAtUTC"
        ] = str(
            completed_at_utc
        )

        project[
            "FinalPackageManifest"
        ] = str(
            package_root
            / "package_manifest.csv"
        )

        project[
            "FinalPackageManifestSHA256"
        ] = sha256_file(
            package_root
            / "package_manifest.csv"
        )


        checks = [
            (
                "Final-package checkpoint SHA-256",
                project[
                    "FinalPackageCheckpointSHA256"
                ],
                checkpoint_sha256,
            ),
            (
                "Final-package checkpoint status",
                project[
                    "ExpectedPackageStatus"
                ],
                checkpoint_status,
            ),
            (
                "Checkpoint raw-root SHA-256",
                project[
                    "RawRootSHA256"
                ],
                str(
                    checkpoint_raw_sha256
                ),
            ),
            (
                "Checkpoint package-root SHA-256",
                project[
                    "FinalPackageRootSHA256"
                ],
                str(
                    checkpoint_package_sha256
                ),
            ),
            (
                "Step 5B checkpoint SHA-256",
                project[
                    "Step5BCheckpointSHA256"
                ],
                step5b_checkpoint_sha256,
            ),
            (
                "Selection checkpoint SHA-256",
                project[
                    "SelectionCheckpointSHA256"
                ],
                selection_checkpoint_sha256,
            ),
            (
                "Current final-package files",
                project[
                    "PackageFiles"
                ],
                package_files,
            ),
            (
                "Current final-package bytes",
                project[
                    "PackageBytes"
                ],
                package_bytes,
            ),
            (
                "Current final-package root SHA-256",
                project[
                    "FinalPackageRootSHA256"
                ],
                package_root_sha256,
            ),
        ]


        if checkpoint_package_files is not None:
            checks.append((
                "Checkpoint final-package files",
                project[
                    "PackageFiles"
                ],
                int(
                    checkpoint_package_files
                ),
            ))


        if checkpoint_package_bytes is not None:
            checks.append((
                "Checkpoint final-package bytes",
                project[
                    "PackageBytes"
                ],
                int(
                    checkpoint_package_bytes
                ),
            ))


        for check_name, expected, actual in checks:
            add_check(
                package_validation_records,
                (
                    f"Project "
                    f"{project['ProjectNumber']} — "
                    f"{check_name}"
                ),
                expected,
                actual,
                actual == expected,
            )


    package_validation = pd.DataFrame(
        package_validation_records
    )

    package_failures = package_validation[
        ~package_validation[
            "Pass"
        ]
    ]


    print("\nFrozen package validation:")

    display(
        package_validation
    )


    if not package_failures.empty:
        raise RuntimeError(
            "Frozen package validation failed. "
            "The completion registry was not modified."
        )


    # ----------------------------------------------------------------------------------------------
    # 5. LOAD AND INSPECT CURRENT REGISTRY
    # ----------------------------------------------------------------------------------------------

    if not REGISTRY_PATH.is_file():
        raise FileNotFoundError(
            "Completion registry is missing:\n"
            f"{REGISTRY_PATH}"
        )


    registry_sha256_before = sha256_file(
        REGISTRY_PATH
    )

    registry = (
        pd.read_csv(
            REGISTRY_PATH,
            dtype=str,
        )
        .fillna("")
    )


    if registry.columns.duplicated().any():
        raise RuntimeError(
            "Completion registry contains duplicate columns."
        )


    column_semantics = {
        column:
            semantic_for_column(
                column
            )
        for column in registry.columns
    }


    project_number_columns = [
        column
        for column, semantic
        in column_semantics.items()
        if semantic == "ProjectNumber"
    ]

    project_columns = [
        column
        for column, semantic
        in column_semantics.items()
        if semantic == "Project"
    ]

    slug_columns = [
        column
        for column, semantic
        in column_semantics.items()
        if semantic == "ProjectSlug"
    ]

    status_columns = [
        column
        for column, semantic
        in column_semantics.items()
        if semantic == "Status"
    ]


    if len(project_number_columns) != 1:
        raise RuntimeError(
            "Could not uniquely identify the registry "
            "ProjectNumber column.\n"
            f"Candidates: {project_number_columns}"
        )


    if len(project_columns) != 1:
        raise RuntimeError(
            "Could not uniquely identify the registry "
            "Project column.\n"
            f"Candidates: {project_columns}"
        )


    if len(slug_columns) != 1:
        raise RuntimeError(
            "Could not uniquely identify the registry "
            "ProjectSlug column.\n"
            f"Candidates: {slug_columns}"
        )


    if len(status_columns) != 1:
        raise RuntimeError(
            "Could not uniquely identify the registry "
            "Status column.\n"
            f"Candidates: {status_columns}"
        )


    project_number_column = (
        project_number_columns[0]
    )

    project_column = (
        project_columns[0]
    )

    slug_column = (
        slug_columns[0]
    )

    status_column = (
        status_columns[0]
    )


    registry_project_numbers = pd.to_numeric(
        registry[
            project_number_column
        ],
        errors="raise",
    ).astype(int)


    existing_project_numbers = set(
        registry_project_numbers.tolist()
    )


    has_project9 = (
        9 in existing_project_numbers
    )

    has_project10 = (
        10 in existing_project_numbers
    )


    if has_project9 != has_project10:
        raise RuntimeError(
            "Only one of Projects 9 and 10 is present "
            "in the registry. Manual inspection is required."
        )


    already_finalised = bool(
        has_project9
        and has_project10
    )


    if not already_finalised:
        if registry_sha256_before != (
            EXPECTED_PRE_INSERTION_REGISTRY_SHA256
        ):
            raise RuntimeError(
                "The pre-insertion registry SHA-256 differs.\n"
                f"Expected: "
                f"{EXPECTED_PRE_INSERTION_REGISTRY_SHA256}\n"
                f"Actual:   {registry_sha256_before}\n\n"
                "The registry was not modified."
            )


        if len(registry) != 8:
            raise RuntimeError(
                "Expected exactly eight registry rows "
                "before inserting Projects 9 and 10.\n"
                f"Actual rows: {len(registry)}"
            )


        if existing_project_numbers != set(
            range(1, 9)
        ):
            raise RuntimeError(
                "Expected registry project numbers 1–8.\n"
                f"Actual: {sorted(existing_project_numbers)}"
            )


        incomplete_existing_rows = registry[
            registry[
                status_column
            ].astype(str).ne(
                COMPLETE_STATUS
            )
        ]


        if not incomplete_existing_rows.empty:
            raise RuntimeError(
                "One or more Projects 1–8 are not marked "
                "COMPLETE_AND_FROZEN."
            )


    # ----------------------------------------------------------------------------------------------
    # 6. BUILD REGISTRY ROWS USING THE EXISTING SCHEMA
    # ----------------------------------------------------------------------------------------------

    new_rows = []
    unknown_variable_columns = []


    for project in PROJECTS:
        row = {
            column:
                ""
            for column in registry.columns
        }


        values = {
            **project,
            "Status":
                COMPLETE_STATUS,
        }


        for column in registry.columns:
            semantic = column_semantics[
                column
            ]


            if (
                semantic is not None
                and semantic in values
            ):
                row[
                    column
                ] = str(
                    values[
                        semantic
                    ]
                )

                continue


            nonempty_values = (
                registry[
                    column
                ]
                .astype(str)
                .str.strip()
            )

            nonempty_values = nonempty_values[
                nonempty_values.ne("")
            ]


            unique_nonempty = sorted(
                nonempty_values.unique().tolist()
            )


            if len(unique_nonempty) == 1:
                # Preserve registry-wide constant metadata.
                row[
                    column
                ] = unique_nonempty[0]

            elif len(unique_nonempty) == 0:
                row[
                    column
                ] = ""

            else:
                unknown_variable_columns.append(
                    column
                )


        new_rows.append(
            row
        )


    unknown_variable_columns = sorted(
        set(
            unknown_variable_columns
        )
    )


    if unknown_variable_columns:
        print(
            "\nUnmapped variable registry columns:"
        )

        display(
            pd.DataFrame({
                "Column":
                    unknown_variable_columns,

                "Normalised":
                    [
                        normalise_name(
                            column
                        )
                        for column
                        in unknown_variable_columns
                    ],
            })
        )

        raise RuntimeError(
            "Registry insertion stopped safely because "
            "variable columns could not be mapped."
        )


    new_rows_frame = pd.DataFrame(
        new_rows,
        columns=registry.columns,
    )


    print("\nRegistry rows prepared for insertion:")

    display(
        new_rows_frame
    )


    # ----------------------------------------------------------------------------------------------
    # 7. ATOMIC INSERTION OR IDEMPOTENT READBACK
    # ----------------------------------------------------------------------------------------------

    timestamp_tag = datetime.now(
        timezone.utc
    ).strftime(
        "%Y%m%dT%H%M%SZ"
    )

    registry_backup_path = (
        NOTES_ROOT
        / (
            "completed_project_registry_"
            f"before_projects_09_10_{timestamp_tag}.csv"
        )
    )


    if not already_finalised:
        shutil.copy2(
            REGISTRY_PATH,
            registry_backup_path,
        )


        if sha256_file(
            registry_backup_path
        ) != registry_sha256_before:
            raise RuntimeError(
                "Registry backup SHA-256 mismatch."
            )


        registry_to_write = pd.concat(
            [
                registry,
                new_rows_frame,
            ],
            ignore_index=True,
        )


        registry_to_write[
            project_number_column
        ] = pd.to_numeric(
            registry_to_write[
                project_number_column
            ],
            errors="raise",
        ).astype(int)


        registry_to_write = (
            registry_to_write.sort_values(
                project_number_column,
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )


        # Recheck immediately before the atomic replacement.
        if sha256_file(
            REGISTRY_PATH
        ) != registry_sha256_before:
            raise RuntimeError(
                "Registry changed after preflight. "
                "Atomic insertion was cancelled."
            )


        temporary_registry_path = (
            REGISTRY_PATH.with_name(
                f".{REGISTRY_PATH.name}."
                f"tmp_{os.getpid()}"
            )
        )


        registry_to_write.to_csv(
            temporary_registry_path,
            index=False,
            lineterminator="\n",
        )


        temporary_readback = (
            pd.read_csv(
                temporary_registry_path,
                dtype=str,
            )
            .fillna("")
        )


        if len(
            temporary_readback
        ) != 10:
            raise RuntimeError(
                "Temporary registry does not contain "
                "exactly ten rows."
            )


        os.replace(
            temporary_registry_path,
            REGISTRY_PATH,
        )

        registry_write_performed = True

    else:
        registry_backup_path = None
        registry_write_performed = False


    # ----------------------------------------------------------------------------------------------
    # 8. FINAL REGISTRY VALIDATION
    # ----------------------------------------------------------------------------------------------

    registry_final = (
        pd.read_csv(
            REGISTRY_PATH,
            dtype=str,
        )
        .fillna("")
    )


    final_project_numbers = pd.to_numeric(
        registry_final[
            project_number_column
        ],
        errors="raise",
    ).astype(int)


    final_project_number_set = set(
        final_project_numbers.tolist()
    )


    project9_rows = registry_final[
        final_project_numbers.eq(9)
    ].copy()

    project10_rows = registry_final[
        final_project_numbers.eq(10)
    ].copy()


    registry_validation_records = []


    add_check(
        registry_validation_records,
        "Registry rows",
        10,
        len(registry_final),
        len(registry_final) == 10,
    )

    add_check(
        registry_validation_records,
        "Registry project-number set",
        list(range(1, 11)),
        sorted(final_project_number_set),
        final_project_number_set
        == set(range(1, 11)),
    )

    add_check(
        registry_validation_records,
        "Project 9 registry rows",
        1,
        len(project9_rows),
        len(project9_rows) == 1,
    )

    add_check(
        registry_validation_records,
        "Project 10 registry rows",
        1,
        len(project10_rows),
        len(project10_rows) == 1,
    )

    add_check(
        registry_validation_records,
        "COMPLETE_AND_FROZEN rows",
        10,
        int(
            registry_final[
                status_column
            ].eq(
                COMPLETE_STATUS
            ).sum()
        ),
        bool(
            registry_final[
                status_column
            ].eq(
                COMPLETE_STATUS
            ).all()
        ),
    )


    for project, project_rows in [
        (
            PROJECT_9,
            project9_rows,
        ),
        (
            PROJECT_10,
            project10_rows,
        ),
    ]:
        if len(project_rows) == 1:
            project_row = project_rows.iloc[0]

            add_check(
                registry_validation_records,
                (
                    f"Project "
                    f"{project['ProjectNumber']} name"
                ),
                project["Project"],
                project_row[
                    project_column
                ],
                project_row[
                    project_column
                ] == project["Project"],
            )

            add_check(
                registry_validation_records,
                (
                    f"Project "
                    f"{project['ProjectNumber']} slug"
                ),
                project["ProjectSlug"],
                project_row[
                    slug_column
                ],
                project_row[
                    slug_column
                ] == project["ProjectSlug"],
            )

            add_check(
                registry_validation_records,
                (
                    f"Project "
                    f"{project['ProjectNumber']} status"
                ),
                COMPLETE_STATUS,
                project_row[
                    status_column
                ],
                project_row[
                    status_column
                ] == COMPLETE_STATUS,
            )


            for column, semantic in (
                column_semantics.items()
            ):
                if semantic in {
                    "RawRootSHA256",
                    "FinalPackageRootSHA256",
                    "FinalPackageCheckpointSHA256",
                }:
                    expected_value = str(
                        project[
                            semantic
                        ]
                    )

                    actual_value = str(
                        project_row[
                            column
                        ]
                    )

                    add_check(
                        registry_validation_records,
                        (
                            f"Project "
                            f"{project['ProjectNumber']} "
                            f"{column}"
                        ),
                        expected_value,
                        actual_value,
                        actual_value
                        == expected_value,
                    )


    registry_validation = pd.DataFrame(
        registry_validation_records
    )


    registry_failures = registry_validation[
        ~registry_validation[
            "Pass"
        ]
    ]


    print("\nFinal registry validation:")

    display(
        registry_validation
    )


    if not registry_failures.empty:
        raise RuntimeError(
            "The final registry did not pass validation."
        )


    registry_sha256_after = sha256_file(
        REGISTRY_PATH
    )


    # ----------------------------------------------------------------------------------------------
    # 9. WRITE FINALISATION CHECKPOINT
    # ----------------------------------------------------------------------------------------------

    completed_at_utc = datetime.now(
        timezone.utc
    ).isoformat()


    finalisation_payload = {
        "Status":
            FINAL_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "Registry":
            str(
                REGISTRY_PATH
            ),

        "RegistrySHA256Before":
            registry_sha256_before,

        "RegistrySHA256After":
            registry_sha256_after,

        "RegistryBackup":
            (
                str(
                    registry_backup_path
                )
                if registry_backup_path
                is not None
                else None
            ),

        "RegistryWritePerformed":
            registry_write_performed,

        "RegistryRows":
            len(
                registry_final
            ),

        "CompleteAndFrozenProjects":
            int(
                registry_final[
                    status_column
                ].eq(
                    COMPLETE_STATUS
                ).sum()
            ),

        "ProjectNumbers":
            sorted(
                final_project_number_set
            ),

        "Project9":
            {
                "Project":
                    PROJECT_9[
                        "Project"
                    ],

                "ProjectSlug":
                    PROJECT_9[
                        "ProjectSlug"
                    ],

                "Status":
                    COMPLETE_STATUS,

                "RawRootSHA256":
                    PROJECT_9[
                        "RawRootSHA256"
                    ],

                "FinalPackageRootSHA256":
                    PROJECT_9[
                        "FinalPackageRootSHA256"
                    ],

                "FinalPackageCheckpointSHA256":
                    PROJECT_9[
                        "FinalPackageCheckpointSHA256"
                    ],
            },

        "Project10":
            {
                "Project":
                    PROJECT_10[
                        "Project"
                    ],

                "ProjectSlug":
                    PROJECT_10[
                        "ProjectSlug"
                    ],

                "Status":
                    COMPLETE_STATUS,

                "RawRootSHA256":
                    PROJECT_10[
                        "RawRootSHA256"
                    ],

                "FinalPackageRootSHA256":
                    PROJECT_10[
                        "FinalPackageRootSHA256"
                    ],

                "FinalPackageCheckpointSHA256":
                    PROJECT_10[
                        "FinalPackageCheckpointSHA256"
                    ],
            },

        "PackageValidationChecks":
            len(
                package_validation
            ),

        "PackageValidationFailures":
            len(
                package_failures
            ),

        "RegistryValidationChecks":
            len(
                registry_validation
            ),

        "RegistryValidationFailures":
            len(
                registry_failures
            ),

        "Projects1To8Preserved":
            True,

        "ExperimentsRerun":
            False,

        "RawResultsModified":
            False,

        "FinalPackagesModified":
            False,
    }


    atomic_write_json(
        FINALISATION_CHECKPOINT_PATH,
        finalisation_payload,
    )


    finalisation_status_payload = {
        "Status":
            FINAL_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "RegistryRows":
            len(
                registry_final
            ),

        "CompleteAndFrozenProjects":
            int(
                registry_final[
                    status_column
                ].eq(
                    COMPLETE_STATUS
                ).sum()
            ),

        "RegistrySHA256":
            registry_sha256_after,

        "Checkpoint":
            str(
                FINALISATION_CHECKPOINT_PATH
            ),

        "CheckpointSHA256":
            sha256_file(
                FINALISATION_CHECKPOINT_PATH
            ),

        "Project9Status":
            COMPLETE_STATUS,

        "Project10Status":
            COMPLETE_STATUS,
    }


    atomic_write_json(
        FINALISATION_STATUS_PATH,
        finalisation_status_payload,
    )


    # ----------------------------------------------------------------------------------------------
    # 10. FINAL READBACK
    # ----------------------------------------------------------------------------------------------

    checkpoint_readback = load_json(
        FINALISATION_CHECKPOINT_PATH
    )

    status_readback = load_json(
        FINALISATION_STATUS_PATH
    )


    if checkpoint_readback.get(
        "Status"
    ) != FINAL_STATUS:
        raise RuntimeError(
            "Registry-finalisation checkpoint readback failed."
        )


    if status_readback.get(
        "Status"
    ) != FINAL_STATUS:
        raise RuntimeError(
            "Registry-finalisation status readback failed."
        )


    if sha256_file(
        REGISTRY_PATH
    ) != registry_sha256_after:
        raise RuntimeError(
            "Registry changed after finalisation."
        )


    # ----------------------------------------------------------------------------------------------
    # 11. FINAL OUTPUT
    # ----------------------------------------------------------------------------------------------

    print("\n")
    print("=" * 132)
    print("=== PROJECTS 9 AND 10 SERIAL REGISTRY FINALISATION RESULT ===")
    print("=" * 132)


    print("\nRegistry:")

    print(
        "Registry path:",
        REGISTRY_PATH,
    )

    print(
        "Registry rows:",
        len(
            registry_final
        ),
    )

    print(
        "Project numbers:",
        sorted(
            final_project_number_set
        ),
    )

    print(
        "COMPLETE_AND_FROZEN projects:",
        int(
            registry_final[
                status_column
            ].eq(
                COMPLETE_STATUS
            ).sum()
        ),
    )

    print(
        "Registry write performed:",
        registry_write_performed,
    )

    print(
        "Registry SHA-256 before:",
        registry_sha256_before,
    )

    print(
        "Registry SHA-256 after:",
        registry_sha256_after,
    )


    print("\nProject 9:")

    print(
        "Project:",
        PROJECT_9[
            "Project"
        ],
    )

    print(
        "Status:",
        COMPLETE_STATUS,
    )

    print(
        "Raw-root SHA-256:",
        PROJECT_9[
            "RawRootSHA256"
        ],
    )

    print(
        "Final-package root SHA-256:",
        PROJECT_9[
            "FinalPackageRootSHA256"
        ],
    )


    print("\nProject 10:")

    print(
        "Project:",
        PROJECT_10[
            "Project"
        ],
    )

    print(
        "Status:",
        COMPLETE_STATUS,
    )

    print(
        "Raw-root SHA-256:",
        PROJECT_10[
            "RawRootSHA256"
        ],
    )

    print(
        "Final-package root SHA-256:",
        PROJECT_10[
            "FinalPackageRootSHA256"
        ],
    )


    print("\nValidation:")

    print(
        "Package validation failures:",
        len(
            package_failures
        ),
    )

    print(
        "Registry validation failures:",
        len(
            registry_failures
        ),
    )


    print("\nFinalisation checkpoint:")

    print(
        FINALISATION_CHECKPOINT_PATH
    )

    print(
        "Checkpoint SHA-256:",
        sha256_file(
            FINALISATION_CHECKPOINT_PATH
        ),
    )


    print(
        "\nSTATUS:",
        FINAL_STATUS,
    )

    print("=" * 132)


finally:
    if LOCK_PATH.exists():
        LOCK_PATH.unlink()

=== SERIAL FINALISATION: REGISTER PROJECTS 9 AND 10 AS COMPLETE_AND_FROZEN ===

Frozen package validation:


,Check,Expected,Actual,Pass
0,Project 9 — Final-package checkpoint SHA-256,209ed665deb40a1946b2507c173676c8e0df98b65c071d...,209ed665deb40a1946b2507c173676c8e0df98b65c071d...,True
1,Project 9 — Final-package checkpoint status,PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_AND_V...,PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_AND_V...,True
2,Project 9 — Checkpoint raw-root SHA-256,c31cb45e1354dc1222b82103bd275c21b10dd2ba617112...,c31cb45e1354dc1222b82103bd275c21b10dd2ba617112...,True
3,Project 9 — Checkpoint package-root SHA-256,600ec7a8b4013764eee49a7351337de4b773da6bc32902...,600ec7a8b4013764eee49a7351337de4b773da6bc32902...,True
4,Project 9 — Step 5B checkpoint SHA-256,a1fff6906ac62870689cbf4f3f38f16cdac04b10e68511...,a1fff6906ac62870689cbf4f3f38f16cdac04b10e68511...,True
5,Project 9 — Selection checkpoint SHA-256,3ef9e66e606772a9679cf727182389e419c9f8821512ba...,3ef9e66e606772a9679cf727182389e419c9f8821512ba...,True
6,Project 9 — Current final-package files,89,89,True
7,Project 9 — Current final-package bytes,5234900,5234900,True
8,Project 9 — Current final-package root SHA-256,600ec7a8b4013764eee49a7351337de4b773da6bc32902...,600ec7a8b4013764eee49a7351337de4b773da6bc32902...,True
9,Project 9 — Checkpoint final-package files,89,89,True


RuntimeError: Could not uniquely identify the registry Status column.
Candidates: ['Status', 'FinalAuditStatus']

In [16]:
# ==================================================================================================
# PROJECTS 9 AND 10 — SERIAL REGISTRY FINALISATION V2
#
# FIX:
# - Uses the exact registry column "Status" as the completion status.
# - Treats "FinalAuditStatus" as a separate final-package audit field.
# - Does not confuse Status with FinalAuditStatus.
#
# SAFETY:
# - Does not rerun any experiment.
# - Does not modify raw results.
# - Does not modify either final package.
# - Creates a backup before changing the registry.
# - Inserts Projects 9 and 10 together in one atomic operation.
# - Can safely detect an already-completed insertion.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import shutil
import time

import pandas as pd


print("=" * 132)
print("=== PROJECTS 9 AND 10 SERIAL REGISTRY FINALISATION V2 ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

LOCK_PATH = (
    NOTES_ROOT
    / ".projects_09_10_registry_finalisation_v2.lock"
)

FINALISATION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "projects_09_10_registry_finalisation_checkpoint.json"
)

FINALISATION_STATUS_PATH = (
    NOTES_ROOT
    / "projects_09_10_registry_finalisation_status.json"
)


EXPECTED_PRE_INSERTION_REGISTRY_SHA256 = (
    "a442446f3ca6207b31213fc422fb0c883"
    "c96608a8e6a3907da969e326808bee7"
)

COMPLETE_STATUS = (
    "COMPLETE_AND_FROZEN"
)

FINAL_STATUS = (
    "PASS_PROJECTS_09_10_REGISTERED_COMPLETE_AND_FROZEN"
)


# --------------------------------------------------------------------------------------------------
# 2. FROZEN PROJECT RECORDS
# --------------------------------------------------------------------------------------------------

PROJECTS = [
    {
        "ProjectNumber":
            9,

        "Project":
            "camunda@camunda-bpm-platform",

        "ProjectSlug":
            "camunda__camunda-bpm-platform",

        "Conditions":
            270,

        "NoiseLevels":
            9,

        "RepetitionSeeds":
            30,

        "MLTechniques":
            4,

        "Baselines":
            3,

        "Techniques":
            7,

        "MLFits":
            1080,

        "RankingRows":
            34_970_670,

        "BuildMetricRows":
            56_700,

        "ProjectRunRows":
            1_890,

        "RawFiles":
            2_160,

        "RawBytes":
            387_066_081,

        "RawRootSHA256":
            (
                "c31cb45e1354dc1222b82103bd275c21"
                "b10dd2ba6171125005d9c0a99ab14724"
            ),

        "RawRoot":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Results/Raw/camunda__camunda-bpm-platform"
            ),

        "RawManifest":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Results/Aggregated/camunda__camunda-bpm-platform/"
                "camunda_step5b_final_audit/"
                "camunda_raw_file_manifest.csv"
            ),

        "PackageFiles":
            89,

        "PackageBytes":
            5_234_900,

        "FinalPackageRootSHA256":
            (
                "600ec7a8b4013764eee49a7351337de4"
                "b773da6bc32902a38a58e95ed79147ff"
            ),

        "FinalPackageDirectory":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Results/Aggregated/camunda__camunda-bpm-platform/"
                "camunda_final_package"
            ),

        "FinalPackageCheckpoint":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Notes/project_09_final_package_checkpoint.json"
            ),

        "FinalPackageCheckpointSHA256":
            (
                "209ed665deb40a1946b2507c173676c8"
                "e0df98b65c071d8cf0d4f242eaf4f0eb"
            ),

        "FinalAuditStatus":
            (
                "PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_"
                "AND_VALIDATED_REGISTRY_PENDING"
            ),

        "Step5BCheckpoint":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Notes/project_09_step5b_checkpoint.json"
            ),

        "Step5BCheckpointSHA256":
            (
                "a1fff6906ac62870689cbf4f3f38f16c"
                "dac04b10e6851190bcb34fbb8d6f8a14"
            ),

        "SelectionCheckpoint":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Notes/project_09_selection_checkpoint.json"
            ),

        "SelectionCheckpointSHA256":
            (
                "3ef9e66e606772a9679cf727182389e4"
                "19c9f8821512ba1492eb9ebc9259948b"
            ),

        "SourceRootSHA256":
            (
                "65201f596c631d0d7472868d3b23aabc"
                "86fa8f9ef4674b8ac9400fd577ab8f07"
            ),
    },

    {
        "ProjectNumber":
            10,

        "Project":
            "spring-cloud@spring-cloud-dataflow",

        "ProjectSlug":
            "spring-cloud__spring-cloud-dataflow",

        "Conditions":
            270,

        "NoiseLevels":
            9,

        "RepetitionSeeds":
            30,

        "MLTechniques":
            4,

        "Baselines":
            3,

        "Techniques":
            7,

        "MLFits":
            1080,

        "RankingRows":
            4_934_790,

        "BuildMetricRows":
            51_030,

        "ProjectRunRows":
            1_890,

        "RawFiles":
            2_160,

        "RawBytes":
            68_638_025,

        "RawRootSHA256":
            (
                "5d336e1c7630cd6dec8ecdd8e17fa1af"
                "5cd6547d6cf90ecc19d0dbdce43fea6c"
            ),

        "RawRoot":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Results/Raw/spring-cloud__spring-cloud-dataflow"
            ),

        "RawManifest":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Results/Aggregated/spring-cloud__spring-cloud-dataflow/"
                "spring_cloud_dataflow_step5b_final_audit/"
                "spring_cloud_dataflow_raw_file_manifest.csv"
            ),

        "PackageFiles":
            92,

        "PackageBytes":
            133_559_115,

        "FinalPackageRootSHA256":
            (
                "2682e51a689abc8a2cfb12b07b7c1fe0"
                "f450d2acf9e9a5ead3943e7ce3fb6ab6"
            ),

        "FinalPackageDirectory":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Results/Aggregated/spring-cloud__spring-cloud-dataflow/"
                "spring_cloud_dataflow_final_package"
            ),

        "FinalPackageCheckpoint":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Notes/project_10_final_package_checkpoint.json"
            ),

        "FinalPackageCheckpointSHA256":
            (
                "73850cf9a6596383214a798b409e3180"
                "38bd08867bff8b5c50fd6c1273863ffc"
            ),

        "FinalAuditStatus":
            (
                "PASS_PROJECT_10_FINAL_PACKAGE_CONSTRUCTED_"
                "AND_VALIDATED_REGISTRY_PENDING"
            ),

        "Step5BCheckpoint":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Notes/project_10_step5b_checkpoint.json"
            ),

        "Step5BCheckpointSHA256":
            (
                "38ed74dac0892ba69efb418397c27e926"
                "c221880e39398d70447aba37d2cb364"
            ),

        "SelectionCheckpoint":
            (
                "/content/drive/MyDrive/Thesis_Experiment/"
                "Notes/project_10_selection_checkpoint.json"
            ),

        "SelectionCheckpointSHA256":
            (
                "97960ad7032d7c8b9dfb0a3bb45ba6d3"
                "cd64b648eadf0ae6169510ba981e7877"
            ),

        "SourceRootSHA256":
            (
                "582f01b3a43b542537b93243e5bb5b8c"
                "ff36c274c6c2a3b12b580090d664206e"
            ),
    },
]


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def normalise_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


def build_tree_manifest(root):
    root = Path(root)

    records = []

    for file_path in sorted(
        [
            path
            for path in root.rglob("*")
            if path.is_file()
        ],
        key=lambda path:
            path.relative_to(
                root
            ).as_posix(),
    ):
        records.append({
            "RelativePath":
                file_path.relative_to(
                    root
                ).as_posix(),

            "SizeBytes":
                int(
                    file_path.stat().st_size
                ),

            "SHA256":
                sha256_file(
                    file_path
                ),
        })

    return pd.DataFrame(
        records
    )


def canonical_root_hash(manifest):
    digest = hashlib.sha256()

    ordered = manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def find_exact_column(
    dataframe,
    normalised_name,
    label,
):
    matches = [
        column
        for column in dataframe.columns
        if normalise_name(column)
        == normalised_name
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected normalised name: {normalised_name}\n"
            f"Matches: {matches}\n"
            f"Registry columns: {dataframe.columns.tolist()}"
        )

    return matches[0]


def find_optional_exact_column(
    dataframe,
    normalised_name,
):
    matches = [
        column
        for column in dataframe.columns
        if normalise_name(column)
        == normalised_name
    ]

    if len(matches) > 1:
        raise RuntimeError(
            f"More than one registry column resolves to "
            f"{normalised_name}: {matches}"
        )

    return (
        matches[0]
        if matches
        else None
    )


def add_check(
    records,
    check,
    expected,
    actual,
    passed,
):
    records.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


def registry_value_for_column(
    column,
    project,
):
    name = normalise_name(
        column
    )

    package_directory = Path(
        project[
            "FinalPackageDirectory"
        ]
    )

    package_manifest = (
        package_directory
        / "package_manifest.csv"
    )

    package_report = (
        package_directory
        / "package_report.json"
    )

    raw_manifest = Path(
        project[
            "RawManifest"
        ]
    )


    exact_values = {
        "projectnumber":
            project[
                "ProjectNumber"
            ],

        "projectno":
            project[
                "ProjectNumber"
            ],

        "projectindex":
            project[
                "ProjectNumber"
            ],

        "project":
            project[
                "Project"
            ],

        "projectname":
            project[
                "Project"
            ],

        "repository":
            project[
                "Project"
            ],

        "repo":
            project[
                "Project"
            ],

        "projectslug":
            project[
                "ProjectSlug"
            ],

        "slug":
            project[
                "ProjectSlug"
            ],

        # Main registry completion status.
        "status":
            COMPLETE_STATUS,

        "completionstatus":
            COMPLETE_STATUS,

        "registrystatus":
            COMPLETE_STATUS,

        "completionregistrystatus":
            COMPLETE_STATUS,

        # Separate final-audit status.
        "finalauditstatus":
            project[
                "FinalAuditStatus"
            ],

        "packagevalidationstatus":
            project[
                "FinalAuditStatus"
            ],

        "finalpackagestatus":
            project[
                "FinalAuditStatus"
            ],

        "conditions":
            project[
                "Conditions"
            ],

        "conditioncount":
            project[
                "Conditions"
            ],

        "expectedconditions":
            project[
                "Conditions"
            ],

        "completedconditions":
            project[
                "Conditions"
            ],

        "finalauditconditions":
            project[
                "Conditions"
            ],

        "finalauditexpectedconditions":
            project[
                "Conditions"
            ],

        "finalauditcompletedconditions":
            project[
                "Conditions"
            ],

        "finalauditcurrentcondition":
            project[
                "Conditions"
            ],

        "noiselevels":
            project[
                "NoiseLevels"
            ],

        "noisecount":
            project[
                "NoiseLevels"
            ],

        "repetitionseeds":
            project[
                "RepetitionSeeds"
            ],

        "seeds":
            project[
                "RepetitionSeeds"
            ],

        "seedcount":
            project[
                "RepetitionSeeds"
            ],

        "mltechniques":
            project[
                "MLTechniques"
            ],

        "mlmodels":
            project[
                "MLTechniques"
            ],

        "modelcount":
            project[
                "MLTechniques"
            ],

        "baselines":
            project[
                "Baselines"
            ],

        "baselinecount":
            project[
                "Baselines"
            ],

        "techniques":
            project[
                "Techniques"
            ],

        "techniquecount":
            project[
                "Techniques"
            ],

        "mlfits":
            project[
                "MLFits"
            ],

        "mlfitcount":
            project[
                "MLFits"
            ],

        "expectedmodelfits":
            project[
                "MLFits"
            ],

        "totalmlfits":
            project[
                "MLFits"
            ],

        "rankingrows":
            project[
                "RankingRows"
            ],

        "fullrunrankingrows":
            project[
                "RankingRows"
            ],

        "finalauditrankingrows":
            project[
                "RankingRows"
            ],

        "buildmetricrows":
            project[
                "BuildMetricRows"
            ],

        "fullrunbuildmetricrows":
            project[
                "BuildMetricRows"
            ],

        "finalauditbuildmetricrows":
            project[
                "BuildMetricRows"
            ],

        "projectrunrows":
            project[
                "ProjectRunRows"
            ],

        "fullrunprojectrunrows":
            project[
                "ProjectRunRows"
            ],

        "finalauditprojectrunrows":
            project[
                "ProjectRunRows"
            ],

        "rawfiles":
            project[
                "RawFiles"
            ],

        "rawfilecount":
            project[
                "RawFiles"
            ],

        "rawresultsfilecount":
            project[
                "RawFiles"
            ],

        "rawbytes":
            project[
                "RawBytes"
            ],

        "rawsizebytes":
            project[
                "RawBytes"
            ],

        "rawresultstotalbytes":
            project[
                "RawBytes"
            ],

        "rawrootsha256":
            project[
                "RawRootSHA256"
            ],

        "rawsha256":
            project[
                "RawRootSHA256"
            ],

        "rawresultsrootsha256":
            project[
                "RawRootSHA256"
            ],

        "rawroot":
            project[
                "RawRoot"
            ],

        "rawdirectory":
            project[
                "RawRoot"
            ],

        "rawresultroot":
            project[
                "RawRoot"
            ],

        "rawresultdirectory":
            project[
                "RawRoot"
            ],

        "projectrawresults":
            project[
                "RawRoot"
            ],

        "rawresultsmanifest":
            str(
                raw_manifest
            ),

        "rawmanifest":
            str(
                raw_manifest
            ),

        "rawresultsmanifestsha256":
            sha256_file(
                raw_manifest
            ),

        "rawmanifestsha256":
            sha256_file(
                raw_manifest
            ),

        "packagefiles":
            project[
                "PackageFiles"
            ],

        "packagefilecount":
            project[
                "PackageFiles"
            ],

        "finalpackagefiles":
            project[
                "PackageFiles"
            ],

        "finalpackagefilecount":
            project[
                "PackageFiles"
            ],

        "finalauditmanifestrows":
            project[
                "PackageFiles"
            ],

        "packagebytes":
            project[
                "PackageBytes"
            ],

        "packagesizebytes":
            project[
                "PackageBytes"
            ],

        "finalpackagebytes":
            project[
                "PackageBytes"
            ],

        "finalpackagetotalbytes":
            project[
                "PackageBytes"
            ],

        "finalpackagerootsha256":
            project[
                "FinalPackageRootSHA256"
            ],

        "packagerootsha256":
            project[
                "FinalPackageRootSHA256"
            ],

        "packagesha256":
            project[
                "FinalPackageRootSHA256"
            ],

        "finalpackagedirectory":
            project[
                "FinalPackageDirectory"
            ],

        "packagedirectory":
            project[
                "FinalPackageDirectory"
            ],

        "finalpackageroot":
            project[
                "FinalPackageDirectory"
            ],

        "finalauditdirectory":
            project[
                "FinalPackageDirectory"
            ],

        "finalpackagecheckpoint":
            project[
                "FinalPackageCheckpoint"
            ],

        "packagecheckpoint":
            project[
                "FinalPackageCheckpoint"
            ],

        "freezerecord":
            project[
                "FinalPackageCheckpoint"
            ],

        "finalpackagecheckpointsha256":
            project[
                "FinalPackageCheckpointSHA256"
            ],

        "packagecheckpointsha256":
            project[
                "FinalPackageCheckpointSHA256"
            ],

        "step5bcheckpoint":
            project[
                "Step5BCheckpoint"
            ],

        "step5bcheckpointsha256":
            project[
                "Step5BCheckpointSHA256"
            ],

        "selectioncheckpoint":
            project[
                "SelectionCheckpoint"
            ],

        "selectioncheckpointsha256":
            project[
                "SelectionCheckpointSHA256"
            ],

        "sourcerootsha256":
            project[
                "SourceRootSHA256"
            ],

        "sourcesha256":
            project[
                "SourceRootSHA256"
            ],

        "finalpackagemanifest":
            str(
                package_manifest
            ),

        "packagemanifest":
            str(
                package_manifest
            ),

        "finalpackagemanifestsha256":
            sha256_file(
                package_manifest
            ),

        "packagemanifestsha256":
            sha256_file(
                package_manifest
            ),

        "finalauditreport":
            str(
                package_report
            ),

        "finalauditmanifestviolations":
            0,

        "finalauditconditionmismatches":
            0,

        "finalauditrankingsignaturemismatches":
            0,

        "finalpackagemissingfiles":
            0,

        "finalpackageunexpectedfiles":
            0,

        "finalpackagesizemismatches":
            0,

        "finalpackagesha256mismatches":
            0,

        "finalauditerror":
            "",

        "finalauditprogress":
            "COMPLETE",

        "donotrerun":
            True,

        "primarymetric":
            "APFDc",

        "secondarymetric":
            "APFD",

        "completedatutc":
            project[
                "CompletedAtUTC"
            ],

        "completiontimeutc":
            project[
                "CompletedAtUTC"
            ],

        "completiontimestamp":
            project[
                "CompletedAtUTC"
            ],

        "frozenatutc":
            project[
                "CompletedAtUTC"
            ],

        "updatedatutc":
            project[
                "CompletedAtUTC"
            ],
    }


    if name in exact_values:
        return (
            str(
                exact_values[
                    name
                ]
            ),
            True,
        )


    # Conservative pattern fallbacks.

    if (
        "finalaudit"
        in name
        and "status"
        in name
    ):
        return (
            project[
                "FinalAuditStatus"
            ],
            True,
        )


    if (
        "status"
        in name
        and "audit"
        not in name
        and "package"
        not in name
    ):
        return (
            COMPLETE_STATUS,
            True,
        )


    if (
        "project"
        in name
        and (
            "number"
            in name
            or "index"
            in name
        )
    ):
        return (
            str(
                project[
                    "ProjectNumber"
                ]
            ),
            True,
        )


    if "slug" in name:
        return (
            project[
                "ProjectSlug"
            ],
            True,
        )


    if (
        "raw"
        in name
        and "sha256"
        in name
    ):
        return (
            project[
                "RawRootSHA256"
            ],
            True,
        )


    if (
        "package"
        in name
        and "checkpoint"
        in name
        and "sha256"
        in name
    ):
        return (
            project[
                "FinalPackageCheckpointSHA256"
            ],
            True,
        )


    if (
        "package"
        in name
        and "checkpoint"
        in name
    ):
        return (
            project[
                "FinalPackageCheckpoint"
            ],
            True,
        )


    if (
        "package"
        in name
        and "sha256"
        in name
    ):
        return (
            project[
                "FinalPackageRootSHA256"
            ],
            True,
        )


    if (
        "package"
        in name
        and "file"
        in name
    ):
        return (
            str(
                project[
                    "PackageFiles"
                ]
            ),
            True,
        )


    if (
        "package"
        in name
        and (
            "byte"
            in name
            or "size"
            in name
        )
    ):
        return (
            str(
                project[
                    "PackageBytes"
                ]
            ),
            True,
        )


    if (
        "package"
        in name
        and (
            "root"
            in name
            or "directory"
            in name
        )
    ):
        return (
            project[
                "FinalPackageDirectory"
            ],
            True,
        )


    if (
        "completed"
        in name
        or "completion"
        in name
        or "frozenat"
        in name
    ) and (
        "utc"
        in name
        or "time"
        in name
        or "date"
        in name
        or "timestamp"
        in name
    ):
        return (
            project[
                "CompletedAtUTC"
            ],
            True,
        )


    return (
        "",
        False,
    )


# --------------------------------------------------------------------------------------------------
# 4. ACQUIRE EXCLUSIVE REGISTRY LOCK
# --------------------------------------------------------------------------------------------------

if LOCK_PATH.exists():
    lock_age_seconds = (
        time.time()
        - LOCK_PATH.stat().st_mtime
    )

    if lock_age_seconds > 1800:
        LOCK_PATH.unlink()

    else:
        raise RuntimeError(
            "A recent registry-finalisation lock exists:\n"
            f"{LOCK_PATH}\n\n"
            "Do not run multiple registry writers concurrently."
        )


lock_descriptor = os.open(
    LOCK_PATH,
    os.O_CREAT
    | os.O_EXCL
    | os.O_WRONLY,
)


try:
    os.write(
        lock_descriptor,
        json.dumps({
            "PID":
                os.getpid(),

            "CreatedAtUTC":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }).encode("utf-8"),
    )

finally:
    os.close(
        lock_descriptor
    )


try:

    # ----------------------------------------------------------------------------------------------
    # 5. VALIDATE BOTH FINAL PACKAGES
    # ----------------------------------------------------------------------------------------------

    package_validation_records = []


    for project in PROJECTS:
        package_root = Path(
            project[
                "FinalPackageDirectory"
            ]
        )

        package_checkpoint = Path(
            project[
                "FinalPackageCheckpoint"
            ]
        )

        step5b_checkpoint = Path(
            project[
                "Step5BCheckpoint"
            ]
        )

        selection_checkpoint = Path(
            project[
                "SelectionCheckpoint"
            ]
        )

        raw_manifest = Path(
            project[
                "RawManifest"
            ]
        )


        required_paths = [
            package_root,
            package_checkpoint,
            step5b_checkpoint,
            selection_checkpoint,
            raw_manifest,
        ]


        missing_paths = [
            str(path)
            for path in required_paths
            if not path.exists()
        ]


        if missing_paths:
            raise FileNotFoundError(
                f"Project {project['ProjectNumber']} "
                "required paths are missing:\n"
                + "\n".join(
                    missing_paths
                )
            )


        checkpoint_sha256 = sha256_file(
            package_checkpoint
        )

        step5b_sha256 = sha256_file(
            step5b_checkpoint
        )

        selection_sha256 = sha256_file(
            selection_checkpoint
        )


        checkpoint_payload = load_json(
            package_checkpoint
        )


        project[
            "CompletedAtUTC"
        ] = str(
            checkpoint_payload.get(
                "CompletedAtUTC",
                "",
            )
        )


        package_tree = build_tree_manifest(
            package_root
        )

        package_files = len(
            package_tree
        )

        package_bytes = int(
            package_tree[
                "SizeBytes"
            ].sum()
        )

        package_root_sha256 = canonical_root_hash(
            package_tree
        )


        checks = [
            (
                "Final-package checkpoint SHA-256",
                project[
                    "FinalPackageCheckpointSHA256"
                ],
                checkpoint_sha256,
            ),

            (
                "Final-package checkpoint status",
                project[
                    "FinalAuditStatus"
                ],
                checkpoint_payload.get(
                    "Status"
                ),
            ),

            (
                "Checkpoint raw-root SHA-256",
                project[
                    "RawRootSHA256"
                ],
                checkpoint_payload.get(
                    "RawRootSHA256"
                ),
            ),

            (
                "Checkpoint package-root SHA-256",
                project[
                    "FinalPackageRootSHA256"
                ],
                checkpoint_payload.get(
                    "FinalPackageRootSHA256"
                ),
            ),

            (
                "Step 5B checkpoint SHA-256",
                project[
                    "Step5BCheckpointSHA256"
                ],
                step5b_sha256,
            ),

            (
                "Selection checkpoint SHA-256",
                project[
                    "SelectionCheckpointSHA256"
                ],
                selection_sha256,
            ),

            (
                "Current package files",
                project[
                    "PackageFiles"
                ],
                package_files,
            ),

            (
                "Current package bytes",
                project[
                    "PackageBytes"
                ],
                package_bytes,
            ),

            (
                "Current package-root SHA-256",
                project[
                    "FinalPackageRootSHA256"
                ],
                package_root_sha256,
            ),
        ]


        for check_name, expected, actual in checks:
            add_check(
                package_validation_records,
                (
                    f"Project "
                    f"{project['ProjectNumber']} — "
                    f"{check_name}"
                ),
                expected,
                actual,
                actual == expected,
            )


    package_validation = pd.DataFrame(
        package_validation_records
    )

    package_failures = package_validation[
        ~package_validation[
            "Pass"
        ]
    ]


    print("\nFrozen package validation:")

    display(
        package_validation
    )


    if not package_failures.empty:
        raise RuntimeError(
            "Frozen package validation failed. "
            "The registry was not modified."
        )


    # ----------------------------------------------------------------------------------------------
    # 6. LOAD CURRENT REGISTRY AND RESOLVE EXACT CORE COLUMNS
    # ----------------------------------------------------------------------------------------------

    if not REGISTRY_PATH.is_file():
        raise FileNotFoundError(
            "Completion registry is missing:\n"
            f"{REGISTRY_PATH}"
        )


    registry_sha256_before = sha256_file(
        REGISTRY_PATH
    )

    registry = (
        pd.read_csv(
            REGISTRY_PATH,
            dtype=str,
        )
        .fillna("")
    )


    if registry.columns.duplicated().any():
        raise RuntimeError(
            "The completion registry has duplicate column names."
        )


    project_number_column = find_exact_column(
        registry,
        "projectnumber",
        "ProjectNumber column",
    )

    project_column = find_exact_column(
        registry,
        "project",
        "Project column",
    )

    project_slug_column = find_exact_column(
        registry,
        "projectslug",
        "ProjectSlug column",
    )

    # Critical V2 fix:
    # Resolve only the exact "Status" column.
    status_column = find_exact_column(
        registry,
        "status",
        "main Status column",
    )

    # FinalAuditStatus is intentionally separate.
    final_audit_status_column = (
        find_optional_exact_column(
            registry,
            "finalauditstatus",
        )
    )


    print("\nResolved registry columns:")

    print(
        "ProjectNumber:",
        project_number_column,
    )

    print(
        "Project:",
        project_column,
    )

    print(
        "ProjectSlug:",
        project_slug_column,
    )

    print(
        "Main completion Status:",
        status_column,
    )

    print(
        "Separate FinalAuditStatus:",
        final_audit_status_column,
    )


    registry_project_numbers = pd.to_numeric(
        registry[
            project_number_column
        ],
        errors="raise",
    ).astype(int)


    existing_project_numbers = set(
        registry_project_numbers.tolist()
    )


    project9_present = (
        9 in existing_project_numbers
    )

    project10_present = (
        10 in existing_project_numbers
    )


    if project9_present != project10_present:
        raise RuntimeError(
            "Only one of Projects 9 and 10 is already "
            "present in the registry. No write was performed."
        )


    already_finalised = bool(
        project9_present
        and project10_present
    )


    if not already_finalised:
        if (
            registry_sha256_before
            != EXPECTED_PRE_INSERTION_REGISTRY_SHA256
        ):
            raise RuntimeError(
                "The pre-insertion registry SHA-256 differs.\n"
                f"Expected: "
                f"{EXPECTED_PRE_INSERTION_REGISTRY_SHA256}\n"
                f"Actual:   {registry_sha256_before}\n\n"
                "The registry was not modified."
            )


        if len(registry) != 8:
            raise RuntimeError(
                "Expected exactly eight rows before insertion.\n"
                f"Actual rows: {len(registry)}"
            )


        if existing_project_numbers != set(
            range(1, 9)
        ):
            raise RuntimeError(
                "Expected registered project numbers 1–8.\n"
                f"Actual: {sorted(existing_project_numbers)}"
            )


        existing_status_values = (
            registry[
                status_column
            ]
            .astype(str)
            .str.strip()
        )


        if not existing_status_values.eq(
            COMPLETE_STATUS
        ).all():
            raise RuntimeError(
                "One or more Projects 1–8 are not marked "
                "COMPLETE_AND_FROZEN."
            )


    # ----------------------------------------------------------------------------------------------
    # 7. BUILD PROJECT 9 AND PROJECT 10 ROWS
    # ----------------------------------------------------------------------------------------------

    new_rows = []
    unmapped_variable_columns = set()


    for project in PROJECTS:
        row = {}


        for column in registry.columns:
            value, mapped = registry_value_for_column(
                column,
                project,
            )


            if mapped:
                row[
                    column
                ] = value

                continue


            existing_values = (
                registry[
                    column
                ]
                .astype(str)
                .str.strip()
            )

            existing_nonempty = existing_values[
                existing_values.ne("")
            ]

            unique_nonempty = sorted(
                existing_nonempty.unique().tolist()
            )


            if len(unique_nonempty) == 1:
                # Preserve registry-wide protocol constants.
                row[
                    column
                ] = unique_nonempty[0]

            else:
                # Leave unknown optional project-specific fields blank
                # rather than copying another project's information.
                row[
                    column
                ] = ""

                if len(unique_nonempty) > 1:
                    unmapped_variable_columns.add(
                        column
                    )


        # Explicitly reinforce the four core fields.
        row[
            project_number_column
        ] = str(
            project[
                "ProjectNumber"
            ]
        )

        row[
            project_column
        ] = project[
            "Project"
        ]

        row[
            project_slug_column
        ] = project[
            "ProjectSlug"
        ]

        row[
            status_column
        ] = COMPLETE_STATUS


        if (
            final_audit_status_column
            is not None
        ):
            row[
                final_audit_status_column
            ] = project[
                "FinalAuditStatus"
            ]


        new_rows.append(
            row
        )


    new_rows_frame = pd.DataFrame(
        new_rows,
        columns=registry.columns,
    )


    print("\nRegistry rows prepared:")

    display(
        new_rows_frame
    )


    if unmapped_variable_columns:
        print(
            "\nOptional variable columns left blank "
            "because they had no safe mapping:"
        )

        display(
            pd.DataFrame({
                "Column":
                    sorted(
                        unmapped_variable_columns
                    )
            })
        )


    # ----------------------------------------------------------------------------------------------
    # 8. BACKUP AND ATOMIC REGISTRY INSERTION
    # ----------------------------------------------------------------------------------------------

    timestamp_tag = datetime.now(
        timezone.utc
    ).strftime(
        "%Y%m%dT%H%M%SZ"
    )


    if not already_finalised:
        registry_backup_path = (
            NOTES_ROOT
            / (
                "completed_project_registry_"
                "before_projects_09_10_"
                f"{timestamp_tag}.csv"
            )
        )


        shutil.copy2(
            REGISTRY_PATH,
            registry_backup_path,
        )


        if (
            sha256_file(
                registry_backup_path
            )
            != registry_sha256_before
        ):
            raise RuntimeError(
                "Registry backup hash validation failed."
            )


        registry_to_write = pd.concat(
            [
                registry,
                new_rows_frame,
            ],
            ignore_index=True,
        )


        registry_to_write[
            project_number_column
        ] = pd.to_numeric(
            registry_to_write[
                project_number_column
            ],
            errors="raise",
        ).astype(int)


        registry_to_write = (
            registry_to_write.sort_values(
                project_number_column,
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )


        if len(registry_to_write) != 10:
            raise RuntimeError(
                "Prepared registry does not have exactly ten rows."
            )


        if sha256_file(
            REGISTRY_PATH
        ) != registry_sha256_before:
            raise RuntimeError(
                "The registry changed after preflight. "
                "Insertion was cancelled."
            )


        temporary_registry_path = (
            REGISTRY_PATH.with_name(
                f".{REGISTRY_PATH.name}."
                f"tmp_{os.getpid()}"
            )
        )


        registry_to_write.to_csv(
            temporary_registry_path,
            index=False,
            lineterminator="\n",
        )


        temporary_readback = (
            pd.read_csv(
                temporary_registry_path,
                dtype=str,
            )
            .fillna("")
        )


        if len(
            temporary_readback
        ) != 10:
            raise RuntimeError(
                "Temporary registry readback did not contain ten rows."
            )


        temporary_numbers = pd.to_numeric(
            temporary_readback[
                project_number_column
            ],
            errors="raise",
        ).astype(int)


        if set(
            temporary_numbers.tolist()
        ) != set(
            range(1, 11)
        ):
            raise RuntimeError(
                "Temporary registry does not contain Projects 1–10."
            )


        os.replace(
            temporary_registry_path,
            REGISTRY_PATH,
        )


        registry_write_performed = True

    else:
        registry_backup_path = None
        registry_write_performed = False


    # ----------------------------------------------------------------------------------------------
    # 9. FINAL REGISTRY VALIDATION
    # ----------------------------------------------------------------------------------------------

    registry_final = (
        pd.read_csv(
            REGISTRY_PATH,
            dtype=str,
        )
        .fillna("")
    )


    final_project_numbers = pd.to_numeric(
        registry_final[
            project_number_column
        ],
        errors="raise",
    ).astype(int)


    final_project_set = set(
        final_project_numbers.tolist()
    )


    final_status_values = (
        registry_final[
            status_column
        ]
        .astype(str)
        .str.strip()
    )


    project9_rows = registry_final[
        final_project_numbers.eq(9)
    ]

    project10_rows = registry_final[
        final_project_numbers.eq(10)
    ]


    registry_validation_records = []


    add_check(
        registry_validation_records,
        "Registry rows",
        10,
        len(registry_final),
        len(registry_final) == 10,
    )

    add_check(
        registry_validation_records,
        "Registered project numbers",
        list(range(1, 11)),
        sorted(final_project_set),
        final_project_set
        == set(range(1, 11)),
    )

    add_check(
        registry_validation_records,
        "COMPLETE_AND_FROZEN projects",
        10,
        int(
            final_status_values.eq(
                COMPLETE_STATUS
            ).sum()
        ),
        final_status_values.eq(
            COMPLETE_STATUS
        ).all(),
    )

    add_check(
        registry_validation_records,
        "Project 9 registry rows",
        1,
        len(project9_rows),
        len(project9_rows) == 1,
    )

    add_check(
        registry_validation_records,
        "Project 10 registry rows",
        1,
        len(project10_rows),
        len(project10_rows) == 1,
    )


    for project, project_rows in [
        (
            PROJECTS[0],
            project9_rows,
        ),
        (
            PROJECTS[1],
            project10_rows,
        ),
    ]:
        if len(project_rows) != 1:
            continue


        project_row = project_rows.iloc[0]


        add_check(
            registry_validation_records,
            (
                f"Project "
                f"{project['ProjectNumber']} name"
            ),
            project[
                "Project"
            ],
            project_row[
                project_column
            ],
            project_row[
                project_column
            ] == project[
                "Project"
            ],
        )

        add_check(
            registry_validation_records,
            (
                f"Project "
                f"{project['ProjectNumber']} slug"
            ),
            project[
                "ProjectSlug"
            ],
            project_row[
                project_slug_column
            ],
            project_row[
                project_slug_column
            ] == project[
                "ProjectSlug"
            ],
        )

        add_check(
            registry_validation_records,
            (
                f"Project "
                f"{project['ProjectNumber']} completion status"
            ),
            COMPLETE_STATUS,
            project_row[
                status_column
            ],
            project_row[
                status_column
            ] == COMPLETE_STATUS,
        )


        if (
            final_audit_status_column
            is not None
        ):
            add_check(
                registry_validation_records,
                (
                    f"Project "
                    f"{project['ProjectNumber']} "
                    "final-audit status"
                ),
                project[
                    "FinalAuditStatus"
                ],
                project_row[
                    final_audit_status_column
                ],
                (
                    project_row[
                        final_audit_status_column
                    ]
                    == project[
                        "FinalAuditStatus"
                    ]
                ),
            )


        for column in registry_final.columns:
            name = normalise_name(
                column
            )


            if name in {
                "rawrootsha256",
                "rawresultsrootsha256",
            }:
                add_check(
                    registry_validation_records,
                    (
                        f"Project "
                        f"{project['ProjectNumber']} "
                        f"{column}"
                    ),
                    project[
                        "RawRootSHA256"
                    ],
                    project_row[
                        column
                    ],
                    (
                        project_row[
                            column
                        ]
                        == project[
                            "RawRootSHA256"
                        ]
                    ),
                )


            if name in {
                "finalpackagerootsha256",
                "packagerootsha256",
            }:
                add_check(
                    registry_validation_records,
                    (
                        f"Project "
                        f"{project['ProjectNumber']} "
                        f"{column}"
                    ),
                    project[
                        "FinalPackageRootSHA256"
                    ],
                    project_row[
                        column
                    ],
                    (
                        project_row[
                            column
                        ]
                        == project[
                            "FinalPackageRootSHA256"
                        ]
                    ),
                )


    registry_validation = pd.DataFrame(
        registry_validation_records
    )

    registry_failures = registry_validation[
        ~registry_validation[
            "Pass"
        ]
    ]


    print("\nFinal registry validation:")

    display(
        registry_validation
    )


    if not registry_failures.empty:
        raise RuntimeError(
            "Final registry validation failed."
        )


    registry_sha256_after = sha256_file(
        REGISTRY_PATH
    )


    # ----------------------------------------------------------------------------------------------
    # 10. WRITE FINALISATION CHECKPOINT AND STATUS
    # ----------------------------------------------------------------------------------------------

    completed_at_utc = datetime.now(
        timezone.utc
    ).isoformat()


    finalisation_checkpoint = {
        "Status":
            FINAL_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "Registry":
            str(
                REGISTRY_PATH
            ),

        "RegistrySHA256Before":
            registry_sha256_before,

        "RegistrySHA256After":
            registry_sha256_after,

        "RegistryBackup":
            (
                str(
                    registry_backup_path
                )
                if registry_backup_path
                is not None
                else None
            ),

        "RegistryWritePerformed":
            registry_write_performed,

        "RegistryRows":
            len(
                registry_final
            ),

        "ProjectNumbers":
            sorted(
                final_project_set
            ),

        "CompleteAndFrozenProjects":
            int(
                final_status_values.eq(
                    COMPLETE_STATUS
                ).sum()
            ),

        "MainStatusColumn":
            status_column,

        "FinalAuditStatusColumn":
            final_audit_status_column,

        "UnmappedOptionalVariableColumns":
            sorted(
                unmapped_variable_columns
            ),

        "Project9": {
            "Project":
                PROJECTS[0][
                    "Project"
                ],

            "Status":
                COMPLETE_STATUS,

            "FinalAuditStatus":
                PROJECTS[0][
                    "FinalAuditStatus"
                ],

            "RawRootSHA256":
                PROJECTS[0][
                    "RawRootSHA256"
                ],

            "FinalPackageRootSHA256":
                PROJECTS[0][
                    "FinalPackageRootSHA256"
                ],
        },

        "Project10": {
            "Project":
                PROJECTS[1][
                    "Project"
                ],

            "Status":
                COMPLETE_STATUS,

            "FinalAuditStatus":
                PROJECTS[1][
                    "FinalAuditStatus"
                ],

            "RawRootSHA256":
                PROJECTS[1][
                    "RawRootSHA256"
                ],

            "FinalPackageRootSHA256":
                PROJECTS[1][
                    "FinalPackageRootSHA256"
                ],
        },

        "PackageValidationChecks":
            len(
                package_validation
            ),

        "PackageValidationFailures":
            len(
                package_failures
            ),

        "RegistryValidationChecks":
            len(
                registry_validation
            ),

        "RegistryValidationFailures":
            len(
                registry_failures
            ),

        "ExperimentsRerun":
            False,

        "RawResultsModified":
            False,

        "FinalPackagesModified":
            False,
    }


    atomic_write_json(
        FINALISATION_CHECKPOINT_PATH,
        finalisation_checkpoint,
    )


    finalisation_status = {
        "Status":
            FINAL_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "RegistryRows":
            len(
                registry_final
            ),

        "CompleteAndFrozenProjects":
            int(
                final_status_values.eq(
                    COMPLETE_STATUS
                ).sum()
            ),

        "RegistrySHA256":
            registry_sha256_after,

        "Project9Status":
            COMPLETE_STATUS,

        "Project10Status":
            COMPLETE_STATUS,

        "Checkpoint":
            str(
                FINALISATION_CHECKPOINT_PATH
            ),

        "CheckpointSHA256":
            sha256_file(
                FINALISATION_CHECKPOINT_PATH
            ),
    }


    atomic_write_json(
        FINALISATION_STATUS_PATH,
        finalisation_status,
    )


    # ----------------------------------------------------------------------------------------------
    # 11. FINAL READBACK
    # ----------------------------------------------------------------------------------------------

    checkpoint_readback = load_json(
        FINALISATION_CHECKPOINT_PATH
    )

    status_readback = load_json(
        FINALISATION_STATUS_PATH
    )


    if checkpoint_readback.get(
        "Status"
    ) != FINAL_STATUS:
        raise RuntimeError(
            "Finalisation checkpoint readback failed."
        )


    if status_readback.get(
        "Status"
    ) != FINAL_STATUS:
        raise RuntimeError(
            "Finalisation status readback failed."
        )


    if (
        sha256_file(
            REGISTRY_PATH
        )
        != registry_sha256_after
    ):
        raise RuntimeError(
            "Registry changed after finalisation."
        )


    # ----------------------------------------------------------------------------------------------
    # 12. FINAL OUTPUT
    # ----------------------------------------------------------------------------------------------

    print("\n")
    print("=" * 132)
    print("=== PROJECTS 9 AND 10 SERIAL REGISTRY FINALISATION V2 RESULT ===")
    print("=" * 132)


    print("\nRegistry:")

    print(
        "Registry rows:",
        len(
            registry_final
        ),
    )

    print(
        "Project numbers:",
        sorted(
            final_project_set
        ),
    )

    print(
        "COMPLETE_AND_FROZEN projects:",
        int(
            final_status_values.eq(
                COMPLETE_STATUS
            ).sum()
        ),
    )

    print(
        "Main completion-status column:",
        status_column,
    )

    print(
        "Separate final-audit-status column:",
        final_audit_status_column,
    )

    print(
        "Registry write performed:",
        registry_write_performed,
    )

    print(
        "Registry SHA-256 before:",
        registry_sha256_before,
    )

    print(
        "Registry SHA-256 after:",
        registry_sha256_after,
    )


    print("\nProject 9:")

    print(
        "Project:",
        PROJECTS[0][
            "Project"
        ],
    )

    print(
        "Status:",
        COMPLETE_STATUS,
    )

    print(
        "Final-audit status:",
        PROJECTS[0][
            "FinalAuditStatus"
        ],
    )


    print("\nProject 10:")

    print(
        "Project:",
        PROJECTS[1][
            "Project"
        ],
    )

    print(
        "Status:",
        COMPLETE_STATUS,
    )

    print(
        "Final-audit status:",
        PROJECTS[1][
            "FinalAuditStatus"
        ],
    )


    print("\nValidation:")

    print(
        "Package validation failures:",
        len(
            package_failures
        ),
    )

    print(
        "Registry validation failures:",
        len(
            registry_failures
        ),
    )


    print("\nFinalisation checkpoint:")

    print(
        FINALISATION_CHECKPOINT_PATH
    )

    print(
        "Checkpoint SHA-256:",
        sha256_file(
            FINALISATION_CHECKPOINT_PATH
        ),
    )


    print(
        "\nSTATUS:",
        FINAL_STATUS,
    )

    print("=" * 132)


finally:
    if LOCK_PATH.exists():
        LOCK_PATH.unlink()

=== PROJECTS 9 AND 10 SERIAL REGISTRY FINALISATION V2 ===

Frozen package validation:


,Check,Expected,Actual,Pass
0,Project 9 — Final-package checkpoint SHA-256,209ed665deb40a1946b2507c173676c8e0df98b65c071d...,209ed665deb40a1946b2507c173676c8e0df98b65c071d...,True
1,Project 9 — Final-package checkpoint status,PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_AND_V...,PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_AND_V...,True
2,Project 9 — Checkpoint raw-root SHA-256,c31cb45e1354dc1222b82103bd275c21b10dd2ba617112...,c31cb45e1354dc1222b82103bd275c21b10dd2ba617112...,True
3,Project 9 — Checkpoint package-root SHA-256,600ec7a8b4013764eee49a7351337de4b773da6bc32902...,600ec7a8b4013764eee49a7351337de4b773da6bc32902...,True
4,Project 9 — Step 5B checkpoint SHA-256,a1fff6906ac62870689cbf4f3f38f16cdac04b10e68511...,a1fff6906ac62870689cbf4f3f38f16cdac04b10e68511...,True
5,Project 9 — Selection checkpoint SHA-256,3ef9e66e606772a9679cf727182389e419c9f8821512ba...,3ef9e66e606772a9679cf727182389e419c9f8821512ba...,True
6,Project 9 — Current package files,89,89,True
7,Project 9 — Current package bytes,5234900,5234900,True
8,Project 9 — Current package-root SHA-256,600ec7a8b4013764eee49a7351337de4b773da6bc32902...,600ec7a8b4013764eee49a7351337de4b773da6bc32902...,True
9,Project 10 — Final-package checkpoint SHA-256,73850cf9a6596383214a798b409e318038bd08867bff8b...,73850cf9a6596383214a798b409e318038bd08867bff8b...,True



Resolved registry columns:
ProjectNumber: ProjectNumber
Project: Project
ProjectSlug: ProjectSlug
Main completion Status: Status
Separate FinalAuditStatus: FinalAuditStatus

Registry rows prepared:


,ProjectNumber,Project,ProjectSlug,Status,Conditions,Seeds,NoiseLevels,Techniques,EvaluationBuilds,EvaluationRows,...,FreezeRecord,ChecksumManifest,LastFreezeValidationAtUTC,RawResultsManifest,FinalPackageManifest,RawResultsRootSHA256,FinalPackageRootSHA256,ModelFits,ManifestRowsAudited,FinalAuditStatus
0,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,COMPLETE_AND_FROZEN,270,30,9,7,,,...,/content/drive/MyDrive/Thesis_Experiment/Notes...,/content/drive/MyDrive/Thesis_Experiment/Resul...,2026-07-25T03:49:02.436302+00:00,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,c31cb45e1354dc1222b82103bd275c21b10dd2ba617112...,600ec7a8b4013764eee49a7351337de4b773da6bc32902...,,5358150.0,PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_AND_V...
1,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,COMPLETE_AND_FROZEN,270,30,9,7,,,...,/content/drive/MyDrive/Thesis_Experiment/Notes...,/content/drive/MyDrive/Thesis_Experiment/Resul...,2026-07-25T03:49:02.436302+00:00,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,5d336e1c7630cd6dec8ecdd8e17fa1af5cd6547d6cf90e...,2682e51a689abc8a2cfb12b07b7c1fe0f450d2acf9e9a5...,,5358150.0,PASS_PROJECT_10_FINAL_PACKAGE_CONSTRUCTED_AND_...



Optional variable columns left blank because they had no safe mapping:


,Column
0,EvaluationBuilds
1,EvaluationFailures
2,EvaluationRows
3,FinalDirectory
4,ModelFits



Final registry validation:


,Check,Expected,Actual,Pass
0,Registry rows,10,10,True
1,Registered project numbers,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",True
2,COMPLETE_AND_FROZEN projects,10,10,True
3,Project 9 registry rows,1,1,True
4,Project 10 registry rows,1,1,True
5,Project 9 name,camunda@camunda-bpm-platform,camunda@camunda-bpm-platform,True
6,Project 9 slug,camunda__camunda-bpm-platform,camunda__camunda-bpm-platform,True
7,Project 9 completion status,COMPLETE_AND_FROZEN,COMPLETE_AND_FROZEN,True
8,Project 9 final-audit status,PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_AND_V...,PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_AND_V...,True
9,Project 9 RawResultsRootSHA256,c31cb45e1354dc1222b82103bd275c21b10dd2ba617112...,c31cb45e1354dc1222b82103bd275c21b10dd2ba617112...,True




=== PROJECTS 9 AND 10 SERIAL REGISTRY FINALISATION V2 RESULT ===

Registry:
Registry rows: 10
Project numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
COMPLETE_AND_FROZEN projects: 10
Main completion-status column: Status
Separate final-audit-status column: FinalAuditStatus
Registry write performed: True
Registry SHA-256 before: a442446f3ca6207b31213fc422fb0c883c96608a8e6a3907da969e326808bee7
Registry SHA-256 after: 847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750

Project 9:
Project: camunda@camunda-bpm-platform
Status: COMPLETE_AND_FROZEN
Final-audit status: PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_AND_VALIDATED_REGISTRY_PENDING

Project 10:
Project: spring-cloud@spring-cloud-dataflow
Status: COMPLETE_AND_FROZEN
Final-audit status: PASS_PROJECT_10_FINAL_PACKAGE_CONSTRUCTED_AND_VALIDATED_REGISTRY_PENDING

Validation:
Package validation failures: 0
Registry validation failures: 0

Finalisation checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/projects_09_10_registry_